# Python Notes 2: Advanced Python

These notes take you from "I can write Python scripts" to "I can design, test and reason about real Python programs".
This is the second of three notebooks:

| Notebook | Topic | Level |
|---|---|---|
| Notebook 1 | Python Fundamentals | Beginner to intermediate |
| **Notebook 2 (this one)** | Advanced Python | Intermediate to advanced |
| Notebook 3 | DSA and Interview Preparation | Interview preparation |

**Before you start:** finish Notebook 1. In particular you need functions (Notebook 1, Part 11),
errors and exceptions (Notebook 1, Part 12) and references and mutability (Notebook 1, Part 10).

**Python version:** the code targets Python 3.12 and newer and was run with Python 3.14.
Newer features are marked in the text, for example "(Python 3.13+)", and guarded so the cells still run on 3.12.

## How to use this notebook

* Read one section, run its cells, then do its **Practice** block before you look at the solutions.
* Change the examples. Break them on purpose. Read the error. Fix it. This is how you really learn.
* Each Part is self-contained: it imports what it needs and does not use variables from other Parts.
* Every Part ends with a **How to debug** section and a **Summary**. Use them for quick revision.

> **Tip:** Parts 1-2 (OOP), 3 (iterators and generators), 4 (decorators) and 7 (standard library)
> are used again and again in Notebook 3 (DSA). Study them carefully.

## Table of contents of Notebook 2

| Part | Title | What it covers |
|---|---|---|
| 1 | OOP - Classes and Objects | classes, objects, `__init__`/`__new__`, attributes, methods, properties, `__str__`/`__repr__` |
| 2 | OOP - Inheritance, Polymorphism and Special Methods | inheritance, `super()`, MRO, ABCs, dunder methods, dataclasses, enums, class design |
| 3 | Iterators and Generators | the iterator protocol, `yield`, generator pipelines |
| 4 | Closures and Decorators | wrapping functions, `functools.wraps`, decorators with arguments |
| 5 | Context Managers | the `with` statement, `@contextmanager`, `ExitStack` |
| 6 | Advanced Language Features | introspection, descriptors, metaclasses, advanced typing, memory |
| 7 | The Standard Library Toolbox | collections, heapq, itertools, functools, datetime, re, logging |
| 8 | Concurrency and Async Programming | threads, processes, the GIL, asyncio |
| 9 | Testing, Code Quality, Security and Performance | unittest, pytest, mock, linters, profiling |
| 10 | Real-World Mini Projects | CLI tool, ETL pipeline, OOP project, capstone practice |

---

# Part 1: OOP - Classes and Objects

**What these notes cover**

* Why object-oriented programming (OOP) exists and when it helps.
* How to write a class, create objects from it and use `self`.
* What `__init__` and `__new__` do, and who calls them.
* The difference between instance attributes and class attributes, and how Python looks up an attribute.
* Instance methods, class methods and static methods, and when to use each.
* Encapsulation: the `_name` convention, `__name` name mangling and properties.
* `__str__` vs `__repr__`, the life of an object, and how to debug class code.

**Before you start:** Notebook 1, Part 11 (Functions) and Part 10 (References, Mutability and Copying).

**Running example:** in this Part we build one class, `Employee`, step by step.
Each section adds one new idea to it. Small extra examples appear where they teach an idea better.

## 1.1 Why object-oriented programming?

Imagine you write a payroll program with only variables and functions.
Each employee has a name, a department and a salary. You might store them like this:

```python
names = ["Asha", "Ravi"]
depts = ["Engineering", "Sales"]
salaries = [85000, 60000]
```

This works for a while. But the data for one person is spread over three lists.
Every function needs all three lists. If you add a new field (say, email), you must change many places.
It is easy to update one list and forget the others.

**Object-oriented programming (OOP)** solves this by putting the **data** and the **functions that work on
that data** together in one unit, called an **object**.

> **In simple words:** a **class** is a blueprint (like the plan of a house).
> An **object** is one real thing built from that blueprint (one actual house).
> You can build many houses from one plan. Each house has its own furniture, but all follow the same plan.

### The main OOP terms

| Term | Meaning in simple words |
|---|---|
| **Class** | A blueprint. It describes what data and behaviour its objects have. |
| **Object / instance** | One concrete thing made from a class, e.g. `asha = Employee("Asha", ...)`. |
| **Attribute** | A variable that belongs to an object or a class, e.g. `asha.name`. |
| **Method** | A function defined inside a class, called on an object, e.g. `asha.describe()`. |
| **Encapsulation** | Keeping data and the code that manages it together, and controlling access. |
| **Inheritance** | A class reusing and extending another class (Part 2). |
| **Polymorphism** | Different classes answering the same method call in their own way (Part 2). |
| **Abstraction** | Showing *what* an object does and hiding *how* it does it. |

> **Note:** "Instance" and "object" mean the same thing here. We say "an instance of `Employee`"
> to stress which class it was made from.

### Everything in Python is already an object

You have used objects since your first line of Python. A string is an object of class `str`.
Its methods (`upper`, `split`, ...) are functions that belong to that class.
Writing your own class means creating your own type, just like `str` or `list`.

In [1]:
# Built-in values are objects: each has a class (type) and methods
text = "hello"
print(type(text))              # <class 'str'>
print(text.upper())            # HELLO
print(str.upper(text))         # HELLO  (the same call, written through the class)

numbers = [3, 1, 2]
numbers.sort()                 # a method that changes this list object
print(numbers)                 # [1, 2, 3]
print(isinstance(numbers, list))   # True

<class 'str'>
HELLO
HELLO
[1, 2, 3]
True


### When is OOP a good choice?

* You have **things with state** (data that changes over time): a bank account, a shopping cart, a game player.
* Several functions keep passing the same group of values around.
* You need many similar objects that behave the same way (100 employees, 1,000 orders).
* You want to model "kinds of things" that share behaviour (Part 2: inheritance and polymorphism).

When is it **not** needed? A short script, or a pure calculation (input in, result out) is often clearer
as plain functions. Python lets you mix both styles.

> **Interview tip:** Interviewers like to hear that you choose OOP for a reason (state plus behaviour),
> not out of habit. "Not everything needs a class" is a good, mature answer.

> **Key points**
> * OOP groups data (attributes) and behaviour (methods) into objects.
> * A class is a blueprint; an object (instance) is one thing built from it.
> * Every value in Python is already an object of some class.
> * Use classes for things with state and behaviour; use plain functions for simple calculations.

---

## 1.2 Your first class: objects, attributes, methods and `self`

A class is created with the `class` keyword. Class names use **CapWords** (`Employee`, `BankAccount`), as PEP 8 says.

The smallest possible class has an empty body. `pass` means "nothing here".
Calling the class like a function, `Employee()`, creates a new object.

In [2]:
# The smallest class, and two objects made from it
class Employee:
    pass


asha = Employee()
ravi = Employee()

print(type(asha).__name__)         # Employee
print(isinstance(asha, Employee))  # True
print(asha is ravi)                # False  (two different objects)

Employee
True
False


### Attributes are names attached to an object

You can attach attributes to an object with dot notation: `object.name = value`.
Each object keeps its own attributes in its own small dictionary, available as `obj.__dict__` or `vars(obj)`.

The cell below does this "by hand" only to show the idea. In real code you set attributes in `__init__`
(section 1.3), so every object gets the same attributes automatically.

In [3]:
# Attaching attributes by hand (only to see how objects store data)
class Employee:
    pass


asha = Employee()
asha.name = "Asha"
asha.dept = "Engineering"

ravi = Employee()
ravi.name = "Ravi"                   # ravi has no 'dept' yet: easy to forget one!

print(asha.name, asha.dept)          # Asha Engineering
print(vars(asha))                    # {'name': 'Asha', 'dept': 'Engineering'}
print(vars(ravi))                    # {'name': 'Ravi'}

Asha Engineering
{'name': 'Asha', 'dept': 'Engineering'}
{'name': 'Ravi'}


### Methods and `self`

A **method** is a function written inside a class. Its first parameter is the object the method is
called on. By convention this parameter is named **`self`**.

`self` is not a keyword. It is just the first parameter. Python fills it in for you:
when you write `asha.describe()`, Python runs `Employee.describe(asha)`.

```text
   asha.describe()
        |
        v
   +---------------------------------------+
   | Python finds 'describe' on the class  |
   | Employee (not on the object itself)   |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | It calls Employee.describe(asha)      |
   | so inside the method: self is asha    |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | The method reads self.name, self.dept |
   | which are asha's own attributes       |
   +---------------------------------------+
```

In [4]:
# A method uses self to reach the data of the object it was called on
class Employee:
    def describe(self):
        return f"{self.name} works in {self.dept}"


asha = Employee()
asha.name = "Asha"
asha.dept = "Engineering"

print(asha.describe())             # Asha works in Engineering
print(Employee.describe(asha))     # Asha works in Engineering  (what Python really does)

Asha works in Engineering
Asha works in Engineering


### Bound methods

`asha.describe` (without brackets) is a **bound method**: the function plus the object it is tied to.
You can store it in a variable and call it later. It still remembers `asha`.

In [5]:
# A bound method remembers its object
class Employee:
    def describe(self):
        return f"{self.name} works in {self.dept}"


asha = Employee()
asha.name, asha.dept = "Asha", "Engineering"

say = asha.describe                 # no () -> we get the bound method, we do not call it
print(say())                        # Asha works in Engineering
print(say.__self__ is asha)         # True  (the object is stored inside the bound method)

Asha works in Engineering
True


### Common mistakes with `self`

* **Forgetting `self` in the method definition.** Python still passes the object, so the call gets one
  argument too many: `TypeError: Employee.describe() takes 0 positional arguments but 1 was given`.
* **Forgetting `self.` when you read or write an attribute.** `name = "Asha"` inside a method creates a
  local variable that disappears when the method ends. Use `self.name = "Asha"`.

In [6]:
# Common mistake: a method without self
class Broken:
    def hello():                     # missing self
        return "hi"


try:
    Broken().hello()
except TypeError as e:
    print("TypeError:", e)           # TypeError: Broken.hello() takes 0 positional arguments but 1 was given


# Fix: always write self as the first parameter
class Fixed:
    def hello(self):
        return "hi"


print(Fixed().hello())               # hi

TypeError: Broken.hello() takes 0 positional arguments but 1 was given
hi


In [7]:
# Common mistake: forgetting "self." creates a local variable, not an attribute
class Counter:
    def start(self):
        count = 0                    # local variable; lost when start() returns

    def start_fixed(self):
        self.count = 0               # attribute stored on the object


c = Counter()
c.start()
print(hasattr(c, "count"))           # False
c.start_fixed()
print(c.count)                       # 0

False
0


> **Key points**
> * `class Name:` creates a class; `Name()` creates a new object.
> * Objects store their own attributes in `obj.__dict__` (also `vars(obj)`).
> * A method is a function in the class. `obj.method()` means `Class.method(obj)`.
> * `self` is the first parameter by convention. It is the object the method was called on.
> * Always use `self.attr` to read or write the object's data inside a method.

---

## 1.3 `__init__` and `__new__`: how an object is created

Setting attributes by hand is error-prone. We want every `Employee` to start with a name and a department.
The special method **`__init__`** does exactly that. Python calls it **automatically** right after a new object is made.

Names with two underscores on both sides, like `__init__`, are called **dunder** ("double underscore")
or **special** methods. Python calls them for you at special moments.

In [8]:
# The running example, step 1: __init__ gives every Employee the same attributes
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name             # instance attribute: one value per object
        self.dept = dept
        self.salary = salary

    def describe(self):
        return f"{self.name} works in {self.dept}"


asha = Employee("Asha", "Engineering", 85000)
ravi = Employee("Ravi", "Sales", 60000)

print(asha.describe())               # Asha works in Engineering
print(ravi.describe())               # Ravi works in Sales
print(vars(ravi))                    # {'name': 'Ravi', 'dept': 'Sales', 'salary': 60000}

Asha works in Engineering
Ravi works in Sales
{'name': 'Ravi', 'dept': 'Sales', 'salary': 60000}


### What really happens when you call `Employee(...)`

Creating an object is a **two-step process**:

1. **`__new__`** *creates* the new, empty object (it allocates it). It receives the class as `cls`.
2. **`__init__`** *initialises* that object: it fills in the attributes. It receives the object as `self`.

Python runs both steps for you. You never call `__init__` yourself in normal code.

```text
        Employee("Asha", "Engineering", 85000)
                         |
                         v
        +----------------------------------+
        | 1. obj = Employee.__new__(       |
        |        Employee, ...)            |
        |    creates an empty object       |
        +----------------+-----------------+
                         |
                         v
        +----------------------------------+    No    +------------------+
        | Is obj an instance of Employee?  |--------->| skip __init__,   |
        +----------------+-----------------+          | return obj       |
                         | Yes                        +------------------+
                         v
        +----------------------------------+
        | 2. obj.__init__("Asha", ...)     |
        |    sets obj.name, obj.dept, ...  |
        |    must return None              |
        +----------------+-----------------+
                         |
                         v
        +----------------------------------+
        | 3. the call returns obj          |
        +----------------------------------+
```

> **In simple words:** `__new__` builds the empty house; `__init__` moves the furniture in.
> People often call `__init__` "the constructor". More exactly, it is the **initialiser**.

In [9]:
# Watch the order: __new__ runs first, then __init__
class Demo:
    def __new__(cls, value):
        print("1. __new__ creates the object")
        return super().__new__(cls)          # ask object.__new__ for a fresh empty object

    def __init__(self, value):
        print("2. __init__ sets value =", value)
        self.value = value


d = Demo(42)
# 1. __new__ creates the object
# 2. __init__ sets value = 42
print(d.value)                               # 42

1. __new__ creates the object
2. __init__ sets value = 42
42


### When do you need `__new__`?

Almost never. Use `__init__` for normal classes. `__new__` is needed in rare cases, for example:

* Subclassing an **immutable** built-in type (`str`, `int`, `tuple`). The value must be decided when the
  object is created, because it cannot be changed afterwards.
* Controlling *which* object is returned (caching, or the singleton pattern).

In [10]:
# A real use of __new__: a str subclass whose value is decided at creation time
class UpperStr(str):
    def __new__(cls, text):
        return super().__new__(cls, text.upper())   # a str cannot change later, so fix it now


code = UpperStr("abc-123")
print(code)                          # ABC-123
print(isinstance(code, str))         # True

ABC-123
True


### `__init__` must return `None`

`__init__` only fills in the object. If it returns a value, Python raises a `TypeError`.

In [11]:
# __init__ must not return a value
class Bad:
    def __init__(self):
        return 5


try:
    Bad()
except TypeError as e:
    print("TypeError:", e)           # TypeError: __init__() should return None, not 'int'

TypeError: __init__() should return None, not 'int'


### Default values and validation in `__init__`

`__init__` is a normal function, so it can have default parameter values and can check its inputs.
Raise an exception early if the data is invalid. Then a broken object can never exist.

In [12]:
# __init__ with a default value and simple validation
class Employee:
    def __init__(self, name, dept, salary=50000):
        if not name:
            raise ValueError("name must not be empty")
        self.name = name
        self.dept = dept
        self.salary = salary


meera = Employee("Meera", "HR")      # salary uses the default
print(meera.salary)                  # 50000

try:
    Employee("", "HR")
except ValueError as e:
    print("ValueError:", e)          # ValueError: name must not be empty

try:
    Employee("Ravi")                 # dept is missing
except TypeError as e:
    print("TypeError:", e)           # TypeError: Employee.__init__() missing 1 required positional argument: 'dept'

50000
ValueError: name must not be empty
TypeError: Employee.__init__() missing 1 required positional argument: 'dept'


> **Common mistake:** calling `obj.__init__(...)` yourself "to create" or "to reset" an object.
> Python already called it. To make a new object, call the class: `Employee("Asha", "HR")`.
> To change data later, write a normal method (for example `update_dept`).

> **Common mistake:** misspelling it as `__int__` or `_init_`. Python then never calls it, and you get
> `TypeError: Employee() takes no arguments`.

> **Key points**
> * `Employee(...)` runs `__new__` (create) and then `__init__` (initialise) for you.
> * Put all instance attributes in `__init__`, so every object has the same shape.
> * `__init__` must return `None`. Validate inputs there and raise early.
> * You almost never need `__new__`; it is for immutable subclasses and controlling object creation.

### Practice 1.3: Classes, objects and `__init__`

**A. Predict the output**

1. What does this print?
   ```python
   class Pen:
       def __init__(self, color):
           self.color = color

   a = Pen("blue")
   b = Pen("red")
   b.color = "green"
   print(a.color, b.color)
   ```
2. What does this print?
   ```python
   class Box:
       def __init__(self):
           size = 10

   print(hasattr(Box(), "size"))
   ```
3. What does this print?
   ```python
   class Greeter:
       def greet(self, name):
           return "Hi " + name

   g = Greeter()
   print(g.greet("Asha") == Greeter.greet(g, "Asha"))
   ```
4. Which error does `Pen()` raise for the `Pen` class of question 1, and why?

**B. Write code**

5. Write a class `Book` with `__init__(self, title, author, pages)` and a method `summary()` that
   returns `"<title> by <author>, <pages> pages"`.
6. Write a class `Rectangle` with `width` and `height` and methods `area()` and `perimeter()`.
   Raise `ValueError` in `__init__` if a side is not positive.
7. Write a class `Counter` with `increment(step=1)`, `reset()` and an attribute `value` that starts at 0.
   `increment` should return the new value.
8. **[Hard]** Write a class `Temperature` that stores Celsius and has methods `to_fahrenheit()` and
   `to_kelvin()`. Reject values below -273.15 with `ValueError`.

**C. Find and fix the bug**

9. This should print `Ravi`, but it fails. Why? Fix it.
   ```python
   class User:
       def __init__(self, name):
           name = name

       def show():
           return self.name

   print(User("Ravi").show())
   ```

**D. Interview questions**

10. What is the difference between a class and an object?
11. What is `self`? Is it a keyword?
12. **[Hard]** What is the difference between `__new__` and `__init__`? When would you write `__new__`?

### Solutions 1.3

1. `blue green`. Each object has its own `color` attribute. Changing `b.color` does not touch `a`.
2. `False`. `size = 10` creates a local variable inside `__init__`. It is not stored on the object.
3. `True`. `g.greet("Asha")` is translated into `Greeter.greet(g, "Asha")`.
4. `TypeError: Pen.__init__() missing 1 required positional argument: 'color'`.
   The arguments of `Pen(...)` are passed to `__init__`, which needs `color`.
9. Two bugs. `name = name` assigns to a local variable; it must be `self.name = name`.
   `show()` is missing the `self` parameter, so the call gives `TypeError: User.show() takes 0 positional
   arguments but 1 was given`. Fixed code is in the cell below.
10. A class is the blueprint (the type). An object is one concrete value made from it, with its own data.
    `Employee` is a class; `Employee("Asha", "HR", 50000)` is an object.
11. `self` is the first parameter of an instance method. Python passes the object the method was called on.
    It is only a naming convention, not a keyword, but always use the name `self`.
12. `__new__` is a static-like method that receives the class and **creates and returns** the new object.
    `__init__` receives that object and **initialises** it (returns `None`). Write `__new__` only when
    you subclass an immutable type (such as `str`, `int`, `tuple`) or need to control which object is
    returned (caching, singleton). Python calls both for you.

In [13]:
# Solutions 1.3 (B and C)

# Q5
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def summary(self):
        return f"{self.title} by {self.author}, {self.pages} pages"


assert Book("Dune", "Frank Herbert", 412).summary() == "Dune by Frank Herbert, 412 pages"


# Q6
class Rectangle:
    def __init__(self, width, height):
        if width <= 0 or height <= 0:
            raise ValueError("sides must be positive")
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


r = Rectangle(3, 4)
assert (r.area(), r.perimeter()) == (12, 14)
for bad_sides in [(0, 4), (3, -1)]:
    try:
        Rectangle(*bad_sides)
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError")


# Q7
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self, step=1):
        self.value += step
        return self.value

    def reset(self):
        self.value = 0


c = Counter()
assert c.increment() == 1
assert c.increment(5) == 6
c.reset()
assert c.value == 0


# Q8
class Temperature:
    ABSOLUTE_ZERO = -273.15

    def __init__(self, celsius):
        if celsius < self.ABSOLUTE_ZERO:
            raise ValueError("below absolute zero")
        self.celsius = celsius

    def to_fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    def to_kelvin(self):
        return self.celsius + 273.15


t = Temperature(100)
assert t.to_fahrenheit() == 212
assert abs(t.to_kelvin() - 373.15) < 1e-9
try:
    Temperature(-300)
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")


# Q9 fixed
class User:
    def __init__(self, name):
        self.name = name

    def show(self):
        return self.name


assert User("Ravi").show() == "Ravi"

print("Practice 1.3: all checks passed")

Practice 1.3: all checks passed


---

## 1.4 Instance attributes vs class attributes

There are two places where an attribute can live:

* An **instance attribute** belongs to one object. It is set with `self.name = ...` (usually in `__init__`).
  It is stored in that object's own dictionary, `obj.__dict__`.
* A **class attribute** belongs to the class. It is written directly in the class body.
  It is stored once, in `Employee.__dict__`, and **shared** by all objects of the class.

Use a class attribute for data that is the same for every object (a company name, a constant, a counter).
Use instance attributes for data that differs per object (name, salary).

In [14]:
# The running example, step 2: a class attribute shared by all employees
class Employee:
    company = "Acme Corp"            # class attribute: stored once, on the class

    def __init__(self, name, dept, salary):
        self.name = name             # instance attributes: stored on each object
        self.dept = dept
        self.salary = salary


asha = Employee("Asha", "Engineering", 85000)
ravi = Employee("Ravi", "Sales", 60000)

print(asha.company, ravi.company)              # Acme Corp Acme Corp
print(vars(asha))                              # {'name': 'Asha', 'dept': 'Engineering', 'salary': 85000}
print("company" in vars(asha))                 # False  (not stored on the object)
print("company" in Employee.__dict__)          # True   (stored on the class)

Acme Corp Acme Corp
{'name': 'Asha', 'dept': 'Engineering', 'salary': 85000}
False
True


### The memory picture

The class attribute is stored **only once**, inside the class object. The instances do **not** contain a copy.
When you read `asha.company`, Python does not find it in `asha`, so it continues to the class.

```text
   +-----------------------------------------------+
   | class object: Employee                        |
   |   Employee.__dict__                           |
   |     company  -> 'Acme Corp'                   |
   |     __init__ -> <function>                    |
   +-----------------------------------------------+
            ^                              ^
            | (lookup continues            | (lookup continues
            |  to the class)               |  to the class)
   +------------------------+     +------------------------+
   | object: asha           |     | object: ravi           |
   |   asha.__dict__        |     |   ravi.__dict__        |
   |     name   -> 'Asha'   |     |     name   -> 'Ravi'   |
   |     dept   -> 'Eng...' |     |     dept   -> 'Sales'  |
   |     salary -> 85000    |     |     salary -> 60000    |
   |   (no 'company' here)  |     |   (no 'company' here)  |
   +------------------------+     +------------------------+
```

Every object also remembers its class (`asha.__class__ is Employee`). That link is what the arrows show.

### How Python looks up `obj.attr` (simplified)

```text
            obj.attr  (reading)
                 |
                 v
   +------------------------------+   Yes   +----------------------+
   | 'attr' in obj.__dict__ ?     |-------->| return that value    |
   +--------------+---------------+         +----------------------+
                  | No
                  v
   +------------------------------+   Yes   +----------------------+
   | 'attr' in the class, or in   |-------->| return it (a method  |
   | a parent class (in MRO       |         | becomes a bound      |
   | order, see Part 2) ?         |         | method)              |
   +--------------+---------------+         +----------------------+
                  | No
                  v
   +------------------------------+
   | raise AttributeError         |
   +------------------------------+
```

> **Note:** this is the simple version and it is right for normal attributes. Properties (section 1.8)
> are found on the class and take priority over the instance dictionary. The full rules
> (descriptors, `__getattr__`) are in Part 6.

### Reading through an instance vs writing through an instance

* **Reading** `asha.company` follows the lookup chain above and finds the class value.
* **Writing** `asha.company = "Other"` always writes into `asha.__dict__`. It creates a **new instance
  attribute** that *hides* (shadows) the class attribute for `asha` only. The class is unchanged.
* To change the shared value for everyone, assign on the class: `Employee.company = "..."`.

In [15]:
# Writing through an instance creates a new instance attribute (shadowing)
class Employee:
    company = "Acme Corp"

    def __init__(self, name):
        self.name = name


asha = Employee("Asha")
ravi = Employee("Ravi")

asha.company = "Globex"                        # creates asha.__dict__['company']
print(asha.company)                            # Globex     (found in asha first)
print(ravi.company)                            # Acme Corp  (ravi still sees the class)
print(Employee.company)                        # Acme Corp  (the class is unchanged)
print(vars(asha))                              # {'name': 'Asha', 'company': 'Globex'}

Globex
Acme Corp
Acme Corp
{'name': 'Asha', 'company': 'Globex'}


In [16]:
# Changing the class attribute affects every object that does not shadow it
Employee.company = "Initech"
print(ravi.company)                            # Initech
print(asha.company)                            # Globex  (asha's own attribute still hides it)

del asha.company                               # remove the instance attribute ...
print(asha.company)                            # Initech (... now the class value is visible again)

Initech
Globex
Initech


### Class attributes as constants and counters

Class attributes are a good home for **constants** that belong to the class (written in `UPPER_CASE`)
and for **class-wide counters**. A method can read a class constant through `self`,
because lookup falls back to the class.

To *change* a class-wide value, do not write `self.counter += 1`: that creates an instance attribute.
Use a class method (section 1.6), or in `__init__` the class itself (`type(self)`), as explained there.

In [17]:
# A class constant read through self
class Employee:
    company = "Acme Corp"
    RAISE_PERCENT = 5                    # constant shared by all employees

    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def apply_raise(self):
        # self.RAISE_PERCENT is found on the class
        self.salary = round(self.salary * (100 + self.RAISE_PERCENT) / 100)


asha = Employee("Asha", 80000)
asha.apply_raise()
print(asha.salary)                      # 84000

84000


> **Key points**
> * Instance attributes live in `obj.__dict__`; class attributes live once in `Class.__dict__`.
> * Reading `obj.x` looks in the object first, then in the class (and its parents).
> * Writing `obj.x = ...` always creates or changes an *instance* attribute (it can shadow the class one).
> * Change a shared value on the class (`Class.x = ...`) or through a class method.

---

## 1.5 The mutable class attribute trap

The "writing creates an instance attribute" rule applies to **assignment** (`obj.x = ...`).
**Mutating** a shared object (for example `obj.items.append(...)`) is not assignment.
It changes the one shared list stored on the class, so **every** object sees the change.

This is the same idea as aliasing in Notebook 1, Part 10: two names pointing to one list.

In [18]:
# Common mistake: a list as a class attribute is shared by every object
class Team:
    members = []                       # ONE list, stored on the class

    def add(self, name):
        self.members.append(name)      # mutation: changes the shared list


backend = Team()
frontend = Team()
backend.add("Asha")

print(frontend.members)                # ['Asha']  (surprise: frontend sees it too)
print(backend.members is frontend.members is Team.members)   # True

['Asha']
True


In [19]:
# Fix: create the list in __init__ so each object gets its own list
class Team:
    def __init__(self):
        self.members = []              # a new list for every object

    def add(self, name):
        self.members.append(name)


backend = Team()
frontend = Team()
backend.add("Asha")

print(backend.members)                 # ['Asha']
print(frontend.members)                # []

['Asha']
[]


```text
   Trap (class attribute)              Fix (instance attribute)

   Team.members --> [ 'Asha' ]         backend.members  --> [ 'Asha' ]
        ^      ^                       frontend.members --> [ ]
        |      |
     backend frontend                  (two separate list objects)
   (both find the same list)
```

> **Warning:** the same trap happens with a mutable default argument, `def __init__(self, members=[])`.
> Use `members=None` and then `self.members = [] if members is None else list(members)`.

In [20]:
# The mutable default argument trap inside __init__, and its fix
class Team:
    def __init__(self, members=None):
        # None means "not given"; build a fresh list for every object
        self.members = [] if members is None else list(members)


a = Team()
b = Team()
a.members.append("Ravi")
print(b.members)                       # []
print(Team(["Meera"]).members)         # ['Meera']

[]
['Meera']


> **Key points**
> * A mutable class attribute (list, dict, set) is shared by all objects.
> * `obj.items.append(x)` mutates the shared object; it does not create an instance attribute.
> * Create per-object containers in `__init__`. Use `None` as the default, never `[]`.

### Practice 1.5: Instance and class attributes

**A. Predict the output**

1. What does this print?
   ```python
   class Dog:
       species = "Canine"

   d1 = Dog()
   d2 = Dog()
   Dog.species = "Wolf"
   d1.species = "Fox"
   print(d1.species, d2.species, Dog.species)
   ```
2. What does this print?
   ```python
   class Cart:
       items = []

       def add(self, item):
           self.items.append(item)

   a = Cart()
   b = Cart()
   a.add("pen")
   b.add("book")
   print(len(a.items), len(Cart.items))
   ```
3. What does this print?
   ```python
   class Config:
       debug = False

   c = Config()
   c.debug = True
   del c.debug
   print(c.debug, "debug" in vars(c))
   ```
4. What does this print?
   ```python
   class Score:
       total = 0

       def add(self, points):
           self.total += points

   s1 = Score()
   s1.add(5)
   s2 = Score()
   print(s1.total, s2.total, Score.total)
   ```

**B. Write code**

5. Write a class `Product` with a class attribute `TAX_RATE = 0.18` and instance attributes `name` and
   `price`. Add `price_with_tax()` (rounded to 2 decimals). Changing `Product.TAX_RATE` must affect all products.
6. Write a class `Classroom` whose `students` list is separate for each object. Add `enroll(name)` and
   accept an optional starting list in `__init__` (copy it, so the caller's list is not changed later).
7. **[Hard]** Write a function `where_is(obj, name)` that returns `"instance"`, `"class"` or `"missing"`
   depending on where the attribute is stored. Only look at `vars(obj)` and `vars(type(obj))`.

**C. Find and fix the bug**

8. Each `Library` should have its own books, but all libraries share them. Fix it.
   ```python
   class Library:
       books = {}

       def add(self, title, copies):
           self.books[title] = copies
   ```
9. The developer wanted to count all orders, but `Order.count` stays 0. Why? Fix it.
   ```python
   class Order:
       count = 0

       def __init__(self, item):
           self.item = item
           self.count += 1
   ```

**D. Interview questions**

10. Where are instance attributes and class attributes stored in memory?
11. Why does `obj.x = 5` never change a class attribute, while `obj.items.append(5)` can?
12. **[Hard]** Describe the steps Python follows to find `obj.name` (simple version).

### Solutions 1.5

1. `Fox Wolf Wolf`. `Dog.species = "Wolf"` changes the class. `d1.species = "Fox"` creates an instance
   attribute on `d1` only. `d2` has no own attribute, so it sees the class value.
2. `2 2`. `items` is one list shared by the class and both objects.
3. `False False`. After `del c.debug` the instance attribute is gone, so the class value `False` shows again.
4. `5 0 0`. `self.total += points` means `self.total = self.total + points`. It reads the class value 0,
   adds 5, and **assigns** the result to a new instance attribute on `s1`. The class stays 0.
8. `books = {}` is a class attribute shared by all libraries. Create it in `__init__`:
   `self.books = {}`.
9. `self.count += 1` creates an instance attribute on each order (same as question 4). Increase the class
   attribute instead: `type(self).count += 1` (or `Order.count += 1`, or better a class method, see section 1.6).
10. Instance attributes are stored in each object's own dictionary, `obj.__dict__`.
    Class attributes are stored once, in the class's dictionary, `Class.__dict__`.
11. Assignment through an object always writes to the object's own dictionary. `append` is not an
    assignment: Python first *reads* `obj.items` (finding the shared list on the class) and then mutates
    that list in place.
12. Python looks in `obj.__dict__`; if not found, it looks in the class, then in the parent classes in MRO
    order; if still not found, it raises `AttributeError`. (Properties and other descriptors are found on
    the class first; see Part 6 for the complete rules.)

In [21]:
# Solutions 1.5 (B and C) and checks for A

# A1 and A4 checks
class Dog:
    species = "Canine"


d1, d2 = Dog(), Dog()
Dog.species = "Wolf"
d1.species = "Fox"
assert (d1.species, d2.species, Dog.species) == ("Fox", "Wolf", "Wolf")


class Score:
    total = 0

    def add(self, points):
        self.total += points


s1 = Score()
s1.add(5)
s2 = Score()
assert (s1.total, s2.total, Score.total) == (5, 0, 0)


# Q5
class Product:
    TAX_RATE = 0.18

    def __init__(self, name, price):
        self.name = name
        self.price = price

    def price_with_tax(self):
        return round(self.price * (1 + self.TAX_RATE), 2)


pen = Product("pen", 100)
assert pen.price_with_tax() == 118.0
Product.TAX_RATE = 0.05
assert pen.price_with_tax() == 105.0
Product.TAX_RATE = 0.18                       # restore


# Q6
class Classroom:
    def __init__(self, students=None):
        self.students = [] if students is None else list(students)   # copy the caller's list

    def enroll(self, name):
        self.students.append(name)


start = ["Asha"]
room_a = Classroom(start)
room_b = Classroom()
room_a.enroll("Ravi")
assert room_a.students == ["Asha", "Ravi"]
assert room_b.students == []
assert start == ["Asha"]                      # caller's list unchanged


# Q7
def where_is(obj, name):
    if name in vars(obj):
        return "instance"
    if name in vars(type(obj)):
        return "class"
    return "missing"


assert where_is(d1, "species") == "instance"
assert where_is(d2, "species") == "class"
assert where_is(d2, "colour") == "missing"


# Q8 fixed
class Library:
    def __init__(self):
        self.books = {}

    def add(self, title, copies):
        self.books[title] = copies


lib1, lib2 = Library(), Library()
lib1.add("Dune", 2)
assert lib2.books == {}


# Q9 fixed
class Order:
    count = 0

    def __init__(self, item):
        self.item = item
        type(self).count += 1                 # change the class attribute, not the instance


Order("pen")
Order("book")
assert Order.count == 2

print("Practice 1.5: all checks passed")

Practice 1.5: all checks passed


---

## 1.6 Instance methods, class methods and static methods

A class can have three kinds of methods. They differ in **what they receive as the first argument**.

| Kind | Decorator | First parameter | Can use | Typical use |
|---|---|---|---|---|
| Instance method | none | `self` (the object) | object data and class data | normal behaviour |
| Class method | `@classmethod` | `cls` (the class) | class data | alternative constructors, class-wide state |
| Static method | `@staticmethod` | none | only its own arguments | helper that belongs to the class topic |

A **decorator** is the `@name` line above a function. It changes how the function behaves.
Here it tells Python what to pass as the first argument. Decorators are explained fully in Part 4.

### Instance methods

This is the normal kind. It receives the object as `self`. Use it whenever the method needs the data of
one particular object. Every method you have written so far was an instance method.

### Class methods

A class method receives **the class** as `cls`. It can be called on the class (`Employee.headcount()`)
or on an object (Python then passes the object's class). Use it for two jobs:

1. **Reading or changing class-wide state**, such as a counter of all employees.
2. **Alternative constructors**: extra ways to build an object, like `Employee.from_string("Asha,HR,50000")`.
   They end with `return cls(...)`.

In [22]:
# The running example, step 3: class-wide state managed by class methods
class Employee:
    company = "Acme Corp"
    _count = 0                                 # number of employees created so far

    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary
        type(self)._register()                 # update class-wide state through the class

    @classmethod
    def _register(cls):
        cls._count += 1

    @classmethod
    def headcount(cls):
        return cls._count

    @classmethod
    def rename_company(cls, new_name):
        cls.company = new_name                 # changes the shared value for everyone


asha = Employee("Asha", "Engineering", 85000)
ravi = Employee("Ravi", "Sales", 60000)

print(Employee.headcount())                    # 2
Employee.rename_company("Acme Global")
print(asha.company, ravi.company)              # Acme Global Acme Global

2
Acme Global Acme Global


> **Common mistake:** changing a class attribute from an instance method with the hard-coded class name,
> for example `Employee.company = new_name` inside `def change(self, new_name)`.
> It works, but it hides the intent (it looks like it changes one object) and it always changes `Employee`,
> even when called on a subclass. Use a `@classmethod` and `cls`, as above.

### Alternative constructors

Sometimes data arrives in a different shape: a CSV line, a dictionary from JSON.
Instead of parsing it outside the class every time, give the class a named constructor.

It is a class method that converts the input and then calls `cls(...)`. Using `cls` (and not `Employee`)
matters: when a subclass calls it, it builds an object of the subclass (Part 2).

In [23]:
# Alternative constructors: build an Employee from other formats
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    @classmethod
    def from_csv(cls, line):
        name, dept, salary = line.split(",")
        return cls(name.strip(), dept.strip(), int(salary))

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["dept"], data["salary"])


meera = Employee.from_csv("Meera, HR, 52000")
print(meera.name, meera.dept, meera.salary)                 # Meera HR 52000

kiran = Employee.from_dict({"name": "Kiran", "dept": "Sales", "salary": 58000})
print(kiran.name, type(kiran).__name__)                     # Kiran Employee

Meera HR 52000
Kiran Employee


The standard library uses this pattern a lot: `dict.fromkeys(...)`, `datetime.fromisoformat(...)`,
`int.from_bytes(...)`.

In [24]:
# Alternative constructors you already know from the standard library
from datetime import date

print(dict.fromkeys(["a", "b"], 0))                         # {'a': 0, 'b': 0}
print(date.fromisoformat("2026-01-15"))                     # 2026-01-15
print(int.from_bytes(b"\x01\x00", "big"))                   # 256

{'a': 0, 'b': 0}
2026-01-15
256


### Static methods

A static method receives **nothing automatically**: no `self`, no `cls`. It is a plain function that
lives inside the class because it belongs to the class's topic. Use it for small helpers such as
validation or unit conversion.

In [25]:
# A static method: a helper that needs no object and no class
class Employee:
    def __init__(self, name, dept, salary):
        if not Employee.is_valid_salary(salary):
            raise ValueError(f"invalid salary: {salary}")
        self.name = name
        self.dept = dept
        self.salary = salary

    @staticmethod
    def is_valid_salary(amount):
        return isinstance(amount, int) and amount > 0


print(Employee.is_valid_salary(50000))       # True
print(Employee.is_valid_salary(-5))          # False

asha = Employee("Asha", "Engineering", 85000)
print(asha.is_valid_salary(10))              # True  (can also be called on an object)

try:
    Employee("Ravi", "Sales", 0)
except ValueError as e:
    print("ValueError:", e)                  # ValueError: invalid salary: 0

True
False
True
ValueError: invalid salary: 0


### Which kind should I use?

```text
                 +-------------------------------+
                 | Does the method need the data |   Yes
                 | of ONE particular object?     |---------> instance method (self)
                 +---------------+---------------+
                                 | No
                                 v
                 +-------------------------------+
                 | Does it need the class        |   Yes
                 | (class data, or cls(...) to   |---------> @classmethod (cls)
                 | build an object)?             |
                 +---------------+---------------+
                                 | No
                                 v
                 +-------------------------------+
                 | Is it clearly part of this    |   Yes
                 | class's topic?                |---------> @staticmethod
                 +---------------+---------------+
                                 | No
                                 v
                        a normal module-level function
```

> **Tip:** if a static method is not really related to the class, a plain function in the module is
> often simpler and easier to test.

In [26]:
# All three kinds side by side; see what each one receives
class Demo:
    def instance_method(self):
        return f"instance method got {type(self).__name__} object"

    @classmethod
    def class_method(cls):
        return f"class method got class {cls.__name__}"

    @staticmethod
    def static_method():
        return "static method got nothing"


d = Demo()
print(d.instance_method())      # instance method got Demo object
print(Demo.class_method())      # class method got class Demo
print(d.class_method())         # class method got class Demo  (called on an object: still gets the class)
print(Demo.static_method())     # static method got nothing

instance method got Demo object
class method got class Demo
class method got class Demo
static method got nothing


> **Key points**
> * Instance methods get `self` and work with one object's data.
> * Class methods get `cls`. Use them for class-wide state and alternative constructors (`return cls(...)`).
> * Static methods get nothing automatically. Use them for small helpers related to the class.
> * Change class attributes through `cls` in a class method, not through a hard-coded class name in an instance method.

### Practice 1.6: Kinds of methods

**A. Predict the output**

1. What does this print?
   ```python
   class Pizza:
       size = "medium"

       @classmethod
       def large(cls):
           cls.size = "large"

   p = Pizza()
   p.large()
   print(Pizza.size, p.size)
   ```
2. What does this print?
   ```python
   class MathTools:
       @staticmethod
       def double(x):
           return x * 2

   print(MathTools.double(4), MathTools().double("ab"))
   ```
3. What does this print?
   ```python
   class Animal:
       @classmethod
       def create(cls):
           return cls()

   class Cat(Animal):
       pass

   print(type(Cat.create()).__name__)
   ```

**B. Write code**

4. Write a class `Date` with `day`, `month`, `year` and an alternative constructor
   `from_string("15-01-2026")` (format DD-MM-YYYY). Add a static method `is_leap(year)`.
5. Write a class `Ticket` that gives every new ticket a unique, increasing `number`, starting at 1.
   Add a class method `issued()` that returns how many tickets were created.
6. Write a class `Converter` with static methods `km_to_miles(km)` and `miles_to_km(miles)`
   (1 mile = 1.609344 km). Round results to 3 decimals.
7. **[Hard]** Write a class `Color` with `r`, `g`, `b` and two alternative constructors:
   `from_hex("#ff8000")` and `from_tuple((255, 128, 0))`. Add `to_hex()` that returns lowercase hex.

**C. Find and fix the bug**

8. The method should change the shared tax rate for all products, and it should work for subclasses
   too. What is wrong with this design? Rewrite it.
   ```python
   class Product:
       tax_rate = 0.18

       def set_tax(self, rate):
           Product.tax_rate = rate
   ```

**D. Interview questions**

9. When do you use a class method instead of a static method?
10. Why should an alternative constructor call `cls(...)` and not the class name?
11. **[Hard]** Can a static method access class attributes? Should it?

### Solutions 1.6

1. `large large`. Calling a class method on an object still passes the class, so `cls.size` changes
   the class attribute. `p` has no own `size`, so it sees `"large"`.
2. `8 abab`. A static method is a plain function; it can be called on the class or an object.
3. `Cat`. `Cat.create()` passes `Cat` as `cls`, so `cls()` builds a `Cat`.
8. The method is an instance method but it changes class-wide state, and it hard-codes `Product`.
   A subclass `Book(Product)` calling `set_tax` would change `Product.tax_rate` instead of `Book.tax_rate`.
   Use `@classmethod def set_tax(cls, rate): cls.tax_rate = rate`.
9. Use a class method when the method needs the class itself: to read or change class attributes,
   or to create objects (`cls(...)`). Use a static method when it needs neither the object nor the class.
10. With `cls(...)` a subclass that calls the inherited constructor gets an object of the subclass.
    A hard-coded name would always build the base class.
11. Technically yes, by writing the class name (`Product.tax_rate`). But if it needs the class,
    it should be a class method: that is clearer and works correctly with subclasses.

In [27]:
# Solutions 1.6 (B and C)

# Q4
class Date:
    def __init__(self, day, month, year):
        self.day = day
        self.month = month
        self.year = year

    @classmethod
    def from_string(cls, text):
        day, month, year = (int(part) for part in text.split("-"))
        return cls(day, month, year)

    @staticmethod
    def is_leap(year):
        return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)


d = Date.from_string("15-01-2026")
assert (d.day, d.month, d.year) == (15, 1, 2026)
assert Date.is_leap(2024) and Date.is_leap(2000)
assert not Date.is_leap(1900) and not Date.is_leap(2026)


# Q5
class Ticket:
    _issued = 0

    def __init__(self):
        type(self)._issued += 1
        self.number = type(self)._issued

    @classmethod
    def issued(cls):
        return cls._issued


t1, t2, t3 = Ticket(), Ticket(), Ticket()
assert (t1.number, t2.number, t3.number) == (1, 2, 3)
assert Ticket.issued() == 3


# Q6
class Converter:
    KM_PER_MILE = 1.609344

    @staticmethod
    def km_to_miles(km):
        return round(km / Converter.KM_PER_MILE, 3)

    @staticmethod
    def miles_to_km(miles):
        return round(miles * Converter.KM_PER_MILE, 3)


assert Converter.km_to_miles(10) == 6.214
assert Converter.miles_to_km(1) == 1.609
assert Converter.km_to_miles(0) == 0


# Q7
class Color:
    def __init__(self, r, g, b):
        for value in (r, g, b):
            if not 0 <= value <= 255:
                raise ValueError(f"channel out of range: {value}")
        self.r, self.g, self.b = r, g, b

    @classmethod
    def from_hex(cls, text):
        text = text.lstrip("#")
        return cls(int(text[0:2], 16), int(text[2:4], 16), int(text[4:6], 16))

    @classmethod
    def from_tuple(cls, rgb):
        return cls(*rgb)

    def to_hex(self):
        return f"#{self.r:02x}{self.g:02x}{self.b:02x}"


orange = Color.from_hex("#ff8000")
assert (orange.r, orange.g, orange.b) == (255, 128, 0)
assert Color.from_tuple((255, 128, 0)).to_hex() == "#ff8000"
assert Color(0, 0, 0).to_hex() == "#000000"


# Q8 fixed
class Product:
    tax_rate = 0.18

    @classmethod
    def set_tax(cls, rate):
        cls.tax_rate = rate


class Book(Product):
    pass


Book.set_tax(0.0)
assert Book.tax_rate == 0.0 and Product.tax_rate == 0.18   # only the subclass changed

print("Practice 1.6: all checks passed")

Practice 1.6: all checks passed


---

## 1.7 Encapsulation: public, `_internal` and `__mangled` names

**Encapsulation** means an object manages its own data. Outside code should use the object's methods
instead of reaching in and changing data directly. This keeps the data valid.

Languages like Java have `private` and `public` keywords. **Python has no real access control.**
It uses naming conventions instead:

| Name | Meaning |
|---|---|
| `name` | Public. Part of the class's interface. Anyone may use it. |
| `_name` | Internal. A convention meaning "please do not use this from outside the class". |
| `__name` | Name mangling. Inside class `C`, Python renames it to `_C__name`. |

> **In simple words:** a single underscore is a "staff only" sign on a door. The door is not locked.
> Python developers respect the sign.

In [28]:
# _balance is internal by convention; outside code should use the methods
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance            # internal: change it only through methods

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount

    def get_balance(self):
        return self._balance


acct = BankAccount("Asha", 100)
acct.deposit(50)
print(acct.get_balance())                  # 150
print(acct._balance)                       # 150  (possible, but against the convention)

150
150


### Name mangling with `__name`

A name that starts with two underscores (and does not end with two) is rewritten by Python inside the class:
`self.__pin` becomes `self._BankAccount__pin`.

The purpose is **not security**. It prevents a *subclass* from accidentally overwriting the parent's
attribute when it uses the same name. Anyone can still reach it as `obj._BankAccount__pin`.

In [29]:
# Name mangling renames __pin to _BankAccount__pin
class BankAccount:
    def __init__(self, owner, pin):
        self.owner = owner
        self.__pin = pin                   # stored as _BankAccount__pin

    def check_pin(self, pin):
        return pin == self.__pin           # inside the class the short name still works


acct = BankAccount("Asha", 1234)
print(list(vars(acct)))                    # ['owner', '_BankAccount__pin']
print(acct.check_pin(1234))                # True

try:
    print(acct.__pin)
except AttributeError as e:
    print("AttributeError:", e)            # AttributeError: 'BankAccount' object has no attribute '__pin'

print(acct._BankAccount__pin)              # 1234  (still reachable: mangling is not security)

['owner', '_BankAccount__pin']
True
AttributeError: 'BankAccount' object has no attribute '__pin'
1234


In [30]:
# Why mangling exists: parent and child can both use "__id" without a clash
class Base:
    def __init__(self):
        self.__id = "base-id"              # becomes _Base__id

    def base_id(self):
        return self.__id


class Child(Base):
    def __init__(self):
        super().__init__()
        self.__id = "child-id"             # becomes _Child__id (a different name)


c = Child()
print(c.base_id())                         # base-id  (the parent's value was not overwritten)
print(sorted(vars(c)))                     # ['_Base__id', '_Child__id']

base-id
['_Base__id', '_Child__id']


> **Tip:** in everyday code use a single underscore (`_balance`) for internal data. Use double underscores
> only when you design a class that others will subclass and you must avoid name clashes.

> **Warning:** never store real secrets (passwords, PINs) in plain attributes and think they are safe.
> Python has no private data. Store a hash instead (see Part 9, secure coding).

> **Key points**
> * Python has no private attributes; it relies on conventions.
> * `_name` means internal: do not use it from outside the class.
> * `__name` is renamed to `_ClassName__name` to avoid clashes in subclasses. It is not security.

---

## 1.8 Properties: getters, setters and computed attributes

In some languages you write `get_salary()` and `set_salary()` for every attribute "just in case".
Python does not need this. Start with a **plain public attribute**. If you later need validation or a
computed value, turn it into a **property**. The calling code does not change: `emp.salary` still works.

A **property** is a method that is used like an attribute:

* The **getter** (`@property`) runs when you read `obj.salary`. It must **return** the value.
* The **setter** (`@salary.setter`) runs when you write `obj.salary = value`. It usually **validates**.
* The real data is stored in an internal attribute such as `self._salary`.

In [31]:
# The running example, step 4: salary becomes a validated property
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary              # goes through the setter, so it is validated here too

    @property
    def salary(self):                     # getter: runs on "emp.salary"
        return self._salary

    @salary.setter
    def salary(self, value):              # setter: runs on "emp.salary = value"
        if not isinstance(value, int) or value <= 0:
            raise ValueError(f"salary must be a positive int, got {value!r}")
        self._salary = value


asha = Employee("Asha", "Engineering", 85000)
print(asha.salary)                        # 85000  (no brackets: it looks like an attribute)

asha.salary = 90000                       # the setter runs
print(asha.salary)                        # 90000

try:
    asha.salary = -10
except ValueError as e:
    print("ValueError:", e)               # ValueError: salary must be a positive int, got -10

print(asha.salary)                        # 90000  (the bad value was rejected)

85000
90000
ValueError: salary must be a positive int, got -10
90000


```text
   asha.salary            -->  Employee.salary getter  -->  returns asha._salary
   asha.salary = 90000    -->  Employee.salary setter  -->  validates, sets asha._salary
```

> **Common mistake:** a getter that prints instead of returning. Then `print(emp.salary)` prints the text
> and then `None`. A getter must `return` the value; the caller decides what to do with it.

### Read-only and computed properties

A property with only a getter (no setter) is **read-only**. Use it for values that are **computed** from
other attributes. The computed value is always up to date, because it is calculated on every read.

In [32]:
# Read-only computed properties: email and monthly_salary
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    @property
    def email(self):
        # computed from name every time, so it is never out of date
        return self.name.lower().replace(" ", ".") + "@acme.example"

    @property
    def monthly_salary(self):
        return round(self.salary / 12, 2)


meera = Employee("Meera Iyer", "HR", 60000)
print(meera.email)                      # meera.iyer@acme.example
print(meera.monthly_salary)             # 5000.0

meera.name = "Meera Rao"
print(meera.email)                      # meera.rao@acme.example  (updated automatically)

try:
    meera.email = "x@y.z"
except AttributeError as e:
    print("AttributeError:", e)         # AttributeError: property 'email' of 'Employee' object has no setter

meera.iyer@acme.example
5000.0
meera.rao@acme.example
AttributeError: property 'email' of 'Employee' object has no setter


### Another example: temperature with validation and a computed property

In [33]:
# Setter validates; fahrenheit is computed and read-only
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius                 # uses the setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print(t.fahrenheit)                            # 77.0
t.celsius = 100
print(t.fahrenheit)                            # 212.0

try:
    t.celsius = -300
except ValueError as e:
    print("ValueError:", e)                    # ValueError: below absolute zero

77.0
212.0
ValueError: below absolute zero


### The deleter (rarely needed)

A property can also define what `del obj.attr` does, with `@name.deleter`. It is used rarely,
for example to clear a cached value.

In [34]:
# A deleter clears a stored value
class Report:
    def __init__(self):
        self._summary = None

    @property
    def summary(self):
        if self._summary is None:
            self._summary = "computed summary"     # pretend this is slow to compute
        return self._summary

    @summary.deleter
    def summary(self):
        self._summary = None                       # next read will compute it again


r = Report()
print(r.summary)                                   # computed summary
del r.summary
print(r._summary)                                  # None

computed summary
None


### Common mistake: infinite recursion in a setter

Inside the setter you must store the value in the **internal** name (`self._salary`).
If you write `self.salary = value`, the setter calls itself again and again until Python stops it
with `RecursionError`.

In [35]:
# Common mistake: the setter assigns to the property itself
class Broken:
    @property
    def size(self):
        return self._size

    @size.setter
    def size(self, value):
        self.size = value                      # calls this setter again -> endless recursion


try:
    Broken().size = 3
except RecursionError as e:
    print("RecursionError:", e)                # RecursionError: maximum recursion depth exceeded (wording varies by version)


# Fix: store the data under the internal name
class Fixed:
    @property
    def size(self):
        return self._size

    @size.setter
    def size(self, value):
        self._size = value


f = Fixed()
f.size = 3
print(f.size)                                  # 3

RecursionError: maximum recursion depth exceeded
3


> **Tip:** `functools.cached_property` computes a value once and then stores it, which is useful for
> expensive values that never change. See Part 7.

> **Key points**
> * Start with plain attributes. Switch to a property later without changing calling code.
> * A getter must `return` the value. A setter validates and stores it in an internal name (`_x`).
> * A property without a setter is read-only. Use it for computed values.
> * In `__init__`, assign through the property (`self.salary = salary`) so validation runs there too.

### Practice 1.8: Encapsulation and properties

**A. Predict the output**

1. What does this print?
   ```python
   class Secret:
       def __init__(self):
           self.__code = 7
           self._hint = "lucky"

   s = Secret()
   print(sorted(vars(s)))
   ```
2. What does this print?
   ```python
   class Circle:
       def __init__(self, r):
           self.r = r

       @property
       def diameter(self):
           return 2 * self.r

   c = Circle(3)
   c.r = 5
   print(c.diameter)
   ```
3. What does this print, and why?
   ```python
   class Item:
       @property
       def price(self):
           print("price is 10")

   print(Item().price)
   ```
4. What happens here?
   ```python
   class Square:
       def __init__(self, side):
           self.side = side

       @property
       def area(self):
           return self.side ** 2

   sq = Square(2)
   sq.area = 10
   ```

**B. Write code**

5. Write a class `Person` with a property `age` that only accepts integers from 0 to 150
   (raise `ValueError` otherwise). Validation must also run in `__init__`.
6. Write a class `Rectangle` with properties `width` and `height` (positive numbers only) and read-only
   properties `area` and `is_square`.
7. Write a class `BankAccount` with a read-only `balance` property, `deposit(amount)` and
   `withdraw(amount)`. Reject non-positive amounts and overdrafts with `ValueError`.
8. **[Hard]** Write a class `User` with a property `username` that strips spaces, lowercases the value and
   rejects names shorter than 3 characters. Add a read-only property `profile_url` that returns
   `"https://example.com/u/<username>"`.

**C. Find and fix the bug**

9. Reading `p.name` gives `RecursionError`. Why? Fix it.
   ```python
   class Pet:
       def __init__(self, name):
           self.name = name

       @property
       def name(self):
           return self.name

       @name.setter
       def name(self, value):
           self._name = value.title()
   ```

**D. Interview questions**

10. Does Python have private attributes? Explain `_x` and `__x`.
11. Why do Python developers not write getters and setters for every attribute?
12. **[Hard]** Why is a property found even though the object's `__dict__` may contain a key with the
    same name? (Hint: where is the property stored, and what kind of object is it?)

### Solutions 1.8

1. `['_Secret__code', '_hint']`. `__code` was mangled to `_Secret__code`; `_hint` is unchanged.
2. `10`. `diameter` is computed on every read, from the current `r`.
3. It prints `price is 10` and then `None`. The getter prints but does not return, so the value is `None`.
   A getter must `return 10`.
4. `AttributeError: property 'area' of 'Square' object has no setter`. `area` is read-only.
9. The getter returns `self.name`, which calls the getter again, forever. It must return `self._name`.
10. No. `_x` is a convention meaning "internal, do not use from outside". `__x` triggers name mangling
    (`_ClassName__x`), which avoids name clashes in subclasses, but the value is still reachable.
11. Because a plain attribute can later be replaced by a property without changing any calling code.
    Writing getters and setters up front only adds noise (this follows YAGNI: "you aren't gonna need it").
12. A property is stored on the **class**, and it is a *data descriptor* (an object with `__get__` and
    `__set__` methods). For attribute lookup, data descriptors on the class take priority over the
    instance dictionary. Details in Part 6.

In [36]:
# Solutions 1.8 (B and C) and checks for A

# A1, A4 checks
class Secret:
    def __init__(self):
        self.__code = 7
        self._hint = "lucky"


assert sorted(vars(Secret())) == ["_Secret__code", "_hint"]


class Square:
    def __init__(self, side):
        self.side = side

    @property
    def area(self):
        return self.side ** 2


try:
    Square(2).area = 10
except AttributeError:
    pass
else:
    raise AssertionError("area should be read-only")


# Q5
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age                      # runs the setter

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not isinstance(value, int) or not 0 <= value <= 150:
            raise ValueError(f"invalid age: {value!r}")
        self._age = value


p = Person("Asha", 30)
p.age = 31
assert p.age == 31
for bad in (-1, 151, "30", 3.5):
    try:
        Person("X", bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"accepted {bad!r}")


# Q6
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @staticmethod
    def _check(value):
        if value <= 0:
            raise ValueError("sides must be positive")
        return value

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        self._width = self._check(value)

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        self._height = self._check(value)

    @property
    def area(self):
        return self.width * self.height

    @property
    def is_square(self):
        return self.width == self.height


r = Rectangle(2, 3)
assert (r.area, r.is_square) == (6, False)
r.height = 2
assert (r.area, r.is_square) == (4, True)
try:
    r.width = 0
except ValueError:
    pass
else:
    raise AssertionError("width 0 accepted")


# Q7
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount


acct = BankAccount("Ravi", 100)
acct.deposit(50)
acct.withdraw(120)
assert acct.balance == 30
for action in (lambda: acct.withdraw(31), lambda: acct.deposit(0)):
    try:
        action()
    except ValueError:
        pass
    else:
        raise AssertionError("bad amount accepted")
try:
    acct.balance = 1_000_000
except AttributeError:
    pass
else:
    raise AssertionError("balance should be read-only")


# Q8
class User:
    def __init__(self, username):
        self.username = username

    @property
    def username(self):
        return self._username

    @username.setter
    def username(self, value):
        cleaned = value.strip().lower()
        if len(cleaned) < 3:
            raise ValueError("username too short")
        self._username = cleaned

    @property
    def profile_url(self):
        return f"https://example.com/u/{self.username}"


u = User("  Meera  ")
assert u.username == "meera"
assert u.profile_url == "https://example.com/u/meera"
try:
    u.username = " ab "
except ValueError:
    pass
else:
    raise AssertionError("short name accepted")


# Q9 fixed
class Pet:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value.title()


assert Pet("rex the dog").name == "Rex The Dog"

print("Practice 1.8: all checks passed")

Practice 1.8: all checks passed


---

## 1.9 `__str__` vs `__repr__`: how objects show themselves

By default, printing an object shows something like `<__main__.Employee object at 0x...>`.
That is not helpful. Two special methods control the text form of an object:

| Method | Called by | Audience | Goal |
|---|---|---|---|
| `__repr__` | `repr(obj)`, the REPL, inside lists/dicts, `f"{obj!r}"` | developers | unambiguous; ideally looks like code that recreates the object |
| `__str__` | `print(obj)`, `str(obj)`, `f"{obj}"` | end users | readable |

If a class defines only `__repr__`, then `str()` and `print()` use it too. So **always write `__repr__`
first**; add `__str__` only when you want a different, friendlier text.

In [37]:
# The running example, step 5: __repr__ for developers, __str__ for users
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    def __repr__(self):
        # !r adds quotes around strings, so the text looks like real Python code
        return f"Employee({self.name!r}, {self.dept!r}, {self.salary!r})"

    def __str__(self):
        return f"{self.name} ({self.dept})"


asha = Employee("Asha", "Engineering", 85000)

print(asha)                        # Asha (Engineering)              <- __str__
print(repr(asha))                  # Employee('Asha', 'Engineering', 85000)  <- __repr__
print(f"{asha} / {asha!r}")        # Asha (Engineering) / Employee('Asha', 'Engineering', 85000)
print([asha])                      # [Employee('Asha', 'Engineering', 85000)]  <- lists use __repr__

Asha (Engineering)
Employee('Asha', 'Engineering', 85000)
Asha (Engineering) / Employee('Asha', 'Engineering', 85000)
[Employee('Asha', 'Engineering', 85000)]


> **Note:** containers (lists, dicts, tuples) always show their items with `repr`, even when you `print`
> the container. That is why a good `__repr__` matters for debugging.

In [38]:
# Only __repr__ defined: print() falls back to it
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"


p = Point(2, 3)
print(p)                           # Point(x=2, y=3)
print(str(p) == repr(p))           # True

Point(x=2, y=3)
True


> **Common mistake:** returning something that is not a string from `__str__` or `__repr__`, or printing
> inside them. They must **return a `str`**. Otherwise you get
> `TypeError: __str__ returned non-string (type NoneType)`.

In [39]:
# Common mistake: __str__ prints instead of returning a string
class Broken:
    def __str__(self):
        print("Broken object")            # returns None


try:
    str(Broken())                         # prints "Broken object", then fails
except TypeError as e:
    print("TypeError:", e)                # TypeError: __str__ returned non-string (type NoneType)

Broken object
TypeError: __str__ returned non-string (type NoneType)


> **Key points**
> * `__repr__` is for developers: unambiguous, ideally `ClassName(args)`. Containers use it.
> * `__str__` is for users: readable. `print()` and `str()` use it, and fall back to `__repr__`.
> * Always implement `__repr__`. Use `{value!r}` inside it so strings get quotes.
> * Both must return a string; never print inside them.

---

## 1.10 The life of an object

Every object goes through the same stages:

```text
   +------------------+     +------------------+     +---------------------+
   | 1. created       | --> | 2. initialised   | --> | 3. used             |
   |    __new__       |     |    __init__      |     |    names refer to it|
   +------------------+     +------------------+     +----------+----------+
                                                                |
                                                                v
                                        +---------------------------------------+
                                        | 4. no name or container refers to it  |
                                        |    any more: it becomes garbage       |
                                        +-------------------+-------------------+
                                                            |
                                                            v
                                        +---------------------------------------+
                                        | 5. Python frees the memory            |
                                        |    (__del__ runs first, if defined)   |
                                        +---------------------------------------+
```

* An object stays alive as long as at least one **reference** (a name, a list item, an attribute) points to it.
* CPython (the standard Python) counts references. When the count drops to zero, the object is freed at once.
* Objects that refer to each other in a cycle are found later by the **cyclic garbage collector**.
* `del name` removes a **name**, not the object. The object is freed only when no references are left.

Memory management is covered in more detail in Part 6.

In [40]:
# del removes a name; the object is freed only when no references remain (CPython)
class Session:
    def __init__(self, user):
        self.user = user

    def __del__(self):
        print(f"session of {self.user} is freed")


s = Session("Asha")
alias = s                      # a second reference to the same object
del s                          # removes the name s only; nothing is printed yet
print("alias still works:", alias.user)   # alias still works: Asha
del alias                      # last reference gone -> __del__ runs now in CPython
# session of Asha is freed

alias still works: Asha
session of Asha is freed


> **Warning:** do not rely on `__del__` for cleanup (closing files, releasing locks). When it runs is an
> implementation detail, and it may not run at all when the program exits. Use a `with` statement and a
> context manager instead (Part 5).

### Identity and default equality

By default, two objects are equal (`==`) only if they are the **same object** (`is`).
Two employees with the same data are still different objects. To compare by value, define `__eq__`
(Part 2, section 2.6) or use a dataclass (Part 2, section 2.7).

In [41]:
# Default == compares identity, not the data
class Employee:
    def __init__(self, name):
        self.name = name


a = Employee("Asha")
b = Employee("Asha")
c = a

print(a == b)          # False  (two different objects with the same data)
print(a == c)          # True   (same object)
print(a is c)          # True

False
True
True


> **Key points**
> * Lifecycle: `__new__` creates, `__init__` initialises, the object lives while referenced, then it is freed.
> * `del x` deletes a name, not necessarily the object.
> * Do not use `__del__` for cleanup; use context managers.
> * By default `==` on your objects means "same object". Define `__eq__` for value equality.

---

## 1.11 Putting it together, common mistakes and debugging

Here is the complete `Employee` class built in this Part. It combines everything:
a class attribute, a class-wide counter, validation through a property, a computed property,
an alternative constructor, a static helper, and good text forms.

In [42]:
# The running example, complete version for Part 1
class Employee:
    company = "Acme Corp"                       # class attribute (shared)
    RAISE_PERCENT = 5                           # class constant
    _count = 0                                  # class-wide state

    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary                    # validated by the property setter
        type(self)._count += 1

    # ----- properties -----
    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, value):
        if not self.is_valid_salary(value):
            raise ValueError(f"invalid salary: {value!r}")
        self._salary = value

    @property
    def email(self):
        return self.name.lower().replace(" ", ".") + "@acme.example"

    # ----- instance methods -----
    def apply_raise(self):
        self.salary = round(self.salary * (100 + self.RAISE_PERCENT) / 100)

    def describe(self):
        return f"{self.name} works in {self.dept} at {self.company}"

    # ----- class methods -----
    @classmethod
    def from_csv(cls, line):
        name, dept, salary = (part.strip() for part in line.split(","))
        return cls(name, dept, int(salary))

    @classmethod
    def headcount(cls):
        return cls._count

    # ----- static methods -----
    @staticmethod
    def is_valid_salary(value):
        return isinstance(value, int) and value > 0

    # ----- text forms -----
    def __repr__(self):
        return f"Employee({self.name!r}, {self.dept!r}, {self.salary!r})"

    def __str__(self):
        return f"{self.name} ({self.dept})"


staff = [
    Employee("Asha Rao", "Engineering", 85000),
    Employee.from_csv("Ravi Kumar, Sales, 60000"),
]
staff[1].apply_raise()

for person in staff:
    print(person, "-", person.email)
# Asha Rao (Engineering) - asha.rao@acme.example
# Ravi Kumar (Sales) - ravi.kumar@acme.example

print(staff)            # [Employee('Asha Rao', 'Engineering', 85000), Employee('Ravi Kumar', 'Sales', 63000)]
print(Employee.headcount())   # 2

Asha Rao (Engineering) - asha.rao@acme.example
Ravi Kumar (Sales) - ravi.kumar@acme.example
[Employee('Asha Rao', 'Engineering', 85000), Employee('Ravi Kumar', 'Sales', 63000)]
2


### Common mistakes with classes

1. Forgetting `self` as the first parameter of a method.
2. Forgetting `self.` when setting an attribute (a local variable is created instead).
3. Using a mutable class attribute (`items = []`) when each object needs its own list.
4. Changing a class attribute through `self.x = ...` (creates an instance attribute instead).
5. Calling `obj.__init__(...)` yourself. Create a new object with `Class(...)` instead.
6. A property getter that prints instead of returning; a setter that assigns to itself (recursion).
7. `__str__` / `__repr__` that do not return a string.
8. Too many getters and setters "just in case". Use plain attributes until you need control.
9. Classes that do too much ("god classes"). One class should have one clear job.
10. Using a class when a simple function would do.

### How to debug class code

Most bugs in class code show up as one of a few errors. Read the **last line** of the traceback first:
it names the error type and the message. Then look at the line in your code that the traceback points to.

| Error message (example) | What it means | Typical fix |
|---|---|---|
| `AttributeError: 'Employee' object has no attribute 'salary'` | The attribute was never set on this object (or is misspelt). | Set it in `__init__`; check spelling; check `self.` |
| `TypeError: Employee.describe() takes 0 positional arguments but 1 was given` | The method has no `self` parameter. | Add `self` as the first parameter. |
| `TypeError: Employee.__init__() missing 1 required positional argument: 'dept'` | Not enough arguments when creating the object. | Pass all arguments, or give a default. |
| `TypeError: Employee() takes no arguments` | `__init__` is missing or misspelt (`__int__`, `_init_`). | Spell it `__init__`, with two underscores on each side. |
| `AttributeError: property 'email' of 'Employee' object has no setter` | Assigning to a read-only property. | Add a setter, or do not assign. |
| `RecursionError: maximum recursion depth exceeded` | A property returns or assigns to itself. | Use the internal name `self._x`. |
| `NameError: name 'salary' is not defined` | Used `salary` inside a method instead of `self.salary`. | Add `self.` |

In [43]:
# Reproducing the most common errors, and printing them clearly
class Employee:
    def __init__(self, name, dept):
        self.name = name
        self.dept = dept

    def yearly_bonus(self):
        return salary * 0.1                 # bug: should be self.salary (and salary is never set)


emp = Employee("Asha", "Engineering")

attempts = [
    lambda: emp.salary,                     # attribute never set
    lambda: emp.yearly_bonus(),             # name without self.
    lambda: Employee("Ravi"),               # missing argument
]
for attempt in attempts:
    try:
        attempt()
    except (AttributeError, NameError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")
# AttributeError: 'Employee' object has no attribute 'salary'
# NameError: name 'salary' is not defined
# TypeError: Employee.__init__() missing 1 required positional argument: 'dept'

AttributeError: 'Employee' object has no attribute 'salary'
NameError: name 'salary' is not defined
TypeError: Employee.__init__() missing 1 required positional argument: 'dept'


### Inspecting an object while debugging

When an object does not behave as expected, look inside it. These tools help:

* `vars(obj)` shows the instance attributes (what this object really stores).
* `type(obj)` and `type(obj).__mro__` show the class and where methods are looked up.
* `hasattr(obj, "x")` and `getattr(obj, "x", default)` test for an attribute safely.
* `dir(obj)` lists every name that can be reached from the object (including inherited ones).
* A good `__repr__` makes `print(obj)` and debugger views useful.

In [44]:
# Inspecting an object to find out what it really contains
class Employee:
    company = "Acme Corp"

    def __init__(self, name):
        self.name = name


emp = Employee("Asha")
emp.nmae = "Asha R"                     # a typo creates a NEW attribute instead of an error

print(vars(emp))                        # {'name': 'Asha', 'nmae': 'Asha R'}  <- the typo is visible here
print(hasattr(emp, "company"))          # True  (found on the class)
print(getattr(emp, "salary", "no salary set"))   # no salary set
print("name" in dir(emp), "company" in dir(emp)) # True True

{'name': 'Asha', 'nmae': 'Asha R'}
True
no salary set
True True


```text
   +--------------------------+
   | Read the last line of    |
   | the traceback (type+msg) |
   +------------+-------------+
                |
                v
   +--------------------------+
   | Go to the line it points |
   | to in YOUR code          |
   +------------+-------------+
                |
                v
   +--------------------------+     +-----------------------------+
   | Inspect the object:      |---->| Attribute missing or wrong? |
   | vars(obj), type(obj)     |     | check __init__, self., typo |
   +------------+-------------+     +-----------------------------+
                |
                v
   +--------------------------+
   | Fix, re-run, and add an  |
   | assert or test for it    |
   +--------------------------+
```

> **Tip:** a typo in an assignment (`emp.nmae = ...`) does not raise an error; it silently creates a new
> attribute. `vars(obj)` reveals it. `__slots__` (Part 2) can prevent this kind of bug.

> **Key points**
> * Read the last line of the traceback first, then the line of your code it points to.
> * `AttributeError`, `TypeError` (wrong number of arguments) and `NameError` (missing `self.`) are the usual suspects.
> * Inspect objects with `vars()`, `type()`, `hasattr()`, `getattr()` and a good `__repr__`.

### Practice 1.11: Classes review and design

**A. Predict the output**

1. What does this print?
   ```python
   class Tag:
       def __init__(self, label):
           self.label = label

       def __repr__(self):
           return f"Tag({self.label!r})"

       def __str__(self):
           return self.label

   tags = [Tag("python"), Tag("oop")]
   print(tags[0], tags)
   ```
2. What does this print?
   ```python
   class A:
       x = 1

       def __init__(self):
           self.y = A.x
           A.x += 1

   a1 = A()
   a2 = A()
   print(a1.y, a2.y, A.x)
   ```
3. What does this print?
   ```python
   class P:
       def __init__(self, v):
           self.v = v

   print(P(1) == P(1), P(1) is P(1))
   ```

**B. Write code**

4. Write a class `Stack` (backed by a list) with `push`, `pop`, `peek`, `is_empty`, `size` and a
   `__repr__` like `Stack([1, 2, 3])`. `pop` and `peek` on an empty stack raise `IndexError("stack is empty")`.
5. Write a class `ShoppingCart` with `add(item, price, qty=1)`, `remove(item)`, a read-only property
   `total` and `__str__` that returns `"Cart(3 items, total=45.50)"` (count all quantities, 2 decimals).
6. **[Hard]** Design a `BankAccount` class: owner, account number generated from a class-wide counter
   (`"ACC-0001"`, `"ACC-0002"`, ...), read-only `balance`, `deposit`, `withdraw` (raise a custom exception
   `InsufficientFundsError` when overdrawn), a transaction history (list of tuples `("deposit", 100)`),
   an alternative constructor `from_dict`, and a useful `__repr__`.
7. **[Hard]** Write a class `Matrix2x2` with `__init__(a, b, c, d)`, a read-only `determinant` property,
   a method `multiply(other)` that returns a new `Matrix2x2`, and a `__repr__`.

**C. Find and fix the bug**

8. This class has four bugs. Find and fix them all.
   ```python
   class Account:
       history = []

       def __init__(self, owner):
           owner = owner

       @property
       def owner_upper(self):
           print(self.owner.upper())

       def add(entry):
           self.history.append(entry)
   ```

**D. Interview questions**

9. What are the four pillars of OOP? Give a one-line Python example of each.
10. When would you *not* use a class?
11. **[Hard]** Why is `__repr__` more important than `__str__` for debugging?

### Solutions 1.11

1. `python [Tag('python'), Tag('oop')]`. `print(tags[0])` uses `__str__`; the list shows its items with `__repr__`.
2. `1 2 3`. Each `__init__` copies the current class value and then increases the class attribute.
3. `False False`. Without `__eq__`, `==` compares identity; two separate objects are never equal.
8. (1) `history = []` is shared by all accounts; create it in `__init__`. (2) `owner = owner` must be
   `self.owner = owner`. (3) The getter prints instead of returning. (4) `add` is missing `self`.
9. **Encapsulation**: `BankAccount` keeps `_balance` and changes it only in `deposit`/`withdraw`.
   **Abstraction**: callers use `acct.withdraw(50)` without knowing how it is stored.
   **Inheritance**: `class Manager(Employee)` reuses `Employee`.
   **Polymorphism**: `shape.area()` works for `Circle` and `Square` (Part 2).
10. When there is no state to keep, for example a pure calculation like `def celsius_to_fahrenheit(c)`.
    Also for small scripts, or when a dict, tuple or dataclass is enough.
11. Containers, debuggers, logs and the REPL all use `repr`. A good `__repr__` shows exactly which object
    and which values you have. `__str__` is optional and falls back to `__repr__`.

In [45]:
# Solutions 1.11 (B and C) and checks for A

# A2 check
class A:
    x = 1

    def __init__(self):
        self.y = A.x
        A.x += 1


a1, a2 = A(), A()
assert (a1.y, a2.y, A.x) == (1, 2, 3)


# Q4
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise IndexError("stack is empty")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("stack is empty")
        return self._items[-1]

    def is_empty(self):
        return not self._items

    def size(self):
        return len(self._items)

    def __repr__(self):
        return f"Stack({self._items!r})"


s = Stack()
for n in (1, 2, 3):
    s.push(n)
assert repr(s) == "Stack([1, 2, 3])"
assert (s.peek(), s.pop(), s.size()) == (3, 3, 2)
s.pop()
s.pop()
assert s.is_empty()
try:
    s.pop()
except IndexError as e:
    assert str(e) == "stack is empty"
else:
    raise AssertionError("pop on empty stack must fail")


# Q5
class ShoppingCart:
    def __init__(self):
        self._lines = {}                  # item -> [price, qty]

    def add(self, item, price, qty=1):
        if item in self._lines:
            self._lines[item][1] += qty
        else:
            self._lines[item] = [price, qty]

    def remove(self, item):
        del self._lines[item]             # KeyError if missing, which is a clear error

    @property
    def total(self):
        return sum(price * qty for price, qty in self._lines.values())

    def __str__(self):
        count = sum(qty for _, qty in self._lines.values())
        return f"Cart({count} items, total={self.total:.2f})"


cart = ShoppingCart()
cart.add("pen", 10.5)
cart.add("book", 35.0)
cart.add("pen", 10.5)
cart.remove("pen")
cart.add("pen", 5.25, qty=2)
assert str(cart) == "Cart(3 items, total=45.50)"
assert str(ShoppingCart()) == "Cart(0 items, total=0.00)"


# Q6
class InsufficientFundsError(Exception):
    """Raised when a withdrawal is larger than the balance."""


class BankAccount:
    _next_number = 1

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.number = f"ACC-{type(self)._next_number:04d}"
        type(self)._next_number += 1
        self._balance = 0
        self.history = []
        if balance:
            self.deposit(balance)

    @classmethod
    def from_dict(cls, data):
        return cls(data["owner"], data.get("balance", 0))

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise InsufficientFundsError(f"balance {self._balance} is less than {amount}")
        self._balance -= amount
        self.history.append(("withdraw", amount))

    def __repr__(self):
        return f"BankAccount(owner={self.owner!r}, number={self.number!r}, balance={self._balance})"


acc1 = BankAccount("Asha", 100)
acc2 = BankAccount.from_dict({"owner": "Ravi"})
acc1.withdraw(40)
assert (acc1.number, acc2.number) == ("ACC-0001", "ACC-0002")
assert acc1.balance == 60 and acc2.balance == 0
assert acc1.history == [("deposit", 100), ("withdraw", 40)]
assert repr(acc2) == "BankAccount(owner='Ravi', number='ACC-0002', balance=0)"
try:
    acc2.withdraw(1)
except InsufficientFundsError as e:
    assert "less than 1" in str(e)
else:
    raise AssertionError("overdraft accepted")


# Q7
class Matrix2x2:
    def __init__(self, a, b, c, d):
        self.a, self.b, self.c, self.d = a, b, c, d

    @property
    def determinant(self):
        return self.a * self.d - self.b * self.c

    def multiply(self, other):
        return Matrix2x2(
            self.a * other.a + self.b * other.c,
            self.a * other.b + self.b * other.d,
            self.c * other.a + self.d * other.c,
            self.c * other.b + self.d * other.d,
        )

    def __repr__(self):
        return f"Matrix2x2({self.a}, {self.b}, {self.c}, {self.d})"


m = Matrix2x2(1, 2, 3, 4)
identity = Matrix2x2(1, 0, 0, 1)
assert m.determinant == -2
assert repr(m.multiply(identity)) == "Matrix2x2(1, 2, 3, 4)"
assert repr(m.multiply(m)) == "Matrix2x2(7, 10, 15, 22)"


# Q8 fixed
class Account:
    def __init__(self, owner):
        self.owner = owner
        self.history = []

    @property
    def owner_upper(self):
        return self.owner.upper()

    def add(self, entry):
        self.history.append(entry)


x, y = Account("asha"), Account("ravi")
x.add("opened")
assert (x.owner_upper, x.history, y.history) == ("ASHA", ["opened"], [])

print("Practice 1.11: all checks passed")

Practice 1.11: all checks passed


### Summary of Part 1

* A class is a blueprint; calling it creates an object. Methods receive the object as `self`.
* `__new__` creates the object and `__init__` initialises it. Python calls both; you never call `__init__` yourself.
* Instance attributes live in `obj.__dict__`; class attributes live once on the class and are shared.
* Reading `obj.x` searches the object, then the class and its parents. Writing `obj.x = ...` always writes to the object.
* Never use a mutable class attribute for per-object data; create it in `__init__`.
* Instance methods use `self`; class methods use `cls` (class-wide state, alternative constructors); static methods are helpers.
* `_x` means internal; `__x` is name-mangled to avoid subclass clashes. Python has no real private data.
* Properties give validated or computed attributes; a getter returns, a setter validates and stores in `_x`.
* Always write `__repr__`; add `__str__` for a user-friendly form.
* Debug class code by reading the last traceback line and inspecting objects with `vars()`, `type()` and `getattr()`.

---

# Part 2: OOP - Inheritance, Polymorphism and Special Methods

**What these notes cover**

* Inheritance: single, multilevel, hierarchical and multiple; overriding methods; `super()`.
* The method resolution order (MRO), the diamond problem and cooperative `super()`.
* Mixins, and when composition is better than inheritance.
* Polymorphism, duck typing and abstract base classes (ABCs).
* Special (dunder) methods: comparison, hashing, arithmetic, containers, `__call__`, `__bool__`.
* Dataclasses, `__slots__` and enums.
* Design principles (SOLID, DRY, KISS) and how to design classes in an interview
  (bank account, LRU cache, parking lot).

**Before you start:** Part 1 (Classes and Objects). Notebook 1, Part 12 (Errors and Exceptions)
helps for the custom exceptions used here.

**Running example:** we continue the company domain from Part 1. A small `Employee` base class gets
subclasses (`Manager`, `Engineer`, `Intern`), and a `Company` class holds employees.

## 2.1 Inheritance basics: reuse and extend a class

**Inheritance** lets a new class (the **child**, **subclass** or **derived class**) reuse the code of an
existing class (the **parent**, **superclass** or **base class**). The child gets all the parent's
attributes and methods, and can:

* **add** new attributes and methods,
* **override** (replace) a parent method with its own version,
* **extend** a parent method: run the parent's version with `super()` and add more.

Inheritance models an **"is-a"** relationship: a manager *is an* employee.

In [46]:
# The base class for Part 2
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def annual_pay(self):
        return self.salary

    def describe(self):
        return f"{self.name} earns {self.annual_pay()}"

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.salary!r})"


# A child class: everything is inherited, nothing added yet
class Engineer(Employee):
    pass


ravi = Engineer("Ravi", 90000)
print(ravi.describe())                 # Ravi earns 90000   (method inherited from Employee)
print(ravi)                            # Engineer('Ravi', 90000)  (type(self) is Engineer)

Ravi earns 90000
Engineer('Ravi', 90000)


> **Tip:** in `__repr__` we wrote `type(self).__name__` instead of the fixed text `"Employee"`.
> Then every subclass shows its own name without writing its own `__repr__`.

### Overriding and extending with `super()`

A manager earns a salary plus a bonus. `Manager` **overrides** `annual_pay`, and **extends** `__init__`:
it first lets `Employee.__init__` set the name and salary through `super().__init__(...)`, then adds its own data.

`super()` returns a helper object that finds the **next class in the method resolution order**
(for single inheritance: simply the parent). So `super().annual_pay()` runs the parent's version.

In [47]:
# Manager extends __init__ and overrides annual_pay
class Manager(Employee):
    def __init__(self, name, salary, bonus):
        super().__init__(name, salary)        # let the parent set name and salary
        self.bonus = bonus                    # then add what is new
        self.reports = []

    def annual_pay(self):                     # override
        return super().annual_pay() + self.bonus   # reuse the parent's result

    def add_report(self, employee):
        self.reports.append(employee)


asha = Manager("Asha", 120000, 15000)
asha.add_report(ravi)

print(asha.describe())          # Asha earns 135000  (describe is inherited, but calls the NEW annual_pay)
print(asha.reports)             # [Engineer('Ravi', 90000)]

Asha earns 135000
[Engineer('Ravi', 90000)]


Notice the last example carefully. `describe()` is defined only in `Employee`, but it calls
`self.annual_pay()`. Because `self` is a `Manager`, Python finds `Manager.annual_pay`.
A parent method automatically uses the child's overrides. This is the heart of polymorphism (section 2.5).

```text
   asha.describe()
        |
        v
   look for 'describe':  Manager?  no   -->  Employee?  yes  -> run Employee.describe(asha)
                                                                     |
                                                                     v
                                                            self.annual_pay()
                                                                     |
                                                                     v
   look for 'annual_pay': Manager?  yes -> run Manager.annual_pay(asha)
                                                |
                                                v
                                     super().annual_pay()  -> Employee.annual_pay(asha)
```

### Common mistake: forgetting `super().__init__()`

If a child defines its own `__init__`, the parent's `__init__` does **not** run automatically.
Forget `super().__init__(...)` and the parent's attributes are never set.

In [48]:
# Common mistake: the child's __init__ replaces the parent's and never calls it
class Intern(Employee):
    def __init__(self, name, salary, school):
        self.school = school                   # forgot super().__init__(name, salary)


meera = Intern("Meera", 20000, "City College")
try:
    print(meera.describe())
except AttributeError as e:
    print("AttributeError:", e)                # AttributeError: 'Intern' object has no attribute 'name'


# Fix: call the parent's initialiser first
class Intern(Employee):
    def __init__(self, name, salary, school):
        super().__init__(name, salary)
        self.school = school


print(Intern("Meera", 20000, "City College").describe())   # Meera earns 20000

AttributeError: 'Intern' object has no attribute 'name'
Meera earns 20000


### Checking relationships: `isinstance` and `issubclass`

* `isinstance(obj, Cls)` is `True` if `obj` is an instance of `Cls` **or of any subclass**.
* `issubclass(Child, Parent)` checks the relationship between classes.
* `type(obj) is Cls` is an exact check that ignores subclasses. It is rarely what you want.
* Every class inherits from `object` in the end.

In [49]:
# isinstance respects inheritance; type() is exact
boss = Manager("Asha", 120000, 15000)

print(isinstance(boss, Manager))        # True
print(isinstance(boss, Employee))       # True   (a Manager is an Employee)
print(type(boss) is Employee)           # False  (exact type is Manager)
print(issubclass(Manager, Employee))    # True
print(issubclass(Employee, object))     # True   (every class inherits from object)
print(Manager.__bases__)                # (<class '__main__.Employee'>,)

True
True
False
True
True
(<class '__main__.Employee'>,)


> **Key points**
> * `class Child(Parent):` inherits all attributes and methods of `Parent`.
> * Override a method by defining it again in the child; extend it by calling `super().method(...)`.
> * If the child defines `__init__`, call `super().__init__(...)` so the parent's attributes are set.
> * Parent methods call `self.method()`, so they automatically use the child's overrides.
> * Use `isinstance` (it respects subclasses), not `type(x) is ...`.

---

## 2.2 Kinds of inheritance

```text
  Single           Multilevel          Hierarchical               Multiple

  Employee         Employee            Employee                   Payable   Printable
     |                |              /    |     \                     \       /
  Manager          Manager       Manager Engineer Intern               Invoice
                      |
                   Director
```

| Kind | Meaning | Example |
|---|---|---|
| Single | one parent, one child | `Manager(Employee)` |
| Multilevel | a chain: grandparent -> parent -> child | `Director(Manager)` |
| Hierarchical | one parent, many children | `Manager`, `Engineer`, `Intern` all from `Employee` |
| Multiple | one child, several parents | `Invoice(Payable, Printable)` |

In [50]:
# Multilevel inheritance: Director -> Manager -> Employee
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def annual_pay(self):
        return self.salary


class Manager(Employee):
    def __init__(self, name, salary, bonus):
        super().__init__(name, salary)
        self.bonus = bonus

    def annual_pay(self):
        return super().annual_pay() + self.bonus


class Director(Manager):
    def __init__(self, name, salary, bonus, shares):
        super().__init__(name, salary, bonus)     # Manager.__init__, which calls Employee.__init__
        self.shares = shares

    def annual_pay(self):
        return super().annual_pay() + self.shares * 10


d = Director("Kiran", 200000, 30000, 1000)
print(d.annual_pay())                             # 240000  (200000 + 30000 + 10000)
print([cls.__name__ for cls in Director.__mro__]) # ['Director', 'Manager', 'Employee', 'object']

240000
['Director', 'Manager', 'Employee', 'object']


In [51]:
# Hierarchical inheritance: several children share one parent
class Engineer(Employee):
    def annual_pay(self):
        return self.salary + 5000            # on-call allowance


class Intern(Employee):
    def annual_pay(self):
        return self.salary // 2              # half year


team = [Employee("Asha", 80000), Engineer("Ravi", 90000), Intern("Meera", 40000)]
for person in team:
    print(type(person).__name__, person.annual_pay())
# Employee 80000
# Engineer 95000
# Intern 20000

Employee 80000
Engineer 95000
Intern 20000


### Multiple inheritance

A class may have several parents: `class Child(ParentA, ParentB)`. It gets the methods of all of them.
The **order of the parents matters**: if two parents define the same method, the one listed first wins.
The exact search order is the MRO (next section).

In [52]:
# Multiple inheritance: an Invoice is both Payable and Printable
class Payable:
    def pay(self):
        return f"paid {self.amount}"

    def describe(self):
        return "payable item"


class Printable:
    def print_copy(self):
        return f"printing {self.title}"

    def describe(self):
        return "printable item"


class Invoice(Payable, Printable):
    def __init__(self, title, amount):
        self.title = title
        self.amount = amount


inv = Invoice("INV-7", 250)
print(inv.pay())            # paid 250
print(inv.print_copy())     # printing INV-7
print(inv.describe())       # payable item   (Payable is listed first)

paid 250
printing INV-7
payable item


> **Key points**
> * Single, multilevel, hierarchical and multiple inheritance are all supported in Python.
> * In a multilevel chain, each `super().__init__(...)` passes control one level up.
> * With multiple parents, the order in `class C(A, B)` decides which method wins.
> * Deep or wide hierarchies are hard to understand. Keep them shallow.

---

## 2.3 The MRO, the diamond problem and cooperative `super()`

### Method resolution order (MRO)

When you call `obj.method()`, Python searches a list of classes, in order, and uses the first match.
This list is the **method resolution order**, or **MRO**. See it with `Class.__mro__` or `Class.mro()`.

Python computes the MRO with an algorithm called **C3 linearisation**. In simple words it guarantees:

1. A class comes **before** its parents.
2. The parents keep the **order you wrote** them in (`class D(B, C)`: `B` before `C`).
3. Every class appears **exactly once**, and `object` is always last.

### The diamond

The **diamond problem** happens when two parents share a common grandparent:

```text
                 +-------+
                 |   A   |          class A:          ...
                 +---+---+          class B(A):       ...
                /         \         class C(A):       ...
         +-----+-+       +-+-----+  class D(B, C):    ...
         |   B   |       |   C   |
         +-----+-+       +-+-----+  MRO of D (C3):
                \         /
                 +---+---+             D  ->  B  ->  C  ->  A  ->  object
                 |   D   |
                 +-------+          A is visited ONCE, after both B and C.
```

Questions: if `B` and `C` both override a method of `A`, which one does `D` use? And does `A` run twice?
C3 answers both: `D` uses `B` first (listed first), then `C`, then `A`, and `A` appears only once.

In [53]:
# The diamond: MRO is D, B, C, A, object
class A:
    def who(self):
        return "A"


class B(A):
    def who(self):
        return "B>" + super().who()


class C(A):
    def who(self):
        return "C>" + super().who()


class D(B, C):
    def who(self):
        return "D>" + super().who()


print([cls.__name__ for cls in D.__mro__])   # ['D', 'B', 'C', 'A', 'object']
print(D().who())                             # D>B>C>A

['D', 'B', 'C', 'A', 'object']
D>B>C>A


### What `super()` really means

Look at the output `D>B>C>A`. Inside `B.who`, `super().who()` called **`C.who`**, not `A.who`,
even though `C` is not a parent of `B`!

`super()` does **not** mean "my parent". It means **"the next class after me in the MRO of the object's
actual type"**. The object is a `D`, so the MRO is `D, B, C, A, object`, and the class after `B` is `C`.

```text
   MRO of type(self) = D:     D  ->  B  ->  C  ->  A  ->  object
                              |      |      |      |
   super() inside D.who  -----+----->|      |      |
   super() inside B.who  ------------+----->|      |
   super() inside C.who  -------------------+----->|
```

This is what makes **cooperative multiple inheritance** possible: every class calls `super()`,
and each class in the chain runs exactly once.

In [54]:
# Same B, but on a plain B object the class after B is A (not C)
print([cls.__name__ for cls in B.__mro__])   # ['B', 'A', 'object']
print(B().who())                             # B>A

['B', 'A', 'object']
B>A


### Why calling parents by name breaks in a diamond

Older code often calls a parent directly: `Base.__init__(self)`. With one parent this works.
In a diamond, the shared base runs **twice**. Using `super()` everywhere runs it **once**.

In [55]:
# Calling parents by name in a diamond: Base.__init__ runs twice
class Base:
    def __init__(self):
        print("  Base")


class Left(Base):
    def __init__(self):
        Base.__init__(self)
        print("  Left")


class Right(Base):
    def __init__(self):
        Base.__init__(self)
        print("  Right")


class Both(Left, Right):
    def __init__(self):
        Left.__init__(self)
        Right.__init__(self)


print("explicit parent calls:")
obj = Both()
# explicit parent calls:
#   Base
#   Left
#   Base      <- runs a second time
#   Right

explicit parent calls:
  Base
  Left
  Base
  Right


In [56]:
# Cooperative super(): every class calls super(), each class runs exactly once
class Base:
    def __init__(self):
        print("  Base")


class Left(Base):
    def __init__(self):
        super().__init__()
        print("  Left")


class Right(Base):
    def __init__(self):
        super().__init__()
        print("  Right")


class Both(Left, Right):
    def __init__(self):
        super().__init__()


print("cooperative super():")
obj = Both()
# cooperative super():
#   Base
#   Right
#   Left
print([cls.__name__ for cls in Both.__mro__])   # ['Both', 'Left', 'Right', 'Base', 'object']

cooperative super():
  Base
  Right
  Left
['Both', 'Left', 'Right', 'Base', 'object']


The prints appear in the order `Base, Right, Left` because each `__init__` calls `super()` **before**
printing. The calls go down the MRO (`Both -> Left -> Right -> Base`), then the prints happen on the way back.

### Cooperative `super()` with `**kwargs`

In a cooperative chain, each class does not know which class comes next, so it does not know which
arguments the next class needs. The common solution:

* Every `__init__` accepts its own **keyword** arguments plus `**kwargs`.
* It takes what it needs and passes the **rest** on with `super().__init__(**kwargs)`.
* The last class before `object` receives an empty `kwargs` (`object.__init__` takes no arguments).

In [57]:
# Cooperative initialisation with **kwargs: each class takes its own arguments
class Person:
    def __init__(self, *, name, **kwargs):
        super().__init__(**kwargs)           # pass the rest along the MRO
        self.name = name


class Worker(Person):
    def __init__(self, *, salary, **kwargs):
        super().__init__(**kwargs)
        self.salary = salary


class Student(Person):
    def __init__(self, *, school, **kwargs):
        super().__init__(**kwargs)
        self.school = school


class WorkingStudent(Worker, Student):
    pass                                     # nothing to add: the chain does everything


ws = WorkingStudent(name="Meera", salary=20000, school="City College")
print(vars(ws))       # {'name': 'Meera', 'school': 'City College', 'salary': 20000}
print([cls.__name__ for cls in WorkingStudent.__mro__])
# ['WorkingStudent', 'Worker', 'Student', 'Person', 'object']

{'name': 'Meera', 'school': 'City College', 'salary': 20000}
['WorkingStudent', 'Worker', 'Student', 'Person', 'object']


### When C3 cannot find an order

Some class definitions make a consistent order impossible. Python then refuses to create the class.
Example: `class Z(X, Y)` where `Y` is already a subclass of `X`. Rule 1 says `Y` before `X`
(child before parent), rule 2 says `X` before `Y` (the written order). Both cannot be true.

In [58]:
# An impossible MRO raises TypeError when the class is created
class X:
    pass


class Y(X):
    pass


try:
    class Z(X, Y):        # X listed before its own subclass Y
        pass
except TypeError as e:
    print("TypeError:", e)
# TypeError: Cannot create a consistent method resolution order (MRO) for bases X, Y


class Z(Y, X):            # fix: list the more specific class first (or just class Z(Y))
    pass


print([cls.__name__ for cls in Z.__mro__])      # ['Z', 'Y', 'X', 'object']

TypeError: Cannot create a consistent method resolution order (MRO) for bases X, Y
['Z', 'Y', 'X', 'object']


> **Tip:** with single inheritance, `super().__init__(...)` is always right. With multiple inheritance,
> either use `super()` in **every** class (cooperative style with `**kwargs`) or, better, avoid complex
> hierarchies and use mixins or composition (next section).

> **Key points**
> * The MRO is the ordered list of classes Python searches; see `Class.__mro__`.
> * C3 keeps children before parents and the written order of parents; each class appears once.
> * `super()` means "the next class in the MRO of `type(self)`", not "my parent".
> * In a diamond, calling parents by name runs the shared base twice; cooperative `super()` runs it once.
> * Use keyword arguments and `**kwargs` to pass arguments along a cooperative chain.

### Practice 2.3: Inheritance, super() and the MRO

**A. Predict the output**

1. What does this print?
   ```python
   class Animal:
       def sound(self):
           return "..."

       def speak(self):
           return "I say " + self.sound()

   class Cat(Animal):
       def sound(self):
           return "meow"

   print(Cat().speak())
   ```
2. What does this print?
   ```python
   class A:
       def hi(self):
           return "A"

   class B(A):
       pass

   class C(A):
       def hi(self):
           return "C"

   class D(B, C):
       pass

   print(D().hi(), [k.__name__ for k in D.__mro__])
   ```
3. What does this print?
   ```python
   class Base:
       def __init__(self):
           self.ready = True

   class Child(Base):
       def __init__(self):
           self.name = "child"

   print(hasattr(Child(), "ready"))
   ```
4. What does this print?
   ```python
   class P:
       def go(self):
           return "P"

   class Q(P):
       def go(self):
           return "Q" + super().go()

   class R(P):
       def go(self):
           return "R" + super().go()

   class S(R, Q):
       def go(self):
           return "S" + super().go()

   print(S().go())
   ```

**B. Write code**

5. Write a base class `Shape` with `name` and a method `describe()` that returns
   `"<name> with area <area rounded to 2>"` using `self.area()`. Write `Rectangle(Shape)` and
   `Circle(Shape)` that call `super().__init__(name)` and implement `area()`.
6. Write a class `Vehicle(make, wheels)` and a subclass `Motorbike(make)` that always has 2 wheels.
   Add `ElectricMotorbike(make, battery_kwh)` (multilevel) whose `describe()` extends the parent's text
   with `", battery <n> kWh"`.
7. **[Hard]** Write cooperative classes `Logged`, `Timestamped` (both use `**kwargs` and `super()`)
   and `Record(Logged, Timestamped)` so that `Record(text="hi", level="INFO", stamp="10:00")` sets all
   three attributes. Each class appends its name to a shared list `self.init_order` when it runs.
   Check that every class ran exactly once.

**C. Find and fix the bug**

8. `Square(4).area()` fails. Why? Fix it with `super()`.
   ```python
   class Rect:
       def __init__(self, w, h):
           self.w = w
           self.h = h

       def area(self):
           return self.w * self.h

   class Square(Rect):
       def __init__(self, side):
           self.side = side
   ```

**D. Interview questions**

9. What is the MRO and how can you see it?
10. What problem does the diamond cause, and how does Python solve it?
11. **[Hard]** Does `super()` always call the parent class? Explain with the diamond.
12. Why do many style guides recommend avoiding deep inheritance trees?

### Solutions 2.3

1. `I say meow`. `speak` is inherited, but `self.sound()` finds `Cat.sound` first.
2. `C ['D', 'B', 'C', 'A', 'object']`. `B` has no `hi`, so the search continues to `C`, which comes before `A`.
3. `False`. `Child.__init__` does not call `super().__init__()`, so `ready` is never set.
4. `SRQP`. The MRO of `S` is `S, R, Q, P, object`; each `super().go()` moves one step along it.
8. `Square.__init__` never sets `w` and `h`. Call `super().__init__(side, side)`.
9. The MRO is the order in which Python searches classes for an attribute. See it with `Cls.__mro__` or `Cls.mro()`.
10. With `D(B, C)` where `B` and `C` both inherit from `A`, it is unclear which override wins, and `A`'s code
    might run twice. Python's C3 MRO gives a single order (`D, B, C, A, object`) where every class appears
    once, and cooperative `super()` follows that order.
11. No. `super()` returns the next class in the MRO of the object's actual type. Inside `B.who`, for a `D`
    object, the next class is `C`, which is a sibling of `B`, not its parent.
12. Deep trees create tight coupling: a change in a base class can break many subclasses, and it becomes
    hard to see where a method really comes from. Composition is usually more flexible.

In [59]:
# Solutions 2.3 (B and C) and checks for A
import math

# A2 and A4 checks
class P:
    def go(self):
        return "P"


class Q(P):
    def go(self):
        return "Q" + super().go()


class R(P):
    def go(self):
        return "R" + super().go()


class S(R, Q):
    def go(self):
        return "S" + super().go()


assert S().go() == "SRQP"


# Q5
class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        raise NotImplementedError("subclasses must implement area()")

    def describe(self):
        return f"{self.name} with area {self.area():.2f}"


class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__("Rectangle")
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        super().__init__("Circle")
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


assert Rectangle(2, 3).describe() == "Rectangle with area 6.00"
assert Circle(1).describe() == "Circle with area 3.14"


# Q6
class Vehicle:
    def __init__(self, make, wheels):
        self.make = make
        self.wheels = wheels

    def describe(self):
        return f"{self.make} with {self.wheels} wheels"


class Motorbike(Vehicle):
    def __init__(self, make):
        super().__init__(make, wheels=2)


class ElectricMotorbike(Motorbike):
    def __init__(self, make, battery_kwh):
        super().__init__(make)
        self.battery_kwh = battery_kwh

    def describe(self):
        return super().describe() + f", battery {self.battery_kwh} kWh"


assert ElectricMotorbike("Volt", 4).describe() == "Volt with 2 wheels, battery 4 kWh"
assert isinstance(ElectricMotorbike("Volt", 4), Vehicle)


# Q7
class Logged:
    def __init__(self, *, level, **kwargs):
        self.init_order = getattr(self, "init_order", [])
        super().__init__(**kwargs)
        self.level = level
        self.init_order.append("Logged")


class Timestamped:
    def __init__(self, *, stamp, **kwargs):
        self.init_order = getattr(self, "init_order", [])
        super().__init__(**kwargs)
        self.stamp = stamp
        self.init_order.append("Timestamped")


class Record(Logged, Timestamped):
    def __init__(self, *, text, **kwargs):
        self.init_order = []
        super().__init__(**kwargs)
        self.text = text
        self.init_order.append("Record")


rec = Record(text="hi", level="INFO", stamp="10:00")
assert (rec.text, rec.level, rec.stamp) == ("hi", "INFO", "10:00")
assert sorted(rec.init_order) == ["Logged", "Record", "Timestamped"]   # each ran exactly once
assert rec.init_order == ["Timestamped", "Logged", "Record"]           # deepest finishes first


# Q8 fixed
class Rect:
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):
        return self.w * self.h


class Square(Rect):
    def __init__(self, side):
        super().__init__(side, side)


assert Square(4).area() == 16

print("Practice 2.3: all checks passed")

Practice 2.3: all checks passed


---

## 2.4 Mixins, and composition vs inheritance

### Mixins

A **mixin** is a small class that adds **one** extra ability to other classes. It is not meant to be used
on its own and usually has no `__init__`. You "mix it in" with multiple inheritance.

Typical mixins: `ToDictMixin` (convert to a dict), `ReprMixin` (automatic `__repr__`), `JsonMixin`,
`ComparableMixin`. By convention the name ends in `Mixin` and mixins are listed **before** the main base class.

In [60]:
# Mixins add one small ability each
import json


class ToDictMixin:
    def to_dict(self):
        # vars(self) is the instance dictionary; skip internal names
        return {key: value for key, value in vars(self).items() if not key.startswith("_")}


class JsonMixin:
    def to_json(self):
        return json.dumps(self.to_dict(), sort_keys=True)   # relies on to_dict from another mixin


class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary
        self._password_hash = "not-for-export"


class ExportableEmployee(JsonMixin, ToDictMixin, Employee):
    pass


e = ExportableEmployee("Asha", 85000)
print(e.to_dict())          # {'name': 'Asha', 'salary': 85000}
print(e.to_json())          # {"name": "Asha", "salary": 85000}

{'name': 'Asha', 'salary': 85000}
{"name": "Asha", "salary": 85000}


### Composition: "has-a" instead of "is-a"

**Composition** means an object *contains* other objects and uses them to do its work.

* Inheritance models **is-a**: a `Manager` *is an* `Employee`.
* Composition models **has-a**: a `Company` *has* employees; a `Car` *has an* engine.

Inheritance creates **tight coupling**: the child depends on every detail of the parent.
Composition is looser: you can swap the part for another object with the same methods (for example a fake
one in tests). That is why the common advice is **"favour composition over inheritance"**.

In [61]:
# Composition: a Car has an engine, and the engine can be swapped
class PetrolEngine:
    def start(self):
        return "petrol engine roars"


class ElectricMotor:
    def start(self):
        return "electric motor hums"


class Car:
    def __init__(self, model, engine):
        self.model = model
        self.engine = engine              # the Car HAS an engine

    def start(self):
        return f"{self.model}: {self.engine.start()}"   # delegate the work to the part


print(Car("Sedan", PetrolEngine()).start())     # Sedan: petrol engine roars
print(Car("Hatch", ElectricMotor()).start())    # Hatch: electric motor hums

Sedan: petrol engine roars
Hatch: electric motor hums


### Common mistake: inheriting from `list` just to reuse code

A `Stack` is not really a list. If it inherits from `list`, it also inherits `insert`, `sort`, `remove`
and indexing. Callers can then break the "last in, first out" rule. Composition exposes only what you choose.

In [62]:
# Common mistake: inheritance exposes methods that break the stack's rules
class BadStack(list):
    def push(self, item):
        self.append(item)


s = BadStack()
s.push(1)
s.push(2)
s.insert(0, 99)              # allowed, but a stack should never do this
print(s)                     # [99, 1, 2]


# Fix: composition - the stack HAS a list and exposes only stack operations
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def __len__(self):
        return len(self._items)


good = Stack()
good.push(1)
print(hasattr(good, "insert"))   # False

[99, 1, 2]
False


### The running example: a `Company` has employees

A company is not a kind of employee, and not a kind of list. It **has** a list of employees.
We will grow this class with special methods in section 2.6.

In [63]:
# Composition in the running example: Company has Employees
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    def __repr__(self):
        return f"Employee({self.name!r}, {self.dept!r}, {self.salary!r})"


class Company:
    def __init__(self, name):
        self.name = name
        self._staff = []                       # composition: the company owns a list

    def hire(self, employee):
        self._staff.append(employee)

    def payroll(self):
        return sum(e.salary for e in self._staff)

    def by_dept(self, dept):
        return [e for e in self._staff if e.dept == dept]


acme = Company("Acme Corp")
acme.hire(Employee("Asha", "Engineering", 85000))
acme.hire(Employee("Ravi", "Sales", 60000))
acme.hire(Employee("Meera", "Engineering", 90000))

print(acme.payroll())                 # 235000
print(acme.by_dept("Engineering"))    # [Employee('Asha', 'Engineering', 85000), Employee('Meera', 'Engineering', 90000)]

235000
[Employee('Asha', 'Engineering', 85000), Employee('Meera', 'Engineering', 90000)]


### Inheritance or composition? A quick test

```text
     +----------------------------------+
     | Is "Child IS A Parent" true in   |    No
     | every situation, and can a Child |----------> use composition
     | be used anywhere a Parent is?    |            (Child HAS a part)
     +----------------+-----------------+
                      | Yes
                      v
     +----------------------------------+    Yes
     | Do you only want to reuse a few  |----------> composition (or a mixin
     | helper methods?                  |            for one small ability)
     +----------------+-----------------+
                      | No
                      v
               use inheritance
```

> **Interview tip:** a classic question is "composition vs inheritance". Answer with "is-a vs has-a",
> mention tight coupling, and give the `Stack(list)` example.

> **Key points**
> * A mixin adds one small ability; it has no `__init__` and is listed before the main base.
> * Composition (has-a) puts objects inside objects and delegates work to them.
> * Prefer composition; inherit only for a true is-a relationship.
> * Inheriting from built-ins like `list` exposes methods that may break your class's rules.

### Practice 2.4: Mixins and composition

**A. Predict the output**

1. What does this print?
   ```python
   class GreetMixin:
       def greet(self):
           return "Hello, " + self.name

   class User:
       def __init__(self, name):
           self.name = name

   class Member(GreetMixin, User):
       pass

   print(Member("Ravi").greet())
   ```
2. What does this print?
   ```python
   class Wheel:
       pass

   class Bike:
       def __init__(self):
           self.wheels = [Wheel(), Wheel()]

   b = Bike()
   print(len(b.wheels), isinstance(b, Wheel))
   ```

**B. Write code**

3. Write a `ReprMixin` whose `__repr__` returns `ClassName(attr1=value1, attr2=value2)` using `vars(self)`
   (in insertion order, values with `repr`). Use it with a `Product(name, price)` class.
4. Write classes `Battery(capacity)` with `charge_percent` and `Phone(model, battery)` using composition.
   `Phone.status()` returns `"<model>: <percent>%"`. Show that you can pass a different battery object.
5. Write a class `Playlist` that **has** a list of song titles (composition) with `add(title)`,
   `remove(title)` and `titles()` (returns a copy). Do not inherit from `list`.
6. **[Hard]** Write a `Logger` class with `log(message)` that stores messages in a list, and a `Service`
   class that receives a logger in `__init__` (dependency injection) and logs `"start"` and `"stop"` in
   `run()`. Write a `FakeLogger` with the same method for tests and show that `Service` works with both.

**C. Find and fix the bug**

7. Why is this design wrong? Rewrite it with composition.
   ```python
   class Engine:
       def start(self):
           return "vroom"

   class Car(Engine):
       pass
   ```

**D. Interview questions**

8. What is a mixin? How is it different from a normal base class?
9. Explain "favour composition over inheritance" with an example.
10. **[Hard]** What is dependency injection, and how does composition make testing easier?

### Solutions 2.4

1. `Hello, Ravi`. The mixin's method uses `self.name`, which `User.__init__` set.
2. `2 False`. A bike *has* wheels; it is not a wheel.
7. A car is not a kind of engine ("is-a" is false). Inheritance also makes it hard to switch engine types.
   Give the car an engine attribute: `self.engine = engine` and delegate `start()` to it.
8. A mixin is a small class that adds one reusable ability (for example `to_dict`) to other classes through
   multiple inheritance. It is not used on its own, usually has no `__init__` and no state of its own.
   A normal base class models what the object *is*.
9. Inheritance couples a class tightly to its parent. With composition the object holds parts and
   delegates to them, so parts can be swapped (petrol engine vs electric motor) without new subclasses.
10. Dependency injection means passing the objects a class needs (its dependencies) into it, usually
    through `__init__`, instead of creating them inside. In tests you can then pass a fake object
    (like `FakeLogger`) that records calls, without touching files, networks or databases.

In [64]:
# Solutions 2.4 (B and C)

# Q3
class ReprMixin:
    def __repr__(self):
        fields = ", ".join(f"{key}={value!r}" for key, value in vars(self).items())
        return f"{type(self).__name__}({fields})"


class Product(ReprMixin):
    def __init__(self, name, price):
        self.name = name
        self.price = price


assert repr(Product("pen", 2.5)) == "Product(name='pen', price=2.5)"


# Q4
class Battery:
    def __init__(self, capacity, charge_percent=100):
        self.capacity = capacity
        self.charge_percent = charge_percent


class Phone:
    def __init__(self, model, battery):
        self.model = model
        self.battery = battery

    def status(self):
        return f"{self.model}: {self.battery.charge_percent}%"


assert Phone("P1", Battery(4000)).status() == "P1: 100%"
assert Phone("P1", Battery(5000, charge_percent=42)).status() == "P1: 42%"


# Q5
class Playlist:
    def __init__(self):
        self._songs = []

    def add(self, title):
        self._songs.append(title)

    def remove(self, title):
        self._songs.remove(title)

    def titles(self):
        return list(self._songs)          # a copy, so callers cannot change our list


pl = Playlist()
pl.add("Song A")
pl.add("Song B")
pl.remove("Song A")
copy_of_titles = pl.titles()
copy_of_titles.append("Hacked")
assert pl.titles() == ["Song B"]
assert not hasattr(pl, "sort")


# Q6
class Logger:
    def __init__(self):
        self.messages = []

    def log(self, message):
        self.messages.append(message)


class FakeLogger:
    def __init__(self):
        self.calls = 0

    def log(self, message):
        self.calls += 1


class Service:
    def __init__(self, logger):
        self.logger = logger              # injected dependency

    def run(self):
        self.logger.log("start")
        self.logger.log("stop")


real = Logger()
Service(real).run()
assert real.messages == ["start", "stop"]
fake = FakeLogger()
Service(fake).run()
assert fake.calls == 2


# Q7 fixed
class Engine:
    def start(self):
        return "vroom"


class Car:
    def __init__(self, engine):
        self.engine = engine

    def start(self):
        return self.engine.start()


assert Car(Engine()).start() == "vroom"
assert not isinstance(Car(Engine()), Engine)

print("Practice 2.4: all checks passed")

Practice 2.4: all checks passed


---

## 2.5 Polymorphism, duck typing and abstract base classes

### Polymorphism

**Polymorphism** ("many forms") means: the same method call works on objects of different classes,
and each class does it in its own way. The calling code does not need `if type(x) == ...` checks.

In [65]:
# Polymorphism: one loop, many classes, each with its own annual_pay
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def annual_pay(self):
        return self.salary


class Manager(Employee):
    def __init__(self, name, salary, bonus):
        super().__init__(name, salary)
        self.bonus = bonus

    def annual_pay(self):
        return self.salary + self.bonus


class Intern(Employee):
    def annual_pay(self):
        return self.salary // 2


def total_payroll(people):
    # no type checks: each object knows how to compute its own pay
    return sum(p.annual_pay() for p in people)


staff = [Employee("Ravi", 60000), Manager("Asha", 120000, 15000), Intern("Meera", 40000)]
print(total_payroll(staff))          # 215000

215000


> **Common mistake:** writing `if isinstance(p, Manager): ... elif isinstance(p, Intern): ...` in the
> caller. Every new class forces you to edit that `if` chain. Put the behaviour in the classes instead
> (this is the "open/closed principle", section 2.10).

### Duck typing

> "If it walks like a duck and quacks like a duck, it is a duck."

Python does not check *what class* an object is. It only checks, at the moment of the call, that the
object *has* the method. So polymorphism works even **without a common base class**. This is called
**duck typing**.

In [66]:
# Duck typing: no shared base class needed, only the same method name
class Contractor:
    def __init__(self, name, hours, rate):
        self.name = name
        self.hours = hours
        self.rate = rate

    def annual_pay(self):
        return self.hours * self.rate


mixed = staff + [Contractor("Kiran", 1000, 50)]
print(total_payroll(mixed))          # 265000  (Contractor is not an Employee, and it still works)

265000


Built-in functions use duck typing too. `len(x)` works for any object with `__len__`; a `for` loop works
for any object with `__iter__`. You will write such methods in section 2.6.

In [67]:
# Built-ins rely on duck typing: anything with __len__ works with len()
class Team:
    def __init__(self, *names):
        self._names = list(names)

    def __len__(self):
        return len(self._names)


print(len(Team("Asha", "Ravi")))     # 2
print(len("abc"), len([1, 2]), len({"k": 1}))   # 3 2 1

2
3 2 1


### Abstract base classes (ABCs)

Duck typing is flexible, but a missing method is only noticed when it is called, maybe much later.
An **abstract base class** lets you declare a **contract**: "every subclass must implement these methods".

* Inherit from `ABC` (in the `abc` module).
* Mark required methods with `@abstractmethod`.
* A class that still has abstract methods **cannot be instantiated**: Python raises `TypeError` at creation time.
* An ABC may also contain normal (concrete) methods that use the abstract ones.

In [68]:
# An ABC defines a contract: every Shape must implement area() and perimeter()
from abc import ABC, abstractmethod
import math


class Shape(ABC):
    @abstractmethod
    def area(self):
        """Return the area."""

    @abstractmethod
    def perimeter(self):
        """Return the perimeter."""

    def describe(self):                       # concrete method built on the abstract ones
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        return math.pi * self.r ** 2

    def perimeter(self):
        return 2 * math.pi * self.r


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

    def perimeter(self):
        return 4 * self.side


for shape in (Circle(1), Square(2)):
    print(shape.describe())
# Circle: area=3.14, perimeter=6.28
# Square: area=4.00, perimeter=8.00

Circle: area=3.14, perimeter=6.28
Square: area=4.00, perimeter=8.00


In [69]:
# You cannot create an ABC, or a subclass that forgot an abstract method
try:
    Shape()
except TypeError as e:
    print("TypeError:", e)
# TypeError: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'


class Triangle(Shape):
    def area(self):
        return 6                              # perimeter() is missing


try:
    Triangle()
except TypeError as e:
    print("TypeError:", e)
# TypeError: Can't instantiate abstract class Triangle without an implementation for abstract method 'perimeter'

TypeError: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'
TypeError: Can't instantiate abstract class Triangle without an implementation for abstract method 'perimeter'


> **Note:** the error is raised when you **create** the object, not when you call the missing method.
> That is the main benefit over duck typing: the bug is found early.

### Abstract properties

You can require a property too: put `@property` above `@abstractmethod`.

In [70]:
# The running example: an abstract Payable contract with an abstract property
from abc import ABC, abstractmethod


class Payable(ABC):
    @property
    @abstractmethod
    def monthly_cost(self):
        """Cost per month for the company."""

    def yearly_cost(self):
        return self.monthly_cost * 12


class SalariedEmployee(Payable):
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    @property
    def monthly_cost(self):
        return self.salary / 12


class Subscription(Payable):                 # not an employee, but still Payable
    def __init__(self, service, price):
        self.service = service
        self.price = price

    @property
    def monthly_cost(self):
        return self.price


costs = [SalariedEmployee("Asha", 84000), Subscription("Cloud storage", 300)]
print(sum(item.yearly_cost() for item in costs))   # 87600.0

87600.0


### ABCs in the standard library

The module `collections.abc` defines ABCs such as `Iterable`, `Sized`, `Sequence`, `Mapping`, `Callable`.
Some of them recognise a class just because it has the right method, even without inheriting.
This is a nice way to test for an ability with `isinstance`.

In [71]:
# collections.abc: test for abilities with isinstance
from collections.abc import Sized, Iterable, Callable


class Team:
    def __init__(self, *names):
        self._names = list(names)

    def __len__(self):
        return len(self._names)


t = Team("Asha")
print(isinstance(t, Sized))          # True   (it has __len__; no inheritance needed)
print(isinstance(t, Iterable))       # False  (no __iter__)
print(isinstance(len, Callable))     # True

True
False
True


> **Tip:** `typing.Protocol` describes a contract by method names only ("structural typing"), checked by
> type checkers such as mypy. It is the typed version of duck typing; see Part 6.

> **Key points**
> * Polymorphism: the same call (`p.annual_pay()`) runs different code for different classes.
> * Duck typing: Python only cares that the object has the method, not what class it is.
> * An ABC (`ABC` + `@abstractmethod`) enforces a contract; incomplete subclasses fail at creation.
> * Put behaviour in the classes, not in `if isinstance(...)` chains in the caller.

### Practice 2.5: Polymorphism and ABCs

**A. Predict the output**

1. What does this print?
   ```python
   class Dog:
       def speak(self):
           return "Woof"

   class Robot:
       def speak(self):
           return "Beep"

   print([x.speak() for x in (Dog(), Robot())])
   ```
2. What happens here?
   ```python
   from abc import ABC, abstractmethod

   class Base(ABC):
       @abstractmethod
       def run(self):
           ...

   class Child(Base):
       def walk(self):
           return "walking"

   Child()
   ```
3. What does this print?
   ```python
   from abc import ABC, abstractmethod

   class Base(ABC):
       @abstractmethod
       def value(self):
           ...

       def double(self):
           return 2 * self.value()

   class Five(Base):
       def value(self):
           return 5

   print(Five().double())
   ```

**B. Write code**

4. Write an ABC `Notifier` with abstract method `send(message)` that returns a string, and two subclasses
   `EmailNotifier(address)` and `SmsNotifier(phone)`. Write `notify_all(notifiers, message)` that returns a
   list of the results.
5. Write an ABC `PaymentMethod` with abstract `pay(amount)` and a concrete method `pay_with_fee(amount)`
   that adds a 2% fee and calls `pay`. Implement `Card` and `Wallet` (the wallet raises `ValueError` if the
   balance is too low).
6. Write a function `area_sum(shapes)` that works for any objects with an `area()` method (duck typing).
   Test it with two classes that do **not** share a base class.
7. **[Hard]** Write an ABC `Storage` with abstract methods `save(key, value)` and `load(key)`, and a
   concrete method `load_or_default(key, default)`. Implement `MemoryStorage` (a dict). Show that a subclass
   missing `load` cannot be created.

**C. Find and fix the bug**

8. Every time a new animal is added, this function must be changed. Rewrite it with polymorphism.
   ```python
   def sound(animal):
       if isinstance(animal, Dog):
           return "Woof"
       elif isinstance(animal, Cat):
           return "Meow"
   ```

**D. Interview questions**

9. What is polymorphism? Give an example.
10. What is duck typing? What is its main risk?
11. **[Hard]** ABC vs duck typing vs `Protocol`: when would you choose each?

### Solutions 2.5

1. `['Woof', 'Beep']`. Duck typing: both have `speak`.
2. `TypeError: Can't instantiate abstract class Child without an implementation for abstract method 'run'`.
3. `10`. The concrete method `double` uses the subclass's `value`.
8. Give each class a `sound()` method and call `animal.sound()`. New animals add their own method;
   the caller never changes.
9. One interface, many implementations: `shape.area()` computes a circle's or a square's area depending on
   the object.
10. Using an object based on the methods it has, not its class. Risk: a missing or misspelt method is found
    only when that line runs (an `AttributeError` at runtime), possibly in production.
11. Duck typing: small scripts and flexible code. ABC: you own the hierarchy and want Python to refuse
    incomplete subclasses at creation time, and to share concrete helper methods. `Protocol`: you want a type
    checker to verify the contract without forcing classes to inherit (for example third-party classes).

In [72]:
# Solutions 2.5 (B and C)
from abc import ABC, abstractmethod
import math


# Q4
class Notifier(ABC):
    @abstractmethod
    def send(self, message):
        """Send the message and return a description."""


class EmailNotifier(Notifier):
    def __init__(self, address):
        self.address = address

    def send(self, message):
        return f"email to {self.address}: {message}"


class SmsNotifier(Notifier):
    def __init__(self, phone):
        self.phone = phone

    def send(self, message):
        return f"sms to {self.phone}: {message}"


def notify_all(notifiers, message):
    return [n.send(message) for n in notifiers]


assert notify_all([EmailNotifier("a@example.com"), SmsNotifier("555-0101")], "hi") == [
    "email to a@example.com: hi",
    "sms to 555-0101: hi",
]
assert notify_all([], "hi") == []


# Q5
class PaymentMethod(ABC):
    FEE = 0.02

    @abstractmethod
    def pay(self, amount):
        """Pay the amount and return a receipt string."""

    def pay_with_fee(self, amount):
        return self.pay(round(amount * (1 + self.FEE), 2))


class Card(PaymentMethod):
    def pay(self, amount):
        return f"card paid {amount}"


class Wallet(PaymentMethod):
    def __init__(self, balance):
        self.balance = balance

    def pay(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money in wallet")
        self.balance -= amount
        return f"wallet paid {amount}"


assert Card().pay_with_fee(100) == "card paid 102.0"
w = Wallet(150)
assert w.pay_with_fee(100) == "wallet paid 102.0" and w.balance == 48.0
try:
    w.pay_with_fee(100)
except ValueError:
    pass
else:
    raise AssertionError("wallet overdraft accepted")


# Q6
def area_sum(shapes):
    return sum(s.area() for s in shapes)


class Sq:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


class Disc:
    def __init__(self, r):
        self.r = r

    def area(self):
        return math.pi * self.r ** 2


assert math.isclose(area_sum([Sq(2), Disc(1)]), 4 + math.pi)
assert area_sum([]) == 0


# Q7
class Storage(ABC):
    @abstractmethod
    def save(self, key, value):
        ...

    @abstractmethod
    def load(self, key):
        ...

    def load_or_default(self, key, default):
        try:
            return self.load(key)
        except KeyError:
            return default


class MemoryStorage(Storage):
    def __init__(self):
        self._data = {}

    def save(self, key, value):
        self._data[key] = value

    def load(self, key):
        return self._data[key]


store = MemoryStorage()
store.save("a", 1)
assert store.load("a") == 1
assert store.load_or_default("missing", 0) == 0


class BrokenStorage(Storage):
    def save(self, key, value):
        pass


try:
    BrokenStorage()
except TypeError as e:
    assert "load" in str(e)
else:
    raise AssertionError("incomplete subclass was created")


# Q8 fixed
class Dog:
    def sound(self):
        return "Woof"


class Cat:
    def sound(self):
        return "Meow"


def sound(animal):
    return animal.sound()


assert [sound(a) for a in (Dog(), Cat())] == ["Woof", "Meow"]

print("Practice 2.5: all checks passed")

Practice 2.5: all checks passed


---

## 2.6 Special (dunder) methods and operator overloading

Python turns operators and built-in functions into **method calls**. When you write `a + b`, Python
calls `a.__add__(b)`. When you write `len(x)`, Python calls `x.__len__()`.
If your class defines these **special methods**, its objects work with normal Python syntax.
This is called **operator overloading**.

| You write | Python calls | Section |
|---|---|---|
| `repr(x)`, `str(x)` | `x.__repr__()`, `x.__str__()` | Part 1, 1.9 |
| `x == y`, `x != y` | `x.__eq__(y)`, `x.__ne__(y)` | equality |
| `hash(x)`, `{x}`, `{x: 1}` | `x.__hash__()` | hashing |
| `x < y`, `x <= y`, `x > y`, `x >= y` | `__lt__`, `__le__`, `__gt__`, `__ge__` | ordering |
| `x + y`, `x - y`, `x * y`, `x / y` | `__add__`, `__sub__`, `__mul__`, `__truediv__` | arithmetic |
| `y + x` when `y` cannot | `x.__radd__(y)` | reflected operators |
| `x += y` | `x.__iadd__(y)` (falls back to `__add__`) | in-place |
| `len(x)`, `x[i]`, `x[i] = v`, `v in x` | `__len__`, `__getitem__`, `__setitem__`, `__contains__` | containers |
| `for v in x` | `x.__iter__()` | containers (more in Part 3) |
| `bool(x)`, `if x:` | `x.__bool__()` (falls back to `__len__`) | truthiness |
| `x(...)` | `x.__call__(...)` | callable objects |
| `with x:` | `__enter__`, `__exit__` | Part 5 |

> **Note:** you never call dunder methods directly in normal code (`len(x)`, not `x.__len__()`).
> You *define* them; Python calls them.

### Equality: `__eq__`

By default, `==` compares identity (Part 1, section 1.10). Define `__eq__` to compare by value.

* Return `True` or `False` for objects you understand.
* Return the special value **`NotImplemented`** for other types. Python then tries the other object's
  method, and finally falls back to identity (so `==` gives `False` instead of crashing).
* `!=` is derived automatically from `__eq__`.

In [73]:
# Value equality: two Employee objects with the same id are equal
class Employee:
    def __init__(self, emp_id, name):
        self.emp_id = emp_id
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Employee):
            return NotImplemented            # "I do not know how to compare with this"
        return self.emp_id == other.emp_id

    def __repr__(self):
        return f"Employee({self.emp_id!r}, {self.name!r})"


a = Employee(7, "Asha")
b = Employee(7, "Asha R.")                   # same person, name spelt differently

print(a == b)          # True
print(a != b)          # False  (derived from __eq__)
print(a == "Asha")     # False  (NotImplemented on both sides -> falls back to identity)

True
False
False


### Hashing: the `__eq__` / `__hash__` rule

A **hash** is a number computed from an object. Dicts and sets use it to find keys quickly.
The rule is: **if `a == b`, then `hash(a) == hash(b)`** must hold.

* When a class defines `__eq__` but not `__hash__`, Python sets `__hash__ = None`.
  The objects become **unhashable**: they cannot be set members or dict keys.
* To make them hashable, define `__hash__` from the **same fields** used in `__eq__`, usually
  `hash((field1, field2))`.
* Only hash fields that never change. If a key's hash changes while it is in a set, the set cannot find it.

In [74]:
# __eq__ without __hash__ makes objects unhashable (Employee from the cell above)
try:
    {a}
except TypeError as e:
    print("TypeError:", e)
# Python 3.14: TypeError: cannot use 'Employee' as a set element (unhashable type: 'Employee')
# Python 3.12 and 3.13 print the shorter message: unhashable type: 'Employee'

TypeError: cannot use 'Employee' as a set element (unhashable type: 'Employee')


In [75]:
# Fix: define __hash__ consistent with __eq__ (same field: emp_id)
class Employee:
    def __init__(self, emp_id, name):
        self.emp_id = emp_id
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Employee):
            return NotImplemented
        return self.emp_id == other.emp_id

    def __hash__(self):
        return hash(self.emp_id)             # equal objects -> equal hashes

    def __repr__(self):
        return f"Employee({self.emp_id!r}, {self.name!r})"


people = {Employee(7, "Asha"), Employee(7, "Asha R."), Employee(8, "Ravi")}
print(len(people))                           # 2  (the two id-7 objects count as one)
salaries = {Employee(8, "Ravi"): 60000}
print(salaries[Employee(8, "R. Kumar")])     # 60000  (found by value)

2
60000


### Ordering: `__lt__` and `functools.total_ordering`

`sorted()`, `min()`, `max()` and `<` need ordering methods. Writing all four (`<`, `<=`, `>`, `>=`) is
boring. The decorator **`@total_ordering`** (from `functools`) generates the missing ones from
`__eq__` and **one** of them (usually `__lt__`).

In [76]:
# Ordering with total_ordering: define __eq__ and __lt__, get the rest for free
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, text):
        self.parts = tuple(int(p) for p in text.split("."))   # "1.10.0" -> (1, 10, 0)

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts == other.parts

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts < other.parts                         # tuples compare item by item

    def __hash__(self):
        return hash(self.parts)

    def __repr__(self):
        return "Version('" + ".".join(map(str, self.parts)) + "')"


print(Version("1.10.0") > Version("1.9.3"))      # True   (a string comparison would say False)
print(Version("2.0") >= Version("2.0"))          # True   (>= generated by total_ordering)
print(sorted([Version("1.10"), Version("1.2"), Version("1.9")]))
# [Version('1.2'), Version('1.9'), Version('1.10')]
print(max(Version("3.12"), Version("3.9")))      # Version('3.12')

True
True
[Version('1.2'), Version('1.9'), Version('1.10')]
Version('3.12')


> **Tip:** if you only need to sort, you do not need ordering methods at all:
> `sorted(staff, key=lambda e: e.salary)` is often simpler. Define `__lt__` when objects have one
> obvious natural order (versions, money, dates).

### Arithmetic operators and `NotImplemented`

Let us build a `Money` class for the company domain. Rules:

* `Money + Money` gives `Money` (same currency only).
* `Money * number` and `number * Money` give `Money`.
* Anything else is not supported, so the method returns `NotImplemented`.

How Python evaluates `x + y`:

```text
                    x + y
                      |
                      v
       +-----------------------------+   result is not    +----------------+
       | r = x.__add__(y)            |--NotImplemented--->| use r          |
       +--------------+--------------+                    +----------------+
                      | r is NotImplemented (or no __add__)
                      v
       +-----------------------------+   result is not    +----------------+
       | r = y.__radd__(x)           |--NotImplemented--->| use r          |
       | (the "reflected" method)    |                    +----------------+
       +--------------+--------------+
                      | r is NotImplemented (or no __radd__)
                      v
       +-----------------------------+
       | raise TypeError:            |
       | unsupported operand type(s) |
       +-----------------------------+
```

> **Common mistake:** raising `TypeError` yourself (or returning `False`) inside `__add__` for unknown
> types. Return `NotImplemented` instead, so the other operand gets a chance.
> Note that `NotImplemented` (a value) is different from `NotImplementedError` (an exception).

In [77]:
# Money: +, -, * with numbers on both sides, and ==
from functools import total_ordering


@total_ordering
class Money:
    def __init__(self, amount, currency="USD"):
        self.amount = round(amount, 2)
        self.currency = currency

    def _check(self, other):
        return isinstance(other, Money) and other.currency == self.currency

    def __add__(self, other):
        if not self._check(other):
            return NotImplemented
        return Money(self.amount + other.amount, self.currency)

    def __sub__(self, other):
        if not self._check(other):
            return NotImplemented
        return Money(self.amount - other.amount, self.currency)

    def __mul__(self, factor):
        if not isinstance(factor, (int, float)):
            return NotImplemented
        return Money(self.amount * factor, self.currency)

    __rmul__ = __mul__                         # number * Money: same result as Money * number

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __lt__(self, other):
        if not self._check(other):
            return NotImplemented
        return self.amount < other.amount

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __repr__(self):
        return f"Money({self.amount!r}, {self.currency!r})"


salary = Money(5000)
bonus = Money(750.5)

print(salary + bonus)            # Money(5750.5, 'USD')
print(salary - bonus)            # Money(4249.5, 'USD')
print(salary * 2)                # Money(10000, 'USD')
print(3 * bonus)                 # Money(2251.5, 'USD')  (int.__mul__ fails, then Money.__rmul__)
print(salary > bonus)            # True

Money(5750.5, 'USD')
Money(4249.5, 'USD')
Money(10000, 'USD')
Money(2251.5, 'USD')
True


In [78]:
# Unsupported combinations raise a clear TypeError (Money from the cell above)
for attempt in (lambda: salary + 100, lambda: salary + Money(1, "EUR"), lambda: salary * None):
    try:
        attempt()
    except TypeError as e:
        print("TypeError:", e)
# TypeError: unsupported operand type(s) for +: 'Money' and 'int'
# TypeError: unsupported operand type(s) for +: 'Money' and 'Money'
# TypeError: unsupported operand type(s) for *: 'Money' and 'NoneType'

TypeError: unsupported operand type(s) for +: 'Money' and 'int'
TypeError: unsupported operand type(s) for +: 'Money' and 'Money'
TypeError: unsupported operand type(s) for *: 'Money' and 'NoneType'


### Making `sum()` work: `__radd__`

`sum(items)` starts from `0` and computes `0 + first_item`. `int.__add__` does not know `Money`, so Python
tries `Money.__radd__(0)`. Two easy solutions:

1. Give `sum` a start value: `sum(items, Money(0))`.
2. Define `__radd__` that accepts `0`.

In [79]:
# sum() and Money
payments = [Money(100), Money(250.25), Money(49.75)]

try:
    sum(payments)
except TypeError as e:
    print("TypeError:", e)                   # TypeError: unsupported operand type(s) for +: 'int' and 'Money'

print(sum(payments, Money(0)))               # Money(400.0, 'USD')   (solution 1: a start value)


class SummableMoney(Money):
    def __radd__(self, other):
        if other == 0:                       # sum() starts with 0
            return self
        return NotImplemented


print(sum([SummableMoney(10), SummableMoney(5)]))   # Money(15, 'USD')  (solution 2: __radd__)

TypeError: unsupported operand type(s) for +: 'int' and 'Money'
Money(400.0, 'USD')
Money(15, 'USD')


### In-place operators: `__iadd__`

`x += y` first tries `x.__iadd__(y)`. If the class has no `__iadd__`, Python uses `x = x + y`
(a new object). Mutable containers define `__iadd__` to change themselves **in place** and must
**return `self`**.

In [80]:
# __iadd__ changes the object in place and returns self
class Cart:
    def __init__(self):
        self.items = []

    def __iadd__(self, item):
        self.items.append(item)
        return self                          # required: the result is assigned back to the name


cart = Cart()
same = cart
cart += "pen"
cart += "book"
print(cart.items)                            # ['pen', 'book']
print(cart is same)                          # True  (same object, changed in place)

total = Money(10)
before = total
total += Money(5)                            # Money has no __iadd__ -> total = total + Money(5)
print(total, total is before)                # Money(15, 'USD') False  (a new object)

['pen', 'book']
True
Money(15, 'USD') False


### Container protocol: `__len__`, `__getitem__`, `__contains__`, `__iter__`

Our `Company` (section 2.4) holds employees. With a few dunder methods it behaves like a built-in container:
`len(company)`, `company[0]`, `"Asha" in company`, `for e in company`.

In [81]:
# The running example: Company behaves like a container
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    def __repr__(self):
        return f"Employee({self.name!r}, {self.dept!r}, {self.salary!r})"


class Company:
    def __init__(self, name, staff=None):
        self.name = name
        self._staff = list(staff) if staff else []

    def hire(self, employee):
        self._staff.append(employee)

    def __len__(self):                           # len(company)
        return len(self._staff)

    def __getitem__(self, index):                # company[0], company[-1], company[0:2]
        result = self._staff[index]
        if isinstance(index, slice):
            return Company(self.name, result)    # a slice gives a smaller Company
        return result

    def __contains__(self, name):                # "Asha" in company
        return any(e.name == name for e in self._staff)

    def __iter__(self):                          # for e in company
        return iter(self._staff)

    def __repr__(self):
        return f"Company({self.name!r}, {len(self)} employees)"


acme = Company("Acme Corp")
acme.hire(Employee("Asha", "Engineering", 85000))
acme.hire(Employee("Ravi", "Sales", 60000))
acme.hire(Employee("Meera", "HR", 52000))

print(len(acme))                    # 3
print(acme[0])                      # Employee('Asha', 'Engineering', 85000)
print(acme[-1].name)                # Meera
print(acme[0:2])                    # Company('Acme Corp', 2 employees)
print("Ravi" in acme)               # True
print([e.name for e in acme])       # ['Asha', 'Ravi', 'Meera']
print(max(acme, key=lambda e: e.salary).name)   # Asha  (works because Company is iterable)

3
Employee('Asha', 'Engineering', 85000)
Meera
Company('Acme Corp', 2 employees)
True
['Asha', 'Ravi', 'Meera']
Asha


> **Note:** if a class has `__getitem__` (accepting 0, 1, 2, ...) but no `__iter__`, a `for` loop still works:
> Python calls `__getitem__(0)`, `__getitem__(1)`, ... until `IndexError`. Defining `__iter__` is clearer.
> Iterators are covered in depth in Part 3.

### Truthiness: `__bool__` and `__len__`

`if obj:` calls `obj.__bool__()`. If there is no `__bool__`, Python uses `__len__()` (zero means false).
If neither exists, every object is true.

In [82]:
# Truthiness: Company has __len__, so an empty company is falsy
empty = Company("Startup")
print(bool(empty), bool(acme))          # False True


class Account:
    def __init__(self, balance):
        self.balance = balance

    def __bool__(self):                  # an account is "true" if it has money
        return self.balance > 0


print(bool(Account(0)), bool(Account(10)))   # False True
if not Account(0):
    print("empty account")               # empty account

False True
False True
empty account


### Callable objects: `__call__`

An object with `__call__` can be called like a function: `obj(args)`. This is useful for a function that
needs to **remember settings or state** between calls.

In [83]:
# A callable object: a tax calculator that remembers its rate and how often it was used
class TaxCalculator:
    def __init__(self, rate):
        self.rate = rate
        self.calls = 0

    def __call__(self, amount):
        self.calls += 1
        return round(amount * self.rate, 2)


income_tax = TaxCalculator(0.2)
print(income_tax(50000))            # 10000.0
print(income_tax(1234.5))           # 246.9
print(income_tax.calls)             # 2
print(callable(income_tax))         # True

10000.0
246.9
2
True


> **Common mistake:** returning a value from `__iadd__` other than `self`, or forgetting `return self`.
> Then `x += y` sets `x` to `None`.

> **Key points**
> * Operators and built-ins call dunder methods; define them to make your objects feel built-in.
> * `__eq__` for value equality; if you define it, define `__hash__` from the same immutable fields.
> * `@total_ordering` + `__eq__` + `__lt__` gives all comparisons.
> * Return `NotImplemented` for unsupported types; use `__radd__`/`__rmul__` for reversed operands.
> * `__len__`, `__getitem__`, `__contains__`, `__iter__` make a container; `__bool__` controls truthiness.
> * `__call__` makes an object callable, like a function with memory.

### Practice 2.6: Special methods

**A. Predict the output**

1. What does this print?
   ```python
   class Box:
       def __init__(self, items):
           self.items = items

       def __len__(self):
           return len(self.items)

   print(len(Box([1, 2, 3])), bool(Box([])))
   ```
2. What does this print?
   ```python
   class P:
       def __init__(self, x):
           self.x = x

       def __eq__(self, other):
           return isinstance(other, P) and self.x == other.x

   print(P(1) == P(1), P(1) in [P(1)], hasattr(P, "__hash__"), P.__hash__)
   ```
3. What does this print?
   ```python
   class Meters:
       def __init__(self, v):
           self.v = v

       def __add__(self, other):
           return Meters(self.v + other.v)

       def __repr__(self):
           return f"Meters({self.v})"

   x = Meters(1)
   x += Meters(2)
   print(x)
   ```
4. Which methods does Python try, and in which order, for `2 * Vector(1, 1)`?

**B. Write code**

5. Write a class `Vector2D` supporting `+`, `-`, `*` by a number on **both sides**, `abs()` (length),
   `==`, `hash` and `__repr__` like `Vector2D(3, 4)`.
6. Write a class `Fraction` (do not use the `fractions` module) with numerator and denominator stored in
   lowest terms (use `math.gcd`, sign on the numerator), `+`, `*`, `==`, `<` (with `total_ordering`),
   `__hash__` and `__repr__` like `Fraction(1, 2)`. Raise `ZeroDivisionError` for denominator 0.
7. Write a class `Inventory` that maps item names to quantities with `inv["apple"]` (returns 0 for unknown
   items), `inv["apple"] = 5`, `del inv["apple"]`, `len(inv)`, `"apple" in inv` and iteration over names in
   sorted order.
8. **[Hard]** Write a class `Polynomial` from a list of coefficients (lowest power first) with `__call__(x)`
   to evaluate it, `+` for two polynomials, `__eq__`, and `__repr__` like `Polynomial([1, 0, 2])`.
   Trailing zero coefficients should be removed so that equal polynomials compare equal.

**C. Find and fix the bug**

9. Putting `Point` objects in a set fails. Fix it.
   ```python
   class Point:
       def __init__(self, x, y):
           self.x = x
           self.y = y

       def __eq__(self, other):
           return (self.x, self.y) == (other.x, other.y)

   print(len({Point(1, 2), Point(1, 2)}))
   ```
10. `Point(1, 2) == "text"` crashes with `AttributeError` for the class above. Why? Fix it.

**D. Interview questions**

11. What is the difference between `NotImplemented` and `NotImplementedError`?
12. Why must equal objects have equal hashes? What goes wrong if the hash uses a field that changes?
13. **[Hard]** How does `x + y` decide between `x.__add__` and `y.__radd__`?

### Solutions 2.6

1. `3 False`. `bool()` falls back to `__len__`, and length 0 is false.
2. `True True True None`. `in` on a list uses `==`. Defining `__eq__` without `__hash__` sets `__hash__` to
   `None` (the attribute exists, but its value is `None`, so instances are unhashable).
3. `Meters(3)`. There is no `__iadd__`, so `x += y` becomes `x = x + y`.
4. First `int.__mul__(2, vector)`, which returns `NotImplemented`; then `Vector.__rmul__(vector, 2)`.
   If both give `NotImplemented`, Python raises `TypeError`.
9. `__eq__` without `__hash__` makes `Point` unhashable. Add `def __hash__(self): return hash((self.x, self.y))`.
   The set then has length 1.
10. `other.x` does not exist on a string. Check the type first and return `NotImplemented`:
    `if not isinstance(other, Point): return NotImplemented`. Then `==` returns `False`.
11. `NotImplemented` is a special **value** returned by binary dunder methods to say "I cannot handle this
    type, try the other operand". `NotImplementedError` is an **exception** you raise in a method that a
    subclass must override (or that is not written yet).
12. Dicts and sets first use the hash to find the bucket, then `==` to confirm. If equal objects had
    different hashes, a lookup would search the wrong bucket and miss. If a field used in the hash changes
    after the object is in a set, the object is in the bucket for its *old* hash, so lookups fail.
13. Python calls `x.__add__(y)`. If that returns `NotImplemented` (or does not exist), it calls
    `y.__radd__(x)`. If that also fails, it raises `TypeError`. Special case: if `y`'s type is a
    **subclass** of `x`'s type and overrides `__radd__`, Python tries `y.__radd__(x)` first.

In [84]:
# Solutions 2.6 (B and C) and checks for A
import math
from functools import total_ordering


# A2 check
class P:
    def __init__(self, x):
        self.x = x

    def __eq__(self, other):
        return isinstance(other, P) and self.x == other.x


assert P.__hash__ is None


# Q5
class Vector2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, k):
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vector2D(self.x * k, self.y * k)

    __rmul__ = __mul__

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __eq__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"


v = Vector2D(3, 4)
assert abs(v) == 5.0
assert v * 2 == 2 * v == Vector2D(6, 8)
assert v + v - v == v
assert repr(v) == "Vector2D(3, 4)"
assert len({Vector2D(1, 1), Vector2D(1, 1)}) == 1
assert (v == "v") is False


# Q6
@total_ordering
class Fraction:
    def __init__(self, num, den=1):
        if den == 0:
            raise ZeroDivisionError("denominator must not be 0")
        if den < 0:                                  # keep the sign on the numerator
            num, den = -num, -den
        g = math.gcd(num, den)
        self.num = num // g
        self.den = den // g

    def __add__(self, other):
        if isinstance(other, int):
            other = Fraction(other)
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.num * other.den + other.num * self.den, self.den * other.den)

    __radd__ = __add__

    def __mul__(self, other):
        if isinstance(other, int):
            other = Fraction(other)
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.num * other.num, self.den * other.den)

    __rmul__ = __mul__

    def __eq__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return (self.num, self.den) == (other.num, other.den)

    def __lt__(self, other):
        if not isinstance(other, Fraction):
            return NotImplemented
        return self.num * other.den < other.num * self.den

    def __hash__(self):
        return hash((self.num, self.den))

    def __repr__(self):
        return f"Fraction({self.num}, {self.den})"


half = Fraction(1, 2)
assert Fraction(2, 4) == half
assert Fraction(1, -2) == Fraction(-1, 2)
assert half + Fraction(1, 3) == Fraction(5, 6)
assert half * 4 == Fraction(2) and 4 * half == Fraction(2)
assert sum([half, half, half]) == Fraction(3, 2)
assert Fraction(1, 3) < half <= half and half > Fraction(-1, 2)
assert sorted([half, Fraction(1, 3), Fraction(0)]) == [Fraction(0), Fraction(1, 3), half]
assert repr(Fraction(6, 8)) == "Fraction(3, 4)"
try:
    Fraction(1, 0)
except ZeroDivisionError:
    pass
else:
    raise AssertionError("zero denominator accepted")


# Q7
class Inventory:
    def __init__(self):
        self._stock = {}

    def __getitem__(self, name):
        return self._stock.get(name, 0)

    def __setitem__(self, name, qty):
        if qty < 0:
            raise ValueError("quantity must not be negative")
        self._stock[name] = qty

    def __delitem__(self, name):
        del self._stock[name]

    def __len__(self):
        return len(self._stock)

    def __contains__(self, name):
        return name in self._stock

    def __iter__(self):
        return iter(sorted(self._stock))


inv = Inventory()
inv["pear"] = 2
inv["apple"] = 5
assert inv["apple"] == 5 and inv["kiwi"] == 0
assert list(inv) == ["apple", "pear"]
assert len(inv) == 2 and "pear" in inv and "kiwi" not in inv
del inv["pear"]
assert list(inv) == ["apple"]


# Q8
class Polynomial:
    def __init__(self, coeffs):
        coeffs = list(coeffs)
        while coeffs and coeffs[-1] == 0:          # drop trailing zeros
            coeffs.pop()
        self.coeffs = coeffs

    def __call__(self, x):
        result = 0
        for c in reversed(self.coeffs):            # Horner's method
            result = result * x + c
        return result

    def __add__(self, other):
        if not isinstance(other, Polynomial):
            return NotImplemented
        n = max(len(self.coeffs), len(other.coeffs))
        a = self.coeffs + [0] * (n - len(self.coeffs))
        b = other.coeffs + [0] * (n - len(other.coeffs))
        return Polynomial([x + y for x, y in zip(a, b)])

    def __eq__(self, other):
        if not isinstance(other, Polynomial):
            return NotImplemented
        return self.coeffs == other.coeffs

    __hash__ = None                                # mutable list inside: keep it unhashable on purpose

    def __repr__(self):
        return f"Polynomial({self.coeffs})"


p = Polynomial([1, 0, 2])                           # 1 + 2x^2
assert p(3) == 19 and p(0) == 1
assert p + Polynomial([0, 1]) == Polynomial([1, 1, 2])
assert Polynomial([1, 2, 0, 0]) == Polynomial([1, 2])
assert p + Polynomial([-1, 0, -2]) == Polynomial([])
assert Polynomial([])(5) == 0
assert repr(p) == "Polynomial([1, 0, 2])"


# Q9 and Q10 fixed
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


assert len({Point(1, 2), Point(1, 2)}) == 1
assert (Point(1, 2) == "text") is False

print("Practice 2.6: all checks passed")

Practice 2.6: all checks passed


---

## 2.7 Dataclasses: classes that mainly hold data

Many classes are mostly **data containers**: an `__init__` that copies arguments to attributes, a
`__repr__`, and an `__eq__`. Writing these by hand is boring and easy to get wrong.

The `@dataclass` decorator (module `dataclasses`) **generates** them from **type annotations**
(`name: str`). A type annotation is a hint about the type; Python does not enforce it.

In [85]:
# The running example as a dataclass: __init__, __repr__ and __eq__ are generated
from dataclasses import dataclass


@dataclass
class Employee:
    name: str
    dept: str
    salary: int = 50000              # a field with a default value

    def monthly_salary(self):        # normal methods still work
        return round(self.salary / 12, 2)


asha = Employee("Asha", "Engineering", 85000)
print(asha)                                             # Employee(name='Asha', dept='Engineering', salary=85000)
print(asha == Employee("Asha", "Engineering", 85000))   # True  (generated __eq__ compares fields)
print(Employee("Meera", "HR"))                          # Employee(name='Meera', dept='HR', salary=50000)
print(asha.monthly_salary())                            # 7083.33

Employee(name='Asha', dept='Engineering', salary=85000)
True
Employee(name='Meera', dept='HR', salary=50000)
7083.33


The decorator wrote roughly this code for you:

```python
def __init__(self, name: str, dept: str, salary: int = 50000):
    self.name = name
    self.dept = dept
    self.salary = salary

def __repr__(self):
    return f"Employee(name={self.name!r}, dept={self.dept!r}, salary={self.salary!r})"

def __eq__(self, other):
    if other.__class__ is self.__class__:
        return (self.name, self.dept, self.salary) == (other.name, other.dept, other.salary)
    return NotImplemented
```

### Mutable defaults need `field(default_factory=...)`

A default like `skills: list = []` would be shared by all objects (the same trap as in Part 1, 1.5).
Dataclasses refuse it with a `ValueError`. Use `field(default_factory=list)`: it calls `list()` to make
a **new** list for every object.

In [86]:
# default_factory creates a fresh list for each object
from dataclasses import dataclass, field

try:
    @dataclass
    class Broken:
        skills: list = []                      # not allowed
except ValueError as e:
    print("ValueError:", e)
# ValueError: mutable default <class 'list'> for field skills is not allowed: use default_factory


@dataclass
class Employee:
    name: str
    skills: list = field(default_factory=list)


a = Employee("Asha")
b = Employee("Ravi")
a.skills.append("Python")
print(a.skills, b.skills)                      # ['Python'] []

ValueError: mutable default <class 'list'> for field skills is not allowed: use default_factory
['Python'] []


### `__post_init__`: validation and derived fields

The generated `__init__` only stores the values. To validate them or compute extra fields, define
`__post_init__`. It runs right after the generated `__init__`.

`field(init=False)` creates a field that is not an `__init__` parameter (you set it in `__post_init__`).
`field(repr=False)` hides a field from the repr; `field(compare=False)` leaves it out of `==`.

In [87]:
# __post_init__ validates and fills a derived field
from dataclasses import dataclass, field


@dataclass
class Employee:
    name: str
    dept: str
    salary: int
    email: str = field(init=False)                       # computed, not passed in
    notes: str = field(default="", repr=False, compare=False)

    def __post_init__(self):
        if self.salary <= 0:
            raise ValueError(f"salary must be positive, got {self.salary}")
        self.email = self.name.lower() + "@acme.example"


ravi = Employee("Ravi", "Sales", 60000, notes="joined in March")
print(ravi)            # Employee(name='Ravi', dept='Sales', salary=60000, email='ravi@acme.example')
print(ravi == Employee("Ravi", "Sales", 60000))          # True  (notes is not compared)

try:
    Employee("Kiran", "HR", -1)
except ValueError as e:
    print("ValueError:", e)                              # ValueError: salary must be positive, got -1

Employee(name='Ravi', dept='Sales', salary=60000, email='ravi@acme.example')
True
ValueError: salary must be positive, got -1


### Useful options

| Option | Effect |
|---|---|
| `@dataclass(order=True)` | generates `<`, `<=`, `>`, `>=` comparing fields as a tuple, in field order |
| `@dataclass(frozen=True)` | objects are read-only; assigning raises `FrozenInstanceError`; they become hashable |
| `@dataclass(slots=True)` | uses `__slots__` (section 2.8): less memory, no new attributes (Python 3.10+) |
| `@dataclass(kw_only=True)` | all fields must be passed as keywords (Python 3.10+) |
| `eq=False` | keep identity comparison |

In [88]:
# order=True: compare and sort by fields in order (here: priority first, then title)
from dataclasses import dataclass, field


@dataclass(order=True)
class Task:
    priority: int
    title: str = field(compare=False)        # not used for ordering or ==


tasks = [Task(3, "write report"), Task(1, "fix outage"), Task(2, "review code")]
print([t.title for t in sorted(tasks)])      # ['fix outage', 'review code', 'write report']
print(Task(1, "a") == Task(1, "b"))          # True  (title is ignored)

['fix outage', 'review code', 'write report']
True


In [89]:
# frozen=True: read-only and hashable, good for keys and value objects
from dataclasses import dataclass, FrozenInstanceError, replace


@dataclass(frozen=True)
class Point:
    x: int
    y: int


p = Point(1, 2)
try:
    p.x = 10
except FrozenInstanceError as e:
    print("FrozenInstanceError:", e)         # FrozenInstanceError: cannot assign to field 'x'

visited = {p, Point(1, 2), Point(3, 4)}      # hashable, so it can go in a set
print(len(visited))                          # 2

moved = replace(p, x=10)                     # "change" a frozen object by making a new one
print(p, moved)                              # Point(x=1, y=2) Point(x=10, y=2)

FrozenInstanceError: cannot assign to field 'x'
2
Point(x=1, y=2) Point(x=10, y=2)


In [90]:
# kw_only=True and slots=True (Python 3.10+)
from dataclasses import dataclass


@dataclass(kw_only=True, slots=True)
class Config:
    host: str
    port: int = 8080
    debug: bool = False


cfg = Config(host="localhost", debug=True)
print(cfg)                                   # Config(host='localhost', port=8080, debug=True)

try:
    Config("localhost")                      # positional argument not allowed
except TypeError as e:
    print("TypeError:", e)                   # TypeError: Config.__init__() takes 1 positional argument but 2 were given

Config(host='localhost', port=8080, debug=True)
TypeError: Config.__init__() takes 1 positional argument but 2 were given


### Helpers: `asdict`, `astuple`, `fields`, `replace`

In [91]:
# Converting dataclasses to dicts and tuples
from dataclasses import dataclass, asdict, astuple, fields
import json


@dataclass
class Employee:
    name: str
    dept: str
    salary: int


e = Employee("Meera", "HR", 52000)
print(asdict(e))                             # {'name': 'Meera', 'dept': 'HR', 'salary': 52000}
print(astuple(e))                            # ('Meera', 'HR', 52000)
print([f.name for f in fields(e)])           # ['name', 'dept', 'salary']
print(json.dumps(asdict(e)))                 # {"name": "Meera", "dept": "HR", "salary": 52000}

{'name': 'Meera', 'dept': 'HR', 'salary': 52000}
('Meera', 'HR', 52000)
['name', 'dept', 'salary']
{"name": "Meera", "dept": "HR", "salary": 52000}


### Inheritance with dataclasses, and a common error

A dataclass can inherit from another dataclass; the fields are combined, parent first.
Fields **without** a default cannot come after fields **with** a default (the generated `__init__` would be
invalid). `kw_only=True` is the usual fix.

In [92]:
# Dataclass inheritance; field order rule
from dataclasses import dataclass


@dataclass
class Employee:
    name: str
    salary: int = 50000


try:
    @dataclass
    class Manager(Employee):
        bonus: int                           # no default, but comes after salary (which has one)
except TypeError as e:
    print("TypeError:", e)
# TypeError: non-default argument 'bonus' follows default argument 'salary'  (wording varies by version)


@dataclass(kw_only=True)
class Manager(Employee):
    bonus: int                               # keyword-only fields may come in any order


m = Manager("Asha", 120000, bonus=15000)
print(m)                                     # Manager(name='Asha', salary=120000, bonus=15000)

TypeError: non-default argument 'bonus' follows default argument 'salary'
Manager(name='Asha', salary=120000, bonus=15000)


> **Tip:** use a dataclass when the class is mainly data. Use a normal class when the class is mainly
> behaviour, or when you need full control of `__init__`. `typing.NamedTuple` is an immutable, tuple-based
> alternative (see Part 7, `namedtuple`).

> **Key points**
> * `@dataclass` generates `__init__`, `__repr__` and `__eq__` from annotated fields.
> * Use `field(default_factory=list)` for mutable defaults; `__post_init__` for validation.
> * `order=True` adds comparisons; `frozen=True` makes objects read-only and hashable.
> * `slots=True` and `kw_only=True` are available from Python 3.10.
> * `asdict`, `astuple`, `replace` and `fields` help convert and copy dataclasses.

---

## 2.8 `__slots__`: fixed attributes and less memory

Normally each object has a `__dict__` to store its attributes. A dict is flexible but uses memory.
If a class sets **`__slots__`** to a tuple of names:

* objects get no `__dict__`; attributes are stored in fixed slots (less memory, slightly faster access);
* you **cannot add** attributes that are not listed (typos raise `AttributeError` instead of silently creating a new attribute).

Use it for classes with **many** small objects (millions of points, nodes, records).

In [93]:
# __slots__: no __dict__, and unknown attributes are rejected
class PointSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


p = PointSlots(1, 2)
print(hasattr(p, "__dict__"))            # False

try:
    p.z = 3                              # not listed in __slots__
except AttributeError as e:
    print("AttributeError:", e)
# AttributeError: 'PointSlots' object has no attribute 'z' and no __dict__ for setting new attributes
# (older Python versions show only the first part of this message)

False
AttributeError: 'PointSlots' object has no attribute 'z' and no __dict__ for setting new attributes


In [94]:
# Memory comparison: the object itself plus its __dict__ (exact numbers vary by Python version)
import sys


class PointDict:
    def __init__(self, x, y):
        self.x = x
        self.y = y


d_obj = PointDict(1, 2)
s_obj = PointSlots(1, 2)
size_with_dict = sys.getsizeof(d_obj) + sys.getsizeof(d_obj.__dict__)
size_with_slots = sys.getsizeof(s_obj)
print("slots object is smaller:", size_with_slots < size_with_dict)   # slots object is smaller: True

slots object is smaller: True


**Things to know about `__slots__`:**

* A subclass without its own `__slots__` gets a `__dict__` again.
* Class attributes with the same name as a slot are not allowed (so no class-level defaults for slots).
* `@dataclass(slots=True)` creates the slots for you (Python 3.10+).
* Do not add `__slots__` everywhere "for speed". Use it when you have measured a memory problem, or when
  a fixed set of attributes is part of the design.

> **Key points**
> * `__slots__ = ("a", "b")` removes the per-object `__dict__` and fixes the allowed attributes.
> * It saves memory for many small objects and turns attribute typos into errors.
> * Combine with dataclasses via `@dataclass(slots=True)`.

### Practice 2.8: Dataclasses and `__slots__`

**A. Predict the output**

1. What does this print?
   ```python
   from dataclasses import dataclass

   @dataclass
   class Item:
       name: str
       qty: int = 1

   print(Item("pen"), Item("pen") == Item("pen", 1))
   ```
2. What does this print?
   ```python
   from dataclasses import dataclass

   @dataclass(order=True)
   class Score:
       points: int
       player: str

   print(max([Score(5, "Ravi"), Score(9, "Asha"), Score(9, "Meera")]))
   ```
3. What happens here?
   ```python
   from dataclasses import dataclass

   @dataclass(frozen=True)
   class Color:
       r: int
       g: int
       b: int

   c = Color(1, 2, 3)
   c.r = 0
   ```
4. What does this print?
   ```python
   class A:
       __slots__ = ("x",)

   class B(A):
       pass

   b = B()
   b.y = 1
   print(b.y)
   ```

**B. Write code**

5. Write a dataclass `Product(name, price, tags)` where `tags` defaults to a new empty list, and
   `__post_init__` rejects a negative price and strips spaces from `name`.
6. Write a frozen dataclass `Coordinate(lat, lon)` that validates ranges (lat -90..90, lon -180..180) in
   `__post_init__`. Use it as a dict key to count visits to places.
7. Write a dataclass `Student(name, grade)` with `order` based on grade only (the name must not be compared).
   Sort students from highest to lowest grade.
8. **[Hard]** Write a dataclass `Order(order_id, items)` where `items` is a list of `(name, price, qty)`
   tuples (default empty), with a read-only property `total`, and a method `to_json()` that returns JSON
   including the total (use `asdict`).

**C. Find and fix the bug**

9. This fails when the class is defined. Why? Fix it in two different ways.
   ```python
   from dataclasses import dataclass

   @dataclass
   class Book:
       title: str
       year: int = 2000
       author: str
   ```

**D. Interview questions**

10. What does `@dataclass` generate? When would you still write a normal class?
11. Why is a frozen dataclass hashable but a normal dataclass is not?
12. **[Hard]** What are the benefits and the limits of `__slots__`?

### Solutions 2.8

1. `Item(name='pen', qty=1) True`.
2. `Score(points=9, player='Meera')`. Order compares `(points, player)` as tuples; with equal points,
   `'Meera' > 'Asha'`.
3. `dataclasses.FrozenInstanceError: cannot assign to field 'r'`.
4. `1`. `B` does not define `__slots__`, so its objects get a `__dict__` again.
9. A field without a default (`author`) follows a field with a default (`year`):
   `TypeError: non-default argument 'author' follows default argument 'year'`.
   Fix 1: move `author` before `year`. Fix 2: `@dataclass(kw_only=True)`.
10. `__init__`, `__repr__` and `__eq__` (plus ordering with `order=True`, `__hash__` with `frozen=True`,
    and `__slots__` with `slots=True`). Write a normal class when it is mostly behaviour, when `__init__`
    needs to work very differently from the fields, or when you need full control.
11. By default a dataclass generates `__eq__`, which sets `__hash__` to `None` (mutable objects should
    not be hashed by value). With `frozen=True` the fields cannot change, so a hash from the fields is
    safe, and the dataclass generates `__hash__`.
12. Benefits: less memory per object, slightly faster attribute access, typos become errors.
    Limits: no new attributes, no `__dict__` (some tools expect it), subclasses must also define
    `__slots__` to keep the benefit, and no class-level default values with the same names.

In [95]:
# Solutions 2.8 (B and C) and checks for A
from dataclasses import dataclass, field, asdict, FrozenInstanceError
import json


# A2 and A3 checks
@dataclass(order=True)
class Score:
    points: int
    player: str


assert max([Score(5, "Ravi"), Score(9, "Asha"), Score(9, "Meera")]) == Score(9, "Meera")


@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int


try:
    Color(1, 2, 3).r = 0
except FrozenInstanceError:
    pass
else:
    raise AssertionError("frozen dataclass was changed")


# Q5
@dataclass
class Product:
    name: str
    price: float
    tags: list = field(default_factory=list)

    def __post_init__(self):
        if self.price < 0:
            raise ValueError("price must not be negative")
        self.name = self.name.strip()


p1, p2 = Product("  pen ", 2.5), Product("book", 10)
p1.tags.append("office")
assert p1.name == "pen" and p2.tags == []
try:
    Product("x", -1)
except ValueError:
    pass
else:
    raise AssertionError("negative price accepted")


# Q6
@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

    def __post_init__(self):
        if not -90 <= self.lat <= 90 or not -180 <= self.lon <= 180:
            raise ValueError(f"invalid coordinate: {self.lat}, {self.lon}")


visits = {}
for place in [Coordinate(28.6, 77.2), Coordinate(19.1, 72.9), Coordinate(28.6, 77.2)]:
    visits[place] = visits.get(place, 0) + 1
assert visits[Coordinate(28.6, 77.2)] == 2 and len(visits) == 2
try:
    Coordinate(91, 0)
except ValueError:
    pass
else:
    raise AssertionError("invalid latitude accepted")


# Q7
@dataclass(order=True)
class Student:
    grade: int
    name: str = field(compare=False)


students = [Student(70, "Asha"), Student(95, "Ravi"), Student(85, "Meera")]
ranked = sorted(students, reverse=True)
assert [s.name for s in ranked] == ["Ravi", "Meera", "Asha"]
assert Student(80, "A") == Student(80, "B")        # names are not compared


# Q8
@dataclass
class Order:
    order_id: str
    items: list = field(default_factory=list)       # (name, price, qty) tuples

    @property
    def total(self):
        return round(sum(price * qty for _, price, qty in self.items), 2)

    def to_json(self):
        data = asdict(self)
        data["total"] = self.total
        return json.dumps(data)


order = Order("A-1", [("pen", 2.5, 4), ("book", 12.25, 1)])
assert order.total == 22.25
assert json.loads(order.to_json()) == {
    "order_id": "A-1",
    "items": [["pen", 2.5, 4], ["book", 12.25, 1]],   # JSON turns tuples into lists
    "total": 22.25,
}
assert Order("A-2").total == 0


# Q9 fixed (two ways)
@dataclass
class Book:
    title: str
    author: str
    year: int = 2000


@dataclass(kw_only=True)
class BookKw:
    title: str
    year: int = 2000
    author: str


assert Book("Dune", "Herbert").year == 2000
assert BookKw(title="Dune", author="Herbert").year == 2000

print("Practice 2.8: all checks passed")

Practice 2.8: all checks passed


---

## 2.9 Enums: named constants

An **enumeration** (`Enum`) is a fixed set of named values, such as departments, order states or
weekdays. Using plain strings (`"enginering"`) allows typos that nobody notices. With an enum, a typo is
an immediate `AttributeError`, editors can autocomplete, and all allowed values are listed in one place.

In [96]:
# The running example: departments as an Enum
from enum import Enum


class Department(Enum):
    ENGINEERING = "eng"
    SALES = "sales"
    HR = "hr"


dept = Department.SALES
print(dept)                          # Department.SALES
print(dept.name, dept.value)         # SALES sales
print(Department("hr"))              # Department.HR        (look up by value)
print(Department["ENGINEERING"])     # Department.ENGINEERING  (look up by name)
print([d.name for d in Department])  # ['ENGINEERING', 'SALES', 'HR']  (definition order)
print(dept is Department.SALES)      # True  (each member is a single object)

Department.SALES
SALES sales
Department.HR
Department.ENGINEERING
['ENGINEERING', 'SALES', 'HR']
True


In [97]:
# Errors with enums are clear and early
for attempt in (lambda: Department("finance"), lambda: Department["FINANCE"], lambda: Department.FINANSE):
    try:
        attempt()
    except (ValueError, KeyError, AttributeError) as e:
        print(f"{type(e).__name__}: {e}")
# ValueError: 'finance' is not a valid Department
# KeyError: 'FINANCE'
# AttributeError: type object 'Department' has no attribute 'FINANSE'

ValueError: 'finance' is not a valid Department
KeyError: 'FINANCE'
AttributeError: type object 'Department' has no attribute 'FINANSE'


### `auto()`, methods and `match`

* `auto()` picks the values for you (1, 2, 3, ... for `Enum`) when the value itself does not matter.
* Enums are classes, so they can have methods and properties.
* Compare members with `is` or `==`. They work well with `match` (Notebook 1, Part 5).

In [98]:
# auto(), a method on the enum, and match/case
from enum import Enum, auto


class OrderStatus(Enum):
    NEW = auto()
    PAID = auto()
    SHIPPED = auto()
    CANCELLED = auto()

    @property
    def is_final(self):
        return self in (OrderStatus.SHIPPED, OrderStatus.CANCELLED)


def next_step(status):
    match status:
        case OrderStatus.NEW:
            return "wait for payment"
        case OrderStatus.PAID:
            return "pack and ship"
        case OrderStatus.SHIPPED | OrderStatus.CANCELLED:
            return "nothing to do"


print([s.value for s in OrderStatus])          # [1, 2, 3, 4]
print(OrderStatus.PAID.is_final)               # False
print(next_step(OrderStatus.PAID))             # pack and ship
print(next_step(OrderStatus.CANCELLED))        # nothing to do

[1, 2, 3, 4]
False
pack and ship
nothing to do


### `IntEnum`, `StrEnum` and `Flag`

| Type | Members behave like | Use for |
|---|---|---|
| `Enum` | only themselves (`Color.RED != 1`) | most cases: safest |
| `IntEnum` | `int` (`Priority.HIGH == 3`, can do arithmetic, sort with ints) | compatibility with code that expects numbers |
| `StrEnum` (Python 3.11+) | `str` (`Dept.HR == "hr"`); `auto()` gives the lowercase name | values sent to JSON, APIs, databases |
| `Flag` | bit flags combined with `\|` | sets of options such as permissions |

In [99]:
# IntEnum and StrEnum mix with int and str
from enum import Enum, IntEnum, StrEnum, auto


class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3


class Dept(StrEnum):
    ENGINEERING = auto()             # value becomes "engineering"
    HR = auto()


class Plain(Enum):
    ONE = 1


print(Priority.HIGH == 3, Priority.HIGH > Priority.LOW)   # True True
print(sorted([3, Priority.LOW, 2]))                       # [<Priority.LOW: 1>, 2, 3]
print(Dept.HR == "hr", Dept.ENGINEERING.upper())          # True ENGINEERING
print(f"dept={Dept.HR}")                                  # dept=hr
print(Plain.ONE == 1)                                     # False  (a plain Enum is not an int)

True True
[<Priority.LOW: 1>, 2, 3]
True ENGINEERING
dept=hr
False


In [100]:
# Flag: combine options with |, test with in
from enum import Flag, auto


class Permission(Flag):
    READ = auto()        # 1
    WRITE = auto()       # 2
    DELETE = auto()      # 4


editor = Permission.READ | Permission.WRITE
print(editor)                                  # Permission.READ|WRITE
print(Permission.WRITE in editor)              # True
print(Permission.DELETE in editor)             # False
print([p.name for p in editor])                # ['READ', 'WRITE']   (iterating a Flag needs Python 3.11+)
print(bool(Permission(0)))                     # False  (no permissions)

Permission.READ|WRITE
True
False
['READ', 'WRITE']
False


> **Tip:** add `@enum.unique` above an enum class to forbid two names with the same value
> (otherwise the second name silently becomes an alias of the first).

> **Key points**
> * An `Enum` is a fixed set of named members; each has `.name` and `.value`.
> * Look up by value with `Cls(value)` and by name with `Cls["NAME"]`; bad input fails early.
> * `auto()` chooses values; enums can have methods and work with `match`.
> * `IntEnum`/`StrEnum` mix with `int`/`str`; `Flag` combines options with `|`.

### Practice 2.9: Enums

**A. Predict the output**

1. What does this print?
   ```python
   from enum import Enum

   class Size(Enum):
       S = 1
       M = 2
       L = 3

   print(Size(2).name, Size.L.value, len(Size))
   ```
2. What does this print?
   ```python
   from enum import Enum, IntEnum

   class A(Enum):
       X = 1

   class B(IntEnum):
       X = 1

   print(A.X == 1, B.X == 1)
   ```
3. What does this print?
   ```python
   from enum import Enum

   class Color(Enum):
       RED = 1
       CRIMSON = 1
       BLUE = 2

   print(Color.CRIMSON, [c.name for c in Color])
   ```

**B. Write code**

4. Write an enum `Weekday` (MONDAY..SUNDAY with values 1..7) with a property `is_weekend` and a class method
   `from_name(text)` that accepts any case (`"monday"`, `"Monday"`).
5. Write an enum `TrafficLight` (RED, GREEN, YELLOW) with a method `next()` that returns the next light in
   the cycle RED -> GREEN -> YELLOW -> RED.
6. **[Hard]** Write an `OrderStatus` enum and an `Order` class whose method `move_to(new_status)` allows only
   these transitions: NEW -> PAID or CANCELLED, PAID -> SHIPPED or CANCELLED. Any other transition raises
   `ValueError`. Keep the allowed transitions in a dict.

**C. Find and fix the bug**

7. The check below is always `False`. Why? Fix it.
   ```python
   from enum import Enum

   class Role(Enum):
       ADMIN = "admin"
       USER = "user"

   def is_admin(role):
       return role == "admin"

   print(is_admin(Role.ADMIN))
   ```

**D. Interview questions**

8. Why use an enum instead of string constants?
9. **[Hard]** When would you use `StrEnum` or `IntEnum` instead of a plain `Enum`, and what is the risk?

### Solutions 2.9

1. `M 3 3`.
2. `False True`. A plain `Enum` member is not equal to its value; an `IntEnum` member is an `int`.
3. `Color.RED ['RED', 'BLUE']`. `CRIMSON` has the same value as `RED`, so it becomes an **alias** of `RED`;
   aliases are not listed when iterating. `@unique` would turn this into an error.
7. A plain `Enum` member is never equal to a string. Compare with the member: `role is Role.ADMIN`
   (or use a `StrEnum` if you must compare with strings).
8. Typos fail immediately, all allowed values are in one place, editors can autocomplete, and members
   carry a name and a value and can have methods.
9. When values must interoperate with code or data that expects plain `int` or `str` (JSON, databases,
   old APIs). The risk: they compare equal to plain values, so mistakes such as comparing two unrelated
   enums that share the value `1` are no longer caught.

In [101]:
# Solutions 2.9 (B and C) and checks for A
from enum import Enum, IntEnum, auto


# A3 check
class Color(Enum):
    RED = 1
    CRIMSON = 1
    BLUE = 2


assert Color.CRIMSON is Color.RED and [c.name for c in Color] == ["RED", "BLUE"]


# Q4
class Weekday(Enum):
    MONDAY = 1
    TUESDAY = 2
    WEDNESDAY = 3
    THURSDAY = 4
    FRIDAY = 5
    SATURDAY = 6
    SUNDAY = 7

    @property
    def is_weekend(self):
        return self.value >= 6

    @classmethod
    def from_name(cls, text):
        return cls[text.strip().upper()]


assert Weekday.from_name("monday") is Weekday.MONDAY
assert Weekday.from_name(" Sunday ").is_weekend
assert not Weekday.FRIDAY.is_weekend
try:
    Weekday.from_name("funday")
except KeyError:
    pass
else:
    raise AssertionError("unknown day accepted")


# Q5
class TrafficLight(Enum):
    RED = auto()
    GREEN = auto()
    YELLOW = auto()

    def next(self):
        members = list(type(self))
        return members[(members.index(self) + 1) % len(members)]


assert TrafficLight.RED.next() is TrafficLight.GREEN
assert TrafficLight.YELLOW.next() is TrafficLight.RED


# Q6
class OrderStatus(Enum):
    NEW = auto()
    PAID = auto()
    SHIPPED = auto()
    CANCELLED = auto()


ALLOWED = {
    OrderStatus.NEW: {OrderStatus.PAID, OrderStatus.CANCELLED},
    OrderStatus.PAID: {OrderStatus.SHIPPED, OrderStatus.CANCELLED},
}


class Order:
    def __init__(self):
        self.status = OrderStatus.NEW

    def move_to(self, new_status):
        if new_status not in ALLOWED.get(self.status, set()):
            raise ValueError(f"cannot go from {self.status.name} to {new_status.name}")
        self.status = new_status


o = Order()
o.move_to(OrderStatus.PAID)
o.move_to(OrderStatus.SHIPPED)
assert o.status is OrderStatus.SHIPPED
try:
    o.move_to(OrderStatus.CANCELLED)                # SHIPPED is final
except ValueError as e:
    assert str(e) == "cannot go from SHIPPED to CANCELLED"
else:
    raise AssertionError("invalid transition accepted")
try:
    Order().move_to(OrderStatus.SHIPPED)            # cannot skip PAID
except ValueError:
    pass
else:
    raise AssertionError("invalid transition accepted")


# Q7 fixed
class Role(Enum):
    ADMIN = "admin"
    USER = "user"


def is_admin(role):
    return role is Role.ADMIN


assert is_admin(Role.ADMIN) and not is_admin(Role.USER)

print("Practice 2.9: all checks passed")

Practice 2.9: all checks passed


---

## 2.10 OOP design principles in simple words

Design principles are rules of thumb that keep code easy to change. You will hear these names in interviews.

### SOLID

| Letter | Principle | In simple words |
|---|---|---|
| S | Single responsibility | A class should have one job (one reason to change). |
| O | Open/closed | Add new behaviour by adding new classes, not by editing working code. |
| L | Liskov substitution | A subclass must work everywhere its parent works, without surprises. |
| I | Interface segregation | Many small interfaces are better than one big one; do not force classes to implement methods they do not need. |
| D | Dependency inversion | Depend on abstractions (what something does), not on concrete classes; pass dependencies in. |

### Other short rules

* **DRY** (Don't Repeat Yourself): one piece of knowledge lives in one place. Copy-pasted logic will drift apart.
* **KISS** (Keep It Simple): choose the simplest design that works. A function is fine if a class is not needed.
* **YAGNI** (You Aren't Gonna Need It): do not build features "for the future". Add them when needed.
* **Composition over inheritance** (section 2.4).

### S: single responsibility

A class that computes a payroll *and* formats it *and* saves it has three reasons to change.
Split it: each class does one job, and they work together.

In [102]:
# Single responsibility: calculation, formatting and storage are separate classes
class PayrollCalculator:
    def total(self, salaries):
        return sum(salaries.values())


class PayrollFormatter:
    def to_text(self, salaries):
        lines = [f"{name:<6}{amount:>8}" for name, amount in sorted(salaries.items())]
        return "\n".join(lines)


class InMemoryStore:
    def __init__(self):
        self.saved = []

    def save(self, text):
        self.saved.append(text)


salaries = {"Ravi": 60000, "Asha": 85000}
store = InMemoryStore()
store.save(PayrollFormatter().to_text(salaries))
print(PayrollCalculator().total(salaries))      # 145000
print(store.saved[0])
# Asha     85000
# Ravi     60000

145000
Asha     85000
Ravi     60000


### O: open/closed, with polymorphism

New discount types should not require editing a long `if/elif` chain. Each discount is its own class with
the same method. Adding a new one means adding a class; the checkout code stays unchanged.

In [103]:
# Open/closed: new discounts are new classes; checkout() never changes
class NoDiscount:
    def apply(self, amount):
        return amount


class PercentOff:
    def __init__(self, percent):
        self.percent = percent

    def apply(self, amount):
        return round(amount * (100 - self.percent) / 100, 2)


class FlatOff:
    def __init__(self, value):
        self.value = value

    def apply(self, amount):
        return max(0, amount - self.value)


def checkout(amount, discount):
    return discount.apply(amount)


for d in (NoDiscount(), PercentOff(10), FlatOff(30)):
    print(type(d).__name__, checkout(200, d))
# NoDiscount 200
# PercentOff 180.0
# FlatOff 170

NoDiscount 200
PercentOff 180.0
FlatOff 170


### L: Liskov substitution, the classic square and rectangle problem

Mathematically a square *is a* rectangle. But if `Rectangle` lets you set width and height
independently, a `Square` subclass must break that promise. Code written for rectangles then gives wrong
results with squares. "Is-a" in real life does not always mean "is-a" in code.

In [104]:
# Liskov violation: Square changes the behaviour that Rectangle promised
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def set_width(self, width):
        self.width = width

    def area(self):
        return self.width * self.height


class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)

    def set_width(self, width):            # to stay a square it must also change the height
        self.width = width
        self.height = width


def stretch(rect):
    # written for rectangles: doubling the width should double the area
    before = rect.area()
    rect.set_width(rect.width * 2)
    return rect.area() == 2 * before


print(stretch(Rectangle(2, 3)))            # True
print(stretch(Square(2)))                  # False  (a Square cannot be used where a Rectangle is expected)

True
False


**Fix:** do not make `Square` a subclass of a *mutable* `Rectangle`. Make both implement a common `Shape`
interface (with `area()`), or make shapes immutable.

### I and D: small interfaces, and dependencies passed in

* **Interface segregation:** an ABC `MultiFunctionDevice` with `print`, `scan` and `fax` forces a simple
  printer to implement `fax`. Split it into `Printer`, `Scanner`, `Fax`; a class inherits only what it supports.
* **Dependency inversion:** a `ReportService` should receive "something with a `send(message)` method"
  (an email sender, an SMS sender, a fake for tests) instead of creating an `EmailSender` itself.
  You saw this in Practice 2.4 (dependency injection).

> **Interview tip:** you do not need to recite SOLID word for word. Show it in your design: small classes
> with one job, polymorphism instead of type checks, dependencies passed in through `__init__`.

> **Key points**
> * SOLID: one job per class; extend by adding classes; subclasses must keep the parent's promises;
>   small interfaces; depend on abstractions.
> * DRY, KISS and YAGNI keep designs small and easy to change.
> * A real-world "is-a" does not guarantee a correct subclass (square and rectangle).

---

## 2.11 Designing classes in an interview

"Object-oriented design" (OOD) questions ask you to model a small system: a parking lot, a library,
an elevator, a vending machine, an LRU cache. There is no single right answer. The interviewer watches
**how you think**. A reliable process:

```text
   +---------------------------------------+
   | 1. Clarify requirements and scope     |
   |    (who uses it? which operations?)   |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | 2. Find the nouns -> classes          |
   |    find the verbs -> methods          |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | 3. Relationships: is-a (inherit),     |
   |    has-a (compose); enums for states  |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | 4. Choose data structures for the     |
   |    key operations (state the Big-O)   |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | 5. Write the core code; handle errors |
   |    with clear custom exceptions       |
   +-------------------+-------------------+
                       |
                       v
   +---------------------------------------+
   | 6. Walk through an example; test edge |
   |    cases; discuss extensions          |
   +---------------------------------------+
```

### Design 1: LRU cache

**Problem:** design a cache with a fixed `capacity`. `get(key)` returns the value or `-1`.
`put(key, value)` inserts or updates. When the cache is full, it removes the **least recently used** item.
Both operations must be **O(1)**.

**Idea:** we need fast lookup by key (a dict) and a fast way to know the oldest-used key (an ordered
structure). `collections.OrderedDict` gives both: `move_to_end(key)` marks a key as recently used, and
`popitem(last=False)` removes the oldest. Both are O(1). (`OrderedDict` is covered in Part 7; Notebook 3,
Part 9 builds the same cache from a dict and a doubly linked list.)

```text
   least recently used                         most recently used
        |                                               |
        v                                               v
      [ "a" ] <-> [ "b" ] <-> [ "c" ]     get("a")  ->  [ "b" ] <-> [ "c" ] <-> [ "a" ]
                                          put("d") on a full cache (capacity 3)
                                                    ->  [ "c" ] <-> [ "a" ] <-> [ "d" ]   ("b" evicted)
```

In [105]:
# LRU cache with OrderedDict: get and put are O(1)
from collections import OrderedDict


class LRUCache:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._data = OrderedDict()

    def get(self, key):
        if key not in self._data:
            return -1
        self._data.move_to_end(key)                 # mark as most recently used
        return self._data[key]

    def put(self, key, value):
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)          # remove the least recently used

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"LRUCache({list(self._data.items())})"


cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
print(cache.get("a"))          # 1     ("a" is now the most recent)
cache.put("c", 3)              # full: evicts "b", the least recently used
print(cache.get("b"))          # -1
print(cache)                   # LRUCache([('a', 1), ('c', 3)])

1
-1
LRUCache([('a', 1), ('c', 3)])


### Design 2: bank account system

**Requirements (after clarifying):** accounts have an owner and a generated number; deposit, withdraw and
transfer; no overdraft for normal accounts; savings accounts pay interest; keep a transaction history;
clear errors.

**Classes:** `Account` (base), `SavingsAccount` (is-a account), `Transaction` (a frozen dataclass),
`TransactionType` (enum), `Bank` (has accounts), and custom exceptions.

In [106]:
# Bank account design: enum + frozen dataclass + inheritance + custom exceptions
import itertools
from dataclasses import dataclass
from enum import Enum


class BankError(Exception):
    """Base class for all bank errors, so callers can catch them together."""


class InsufficientFundsError(BankError):
    pass


class AccountNotFoundError(BankError):
    pass


class TransactionType(Enum):
    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"
    INTEREST = "interest"


@dataclass(frozen=True)
class Transaction:
    kind: TransactionType
    amount: float


class Account:
    _numbers = itertools.count(1)            # one shared counter for ALL account types (on purpose)

    def __init__(self, owner):
        self.owner = owner
        self.number = f"AC{next(self._numbers):04d}"   # next() advances the shared counter
        self._balance = 0.0
        self._history = []

    @property
    def balance(self):
        return self._balance

    @property
    def history(self):
        return tuple(self._history)          # read-only view for callers

    def _record(self, kind, amount):
        self._history.append(Transaction(kind, amount))

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount
        self._record(TransactionType.DEPOSIT, amount)

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise InsufficientFundsError(f"{self.number}: balance {self._balance} < {amount}")
        self._balance -= amount
        self._record(TransactionType.WITHDRAW, amount)

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, number={self.number!r}, balance={self._balance})"


class SavingsAccount(Account):
    def __init__(self, owner, rate):
        super().__init__(owner)
        self.rate = rate

    def add_interest(self):
        interest = round(self._balance * self.rate, 2)
        self._balance += interest
        self._record(TransactionType.INTEREST, interest)


class Bank:
    def __init__(self):
        self._accounts = {}                  # number -> Account: O(1) lookup

    def open(self, account):
        self._accounts[account.number] = account
        return account

    def get(self, number):
        try:
            return self._accounts[number]
        except KeyError:
            raise AccountNotFoundError(number) from None

    def transfer(self, from_number, to_number, amount):
        source, target = self.get(from_number), self.get(to_number)
        source.withdraw(amount)              # raises before anything changes if funds are low
        target.deposit(amount)

In [107]:
# Using the bank design (classes from the cell above)
bank = Bank()
asha = bank.open(Account("Asha"))
ravi = bank.open(SavingsAccount("Ravi", rate=0.05))

asha.deposit(500)
bank.transfer(asha.number, ravi.number, 200)
ravi.add_interest()

print(asha)                                  # Account('Asha', number='AC0001', balance=300.0)
print(ravi)                                  # SavingsAccount('Ravi', number='AC0002', balance=210.0)
print([t.kind.value for t in ravi.history])  # ['deposit', 'interest']

for action in (lambda: bank.transfer(asha.number, ravi.number, 1000),
               lambda: bank.get("AC9999")):
    try:
        action()
    except BankError as e:                   # one except catches every bank error
        print(f"{type(e).__name__}: {e}")
# InsufficientFundsError: AC0001: balance 300.0 < 1000
# AccountNotFoundError: AC9999

Account('Asha', number='AC0001', balance=300.0)
SavingsAccount('Ravi', number='AC0002', balance=210.0)
['deposit', 'interest']
InsufficientFundsError: AC0001: balance 300.0 < 1000
AccountNotFoundError: AC9999


### Design 3: parking lot

**Requirements:** a lot has spots of three sizes (small, medium, large). A motorbike fits any spot, a car
needs medium or large, a bus needs large. `park(vehicle)` puts the vehicle in the **smallest free spot that
fits** and returns the spot id; `leave(plate)` frees the spot; `free_spots()` reports free spots per size.

**Classes:** `Size` (an `IntEnum`, so sizes can be compared), `Vehicle` and its subclasses, `ParkingSpot`,
`ParkingLot`, `ParkingFullError`.

In [108]:
# Parking lot design
from enum import IntEnum


class Size(IntEnum):
    SMALL = 1
    MEDIUM = 2
    LARGE = 3


class ParkingFullError(Exception):
    pass


class Vehicle:
    size = Size.MEDIUM                           # subclasses override this class attribute

    def __init__(self, plate):
        self.plate = plate

    def __repr__(self):
        return f"{type(self).__name__}({self.plate!r})"


class Motorbike(Vehicle):
    size = Size.SMALL


class Car(Vehicle):
    size = Size.MEDIUM


class Bus(Vehicle):
    size = Size.LARGE


class ParkingSpot:
    def __init__(self, spot_id, size):
        self.spot_id = spot_id
        self.size = size
        self.vehicle = None

    @property
    def is_free(self):
        return self.vehicle is None

    def fits(self, vehicle):
        return self.is_free and vehicle.size <= self.size


class ParkingLot:
    def __init__(self, sizes):
        # spots sorted from small to large, so the first fit is the smallest fit
        self._spots = sorted((ParkingSpot(i, s) for i, s in enumerate(sizes)), key=lambda sp: sp.size)
        self._where = {}                         # plate -> spot, O(1) for leave()

    def park(self, vehicle):
        if vehicle.plate in self._where:
            raise ValueError(f"{vehicle.plate} is already parked")
        for spot in self._spots:                 # O(number of spots)
            if spot.fits(vehicle):
                spot.vehicle = vehicle
                self._where[vehicle.plate] = spot
                return spot.spot_id
        raise ParkingFullError(f"no free spot for {vehicle!r}")

    def leave(self, plate):
        spot = self._where.pop(plate)            # KeyError if the car is not here
        spot.vehicle = None
        return spot.spot_id

    def free_spots(self):
        counts = {size.name: 0 for size in Size}
        for spot in self._spots:
            if spot.is_free:
                counts[spot.size.name] += 1
        return counts

In [109]:
# Using the parking lot (classes from the cell above)
lot = ParkingLot([Size.LARGE, Size.SMALL, Size.MEDIUM, Size.MEDIUM])

print(lot.park(Motorbike("MB-1")))       # 1   (the small spot)
print(lot.park(Car("CAR-1")))            # 2   (first medium spot)
print(lot.park(Bus("BUS-1")))            # 0   (the only large spot)
print(lot.free_spots())                  # {'SMALL': 0, 'MEDIUM': 1, 'LARGE': 0}

try:
    lot.park(Bus("BUS-2"))
except ParkingFullError as e:
    print("ParkingFullError:", e)        # ParkingFullError: no free spot for Bus('BUS-2')

print(lot.leave("BUS-1"))                # 0
print(lot.park(Bus("BUS-2")))            # 0

1
2
0
{'SMALL': 0, 'MEDIUM': 1, 'LARGE': 0}
ParkingFullError: no free spot for Bus('BUS-2')
0
0


**Possible extensions to discuss** (the interviewer often asks "what if ..."):

* Several floors: a `Level` class that has spots; the lot has levels (composition).
* Faster search: keep one free-spot heap or list **per size**, so `park` is O(log n) or O(1).
* Fees: a `Ticket` dataclass with entry time and a pricing strategy object (open/closed principle).
* Concurrency: two cars must not get the same spot; protect `park` with a lock (Part 8).

> **Interview tip:** talk while you design. Say which classes you chose and why, which data structure
> makes each operation fast, and which edge cases you handle (full lot, same car twice, unknown plate).

> **Key points**
> * Process: clarify, nouns to classes, verbs to methods, relationships, data structures, code, test.
> * Use enums for fixed categories and states, dataclasses for records, custom exceptions for errors.
> * Choose data structures for the key operations and state their Big-O.
> * Prepare to extend your design; composition and small classes make that easy.

---

## 2.12 Debugging and OOP design practice

### How to debug inheritance and special methods

| Error message (example) | What it means | Typical fix |
|---|---|---|
| `AttributeError: 'Intern' object has no attribute 'name'` | The child's `__init__` did not call `super().__init__()`. | Call `super().__init__(...)` first. |
| `TypeError: Can't instantiate abstract class Shape without an implementation for abstract method 'area'` | A subclass forgot (or misspelt) an abstract method. | Implement the method with the exact name. |
| `TypeError: Cannot create a consistent method resolution order (MRO) for bases X, Y` | The order of base classes is impossible. | List subclasses before their parents. |
| `TypeError: object.__init__() takes exactly one argument (the instance to initialize)` | In a cooperative chain a keyword argument was not consumed, so it reached `object`. | Each class takes its own keyword and passes `**kwargs` on. |
| `TypeError: unhashable type: 'Point'` (3.14: "cannot use 'Point' as a set element ...") | `__eq__` defined without `__hash__`. | Add `__hash__` from the same fields, or use `frozen=True`. |
| `TypeError: unsupported operand type(s) for +: 'Money' and 'int'` | No method handled this type combination. | Support the type, add `__radd__`, or convert the value first. |
| `RecursionError` inside `__repr__` or `__eq__` | The method calls itself, e.g. `f"{self}"` inside `__repr__`. | Build the text from attributes, not from `self`. |

In [110]:
# Reproducing two typical errors: an unconsumed keyword, and a recursive __repr__
class Person:
    def __init__(self, *, name, **kwargs):
        super().__init__(**kwargs)
        self.name = name


class Worker(Person):
    def __init__(self, *, salary, **kwargs):
        super().__init__(**kwargs)
        self.salary = salary


try:
    Person(name="Asha", salary=5)            # Person does not take salary, so it reaches object
except TypeError as e:
    print("TypeError:", e)                   # TypeError: object.__init__() takes exactly one argument (the instance to initialize)

print(vars(Worker(name="Asha", salary=5)))   # {'name': 'Asha', 'salary': 5}


class Loop:
    def __repr__(self):
        return f"Loop({self})"               # f"{self}" calls str(), which falls back to __repr__ again


try:
    repr(Loop())
except RecursionError:
    print("RecursionError in __repr__")      # RecursionError in __repr__

TypeError: object.__init__() takes exactly one argument (the instance to initialize)
{'name': 'Asha', 'salary': 5}
RecursionError in __repr__


### Finding where a method really comes from

With inheritance, the hardest question is often "**which** version of this method runs?".
Walk the MRO and look in each class's own `__dict__`. The first class that has the name is the one Python uses.

In [111]:
# A small debugging helper: which class provides a method for this object?
def where_defined(obj, name):
    for cls in type(obj).__mro__:
        if name in vars(cls):
            return cls.__name__
    return None


class Employee:
    def annual_pay(self):
        return 1

    def describe(self):
        return "employee"


class Manager(Employee):
    def annual_pay(self):
        return 2


class Director(Manager):
    pass


d = Director()
print([c.__name__ for c in Director.__mro__])   # ['Director', 'Manager', 'Employee', 'object']
print(where_defined(d, "annual_pay"))           # Manager
print(where_defined(d, "describe"))             # Employee
print(where_defined(d, "__repr__"))             # object
print(Director.annual_pay.__qualname__)         # Manager.annual_pay  (another quick way)

['Director', 'Manager', 'Employee', 'object']
Manager
Employee
object
Manager.annual_pay


```text
   Wrong method runs, or an attribute is missing?
                     |
                     v
   +-----------------------------------+
   | print(type(obj).__mro__)          |  <- in which order are classes searched?
   +-----------------+-----------------+
                     |
                     v
   +-----------------------------------+
   | which class defines the method?   |  <- where_defined(), method.__qualname__
   +-----------------+-----------------+
                     |
                     v
   +-----------------------------------+
   | print(vars(obj))                  |  <- did every __init__ in the chain run?
   +-----------------+-----------------+
                     |
                     v
   +-----------------------------------+
   | fix (super() call, method name,   |
   | base order), re-run, add a test   |
   +-----------------------------------+
```

**Checklist for dunder methods:**

* Binary operators return `NotImplemented` for unknown types (never `False`, never raise).
* `__eq__` and `__hash__` use the same fields; hashed fields never change.
* `__iadd__` returns `self`.
* `__repr__` and `__str__` return strings built from attributes.
* `__len__` returns a non-negative `int`; `__bool__` returns a `bool`.
* Test each dunder directly with `assert` (for example `assert a + b == expected`).

> **Key points**
> * Read the error type first: most inheritance bugs are a missing `super().__init__()`, a misspelt
>   abstract method, or a wrong base-class order.
> * Use `__mro__`, `vars(cls)` and `method.__qualname__` to see which code really runs.
> * Most dunder bugs come from not returning `NotImplemented`, `self` or a string.

### Practice 2.12: OOP design interview practice

**A. Predict the output**

1. What does this print?
   ```python
   class Base:
       def __init__(self):
           self.items = []

       def add(self, x):
           self.items.append(x)
           return self

   class Loud(Base):
       def add(self, x):
           return super().add(str(x).upper())

   print(Loud().add("a").add("b").items)
   ```
2. What does this print?
   ```python
   from collections import OrderedDict

   d = OrderedDict(a=1, b=2, c=3)
   d.move_to_end("a")
   d.popitem(last=False)
   print(list(d))
   ```

**B. Write code (design questions)**

3. **Min stack.** Design a `MinStack` with `push(x)`, `pop()`, `top()` and `get_min()`, all **O(1)**.
   Raise `IndexError` on an empty stack.
4. **Library system.** Design `Book` (dataclass: isbn, title), `Member` (name, borrowed books, limit of 2
   books) and `Library` with `add_book`, `borrow(member, isbn)` and `return_book(member, isbn)`.
   Use custom exceptions `BookNotAvailableError` and `LimitReachedError` (both inherit from `LibraryError`).
5. **Deck of cards.** Use an `Enum` for suits and a frozen dataclass `Card(rank, suit)` with ranks 2..14.
   `Deck` builds 52 cards, has `__len__`, `shuffle(seed)` (use `random.Random(seed)` so it is
   deterministic) and `deal(n)` that removes and returns `n` cards (raise `ValueError` if not enough).
   Find the highest card of a hand with `max(..., key=...)`.
6. **[Hard] Rate limiter.** Design a callable class `RateLimiter(limit, window_seconds, clock)`.
   Calling `limiter(user)` returns `True` if the user made fewer than `limit` calls in the last
   `window_seconds`, else `False`. `clock` is a function returning the current time (inject a fake clock in
   tests). Use a `deque` of timestamps per user.
7. **[Hard] Versioned key-value store.** Design `TimeMap` with `set(key, value, timestamp)` and
   `get(key, timestamp)` that returns the value with the largest timestamp `<=` the given one (or `""`).
   Timestamps for each key arrive in increasing order. `get` must be O(log n) (use `bisect`).
8. **[Hard] Vending machine.** States (enum): `IDLE`, `HAS_MONEY`. `insert(amount)`, `select(item)` returns
   `(item, change)` if enough money and the item is in stock, `cancel()` returns the inserted money.
   Invalid actions raise `VendingError`.

**C. Find and fix the bug**

9. This LRU cache never evicts the right key. Find the two bugs.
   ```python
   from collections import OrderedDict

   class LRU:
       def __init__(self, capacity):
           self.capacity = capacity
           self.data = OrderedDict()

       def get(self, key):
           return self.data.get(key, -1)

       def put(self, key, value):
           self.data[key] = value
           if len(self.data) > self.capacity:
               self.data.popitem()
   ```

**D. Interview questions**

10. Explain the Liskov substitution principle with an example.
11. You are asked to design an elevator system. List the classes, enums and key methods you would start with.
12. **[Hard]** Your `ParkingLot.park` is O(number of spots). How would you make it faster?

### Solutions 2.12

1. `['A', 'B']`. `Loud.add` upper-cases the value and calls the parent, which returns `self`, so calls chain.
2. `['c', 'a']`. Moving `a` to the end gives `b, c, a`; `popitem(last=False)` removes the first, `b`.
9. (1) `get` does not mark the key as recently used: call `move_to_end(key)` when the key exists.
   (2) `popitem()` removes the **last** (most recent) item; use `popitem(last=False)`.
   Also, `put` on an existing key should `move_to_end(key)` (assigning to an existing key keeps its old position).
10. A subclass object must be usable anywhere a parent object is expected, without breaking the caller's
    expectations. Violation: `Square(Rectangle)` whose `set_width` also changes the height, so code that
    doubles the width expecting double the area breaks.
11. `Direction` enum (UP, DOWN, IDLE), `DoorState` enum, `Request` dataclass (floor, direction),
    `Elevator` (current floor, direction, a set or heaps of target floors; `step()`, `add_stop(floor)`),
    `Controller` / `Building` (has elevators; `request(floor, direction)` chooses the best elevator with a
    strategy object). Discuss scheduling (nearest car, SCAN algorithm) and edge cases.
12. Keep a separate collection of **free** spots per size (for example a list used as a stack, or a heap of
    spot ids). `park` checks the sizes that fit, from small to large (only 3 sizes), and pops a free spot:
    O(1) with stacks or O(log n) with heaps. `leave` pushes the spot back.

In [112]:
# Solutions 2.12 (B and C) and checks for A
import bisect
import random
from collections import OrderedDict, deque
from dataclasses import dataclass, field
from enum import Enum


# Q3: MinStack - a second stack stores the minimum at each depth
class MinStack:
    def __init__(self):
        self._items = []
        self._mins = []

    def push(self, x):
        self._items.append(x)
        self._mins.append(x if not self._mins else min(x, self._mins[-1]))

    def pop(self):
        if not self._items:
            raise IndexError("pop from empty stack")
        self._mins.pop()
        return self._items.pop()

    def top(self):
        if not self._items:
            raise IndexError("top from empty stack")
        return self._items[-1]

    def get_min(self):
        if not self._mins:
            raise IndexError("min from empty stack")
        return self._mins[-1]


ms = MinStack()
for x in (5, 3, 7, 3, 1):
    ms.push(x)
assert ms.get_min() == 1
ms.pop()
assert ms.get_min() == 3 and ms.top() == 3
ms.pop()
assert ms.get_min() == 3
ms.pop()
ms.pop()
assert ms.get_min() == 5
ms.pop()
try:
    ms.get_min()
except IndexError:
    pass
else:
    raise AssertionError("empty MinStack must raise")


# Q4: library system
class LibraryError(Exception):
    pass


class BookNotAvailableError(LibraryError):
    pass


class LimitReachedError(LibraryError):
    pass


@dataclass(frozen=True)
class Book:
    isbn: str
    title: str


@dataclass
class Member:
    name: str
    borrowed: set = field(default_factory=set)
    LIMIT = 2                                        # no annotation: a class constant, not a field


class Library:
    def __init__(self):
        self._books = {}                             # isbn -> Book
        self._available = set()                      # isbns on the shelf

    def add_book(self, book):
        self._books[book.isbn] = book
        self._available.add(book.isbn)

    def borrow(self, member, isbn):
        if isbn not in self._available:
            raise BookNotAvailableError(isbn)
        if len(member.borrowed) >= Member.LIMIT:
            raise LimitReachedError(member.name)
        self._available.remove(isbn)
        member.borrowed.add(isbn)

    def return_book(self, member, isbn):
        member.borrowed.remove(isbn)
        self._available.add(isbn)


lib = Library()
for isbn, title in [("1", "Dune"), ("2", "Emma"), ("3", "Ulysses")]:
    lib.add_book(Book(isbn, title))
asha, ravi = Member("Asha"), Member("Ravi")
lib.borrow(asha, "1")
lib.borrow(asha, "2")
for action, error in [(lambda: lib.borrow(ravi, "1"), BookNotAvailableError),
                      (lambda: lib.borrow(asha, "3"), LimitReachedError)]:
    try:
        action()
    except error:
        pass
    else:
        raise AssertionError(f"expected {error.__name__}")
lib.return_book(asha, "1")
lib.borrow(ravi, "1")
assert asha.borrowed == {"2"} and ravi.borrowed == {"1"}


# Q5: deck of cards
class Suit(Enum):
    CLUBS = "C"
    DIAMONDS = "D"
    HEARTS = "H"
    SPADES = "S"


@dataclass(frozen=True)
class Card:
    rank: int                                        # 2..14 (14 = ace)
    suit: Suit


class Deck:
    def __init__(self):
        self._cards = [Card(rank, suit) for suit in Suit for rank in range(2, 15)]

    def __len__(self):
        return len(self._cards)

    def shuffle(self, seed):
        random.Random(seed).shuffle(self._cards)     # own generator: deterministic, no global effect

    def deal(self, n):
        if n > len(self._cards):
            raise ValueError(f"only {len(self._cards)} cards left")
        hand, self._cards = self._cards[:n], self._cards[n:]
        return hand


deck = Deck()
assert len(deck) == 52 and len(set(deck._cards)) == 52
deck.shuffle(seed=7)
hand = deck.deal(5)
assert len(hand) == 5 and len(deck) == 47
other = Deck()
other.shuffle(seed=7)
assert other.deal(5) == hand                         # same seed -> same order
highest = max(hand, key=lambda card: card.rank)
assert all(highest.rank >= card.rank for card in hand)
try:
    deck.deal(48)
except ValueError:
    pass
else:
    raise AssertionError("dealing too many cards must fail")


# Q6: rate limiter with an injected clock
class RateLimiter:
    def __init__(self, limit, window_seconds, clock):
        self.limit = limit
        self.window = window_seconds
        self.clock = clock
        self._calls = {}                             # user -> deque of timestamps

    def __call__(self, user):
        now = self.clock()
        calls = self._calls.setdefault(user, deque())
        while calls and calls[0] <= now - self.window:   # drop timestamps outside the window
            calls.popleft()
        if len(calls) < self.limit:
            calls.append(now)
            return True
        return False


class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now


clock = FakeClock()
allow = RateLimiter(limit=2, window_seconds=10, clock=clock)
assert allow("asha") and allow("asha") and not allow("asha")
assert allow("ravi")                                 # users are independent
clock.now = 9.9
assert not allow("asha")
clock.now = 10.0                                     # the first two calls are now outside the window
assert allow("asha") and allow("asha") and not allow("asha")


# Q7: versioned key-value store
class TimeMap:
    def __init__(self):
        self._times = {}                             # key -> [timestamps]
        self._values = {}                            # key -> [values]

    def set(self, key, value, timestamp):
        self._times.setdefault(key, []).append(timestamp)
        self._values.setdefault(key, []).append(value)

    def get(self, key, timestamp):
        times = self._times.get(key, [])
        i = bisect.bisect_right(times, timestamp)    # number of timestamps <= timestamp
        return self._values[key][i - 1] if i else ""


tm = TimeMap()
tm.set("price", "10", 1)
tm.set("price", "12", 4)
assert tm.get("price", 0) == ""
assert tm.get("price", 1) == "10" and tm.get("price", 3) == "10"
assert tm.get("price", 4) == "12" and tm.get("price", 100) == "12"
assert tm.get("missing", 5) == ""


# Q8: vending machine with a state enum
class VendingError(Exception):
    pass


class State(Enum):
    IDLE = "idle"
    HAS_MONEY = "has money"


class VendingMachine:
    def __init__(self, prices, stock):
        self.prices = dict(prices)
        self.stock = dict(stock)
        self.state = State.IDLE
        self.inserted = 0

    def insert(self, amount):
        if amount <= 0:
            raise VendingError("amount must be positive")
        self.inserted += amount
        self.state = State.HAS_MONEY

    def select(self, item):
        if self.state is not State.HAS_MONEY:
            raise VendingError("insert money first")
        if self.stock.get(item, 0) == 0:
            raise VendingError(f"{item} is sold out")
        price = self.prices[item]
        if self.inserted < price:
            raise VendingError(f"need {price - self.inserted} more")
        self.stock[item] -= 1
        change, self.inserted, self.state = self.inserted - price, 0, State.IDLE
        return item, change

    def cancel(self):
        if self.state is not State.HAS_MONEY:
            raise VendingError("nothing to return")
        refund, self.inserted, self.state = self.inserted, 0, State.IDLE
        return refund


vm = VendingMachine(prices={"tea": 30, "juice": 50}, stock={"tea": 1, "juice": 0})
vm.insert(20)
vm.insert(20)
assert vm.select("tea") == ("tea", 10) and vm.state is State.IDLE
for action in (lambda: vm.select("tea"), lambda: vm.cancel()):
    try:
        action()
    except VendingError:
        pass
    else:
        raise AssertionError("expected VendingError")
vm.insert(100)
try:
    vm.select("juice")
except VendingError as e:
    assert str(e) == "juice is sold out"
assert vm.cancel() == 100


# Q9 fixed
class LRU:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)


lru = LRU(2)
lru.put(1, "a")
lru.put(2, "b")
lru.get(1)
lru.put(3, "c")                                       # evicts 2
assert lru.get(2) == -1 and lru.get(1) == "a" and lru.get(3) == "c"

print("Practice 2.12: all checks passed")

Practice 2.12: all checks passed


### Summary of Part 2

* Inheritance (`class Child(Parent)`) reuses code; override methods and extend them with `super()`.
* The MRO (C3) is the search order; `super()` means "next class in the MRO", which makes cooperative
  multiple inheritance (with `**kwargs`) work and runs a diamond's base only once.
* Prefer composition ("has-a") and small mixins over deep inheritance trees.
* Polymorphism and duck typing let one call work on many types; ABCs enforce a contract at creation time.
* Dunder methods make objects work with operators and built-ins. Return `NotImplemented` for unknown types,
  keep `__eq__` and `__hash__` consistent, use `@total_ordering`.
* Dataclasses remove boilerplate (`field(default_factory=...)`, `__post_init__`, `frozen`, `order`, `slots`, `kw_only`).
* `__slots__` saves memory and blocks unknown attributes; enums replace magic strings and numbers.
* SOLID, DRY, KISS and YAGNI guide designs; for interview design questions, clarify, model, choose data
  structures, code, and test.
* Debug class hierarchies with `__mro__`, `vars()` and `__qualname__`.

---

# Part 3: Iterators and Generators

**What these notes cover**

* The difference between an **iterable** and an **iterator**, and the protocol that connects them.
* What a `for` loop really does behind the scenes (with a flowchart).
* How to write your own iterator classes and iterable classes.
* Generator functions (`yield`): how they pause and resume, and why they save memory.
* Generator expressions, infinite generators, `yield from`, and generator pipelines.
* The advanced generator methods `send()`, `throw()` and `close()`.
* When to use generators, when not to, and how to debug the typical iterator bugs.

**Before you start:** Notebook 1, Part 6 (Loops), Part 8 (Lists and Tuples) and Part 11 (Functions). Notebook 2, Part 1 (classes) helps for section 3.2.

## 3.1 Iterables and iterators

Many objects in Python can be looped over: lists, strings, dicts, sets, files, `range`. They all follow one small set of rules called the **iteration protocol**. A *protocol* is an agreement: "if your object has these methods, Python syntax will work with it".

There are two roles:

| Term | Meaning in simple words | Required methods | Examples |
|---|---|---|---|
| **Iterable** | Something you *can* loop over | `__iter__()` | `list`, `str`, `dict`, `set`, `range` |
| **Iterator** | The object that *does* the looping and remembers the position | `__next__()` and `__iter__()` | `iter([1, 2])`, `zip(...)`, `map(...)`, files |

* `iter(obj)` asks an iterable for a fresh iterator (it calls `obj.__iter__()`).
* `next(it)` asks the iterator for the next value (it calls `it.__next__()`).
* When there are no more values, `__next__()` raises the exception **`StopIteration`**.

**In simple words:** a book is an iterable. A bookmark is an iterator. You can put many bookmarks in the same book, and each bookmark remembers its own page.

In [113]:
# iter() gets an iterator from an iterable; next() takes one value at a time
nums = [10, 20, 30]
it = iter(nums)

print(next(it))   # 10
print(next(it))   # 20
print(next(it))   # 30

try:
    next(it)      # nothing left
except StopIteration:
    print("StopIteration: the iterator is exhausted")

10
20
30
StopIteration: the iterator is exhausted


In [114]:
# An iterator returns itself from iter(); an iterable returns a NEW iterator each time
nums = [10, 20, 30]
it = iter(nums)

print(iter(it) is it)            # True  - the iterator is its own iterator
print(iter(nums) is nums)        # False - the list makes a new iterator
print(iter(nums) is iter(nums))  # False - two separate bookmarks

print(type(it).__name__)         # list_iterator

True
False
False
list_iterator


### What a for loop does internally

A `for` loop is a short way of writing "get an iterator, then call `next()` until `StopIteration`". Python handles the `StopIteration` for you, so you never see it.

```text
        +---------------------------+
        |  for item in obj:         |
        +-------------+-------------+
                      |
                      v
        +---------------------------+
        |  it = iter(obj)           |   (calls obj.__iter__())
        +-------------+-------------+
                      |
                      v
        +---------------------------+
   +--->|  item = next(it)          |   (calls it.__next__())
   |    +-------------+-------------+
   |                  |
   |                  v
   |    +---------------------------+  Yes  +---------------------+
   |    | StopIteration raised?     |------>| leave the loop      |
   |    +-------------+-------------+       | (run loop "else")   |
   |                  | No                  +---------------------+
   |                  v
   |    +---------------------------+
   |    |  run the loop body        |
   |    +-------------+-------------+
   |                  |
   +------------------+
```

In [115]:
# The same loop written by hand: this is what "for item in nums" does
nums = [1, 2, 3]

it = iter(nums)
while True:
    try:
        item = next(it)
    except StopIteration:
        break
    print(item, end=" ")      # 1 2 3
print()

# The normal, readable way
for item in nums:
    print(item, end=" ")      # 1 2 3
print()

1 2 3 
1 2 3 


### next() with a default, and iter() with a sentinel

* `next(it, default)` returns `default` instead of raising `StopIteration`. This is handy when you want "the first item, or None".
* `iter(callable, sentinel)` calls `callable()` again and again and stops when the result equals `sentinel`. A **sentinel** is a special value that means "stop".

In [116]:
# next() with a default value
it = iter([])
print(next(it, "empty"))                  # empty

# First even number, or None if there is none
numbers = [3, 7, 8, 11, 12]
first_even = next((n for n in numbers if n % 2 == 0), None)
print(first_even)                          # 8

empty
8


In [117]:
# iter(callable, sentinel): keep calling until the sentinel value appears
readings = [5, 3, 9, -1, 4]               # -1 means "end of data"
position = 0

def read_next():
    global position
    value = readings[position]
    position += 1
    return value

print(list(iter(read_next, -1)))          # [5, 3, 9]

[5, 3, 9]


### Many built-ins return lazy iterators

**Lazy** means "do the work only when a value is asked for". `map`, `filter`, `zip`, `enumerate`, `reversed`, files and generators are lazy iterators. They do not build a full list in memory.

A consequence: an iterator can be used **only once**. After it is exhausted it stays empty.

In [118]:
# zip, map and enumerate return one-shot iterators
names = ["Asha", "Ravi", "Meera"]
scores = [91, 78, 85]

pairs = zip(names, scores)
print(type(pairs).__name__)      # zip
print(list(pairs))               # [('Asha', 91), ('Ravi', 78), ('Meera', 85)]
print(list(pairs))               # []  - already exhausted

upper = map(str.upper, names)
print(next(upper))               # ASHA
print(list(upper))               # ['RAVI', 'MEERA']  - continues from where it stopped

zip
[('Asha', 91), ('Ravi', 78), ('Meera', 85)]
[]
ASHA
['RAVI', 'MEERA']


In [119]:
# range is an iterable (re-usable), NOT an iterator
r = range(3)
print(list(r), list(r))          # [0, 1, 2] [0, 1, 2]

try:
    next(r)
except TypeError as e:
    print("TypeError:", e)       # TypeError: 'range' object is not an iterator

[0, 1, 2] [0, 1, 2]
TypeError: 'range' object is not an iterator


In [120]:
# How to check what you have: the abstract base classes in collections.abc
from collections.abc import Iterable, Iterator

data = [1, 2, 3]
print(isinstance(data, Iterable), isinstance(data, Iterator))              # True False
print(isinstance(iter(data), Iterable), isinstance(iter(data), Iterator))  # True True
print(isinstance(5, Iterable))                                             # False

True False
True True
False


> **Tip:** The most reliable way to test "can I loop over this?" is to try `iter(obj)` and catch `TypeError`. Some old-style classes are iterable through `__getitem__` and do not register as `Iterable`.

> **Common mistake:** Passing the same iterator to two consumers (for example two loops, or `sum()` then `max()`). The second consumer gets nothing. Convert to a list first if you need several passes.

> **Key points**
> * An iterable has `__iter__()`; an iterator has `__next__()` and `__iter__()` (returning itself).
> * `for` calls `iter()` once, then `next()` until `StopIteration`.
> * Iterators are one-shot. Iterables such as lists and `range` can be looped many times.
> * `next(it, default)` avoids `StopIteration`.

---

## 3.2 Writing your own iterators

You can make any class work with `for` by implementing the protocol. There are two designs:

1. **Iterator class**: the object has `__next__()` and keeps the position itself. It is one-shot.
2. **Iterable class**: the object has `__iter__()` that returns a *new* iterator each time. It can be looped many times.

Most real containers are iterables, not iterators.

In [121]:
# An ITERATOR class: it stores the position, so it can be used only once
class CountdownIterator:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self                      # an iterator is its own iterator

    def __next__(self):
        if self.current <= 0:
            raise StopIteration          # signal "no more values"
        value = self.current
        self.current -= 1
        return value

it = CountdownIterator(3)
print(list(it))     # [3, 2, 1]
print(list(it))     # []  - one-shot

[3, 2, 1]
[]


In [122]:
# An ITERABLE class: each loop gets a fresh iterator, so it can be looped many times
class CountdownIterator:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)   # a new bookmark every time

c = Countdown(3)
print(list(c))      # [3, 2, 1]
print(list(c))      # [3, 2, 1]  - works again

[3, 2, 1]
[3, 2, 1]


```text
   Iterable (Countdown)                Iterators (bookmarks)
  +----------------------+   iter()   +-------------------------+
  | start = 3            |----------->| CountdownIterator       |
  | __iter__()           |            | current = 3 -> 2 -> 1   |
  +----------------------+   iter()   +-------------------------+
                         \----------->| CountdownIterator       |
                                      | current = 3 (fresh)     |
                                      +-------------------------+
```

The easiest way to write `__iter__` is as a generator method (you will learn `yield` in section 3.3). It is shown here so you can recognise it:

In [123]:
# The most common real-world style: __iter__ written with yield
class Playlist:
    def __init__(self, *songs):
        self._songs = list(songs)

    def __iter__(self):
        for song in self._songs:
            yield song                      # each loop gets a new generator

    def __len__(self):
        return len(self._songs)

p = Playlist("Intro", "Rain", "Outro")
print(list(p))          # ['Intro', 'Rain', 'Outro']
print(len(p))           # 3
print("Rain" in p)      # True  - "in" uses __iter__ when there is no __contains__

['Intro', 'Rain', 'Outro']
3
True


In [124]:
# A realistic iterator: walk through a list in fixed-size pages
class Pages:
    def __init__(self, items, page_size):
        self.items = items
        self.page_size = page_size

    def __iter__(self):
        for start in range(0, len(self.items), self.page_size):
            yield self.items[start:start + self.page_size]

orders = ["o1", "o2", "o3", "o4", "o5"]
for page in Pages(orders, 2):
    print(page)
# ['o1', 'o2']
# ['o3', 'o4']
# ['o5']

['o1', 'o2']
['o3', 'o4']
['o5']


> **Common mistake:** Forgetting to raise `StopIteration` in `__next__`. The loop then never ends, or returns `None` forever.

> **Common mistake:** Making a container class its own iterator (`__iter__` returns `self`). Then two nested loops over the same object share one position and give wrong results.

> **Key points**
> * An iterator class needs `__iter__` (return `self`) and `__next__` (return a value or raise `StopIteration`).
> * An iterable class needs only `__iter__`, returning a new iterator each time.
> * Writing `__iter__` with `yield` is the shortest and safest way.

---

### Practice 3.1: Iterables and iterators

**A. Predict the output**

1. What does this print?
   ```python
   it = iter([1, 2, 3])
   next(it)
   print(list(it), list(it))
   ```
2. What does this print?
   ```python
   letters = "abc"
   print(iter(letters) is iter(letters))
   ```
3. What does this print?
   ```python
   pairs = zip("ab", [1, 2])
   print(dict(pairs), dict(pairs))
   ```
4. What does this print?
   ```python
   it = iter([5])
   print(next(it, 0), next(it, 0))
   ```

**B. Write code**

5. Write an iterator class `Squares(n)` that produces `0, 1, 4, ..., (n-1)**2` and then stops. It is one-shot.
6. Write an iterable class `Range2(start, stop, step=1)` that behaves like `range` for positive steps and can be looped over many times.
7. Write a function `first(iterable, default=None)` that returns the first item of any iterable, or `default` if it is empty.
8. **[Hard]** Write an iterable class `Cycle(items, times)` that yields the items in order, `times` times in total (for example `Cycle("ab", 2)` gives `a b a b`). It must also work when `items` is itself a one-shot iterator.

**C. Find and fix the bug**

9. This loop prints only the first total correctly. Why?
   ```python
   rows = map(int, ["1", "2", "3"])
   print(sum(rows))
   print(max(rows))
   ```
10. This class causes an infinite loop. Fix it.
    ```python
    class Upto:
        def __init__(self, n):
            self.n = n
            self.i = 0

        def __iter__(self):
            return self

        def __next__(self):
            self.i += 1
            return self.i
    ```

**D. Interview questions**

11. What is the difference between an iterable and an iterator? Give one example of each.
12. What exactly does `for x in obj:` do?
13. **[Hard]** Why should a container's `__iter__` return a new object instead of `self`?

### Solutions 3.1

1. `[2, 3] []`. The first `next` takes `1`; the first `list` takes the rest; the iterator is then empty.
2. `False`. A string is an iterable, so every `iter()` call makes a new iterator object.
3. `{'a': 1, 'b': 2} {}`. `zip` is a one-shot iterator.
4. `5 0`. The second call finds nothing and returns the default.
5. to 8. See the code below.
9. `map` returns a one-shot iterator. `sum` consumed it, so `max` gets an empty iterator and raises `ValueError: max() iterable argument is empty`. Fix: `rows = list(map(int, [...]))`.
10. `__next__` never raises `StopIteration`. Add `if self.i >= self.n: raise StopIteration` before increasing `self.i`.
11. An iterable can produce an iterator (`list`, `str`, `range`). An iterator produces values one by one and remembers its position (`iter(list)`, `zip`, a generator). Every iterator is also an iterable, but not the other way round.
12. It calls `iter(obj)` once, then repeatedly calls `next()` on the result and runs the body for each value, and stops quietly when `StopIteration` is raised.
13. If `__iter__` returns `self`, all loops share one position. Nested loops over the same container break, and a second loop finds nothing. A new iterator per loop gives each loop its own position.

In [125]:
# Solutions 3.1
# Q1-Q4: checks
it = iter([1, 2, 3])
next(it)
assert (list(it), list(it)) == ([2, 3], [])
letters = "abc"
assert (iter(letters) is iter(letters)) is False
pairs = zip("ab", [1, 2])
assert (dict(pairs), dict(pairs)) == ({"a": 1, "b": 2}, {})
it = iter([5])
assert (next(it, 0), next(it, 0)) == (5, 0)


# Q5: one-shot iterator class
class Squares:
    def __init__(self, n):
        self.n = n
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        value = self.i * self.i
        self.i += 1
        return value

sq = Squares(4)
assert list(sq) == [0, 1, 4, 9]
assert list(sq) == []            # one-shot
assert list(Squares(0)) == []    # edge case: empty


# Q6: re-usable iterable class
class Range2:
    def __init__(self, start, stop, step=1):
        if step <= 0:
            raise ValueError("step must be positive")
        self.start, self.stop, self.step = start, stop, step

    def __iter__(self):
        value = self.start
        while value < self.stop:
            yield value
            value += self.step

r = Range2(1, 8, 3)
assert list(r) == [1, 4, 7]
assert list(r) == [1, 4, 7]      # works again
assert list(Range2(5, 5)) == []


# Q7: first item or a default
def first(iterable, default=None):
    return next(iter(iterable), default)

assert first([9, 8]) == 9
assert first([]) is None
assert first("", "none") == "none"
assert first(x for x in range(10) if x > 6) == 7


# Q8: Cycle that also works with one-shot input (store the items once)
class Cycle:
    def __init__(self, items, times):
        self.items = list(items)     # copy, so a one-shot iterator is not lost
        self.times = times

    def __iter__(self):
        for _ in range(self.times):
            yield from self.items    # yield from is explained in section 3.5

assert list(Cycle("ab", 2)) == ["a", "b", "a", "b"]
c = Cycle(iter([1, 2]), 3)
assert list(c) == [1, 2, 1, 2, 1, 2]
assert list(c) == [1, 2, 1, 2, 1, 2]
assert list(Cycle([], 5)) == []


# Q9: fixed version
rows = list(map(int, ["1", "2", "3"]))
assert (sum(rows), max(rows)) == (6, 3)


# Q10: fixed version
class Upto:
    def __init__(self, n):
        self.n = n
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        self.i += 1
        return self.i

assert list(Upto(3)) == [1, 2, 3]
print("Practice 3.1: all checks passed")

Practice 3.1: all checks passed


## 3.3 Generator functions

Writing iterator classes is a lot of typing. A **generator function** gives you an iterator with almost no code. Any function that contains the keyword `yield` is a generator function.

How it behaves:

1. Calling it does **not** run the body. It returns a **generator object** (which is an iterator).
2. Each `next()` runs the body until the next `yield`. The yielded value is returned by `next()`.
3. The function then **pauses**. All its local variables are kept.
4. The next `next()` **resumes** right after that `yield`.
5. When the body ends (or hits `return`), `StopIteration` is raised.

**In simple words:** `return` is "here is the answer, I am done". `yield` is "here is one answer, call me again for the next one".

In [126]:
# Calling a generator function only creates the generator; the body runs on next()
def demo():
    print("  [start]")
    yield 1
    print("  [middle]")
    yield 2
    print("  [end]")

g = demo()
print("created:", type(g).__name__)   # created: generator   (nothing else printed yet)

print(next(g))                         # prints "  [start]" then 1
print(next(g))                         # prints "  [middle]" then 2

try:
    next(g)                            # prints "  [end]" then StopIteration
except StopIteration:
    print("finished")

created: generator
  [start]
1
  [middle]
2
  [end]
finished


### Generator pause and resume

```text
  caller                          generator body
  ------                          --------------
  g = demo()  --------------->    (created, nothing runs)

  next(g)     --------------->    print("[start]")
              <---- 1 ---------   yield 1          (PAUSE, locals kept)

  next(g)     --------------->    resume after yield 1
                                  print("[middle]")
              <---- 2 ---------   yield 2          (PAUSE)

  next(g)     --------------->    resume after yield 2
                                  print("[end]")
              <-- StopIteration   end of function  (FINISHED)
```

The same flow as a flowchart:

```text
        +---------------------------+
        |  next(gen) is called      |
        +-------------+-------------+
                      |
                      v
        +---------------------------+
        | resume where it paused    |
        | (or start at the top)     |
        +-------------+-------------+
                      |
                      v
        +---------------------------+  yield v  +----------------------+
        | run code until "yield"    |---------->| pause; next() gives v|
        | or the end of the body    |           +----------------------+
        +-------------+-------------+
                      | end / return
                      v
        +---------------------------+
        | raise StopIteration       |
        +---------------------------+
```

In [127]:
# Local variables survive between pauses
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b      # a and b are remembered while paused

print(list(fibonacci(8)))    # [0, 1, 1, 2, 3, 5, 8, 13]

for value in fibonacci(4):
    print(value, end=" ")    # 0 1 1 2
print()

[0, 1, 1, 2, 3, 5, 8, 13]
0 1 1 2 


In [128]:
# A generator replaces the whole iterator class from section 3.2
def countdown(start):
    while start > 0:
        yield start
        start -= 1

print(list(countdown(3)))    # [3, 2, 1]

g = countdown(2)
print(list(g), list(g))      # [2, 1] []  - a generator object is one-shot too

[3, 2, 1]
[2, 1] []


In [129]:
# The generator states: created -> suspended -> closed
import inspect

def two_values():
    yield "a"
    yield "b"

g = two_values()
print(inspect.getgeneratorstate(g))   # GEN_CREATED
next(g)
print(inspect.getgeneratorstate(g))   # GEN_SUSPENDED
list(g)
print(inspect.getgeneratorstate(g))   # GEN_CLOSED

GEN_CREATED
GEN_SUSPENDED
GEN_CLOSED


### return inside a generator

A `return value` inside a generator ends it. The value is not yielded; it is stored in the `StopIteration` exception (`e.value`). A `for` loop ignores it. It is mainly used with `yield from` (section 3.5).

In [130]:
# return in a generator stops it; the value travels inside StopIteration
def take_until_negative(numbers):
    count = 0
    for n in numbers:
        if n < 0:
            return count          # stop here
        count += 1
        yield n

print(list(take_until_negative([4, 7, -1, 9])))   # [4, 7]

g = take_until_negative([1, -5])
next(g)
try:
    next(g)
except StopIteration as e:
    print("returned value:", e.value)            # returned value: 1

[4, 7]
returned value: 1


| `return` | `yield` |
|---|---|
| Ends the function for good | Pauses the function; it can resume later |
| Gives one result | Gives zero, one or many results over time |
| Normal function | Turns the function into a generator function |

> **Warning:** A `yield` anywhere in the body (even in code that never runs) makes the whole function a generator function. Calling it then returns a generator and runs nothing.

> **Key points**
> * A function with `yield` returns a generator object when called; the body runs only on `next()`.
> * Each `yield` hands out one value and pauses; local variables are kept.
> * The end of the body (or `return`) raises `StopIteration`.
> * Generator objects are iterators, so they are one-shot.

---

## 3.4 Generator expressions and memory

A **generator expression** looks like a list comprehension but uses round brackets. It produces values lazily, one at a time.

```python
squares_list = [x * x for x in range(5)]   # builds the whole list now
squares_gen  = (x * x for x in range(5))   # builds nothing yet
```

When a generator expression is the only argument of a function, you can drop the extra brackets: `sum(x * x for x in range(5))`.

In [131]:
# Generator expression vs list comprehension
squares_list = [x * x for x in range(5)]
squares_gen = (x * x for x in range(5))

print(squares_list)                  # [0, 1, 4, 9, 16]
print(type(squares_gen).__name__)    # generator
print(next(squares_gen))             # 0
print(list(squares_gen))             # [1, 4, 9, 16]

# Only argument: no extra brackets needed
print(sum(x * x for x in range(5)))  # 30

# With other arguments, the brackets are required
print(sorted((x % 3 for x in range(6)), reverse=True))   # [2, 2, 1, 1, 0, 0]

[0, 1, 4, 9, 16]
generator
0
[1, 4, 9, 16]
30
[2, 2, 1, 1, 0, 0]


### Why generators save memory

A list stores every element at once. A generator stores only its current state (a few local variables), no matter how many values it will produce.

```text
  list:       [ 0 | 1 | 4 | 9 | 16 | ... | 9999800001 ]   all in memory

  generator:  ( current x = 7 )  -> computes the next value on request
```

In [132]:
# Memory: a list grows with n, a generator stays tiny (sizes depend on the Python build)
import sys

as_list = [x * x for x in range(100_000)]
as_gen = (x * x for x in range(100_000))

print("list is larger than 100 KB:", sys.getsizeof(as_list) > 100_000)   # True
print("generator is under 1 KB:", sys.getsizeof(as_gen) < 1_000)         # True

print(sum(as_gen))      # 333328333350000
print(sum(as_gen))      # 0  - the generator is already exhausted

list is larger than 100 KB: True
generator is under 1 KB: True
333328333350000
0


In [133]:
# Lazy evaluation: values are computed only when needed
def loud_square(x):
    print(f"  computing {x}")
    return x * x

gen = (loud_square(x) for x in range(1_000_000))   # nothing computed yet
print("before next")
print(next(gen))     # prints "  computing 0" then 0
print(next(gen))     # prints "  computing 1" then 1
# The other 999,998 values are never computed.

before next
  computing 0
0
  computing 1
1


In [134]:
# any() and all() stop early, which works well with generator expressions
orders = [120, 80, 450, 30, 990]
print(any(amount > 400 for amount in orders))   # True  (stops at 450)
print(all(amount > 0 for amount in orders))     # True
print(max(amount for amount in orders if amount < 100))   # 80

True
True
80


| Use a list comprehension when | Use a generator expression when |
|---|---|
| You need the result several times | You go through the values once |
| You need `len()`, indexing or slicing | The data is large or endless |
| The data is small | You feed it straight into `sum`, `max`, `any`, `"".join` |

> **Common mistake:** Expecting `len()` or indexing to work on a generator. They raise `TypeError` (see section 3.9).

> **Note:** The variable inside a generator expression or comprehension does not leak out to the surrounding code. The first `for` part is evaluated immediately, the rest lazily.

> **Key points**
> * `(expr for x in data)` is a lazy generator; `[expr for x in data]` builds a list now.
> * Generators use constant memory, but you can loop over them only once.
> * Feed generator expressions directly to `sum`, `min`, `max`, `any`, `all`, `"".join`.

---

### Practice 3.2: Generator functions and expressions

**A. Predict the output**

1. What is printed by the first line alone, and by `next(g)`?
   ```python
   def demo():
       print("start")
       yield 1

   g = demo()
   print(next(g))
   ```
2. What does this print?
   ```python
   gen = (x for x in range(3))
   print(list(gen), list(gen), 2 in gen)
   ```
3. What does this print?
   ```python
   def f():
       yield 1
       return 2
       yield 3

   print(list(f()))
   ```
4. What does this print?
   ```python
   def maybe(flag):
       if flag:
           yield "yes"
       return "no"

   print(maybe(False))
   ```
   (Describe it; the address part changes.)

**B. Write code**

5. Write a generator `countdown(n)` that yields `n, n-1, ..., 1`.
6. Write a generator `evens(limit)` that yields even numbers from 0 up to and including `limit`.
7. Write a generator `char_runs(text)` that yields `(char, count)` for each run of equal characters: `"aaabcc"` gives `('a', 3), ('b', 1), ('c', 2)`.
8. Using one generator expression, compute the total length of all words in `["data", "is", "fun"]` that have more than 2 letters.
9. **[Hard]** Write a generator `batches(iterable, size)` that yields lists of `size` items (the last list may be shorter). It must work with any iterable, including another generator.

**C. Find and fix the bug**

10. This function was meant to return a list of doubled values, but the caller gets a generator. Why?
    ```python
    def doubled(values):
        result = []
        for v in values:
            result.append(v * 2)
            yield v
        return result
    ```
11. The second loop prints nothing. Fix it.
    ```python
    names = (n.strip() for n in [" Asha", "Ravi "])
    for n in names:
        print(n)
    for n in names:
        print(n.upper())
    ```

**D. Interview questions**

12. What happens when you call a generator function? When does its body start running?
13. **[Hard]** Why is `sum(x * x for x in range(10**7))` better for memory than `sum([x * x for x in range(10**7)])`? Is it always faster?

### Solutions 3.2

1. `g = demo()` prints nothing. `print(next(g))` prints `start`, then `1`.
2. `[0, 1, 2] [] False`. The generator is empty after the first `list`.
3. `[1]`. `return` ends the generator; its value `2` is not yielded, and `yield 3` never runs.
4. It prints a generator object like `<generator object maybe at 0x...>`. The function contains `yield`, so calling it never runs the body; the `return "no"` value is not printed.
5. to 9. See the code below.
10. The `yield` makes it a generator function, so calling it returns a generator and the returned list is hidden inside `StopIteration`. Remove the `yield` (normal function returning a list), or yield `v * 2` and drop the list.
11. A generator is one-shot. Use a list: `names = [n.strip() for n in [...]]`.
12. You get a generator object; nothing in the body runs. The body starts at the first `next()` (or the first loop step) and runs until the first `yield`.
13. The generator version keeps one value at a time, so memory is constant; the list version builds ten million objects first. Speed is similar; for small data a list comprehension can even be slightly faster. Choose by memory and by how many passes you need.

In [135]:
# Solutions 3.2
# Q2 and Q3 checks
gen = (x for x in range(3))
assert (list(gen), list(gen), 2 in gen) == ([0, 1, 2], [], False)

def f():
    yield 1
    return 2
    yield 3

assert list(f()) == [1]


# Q5
def countdown(n):
    while n > 0:
        yield n
        n -= 1

assert list(countdown(3)) == [3, 2, 1]
assert list(countdown(0)) == []


# Q6
def evens(limit):
    for n in range(0, limit + 1, 2):
        yield n

assert list(evens(6)) == [0, 2, 4, 6]
assert list(evens(5)) == [0, 2, 4]
assert list(evens(-1)) == []


# Q7: runs of equal characters
def char_runs(text):
    if not text:
        return
    current, count = text[0], 1
    for ch in text[1:]:
        if ch == current:
            count += 1
        else:
            yield current, count
            current, count = ch, 1
    yield current, count

assert list(char_runs("aaabcc")) == [("a", 3), ("b", 1), ("c", 2)]
assert list(char_runs("")) == []
assert list(char_runs("z")) == [("z", 1)]


# Q8
words = ["data", "is", "fun"]
total = sum(len(w) for w in words if len(w) > 2)
assert total == 7


# Q9: batches from any iterable
def batches(iterable, size):
    if size < 1:
        raise ValueError("size must be at least 1")
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:                      # the last, shorter batch
        yield batch

assert list(batches(range(5), 2)) == [[0, 1], [2, 3], [4]]
assert list(batches((c for c in "abc"), 5)) == [["a", "b", "c"]]
assert list(batches([], 3)) == []


# Q10: fixed (two correct options)
def doubled_list(values):
    return [v * 2 for v in values]

def doubled_gen(values):
    for v in values:
        yield v * 2

assert doubled_list([1, 2]) == [2, 4]
assert list(doubled_gen([1, 2])) == [2, 4]


# Q11: fixed
names = [n.strip() for n in [" Asha", "Ravi "]]
assert [n.upper() for n in names] == ["ASHA", "RAVI"]
assert [n.upper() for n in names] == ["ASHA", "RAVI"]   # second pass works
print("Practice 3.2: all checks passed")

Practice 3.2: all checks passed


## 3.5 Infinite generators and yield from

### Infinite generators

Because a generator computes values only on request, it can describe an **endless** sequence. This is safe as long as the consumer stops: use `break`, `next()` a fixed number of times, or `itertools.islice` (take the first n items of any iterator).

In [136]:
# An infinite generator and two safe ways to consume it
from itertools import islice

def naturals(start=0):
    n = start
    while True:          # endless, but lazy
        yield n
        n += 1

print(list(islice(naturals(), 5)))        # [0, 1, 2, 3, 4]
print(list(islice(naturals(10), 3)))      # [10, 11, 12]

for n in naturals(1):
    if n * n > 30:
        break
    print(n, end=" ")                     # 1 2 3 4 5
print()

[0, 1, 2, 3, 4]
[10, 11, 12]
1 2 3 4 5 


In [137]:
# Combining infinite generators with filters
from itertools import islice

def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

even_fibs = (n for n in fibonacci() if n % 2 == 0)
print(list(islice(even_fibs, 6)))         # [0, 2, 8, 34, 144, 610]

[0, 2, 8, 34, 144, 610]


In [138]:
# Useful infinite iterators from itertools (more in Part 7)
from itertools import count, cycle, repeat, islice

print(list(islice(count(100, 5), 3)))     # [100, 105, 110]
print(list(islice(cycle("AB"), 5)))       # ['A', 'B', 'A', 'B', 'A']
print(list(repeat("x", 3)))               # ['x', 'x', 'x']

# Give each order an ID without a manual counter
order_ids = count(1001)
for item in ["pen", "book"]:
    print(next(order_ids), item)          # 1001 pen / 1002 book

[100, 105, 110]
['A', 'B', 'A', 'B', 'A']
['x', 'x', 'x']
1001 pen
1002 book


> **Warning:** Never call `list()`, `sum()`, `sorted()` or `len()`-like functions on an infinite generator. They try to read everything and never finish.

### yield from: delegating to another iterable

`yield from iterable` yields every value from `iterable`, one by one. It replaces this loop:

```python
for value in iterable:
    yield value
```

It also passes `send()` / `throw()` through and gives back the sub-generator's `return` value. This makes it ideal for **recursive** generators (trees, nested lists).

In [139]:
# yield from replaces a simple inner loop
def chain_all(*iterables):
    for iterable in iterables:
        yield from iterable

print(list(chain_all([1, 2], "ab", range(2))))   # [1, 2, 'a', 'b', 0, 1]

[1, 2, 'a', 'b', 0, 1]


In [140]:
# Recursive generator: flatten any depth of nested lists
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # delegate to the recursive call
        else:
            yield item

print(list(flatten([1, [2, [3, [4]]], 5])))    # [1, 2, 3, 4, 5]
print(list(flatten([])))                        # []

[1, 2, 3, 4, 5]
[]


In [141]:
# Walking a tree (folder structure) with a recursive generator
tree = {
    "name": "root",
    "children": [
        {"name": "docs", "children": [{"name": "a.txt", "children": []}]},
        {"name": "b.txt", "children": []},
    ],
}

def walk(node, depth=0):
    yield depth, node["name"]
    for child in node["children"]:
        yield from walk(child, depth + 1)

for depth, name in walk(tree):
    print("  " * depth + name)
# root
#   docs
#     a.txt
#   b.txt

root
  docs
    a.txt
  b.txt


In [142]:
# yield from also returns the sub-generator's return value
def read_block(lines):
    count = 0
    for line in lines:
        yield line
        count += 1
    return count                      # becomes the value of "yield from"

def read_all(blocks):
    total = 0
    for block in blocks:
        total += yield from read_block(block)
    print("  lines read:", total)

print(list(read_all([["a", "b"], ["c"]])))
#   lines read: 3
# ['a', 'b', 'c']

  lines read: 3
['a', 'b', 'c']


> **Key points**
> * Infinite generators are fine when the consumer stops (`break`, `islice`, `next`).
> * `yield from it` yields everything from `it`; it is the clean way to write recursive generators.
> * `itertools.count`, `cycle` and `repeat` are ready-made infinite iterators.

---

## 3.6 Generator pipelines

A **pipeline** connects small generators like pipes: each stage takes an iterator, transforms it, and yields results to the next stage. Only one item travels through the pipeline at a time, so even huge files use very little memory.

```text
  raw lines --> [ parse ] --> [ keep errors ] --> [ get message ] --> consumer
               one item at a time flows through every stage
```

This is the same idea as Unix pipes (`cat log | grep ERROR | cut ...`).

In [143]:
# A log-processing pipeline built from small generators
log_lines = [
    "2026-10-01 INFO  start",
    "2026-10-01 ERROR disk full",
    "2026-10-02 INFO  ok",
    "2026-10-02 ERROR network down",
    "2026-10-03 WARN  slow",
]

def parse(lines):
    for line in lines:
        date, level, message = line.split(maxsplit=2)
        yield {"date": date, "level": level, "message": message}

def only_level(records, level):
    for record in records:
        if record["level"] == level:
            yield record

def messages(records):
    for record in records:
        yield record["message"]

pipeline = messages(only_level(parse(log_lines), "ERROR"))
print(list(pipeline))     # ['disk full', 'network down']

['disk full', 'network down']


In [144]:
# The same pipeline with generator expressions
log_lines = ["INFO start", "ERROR disk", "INFO ok", "ERROR net"]

parsed = (line.split(maxsplit=1) for line in log_lines)
errors = (msg for level, msg in parsed if level == "ERROR")
upper = (msg.upper() for msg in errors)

print(list(upper))        # ['DISK', 'NET']

['DISK', 'NET']


In [145]:
# See the "one item at a time" order: stages interleave
def source(items):
    for item in items:
        print(f"  source -> {item}")
        yield item

def double(items):
    for item in items:
        print(f"  double -> {item * 2}")
        yield item * 2

for result in double(source([1, 2])):
    print("consumer got", result)
#   source -> 1
#   double -> 2
# consumer got 2
#   source -> 2
#   double -> 4
# consumer got 4

  source -> 1
  double -> 2
consumer got 2
  source -> 2
  double -> 4
consumer got 4


In [146]:
# Pipeline over a real file, read line by line (the file is never fully loaded)
import os
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "sales.csv")
    with open(path, "w", encoding="utf-8") as f:
        f.write("item,amount\npen,10\nbook,250\nbag,90\n")

    def read_lines(file_path):
        with open(file_path, encoding="utf-8") as f:
            yield from f                       # a file object is a lazy iterator of lines

    lines = read_lines(path)
    next(lines)                                # skip the header
    amounts = (int(line.split(",")[1]) for line in lines)
    print(sum(amounts))                        # 350

350


> **Tip:** Keep each stage small and single-purpose. Small stages are easy to test with a short list, and easy to reuse.

> **Key points**
> * A pipeline is a chain of generators; each pulls one item from the previous one.
> * Memory use stays constant, whatever the input size.
> * Stages run interleaved, not one after another.

---

## 3.7 send, throw and close (advanced)

Generators can also **receive** values. This is rarely needed in everyday code, but you should recognise it (it is how coroutines were built before `async`/`await`, see Part 8).

* `gen.send(value)` resumes the generator and makes the paused `yield` expression evaluate to `value`.
* The first call must be `next(gen)` (or `gen.send(None)`) to run up to the first `yield`. This is called **priming**.
* `gen.throw(exc)` raises `exc` inside the generator at the paused `yield`.
* `gen.close()` raises `GeneratorExit` inside the generator so it can clean up, then finishes it.

In [147]:
# send(): a running average that receives numbers
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        value = yield average      # pause; send() puts a value here
        total += value
        count += 1
        average = total / count

avg = running_average()
next(avg)                          # prime: run to the first yield
print(avg.send(10))                # 10.0
print(avg.send(20))                # 15.0
print(avg.send(30))                # 20.0

10.0
15.0
20.0


In [148]:
# Sending a value to a generator that was not primed is an error
def echo():
    while True:
        received = yield
        print("got", received)

g = echo()
try:
    g.send("hello")
except TypeError as e:
    print("TypeError:", e)   # TypeError: can't send non-None value to a just-started generator

next(g)                      # prime it
g.send("hello")              # got hello

TypeError: can't send non-None value to a just-started generator
got hello


In [149]:
# close() and cleanup: finally runs when the generator is closed early
def reader():
    print("  open resource")
    try:
        for i in range(100):
            yield i
    finally:
        print("  close resource")    # runs on close(), on error, or at the normal end

r = reader()
print(next(r), next(r))     # open resource / 0 1
r.close()                   # close resource
print("after close")

  open resource
0 1
  close resource
after close


In [150]:
# throw(): raise an exception at the paused yield
def worker():
    while True:
        try:
            job = yield
            print("  working on", job)
        except ValueError as e:
            print("  skipped bad job:", e)

w = worker()
next(w)
w.send("report")                    #   working on report
w.throw(ValueError("empty name"))   #   skipped bad job: empty name
w.send("invoice")                   #   working on invoice
w.close()

  working on report
  skipped bad job: empty name
  working on invoice


> **Note:** For new code that waits for I/O, use `async def` and `await` (Part 8), not `send()`-based coroutines.

> **Key points**
> * `send(v)` resumes a generator and makes the current `yield` return `v`; prime with `next()` first.
> * `throw()` raises an exception inside the generator; `close()` stops it and runs its `finally` blocks.
> * Put cleanup code of a generator in `try/finally`.

---

### Practice 3.3: Infinite generators, yield from, pipelines and send

**A. Predict the output**

1. What does this print?
   ```python
   from itertools import islice

   def gen():
       n = 1
       while True:
           yield n
           n *= 2

   print(list(islice(gen(), 5)))
   ```
2. What does this print?
   ```python
   def inner():
       yield 1
       yield 2

   def outer():
       yield 0
       yield from inner()
       yield 3

   print(list(outer()))
   ```
3. What does this print (order of lines)?
   ```python
   def a():
       for i in range(2):
           print("a", i)
           yield i

   def b(items):
       for i in items:
           print("b", i)
           yield i

   list(b(a()))
   ```

**B. Write code**

4. Write an infinite generator `primes()` that yields 2, 3, 5, 7, 11, ... Use it with `islice` to get the first 10 primes.
5. Write a recursive generator `leaves(tree)` for nested tuples/lists that yields only the non-list, non-tuple items: `leaves((1, [2, (3,)], 4))` gives `1, 2, 3, 4`.
6. Build a pipeline: from a list of strings like `"Asha,91"`, keep scores of at least 80, and yield names in upper case.
7. Write a generator `windowed(iterable, k)` that yields sliding windows as tuples: `[1, 2, 3, 4], 3` gives `(1, 2, 3), (2, 3, 4)`.
8. **[Hard]** Write a generator-based `max_tracker()` that receives numbers via `send()` and yields the largest number seen so far.

**C. Find and fix the bug**

9. This flatten function returns generator objects instead of numbers. Why?
   ```python
   def flatten(items):
       for item in items:
           if isinstance(item, list):
               yield flatten(item)
           else:
               yield item
   ```
10. This program never finishes. Why?
    ```python
    from itertools import count

    squares = (n * n for n in count())
    small = [s for s in squares if s < 100]
    ```

**D. Interview questions**

11. What is the difference between `yield` and `yield from`?
12. **[Hard]** How would you process a 50 GB log file on a machine with 8 GB of RAM in Python?

### Solutions 3.3

1. `[1, 2, 4, 8, 16]`.
2. `[0, 1, 2, 3]`.
3. `a 0`, `b 0`, `a 1`, `b 1`. Stages interleave: each item flows through both before the next item starts.
4. to 8. See the code below.
9. `yield flatten(item)` yields the generator object itself. Use `yield from flatten(item)` to yield its values.
10. The list comprehension keeps asking `count()` for more values; the condition filters values but never stops the loop. Use `itertools.takewhile(lambda s: s < 100, squares)` or a loop with `break`.
11. `yield x` produces one value. `yield from it` produces every value of `it`, forwards `send()`/`throw()` to it, and evaluates to its `return` value.
12. Read the file line by line (a file object is a lazy iterator), and build a generator pipeline: parse, filter, transform, then aggregate with a running counter or `collections.Counter`. Memory stays constant because only one line is held at a time. For more speed, split the file into byte ranges and use several processes (Part 8).

In [151]:
# Solutions 3.3
from itertools import islice, takewhile, count
from collections import deque

# Q1-Q3 checks
def gen():
    n = 1
    while True:
        yield n
        n *= 2

assert list(islice(gen(), 5)) == [1, 2, 4, 8, 16]


# Q4: infinite primes (trial division by the primes found so far)
def primes():
    found = []
    candidate = 2
    while True:
        if all(candidate % p != 0 for p in found if p * p <= candidate):
            found.append(candidate)
            yield candidate
        candidate += 1

assert list(islice(primes(), 10)) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


# Q5: leaves of a nested structure
def leaves(tree):
    for item in tree:
        if isinstance(item, (list, tuple)):
            yield from leaves(item)
        else:
            yield item

assert list(leaves((1, [2, (3,)], 4))) == [1, 2, 3, 4]
assert list(leaves([])) == []


# Q6: pipeline
rows = ["Asha,91", "Ravi,78", "Meera,85"]

def parse_rows(lines):
    for line in lines:
        name, score = line.split(",")
        yield name, int(score)

def at_least(records, minimum):
    for name, score in records:
        if score >= minimum:
            yield name, score

def upper_names(records):
    for name, _ in records:
        yield name.upper()

assert list(upper_names(at_least(parse_rows(rows), 80))) == ["ASHA", "MEERA"]


# Q7: sliding windows with a deque of fixed length
def windowed(iterable, k):
    window = deque(maxlen=k)
    for item in iterable:
        window.append(item)
        if len(window) == k:
            yield tuple(window)

assert list(windowed([1, 2, 3, 4], 3)) == [(1, 2, 3), (2, 3, 4)]
assert list(windowed([1], 3)) == []


# Q8: max tracker with send()
def max_tracker():
    largest = None
    while True:
        value = yield largest
        if largest is None or value > largest:
            largest = value

tracker = max_tracker()
next(tracker)
assert [tracker.send(v) for v in [3, -1, 7, 5]] == [3, 3, 7, 7]


# Q9: fixed
def flatten(items):
    for item in items:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

assert list(flatten([1, [2, [3]]])) == [1, 2, 3]


# Q10: fixed with takewhile (stops at the first value that fails)
squares = (n * n for n in count())
small = list(takewhile(lambda s: s < 100, squares))
assert small == [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
print("Practice 3.3: all checks passed")

Practice 3.3: all checks passed


## 3.8 When to use generators (and when not to)

Generators are a great default for "produce values one by one", but they are not always the right choice.

| Use a generator when | Use a list (or other container) when |
|---|---|
| The data is large, streamed, or endless | The data is small and fits easily in memory |
| You go through the values once | You need several passes over the data |
| You want to stop early (search, first match) | You need `len()`, indexing, slicing, or sorting |
| You build a processing pipeline | You need to print or debug the whole result |
| Values are expensive and may not all be needed | You need random access (`data[i]`) |

**Rules of thumb:**

* Return a list from a function when callers will usually need all items and the data is small.
* Return a generator (or accept any iterable) when the data can be large.
* If a caller needs a list, it can always write `list(gen)`.
* Many tools in `itertools` (Part 7) save you from writing generators by hand: `chain`, `islice`, `takewhile`, `groupby`, `pairwise`, `batched` (3.12+).

In [152]:
# A generator is the natural choice for "stop at the first match"
def find_first_large(orders, limit):
    return next((o for o in orders if o["total"] > limit), None)

orders = [{"id": 1, "total": 40}, {"id": 2, "total": 900}, {"id": 3, "total": 950}]
print(find_first_large(orders, 500))    # {'id': 2, 'total': 900}
print(find_first_large(orders, 5000))   # None

{'id': 2, 'total': 900}
None


In [153]:
# A list is the natural choice when you need several passes and len()
scores = [72, 91, 85, 60]
average = sum(scores) / len(scores)
above = [s for s in scores if s > average]
print(average, above)                    # 77.0 [91, 85]

77.0 [91, 85]


In [154]:
# itertools.batched (Python 3.12+) does what the batches() practice generator does
import sys
from itertools import batched

print(list(batched("abcde", 2)))          # [('a', 'b'), ('c', 'd'), ('e',)]

# Python 3.13 added strict=True, which raises if the last batch is short
if sys.version_info >= (3, 13):
    try:
        list(batched("abcde", 2, strict=True))
    except ValueError as e:
        print("ValueError:", e)           # ValueError: batched(): incomplete batch

[('a', 'b'), ('c', 'd'), ('e',)]
ValueError: batched(): incomplete batch


> **Interview tip:** If asked "how would you handle input that does not fit in memory?", answer with streaming (iterate line by line), generators and pipelines, and mention the trade-off: one pass only, no random access.

> **Key points**
> * Generators: big, streamed, endless data; one pass; early stop.
> * Lists: small data; many passes; `len`, indexing, sorting.
> * Check `itertools` before writing your own generator.

---

## 3.9 How to debug iterators and generators

Most iterator bugs fall into a few groups. Learn the error messages and you will spot them quickly.

| Symptom or error | What it means | Fix |
|---|---|---|
| Second loop prints nothing | The iterator or generator was already exhausted | Make a list, or create a new generator |
| `TypeError: 'list' object is not an iterator` | You called `next()` on an iterable | Call `iter()` first |
| `TypeError: 'int' object is not iterable` | You looped over something that has no `__iter__` | Check the type; maybe you meant `range(n)` |
| `TypeError: object of type 'generator' has no len()` | Generators do not know their length | Use `list()`, or count while looping |
| `TypeError: 'generator' object is not subscriptable` | No indexing on generators | Use `itertools.islice` or a list |
| `RuntimeError: generator raised StopIteration` | A bare `next()` inside a generator ran out (PEP 479) | Use `next(it, default)` or catch `StopIteration` |
| `RuntimeError: dictionary changed size during iteration` | You changed a dict while looping over it | Loop over `list(d)` or build a new dict |
| Function "does nothing" | It contains `yield`, so calling it only creates a generator | Iterate over it, or remove `yield` |

In [155]:
# Reproduce the common errors and read the messages
gen = (x for x in range(3))

for label, action in [
    ("next on a list", lambda: next([1, 2])),
    ("loop over an int", lambda: iter(5)),
    ("len of a generator", lambda: len(gen)),
    ("index a generator", lambda: gen[0]),
]:
    try:
        action()
    except TypeError as e:
        print(f"{label:20} -> {type(e).__name__}: {e}")
# next on a list       -> TypeError: 'list' object is not an iterator
# loop over an int     -> TypeError: 'int' object is not iterable
# len of a generator   -> TypeError: object of type 'generator' has no len()
# index a generator    -> TypeError: 'generator' object is not subscriptable

next on a list       -> TypeError: 'list' object is not an iterator
loop over an int     -> TypeError: 'int' object is not iterable
len of a generator   -> TypeError: object of type 'generator' has no len()
index a generator    -> TypeError: 'generator' object is not subscriptable


In [156]:
# PEP 479: a StopIteration escaping from a generator body becomes RuntimeError
def first_of_each(groups):
    for group in groups:
        yield next(iter(group))       # Common mistake: crashes on an empty group

try:
    print(list(first_of_each([[1, 2], [], [3]])))
except RuntimeError as e:
    print("RuntimeError:", e)         # RuntimeError: generator raised StopIteration

# Fix: use a default (or skip empty groups explicitly)
def first_of_each_fixed(groups):
    for group in groups:
        value = next(iter(group), None)
        if value is not None:
            yield value

print(list(first_of_each_fixed([[1, 2], [], [3]])))   # [1, 3]

RuntimeError: generator raised StopIteration
[1, 3]


In [157]:
# Changing a dict while iterating over it
stock = {"pen": 0, "book": 4, "bag": 0}

try:
    for item in stock:
        if stock[item] == 0:
            del stock[item]
except RuntimeError as e:
    print("RuntimeError:", e)     # RuntimeError: dictionary changed size during iteration

# Fix: iterate over a snapshot of the keys, or build a new dict
stock = {"pen": 0, "book": 4, "bag": 0}
for item in list(stock):
    if stock[item] == 0:
        del stock[item]
print(stock)                      # {'book': 4}

RuntimeError: dictionary changed size during iteration
{'book': 4}


In [158]:
# Debugging trick: peek into a pipeline without breaking it
def tap(items, label):
    """Print every item that passes, then pass it on unchanged."""
    for item in items:
        print(f"  [{label}] {item!r}")
        yield item

raw = ["10", "x", "30"]
numbers = (int(s) for s in tap(raw, "raw") if s.isdigit())
print(sum(tap(numbers, "int")))
#   [raw] '10'
#   [int] 10
#   [raw] 'x'
#   [raw] '30'
#   [int] 30
# 40

  [raw] '10'
  [int] 10
  [raw] 'x'
  [raw] '30'
  [int] 30
40


### Debugging flowchart for "my loop gives no or wrong values"

```text
        +-------------------------------+
        | Loop gives no / wrong values  |
        +---------------+---------------+
                        |
                        v
        +-------------------------------+  Yes  +------------------------+
        | Was this iterator used before?|------>| Use a list or create a |
        | (print(list(it)) consumes it!)|       | fresh generator        |
        +---------------+---------------+       +------------------------+
                        | No
                        v
        +-------------------------------+  Yes  +------------------------+
        | Is it a generator that never  |------>| Iterate it: for / list |
        | got iterated?                 |       +------------------------+
        +---------------+---------------+
                        | No
                        v
        +-------------------------------+
        | Add a tap() stage or print    |
        | inside each stage; test each  |
        | stage alone with a tiny list  |
        +---------------+---------------+
                        |
                        v
        +-------------------------------+
        | Fix the stage, re-run, then   |
        | add an assert-based test      |
        +-------------------------------+
```

> **Warning:** `print(list(gen))` while debugging **consumes** the generator. The code after it then sees an empty generator. Convert to a list once, store it, and use the list for both printing and the rest of the code.

> **Tip:** In a debugger, a generator object shows no values. Step *into* the generator body (step into on the `next()` or the `for` line) to see its local variables.

> **Key points**
> * Most bugs are "already exhausted", "not an iterator", or "generator never iterated".
> * A bare `next()` inside a generator can turn into `RuntimeError` (PEP 479).
> * Use a `tap()` stage or small test lists to inspect pipelines.

---

### Practice 3.4: Choosing and debugging

**A. Predict the output**

1. What does this print?
   ```python
   data = (n for n in [3, 1, 2])
   print(sorted(data), sorted(data))
   ```
2. What does this print?
   ```python
   def gen():
       yield next(iter([]))

   try:
       list(gen())
   except Exception as e:
       print(type(e).__name__)
   ```

**B. Write code**

3. Write `count_items(iterable)` that returns how many items an iterable has, without building a list (it may consume the iterable).
4. Write `peek(iterable)` that returns `(first_item, iterator)` where the returned iterator still yields **all** items including the first one. Return `(None, empty iterator)` for empty input. Hint: `itertools.chain`.
5. **[Hard]** Write `unique_everseen(iterable)` that yields items in their first-seen order, skipping duplicates, using O(1) memory per distinct item.

**C. Find and fix the bug**

6. Why is `total` zero?
   ```python
   prices = (p for p in [10, 20, 30])
   print("count:", len(list(prices)))
   total = sum(prices)
   ```
7. Nothing is printed. Why?
   ```python
   def show(items):
       for item in items:
           print(item)
           yield item

   show(["a", "b"])
   ```

**D. Interview questions**

8. When would you return a list instead of a generator from a function?
9. **[Hard]** What is PEP 479 and what problem did it fix?

### Solutions 3.4

1. `[1, 2, 3] []`. `sorted` consumes the generator.
2. `RuntimeError`. The `StopIteration` from the inner `next()` is converted by PEP 479.
3. to 5. See the code below.
6. `len(list(prices))` consumed the generator. Store `prices = list(...)` once, then use `len(prices)` and `sum(prices)`.
7. Calling a generator function only creates a generator; the body never runs. Iterate it: `for _ in show([...]): pass` or `list(show([...]))`, or make it a normal function.
8. When the data is small, when callers need `len`, indexing or several passes, or when you want errors to happen inside the function (a generator raises its errors later, at iteration time).
9. Before Python 3.7, a `StopIteration` raised by accident inside a generator (for example from a bare `next()`) silently ended the generator, hiding bugs. PEP 479 turns it into `RuntimeError`, so the bug is visible.

In [159]:
# Solutions 3.4
from itertools import chain

# Q1 check
data = (n for n in [3, 1, 2])
assert (sorted(data), sorted(data)) == ([1, 2, 3], [])


# Q3
def count_items(iterable):
    return sum(1 for _ in iterable)

assert count_items(x for x in range(5)) == 5
assert count_items([]) == 0
assert count_items("abc") == 3


# Q4: look at the first item without losing it
def peek(iterable):
    it = iter(iterable)
    first = next(it, None)
    if first is None:
        return None, iter(())
    return first, chain([first], it)

head, rest = peek(x * 10 for x in range(3))
assert head == 0 and list(rest) == [0, 10, 20]
head, rest = peek([])
assert head is None and list(rest) == []


# Q5: keep first-seen order, skip duplicates
def unique_everseen(iterable):
    seen = set()
    for item in iterable:
        if item not in seen:
            seen.add(item)
            yield item

assert list(unique_everseen("abracadabra")) == ["a", "b", "r", "c", "d"]
assert list(unique_everseen([])) == []


# Q6: fixed
prices = list(p for p in [10, 20, 30])
assert (len(prices), sum(prices)) == (3, 60)
print("Practice 3.4: all checks passed")

Practice 3.4: all checks passed


### Summary of Part 3

* An **iterable** gives an iterator through `iter()`; an **iterator** gives values through `next()` and raises `StopIteration` at the end.
* A `for` loop is `iter()` once plus `next()` until `StopIteration`.
* Iterators and generators are **one-shot**; lists and `range` can be looped many times.
* A function with `yield` is a **generator function**: calling it creates a generator; each `next()` runs to the next `yield` and pauses.
* Generator expressions `(... for ...)` are lazy and use constant memory.
* Infinite generators are safe with `islice`, `break` or `next()`; `yield from` delegates to another iterable and makes recursive generators easy.
* Pipelines of small generators process large data one item at a time.
* `send`, `throw` and `close` exist for advanced use; put generator cleanup in `try/finally`.
* Debug by checking for exhaustion first, and use small test inputs or a `tap()` stage.

---

# Part 4: Closures and Decorators

**What these notes cover**

* Why functions are objects, and what that makes possible.
* How closures remember variables, and the late-binding trap.
* What a decorator really is: a function that takes a function and returns a function.
* The difference between **definition time** and **call time** (with a flowchart).
* How to write correct decorators with `*args`, `**kwargs`, `return` and `functools.wraps`.
* Decorators with arguments, stacking order, and class-based decorators.
* Practical decorators: timer, logging, retry, cache, registry, validation.
* `functools.cache` and `functools.lru_cache`, decorating methods, and how to debug decorators.

**Before you start:** Notebook 1, Part 11 (Functions: scope, closures, `*args`/`**kwargs`, higher-order functions). Notebook 2, Part 2 (`__call__`) helps for section 4.7.

## 4.1 Functions are objects

In Python a function is an object, like a number or a list. This is often called "functions are **first-class** objects". It means you can:

* assign a function to another name,
* pass a function as an argument,
* return a function from another function,
* store functions in lists and dicts,
* attach attributes to a function.

A function that takes or returns another function is a **higher-order function**. Decorators are built entirely on this idea.

In [160]:
# A function is an object with a type, a name and attributes
def greet(name):
    """Return a greeting."""
    return f"Hello, {name}"

say_hello = greet                 # a second name for the same function object
print(say_hello("Asha"))          # Hello, Asha
print(say_hello is greet)         # True
print(type(greet).__name__)       # function
print(greet.__name__)             # greet
print(greet.__doc__)              # Return a greeting.

Hello, Asha
True
function
greet
Return a greeting.


In [161]:
# Passing functions in, storing them in a dict, returning them
def apply_twice(func, value):
    return func(func(value))

print(apply_twice(lambda x: x * 3, 2))     # 18

operations = {"add": lambda a, b: a + b, "mul": lambda a, b: a * b}
print(operations["mul"](4, 5))             # 20

def make_adder(n):
    def add(x):
        return x + n
    return add                             # return the function, do not call it

add_ten = make_adder(10)
print(add_ten(5))                          # 15

18
20
15


> **Common mistake:** Writing `return add()` instead of `return add`. The first calls the function and returns its result; the second returns the function itself.

> **Key points**
> * Functions are objects: you can name, pass, return and store them.
> * `func` is the function; `func()` is a call to it.
> * Decorators use exactly these abilities.

---

## 4.2 Closures recap

A **closure** is an inner function that remembers variables from the function that created it, even after that outer function has returned. (First introduced in Notebook 1, Part 11.)

The remembered variables are called **free variables**. Python keeps them in "cells" attached to the inner function (`__closure__`).

```text
  make_multiplier(3) runs and returns:

   +-------------------------+        +-------------------+
   | function multiply       |------->| cell: factor = 3  |
   | (x) -> x * factor       |        +-------------------+
   +-------------------------+
     the cell keeps "factor" alive after make_multiplier ended
```

In [162]:
# A closure remembers "factor" after make_multiplier has finished
def make_multiplier(factor):
    def multiply(x):
        return x * factor
    return multiply

triple = make_multiplier(3)
double = make_multiplier(2)

print(triple(10), double(10))                    # 30 20
print(triple.__code__.co_freevars)               # ('factor',)
print(triple.__closure__[0].cell_contents)       # 3

30 20
('factor',)
3


### Changing a remembered variable: nonlocal

Reading a free variable works automatically. To **assign** to it, declare it `nonlocal`. Without `nonlocal`, the assignment creates a new local variable and you get `UnboundLocalError`.

In [163]:
# A counter that keeps private state in a closure
def make_counter():
    count = 0

    def increment():
        nonlocal count            # we assign to count, so declare it nonlocal
        count += 1
        return count

    return increment

counter = make_counter()
print(counter(), counter(), counter())   # 1 2 3

other = make_counter()                   # independent state
print(other())                           # 1

1 2 3
1


In [164]:
# Common mistake: forgetting nonlocal
def make_broken_counter():
    count = 0

    def increment():
        count += 1                       # assignment makes "count" local here
        return count

    return increment

try:
    make_broken_counter()()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)
# UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value


### The late-binding trap

A closure remembers the **variable**, not its value at the time the function was made. Functions created in a loop all see the **final** value of the loop variable.

In [165]:
# Common mistake: all three functions see the last value of i
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])          # [2, 2, 2]

# Fix 1: bind the current value as a default argument
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])          # [0, 1, 2]

# Fix 2: use a factory function (each call has its own variable)
def make_const(value):
    return lambda: value

funcs = [make_const(i) for i in range(3)]
print([f() for f in funcs])          # [0, 1, 2]

[2, 2, 2]
[0, 1, 2]
[0, 1, 2]


> **Key points**
> * A closure is an inner function plus the variables it remembers from the outer function.
> * Use `nonlocal` to assign to a remembered variable.
> * Closures capture variables, not values: watch out for loops (late binding).

---

### Practice 4.1: Functions as objects and closures

**A. Predict the output**

1. What does this print?
   ```python
   def outer():
       x = "outer"

       def inner():
           return x

       return inner

   f = outer()
   print(f())
   ```
2. What does this print?
   ```python
   def make():
       items = []

       def add(x):
           items.append(x)
           return len(items)

       return add

   a = make()
   b = make()
   print(a(1), a(2), b(3))
   ```
3. What does this print?
   ```python
   funcs = []
   for n in [1, 2, 3]:
       funcs.append(lambda: n * 10)
   print([f() for f in funcs])
   ```

**B. Write code**

4. Write `make_accumulator(start)` that returns a function `add(amount)`. Each call adds `amount` to a running total and returns the new total.
5. Write `compose(f, g)` that returns a new function `h(x) = f(g(x))`.
6. Write `make_rate_limiter(limit)` returning a function `allow()` that returns `True` for the first `limit` calls and `False` afterwards.
7. **[Hard]** Write `memo_counter()` that returns two functions `(record, report)`: `record(word)` counts a word, and `report()` returns a dict copy of the counts. Both must share the same private state.

**C. Find and fix the bug**

8. Fix the `UnboundLocalError`.
   ```python
   def make_total():
       total = 0

       def add(x):
           total = total + x
           return total

       return add
   ```
9. The buttons all print `button 3`. Fix it.
   ```python
   handlers = {}
   for i in range(1, 4):
       handlers[i] = lambda: f"button {i}"
   print(handlers[1]())
   ```

**D. Interview questions**

10. What is a closure? Give a practical use.
11. **[Hard]** Where does Python store the variables that a closure remembers, and why are they not lost when the outer function returns?

### Solutions 4.1

1. `outer`.
2. `1 2 1`. Each call to `make()` creates its own `items` list.
3. `[30, 30, 30]`. Late binding: all lambdas read `n` after the loop has finished.
4. to 7. See the code below.
8. Add `nonlocal total` at the start of `add`.
9. Bind the value at creation time: `lambda i=i: f"button {i}"`.
10. A closure is an inner function that remembers variables from its enclosing function. Uses: counters and other private state, function factories (`make_multiplier`), callbacks with configuration, and decorators.
11. In **cell objects** referenced by the inner function's `__closure__` tuple. The inner function keeps references to these cells, so the values stay alive as long as the inner function exists, even after the outer call's frame is gone.

In [166]:
# Solutions 4.1
# Q4
def make_accumulator(start):
    total = start

    def add(amount):
        nonlocal total
        total += amount
        return total

    return add

acc = make_accumulator(100)
assert (acc(10), acc(-30)) == (110, 80)


# Q5
def compose(f, g):
    def h(x):
        return f(g(x))
    return h

inc_then_double = compose(lambda x: x * 2, lambda x: x + 1)
assert inc_then_double(3) == 8


# Q6
def make_rate_limiter(limit):
    calls = 0

    def allow():
        nonlocal calls
        calls += 1
        return calls <= limit

    return allow

allow = make_rate_limiter(2)
assert [allow() for _ in range(4)] == [True, True, False, False]
assert make_rate_limiter(0)() is False


# Q7: two closures sharing one private dict
def memo_counter():
    counts = {}

    def record(word):
        counts[word] = counts.get(word, 0) + 1   # mutating, not rebinding: no nonlocal needed

    def report():
        return dict(counts)

    return record, report

record, report = memo_counter()
for w in ["data", "ai", "data"]:
    record(w)
snapshot = report()
snapshot["ai"] = 99                               # changing the copy does not change the state
assert report() == {"data": 2, "ai": 1}


# Q8: fixed
def make_total():
    total = 0

    def add(x):
        nonlocal total
        total = total + x
        return total

    return add

add = make_total()
assert (add(5), add(5)) == (5, 10)


# Q9: fixed
handlers = {}
for i in range(1, 4):
    handlers[i] = lambda i=i: f"button {i}"
assert handlers[1]() == "button 1" and handlers[3]() == "button 3"
print("Practice 4.1: all checks passed")

Practice 4.1: all checks passed


## 4.3 What a decorator is

A **decorator** is a function that takes a function and returns a function (usually a new "wrapper" that adds behaviour around the original).

```python
@my_decorator
def work():
    ...
```

is exactly the same as:

```python
def work():
    ...
work = my_decorator(work)
```

**In simple words:** a decorator is gift wrapping. The gift (your function) is unchanged, but whoever receives it first meets the wrapping (extra code that runs before and after).

Typical uses: logging, timing, caching, retrying, checking permissions, validating arguments, registering functions (web routes, plugins, test fixtures).

In [167]:
# Decorating by hand first, so there is no magic
def shout(func):
    def wrapper():
        result = func()
        return result.upper() + "!"
    return wrapper

def greet():
    return "hello"

greet = shout(greet)          # replace greet with the wrapper
print(greet())                # HELLO!
print(greet.__name__)         # wrapper  (fixed with functools.wraps in 4.4)

HELLO!
wrapper


In [168]:
# The same with @ syntax
def shout(func):
    def wrapper():
        result = func()
        return result.upper() + "!"
    return wrapper

@shout                        # means: greet = shout(greet)
def greet():
    return "hello"

print(greet())                # HELLO!

HELLO!


### Definition time vs call time

A decorator has two moments:

* **Definition time:** when Python executes the `def` line with `@decorator` above it (usually when the module is imported). The decorator function runs **once** and returns the wrapper.
* **Call time:** every time your code calls the function. Only the **wrapper** runs; inside it, the original function is called.

```text
  DEFINITION TIME (once, when "def" runs)
  +-------------------------+     +---------------------------+
  | def work(): ...         |---->| decorator(work) runs      |
  | (original function)     |     | builds and returns wrapper|
  +-------------------------+     +-------------+-------------+
                                                |
                                                v
                                  +---------------------------+
                                  | name "work" now points to |
                                  | the wrapper               |
                                  +---------------------------+

  CALL TIME (every call: work(5))
  +-------------------+    +---------------------+    +----------------+
  | caller: work(5)   |--->| wrapper(5)          |--->| original       |
  +-------------------+    | 1. code before      |    | work(5) runs   |
                           |                     |<---| returns result |
                           | 2. code after       |    +----------------+
                           | 3. return result    |
                           +----------+----------+
                                      |
                                      v
                             result back to caller
```

In [169]:
# Watch both moments happen
def trace(func):
    print(f"  [definition time] decorating {func.__name__}")
    def wrapper(x):
        print(f"  [call time] before {func.__name__}({x})")
        result = func(x)
        print(f"  [call time] after, result={result}")
        return result
    return wrapper

print("about to define square")
@trace
def square(x):
    return x * x
print("square is defined")

print(square(4))
print(square(5))
# about to define square
#   [definition time] decorating square
# square is defined
#   [call time] before square(4)
#   [call time] after, result=16
# 16
# ... the same three lines for 5 -> 25

about to define square
  [definition time] decorating square
square is defined
  [call time] before square(4)
  [call time] after, result=16
16
  [call time] before square(5)
  [call time] after, result=25
25


> **Note:** Because decorators run at definition time, a decorator that does slow work (or prints) in its outer body does that work on **import**. Keep the outer body light; put per-call work inside the wrapper.

> **Key points**
> * `@deco` above `def f` means `f = deco(f)`.
> * The decorator runs once at definition time; the wrapper runs on every call.
> * After decoration, the name points to the wrapper, which calls the original.

---

## 4.4 Writing decorators properly

A decorator that works for any function must follow four rules:

1. The wrapper accepts **any arguments**: `def wrapper(*args, **kwargs)`.
2. It passes them on: `func(*args, **kwargs)`.
3. It **returns** the original function's result.
4. It copies the original's name, docstring and other metadata with **`@functools.wraps(func)`**.

This is the template you should memorise:

```python
import functools

def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # code before
        result = func(*args, **kwargs)
        # code after
        return result
    return wrapper
```

In [170]:
# The template in action: a call logger
import functools

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        shown = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        result = func(*args, **kwargs)
        print(f"  {func.__name__}({', '.join(shown)}) -> {result!r}")
        return result
    return wrapper

@log_calls
def add(a, b):
    """Add two numbers."""
    return a + b

@log_calls
def greet(name, punctuation="."):
    return f"Hi {name}{punctuation}"

print(add(2, 3))                       #   add(2, 3) -> 5   then 5
print(greet("Ravi", punctuation="!"))  #   greet('Ravi', punctuation='!') -> 'Hi Ravi!'   then Hi Ravi!

  add(2, 3) -> 5
5
  greet('Ravi', punctuation='!') -> 'Hi Ravi!'
Hi Ravi!


In [171]:
# Why functools.wraps matters: without it, the metadata of the wrapper replaces yours
import functools

def plain(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def wrapped(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@plain
def area(w, h):
    """Return the area of a rectangle."""
    return w * h

@wrapped
def volume(w, h, d):
    """Return the volume of a box."""
    return w * h * d

print(area.__name__, "|", area.__doc__)        # wrapper | None
print(volume.__name__, "|", volume.__doc__)    # volume | Return the volume of a box.
print(volume.__wrapped__(1, 2, 3))             # 6  - wraps also stores the original

wrapper | None
volume | Return the volume of a box.
6


`functools.wraps` copies `__name__`, `__qualname__`, `__doc__`, `__module__`, `__annotations__` (and a few more), updates `__dict__`, and sets `__wrapped__` to the original function. Tools such as `help()`, `inspect.signature()`, debuggers, test frameworks and web frameworks rely on this information.

In [172]:
# inspect.signature follows __wrapped__, so help() shows the real parameters
import functools
import inspect

def wrapped(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@wrapped
def ship(order_id: int, express: bool = False) -> str:
    return f"order {order_id} shipped"

print(inspect.signature(ship))      # (order_id: int, express: bool = False) -> str

(order_id: int, express: bool = False) -> str


In [173]:
# Common mistake: forgetting "return result" makes every call return None
import functools

def broken_logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)          # result is thrown away
    return wrapper

@broken_logger
def total(prices):
    return sum(prices)

print(total([10, 20]))                 # None   <- the bug

# Fix: return the result of the original call
def fixed_logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@fixed_logger
def total(prices):
    return sum(prices)

print(total([10, 20]))                 # 30

None
30


> **Common mistake:** Writing `def wrapper(x)` instead of `def wrapper(*args, **kwargs)`. The decorator then only works for functions with exactly one positional argument.

> **Key points**
> * Wrapper: `*args, **kwargs` in, `func(*args, **kwargs)` inside, `return result` out.
> * Always use `@functools.wraps(func)` on the wrapper.
> * `__wrapped__` gives access to the undecorated function (handy in tests).

---

### Practice 4.2: Decorator basics

**A. Predict the output**

1. What does this print?
   ```python
   def deco(func):
       print("decorating")
       return func

   @deco
   def f():
       return 1

   print("defined")
   print(f())
   ```
2. What does this print?
   ```python
   def twice(func):
       def wrapper():
           func()
           func()
       return wrapper

   @twice
   def hello():
       print("hi")

   result = hello()
   print(result)
   ```
3. Without `functools.wraps`, what is `f.__name__` after decoration with a wrapper called `inner`?

**B. Write code**

4. Write a decorator `uppercase` that converts the string result of a function to upper case.
5. Write a decorator `count_calls` that counts how many times a function is called and stores the number in `wrapper.calls`.
6. Write a decorator `not_none` that raises `ValueError` if any positional or keyword argument is `None`.
7. **[Hard]** Write a decorator `debug` that prints the call, the result and the exception (if any), and re-raises the exception unchanged.

**C. Find and fix the bug**

8. This decorator breaks every function with arguments. Fix it.
   ```python
   def announce(func):
       def wrapper():
           print("calling", func.__name__)
           return func()
       return wrapper
   ```
9. `@timer` makes `help(report)` show `wrapper`. Fix it.
   ```python
   import time

   def timer(func):
       def wrapper(*args, **kwargs):
           start = time.perf_counter()
           result = func(*args, **kwargs)
           print(time.perf_counter() - start)
           return result
       return wrapper
   ```

**D. Interview questions**

10. Rewrite `@cache` above `def f(n): ...` without the `@` syntax.
11. **[Hard]** Explain the difference between definition time and call time for a decorator. Why does it matter?

### Solutions 4.2

1. `decorating`, then `defined`, then `1`. The decorator runs when `def` executes.
2. `hi`, `hi`, then `None`. The wrapper does not return anything.
3. `inner`.
4. to 7. See the code below.
8. Make the wrapper accept and pass on any arguments: `def wrapper(*args, **kwargs): ... return func(*args, **kwargs)`, and add `@functools.wraps(func)`.
9. Add `import functools` and put `@functools.wraps(func)` above `def wrapper`.
10. `def f(n): ...` then `f = cache(f)`.
11. Definition time: the decorator function runs once when the `def` statement runs (usually at import) and returns the wrapper. Call time: the wrapper runs on every call. It matters because setup code (registration, reading config) belongs in the outer body and runs once, while per-call work (timing, logging) must be inside the wrapper. Mixing them up gives work done once instead of per call, or slow imports.

In [174]:
# Solutions 4.2
import functools

# Q4
def uppercase(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@uppercase
def greet(name):
    return f"hello {name}"

assert greet("asha") == "HELLO ASHA" and greet.__name__ == "greet"


# Q5
def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def ping():
    return "pong"

ping()
ping()
assert ping.calls == 2


# Q6
def not_none(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for value in list(args) + list(kwargs.values()):
            if value is None:
                raise ValueError(f"{func.__name__}() got a None argument")
        return func(*args, **kwargs)
    return wrapper

@not_none
def full_name(first, last=""):
    return f"{first} {last}".strip()

assert full_name("Meera", last="Iyer") == "Meera Iyer"
try:
    full_name("Meera", last=None)
    raise AssertionError("expected ValueError")
except ValueError as e:
    assert "None argument" in str(e)


# Q7: report calls, results and exceptions, then re-raise
messages = []

def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        messages.append(f"call {func.__name__}{args}")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            messages.append(f"raised {type(e).__name__}: {e}")
            raise                         # re-raise the same exception
        messages.append(f"returned {result!r}")
        return result
    return wrapper

@debug
def divide(a, b):
    return a / b

assert divide(6, 3) == 2.0
try:
    divide(1, 0)
except ZeroDivisionError:
    pass
assert messages == [
    "call divide(6, 3)", "returned 2.0",
    "call divide(1, 0)", "raised ZeroDivisionError: division by zero",
]


# Q8: fixed
def announce(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@announce
def add(a, b):
    return a + b

assert add(1, b=2) == 3 and add.__name__ == "add"
print("Practice 4.2: all checks passed")

Practice 4.2: all checks passed


## 4.5 Decorators with arguments

Sometimes the decorator needs settings: `@repeat(3)`, `@retry(times=5)`, `@route("/home")`. Here `repeat(3)` is **called first**. Its result must be the real decorator, which is then applied to the function.

So you need **three** nested functions:

```text
  @repeat(3)                       hello = repeat(3)(hello)
  def hello(): ...

  +---------------------+  returns  +---------------------+  returns  +--------------------+
  | repeat(times)       |---------->| decorator(func)     |---------->| wrapper(*a, **kw)  |
  | factory: gets the   |           | gets the function   |           | gets the call's    |
  | decorator settings  |           |                     |           | arguments          |
  +---------------------+           +---------------------+           +--------------------+
     definition time                   definition time                    call time
```

In [175]:
# A decorator factory: factory -> decorator -> wrapper
import functools

def repeat(times):                          # 1. factory: receives the settings
    def decorator(func):                    # 2. decorator: receives the function
        @functools.wraps(func)
        def wrapper(*args, **kwargs):       # 3. wrapper: receives the call arguments
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(3)                                  # same as: roll = repeat(3)(roll)
def roll(name):
    return f"{name} rolled"

print(roll("Asha"))     # ['Asha rolled', 'Asha rolled', 'Asha rolled']

['Asha rolled', 'Asha rolled', 'Asha rolled']


In [176]:
# Another example: prefix output with a configurable tag
import functools

def tag(label, upper=False):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            text = func(*args, **kwargs)
            if upper:
                text = text.upper()
            return f"[{label}] {text}"
        return wrapper
    return decorator

@tag("INFO")
def status():
    return "service is running"

@tag("WARN", upper=True)
def disk():
    return "disk almost full"

print(status())     # [INFO] service is running
print(disk())       # [WARN] DISK ALMOST FULL

[INFO] service is running
[WARN] DISK ALMOST FULL


### A decorator that works with and without arguments

Sometimes you want both `@tag` and `@tag(label="X")` to work. The trick: the first parameter is the function (or `None`), and the settings are **keyword-only**.

In [177]:
# Optional arguments: works as @shout and as @shout(mark="?")
import functools

def shout(func=None, *, mark="!"):
    if func is None:                      # called as @shout(...): return the real decorator
        return lambda f: shout(f, mark=mark)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper() + mark
    return wrapper

@shout
def hello():
    return "hello"

@shout(mark="?")
def really():
    return "really"

print(hello(), really())     # HELLO! REALLY?

HELLO! REALLY?


> **Common mistake:** Writing `@repeat` without parentheses for a decorator that needs arguments. The function is passed as `times`, and calling the "decorated" function then fails (see section 4.11).

> **Key points**
> * `@factory(args)` means `func = factory(args)(func)`.
> * Three levels: factory (settings) -> decorator (function) -> wrapper (call arguments).
> * The optional-arguments pattern uses `func=None` plus keyword-only settings.

---

## 4.6 Stacking decorators

You can put several decorators on one function. They are **applied bottom-up** (the one closest to `def` first), so at call time the **top** decorator's wrapper runs first.

```python
@bold
@italic
def text():
    return "hi"
```

means `text = bold(italic(text))`.

```text
  call text()
     |
     v
  +--------------------------------------------+
  | bold wrapper   (outermost, runs first)     |
  |   +------------------------------------+   |
  |   | italic wrapper                     |   |
  |   |   +----------------------------+   |   |
  |   |   | original text() -> "hi"    |   |   |
  |   |   +----------------------------+   |   |
  |   |   returns "<i>hi</i>"              |   |
  |   +------------------------------------+   |
  |   returns "<b><i>hi</i></b>"               |
  +--------------------------------------------+
```

In [178]:
# Stacked decorators: applied bottom-up, run top-down
import functools

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<b>" + func(*args, **kwargs) + "</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<i>" + func(*args, **kwargs) + "</i>"
    return wrapper

@bold
@italic
def text():
    return "hi"

@italic
@bold
def text_swapped():
    return "hi"

print(text())           # <b><i>hi</i></b>
print(text_swapped())   # <i><b>hi</b></i>

<b><i>hi</i></b>
<i><b>hi</b></i>


In [179]:
# Order matters for behaviour, not only for output
import functools

calls = []

def log(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        calls.append(func.__name__)
        return func(*args, **kwargs)
    return wrapper

@functools.cache        # outer: a cache hit returns early, so log does not run
@log
def cached_then_logged(n):
    return n * 2

cached_then_logged(5)
cached_then_logged(5)
print(calls)            # ['cached_then_logged']  - logged only once

['cached_then_logged']


> **Tip:** Read a stack of decorators from the bottom up to know how it was built, and from the top down to know what happens first when the function is called.

> **Key points**
> * `@a @b def f` means `f = a(b(f))`.
> * The top decorator's wrapper runs first at call time.
> * Order changes behaviour: for example caching above logging hides repeated calls from the log.

---

### Practice 4.3: Decorator factories and stacking

**A. Predict the output**

1. What does this print?
   ```python
   def add_suffix(suffix):
       def decorator(func):
           def wrapper():
               return func() + suffix
           return wrapper
       return decorator

   @add_suffix("!")
   @add_suffix("?")
   def word():
       return "ok"

   print(word())
   ```
2. In what order are the lines printed?
   ```python
   def deco(name):
       print("factory", name)
       def decorator(func):
           print("decorate", name)
           return func
       return decorator

   @deco("A")
   @deco("B")
   def f():
       pass
   ```

**B. Write code**

3. Write `repeat(times)` that calls the function `times` times and returns the **last** result.
4. Write `clamp(low, high)` that limits a numeric result to the range `[low, high]`.
5. Write `default_on_error(value, exceptions=(Exception,))` that returns `value` when one of `exceptions` is raised.
6. **[Hard]** Write `prefix` that works as `@prefix` (adds `">> "`) and as `@prefix(text="## ")`.

**C. Find and fix the bug**

7. Calling `hello()` raises a `TypeError`. Why? Fix it.
   ```python
   def repeat(times):
       def decorator(func):
           def wrapper(*args, **kwargs):
               for _ in range(times):
                   func(*args, **kwargs)
           return wrapper
       return decorator

   @repeat
   def hello():
       print("hi")
   ```

**D. Interview questions**

8. Rewrite `@retry(times=3)` above `def fetch(): ...` without the `@` syntax.
9. **[Hard]** You stack `@login_required` and `@cache` on a view function. Which order is correct and why?

### Solutions 4.3

1. `ok?!`. The bottom decorator adds `?` first; the top one adds `!` to that.
2. `factory A`, `factory B`, `decorate B`, `decorate A`. Decorator expressions are evaluated top-down, then applied bottom-up.
3. to 6. See the code below.
7. `@repeat` passes `hello` as `times`, so `hello` becomes the inner `decorator` function. Calling `hello()` then fails with `TypeError: ... missing 1 required positional argument: 'func'`. Use `@repeat(2)`.
8. `def fetch(): ...` then `fetch = retry(times=3)(fetch)`.
9. `@login_required` on top, `@cache` below. The permission check must run on every call; if `@cache` were on top, a cached response could be returned to a user who is not logged in.

In [180]:
# Solutions 4.3
import functools

# Q1 check
def add_suffix(suffix):
    def decorator(func):
        @functools.wraps(func)
        def wrapper():
            return func() + suffix
        return wrapper
    return decorator

@add_suffix("!")
@add_suffix("?")
def word():
    return "ok"

assert word() == "ok?!"


# Q3
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

counter = {"n": 0}

@repeat(3)
def bump():
    counter["n"] += 1
    return counter["n"]

assert bump() == 3 and counter["n"] == 3


# Q4
def clamp(low, high):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return max(low, min(high, func(*args, **kwargs)))
        return wrapper
    return decorator

@clamp(0, 100)
def percent(part, whole):
    return part / whole * 100

assert percent(1, 2) == 50.0
assert percent(3, 2) == 100
assert percent(-1, 2) == 0


# Q5
def default_on_error(value, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            try:
                return func(*args, **kwargs)
            except exceptions:
                return value
        return wrapper
    return decorator

@default_on_error(0, (ValueError,))
def parse_qty(text):
    return int(text)

assert parse_qty("7") == 7 and parse_qty("seven") == 0
try:
    parse_qty(None)                 # TypeError is not in the list, so it is NOT hidden
    raise AssertionError("expected TypeError")
except TypeError:
    pass


# Q6
def prefix(func=None, *, text=">> "):
    if func is None:
        return lambda f: prefix(f, text=text)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return text + func(*args, **kwargs)
    return wrapper

@prefix
def a():
    return "plain"

@prefix(text="## ")
def b():
    return "title"

assert (a(), b()) == (">> plain", "## title")
assert a.__name__ == "a"
print("Practice 4.3: all checks passed")

Practice 4.3: all checks passed


## 4.7 Class-based decorators and decorators with state

A decorator only has to be **callable**. A class is callable (calling it creates an instance), and an instance with `__call__` is callable too. So a class can be a decorator:

* `__init__(self, func)` runs at definition time and stores the function.
* `__call__(self, *args, **kwargs)` runs at call time.

Classes are a good choice when the decorator keeps **state** (counters, caches, settings) that you want to inspect or reset.

In [181]:
# A class-based decorator that counts calls
import functools

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)   # like @wraps, for a class instance
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

@CountCalls                                     # ping = CountCalls(ping)
def ping():
    return "pong"

ping()
ping()
print(ping.calls, ping.__name__)                # 2 ping
print(type(ping).__name__)                      # CountCalls

2 ping
CountCalls


In [182]:
# A class-based decorator WITH arguments: __init__ takes the settings, __call__ takes the function
import functools

class Limit:
    def __init__(self, max_calls):
        self.max_calls = max_calls

    def __call__(self, func):
        calls = 0

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal calls
            if calls >= self.max_calls:
                raise RuntimeError(f"{func.__name__} called more than {self.max_calls} times")
            calls += 1
            return func(*args, **kwargs)
        return wrapper

@Limit(2)
def send_otp(phone):
    return f"OTP sent to {phone}"

print(send_otp("98xxxx01"))       # OTP sent to 98xxxx01
print(send_otp("98xxxx01"))       # OTP sent to 98xxxx01
try:
    send_otp("98xxxx01")
except RuntimeError as e:
    print("RuntimeError:", e)     # RuntimeError: send_otp called more than 2 times

OTP sent to 98xxxx01
OTP sent to 98xxxx01
RuntimeError: send_otp called more than 2 times


**Three ways to keep state in a decorator**

| Way | Example | Good for |
|---|---|---|
| Closure variable with `nonlocal` | `calls += 1` | Private state |
| Attribute on the wrapper function | `wrapper.calls += 1` | State the caller may read |
| Class-based decorator | `self.calls += 1` | Rich state, extra methods (`reset()`) |

> **Warning:** A plain class-based decorator (with `__call__` only) does **not** work correctly on methods, because instances of your class do not bind `self` like functions do. For methods, prefer a function-based decorator (section 4.10).

> **Key points**
> * Any callable can be a decorator; a class with `__call__` is one option.
> * Use `functools.update_wrapper(self, func)` in a class-based decorator.
> * Choose closures for simple state, classes when you need methods or inspection.

---

## 4.8 Practical decorators

These are the decorators you will meet in real projects and in interviews. Each one follows the template from section 4.4.

### Timer

In [183]:
# Timer: measure how long a call takes (store it instead of printing exact numbers)
import functools
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:                                   # record the time even if func raises
            wrapper.last_seconds = time.perf_counter() - start
    wrapper.last_seconds = None
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))                         # 499999500000
print("time recorded:", slow_sum.last_seconds > 0) # time recorded: True  (the exact value varies)

499999500000
time recorded: True


### Retry

A **retry** decorator calls the function again when a temporary error occurs (network timeouts, busy databases). A real one waits between attempts (often with **exponential backoff**: 1s, 2s, 4s ...). Here the wait is tiny so the cell stays fast.

In [184]:
# Retry: call again on selected exceptions, re-raise after the last attempt
import functools
import time

def retry(times=3, exceptions=(Exception,), delay=0.0, backoff=2.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            wait = delay
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == times:
                        raise                       # give up: let the caller see the error
                    print(f"  attempt {attempt} failed ({e}); retrying")
                    time.sleep(wait)
                    wait *= backoff
        return wrapper
    return decorator

attempts = {"count": 0}

@retry(times=3, exceptions=(ConnectionError,), delay=0.001)
def fetch_rates():
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ConnectionError("server busy")
    return {"USD": 1.0, "INR": 88.2}

print(fetch_rates())
#   attempt 1 failed (server busy); retrying
#   attempt 2 failed (server busy); retrying
# {'USD': 1.0, 'INR': 88.2}

  attempt 1 failed (server busy); retrying
  attempt 2 failed (server busy); retrying
{'USD': 1.0, 'INR': 88.2}


> **Warning:** Only retry errors that are **temporary** and operations that are safe to repeat (**idempotent**: doing them twice has the same effect as once). Retrying "charge the customer" can charge twice.

### Memoisation (cache)

**Memoisation** means remembering the results of earlier calls so that the same input is not computed twice. It needs hashable arguments, because they become dict keys.

In [185]:
# A hand-written cache decorator (functools.cache does this for you, see 4.9)
import functools

def memoize(func):
    cache = {}

    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    wrapper.cache = cache                  # expose it for inspection
    return wrapper

@memoize
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))              # 23416728348467685
print(len(fib.cache))       # 81  - each n from 0 to 80 computed once

23416728348467685
81


### Registry

A **registry** decorator records the function somewhere and returns it **unchanged**. Web frameworks use this for routes (`@app.get("/users")`), and plugin systems for handlers.

In [186]:
# Registry: map command names to functions
COMMANDS = {}

def command(name):
    def decorator(func):
        COMMANDS[name] = func       # register at definition time
        return func                 # no wrapper needed
    return decorator

@command("add")
def add_numbers(a, b):
    return a + b

@command("mul")
def multiply(a, b):
    return a * b

def run(name, *args):
    if name not in COMMANDS:
        return f"unknown command: {name}"
    return COMMANDS[name](*args)

print(sorted(COMMANDS))       # ['add', 'mul']
print(run("mul", 6, 7))       # 42
print(run("div", 1, 2))       # unknown command: div

['add', 'mul']
42
unknown command: div


### Validation and access control

In [187]:
# Validation: check argument types from the function's annotations
import functools
import inspect

def check_types(func):
    sig = inspect.signature(func)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)          # match arguments to parameter names
        for name, value in bound.arguments.items():
            expected = func.__annotations__.get(name)
            if isinstance(expected, type) and not isinstance(value, expected):
                raise TypeError(f"{name} must be {expected.__name__}, got {type(value).__name__}")
        return func(*args, **kwargs)
    return wrapper

@check_types
def add_stock(item: str, quantity: int) -> str:
    return f"{quantity} x {item}"

print(add_stock("pen", 5))            # 5 x pen
try:
    add_stock("pen", "five")
except TypeError as e:
    print("TypeError:", e)            # TypeError: quantity must be int, got str

5 x pen
TypeError: quantity must be int, got str


In [188]:
# Access control: only allow users with a given role
import functools

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(user, *args, **kwargs):
            if role not in user.get("roles", []):
                raise PermissionError(f"{user['name']} needs role '{role}'")
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@require_role("admin")
def delete_order(user, order_id):
    return f"order {order_id} deleted by {user['name']}"

asha = {"name": "Asha", "roles": ["admin"]}
ravi = {"name": "Ravi", "roles": ["viewer"]}

print(delete_order(asha, 17))         # order 17 deleted by Asha
try:
    delete_order(ravi, 17)
except PermissionError as e:
    print("PermissionError:", e)      # PermissionError: Ravi needs role 'admin'

order 17 deleted by Asha
PermissionError: Ravi needs role 'admin'


### Deprecation warning

In [189]:
# Mark old functions as deprecated (warnings are captured here so the output is predictable)
import functools
import warnings

def deprecated(reason):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            warnings.warn(f"{func.__name__} is deprecated: {reason}",
                          DeprecationWarning, stacklevel=2)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@deprecated("use total_price() instead")
def old_total(prices):
    return sum(prices)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print(old_total([1, 2]))                  # 3
print(caught[0].category.__name__)            # DeprecationWarning
print(caught[0].message)                      # old_total is deprecated: use total_price() instead

3
DeprecationWarning
old_total is deprecated: use total_price() instead


> **Note:** Python 3.13 added a standard `warnings.deprecated` decorator (PEP 702). It also lets type checkers warn about calls to deprecated functions. Use it when you can require 3.13+.

> **Key points**
> * Timer, retry, cache, registry, validation and access control all follow the same template.
> * A registry decorator returns the original function; it does not need a wrapper.
> * Retry only temporary errors and idempotent operations; always re-raise after the last attempt.

---

## 4.9 functools.cache and functools.lru_cache

The standard library already has a high-quality memoisation decorator:

* `@functools.lru_cache(maxsize=128)` keeps at most `maxsize` results. When full, it removes the **least recently used** entry (LRU).
* `@functools.cache` (Python 3.9+) is `lru_cache(maxsize=None)`: unlimited size, slightly faster.
* `func.cache_info()` shows hits, misses and size; `func.cache_clear()` empties the cache.

The arguments must be **hashable** (they are used as dict keys).

In [190]:
# functools.cache turns exponential recursion into linear time
import functools

@functools.cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(100))            # 354224848179261915075
print(fib.cache_info())    # CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)

fib.cache_clear()
print(fib.cache_info().currsize)   # 0

354224848179261915075
CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)
0


In [191]:
# lru_cache with a small maxsize: the least recently used entry is dropped
import functools

@functools.lru_cache(maxsize=2)
def price_of(item):
    print(f"  looking up {item}")
    return {"pen": 10, "book": 250, "bag": 90}[item]

price_of("pen")      #   looking up pen
price_of("book")     #   looking up book
price_of("pen")      # cache hit (pen is now the most recently used)
price_of("bag")      #   looking up bag   -> "book" is evicted
price_of("book")     #   looking up book  (it had been dropped)
print(price_of.cache_info())   # CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)

  looking up pen
  looking up book
  looking up bag
  looking up book
CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)


In [192]:
# Arguments must be hashable
import functools

@functools.cache
def total(items):
    return sum(items)

print(total((1, 2, 3)))         # 6  - a tuple is hashable
try:
    total([1, 2, 3])
except TypeError as e:
    print("TypeError:", e)      # TypeError: unhashable type: 'list'

6
TypeError: unhashable type: 'list'


> **Warning:** Do not put `@lru_cache` or `@cache` directly on **instance methods** of long-lived programs. `self` becomes part of the cache key, so the cache keeps every instance alive (a memory leak), and stale results remain after the object changes. Use `functools.cached_property` for per-instance computed values (Part 7), or cache a module-level function.

> **Warning:** A cached function should be **pure**: same input, same output, no side effects. Caching a function that reads the clock or a database returns old data.

> **Interview tip:** `@cache` is the fastest way to turn a recursive solution into top-down dynamic programming (Notebook 3, Part 16). Mention that the recursion depth limit (about 1000) still applies.

> **Key points**
> * `@cache` = unlimited memoisation; `@lru_cache(maxsize=n)` = bounded with LRU eviction.
> * Arguments must be hashable; results should depend only on the arguments.
> * `cache_info()` and `cache_clear()` help with testing and debugging.

---

## 4.10 Decorating methods and the built-in decorators

### Function decorators work on methods

A method is just a function whose first argument is `self`. A wrapper with `*args, **kwargs` receives `self` as `args[0]` and passes it on, so normal decorators work on methods.

In [193]:
# A function-based decorator on a method: self is simply the first positional argument
import functools

def log_method(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"  {type(self).__name__}.{func.__name__}({', '.join(map(repr, args))})")
        return func(self, *args, **kwargs)
    return wrapper

class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    @log_method
    def deposit(self, amount):
        self.balance += amount
        return self.balance

acct = Account("Meera")
print(acct.deposit(500))      #   Account.deposit(500)   then 500

  Account.deposit(500)
500


### The built-in method decorators

You have already used three decorators from the language itself (Notebook 2, Parts 1 and 2):

| Decorator | What it does |
|---|---|
| `@property` | Turns a method into a computed attribute (getter, setter, deleter) |
| `@classmethod` | First argument is the class (`cls`), not the instance |
| `@staticmethod` | No automatic first argument |
| `@functools.cached_property` | Computes once per instance, then stores the value |
| `@abc.abstractmethod` | Subclasses must implement the method |
| `@typing.override` (3.12+) | Tells type checkers the method overrides a parent method |

When you combine your own decorator with `@classmethod` or `@staticmethod`, put **`@classmethod`/`@staticmethod` on top** (outermost), so your decorator wraps the plain function.

In [194]:
# Combining a custom decorator with @classmethod: classmethod goes on top
import functools

def log_method(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"  calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod          # outermost
    @log_method           # wraps the plain function
    def from_fahrenheit(cls, f):
        return cls(round((f - 32) * 5 / 9, 1))

t = Temperature.from_fahrenheit(98.6)
print(t.celsius)          #   calling from_fahrenheit   then 37.0

  calling from_fahrenheit
37.0


### Class decorators

A decorator can also take a **class** and return a class (often the same one, modified). `@dataclass` and `@functools.total_ordering` are class decorators. You will write your own in Part 6.

In [195]:
# A tiny class decorator: add a describe() method and register the class
REGISTERED = []

def register_class(cls):
    def describe(self):
        return f"{cls.__name__} with {sorted(vars(self))}"
    cls.describe = describe
    REGISTERED.append(cls.__name__)
    return cls

@register_class
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

print(Product("pen", 10).describe())   # Product with ['name', 'price']
print(REGISTERED)                      # ['Product']

Product with ['name', 'price']
['Product']


> **Key points**
> * A wrapper with `*args, **kwargs` works on methods; `self` is the first argument.
> * Put `@classmethod`/`@staticmethod`/`@property` outermost when combining with your own decorators.
> * Class decorators receive a class and return a class.

---

### Practice 4.4: Stateful and practical decorators

**A. Predict the output**

1. What does this print?
   ```python
   import functools

   @functools.lru_cache(maxsize=None)
   def sq(n):
       return n * n

   sq(3)
   sq(3)
   sq(4)
   print(sq.cache_info().hits, sq.cache_info().misses)
   ```
2. What does this print?
   ```python
   REG = {}

   def register(func):
       REG[func.__name__] = func
       return func

   @register
   def a():
       return "A"

   print(a(), list(REG))
   ```

**B. Write code**

3. Write a class-based decorator `CallHistory` that stores the arguments of every call in `self.history` (as `(args, kwargs)` pairs) and has a `reset()` method.
4. Write `retry_on(exceptions, times)` (no sleeping) and test it with a function that fails twice with `TimeoutError` and then succeeds.
5. Write `validate_positive` that raises `ValueError` if any numeric positional argument is negative.
6. **[Hard]** Write `rate_limited(max_calls, clock)` where `clock` is a function returning the current time in seconds. Allow at most `max_calls` calls in any 1-second window; otherwise raise `RuntimeError`. Test it with a fake clock (no real waiting).

**C. Find and fix the bug**

7. The cache never hits for `get_user(["Asha"])` and actually raises an error. Why? Fix it.
   ```python
   import functools

   @functools.cache
   def get_user(names):
       return [n.upper() for n in names]
   ```
8. This logs "calling" for `from_string`, but `from_string("x")` fails. Fix the order.
   ```python
   class User:
       @log_method
       @classmethod
       def from_string(cls, text):
           return cls()
   ```
   (`log_method` is a normal function-based decorator.)

**D. Interview questions**

9. What is the difference between `functools.cache` and `functools.lru_cache`?
10. **[Hard]** Why is putting `@lru_cache` on an instance method risky?

### Solutions 4.4

1. `1 2`. The second `sq(3)` is a hit; `sq(3)` and `sq(4)` the first time are misses.
2. `A ['a']`. `register` returns the original function unchanged.
3. to 6. See the code below.
7. A list is unhashable, so the cache raises `TypeError: unhashable type: 'list'`. Pass a tuple instead (`get_user(("Asha",))`), or convert inside a thin wrapper before calling the cached function.
8. `@classmethod` must be on top. Otherwise `log_method` receives a `classmethod` object instead of a plain function. Its wrapper then tries to call that object directly, which fails with `TypeError: 'classmethod' object is not callable`. Swap the two lines.
9. `cache` has no size limit (`lru_cache(maxsize=None)`) and is a little faster. `lru_cache(maxsize=n)` keeps at most `n` results and evicts the least recently used one.
10. `self` becomes part of the key, so the cache holds a strong reference to every instance ever used (they are never freed), and results may be stale if the instance's data changes. Use `cached_property` or a module-level cached function instead.

In [196]:
# Solutions 4.4
import functools

# Q3
class CallHistory:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.history = []

    def __call__(self, *args, **kwargs):
        self.history.append((args, kwargs))
        return self.func(*args, **kwargs)

    def reset(self):
        self.history.clear()

@CallHistory
def greet(name, polite=False):
    return f"Dear {name}" if polite else f"Hi {name}"

greet("Asha")
greet("Ravi", polite=True)
assert greet.history == [(("Asha",), {}), (("Ravi",), {"polite": True})]
greet.reset()
assert greet.history == [] and greet.__name__ == "greet"


# Q4
def retry_on(exceptions, times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
        return wrapper
    return decorator

state = {"calls": 0}

@retry_on((TimeoutError,), times=3)
def flaky():
    state["calls"] += 1
    if state["calls"] < 3:
        raise TimeoutError("slow")
    return "ok"

assert flaky() == "ok" and state["calls"] == 3

@retry_on((TimeoutError,), times=2)
def always_fails():
    raise TimeoutError("down")

try:
    always_fails()
    raise AssertionError("expected TimeoutError")
except TimeoutError:
    pass


# Q5
def validate_positive(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for value in args:
            if isinstance(value, (int, float)) and value < 0:
                raise ValueError(f"negative argument: {value}")
        return func(*args, **kwargs)
    return wrapper

@validate_positive
def area(w, h):
    return w * h

assert area(2, 3) == 6
try:
    area(-2, 3)
    raise AssertionError("expected ValueError")
except ValueError as e:
    assert "negative" in str(e)


# Q6: sliding 1-second window with a fake clock
from collections import deque

def rate_limited(max_calls, clock):
    def decorator(func):
        recent = deque()

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            now = clock()
            while recent and now - recent[0] >= 1.0:
                recent.popleft()                 # forget calls older than 1 second
            if len(recent) >= max_calls:
                raise RuntimeError("rate limit exceeded")
            recent.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

fake_time = [0.0]

@rate_limited(2, clock=lambda: fake_time[0])
def api():
    return "ok"

assert api() == "ok" and api() == "ok"
try:
    api()
    raise AssertionError("expected RuntimeError")
except RuntimeError:
    pass
fake_time[0] = 1.5                                # time moves on
assert api() == "ok"


# Q7: fixed (tuple argument)
@functools.cache
def get_user(names):
    return [n.upper() for n in names]

assert get_user(("Asha",)) == ["ASHA"]
print("Practice 4.4: all checks passed")

Practice 4.4: all checks passed


## 4.11 How to debug decorators

Decorator bugs are confusing because the code that fails is not the code you see. Learn the typical symptoms:

| Symptom or error | Likely cause | Fix |
|---|---|---|
| Decorated function returns `None` | Wrapper does not `return` the result | `return func(*args, **kwargs)` |
| `TypeError: wrapper() takes 0 positional arguments but 1 was given` | Wrapper has a fixed signature | Use `*args, **kwargs` |
| `TypeError: ... decorator() missing 1 required positional argument: 'func'` | `@factory` used without `()` | Write `@factory(...)` |
| `TypeError: 'NoneType' object is not callable` | Decorator forgot `return wrapper` | Return the wrapper |
| `__name__` is `wrapper`; tracebacks and logs show "wrapper" | Missing `functools.wraps` | Add `@functools.wraps(func)` |
| `TypeError: unhashable type: 'list'` | List argument to a cached function | Pass tuples / frozensets |
| Behaviour order is wrong | Stacking order | Remember `a(b(f))`: the top runs first |

In [197]:
# Reproduce three classic decorator errors
import functools

def no_return_wrapper(func):            # Common mistake: forgot "return wrapper"
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

@no_return_wrapper
def report():
    return "ok"

try:
    report()
except TypeError as e:
    print("1.", type(e).__name__ + ":", e)    # 1. TypeError: 'NoneType' object is not callable


def fixed_signature(func):              # Common mistake: wrapper takes no arguments
    def wrapper():
        return func()
    return wrapper

@fixed_signature
def double(x):
    return 2 * x

try:
    double(4)
except TypeError as e:
    print("2.", type(e).__name__ + ":", e)
# 2. TypeError: fixed_signature.<locals>.wrapper() takes 0 positional arguments but 1 was given


def repeat(times):                      # Common mistake: used without parentheses
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorator

@repeat
def hello():
    return "hi"

try:
    hello()
except TypeError as e:
    print("3.", type(e).__name__ + ":", e)
# 3. TypeError: repeat.<locals>.decorator() missing 1 required positional argument: 'func'

1. TypeError: 'NoneType' object is not callable
2. TypeError: fixed_signature.<locals>.wrapper() takes 0 positional arguments but 1 was given
3. TypeError: repeat.<locals>.decorator() missing 1 required positional argument: 'func'


### Debugging techniques

1. **Read the traceback bottom-up.** Lines inside `wrapper` show which decorator is involved. With `functools.wraps`, the error message names your real function.
2. **Test the undecorated function** through `func.__wrapped__`. If it works, the bug is in the decorator.
3. **Apply the decorator by hand** in a small test: `f = deco(original)`, then inspect `f`, `type(f)`, `f.__name__`.
4. **Remove decorators one at a time** from a stack to find which one changes the behaviour.
5. In a debugger, **step into** the call: you land in the wrapper first, then step into `func(*args, **kwargs)` to reach your code.

In [198]:
# Technique: compare the decorated and the undecorated function
import functools

def round_result(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return round(func(*args, **kwargs))      # suspicious: rounds to an int
    return wrapper

@round_result
def average(values):
    return sum(values) / len(values)

print(average([1, 2, 2]))               # 2      <- unexpected?
print(average.__wrapped__([1, 2, 2]))   # 1.6666666666666667  -> the decorator changes it

2
1.6666666666666667


```text
        +-------------------------------+
        | Decorated function misbehaves |
        +---------------+---------------+
                        |
                        v
        +-------------------------------+  works  +----------------------+
        | Call f.__wrapped__(...)       |-------->| Bug is in the        |
        | (the original function)       |         | decorator: check     |
        +---------------+---------------+         | return, args, wraps  |
                        | fails                   +----------------------+
                        v
        +-------------------------------+
        | Bug is in your function:      |
        | debug it as normal code       |
        +---------------+---------------+
                        |
                        v
        +-------------------------------+
        | Fix, re-run, add a test for   |
        | both the decorator and f      |
        +-------------------------------+
```

> **Tip:** Write tests for decorators on a tiny dummy function (for example `lambda x: x`). Then you test the decorator alone, without the noise of business logic.

> **Key points**
> * Most decorator bugs are: missing `return`, fixed wrapper signature, missing `()` on a factory, missing `wraps`.
> * `__wrapped__` lets you separate decorator bugs from function bugs.
> * Remove or reorder decorators one at a time to isolate the problem.

---

### Practice 4.5: Debugging and review

**A. Predict the output**

1. What does this print?
   ```python
   def deco(func):
       def wrapper(*args):
           func(*args)
       return wrapper

   @deco
   def add(a, b):
       return a + b

   print(add(1, 2))
   ```
2. What does this print?
   ```python
   import functools

   def deco(func):
       @functools.wraps(func)
       def wrapper():
           return func()
       return wrapper

   @deco
   def task():
       "Run the task."
       return 1

   print(task.__name__, task.__doc__, task.__wrapped__())
   ```

**B. Write code**

3. Write `once(func)`: the wrapped function runs only on the first call; later calls return the first result without running the function again.
4. Write `trace_exceptions(log)` that appends `"<name>: <ExceptionType>"` to the list `log` when the function raises, and re-raises.
5. **[Hard]** Write `typed_cache(func)` that caches results but treats `1` and `1.0` as different keys (hint: include `type(arg)` in the key). Compare with `functools.lru_cache(typed=True)`.

**C. Find and fix the bug**

6. Fix this decorator so that it works and keeps the metadata.
   ```python
   def require_positive(func):
       def wrapper(x):
           if x <= 0:
               raise ValueError("x must be positive")
           func(x)
   ```

**D. Interview questions**

7. Name four real uses of decorators.
8. **[Hard]** How would you unit-test a decorator?

### Solutions 4.5

1. `None`. The wrapper calls `func` but does not return its result.
2. `task Run the task. 1`.
3. to 5. See the code below.
6. Three bugs: the wrapper's result is not returned, `return wrapper` is missing, and `functools.wraps` is missing. See the fixed version below.
7. Logging and tracing, timing, caching, retrying, access control (login required), input validation, registering routes or plugins, deprecation warnings, rate limiting.
8. Apply the decorator to small dummy functions and assert: the result passes through, arguments are forwarded (positional and keyword), metadata is kept (`__name__`, `__doc__`, `__wrapped__`), exceptions behave as intended, and the extra behaviour happens (counter increases, log entry added). Use fake clocks or mocks instead of real time or network.

In [199]:
# Solutions 4.5
import functools

# Q3
def once(func):
    done = False
    result = None

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal done, result
        if not done:
            result = func(*args, **kwargs)
            done = True
        return result
    return wrapper

runs = []

@once
def init_db(url):
    runs.append(url)
    return f"connected to {url}"

assert init_db("db1") == "connected to db1"
assert init_db("db2") == "connected to db1"      # not run again
assert runs == ["db1"]


# Q4
def trace_exceptions(log):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                log.append(f"{func.__name__}: {type(e).__name__}")
                raise
        return wrapper
    return decorator

errors = []

@trace_exceptions(errors)
def parse(text):
    return int(text)

assert parse("5") == 5
try:
    parse("five")
except ValueError:
    pass
assert errors == ["parse: ValueError"]


# Q5
def typed_cache(func):
    cache = {}

    @functools.wraps(func)
    def wrapper(*args):
        key = tuple((type(a), a) for a in args)
        if key not in cache:
            cache[key] = func(*args)
        return cache[key]

    wrapper.cache = cache
    return wrapper

@typed_cache
def describe(x):
    return f"{type(x).__name__}:{x}"

assert describe(1) == "int:1" and describe(1.0) == "float:1.0"
assert len(describe.cache) == 2

@functools.lru_cache(typed=True)
def describe2(x):
    return f"{type(x).__name__}:{x}"

assert describe2(1) == "int:1" and describe2(1.0) == "float:1.0"


# Q6: fixed
def require_positive(func):
    @functools.wraps(func)
    def wrapper(x):
        if x <= 0:
            raise ValueError("x must be positive")
        return func(x)
    return wrapper

@require_positive
def sqrt(x):
    return x ** 0.5

assert sqrt(9) == 3.0 and sqrt.__name__ == "sqrt"
print("Practice 4.5: all checks passed")

Practice 4.5: all checks passed


### Summary of Part 4

* Functions are objects: they can be assigned, passed, returned and stored.
* A **closure** remembers variables of its enclosing function; use `nonlocal` to change them, and watch for late binding in loops.
* `@deco` above `def f` means `f = deco(f)`. The decorator runs once at definition time; the wrapper runs at every call.
* The correct template: `@functools.wraps(func)`, `def wrapper(*args, **kwargs)`, `return func(*args, **kwargs)`.
* Decorators with arguments need three levels: factory -> decorator -> wrapper.
* Stacked decorators are applied bottom-up and run top-down; order changes behaviour.
* Class-based decorators keep rich state; registry decorators return the function unchanged.
* `functools.cache` and `lru_cache` give fast memoisation for pure functions with hashable arguments.
* Debug decorators by calling `__wrapped__`, applying them by hand, and removing them one at a time.

---

# Part 5: Context Managers

**What these notes cover**

* Why cleanup is hard to get right, and how the `with` statement solves it.
* Exactly what `with` does: `__enter__`, `__exit__`, and the path taken when an exception happens (flowchart).
* How to write class-based context managers, and how (and when) to suppress exceptions.
* How to write context managers quickly with `@contextlib.contextmanager`.
* The `contextlib` toolbox: `suppress`, `redirect_stdout`, `closing`, `nullcontext`, `ExitStack`, `chdir`.
* Real uses: files, locks, database transactions, timers, temporary state, and how to debug context managers.

**Before you start:** Notebook 1, Part 12 (Exceptions: `try/except/finally`) and Part 13 (Files and `with open`). Part 3 of this notebook (generators) for section 5.4, and Part 4 (decorators) for `@contextmanager`.

## 5.1 Why context managers exist

Many resources must be **released** after use: files must be closed, locks released, database transactions committed or rolled back, temporary settings restored. If an exception happens in the middle, the cleanup code is easy to skip by accident.

The classic safe pattern is `try/finally`:

```python
f = open("data.txt", encoding="utf-8")
try:
    data = f.read()
finally:
    f.close()          # runs even if read() fails
```

This works, but it is long and easy to forget. A **context manager** packages the setup and the cleanup into one object, and the `with` statement guarantees the cleanup:

```python
with open("data.txt", encoding="utf-8") as f:
    data = f.read()
# here the file is closed, whatever happened inside the block
```

**In simple words:** a context manager is like a hotel room door that locks itself when you leave, even if you leave in a hurry.

In [200]:
# The file is closed after the with-block, even when an error happens inside it
import os
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "notes.txt")

    with open(path, "w", encoding="utf-8") as f:
        f.write("line 1\n")
    print(f.closed)                     # True

    try:
        with open(path, encoding="utf-8") as f:
            raise ValueError("problem while reading")
    except ValueError as e:
        print("caught:", e)             # caught: problem while reading
    print(f.closed)                     # True - closed even after the error

True
caught: problem while reading
True


> **Key points**
> * Resources need cleanup even when errors happen.
> * `try/finally` is correct but verbose; `with` packages it into one line.
> * A context manager is an object that knows how to set up and clean up.

---

## 5.2 How the with statement works

A context manager is any object with two methods:

* `__enter__(self)`: runs at the start of the block. Its **return value** is bound to the name after `as` (if any).
* `__exit__(self, exc_type, exc_value, traceback)`: runs when the block ends **for any reason**: normal end, exception, `return`, `break` or `continue`.

The three arguments of `__exit__` describe the exception. They are all `None` if the block ended normally.

If `__exit__` returns a **true** value, the exception is **suppressed** (swallowed). If it returns `False` or `None`, the exception continues to propagate.

### The enter/exit path, including exceptions

```text
        +----------------------------------+
        | with manager as name:            |
        +----------------+-----------------+
                         |
                         v
        +----------------------------------+
        | name = manager.__enter__()       |
        +----------------+-----------------+
                         |
                         v
        +----------------------------------+
        | run the block                    |
        +----------------+-----------------+
                         |
                         v
        +----------------------------------+   No   +--------------------------+
        | exception raised in the block?   |------->| __exit__(None,None,None) |
        +----------------+-----------------+        | continue after "with"    |
                         | Yes                      +--------------------------+
                         v
        +----------------------------------+
        | __exit__(type, value, traceback) |
        +----------------+-----------------+
                         |
                         v
        +----------------------------------+  true  +--------------------------+
        | what did __exit__ return?        |------->| exception suppressed;    |
        +----------------+-----------------+        | continue after "with"    |
                         | False / None             +--------------------------+
                         v
        +----------------------------------+
        | exception propagates to caller   |
        +----------------------------------+
```

> **Note:** If `__enter__` itself raises, the block does not run and `__exit__` is **not** called. Setup that can fail must clean up after itself.

### What with really does (simplified)

```python
with manager as name:
    block
```

is roughly:

```python
mgr = manager
name = mgr.__enter__()
try:
    block
except BaseException as e:
    if not mgr.__exit__(type(e), e, e.__traceback__):
        raise                 # not suppressed: re-raise
else:
    mgr.__exit__(None, None, None)
```

(The real code looks the special methods up on the type, and also handles `return`/`break` the same way as a normal exit.)

In [201]:
# Watch the order of calls
class Show:
    def __enter__(self):
        print("  __enter__")
        return "resource"

    def __exit__(self, exc_type, exc_value, tb):
        print(f"  __exit__ exc_type={exc_type.__name__ if exc_type else None}")
        return False                      # do not suppress

with Show() as value:
    print("  inside, value =", value)
print("after the block")
#   __enter__
#   inside, value = resource
#   __exit__ exc_type=None
# after the block

  __enter__
  inside, value = resource
  __exit__ exc_type=None
after the block


In [202]:
# __exit__ runs on exception, return and break too
class Show:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        print(f"  exit ({exc_type.__name__ if exc_type else 'no error'})")
        return False

try:
    with Show():
        raise KeyError("missing")
except KeyError as e:
    print("caught outside:", e)          # caught outside: 'missing'

def find_first_even(numbers):
    with Show():
        for n in numbers:
            if n % 2 == 0:
                return n                 # __exit__ still runs before returning

print(find_first_even([1, 4, 6]))        #   exit (no error)   then 4

  exit (KeyError)
caught outside: 'missing'
  exit (no error)
4


> **Key points**
> * `with m as x:` calls `m.__enter__()` (result goes to `x`) and always calls `m.__exit__(...)` at the end.
> * `__exit__` receives the exception details, or three `None`s.
> * Return a true value from `__exit__` only if you really want to swallow the exception.

---

## 5.3 Writing a class-based context manager

Write a class when the manager has state you want to keep or inspect (for example, a timer's elapsed time).

In [203]:
# A resource that is "opened" and "closed"
class Connection:
    def __init__(self, name):
        self.name = name
        self.is_open = False

    def __enter__(self):
        self.is_open = True
        print(f"  open {self.name}")
        return self                         # usually return self

    def __exit__(self, exc_type, exc_value, tb):
        self.is_open = False
        print(f"  close {self.name}")
        return False

with Connection("orders-db") as conn:
    print("  using", conn.name, "open:", conn.is_open)
print("open after block:", conn.is_open)
#   open orders-db
#   using orders-db open: True
#   close orders-db
# open after block: False

  open orders-db
  using orders-db open: True
  close orders-db
open after block: False


In [204]:
# A timer context manager: elapsed time is available after the block
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        self.elapsed = None
        return self

    def __exit__(self, exc_type, exc_value, tb):
        self.elapsed = time.perf_counter() - self.start
        return False

with Timer() as t:
    total = sum(range(200_000))

print(total)                            # 19999900000
print("measured:", t.elapsed > 0)       # measured: True  (the exact time varies)

19999900000
measured: True


### Suppressing exceptions on purpose

Returning `True` from `__exit__` swallows the exception. Do this only for **specific, expected** exceptions. Check `exc_type` with `issubclass`.

In [205]:
# Suppress only the exception types you list
class Ignore:
    def __init__(self, *exception_types):
        self.exception_types = exception_types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        # exc_type is None when there was no exception; issubclass needs a class
        return exc_type is not None and issubclass(exc_type, self.exception_types)

with Ignore(KeyError):
    prices = {"pen": 10}
    print(prices["book"])               # KeyError is raised and suppressed
print("continued after KeyError")       # continued after KeyError

try:
    with Ignore(KeyError):
        int("abc")                      # ValueError is NOT in the list
except ValueError as e:
    print("ValueError still raised:", e)

continued after KeyError
ValueError still raised: invalid literal for int() with base 10: 'abc'


> **Warning:** A `__exit__` that always returns `True` hides every bug inside the block, including typos (`NameError`). Return `False` (or nothing) by default.

### A transaction-style manager

A **transaction** groups changes so they either all happen (**commit**) or none happen (**rollback**). A context manager is the natural way to express this.

In [206]:
# Commit on success, roll back on error
class Transaction:
    def __init__(self, data):
        self.data = data

    def __enter__(self):
        self.snapshot = dict(self.data)    # remember the state before changes
        return self.data

    def __exit__(self, exc_type, exc_value, tb):
        if exc_type is not None:
            self.data.clear()
            self.data.update(self.snapshot)   # rollback
            print("  rolled back")
        else:
            print("  committed")
        return False                          # let the error reach the caller

accounts = {"Asha": 100, "Ravi": 50}

with Transaction(accounts) as acc:
    acc["Asha"] -= 30
    acc["Ravi"] += 30
print(accounts)                 #   committed   then {'Asha': 70, 'Ravi': 80}

try:
    with Transaction(accounts) as acc:
        acc["Asha"] -= 500
        if acc["Asha"] < 0:
            raise ValueError("insufficient funds")
except ValueError as e:
    print("error:", e)
print(accounts)                 #   rolled back / error: insufficient funds / {'Asha': 70, 'Ravi': 80}

  committed
{'Asha': 70, 'Ravi': 80}
  rolled back
error: insufficient funds
{'Asha': 70, 'Ravi': 80}


> **Common mistake:** Returning nothing from `__enter__` and then using `as`. The name gets `None`. Return `self` (or the resource) from `__enter__`.

> **Key points**
> * Class-based managers: `__enter__` sets up and returns the resource; `__exit__` cleans up.
> * Return `True` from `__exit__` only for specific, expected exceptions.
> * Commit/rollback is a classic context manager use.

---

### Practice 5.1: The with statement and class-based managers

**A. Predict the output**

1. What does this print?
   ```python
   class M:
       def __enter__(self):
           print("enter")
           return 42

       def __exit__(self, *exc):
           print("exit")

   with M() as x:
       print(x)
   ```
2. What does this print?
   ```python
   class Quiet:
       def __enter__(self):
           return self

       def __exit__(self, *exc):
           return True

   with Quiet():
       print("a")
       1 / 0
       print("b")
   print("c")
   ```
3. What is `x` after this code?
   ```python
   class M:
       def __enter__(self):
           pass

       def __exit__(self, *exc):
           return False

   with M() as x:
       pass
   ```

**B. Write code**

4. Write a class `Indenter` used as nested `with` blocks; inside, `indenter.print(text)` prints the text indented by 2 spaces per nesting level.
5. Write a class `TempValue(obj, attr, value)` that sets `obj.attr = value` inside the block and restores the old value afterwards (even on error).
6. Write a class `CollectErrors` that suppresses `ValueError` and `KeyError` and stores the suppressed exceptions in a list `self.errors`.
7. **[Hard]** Write a class `Budget(limit)` whose `__enter__` returns an object with a `spend(amount)` method. If the total spent goes over `limit`, `spend` raises `ValueError`, and on exit the manager records `self.ok` as `True` or `False`.

**C. Find and fix the bug**

8. The program prints nothing for errors that happen inside the block. Why? Fix it so only `FileNotFoundError` is hidden.
   ```python
   class Safe:
       def __enter__(self):
           return self

       def __exit__(self, exc_type, exc, tb):
           return True
   ```

**D. Interview questions**

9. What does the `with` statement guarantee?
10. **[Hard]** What happens if `__enter__` raises an exception? What if `__exit__` raises one?

### Solutions 5.1

1. `enter`, `42`, `exit`.
2. `a`, then `c`. The `ZeroDivisionError` is suppressed because `__exit__` returns `True`, so `b` is never printed.
3. `None`, because `__enter__` returns nothing.
4. to 7. See the code below.
8. `__exit__` returns `True` for every exception, so all errors are swallowed. Fix: `return exc_type is not None and issubclass(exc_type, FileNotFoundError)`.
9. That `__exit__` is called when the block is left for any reason (normal end, exception, `return`, `break`, `continue`), so cleanup always runs once `__enter__` has succeeded.
10. If `__enter__` raises, the block is skipped and `__exit__` is not called; the exception propagates. If `__exit__` raises, its new exception replaces any exception from the block (the original one is kept as `__context__`).

In [207]:
# Solutions 5.1
import io
from contextlib import redirect_stdout

# Q2 check
class Quiet:
    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return True

buf = io.StringIO()
with redirect_stdout(buf):
    with Quiet():
        print("a")
        1 / 0
        print("b")
    print("c")
assert buf.getvalue() == "a\nc\n"


# Q4
class Indenter:
    def __init__(self):
        self.level = -1

    def __enter__(self):
        self.level += 1
        return self

    def __exit__(self, *exc):
        self.level -= 1
        return False

    def print(self, text):
        print("  " * self.level + text)

buf = io.StringIO()
with redirect_stdout(buf):
    with Indenter() as ind:
        ind.print("level 0")
        with ind:
            ind.print("level 1")
        ind.print("back to 0")
assert buf.getvalue() == "level 0\n  level 1\nback to 0\n"


# Q5
class TempValue:
    def __init__(self, obj, attr, value):
        self.obj, self.attr, self.value = obj, attr, value

    def __enter__(self):
        self.old = getattr(self.obj, self.attr)
        setattr(self.obj, self.attr, self.value)
        return self.obj

    def __exit__(self, *exc):
        setattr(self.obj, self.attr, self.old)
        return False

class Settings:
    debug = False

s = Settings()
with TempValue(s, "debug", True):
    assert s.debug is True
assert s.debug is False
try:
    with TempValue(s, "debug", True):
        raise RuntimeError("fail")
except RuntimeError:
    pass
assert s.debug is False                 # restored even after an error


# Q6
class CollectErrors:
    def __init__(self):
        self.errors = []

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        if exc_type is not None and issubclass(exc_type, (ValueError, KeyError)):
            self.errors.append(exc_value)
            return True
        return False

collector = CollectErrors()
for text in ["1", "x", "3"]:
    with collector:
        int(text)
assert len(collector.errors) == 1 and isinstance(collector.errors[0], ValueError)


# Q7
class Budget:
    def __init__(self, limit):
        self.limit = limit
        self.spent = 0
        self.ok = None

    def spend(self, amount):
        if self.spent + amount > self.limit:
            raise ValueError("over budget")
        self.spent += amount

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        self.ok = exc_type is None
        return False

with Budget(100) as b:
    b.spend(60)
    b.spend(40)
assert b.ok is True and b.spent == 100

b2 = Budget(50)
try:
    with b2:
        b2.spend(30)
        b2.spend(30)
except ValueError:
    pass
assert b2.ok is False and b2.spent == 30


# Q8: fixed
class Safe:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        return exc_type is not None and issubclass(exc_type, FileNotFoundError)

with Safe():
    open("surely_missing_file_12345.txt", encoding="utf-8")
try:
    with Safe():
        int("x")
    raise AssertionError("ValueError should not be hidden")
except ValueError:
    pass
print("Practice 5.1: all checks passed")

Practice 5.1: all checks passed


## 5.4 Context managers from generators: @contextmanager

Writing a class with two methods is verbose. `contextlib.contextmanager` turns a **generator function** (Part 3) into a context manager:

* Code **before** `yield` is the setup (`__enter__`).
* The **yielded value** is bound to the `as` name.
* Code **after** `yield` is the cleanup (`__exit__`).
* If the block raises, the exception is raised **at the `yield` line** inside your generator. So put the cleanup in `try/finally`.

```text
  @contextmanager
  def manager():
      setup            <- runs on enter
      try:
          yield value  <- the with-block runs here (paused)
      finally:
          cleanup      <- runs on exit, even after an exception
```

In [208]:
# A generator-based context manager with guaranteed cleanup
from contextlib import contextmanager

@contextmanager
def opened(name):
    print(f"  open {name}")
    try:
        yield name.upper()              # value for "as"
    finally:
        print(f"  close {name}")        # always runs

with opened("report") as handle:
    print("  using", handle)
#   open report
#   using REPORT
#   close report

try:
    with opened("invoice"):
        raise ValueError("bad data")
except ValueError as e:
    print("caught:", e)
#   open invoice
#   close invoice
# caught: bad data

  open report
  using REPORT
  close report
  open invoice
  close invoice
caught: bad data


In [209]:
# Common mistake: no try/finally, so cleanup is skipped on error
from contextlib import contextmanager

state = {"locked": False}

@contextmanager
def broken_lock():
    state["locked"] = True
    yield                              # an exception in the block is raised HERE
    state["locked"] = False            # ...so this line never runs

try:
    with broken_lock():
        raise RuntimeError("crash")
except RuntimeError:
    pass
print("still locked:", state["locked"])     # still locked: True   <- the bug

# Fix: try/finally around the yield
@contextmanager
def safe_lock():
    state["locked"] = True
    try:
        yield
    finally:
        state["locked"] = False

state["locked"] = False
try:
    with safe_lock():
        raise RuntimeError("crash")
except RuntimeError:
    pass
print("still locked:", state["locked"])     # still locked: False

still locked: True
still locked: False


In [210]:
# Temporarily change the working directory (contextlib.chdir does this since 3.11)
import os
import tempfile
from contextlib import contextmanager

@contextmanager
def working_dir(path):
    old = os.getcwd()
    os.chdir(path)
    try:
        yield path
    finally:
        os.chdir(old)                    # always restored

before = os.getcwd()
with tempfile.TemporaryDirectory() as folder:
    with working_dir(folder):
        print("inside folder:", os.path.samefile(os.getcwd(), folder))   # True
print("restored:", os.getcwd() == before)                                 # True

inside folder: True
restored: True


In [211]:
# Handling or suppressing an exception inside a generator-based manager
from contextlib import contextmanager

@contextmanager
def log_errors(log):
    try:
        yield
    except ZeroDivisionError as e:       # handle this type: it is suppressed
        log.append(f"ignored {type(e).__name__}")
    # other exceptions propagate normally

log = []
with log_errors(log):
    1 / 0
print(log)                               # ['ignored ZeroDivisionError']

try:
    with log_errors(log):
        [][1]
except IndexError as e:
    print("IndexError propagated:", e)   # IndexError propagated: list index out of range

['ignored ZeroDivisionError']
IndexError propagated: list index out of range


### A contextmanager can also be used as a decorator

Objects created by `@contextmanager` also work as decorators (they inherit from `contextlib.ContextDecorator`). The whole function body then runs inside the `with`.

In [212]:
# The same manager used with "with" and as "@decorator"
from contextlib import contextmanager

events = []

@contextmanager
def audit(action):
    events.append(f"start {action}")
    try:
        yield
    finally:
        events.append(f"end {action}")

@audit("nightly-report")
def build_report():
    events.append("building")

build_report()
print(events)    # ['start nightly-report', 'building', 'end nightly-report']

['start nightly-report', 'building', 'end nightly-report']


| Class-based | `@contextmanager` |
|---|---|
| More code (two methods) | Short: one generator function |
| Easy to keep state as attributes | State lives in local variables |
| Can be reused many times (if designed so) | Each call creates a one-shot manager |
| Full control of suppression via return value | Suppress by catching the exception around `yield` |

> **Warning:** The generator must `yield` **exactly once**. Zero yields gives `RuntimeError: generator didn't yield`; two yields gives `RuntimeError: generator didn't stop` (see section 5.7).

> **Key points**
> * `@contextmanager`: setup, `yield value`, cleanup.
> * Always wrap `yield` in `try/finally` when there is cleanup.
> * Catch an exception around `yield` to suppress it; let others propagate.

---

## 5.5 The contextlib toolbox

The `contextlib` module has ready-made managers for common jobs.

| Tool | What it does |
|---|---|
| `suppress(*exceptions)` | Ignore specific exceptions inside the block |
| `redirect_stdout(f)` / `redirect_stderr(f)` | Send `print` output (or errors) to another file object |
| `closing(obj)` | Call `obj.close()` at the end, for objects that have `close()` but no `with` support |
| `nullcontext(value)` | A "do nothing" manager; useful when a manager is optional |
| `ExitStack()` | Combine a dynamic number of managers and cleanup callbacks |
| `chdir(path)` (3.11+) | Temporarily change the working directory |

In [213]:
# suppress: a clean replacement for try/except/pass of ONE expected exception
import os
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("this_file_does_not_exist.tmp")
print("no crash")                       # no crash

# Same as:
try:
    os.remove("this_file_does_not_exist.tmp")
except FileNotFoundError:
    pass

no crash


In [214]:
# redirect_stdout: capture what print() writes (useful in tests)
import io
from contextlib import redirect_stdout

def report():
    print("Total orders: 3")

buffer = io.StringIO()
with redirect_stdout(buffer):
    report()
print(repr(buffer.getvalue()))          # 'Total orders: 3\n'

'Total orders: 3\n'


In [215]:
# closing: for objects with close() but without __enter__/__exit__
from contextlib import closing

class LegacyClient:
    def __init__(self):
        self.closed = False

    def fetch(self):
        return "data"

    def close(self):
        self.closed = True

with closing(LegacyClient()) as client:
    print(client.fetch())               # data
print(client.closed)                    # True

data
True


In [216]:
# nullcontext: make a manager optional without duplicating code
import threading
from contextlib import nullcontext

def update(counter, lock=None):
    with lock if lock is not None else nullcontext():
        counter["n"] += 1

counter = {"n": 0}
update(counter)                          # no lock
update(counter, threading.Lock())        # with a lock
print(counter)                           # {'n': 2}

{'n': 2}


### ExitStack: a dynamic number of managers

When you do not know in advance how many resources you will open (for example, a list of files), you cannot write a fixed number of `with` statements. `ExitStack` collects them and closes **all** of them, in reverse order, when the block ends. If opening the third file fails, the first two are still closed.

In [217]:
# ExitStack: open any number of files and close them all safely
import os
import tempfile
from contextlib import ExitStack

with tempfile.TemporaryDirectory() as folder:
    paths = [os.path.join(folder, f"part{i}.txt") for i in range(3)]

    with ExitStack() as stack:
        files = [stack.enter_context(open(p, "w", encoding="utf-8")) for p in paths]
        for i, f in enumerate(files):
            f.write(f"chunk {i}\n")
        print("all open:", all(not f.closed for f in files))    # all open: True

    print("all closed:", all(f.closed for f in files))          # all closed: True

all open: True
all closed: True


In [218]:
# ExitStack.callback: register any cleanup function; they run in reverse order
from contextlib import ExitStack

with ExitStack() as stack:
    stack.callback(print, "  cleanup 1 (registered first, runs last)")
    stack.callback(print, "  cleanup 2 (registered last, runs first)")
    print("  working")
#   working
#   cleanup 2 (registered last, runs first)
#   cleanup 1 (registered first, runs last)

  working
  cleanup 2 (registered last, runs first)
  cleanup 1 (registered first, runs last)


In [219]:
# contextlib.chdir (Python 3.11+): built-in version of working_dir from 5.4
import os
import tempfile
from contextlib import chdir

before = os.getcwd()
with tempfile.TemporaryDirectory() as folder:
    with chdir(folder):
        print(os.path.samefile(os.getcwd(), folder))   # True
print(os.getcwd() == before)                           # True

True
True


> **Tip:** `suppress` should list the narrowest exception possible. `suppress(Exception)` is as dangerous as a bare `except:`.

> **Key points**
> * `suppress`, `redirect_stdout`, `closing`, `nullcontext`, `chdir` cover most small needs.
> * `ExitStack` handles a dynamic number of managers and cleanup callbacks, in reverse (LIFO) order.

---

### Practice 5.2: @contextmanager and contextlib

**A. Predict the output**

1. What does this print?
   ```python
   from contextlib import contextmanager

   @contextmanager
   def tag(name):
       print(f"<{name}>")
       yield
       print(f"</{name}>")

   with tag("p"):
       print("text")
   ```
2. What does this print?
   ```python
   from contextlib import ExitStack

   with ExitStack() as stack:
       for i in range(3):
           stack.callback(print, i)
   ```
3. What does this print?
   ```python
   from contextlib import suppress

   with suppress(KeyError):
       print("start")
       {}["x"]
       print("never")
   print("end")
   ```

**B. Write code**

4. With `@contextmanager`, write `temp_env(key, value)` that sets `os.environ[key]` inside the block and restores the old state afterwards, including "the key did not exist".
5. With `@contextmanager`, write `capture_output()` that yields a list; after the block the list contains the lines printed inside the block.
6. Write `open_all(paths)` using `ExitStack` that yields a list of open file objects (for reading) and closes them all at the end.
7. **[Hard]** With `@contextmanager`, write `atomic_write(path)` that yields a file object for a temporary file in the same folder, and on success replaces `path` with it (`os.replace`). On error, the temporary file is removed and `path` is unchanged.

**C. Find and fix the bug**

8. After an error in the block, the timer never prints. Fix it.
   ```python
   import time
   from contextlib import contextmanager

   @contextmanager
   def timed(label):
       start = time.perf_counter()
       yield
       print(label, time.perf_counter() - start)
   ```

**D. Interview questions**

9. When would you write a class-based context manager instead of using `@contextmanager`?
10. **[Hard]** What problem does `ExitStack` solve? Give an example.

### Solutions 5.2

1. `<p>`, `text`, `</p>`.
2. `2`, `1`, `0`. Callbacks run in reverse order of registration.
3. `start`, then `end`.
4. to 7. See the code below.
8. Put `yield` in `try/finally` so the print runs even when the block raises:
   `try: yield` / `finally: print(...)`.
9. When the manager needs rich state that callers inspect afterwards (like a timer's `elapsed`), extra methods, reuse of the same object for several `with` blocks, or when it is part of a larger class (a connection object that is also a context manager).
10. It manages a number of context managers that is only known at run time, and guarantees that all already-entered managers are exited, in reverse order, even if entering a later one fails. Example: opening every file in a user-supplied list, or acquiring a variable number of locks.

In [220]:
# Solutions 5.2
import io
import os
import tempfile
from contextlib import contextmanager, ExitStack, redirect_stdout

# Q4
@contextmanager
def temp_env(key, value):
    old = os.environ.get(key)
    os.environ[key] = value
    try:
        yield
    finally:
        if old is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = old

with temp_env("NOTES_PRACTICE_VAR", "on"):
    assert os.environ["NOTES_PRACTICE_VAR"] == "on"
assert "NOTES_PRACTICE_VAR" not in os.environ


# Q5
@contextmanager
def capture_output():
    lines = []
    buffer = io.StringIO()
    try:
        with redirect_stdout(buffer):
            yield lines
    finally:
        lines.extend(buffer.getvalue().splitlines())

with capture_output() as out:
    print("hello")
    print("world")
assert out == ["hello", "world"]


# Q6
@contextmanager
def open_all(paths):
    with ExitStack() as stack:
        yield [stack.enter_context(open(p, encoding="utf-8")) for p in paths]


# Q7
@contextmanager
def atomic_write(path):
    folder = os.path.dirname(os.path.abspath(path))
    fd, tmp_path = tempfile.mkstemp(dir=folder, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            yield f
        os.replace(tmp_path, path)          # only reached when the block succeeded
    except BaseException:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise


with tempfile.TemporaryDirectory() as folder:
    # Q6 test
    paths = []
    for i in range(2):
        p = os.path.join(folder, f"f{i}.txt")
        with open(p, "w", encoding="utf-8") as f:
            f.write(f"file {i}")
        paths.append(p)
    with open_all(paths) as files:
        assert [f.read() for f in files] == ["file 0", "file 1"]
    assert all(f.closed for f in files)

    # Q7 tests
    target = os.path.join(folder, "config.txt")
    with atomic_write(target) as f:
        f.write("version=1")
    with open(target, encoding="utf-8") as f:
        assert f.read() == "version=1"
    try:
        with atomic_write(target) as f:
            f.write("version=2 (half written)")
            raise RuntimeError("crash in the middle")
    except RuntimeError:
        pass
    with open(target, encoding="utf-8") as f:
        assert f.read() == "version=1"                     # unchanged
    assert not [n for n in os.listdir(folder) if n.endswith(".tmp")]


# Q8: fixed
import time

@contextmanager
def timed(label, results):
    start = time.perf_counter()
    try:
        yield
    finally:
        results.append((label, time.perf_counter() - start))

results = []
try:
    with timed("load", results):
        raise ValueError("bad")
except ValueError:
    pass
assert results[0][0] == "load"
print("Practice 5.2: all checks passed")

Practice 5.2: all checks passed


## 5.6 Real-world uses

You will meet context managers everywhere. Typical cases:

| Use | Example |
|---|---|
| Files | `with open(path, encoding="utf-8") as f:` |
| Locks (threads) | `with lock:` (acquire, then always release) |
| Database transactions | `with conn:` in `sqlite3` (commit or rollback) |
| Temporary folders and files | `with tempfile.TemporaryDirectory() as d:` |
| Temporary settings | `decimal.localcontext()`, `warnings.catch_warnings()`, `np.errstate` |
| Tests | `with unittest.mock.patch(...)`, `with self.assertRaises(...)` (Part 9) |
| Timing and logging | a `Timer` manager, or a manager that logs start/end |
| Network | sessions and sockets (`with socket.socket() as s:`) |

In [221]:
# Locks: "with lock" acquires and always releases (threads are covered in Part 8)
import threading

lock = threading.Lock()

with lock:
    print("inside: locked =", lock.locked())    # inside: locked = True
print("after: locked =", lock.locked())         # after: locked = False

try:
    with lock:
        raise RuntimeError("error while holding the lock")
except RuntimeError:
    pass
print("released after error:", not lock.locked())   # released after error: True

inside: locked = True
after: locked = False
released after error: True


In [222]:
# sqlite3: "with conn" commits on success and rolls back on error (it does NOT close)
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as conn:       # closing() closes the connection
    conn.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, total REAL)")

    with conn:                                           # transaction 1: commit
        conn.execute("INSERT INTO orders (total) VALUES (?)", (250.0,))

    try:
        with conn:                                       # transaction 2: rollback
            conn.execute("INSERT INTO orders (total) VALUES (?)", (99.0,))
            raise ValueError("payment failed")
    except ValueError as e:
        print("error:", e)                               # error: payment failed

    rows = conn.execute("SELECT id, total FROM orders").fetchall()
    print(rows)                                          # [(1, 250.0)]

error: payment failed
[(1, 250.0)]


In [223]:
# Temporary settings: decimal precision only inside the block
from decimal import Decimal, localcontext

with localcontext() as ctx:
    ctx.prec = 5
    print(Decimal(1) / Decimal(7))      # 0.14286
print(Decimal(1) / Decimal(7))          # 0.1428571428571428571428571429

0.14286
0.1428571428571428571428571429


In [224]:
# Tests: patch an object only inside the block (more in Part 9)
from unittest import mock
import random

original = random.randint
with mock.patch("random.randint", return_value=4):
    print(random.randint(1, 6))          # 4  - patched
    print(random.randint is original)    # False
print(random.randint is original)        # True - the real function is back

4
False
True


### Several managers in one with statement

You can open several managers in one `with`. They are entered left to right and exited right to left. Since Python 3.10 you can wrap them in parentheses over several lines.

```python
with (
    open("input.txt", encoding="utf-8") as src,
    open("output.txt", "w", encoding="utf-8") as dst,
):
    dst.write(src.read())
```

In [225]:
# Several managers: entered left to right, exited right to left
from contextlib import contextmanager

@contextmanager
def step(name):
    print(f"  enter {name}")
    try:
        yield name
    finally:
        print(f"  exit {name}")

with (
    step("A") as a,
    step("B") as b,
):
    print(f"  body uses {a} and {b}")
#   enter A
#   enter B
#   body uses A and B
#   exit B
#   exit A

  enter A
  enter B
  body uses A and B
  exit B
  exit A


### Asynchronous context managers (preview)

Code that uses `async`/`await` (Part 8) has its own version: `async with`, implemented by `__aenter__` and `__aexit__`, or by `@contextlib.asynccontextmanager`. Examples: async database connections, `asyncio.timeout()`, `asyncio.TaskGroup()`.

The next cell uses top-level `await`, which works in Jupyter. In a script, put the code in an `async def main()` and call `asyncio.run(main())`.

In [226]:
# An async context manager (top-level await works in Jupyter notebooks)
import asyncio
from contextlib import asynccontextmanager

@asynccontextmanager
async def async_session(name):
    print(f"  connect {name}")
    try:
        yield name
    finally:
        print(f"  disconnect {name}")

async def main():
    async with async_session("api") as s:
        await asyncio.sleep(0)               # pretend to do I/O
        print("  working with", s)

await main()
#   connect api
#   working with api
#   disconnect api

  connect api
  working with api
  disconnect api


> **Key points**
> * Use `with` for files, locks, transactions, temporary folders, temporary settings and test patches.
> * `with conn:` in `sqlite3` manages a transaction, not the connection's lifetime; use `closing()` to close it.
> * Several managers in one `with` are exited in reverse order.
> * `async with` is the asynchronous version (Part 8).

---

## 5.7 How to debug context managers

| Symptom or error | Cause | Fix |
|---|---|---|
| `TypeError: '...' object does not support the context manager protocol` | The object has no `__enter__`/`__exit__` | Use `closing()`, or add the methods |
| `as` name is `None` | `__enter__` returns nothing | `return self` (or the resource) |
| Errors disappear silently | `__exit__` returns `True`, or `suppress(Exception)` | Return `False`; suppress only specific types |
| `RuntimeError: generator didn't yield` | `@contextmanager` function never reached `yield` | Make sure every path yields once |
| `RuntimeError: generator didn't stop` | `@contextmanager` function yields twice | Yield exactly once (no loop around `yield`) |
| Cleanup skipped after an error | No `try/finally` around `yield` | Add `try/finally` |
| `ValueError: I/O operation on closed file.` | Using a file after its `with` block | Do the work inside the block |

In [227]:
# Reproduce the typical errors (exact wording can differ slightly between Python versions)
from contextlib import contextmanager

class NotAManager:
    pass

try:
    with NotAManager():
        pass
except TypeError as e:
    print("1.", type(e).__name__ + ":", e)

@contextmanager
def never_yields(flag):
    if flag:
        yield                        # Common mistake: when flag is False, nothing is yielded

try:
    with never_yields(False):
        pass
except RuntimeError as e:
    print("2.", type(e).__name__ + ":", e)     # 2. RuntimeError: generator didn't yield

@contextmanager
def yields_twice():
    yield 1
    yield 2                          # Common mistake: a second yield

try:
    with yields_twice():
        pass
except RuntimeError as e:
    print("3.", type(e).__name__ + ":", e)     # 3. RuntimeError: generator didn't stop

1. TypeError: 'NotAManager' object does not support the context manager protocol (missed __exit__ method)
2. RuntimeError: generator didn't yield
3. RuntimeError: generator didn't stop


In [228]:
# Using a file after the with-block ended
import os
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "log.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("start\n")

    try:
        f.write("too late\n")                  # the file is already closed
    except ValueError as e:
        print(type(e).__name__ + ":", e)       # ValueError: I/O operation on closed file.

ValueError: I/O operation on closed file.


### A tracing manager for debugging

When you are not sure whether `__enter__`/`__exit__` run, or which exception reaches `__exit__`, wrap the block in a small tracing manager.

In [229]:
# Debug helper: show when a block starts, ends, and with which exception
from contextlib import contextmanager

@contextmanager
def trace_block(name):
    print(f"  -> enter {name}")
    try:
        yield
    except BaseException as e:
        print(f"  !! {name} raised {type(e).__name__}: {e}")
        raise                                   # never hide the error
    else:
        print(f"  <- exit {name} (ok)")

try:
    with trace_block("import orders"):
        int("12x")
except ValueError:
    pass
#   -> enter import orders
#   !! import orders raised ValueError: invalid literal for int() with base 10: '12x'

  -> enter import orders
  !! import orders raised ValueError: invalid literal for int() with base 10: '12x'


```text
        +--------------------------------+
        | Resource not cleaned up, or    |
        | error vanished / unexpected    |
        +---------------+----------------+
                        |
                        v
        +--------------------------------+  No  +---------------------------+
        | Is the cleanup in __exit__ or  |----->| Move cleanup into finally |
        | in a finally around yield?     |      +---------------------------+
        +---------------+----------------+
                        | Yes
                        v
        +--------------------------------+  Yes +---------------------------+
        | Does __exit__ return True, or  |----->| Return False; suppress    |
        | suppress() a broad type?       |      | only specific exceptions  |
        +---------------+----------------+      +---------------------------+
                        | No
                        v
        +--------------------------------+
        | Add trace_block / prints in    |
        | __enter__ and __exit__; check  |
        | the arguments __exit__ gets    |
        +---------------+----------------+
                        |
                        v
        +--------------------------------+
        | Fix, re-run, add a test that   |
        | raises inside the block        |
        +--------------------------------+
```

> **Tip:** Always test a context manager twice: once with a normal block and once with a block that raises. Most bugs show up only in the second test.

> **Key points**
> * Know the messages: "does not support the context manager protocol", "generator didn't yield", "generator didn't stop", "I/O operation on closed file".
> * Hidden errors usually mean a `__exit__` that returns `True` or a too-broad `suppress`.
> * Test both the success path and the exception path.

---

### Practice 5.3: Real uses and debugging

**A. Predict the output**

1. What does this print?
   ```python
   from contextlib import contextmanager

   @contextmanager
   def box(name):
       print("open", name)
       try:
           yield
       finally:
           print("close", name)

   with box("outer"), box("inner"):
       print("work")
   ```
2. What does this print?
   ```python
   import threading

   lock = threading.Lock()
   with lock:
       pass
   print(lock.locked())
   ```

**B. Write code**

3. Write a `@contextmanager` named `rollback_on_error(items)` for a list: on error, restore the list to its content before the block (in place), then re-raise.
4. Write a class `Stopwatch` that can be used many times; each `with` block adds its time to `self.total` and increases `self.laps`.
5. **[Hard]** Write a `@contextmanager` named `limit_recursion(n)` that temporarily sets `sys.setrecursionlimit(n)` and always restores the previous limit.

**C. Find and fix the bug**

6. The lock is never released if `process()` raises. Rewrite it with `with`.
   ```python
   lock.acquire()
   process()
   lock.release()
   ```

**D. Interview questions**

7. Name five places where Python code commonly uses `with`.
8. **[Hard]** Why does `with sqlite3.connect(...) as conn:` not close the connection, and how do you close it safely?

### Solutions 5.3

1. `open outer`, `open inner`, `work`, `close inner`, `close outer`.
2. `False`.
3. to 5. See the code below.
6. Use `with lock:` followed by an indented `process()` line. The lock is released even if `process()` raises.
7. Files, thread locks, database transactions, temporary folders/files, `mock.patch` in tests, `decimal.localcontext`, `warnings.catch_warnings`, `asyncio.timeout`, `redirect_stdout`.
8. In `sqlite3`, the connection's context manager controls a **transaction** (commit on success, rollback on error); it was designed that way before context managers were common. Close it with `contextlib.closing(sqlite3.connect(...))`, or call `conn.close()` in a `finally`.

In [230]:
# Solutions 5.3
import sys
import threading
import time
from contextlib import contextmanager

# Q3
@contextmanager
def rollback_on_error(items):
    snapshot = list(items)
    try:
        yield items
    except BaseException:
        items[:] = snapshot            # restore in place
        raise

cart = ["pen"]
with rollback_on_error(cart):
    cart.append("book")
assert cart == ["pen", "book"]
try:
    with rollback_on_error(cart):
        cart.append("bag")
        raise ValueError("out of stock")
except ValueError:
    pass
assert cart == ["pen", "book"]


# Q4
class Stopwatch:
    def __init__(self):
        self.total = 0.0
        self.laps = 0

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.total += time.perf_counter() - self._start
        self.laps += 1
        return False

sw = Stopwatch()
for _ in range(3):
    with sw:
        sum(range(1000))
assert sw.laps == 3 and sw.total > 0


# Q5
@contextmanager
def limit_recursion(n):
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(n)
    try:
        yield
    finally:
        sys.setrecursionlimit(old)

before = sys.getrecursionlimit()
with limit_recursion(before + 500):
    assert sys.getrecursionlimit() == before + 500
assert sys.getrecursionlimit() == before


# Q6: fixed
lock = threading.Lock()

def process():
    raise RuntimeError("failed")

try:
    with lock:
        process()
except RuntimeError:
    pass
assert not lock.locked()
print("Practice 5.3: all checks passed")

Practice 5.3: all checks passed


### Summary of Part 5

* A context manager packages setup and cleanup; `with` guarantees the cleanup.
* `with m as x:` calls `__enter__` (its result goes to `x`) and always calls `__exit__(exc_type, exc, tb)` at the end.
* A true return value from `__exit__` suppresses the exception; do this only for specific types.
* `@contextmanager` builds a manager from a generator: setup, `yield` once inside `try/finally`, cleanup.
* `contextlib` gives `suppress`, `redirect_stdout`, `closing`, `nullcontext`, `ExitStack` and `chdir`.
* Real uses: files, locks, transactions, temporary folders and settings, test patches, timers; `async with` is the async version.
* Debug by testing both the normal path and the exception path, and by tracing enter/exit.

---

# Part 6: Advanced Language Features

**What these notes cover**

* What "everything is an object" really means, and how to inspect objects at run time (**introspection**).
* The exact order Python uses to look up `obj.attr` (with a flowchart), and the hooks `__getattribute__`, `__getattr__`, `__setattr__`.
* **Descriptors**: the mechanism behind `property`, methods, `classmethod` and `staticmethod`.
* How classes are created: `type()`, `__init_subclass__`, class decorators and metaclasses (and when not to use them).
* Advanced type hints: generics, the Python 3.12 generic syntax (PEP 695), `Protocol`, `TypedDict`, `Literal`, `Final`, `Callable`, `ParamSpec`, and static checkers (mypy, pyright).
* Pattern matching with classes.
* Memory management: reference counting, the cyclic garbage collector, weak references.
* The copy protocol (`__copy__`, `__deepcopy__`, `copy.replace`), and how to debug all of the above.

**Before you start:** Parts 1 and 2 of this notebook (classes, inheritance, properties, dataclasses, dunder methods) and Part 4 (decorators). Notebook 1, Part 10 (references and copying).

> **Note:** Sections 6.1 to 6.4 and 6.6 to 6.8 are core knowledge for an advanced Python developer. Section 6.5 (metaclasses) is specialised: know what it is, but prefer simpler tools.

## 6.1 Everything is an object

In Python, **everything** is an object: numbers, strings, functions, classes, modules, even types themselves. Every object has three things:

* an **identity** (which object it is; compare with `is`),
* a **type** (what kind of object; `type(obj)`),
* a **value** (its data).

Classes are objects too. The type of a normal class is `type`. So `type` is "the class of classes", which is called a **metaclass** (section 6.5).

```text
   5  ---- instance of ---->  int  ---- instance of ---->  type
   "hi" -- instance of ---->  str  ---- instance of ---->  type
   len  -- instance of ---->  builtin_function_or_method
   type -- instance of ---->  type   (type is its own type)
```

In [231]:
# Everything has a type, including types
x = 5
print(type(x))                    # <class 'int'>
print(type(int))                  # <class 'type'>
print(type(type))                 # <class 'type'>
print(type(len).__name__)         # builtin_function_or_method
print(type(lambda: 0).__name__)   # function

import math
print(type(math).__name__)        # module

<class 'int'>
<class 'type'>
<class 'type'>
builtin_function_or_method
function
module


In [232]:
# Classes are objects: you can store them, pass them, and give them attributes
class Invoice:
    pass

class Receipt:
    pass

document_types = {"invoice": Invoice, "receipt": Receipt}   # classes stored in a dict
doc = document_types["receipt"]()                          # create an instance from the dict
print(type(doc).__name__)                                  # Receipt

def make(cls):
    return cls()                                           # a class passed as an argument

print(isinstance(make(Invoice), Invoice))                  # True
print(isinstance(Invoice, type))                           # True - a class is an instance of type

Receipt
True
True


In [233]:
# Functions are objects with attributes too
def discount(price, rate=0.1):
    """Apply a discount."""
    return price * (1 - rate)

print(discount.__name__)          # discount
print(discount.__defaults__)      # (0.1,)
print(discount.__doc__)           # Apply a discount.

discount.version = 2              # any function can carry extra attributes
print(discount.version)           # 2

discount
(0.1,)
Apply a discount.
2


> **Key points**
> * Every object has an identity, a type and a value.
> * Classes and functions are objects: you can store, pass and inspect them.
> * The type of a class is `type`, the default metaclass.

---

## 6.2 Introspection

**Introspection** means a program examining objects while it runs: their type, attributes, methods and signatures. Debuggers, IDEs, test frameworks, serializers and web frameworks all depend on it.

| Tool | What it gives |
|---|---|
| `type(obj)`, `isinstance(obj, cls)` | The type; whether obj is an instance of a class (or subclass) |
| `dir(obj)` | All attribute names reachable from obj (sorted list) |
| `vars(obj)` / `obj.__dict__` | The object's own attribute dictionary |
| `getattr(obj, name, default)` | Read an attribute by its name (a string) |
| `setattr`, `hasattr`, `delattr` | Write, test and delete attributes by name |
| `callable(obj)` | Whether obj can be called |
| `cls.__mro__`, `cls.__bases__` | Method resolution order, direct parents |
| `inspect` module | Signatures, source code, members, kinds of objects |

In [234]:
# dir, vars, getattr, setattr, hasattr
class Product:
    category = "general"                 # class attribute

    def __init__(self, name, price):
        self.name = name                 # instance attributes
        self.price = price

    def label(self):
        return f"{self.name}: {self.price}"

p = Product("pen", 10)

print(vars(p))                                         # {'name': 'pen', 'price': 10}
print([n for n in dir(p) if not n.startswith("__")])   # ['category', 'label', 'name', 'price']
print(getattr(p, "price"))                             # 10
print(getattr(p, "colour", "unknown"))                 # unknown  (default instead of an error)
print(hasattr(p, "label"), callable(p.label))          # True True

setattr(p, "stock", 25)                                # add an attribute by name
print(p.stock)                                         # 25
delattr(p, "stock")
print(hasattr(p, "stock"))                             # False

{'name': 'pen', 'price': 10}
['category', 'label', 'name', 'price']
10
unknown
True True
25
False


In [235]:
# Practical use: call a method chosen at run time by its name
class Report:
    def to_csv(self):
        return "a,b"

    def to_json(self):
        return '{"a": "b"}'

def export(report, fmt):
    method = getattr(report, f"to_{fmt}", None)
    if method is None:
        raise ValueError(f"unsupported format: {fmt}")
    return method()

print(export(Report(), "csv"))          # a,b
print(export(Report(), "json"))         # {"a": "b"}
try:
    export(Report(), "xml")
except ValueError as e:
    print("ValueError:", e)             # ValueError: unsupported format: xml

a,b
{"a": "b"}
ValueError: unsupported format: xml


In [236]:
# inspect: signatures, kinds of objects, members, documentation, source
import inspect
import textwrap

def create_user(name, age=18, *roles, active=True, **extra):
    """Create a user record."""
    return {"name": name}

sig = inspect.signature(create_user)
print(sig)                                    # (name, age=18, *roles, active=True, **extra)
for param in sig.parameters.values():
    print(f"  {param.name:7} {param.kind.name}")
#   name    POSITIONAL_OR_KEYWORD
#   age     POSITIONAL_OR_KEYWORD
#   roles   VAR_POSITIONAL
#   active  KEYWORD_ONLY
#   extra   VAR_KEYWORD

print(inspect.isfunction(create_user), inspect.isclass(create_user))   # True False
print(inspect.getdoc(create_user))                                     # Create a user record.
print(inspect.getsource(textwrap.dedent).splitlines()[0])              # def dedent(text):

(name, age=18, *roles, active=True, **extra)
  name    POSITIONAL_OR_KEYWORD
  age     POSITIONAL_OR_KEYWORD
  roles   VAR_POSITIONAL
  active  KEYWORD_ONLY
  extra   VAR_KEYWORD
True False
Create a user record.
def dedent(text):


In [237]:
# Class introspection: bases, MRO, subclasses, members
import inspect

class Animal:
    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self):
        return "Woof"

class Puppy(Dog):
    pass

print(Puppy.__bases__)                              # (<class '__main__.Dog'>,)
print([c.__name__ for c in Puppy.__mro__])          # ['Puppy', 'Dog', 'Animal', 'object']
print([c.__name__ for c in Animal.__subclasses__()])  # ['Dog']
methods = [name for name, _ in inspect.getmembers(Puppy, inspect.isfunction)]
print(methods)                                      # ['speak']

(<class '__main__.Dog'>,)
['Puppy', 'Dog', 'Animal', 'object']
['Dog']
['speak']


> **Warning:** `getattr`/`setattr` with names that come from users can reach attributes you did not intend (for example `__class__`). Check names against an allow-list.

> **Tip:** In the debugger or REPL, `vars(obj)` and `dir(obj)` are the fastest way to see what an object really contains.

> **Key points**
> * `vars()` shows an object's own data; `dir()` shows everything reachable.
> * `getattr(obj, name, default)` reads attributes by name, safely.
> * `inspect.signature` describes parameters; `__mro__` shows the lookup order of classes.

---

### Practice 6.1: Objects and introspection

**A. Predict the output**

1. What does this print?
   ```python
   print(type(type(3.5)).__name__, isinstance(int, type), isinstance(3, type))
   ```
2. What does this print?
   ```python
   class A:
       x = 1

   a = A()
   a.y = 2
   print(vars(a), hasattr(a, "x"))
   ```
3. What does this print?
   ```python
   def f(a, /, b, *, c=3):
       pass

   import inspect
   print([p.kind.name for p in inspect.signature(f).parameters.values()])
   ```

**B. Write code**

4. Write `public_attrs(obj)` that returns a dict of the instance's own attributes whose names do not start with `_`.
5. Write `call_by_name(obj, name, *args)` that calls a public method chosen by name, and raises `AttributeError` for names starting with `_` or for non-callable attributes.
6. Write `required_params(func)` that returns the names of parameters without default values (ignore `*args` and `**kwargs`).
7. **[Hard]** Write `describe_class(cls)` that returns a dict with keys `"bases"` (names), `"mro"` (names), and `"methods"` (sorted names of functions defined **directly** in the class, not inherited).

**C. Find and fix the bug**

8. This crashes when the attribute is missing. Fix it without `try/except`.
   ```python
   def get_colour(item):
       return getattr(item, "colour")
   ```

**D. Interview questions**

9. What is the difference between `dir(obj)` and `vars(obj)`?
10. **[Hard]** What is introspection used for in real frameworks? Give two examples.

### Solutions 6.1

1. `type True False`.
2. `{'y': 2} True`. `vars` shows only the instance dict; `hasattr` also finds class attributes.
3. `['POSITIONAL_ONLY', 'POSITIONAL_OR_KEYWORD', 'KEYWORD_ONLY']`.
4. to 7. See the code below.
8. Give a default: `getattr(item, "colour", None)`.
9. `vars(obj)` is the object's own `__dict__` (its own data). `dir(obj)` lists all names reachable from the object, including class attributes, methods and inherited dunder methods.
10. Test frameworks find test functions by name (`test_*`) and inspect fixtures' parameters; web frameworks read a view function's signature and type hints to parse request data (FastAPI); `dataclasses` read class annotations to generate `__init__`; ORMs and serializers read attributes to build SQL or JSON.

In [238]:
# Solutions 6.1
import inspect

# Q1-Q3 checks
assert (type(type(3.5)).__name__, isinstance(int, type), isinstance(3, type)) == ("type", True, False)

def f(a, /, b, *, c=3):
    pass

assert [p.kind.name for p in inspect.signature(f).parameters.values()] == [
    "POSITIONAL_ONLY", "POSITIONAL_OR_KEYWORD", "KEYWORD_ONLY"]


# Q4
def public_attrs(obj):
    return {k: v for k, v in vars(obj).items() if not k.startswith("_")}

class Item:
    def __init__(self):
        self.name = "pen"
        self._cost = 4

assert public_attrs(Item()) == {"name": "pen"}


# Q5
def call_by_name(obj, name, *args):
    if name.startswith("_"):
        raise AttributeError(f"private name not allowed: {name}")
    attr = getattr(obj, name)                    # AttributeError if missing
    if not callable(attr):
        raise AttributeError(f"{name} is not a method")
    return attr(*args)

class Calc:
    unit = "cm"

    def add(self, a, b):
        return a + b

assert call_by_name(Calc(), "add", 2, 3) == 5
for bad in ["_secret", "unit", "missing"]:
    try:
        call_by_name(Calc(), bad)
        raise AssertionError("expected AttributeError")
    except AttributeError:
        pass


# Q6
def required_params(func):
    kinds = (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD)
    return [p.name for p in inspect.signature(func).parameters.values()
            if p.default is inspect.Parameter.empty and p.kind not in kinds]

def g(a, b=1, *args, c, d=2, **kw):
    pass

assert required_params(g) == ["a", "c"]
assert required_params(lambda: None) == []


# Q7
def describe_class(cls):
    return {
        "bases": [b.__name__ for b in cls.__bases__],
        "mro": [c.__name__ for c in cls.__mro__],
        "methods": sorted(n for n, v in vars(cls).items() if inspect.isfunction(v)),
    }

class Base:
    def a(self):
        pass

class Child(Base):
    def c(self):
        pass

    def b(self):
        pass

assert describe_class(Child) == {"bases": ["Base"], "mro": ["Child", "Base", "object"], "methods": ["b", "c"]}


# Q8: fixed
def get_colour(item):
    return getattr(item, "colour", None)

assert get_colour(object()) is None
print("Practice 6.1: all checks passed")

Practice 6.1: all checks passed


## 6.3 Attribute lookup

When you write `obj.name`, Python calls `type(obj).__getattribute__(obj, "name")`. The default implementation (from `object`) follows a fixed order. To understand it you need one new word: a **descriptor** is a class attribute that has `__get__` (and maybe `__set__`/`__delete__`). A **data descriptor** has `__set__` or `__delete__` (for example a `property`). A **non-data descriptor** has only `__get__` (for example a plain function). Section 6.4 explains them in detail.

### Lookup order for reading obj.name

```text
        +--------------------------------------+
        | obj.name                             |
        | -> type(obj).__getattribute__(...)   |
        +------------------+-------------------+
                           |
                           v
        +--------------------------------------+  Yes  +----------------------+
        | 1. Found on the class (MRO) as a     |------>| return               |
        |    DATA descriptor (has __set__)?    |       | desc.__get__(obj,cls)|
        +------------------+-------------------+       +----------------------+
                           | No
                           v
        +--------------------------------------+  Yes  +----------------------+
        | 2. "name" in obj.__dict__ ?          |------>| return that value    |
        +------------------+-------------------+       +----------------------+
                           | No
                           v
        +--------------------------------------+  Yes  +----------------------+
        | 3. Found on the class (MRO) as a     |------>| desc.__get__ (e.g.   |
        |    non-data descriptor or a plain    |       | bound method) or the |
        |    class attribute?                  |       | plain value          |
        +------------------+-------------------+       +----------------------+
                           | No
                           v
        +--------------------------------------+  Yes  +----------------------+
        | 4. AttributeError raised. Does the   |------>| return               |
        |    class define __getattr__ ?        |       | __getattr__(obj,name)|
        +------------------+-------------------+       +----------------------+
                           | No
                           v
        +--------------------------------------+
        | AttributeError propagates            |
        +--------------------------------------+
```

**In simple words:** properties win over instance data; instance data wins over methods and class attributes; `__getattr__` is the last resort.

In [239]:
# See the order in action
class Demo:
    shared = "class attribute"

    @property
    def size(self):                          # data descriptor (property has __set__)
        return "from property"

    def greet(self):                         # non-data descriptor (function)
        return "from method"

d = Demo()
d.__dict__["size"] = "from instance dict"    # bypass the property's setter on purpose
d.__dict__["greet"] = lambda: "from instance dict"
d.__dict__["shared"] = "from instance dict"

print(d.size)        # from property        (1. data descriptor wins)
print(d.greet())     # from instance dict   (2. instance dict beats a method)
print(d.shared)      # from instance dict   (2. instance dict beats a class attribute)
print(Demo.shared)   # class attribute

from property
from instance dict
from instance dict
class attribute


### The attribute hooks

| Hook | When it runs | Typical use |
|---|---|---|
| `__getattribute__(self, name)` | On **every** attribute read | Tracing, proxies (rarely needed; easy to cause infinite recursion) |
| `__getattr__(self, name)` | Only when normal lookup **fails** | Delegation, defaults, lazy or computed attributes |
| `__setattr__(self, name, value)` | On every assignment `obj.name = value` | Validation, read-only objects, change tracking |
| `__delattr__(self, name)` | On `del obj.name` | Protecting attributes |

In [240]:
# __getattr__: a fallback only for missing attributes
class Config:
    def __init__(self, **values):
        self.debug = False
        self._values = values

    def __getattr__(self, name):              # called only when normal lookup fails
        try:
            return self._values[name]
        except KeyError:
            raise AttributeError(f"Config has no setting {name!r}") from None

cfg = Config(host="localhost", port=8080)
print(cfg.debug)          # False  (normal attribute; __getattr__ not called)
print(cfg.host, cfg.port) # localhost 8080  (fallback)
try:
    cfg.timeout
except AttributeError as e:
    print("AttributeError:", e)    # AttributeError: Config has no setting 'timeout'

False
localhost 8080
AttributeError: Config has no setting 'timeout'


In [241]:
# Delegation with __getattr__: a wrapper that forwards everything it does not define
class LoggedList:
    def __init__(self):
        self._items = []
        self.log = []

    def append(self, item):                     # defined here: custom behaviour
        self.log.append(f"append {item!r}")
        self._items.append(item)

    def __getattr__(self, name):                # everything else goes to the real list
        return getattr(self._items, name)

    def __len__(self):                          # special methods are looked up on the TYPE,
        return len(self._items)                 # so __getattr__ does not cover len()

orders = LoggedList()
orders.append("o1")
orders.append("o2")
print(orders.count("o1"), orders.index("o2"))   # 1 1   (forwarded to the list)
print(len(orders), orders.log)                  # 2 ["append 'o1'", "append 'o2'"]

1 1
2 ["append 'o1'", "append 'o2'"]


> **Note:** Special method calls made by Python itself (`len(x)`, `x + y`, `for` loops) look up the dunder method on the **type** and skip `__getattr__` and the instance dict. A proxy must define those dunders explicitly.

In [242]:
# __setattr__: validate every assignment (use super().__setattr__ to actually store)
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius                 # goes through __setattr__ too

    def __setattr__(self, name, value):
        if name == "celsius" and value < -273.15:
            raise ValueError("below absolute zero")
        super().__setattr__(name, value)       # NOT self.name = value (infinite recursion)

t = Temperature(25)
t.celsius = 30
print(t.celsius)                    # 30
try:
    t.celsius = -300
except ValueError as e:
    print("ValueError:", e)         # ValueError: below absolute zero

30
ValueError: below absolute zero


In [243]:
# __getattribute__: runs on EVERY read, so always delegate to super()
class Traced:
    def __init__(self):
        self.x = 1

    def __getattribute__(self, name):
        if not name.startswith("__"):
            print(f"  read {name}")
        return super().__getattribute__(name)    # correct way to do the real lookup

t = Traced()
print(t.x)          #   read x   then 1

  read x
1


> **Common mistake:** Inside `__getattribute__`, `__getattr__` or `__setattr__`, using `self.something` can call the same hook again and recurse forever (`RecursionError`). Use `super().__getattribute__(name)` / `super().__setattr__(name, value)`, or `object.__getattribute__(self, name)`.

> **Key points**
> * Reading order: data descriptor on the class, then instance `__dict__`, then non-data descriptor or class attribute, then `__getattr__`.
> * `__getattr__` is a safe fallback; `__getattribute__` intercepts everything and is rarely needed.
> * In `__setattr__`, store values with `super().__setattr__`.

---

## 6.4 Descriptors

A **descriptor** is an object stored as a **class attribute** that controls what happens when the attribute is read, written or deleted on instances. It implements some of:

* `__get__(self, obj, objtype=None)`: on read. `obj` is the instance, or `None` when accessed on the class.
* `__set__(self, obj, value)`: on write.
* `__delete__(self, obj)`: on `del`.
* `__set_name__(self, owner, name)`: called once when the class is created, telling the descriptor its attribute name.

Descriptors are how `property`, methods, `classmethod`, `staticmethod`, `functools.cached_property` and `__slots__` work, and how ORMs (Django, SQLAlchemy) define fields.

**In simple words:** a descriptor is a "smart attribute" that one class attribute provides for every instance.

In [244]:
# A reusable validated attribute: one descriptor class, used for many fields
class Positive:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage = "_" + name             # where the value is stored on the instance

    def __get__(self, obj, objtype=None):
        if obj is None:                       # accessed on the class: return the descriptor
            return self
        return getattr(obj, self.storage)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive, got {value}")
        setattr(obj, self.storage, value)

class Rectangle:
    width = Positive()                        # one descriptor object per field,
    height = Positive()                       # shared by all instances

    def __init__(self, width, height):
        self.width = width                    # calls Positive.__set__
        self.height = height

r = Rectangle(3, 4)
print(r.width * r.height)       # 12
print(vars(r))                  # {'_width': 3, '_height': 4}
try:
    r.height = -1
except ValueError as e:
    print("ValueError:", e)     # ValueError: height must be positive, got -1
print(type(Rectangle.width).__name__)    # Positive

12
{'_width': 3, '_height': 4}
ValueError: height must be positive, got -1
Positive


### Data vs non-data descriptors

| | Data descriptor | Non-data descriptor |
|---|---|---|
| Defines | `__get__` and `__set__` (or `__delete__`) | only `__get__` |
| Priority vs instance `__dict__` | Wins | Loses (instance value hides it) |
| Examples | `property`, `Positive` above, `__slots__` members | functions (methods), `classmethod`, `staticmethod`, `cached_property` |

`functools.cached_property` uses the "loses" rule on purpose: the first read computes the value and stores it in the instance dict; later reads find it there and never call the descriptor again.

In [245]:
# A non-data descriptor: computes once, then the instance dict hides it (like cached_property)
class Lazy:
    def __init__(self, func):
        self.func = func

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.func(obj)
        obj.__dict__[self.name] = value       # next time the instance dict wins
        return value

class Report:
    def __init__(self, rows):
        self.rows = rows

    @Lazy
    def total(self):
        print("  computing total")
        return sum(self.rows)

rep = Report([5, 10, 20])
print(rep.total)      #   computing total   then 35
print(rep.total)      # 35  (no recomputation)

  computing total
35
35


### How property works

`property` is simply a data descriptor class from the standard library. A simplified version written in Python looks like this:

In [246]:
# A minimal re-implementation of property, to show there is no magic
class MyProperty:
    def __init__(self, fget=None, fset=None):
        self.fget = fget
        self.fset = fset

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("read-only attribute")
        self.fset(obj, value)

    def setter(self, fset):
        return MyProperty(self.fget, fset)    # return a new descriptor with the setter

class Circle:
    def __init__(self, radius):
        self._radius = radius

    @MyProperty
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("radius must be >= 0")
        self._radius = value

c = Circle(2)
c.radius = 5            # Circle.radius.__set__(c, 5)
print(c.radius)         # 5
try:
    c.radius = -1
except ValueError as e:
    print("ValueError:", e)    # ValueError: radius must be >= 0

5
ValueError: radius must be >= 0


### How methods become bound methods

A plain function is a non-data descriptor. When you read `obj.method`, `function.__get__(obj, type(obj))` returns a **bound method**: a small object that remembers both the function and the instance. Calling it passes the instance as `self`.

```text
  acct.deposit           ->  Account.__dict__["deposit"].__get__(acct, Account)
                         ->  bound method (function=deposit, self=acct)
  acct.deposit(50)       ->  deposit(acct, 50)
```

In [247]:
# Functions are descriptors; reading through an instance gives a bound method
class Account:
    def __init__(self):
        self.balance = 0

    def deposit(self, amount):
        self.balance += amount
        return self.balance

acct = Account()
func = Account.__dict__["deposit"]                  # the plain function
print(type(func).__name__)                          # function
print(hasattr(func, "__get__"))                     # True - functions are descriptors

bound = func.__get__(acct, Account)                 # what acct.deposit does
print(type(bound).__name__)                         # method
print(bound.__self__ is acct)                       # True
print(bound(50))                                    # 50
print(acct.deposit(25))                             # 75

function
True
method
True
50
75


> **Tip:** Use `@property` for one computed or validated attribute. Write a descriptor class when the **same** behaviour is needed on many attributes or many classes.

> **Key points**
> * A descriptor is a class attribute with `__get__`/`__set__`/`__delete__`; `__set_name__` tells it its name.
> * Data descriptors beat the instance dict; non-data descriptors lose to it.
> * `property`, methods, `classmethod`, `staticmethod` and `cached_property` are all descriptors.

---

### Practice 6.2: Attribute lookup and descriptors

**A. Predict the output**

1. What does this print?
   ```python
   class A:
       x = "class"

       def __getattr__(self, name):
           return "fallback"

   a = A()
   a.y = "instance"
   print(a.x, a.y, a.z)
   ```
2. What does this print?
   ```python
   class A:
       def hello(self):
           return "method"

   a = A()
   a.hello = lambda: "instance"
   print(a.hello(), A().hello())
   ```
3. What does this print?
   ```python
   class A:
       @property
       def v(self):
           return 1

   a = A()
   try:
       a.v = 5
   except AttributeError:
       print("cannot set")
   print(a.v)
   ```

**B. Write code**

4. Write a descriptor `Typed(expected_type)` that raises `TypeError` when a value of the wrong type is assigned. Use it on a `Person` class with `name` (str) and `age` (int).
5. Write a descriptor `Bounded(low, high)` that only accepts numbers in the range `[low, high]`.
6. Write a class `ReadOnlyAfterInit` whose attributes can be set in `__init__` but raise `AttributeError` on any later assignment (use `__setattr__` and a flag).
7. **[Hard]** Write a class `Record` that stores fields in a private dict and supports `rec.field` reads (via `__getattr__`) and `rec.field = value` writes (via `__setattr__`), while keeping a list of changed field names in `rec.changes`.

**C. Find and fix the bug**

8. This raises `RecursionError`. Fix it.
   ```python
   class Logged:
       def __setattr__(self, name, value):
           print("set", name)
           self.__dict__[name] = value if False else value
           setattr(self, name, value)
   ```

**D. Interview questions**

9. What is a descriptor? Name three built-in features that use them.
10. **[Hard]** Explain how `obj.prop = 5` ends up calling a property's setter, and why an instance attribute cannot hide a property.

### Solutions 6.2

1. `class instance fallback`. Only `z` is missing, so only `z` reaches `__getattr__`.
2. `instance method`. Functions are non-data descriptors, so the instance dict wins for `a`; a fresh instance uses the method.
3. `cannot set`, then `1`. A property without a setter is read-only.
4. to 7. See the code below.
8. `setattr(self, name, value)` calls `__setattr__` again, forever. Store with `super().__setattr__(name, value)` (and remove the confusing `__dict__` line).
9. A descriptor is a class attribute that defines `__get__`, `__set__` or `__delete__` to control attribute access on instances. Used by `property`, methods (bound methods), `classmethod`, `staticmethod`, `functools.cached_property`, `__slots__`.
10. Assignment calls `type(obj).__setattr__`, which (in `object`) first looks for a data descriptor named `prop` on the class. `property` defines `__set__`, so Python calls `property.__set__(obj, 5)`, which calls the setter function. Because data descriptors are checked **before** the instance dict for both reading and writing, an instance attribute with the same name cannot hide it.

In [248]:
# Solutions 6.2
# Q1 check
class A:
    x = "class"

    def __getattr__(self, name):
        return "fallback"

a = A()
a.y = "instance"
assert (a.x, a.y, a.z) == ("class", "instance", "fallback")


# Q4
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name
        self.storage = "_" + name

    def __get__(self, obj, objtype=None):
        return self if obj is None else getattr(obj, self.storage)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} must be {self.expected_type.__name__}, "
                            f"got {type(value).__name__}")
        setattr(obj, self.storage, value)

class Person:
    name = Typed(str)
    age = Typed(int)

    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Asha", 30)
assert (p.name, p.age) == ("Asha", 30)
try:
    p.age = "thirty"
    raise AssertionError("expected TypeError")
except TypeError as e:
    assert str(e) == "age must be int, got str"


# Q5
class Bounded:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.name = name
        self.storage = "_" + name

    def __get__(self, obj, objtype=None):
        return self if obj is None else getattr(obj, self.storage)

    def __set__(self, obj, value):
        if not (self.low <= value <= self.high):
            raise ValueError(f"{self.name} must be in [{self.low}, {self.high}]")
        setattr(obj, self.storage, value)

class Exam:
    score = Bounded(0, 100)

    def __init__(self, score):
        self.score = score

assert Exam(0).score == 0 and Exam(100).score == 100
try:
    Exam(101)
    raise AssertionError("expected ValueError")
except ValueError:
    pass


# Q6
class ReadOnlyAfterInit:
    def __init__(self, **fields):
        for key, value in fields.items():
            setattr(self, key, value)
        super().__setattr__("_frozen", True)

    def __setattr__(self, name, value):
        if getattr(self, "_frozen", False):
            raise AttributeError(f"cannot modify {name!r}: object is read-only")
        super().__setattr__(name, value)

point = ReadOnlyAfterInit(x=1, y=2)
assert (point.x, point.y) == (1, 2)
try:
    point.x = 10
    raise AssertionError("expected AttributeError")
except AttributeError:
    pass


# Q7
class Record:
    def __init__(self, **fields):
        super().__setattr__("_fields", dict(fields))
        super().__setattr__("changes", [])

    def __getattr__(self, name):                       # only for names not found normally
        fields = super().__getattribute__("_fields")
        if name in fields:
            return fields[name]
        raise AttributeError(f"Record has no field {name!r}")

    def __setattr__(self, name, value):
        self._fields[name] = value
        self.changes.append(name)

rec = Record(city="Pune")
rec.city = "Delhi"
rec.zip_code = "110001"
assert (rec.city, rec.zip_code, rec.changes) == ("Delhi", "110001", ["city", "zip_code"])
try:
    rec.country
    raise AssertionError("expected AttributeError")
except AttributeError:
    pass


# Q8: fixed
class Logged:
    def __init__(self):
        self.messages = []

    def __setattr__(self, name, value):
        if name != "messages":
            self.messages.append(f"set {name}")
        super().__setattr__(name, value)

obj = Logged()
obj.x = 1
assert obj.x == 1 and obj.messages == ["set x"]
print("Practice 6.2: all checks passed")

Practice 6.2: all checks passed


## 6.5 How classes are created

### What the class statement does

When Python runs a `class` statement it:

1. Runs the class body as code, collecting the names it defines into a namespace dict.
2. Calls the **metaclass** (by default `type`) as `type(name, bases, namespace)` to build the class object.
3. Calls `__set_name__` on descriptors in the namespace, then `__init_subclass__` of the parent class.
4. Applies class decorators (bottom-up) and binds the final result to the class name.

```text
  class Dog(Animal):          1. run body -> namespace {"speak": <function>, ...}
      def speak(self): ...    2. Dog = type("Dog", (Animal,), namespace)
                              3. descriptors' __set_name__, Animal.__init_subclass__(Dog)
                              4. class decorators, then name "Dog" bound to the class
```

In [249]:
# type(name, bases, namespace) creates a class at run time, like a class statement
def describe(self):
    return f"{type(self).__name__} with sides={self.sides}"

Square = type("Square", (object,), {"sides": 4, "describe": describe})

s = Square()
print(s.describe())                 # Square with sides=4
print(type(Square).__name__)        # type

# The equivalent class statement:
class Square2:
    sides = 4

    def describe(self):
        return f"{type(self).__name__} with sides={self.sides}"

print(Square2().describe())         # Square2 with sides=4

Square with sides=4
type
Square2 with sides=4


### Customising class creation: from simplest to most powerful

Choose the **simplest** tool that does the job:

| Tool | Runs when | Good for |
|---|---|---|
| Plain function / factory | You call it | Building objects or classes with settings |
| Class decorator | Right after one class is created | Adding methods, registering, validating one class |
| `__init_subclass__` | Each time a **subclass** is created | Registering or checking all subclasses of a base |
| Descriptor | On attribute access | Per-attribute behaviour (section 6.4) |
| Metaclass | When a class is created, and on class-level operations | Frameworks that must control class creation itself |

In [250]:
# __init_subclass__: register every subclass automatically (plugin pattern)
class Exporter:
    registry = {}

    def __init_subclass__(cls, format_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if format_name is None:
            raise TypeError(f"{cls.__name__} must define format_name")
        Exporter.registry[format_name] = cls

    def export(self, rows):
        raise NotImplementedError

class CsvExporter(Exporter, format_name="csv"):     # class keyword arguments go to __init_subclass__
    def export(self, rows):
        return "\n".join(",".join(map(str, r)) for r in rows)

class TsvExporter(Exporter, format_name="tsv"):
    def export(self, rows):
        return "\n".join("\t".join(map(str, r)) for r in rows)

print(sorted(Exporter.registry))                           # ['csv', 'tsv']
print(Exporter.registry["csv"]().export([[1, 2], [3, 4]])) # 1,2 / 3,4 on two lines

try:
    class BrokenExporter(Exporter):
        pass
except TypeError as e:
    print("TypeError:", e)          # TypeError: BrokenExporter must define format_name

['csv', 'tsv']
1,2
3,4
TypeError: BrokenExporter must define format_name


In [251]:
# A class decorator: add a __repr__ built from the instance attributes
def auto_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{cls.__name__}({fields})"
    cls.__repr__ = __repr__
    return cls

@auto_repr
class Order:
    def __init__(self, order_id, total):
        self.order_id = order_id
        self.total = total

print(Order(7, 250.0))     # Order(order_id=7, total=250.0)

Order(order_id=7, total=250.0)


### Metaclasses

A **metaclass** is the class of a class. A class is created by calling its metaclass, just as an instance is created by calling its class. To write one, subclass `type` and override `__new__` (or `__init__`, or `__call__`).

```text
   instance  <--- created by calling --- class  <--- created by calling --- metaclass
   Dog()                                 Dog                                type (default)
```

Real uses: `abc.ABCMeta` (abstract classes), `enum.EnumMeta`, ORMs that turn class attributes into database columns. In application code, `__init_subclass__` or a class decorator is almost always enough.

In [252]:
# A metaclass that checks every class it creates
class RequireDocstring(type):
    def __new__(mcs, name, bases, namespace):
        if not namespace.get("__doc__"):
            raise TypeError(f"class {name} needs a docstring")
        return super().__new__(mcs, name, bases, namespace)

class Service(metaclass=RequireDocstring):
    """Base class for services."""

class EmailService(Service):          # the metaclass is inherited by subclasses
    """Sends emails."""

print(type(EmailService).__name__)    # RequireDocstring

try:
    class SmsService(Service):
        pass
except TypeError as e:
    print("TypeError:", e)            # TypeError: class SmsService needs a docstring

RequireDocstring
TypeError: class SmsService needs a docstring


In [253]:
# Metaclass __call__ controls instance creation: the classic singleton
class Singleton(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):       # runs on AppConfig(...)
        if cls not in Singleton._instances:
            Singleton._instances[cls] = super().__call__(*args, **kwargs)
        return Singleton._instances[cls]

class AppConfig(metaclass=Singleton):
    def __init__(self, env="prod"):
        self.env = env

a = AppConfig("dev")
b = AppConfig("test")                         # returns the existing object
print(a is b, b.env)                          # True dev

True dev


> **Interview tip:** If asked about singletons in Python, also mention the simpler options: a module-level object (modules are created once and cached), or a function with `functools.cache`. A metaclass singleton is often more than needed.

> **Warning:** A class cannot have two unrelated metaclasses from its bases (you get `TypeError: metaclass conflict`). This is one more reason to avoid custom metaclasses in libraries.

> **Key points**
> * A `class` statement runs the body, then calls the metaclass `type(name, bases, namespace)`.
> * Prefer, in order: function, class decorator, `__init_subclass__`, descriptor, metaclass.
> * Metaclasses subclass `type`; `__new__` controls class creation, `__call__` controls instance creation.

---

### Practice 6.3: Class creation

**A. Predict the output**

1. What does this print?
   ```python
   Point = type("Point", (), {"x": 0})
   p = Point()
   print(Point.__name__, p.x, type(Point) is type)
   ```
2. What does this print?
   ```python
   class Base:
       names = []

       def __init_subclass__(cls, **kwargs):
           super().__init_subclass__(**kwargs)
           Base.names.append(cls.__name__)

   class A(Base):
       pass

   class B(A):
       pass

   print(Base.names)
   ```

**B. Write code**

3. Write a base class `Plugin` that registers every subclass by its `name` class attribute (skip subclasses without a `name`), and a class method `get(name)` that returns the class.
4. Write a class decorator `count_instances` that adds a class attribute `instances` counting how many objects were created (wrap `__init__`).
5. Write a function `make_record_class(name, fields)` that uses `type()` to create a class whose `__init__` takes the fields as keyword arguments and stores them.
6. **[Hard]** Write a metaclass `UpperAttrs` that turns every non-dunder class attribute name into upper case (`rate = 5` becomes `RATE = 5`).

**C. Find and fix the bug**

7. This registry works in a simple hierarchy, but it breaks when a subclass also inherits from another base class that has its own `__init_subclass__`, and it rejects class keyword arguments meant for other bases. What is missing?
   ```python
   class Base:
       registry = []

       def __init_subclass__(cls, **kwargs):
           cls.registry.append(cls)
   ```

**D. Interview questions**

8. What is a metaclass? When would you use one?
9. **[Hard]** Compare a class decorator and `__init_subclass__`: when does each run, and which classes does each affect?

### Solutions 6.3

1. `Point 0 True`.
2. `['A', 'B']`. `__init_subclass__` runs for every subclass, including indirect ones.
3. to 6. See the code below.
7. It does not call `super().__init_subclass__(**kwargs)`. Without that call, the `__init_subclass__` hooks of other classes later in the MRO never run, and keyword arguments meant for them are dropped. Always call `super().__init_subclass__(**kwargs)` (see the fixed version below).
8. A metaclass is the class of a class: calling it creates a class. Use one only when you must control class creation or class-level behaviour that simpler tools cannot handle, for example an ORM that turns attributes into columns, or `abc.ABCMeta`. Prefer `__init_subclass__` or class decorators.
9. A class decorator runs once, right after the decorated class is created, and affects only that class (subclasses are not decorated automatically). `__init_subclass__` is defined on a base class and runs every time a subclass is created, affecting all descendants but not the base itself.

In [254]:
# Solutions 6.3
import functools

# Q1 and Q2 checks
Point = type("Point", (), {"x": 0})
assert (Point.__name__, Point().x, type(Point) is type) == ("Point", 0, True)


# Q3
class Plugin:
    _registry = {}
    name = None

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if cls.name:
            Plugin._registry[cls.name] = cls

    @classmethod
    def get(cls, name):
        return cls._registry[name]

class JsonPlugin(Plugin):
    name = "json"

class XmlPlugin(Plugin):
    name = "xml"

class Helper(Plugin):              # no name: not registered
    pass

assert Plugin.get("xml") is XmlPlugin
assert sorted(Plugin._registry) == ["json", "xml"]


# Q4
def count_instances(cls):
    original_init = cls.__init__

    @functools.wraps(original_init)
    def __init__(self, *args, **kwargs):
        original_init(self, *args, **kwargs)
        cls.instances += 1

    cls.instances = 0
    cls.__init__ = __init__
    return cls

@count_instances
class Ticket:
    def __init__(self, number):
        self.number = number

Ticket(1)
Ticket(2)
assert Ticket.instances == 2


# Q5
def make_record_class(name, fields):
    def __init__(self, **kwargs):
        missing = set(fields) - set(kwargs)
        if missing:
            raise TypeError(f"missing fields: {sorted(missing)}")
        for field in fields:
            setattr(self, field, kwargs[field])
    return type(name, (), {"__init__": __init__, "fields": tuple(fields)})

City = make_record_class("City", ["name", "population"])
c = City(name="Pune", population=7_000_000)
assert (type(c).__name__, c.name, City.fields) == ("City", "Pune", ("name", "population"))


# Q6
class UpperAttrs(type):
    def __new__(mcs, name, bases, namespace):
        new_ns = {}
        for key, value in namespace.items():
            if key.startswith("__"):
                new_ns[key] = value
            else:
                new_ns[key.upper()] = value
        return super().__new__(mcs, name, bases, new_ns)

class Tax(metaclass=UpperAttrs):
    rate = 5

assert Tax.RATE == 5 and not hasattr(Tax, "rate")


# Q7: fixed
class Base:
    registry = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.registry.append(cls)

class Child(Base):
    pass

assert Base.registry == [Child]
print("Practice 6.3: all checks passed")

Practice 6.3: all checks passed


## 6.6 Advanced type hints

Type hints (annotations) describe the expected types for **humans, IDEs and static type checkers** such as `mypy` and `pyright`. Python itself does **not** enforce them at run time.

```text
  your code with hints --> mypy / pyright (before running) --> list of type errors
  your code with hints --> python (running)                 --> hints are ignored
```

Quick reference:

| Hint | Meaning |
|---|---|
| `list[int]`, `dict[str, float]`, `tuple[int, str]`, `tuple[int, ...]` | Built-in generic containers (3.9+) |
| `int \| None` | Either an int or None (3.10+; older code uses `Optional[int]`) |
| `Callable[[int, str], bool]` | A function taking (int, str) and returning bool |
| `Literal["r", "w"]` | Only these exact values |
| `Final` | A constant: must not be reassigned |
| `TypedDict` | A dict with a fixed set of keys and value types |
| `Protocol` | Structural typing: "any object with these methods" |
| `TypeVar` / `[T]` syntax | Generics: a type that is filled in by the caller |
| `Self` (3.11+) | "An instance of the current class" |
| `Any` | Turns checking off for this value |

In [255]:
# Hints are not enforced at run time: Python happily runs this
def add_tax(price: float, rate: float = 0.18) -> float:
    return price * (1 + rate)

print(add_tax(100))                 # 118.0
print(add_tax.__annotations__)      # {'price': <class 'float'>, 'rate': <class 'float'>, 'return': <class 'float'>}

def repeat_text(text: str, times: int) -> str:
    return text * times

print(repeat_text(3, 2))            # 6   - wrong types, no error at run time (mypy would complain)

118.0
{'price': <class 'float'>, 'rate': <class 'float'>, 'return': <class 'float'>}
6


A static checker reports the bad call above **before** the code runs. A typical mypy session (shown as text; install with `python -m pip install mypy`):

```text
$ python -m mypy shop.py
shop.py:7: error: Argument 1 to "repeat_text" has incompatible type "int"; expected "str"  [arg-type]
Found 1 error in 1 file (checked 1 source file)
```

### Generics with TypeVar and the Python 3.12 syntax

A **generic** function or class works with many types while keeping the relationship between them: "returns the same type as the list elements".

In [256]:
# Classic TypeVar style (works on all supported versions)
from typing import TypeVar

T = TypeVar("T")

def first(items: list[T]) -> T:          # if items is list[str], the result is str
    return items[0]

print(first(["a", "b"]), first([10, 20]))   # a 10

a 10


In [257]:
# Python 3.12+ generic syntax (PEP 695): type parameters in square brackets
def last[T](items: list[T]) -> T:
    return items[-1]

class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

s = Stack[int]()
s.push(1)
s.push(2)
print(last([1, 2, 3]), s.pop())        # 3 2
print(last.__type_params__)            # (T,)

# Bounds and constraints
def biggest[N: (int, float)](a: N, b: N) -> N:     # N must be int or float
    return a if a > b else b

print(biggest(3, 7))                   # 7

# The "type" statement creates a type alias (3.12+)
type Pair[A, B] = tuple[A, B]
type UserId = int

p: Pair[str, int] = ("Asha", 1)
print(p, UserId.__value__)             # ('Asha', 1) <class 'int'>

3 2
(T,)
7
('Asha', 1) <class 'int'>


> **Note:** Python 3.13 added default values for type parameters (PEP 696). This syntax does not compile on 3.12, so it is only shown here:
>
> ```python
> class Box[T = int]:      # Python 3.13+
>     ...
> ```

### Protocol: structural typing

A `Protocol` describes "anything that has these methods". A class matches the protocol **without inheriting** from it (duck typing that a checker can verify). With `@runtime_checkable`, `isinstance` checks that the methods exist (it does not check signatures).

In [258]:
# Protocol: any class with a matching method fits, no inheritance needed
from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsArea(Protocol):
    def area(self) -> float: ...

class Square:
    def __init__(self, side: float) -> None:
        self.side = side

    def area(self) -> float:
        return self.side ** 2

class Circle:
    def __init__(self, r: float) -> None:
        self.r = r

    def area(self) -> float:
        return round(3.14159 * self.r ** 2, 2)

def total_area(shapes: list[SupportsArea]) -> float:
    return sum(s.area() for s in shapes)

print(round(total_area([Square(2), Circle(1)]), 2))   # 7.14
print(isinstance(Square(1), SupportsArea))         # True
print(isinstance("text", SupportsArea))            # False

7.14
True
False


In [259]:
# TypedDict, Literal, Final
from typing import TypedDict, NotRequired, Literal, Final

class Movie(TypedDict):
    title: str
    year: int
    rating: NotRequired[float]          # this key may be missing

m: Movie = {"title": "Metropolis", "year": 1927}
print(type(m).__name__)                 # dict  - still a plain dict at run time
print(sorted(Movie.__required_keys__), sorted(Movie.__optional_keys__))   # ['title', 'year'] ['rating']

Mode = Literal["r", "w", "a"]

def open_log(mode: Mode) -> str:
    return f"opened in {mode!r} mode"

print(open_log("a"))                    # opened in 'a' mode

MAX_RETRIES: Final = 3                  # a checker reports any reassignment
print(MAX_RETRIES)                      # 3

dict
['title', 'year'] ['rating']
opened in 'a' mode
3


In [260]:
# Callable and ParamSpec: typing higher-order functions and decorators (3.12 syntax)
import functools
from collections.abc import Callable

def apply(func: Callable[[int], int], value: int) -> int:
    return func(value)

print(apply(lambda n: n + 1, 41))       # 42

# **P captures the parameters, R the return type: the decorated function keeps its signature for checkers
def logged[**P, R](func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        return func(*args, **kwargs)
    return wrapper

@logged
def scale(value: float, factor: float = 2.0) -> float:
    return value * factor

print(scale(1.5))                       # 3.0

42
3.0


In [261]:
# overload, Self and override
from typing import overload, Self, override

@overload
def parse(value: str) -> int: ...
@overload
def parse(value: bytes) -> int: ...
def parse(value):                                  # the real implementation
    return int(value)

print(parse("12"), parse(b"7"))                    # 12 7

class Builder:
    def __init__(self) -> None:
        self.parts: list[str] = []

    def add(self, part: str) -> Self:              # returns the same class (also for subclasses)
        self.parts.append(part)
        return self

class HtmlBuilder(Builder):
    @override                                      # 3.12+: checker verifies a parent method exists
    def add(self, part: str) -> Self:
        return super().add(f"<p>{part}</p>")

print(HtmlBuilder().add("a").add("b").parts)       # ['<p>a</p>', '<p>b</p>']

12 7
['<p>a</p>', '<p>b</p>']


In [262]:
# Narrowing helpers: TypeGuard (3.10+) and TypeIs (3.13+, version-guarded)
import sys
from typing import TypeGuard

def all_strings(items: list[object]) -> TypeGuard[list[str]]:
    return all(isinstance(x, str) for x in items)

values: list[object] = ["a", "b"]
if all_strings(values):
    print(", ".join(values))             # a, b   (checker now knows: list[str])

if sys.version_info >= (3, 13):
    from typing import TypeIs

    def is_int(x: object) -> TypeIs[int]:
        return isinstance(x, int)

    print(is_int(5), is_int("5"))        # True False

a, b
True False


### Annotations are evaluated lazily from Python 3.14

In Python 3.12 and 3.13, annotations are evaluated when the `def` runs, so a hint that names a class defined **later** raises `NameError` (unless you write it as a string `"Order"` or add `from __future__ import annotations`). Python 3.14 evaluates annotations **lazily** (PEP 649 and PEP 749), only when something asks for them. The new `annotationlib` module reads them in different formats.

In [263]:
# Forward references: behaviour depends on the Python version
import sys

source = "def ship(order: Order) -> None: ...\n"     # Order is not defined anywhere yet
namespace = {}
try:
    exec(source, namespace)
    print("defined without error (3.14+ lazy annotations)")
except NameError as e:
    print("NameError (3.12/3.13):", e)

if sys.version_info >= (3, 14):
    import annotationlib
    ship = namespace["ship"]
    print(annotationlib.get_annotations(ship, format=annotationlib.Format.STRING))
    # {'order': 'Order', 'return': 'None'}

defined without error (3.14+ lazy annotations)
{'order': 'Order', 'return': 'None'}


> **Tip:** Use `typing.get_type_hints(obj)` (or `annotationlib.get_annotations` on 3.14+) instead of reading `__annotations__` directly; it resolves string annotations for you.

> **Interview tip:** Say that type hints are for tools, not for the interpreter; if you need run-time validation, use explicit checks or a library such as `pydantic`.

> **Key points**
> * Hints are checked by mypy/pyright, not by Python at run time.
> * Generics: `TypeVar`, or the 3.12 syntax `def f[T](x: T) -> T` and `class Box[T]`; `type Alias = ...` for aliases.
> * `Protocol` = structural typing; `TypedDict` = typed dict keys; `Literal`, `Final`, `Callable`, `ParamSpec`, `Self`, `override` cover most other needs.
> * From 3.14, annotations are evaluated lazily, so forward references just work.

---

### Practice 6.4: Type hints

**A. Predict the output**

1. What does this print?
   ```python
   def double(x: int) -> int:
       return x * 2

   print(double("ab"))
   ```
2. What does this print?
   ```python
   from typing import Protocol, runtime_checkable

   @runtime_checkable
   class Closable(Protocol):
       def close(self) -> None: ...

   class File:
       def close(self):
           pass

   print(isinstance(File(), Closable), isinstance(42, Closable))
   ```

**B. Write code**

3. Write a generic function `chunk[T](items: list[T], size: int) -> list[list[T]]` (3.12 syntax).
4. Write a `TypedDict` called `User` with `name: str`, `age: int` and an optional `email: str`, plus a function `is_adult(user: User) -> bool`.
5. Write a `Protocol` `Drawable` with `draw() -> str` and a function `render_all(items: list[Drawable]) -> list[str]`; show it with two unrelated classes.
6. **[Hard]** Write a generic class `Cache[K, V]` (3.12 syntax) with `get(key: K) -> V | None` and `put(key: K, value: V) -> None`, limited to `capacity` items (drop the oldest key when full).

**C. Find and fix the bug**

7. A teammate expects a `TypeError` here because of the hint. Explain why there is none, and add a run-time check.
   ```python
   def set_age(age: int) -> int:
       return age

   set_age("ten")
   ```

**D. Interview questions**

8. What is the difference between nominal typing (ABCs) and structural typing (Protocols)?
9. **[Hard]** Why are generics useful? Explain `def first[T](items: list[T]) -> T` in plain words.

### Solutions 6.4

1. `abab`. Hints are not enforced; `"ab" * 2` is valid.
2. `True False`.
3. to 6. See the code below.
7. Python ignores type hints at run time; only static checkers such as mypy would report it. Add an explicit check: `if not isinstance(age, int): raise TypeError(...)`.
8. Nominal typing: a class matches only if it inherits from (or registers with) the base, for example an ABC. Structural typing: a class matches if it has the required methods and attributes, no matter what it inherits from. Protocols bring duck typing into static checking.
9. Generics keep track of the relationship between types. `first` works for any element type `T`, and a checker knows that `first(["a"])` returns `str` while `first([1])` returns `int`, instead of losing that information with `Any`.

In [264]:
# Solutions 6.4
from typing import NotRequired, Protocol, TypedDict

# Q1 check
def double(x: int) -> int:
    return x * 2

assert double("ab") == "abab"


# Q3
def chunk[T](items: list[T], size: int) -> list[list[T]]:
    return [items[i:i + size] for i in range(0, len(items), size)]

assert chunk([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]
assert chunk([], 3) == []


# Q4
class User(TypedDict):
    name: str
    age: int
    email: NotRequired[str]

def is_adult(user: User) -> bool:
    return user["age"] >= 18

assert is_adult({"name": "Ravi", "age": 20})
assert not is_adult({"name": "Meera", "age": 15, "email": "m@example.com"})


# Q5
class Drawable(Protocol):
    def draw(self) -> str: ...

class Button:
    def draw(self) -> str:
        return "[button]"

class Label:
    def draw(self) -> str:
        return "label"

def render_all(items: list[Drawable]) -> list[str]:
    return [item.draw() for item in items]

assert render_all([Button(), Label()]) == ["[button]", "label"]


# Q6
class Cache[K, V]:
    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self._data: dict[K, V] = {}

    def get(self, key: K) -> V | None:
        return self._data.get(key)

    def put(self, key: K, value: V) -> None:
        if key not in self._data and len(self._data) >= self.capacity:
            oldest = next(iter(self._data))         # dicts keep insertion order
            del self._data[oldest]
        self._data[key] = value

cache = Cache[str, int](2)
cache.put("a", 1)
cache.put("b", 2)
cache.put("c", 3)
assert cache.get("a") is None and cache.get("c") == 3


# Q7: run-time check added
def set_age(age: int) -> int:
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    return age

try:
    set_age("ten")
    raise AssertionError("expected TypeError")
except TypeError as e:
    assert "got str" in str(e)
print("Practice 6.4: all checks passed")

Practice 6.4: all checks passed


## 6.7 Pattern matching with classes

`match`/`case` (Notebook 1, Part 5) can also **destructure objects**: check the type and pull out attributes in one step.

* `case Point(x=0, y=0):` keyword pattern: the object is a `Point` whose `x` and `y` equal 0.
* `case Point(x, y):` positional pattern: needs `__match_args__` on the class, which tells Python which attributes positions refer to. Dataclasses create `__match_args__` automatically.
* `case Point():` just a type check.
* Guards (`if ...`) and nesting work as usual.

In [265]:
# Class patterns with a dataclass (which provides __match_args__)
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

def where(p):
    match p:
        case Point(x=0, y=0):
            return "origin"
        case Point(x, 0):
            return f"on the x-axis at {x}"
        case Point(x, y) if x == y:
            return "on the diagonal"
        case Point():
            return "somewhere else"
        case _:
            return "not a point"

for value in [Point(0, 0), Point(5, 0), Point(2, 2), Point(1, 2), "hello"]:
    print(where(value))
# origin / on the x-axis at 5 / on the diagonal / somewhere else / not a point

origin
on the x-axis at 5
on the diagonal
somewhere else
not a point


In [266]:
# A normal class needs __match_args__ for positional patterns
class Order:
    __match_args__ = ("status", "total")

    def __init__(self, status, total):
        self.status = status
        self.total = total

def action(order):
    match order:
        case Order("paid", total) if total > 1000:
            return "ship with insurance"
        case Order("paid", _):
            return "ship"
        case Order("cancelled" | "refunded", _):
            return "archive"
        case Order(status=s):
            return f"wait ({s})"

print(action(Order("paid", 5000)))     # ship with insurance
print(action(Order("paid", 20)))       # ship
print(action(Order("refunded", 20)))   # archive
print(action(Order("pending", 20)))    # wait (pending)

ship with insurance
ship
archive
wait (pending)


In [267]:
# Nested class patterns: evaluate a small expression tree
from dataclasses import dataclass

@dataclass
class Num:
    value: float

@dataclass
class Add:
    left: object
    right: object

@dataclass
class Mul:
    left: object
    right: object

def evaluate(node):
    match node:
        case Num(value):
            return value
        case Add(left, right):
            return evaluate(left) + evaluate(right)
        case Mul(Num(0), _) | Mul(_, Num(0)):          # shortcut: anything times zero
            return 0
        case Mul(left, right):
            return evaluate(left) * evaluate(right)
        case _:
            raise TypeError(f"unknown node: {node!r}")

expr = Add(Num(2), Mul(Num(3), Num(4)))      # 2 + 3 * 4
print(evaluate(expr))                        # 14
print(evaluate(Mul(Num(0), expr)))           # 0

14
0


> **Common mistake:** `case Point(x, y)` on a class without `__match_args__` raises `TypeError: Point() accepts 0 positional sub-patterns (2 given)`. Add `__match_args__` or use keyword patterns.

> **Key points**
> * Class patterns check the type and extract attributes at the same time.
> * Positional class patterns use `__match_args__`; dataclasses define it for you.
> * Combine with guards, `|` alternatives and nesting for clear tree-processing code.

---

## 6.8 Memory management and weak references

### Reference counting

In **CPython** (the standard Python), each object keeps a **reference count**: how many names, containers or other objects refer to it. When the count drops to **zero**, the object is freed immediately.

```text
  a = []          refcount of the list = 1   (name a)
  b = a           refcount = 2               (names a and b)
  del a           refcount = 1
  b = None        refcount = 0  -> the list is freed
```

In [268]:
# sys.getrefcount shows the count (the call itself adds a temporary reference)
import sys

data = []
before = sys.getrefcount(data)
alias = data
holder = [data]
print(sys.getrefcount(data) - before)   # 2  (alias and the list "holder")
del alias, holder
print(sys.getrefcount(data) - before)   # 0

2
0


### Reference cycles and the garbage collector

Reference counting cannot free a **cycle**: A refers to B and B refers to A. Even with no outside names, both counts stay above zero. CPython therefore also has a **cyclic garbage collector** (the `gc` module) that runs from time to time and frees unreachable cycles.

```text
     +---------+  other  +---------+
     | node a  |-------->| node b  |
     |         |<--------|         |
     +---------+  other  +---------+
   no outside names point here, but each count is 1 -> needs gc
```

In [269]:
# A cycle survives "del" until the garbage collector runs
import gc
import weakref

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None

gc.disable()                        # keep the demo deterministic (no automatic collection)
try:
    a, b = Node("a"), Node("b")
    a.other, b.other = b, a         # a reference cycle
    watcher = weakref.ref(a)        # watches a without keeping it alive

    del a, b
    print("alive after del:", watcher() is not None)        # alive after del: True
    gc.collect()
    print("alive after gc.collect():", watcher() is not None)  # alive after gc.collect(): False
finally:
    gc.enable()

alive after del: True
alive after gc.collect(): False


> **Warning:** Do not rely on `__del__` (the finaliser method) for cleanup. When it runs depends on the implementation (PyPy has no reference counting) and on cycles. Use `with` statements (Part 5) or `weakref.finalize` instead.

### Weak references

A **weak reference** points to an object **without** increasing its reference count, so it does not keep the object alive. When the object is freed, the weak reference returns `None`. Typical uses: caches, observer lists, parent links in trees.

In [270]:
# weakref.ref, WeakValueDictionary and weakref.finalize
import weakref

class Image:
    def __init__(self, name):
        self.name = name

img = Image("logo.png")
ref = weakref.ref(img)
print(ref() is img)                 # True
del img
print(ref())                        # None - the object is gone

cache = weakref.WeakValueDictionary()  # entries disappear when the value is freed
banner = Image("banner.png")
cache["banner"] = banner
print("banner" in cache)            # True
del banner
print("banner" in cache)            # False

events = []
photo = Image("photo.png")
weakref.finalize(photo, events.append, "photo freed")   # a safer alternative to __del__
del photo
print(events)                       # ['photo freed']

True
None
True
False
['photo freed']


In [271]:
# Not every object supports weak references
import weakref

for value in [42, "text", (1, 2)]:
    try:
        weakref.ref(value)
    except TypeError as e:
        print("TypeError:", e)
# TypeError: cannot create weak reference to 'int' object
# TypeError: cannot create weak reference to 'str' object
# TypeError: cannot create weak reference to 'tuple' object

TypeError: cannot create weak reference to 'int' object
TypeError: cannot create weak reference to 'str' object
TypeError: cannot create weak reference to 'tuple' object


> **Note:** Classes with `__slots__` (Part 2) do not support weak references unless `"__weakref__"` is in the slots.

**Memory tips:**

* Most memory problems in Python are caused by **references you forgot**: global lists that only grow, caches without limits, `lru_cache` on methods.
* Use generators (Part 3) for large data, and `__slots__` for millions of small objects.
* Measure memory with `tracemalloc` (Part 9) before optimising.

> **Key points**
> * CPython frees objects when their reference count reaches zero.
> * The `gc` module frees reference cycles that reference counting cannot.
> * Weak references do not keep objects alive; use them for caches and observers.
> * Use `with` or `weakref.finalize` for cleanup, not `__del__`.

---

## 6.9 The copy protocol

`copy.copy(obj)` makes a **shallow copy** (a new outer object sharing the inner objects); `copy.deepcopy(obj)` copies everything recursively (Notebook 1, Part 10). A class can control both:

* `__copy__(self)`: return the shallow copy.
* `__deepcopy__(self, memo)`: return the deep copy. `memo` is a dict that remembers objects already copied, so shared objects and cycles are copied only once. Pass it on to `copy.deepcopy(child, memo)`.

Use this when some attributes must **not** be copied: a database connection, a lock, a large read-only cache, or an ID that must be unique.

In [272]:
# Custom deep copy: copy the data, share the connection, give a new ID
import copy
import itertools

_ids = itertools.count(1)

class Connection:
    pass

class Session:
    def __init__(self, conn, cart):
        self.id = next(_ids)
        self.conn = conn              # must be shared, never copied
        self.cart = cart              # must be copied

    def __deepcopy__(self, memo):
        new = Session.__new__(Session)              # create without calling __init__
        memo[id(self)] = new                        # register before copying children
        new.id = next(_ids)
        new.conn = self.conn                        # shared on purpose
        new.cart = copy.deepcopy(self.cart, memo)   # pass memo on
        return new

original = Session(Connection(), ["pen", "book"])
clone = copy.deepcopy(original)
clone.cart.append("bag")

print(original.cart, clone.cart)        # ['pen', 'book'] ['pen', 'book', 'bag']
print(clone.conn is original.conn)      # True  - shared
print(original.id, clone.id)            # 1 2

['pen', 'book'] ['pen', 'book', 'bag']
True
1 2


In [273]:
# copy.replace (Python 3.13+): a copy with some fields changed (uses __replace__)
import sys
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Price:
    amount: float
    currency: str

p = Price(100.0, "INR")
print(replace(p, amount=120.0))           # Price(amount=120.0, currency='INR')  (dataclasses.replace, all versions)

if sys.version_info >= (3, 13):
    import copy
    print(copy.replace(p, currency="USD"))  # Price(amount=100.0, currency='USD')

Price(amount=120.0, currency='INR')
Price(amount=100.0, currency='USD')


> **Key points**
> * Implement `__copy__` / `__deepcopy__(memo)` to control what is copied and what is shared.
> * In `__deepcopy__`, register the new object in `memo` and pass `memo` to nested `deepcopy` calls.
> * `dataclasses.replace` (all versions) and `copy.replace` (3.13+) create modified copies of immutable objects.

---

### Practice 6.5: Pattern matching, memory and copying

**A. Predict the output**

1. What does this print?
   ```python
   from dataclasses import dataclass

   @dataclass
   class Pt:
       x: int
       y: int

   match Pt(0, 3):
       case Pt(0, y):
           print("y-axis", y)
       case _:
           print("other")
   ```
2. What does this print?
   ```python
   import weakref

   class A:
       pass

   a = A()
   r = weakref.ref(a)
   b = a
   del a
   print(r() is b)
   ```
3. What does this print?
   ```python
   import copy

   data = {"tags": ["x"]}
   shallow = copy.copy(data)
   shallow["tags"].append("y")
   print(data["tags"])
   ```

**B. Write code**

4. Using `match` and dataclasses `Circle(r)`, `Rect(w, h)` and `Square(side)`, write `area(shape)`; raise `TypeError` for unknown shapes.
5. Write `get_or_create(key, factory)` backed by a `weakref.WeakValueDictionary`: return the existing object while it is alive, otherwise create a new one.
6. Write a class `Document` with a `title` and a list `pages`, where `copy.copy` produces a copy with its **own** list of pages (implement `__copy__`).
7. **[Hard]** Write a function `handle(event)` with `match` that accepts dicts like `{"type": "click", "x": 1, "y": 2}`, `{"type": "key", "key": "q"}` and dataclass `Resize(width, height)` objects, returning a short description string.

**C. Find and fix the bug**

8. This raises `TypeError: Order() accepts 0 positional sub-patterns (1 given)`. Fix it in two different ways.
   ```python
   class Order:
       def __init__(self, status):
           self.status = status

   match Order("paid"):
       case Order("paid"):
           print("paid")
   ```

**D. Interview questions**

9. How does CPython manage memory? What is the role of the `gc` module?
10. **[Hard]** What is the `memo` argument of `__deepcopy__` for?

### Solutions 6.5

1. `y-axis 3`.
2. `True`. `b` still refers to the object, so it is alive.
3. `['x', 'y']`. A shallow copy shares the inner list.
4. to 7. See the code below.
8. Either add `__match_args__ = ("status",)` to the class, or use a keyword pattern: `case Order(status="paid"):`.
9. Mainly reference counting: each object counts its references and is freed as soon as the count reaches zero. Because reference counting cannot free cycles, a generational cyclic garbage collector (`gc`) runs periodically to find and free unreachable cycles. Memory is also reused through internal allocators for small objects.
10. `memo` maps the `id` of objects already copied to their copies. It makes sure an object that appears several times is copied once (shared references stay shared in the copy) and prevents infinite recursion on cycles.

In [274]:
# Solutions 6.5
import copy
import gc
import math
import weakref
from dataclasses import dataclass

# Q4
@dataclass
class Circle:
    r: float

@dataclass
class Rect:
    w: float
    h: float

@dataclass
class Square:
    side: float

def area(shape):
    match shape:
        case Circle(r):
            return math.pi * r * r
        case Rect(w, h):
            return w * h
        case Square(side):
            return side * side
        case _:
            raise TypeError(f"unknown shape: {shape!r}")

assert area(Rect(2, 3)) == 6 and area(Square(4)) == 16
assert math.isclose(area(Circle(1)), math.pi)
try:
    area("triangle")
    raise AssertionError("expected TypeError")
except TypeError:
    pass


# Q5
class Thing:
    def __init__(self, key):
        self.key = key

_instances = weakref.WeakValueDictionary()

def get_or_create(key, factory):
    obj = _instances.get(key)
    if obj is None:
        obj = factory(key)
        _instances[key] = obj
    return obj

t1 = get_or_create("a", Thing)
assert get_or_create("a", Thing) is t1
del t1
gc.collect()
assert "a" not in _instances


# Q6
class Document:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __copy__(self):
        return Document(self.title, list(self.pages))   # new list, same page objects

doc = Document("Guide", ["p1"])
dup = copy.copy(doc)
dup.pages.append("p2")
assert doc.pages == ["p1"] and dup.pages == ["p1", "p2"]


# Q7
@dataclass
class Resize:
    width: int
    height: int

def handle(event):
    match event:
        case {"type": "click", "x": x, "y": y}:
            return f"click at ({x}, {y})"
        case {"type": "key", "key": "q"}:
            return "quit"
        case {"type": "key", "key": k}:
            return f"key {k}"
        case Resize(w, h) if w > 0 and h > 0:
            return f"resize to {w}x{h}"
        case _:
            return "ignored"

assert handle({"type": "click", "x": 1, "y": 2}) == "click at (1, 2)"
assert handle({"type": "key", "key": "q"}) == "quit"
assert handle({"type": "key", "key": "a"}) == "key a"
assert handle(Resize(800, 600)) == "resize to 800x600"
assert handle(Resize(0, 600)) == "ignored"


# Q8: two fixes
class Order:
    __match_args__ = ("status",)

    def __init__(self, status):
        self.status = status

def is_paid(order):
    match order:
        case Order("paid"):                 # fix 1: positional, thanks to __match_args__
            return True
    return False

def is_paid_kw(order):
    match order:
        case Order(status="paid"):          # fix 2: keyword pattern (works without __match_args__)
            return True
    return False

assert is_paid(Order("paid")) and is_paid_kw(Order("paid")) and not is_paid(Order("new"))
print("Practice 6.5: all checks passed")

Practice 6.5: all checks passed


## 6.10 How to debug advanced features

| Symptom or error | Cause | Fix |
|---|---|---|
| `RecursionError: maximum recursion depth exceeded` in `__getattribute__`/`__setattr__`/`__getattr__` | The hook uses `self.x`, which calls the hook again | Use `super().__getattribute__` / `super().__setattr__` |
| `AttributeError: 'X' object has no attribute 'y'` | Name missing, typo, or a `__getattr__` that raises | `dir(obj)`, `vars(obj)`; check spelling and `__getattr__` |
| A property "does nothing" when assigned | No setter defined, or a typo in the setter name | Define `@name.setter` with the **same** name |
| Descriptor value shared between instances | Value stored on the descriptor (`self.value`) instead of on `obj` | Store per instance: `setattr(obj, self.storage, value)` |
| `TypeError: metaclass conflict` | Bases with unrelated metaclasses | Avoid custom metaclasses; use `__init_subclass__` |
| Wrong types accepted silently | Type hints are not enforced | Run mypy/pyright; add explicit checks |
| `TypeError: cannot create weak reference to 'int' object` | Built-in immutables do not support weakrefs | Wrap in a class, or use a normal dict |
| Memory keeps growing | Forgotten references (global lists, unbounded caches) | `tracemalloc`, `gc.get_referrers`, weakrefs |

In [275]:
# Reproduce two classic errors
class Broken:
    def __getattribute__(self, name):
        return self.__dict__[name]          # Common mistake: self.__dict__ calls __getattribute__ again

try:
    Broken().x
except RecursionError as e:
    print("RecursionError:", e)             # RecursionError: maximum recursion depth exceeded

class Fixed:
    def __init__(self):
        self.x = 1

    def __getattribute__(self, name):
        return super().__getattribute__(name)

print(Fixed().x)                            # 1

class Meta1(type):
    pass

class Meta2(type):
    pass

class A(metaclass=Meta1):
    pass

class B(metaclass=Meta2):
    pass

try:
    class C(A, B):
        pass
except TypeError as e:
    print("TypeError:", str(e).split(":")[0])   # TypeError: metaclass conflict

RecursionError: maximum recursion depth exceeded
1
TypeError: metaclass conflict


In [276]:
# Common mistake: storing the value on the descriptor shares it between ALL instances
class SharedValue:
    def __get__(self, obj, objtype=None):
        return self.value

    def __set__(self, obj, value):
        self.value = value                  # stored once, on the descriptor

class Account:
    balance = SharedValue()

a, b = Account(), Account()
a.balance = 100
b.balance = 5
print(a.balance)                            # 5   <- the bug: b overwrote a

# Fix: store the value on the instance (see Positive in section 6.4)
class PerInstance:
    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, obj, objtype=None):
        return getattr(obj, self.storage)

    def __set__(self, obj, value):
        setattr(obj, self.storage, value)

class Account:
    balance = PerInstance()

a, b = Account(), Account()
a.balance = 100
b.balance = 5
print(a.balance, b.balance)                 # 100 5

5
100 5


### Tools for finding out "where does this attribute come from?"

In [277]:
# Find which class in the MRO defines an attribute, and what kind of object it is
import inspect

def find_attribute(obj, name):
    if name in vars(obj):
        return "instance __dict__"
    for cls in type(obj).__mro__:
        if name in vars(cls):
            kind = type(inspect.getattr_static(cls, name)).__name__
            return f"{cls.__name__} ({kind})"
    return "not found (maybe __getattr__)"

class Base:
    def save(self):
        pass

    @property
    def status(self):
        return "ok"

class Child(Base):
    limit = 10

c = Child()
c.note = "x"
for attr in ["note", "limit", "save", "status", "missing"]:
    print(f"{attr:8} -> {find_attribute(c, attr)}")
# note     -> instance __dict__
# limit    -> Child (int)
# save     -> Base (function)
# status   -> Base (property)
# missing  -> not found (maybe __getattr__)

note     -> instance __dict__
limit    -> Child (int)
save     -> Base (function)
status   -> Base (property)
missing  -> not found (maybe __getattr__)


`inspect.getattr_static(obj, name)` finds an attribute **without** triggering descriptors, `__getattr__` or `__getattribute__`. It is very useful when a property or proxy is misbehaving.

```text
        +---------------------------------+
        | Attribute has a surprising value|
        | or raises AttributeError        |
        +----------------+----------------+
                         |
                         v
        +---------------------------------+
        | vars(obj): is it in the         |
        | instance dict?                  |
        +----------------+----------------+
                         |
                         v
        +---------------------------------+
        | Walk type(obj).__mro__ with     |
        | inspect.getattr_static: which   |
        | class defines it, what kind?    |
        +----------------+----------------+
                         |
                         v
        +---------------------------------+
        | Descriptor or hook involved?    |
        | Put a print/breakpoint in       |
        | __get__/__set__/__getattr__     |
        +----------------+----------------+
                         |
                         v
        +---------------------------------+
        | Fix, re-run, add a test for     |
        | the attribute behaviour         |
        +---------------------------------+
```

> **Tip:** For type-related bugs, run `python -m mypy yourfile.py` (or enable pyright in VS Code). In code, `reveal_type(x)` asks the checker to print the type it inferred; at run time (`typing.reveal_type`, 3.11+) it just prints the runtime type.

> **Key points**
> * Recursion in attribute hooks: always delegate with `super()`.
> * Descriptors must store data per instance, not on themselves.
> * `vars`, `__mro__` and `inspect.getattr_static` show where an attribute really comes from.

---

### Practice 6.6: Review and debugging

**A. Predict the output**

1. What does this print?
   ```python
   class Counter:
       count = 0

   a, b = Counter(), Counter()
   a.count += 1
   print(Counter.count, a.count, b.count)
   ```
2. What does this print?
   ```python
   class P:
       @property
       def name(self):
           return "fixed"

   p = P()
   try:
       p.name = "new"
   except AttributeError:
       print("read-only")
   ```

**B. Write code**

3. Write `defined_in(cls, name)` that returns the name of the first class in `cls.__mro__` whose own `__dict__` contains `name`, or `None`.
4. Write a descriptor `LoggedAttribute` that records every read and write as strings like `"get price"` / `"set price=10"` in a list passed to its constructor, storing values per instance.
5. **[Hard]** Write a class decorator `frozen_after_init` that makes instances read-only after `__init__` finishes (assignments then raise `AttributeError`). Do not use dataclasses.

**C. Find and fix the bug**

6. Every `Item` shows the price of the last item created. Fix the descriptor.
   ```python
   class NonNegative:
       def __get__(self, obj, objtype=None):
           return self.value

       def __set__(self, obj, value):
           if value < 0:
               raise ValueError("negative")
           self.value = value
   ```

**D. Interview questions**

7. In what order does Python look up `obj.attr`?
8. **[Hard]** You see `RecursionError` inside a class that defines `__setattr__`. What is the likely cause and the fix?

### Solutions 6.6

1. `0 1 0`. `a.count += 1` reads the class attribute (0) and writes a new **instance** attribute on `a`.
2. `read-only`. The property has no setter.
3. to 5. See the code below.
6. The value is stored on the descriptor, which is shared by all instances. Store it on the instance with `__set_name__` and `setattr(obj, self.storage, value)` (see the fixed version below).
7. Data descriptor on the class (MRO), then the instance `__dict__`, then a non-data descriptor or plain class attribute (MRO), and finally `__getattr__` if defined; otherwise `AttributeError`.
8. `__setattr__` assigns with `self.name = value` (or `setattr(self, ...)`), which calls `__setattr__` again forever. Store with `super().__setattr__(name, value)` or `object.__setattr__(self, name, value)`.

In [278]:
# Solutions 6.6
# Q1 check
class Counter:
    count = 0

a, b = Counter(), Counter()
a.count += 1
assert (Counter.count, a.count, b.count) == (0, 1, 0)


# Q3
def defined_in(cls, name):
    for klass in cls.__mro__:
        if name in vars(klass):
            return klass.__name__
    return None

class Base:
    def run(self):
        pass

class Child(Base):
    pass

assert defined_in(Child, "run") == "Base"
assert defined_in(Child, "__init__") == "object"
assert defined_in(Child, "missing") is None


# Q4
class LoggedAttribute:
    def __init__(self, log):
        self.log = log

    def __set_name__(self, owner, name):
        self.name = name
        self.storage = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        self.log.append(f"get {self.name}")
        return getattr(obj, self.storage)

    def __set__(self, obj, value):
        self.log.append(f"set {self.name}={value}")
        setattr(obj, self.storage, value)

access_log = []

class Product:
    price = LoggedAttribute(access_log)

    def __init__(self, price):
        self.price = price

pen, book = Product(10), Product(250)
assert pen.price == 10 and book.price == 250
assert access_log == ["set price=10", "set price=250", "get price", "get price"]


# Q5
def frozen_after_init(cls):
    original_init = cls.__init__

    def __init__(self, *args, **kwargs):
        object.__setattr__(self, "_initialising", True)
        original_init(self, *args, **kwargs)
        object.__setattr__(self, "_initialising", False)

    def __setattr__(self, name, value):
        if not getattr(self, "_initialising", False):
            raise AttributeError(f"{cls.__name__} is frozen; cannot set {name!r}")
        object.__setattr__(self, name, value)

    cls.__init__ = __init__
    cls.__setattr__ = __setattr__
    return cls

@frozen_after_init
class Money:
    def __init__(self, amount, currency):
        self.amount = amount
        self.currency = currency

m = Money(5, "INR")
assert (m.amount, m.currency) == (5, "INR")
try:
    m.amount = 50
    raise AssertionError("expected AttributeError")
except AttributeError as e:
    assert "frozen" in str(e)


# Q6: fixed
class NonNegative:
    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage)

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError("negative")
        setattr(obj, self.storage, value)

class Item:
    price = NonNegative()

    def __init__(self, price):
        self.price = price

i1, i2 = Item(10), Item(99)
assert (i1.price, i2.price) == (10, 99)
print("Practice 6.6: all checks passed")

Practice 6.6: all checks passed


### Summary of Part 6

* Everything is an object with an identity, a type and a value; classes are instances of `type`.
* Introspection (`type`, `dir`, `vars`, `getattr`, `inspect`) lets code examine objects at run time.
* Attribute lookup order: data descriptor, instance `__dict__`, non-data descriptor or class attribute, then `__getattr__`.
* Descriptors (`__get__`, `__set__`, `__set_name__`) power `property`, methods, `classmethod`, `staticmethod` and reusable validated fields.
* Classes are created by calling the metaclass; prefer class decorators and `__init_subclass__` over custom metaclasses.
* Type hints are for tools (mypy, pyright); use generics (3.12 `[T]` syntax), `Protocol`, `TypedDict`, `Literal`, `Final`, `Callable`, `ParamSpec`.
* `match` can destructure objects through `__match_args__` or keyword patterns.
* CPython frees memory by reference counting plus a cyclic garbage collector; weak references do not keep objects alive.
* `__copy__` and `__deepcopy__(memo)` control copying; `copy.replace` (3.13+) makes modified copies.

---

# Part 7: The Standard Library Toolbox

**What these notes cover**

* The containers in `collections` that save you work every day: `Counter`, `defaultdict`, `deque`, `namedtuple`, `OrderedDict` and `ChainMap`.
* Priority queues with `heapq` (including the new max-heap functions of Python 3.14) and binary search with `bisect`.
* Iteration and function tools: `itertools`, `functools` and `operator`.
* Numbers (`math`, `statistics`, `random`, `secrets`, `decimal`, `fractions`), dates and times (`datetime`, `zoneinfo`, `time`).
* Text patterns with regular expressions (`re`).
* Professional diagnostics with `logging` and `pprint`.
* Command-line programs with `argparse` and `sys`, and running other programs with `subprocess`.
* How to debug the typical errors each module gives you.

**Before you start:** Notebook 1 (especially Part 9 on dictionaries and sets, Part 11 on functions, Part 12 on errors and Part 13 on files) and Parts 3 and 4 of this notebook (iterators, generators and decorators).

## Why the standard library matters

Python is called **"batteries included"**. This means a large amount of ready-made, well-tested code ships with every Python installation. You do not need `pip install` for any module in this Part.

**In simple words:** before you write a helper function, check if the standard library already has it. Library code is faster (much of it is written in C), already tested by millions of users, and other programmers recognise it immediately.

In coding interviews this matters a lot. A solution that uses `Counter`, `deque` or `heapq` is shorter, has fewer bugs, and shows the interviewer that you know your tools.

| Need | Module | Section |
|---|---|---|
| Counting, grouping, queues | `collections` | 7.1 |
| Priority queue, top-k | `heapq` | 7.2 |
| Binary search on a sorted list | `bisect` | 7.3 |
| Combinations, chaining, grouping | `itertools` | 7.4 |
| Caching, partial functions | `functools`, `operator` | 7.5, 7.6 |
| Maths, statistics, randomness, exact numbers | `math`, `statistics`, `random`, `secrets`, `decimal`, `fractions` | 7.7 |
| Dates, times, time zones | `datetime`, `zoneinfo`, `time` | 7.8 |
| Text patterns | `re` | 7.9 |
| Diagnostics | `logging`, `pprint` | 7.10 |
| Command-line programs | `argparse`, `sys` | 7.11 |
| Running other programs | `subprocess` | 7.12 |

Other modules are covered elsewhere: files, `pathlib`, `json`, `csv` and `pickle` in Notebook 1, Part 13; `dataclasses` and `enum` in Part 2 of this notebook; `contextlib` in Part 5; `typing` in Part 6; `threading`, `concurrent.futures` and `asyncio` in Part 8; `unittest` and profiling tools in Part 9.

> **Interview tip:** The tools used most often in coding interviews are `Counter`, `defaultdict`, `deque`, `heapq`, `bisect`, `itertools` and `functools.lru_cache`. Know them by heart.

---

## 7.1 collections: specialised containers

The `collections` module gives you container types that solve common problems better than a plain `dict`, `list` or `tuple`.

| Type | What it is | Typical use |
|---|---|---|
| `Counter` | dict that counts things | frequencies, anagrams, top-k |
| `defaultdict` | dict that creates missing values | grouping, adjacency lists |
| `deque` | list with fast operations at both ends | queues, BFS, sliding windows |
| `namedtuple` | tuple with named fields | small read-only records |
| `OrderedDict` | dict with extra ordering methods | LRU cache |
| `ChainMap` | several dicts viewed as one | layered settings |

### 7.1.1 Counter

A **`Counter`** is a dictionary subclass made for counting **hashable** objects (objects Python can compute a fixed hash number for, such as strings, numbers and tuples). Keys are the items. Values are how many times each item was seen.

**In simple words:** give it a list or a string, and it builds the frequency table for you in O(n) time.

In [279]:
# Counter builds a frequency table in one line
from collections import Counter

letters = Counter("abracadabra")
print(letters)                    # Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1})

fruits = ["apple", "banana", "apple", "cherry", "banana", "apple"]
fruit_counts = Counter(fruits)
print(fruit_counts)               # Counter({'apple': 3, 'banana': 2, 'cherry': 1})

print(fruit_counts["apple"])      # 3
print(fruit_counts["mango"])      # 0  (missing items count as 0, no KeyError)
print("mango" in fruit_counts)    # False (reading a missing key does not add it)

Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1})
Counter({'apple': 3, 'banana': 2, 'cherry': 1})
3
0
False


In [280]:
# most_common(k) returns the k most frequent items, highest first
from collections import Counter

letters = Counter("abracadabra")
print(letters.most_common(2))     # [('a', 5), ('b', 2)]
print(letters.most_common()[-1])  # ('d', 1)  (least common item)

# Ties keep the order in which the items were first seen ('b' before 'r')
print(letters.most_common(3))     # [('a', 5), ('b', 2), ('r', 2)]

[('a', 5), ('b', 2)]
('d', 1)
[('a', 5), ('b', 2), ('r', 2)]


A `Counter` can be updated, and it supports arithmetic. This is very handy for "do I have enough letters?" style questions.

In [281]:
# update() adds counts, subtract() removes counts (and may go to zero or below)
from collections import Counter

stock = Counter(apples=3, pears=1)
stock.update(["apples", "kiwis"])        # add one apple and one kiwi
print(stock)                             # Counter({'apples': 4, 'pears': 1, 'kiwis': 1})

stock.subtract({"pears": 2})
print(stock["pears"])                    # -1  (subtract keeps negative counts)

print(stock.total())                     # 4  (sum of all counts, Python 3.10+)
print(sorted(stock.elements()))          # ['apples', 'apples', 'apples', 'apples', 'kiwis']

Counter({'apples': 4, 'pears': 1, 'kiwis': 1})
-1
4
['apples', 'apples', 'apples', 'apples', 'kiwis']


In [282]:
# Counter arithmetic: +, -, & (minimum), | (maximum)
from collections import Counter

a = Counter("aab")
b = Counter("abc")

print(a + b)    # Counter({'a': 3, 'b': 2, 'c': 1})
print(a - b)    # Counter({'a': 1})          (only positive counts are kept)
print(a & b)    # Counter({'a': 1, 'b': 1})  (minimum of each count)
print(a | b)    # Counter({'a': 2, 'b': 1, 'c': 1})  (maximum of each count)

Counter({'a': 3, 'b': 2, 'c': 1})
Counter({'a': 1})
Counter({'a': 1, 'b': 1})
Counter({'a': 2, 'b': 1, 'c': 1})


In [283]:
# Three classic interview checks written with Counter
from collections import Counter

# 1. Valid anagram: same letters with the same counts
print(Counter("listen") == Counter("silent"))      # True

# 2. Ransom note: can we build the note from the magazine letters?
note, magazine = "aab", "baa c"
print(not (Counter(note) - Counter(magazine)))     # True (nothing is missing)

# 3. First unique character: count first, then scan in order
text = "leetcode"
counts = Counter(text)
first = next((i for i, ch in enumerate(text) if counts[ch] == 1), -1)
print(first)                                       # 0  ('l' appears once)

True
True
0


> **Note:** Since Python 3.10, two counters are equal when every item has the same count, and missing items count as zero. Counters also support `<=` and `>=` as "is a sub-multiset of": `Counter("ab") <= Counter("aabc")` is `True`.

> **Interview tip:** Building a `Counter` is O(n). `most_common(k)` uses a heap internally, so it costs about O(n log k). Calling `most_common()` with no argument sorts everything: O(n log n).

### 7.1.2 defaultdict

A **`defaultdict`** is a dictionary that creates a value automatically when you look up a missing key.

You give it a **factory**: a function that is called with no arguments to make the missing value. `int()` returns `0`, `list()` returns `[]`, `set()` returns `set()`.

```text
  dd = defaultdict(list)

  dd["fruit"].append("apple")
        |
        v
  +--------------------+   No   +---------------------------+
  | "fruit" in dd ?    |------->| dd["fruit"] = list()  ->[]|
  +---------+----------+        +-------------+-------------+
            | Yes                             |
            v                                 v
  +-----------------------------------------------------+
  | return dd["fruit"], then .append("apple") runs on it |
  +-----------------------------------------------------+
```

In [284]:
# defaultdict(int): counting without checking "if key in d"
from collections import defaultdict

word_count = defaultdict(int)
for word in ["apple", "banana", "apple", "cherry", "banana", "apple"]:
    word_count[word] += 1          # a missing word starts at int() == 0

print(dict(word_count))            # {'apple': 3, 'banana': 2, 'cherry': 1}

{'apple': 3, 'banana': 2, 'cherry': 1}


In [285]:
# defaultdict(list): grouping items under a key
from collections import defaultdict

pairs = [("fruit", "apple"), ("veggie", "carrot"), ("fruit", "banana")]

groups = defaultdict(list)
for category, item in pairs:
    groups[category].append(item)  # a missing category starts as []

print(dict(groups))                # {'fruit': ['apple', 'banana'], 'veggie': ['carrot']}

{'fruit': ['apple', 'banana'], 'veggie': ['carrot']}


In [286]:
# defaultdict(list) is the standard way to build a graph's adjacency list
from collections import defaultdict

edges = [("A", "B"), ("A", "C"), ("B", "D")]

graph = defaultdict(list)
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)             # undirected graph: add both directions

print(dict(graph))                 # {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A'], 'D': ['B']}

{'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A'], 'D': ['B']}


In [287]:
# Nested defaultdict: the factory can be any function, even a lambda
from collections import defaultdict

stats = defaultdict(lambda: defaultdict(int))
stats["asha"]["clicks"] += 1
stats["asha"]["clicks"] += 1
stats["ravi"]["views"] += 1

print({user: dict(inner) for user, inner in stats.items()})
# {'asha': {'clicks': 2}, 'ravi': {'views': 1}}

{'asha': {'clicks': 2}, 'ravi': {'views': 1}}


In [288]:
# Common mistake: simply READING a missing key creates it
from collections import defaultdict

dd = defaultdict(list)
print(dd["ghost"])           # []  (the key 'ghost' was created by this read)
print(dict(dd))              # {'ghost': []}

# Fix: use "in" or .get() to check, they never create keys
print("other" in dd)         # False
print(dd.get("other"))       # None
print(sorted(dd))            # ['ghost']  ('other' was not added)

[]
{'ghost': []}
False
None
['ghost']


> **Tip:** `dict.setdefault(key, []).append(x)` does the same job on a normal dict, but `defaultdict` is shorter and clearer when you do it many times.

> **Warning:** Convert to a normal dict (`dict(dd)`) before returning it from an API. Otherwise the caller may create keys by accident when reading.

### 7.1.3 deque (double-ended queue)

A **`deque`** (say "deck") is a sequence where adding or removing at **either end** is O(1). A list is fast only at the right end. `list.pop(0)` and `list.insert(0, x)` must shift every other element, so they are O(n).

```text
            appendleft / popleft          append / pop
                    |                          |
                    v                          v
              +----+----+----+----+----+----+
     left --> |  0 |  1 |  2 |  3 |  4 |  5 | <-- right
              +----+----+----+----+----+----+
     Both ends are O(1). Indexing in the middle is O(n).
```

In [289]:
# Basic deque operations at both ends
from collections import deque

dq = deque([1, 2, 3])
dq.append(4)              # add to the right
dq.appendleft(0)          # add to the left
print(dq)                 # deque([0, 1, 2, 3, 4])

print(dq.pop())           # 4  (remove from the right)
print(dq.popleft())       # 0  (remove from the left)
print(dq)                 # deque([1, 2, 3])

print(dq[0], dq[-1])      # 1 3  (peeking at the ends is O(1))

deque([0, 1, 2, 3, 4])
4
0
deque([1, 2, 3])
1 3


In [290]:
# deque as a FIFO queue: first in, first out
from collections import deque

queue = deque()
for customer in ["Asha", "Ravi", "Meera"]:
    queue.append(customer)          # join at the back

while queue:
    print("serving", queue.popleft())  # leave from the front
# serving Asha
# serving Ravi
# serving Meera

serving Asha
serving Ravi
serving Meera


In [291]:
# maxlen: a bounded deque drops the oldest item automatically
from collections import deque

recent = deque(maxlen=3)
for page in ["home", "shop", "cart", "pay", "done"]:
    recent.append(page)

print(recent)             # deque(['cart', 'pay', 'done'], maxlen=3)
print(list(recent))       # ['cart', 'pay', 'done']  (last 3 pages visited)

deque(['cart', 'pay', 'done'], maxlen=3)
['cart', 'pay', 'done']


In [292]:
# rotate() and extendleft()
from collections import deque

d = deque([1, 2, 3, 4, 5])
d.rotate(2)               # move 2 items from the right end to the left
print(d)                  # deque([4, 5, 1, 2, 3])
d.rotate(-2)              # negative: rotate to the left
print(d)                  # deque([1, 2, 3, 4, 5])

# extendleft adds items one by one to the left, so their order is reversed
d.extendleft([0, -1])
print(d)                  # deque([-1, 0, 1, 2, 3, 4, 5])

deque([4, 5, 1, 2, 3])
deque([1, 2, 3, 4, 5])
deque([-1, 0, 1, 2, 3, 4, 5])


**Breadth-first search (BFS)** visits a graph level by level. It always uses a queue, and `deque.popleft()` is the right tool. Full BFS is in Notebook 3, Part 14.

```text
  +--------------------------+
  | queue = deque([start])   |
  +------------+-------------+
               v
  +--------------------------+   empty   +------+
  | while queue:             |---------->| done |
  +------------+-------------+           +------+
               v
  +--------------------------+
  | node = queue.popleft()   |
  | visit node               |
  | append unseen neighbours |
  +------------+-------------+
               |
               +----> back to "while queue"
```

In [293]:
# BFS on a small graph with deque
from collections import deque

graph = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}

def bfs_order(graph, start):
    seen = {start}
    order = []
    queue = deque([start])
    while queue:
        node = queue.popleft()            # O(1); list.pop(0) would be O(n)
        order.append(node)
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return order

print(bfs_order(graph, "A"))              # ['A', 'B', 'C', 'D']

['A', 'B', 'C', 'D']


| Operation | list | deque |
|---|---|---|
| append / pop at the right | O(1) | O(1) |
| insert / pop at the left | O(n) | O(1) |
| index `x[i]` near the ends | O(1) | O(1) |
| index `x[i]` in the middle | O(1) | O(n) |
| slicing `x[a:b]` | supported | not supported |

> **Tip:** Use a list when you need fast random access or slicing. Use a deque when you add and remove at the front.

> **Note:** `append` and `popleft` on a deque are thread-safe, but for producer/consumer code between threads prefer `queue.Queue` (see Part 8).

### 7.1.4 namedtuple

A **`namedtuple`** creates a small tuple class whose fields have names. It is still a tuple: immutable, hashable (if its fields are hashable), unpackable, and light in memory.

In [294]:
# namedtuple: a tuple whose fields can be read by name
from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)

print(p)                   # Point(x=3, y=4)
print(p.x, p[1])           # 3 4  (by name or by index)

x, y = p                   # unpacks like a normal tuple
print(x + y)               # 7

print(p._asdict())         # {'x': 3, 'y': 4}
print(p._replace(x=10))    # Point(x=10, y=4)  (a NEW tuple; p is unchanged)

Point(x=3, y=4)
3 4
7
{'x': 3, 'y': 4}
Point(x=10, y=4)


In [295]:
# namedtuple is immutable, and fields can have defaults
from collections import namedtuple

Employee = namedtuple("Employee", ["name", "dept", "level"], defaults=["junior"])
e = Employee("Meera", "data")
print(e)                   # Employee(name='Meera', dept='data', level='junior')

try:
    e.level = "senior"     # tuples cannot be changed
except AttributeError as err:
    print("AttributeError:", err)

Employee(name='Meera', dept='data', level='junior')
AttributeError: can't set attribute


In [296]:
# typing.NamedTuple: the same thing written as a class, with type hints
from typing import NamedTuple

class Order(NamedTuple):
    order_id: int
    amount: float
    status: str = "new"

o = Order(101, 250.0)
print(o)                   # Order(order_id=101, amount=250.0, status='new')
print(o.amount * 2)        # 500.0

Order(order_id=101, amount=250.0, status='new')
500.0


> **Tip:** For a record that must change, or that needs methods and validation, use a `@dataclass` instead (see Part 2). Use a named tuple for small, fixed records such as points, rows, or return values with several parts.

### 7.1.5 OrderedDict and an LRU cache

Since Python 3.7 a normal `dict` keeps insertion order. So why does `OrderedDict` still exist? It has two extra tools that a plain dict lacks:

* `move_to_end(key, last=True)`: move a key to the end (or to the front with `last=False`) in O(1).
* `popitem(last=False)`: remove the **first** item in O(1).

It also compares order-sensitively: two `OrderedDict` objects with the same items in a different order are not equal.

In [297]:
# OrderedDict extras: move_to_end and popitem(last=False)
from collections import OrderedDict

od = OrderedDict(a=1, b=2, c=3)
od.move_to_end("a")                 # a becomes the newest
print(list(od))                     # ['b', 'c', 'a']

od.move_to_end("c", last=False)     # c moves to the front
print(list(od))                     # ['c', 'b', 'a']

print(od.popitem(last=False))       # ('c', 3)  (removes the OLDEST item)

# Order matters for equality between OrderedDicts, but not for plain dicts
print(OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1))   # False
print(dict(a=1, b=2) == dict(b=2, a=1))                 # True

['b', 'c', 'a']
['c', 'b', 'a']
('c', 3)
False
True


An **LRU cache** (least recently used) keeps a fixed number of items. When it is full and a new item arrives, it throws away the item that was used least recently. This is a very common interview design question.

```text
  get(key) / put(key, value)
             |
             v
  +----------------------+
  | move key to the END  |   END = most recently used
  +----------+-----------+
             v
  +----------------------+   No    +------+
  | size > capacity ?    |-------->| done |
  +----------+-----------+         +------+
             | Yes
             v
  +----------------------+
  | popitem(last=False)  |   FRONT = least recently used
  +----------------------+
```

In [298]:
# LRU cache with OrderedDict: get and put are both O(1)
from collections import OrderedDict

class LRUCache:
    """Keep at most `capacity` items; evict the least recently used one."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)          # mark as most recently used
        return self.data[key]

    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)   # evict the least recently used

cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")              # "a" is now the most recently used
cache.put("c", 3)           # cache is full, so "b" is evicted

print(list(cache.data))     # ['a', 'c']
print(cache.get("b"))       # -1

['a', 'c']
-1


> **Interview tip:** Some interviewers ask you to build the LRU cache without `OrderedDict`, using a dict plus a doubly linked list. That version is in Notebook 3, Part 9. Mention that `OrderedDict` is itself built that way.

### 7.1.6 ChainMap

A **`ChainMap`** groups several dictionaries and searches them in order, without copying them. It is perfect for layered settings: command-line options first, then environment settings, then defaults.

In [299]:
# ChainMap: look up keys in several dicts, first match wins
from collections import ChainMap

defaults = {"theme": "light", "language": "en", "page_size": 20}
user_settings = {"theme": "dark"}
cli_options = {"page_size": 50}

settings = ChainMap(cli_options, user_settings, defaults)
print(settings["theme"])        # dark  (from user_settings)
print(settings["page_size"])    # 50    (from cli_options)
print(settings["language"])     # en    (from defaults)

settings["language"] = "hi"     # writes always go to the FIRST dict
print(cli_options)              # {'page_size': 50, 'language': 'hi'}
print(defaults["language"])     # en  (unchanged)

dark
50
en
{'page_size': 50, 'language': 'hi'}
en


### Common mistakes with collections

* **Reading a missing key of a `defaultdict` creates it.** Use `key in dd` or `dd.get(key)` to check.
* **Using `list.pop(0)` for a queue.** It is O(n). Use `deque.popleft()`.
* **Expecting `Counter.most_common()` to break ties alphabetically.** Ties keep first-seen order. Sort with a key like `(-count, word)` if you need alphabetical ties.
* **Trying to change a namedtuple field.** Use `_replace()` to get a new tuple, or use a dataclass.
* **`deque.extendleft([1, 2])` gives `2, 1` at the front**, because items are added one at a time.
* **Slicing a deque** (`dq[1:3]`) raises `TypeError`. Use `itertools.islice` or convert to a list.

### How to debug collections code

| Error or symptom | What it means | Fix |
|---|---|---|
| A `defaultdict` has keys you never inserted | some code read `dd[key]` to check it | check with `in` or `.get()` |
| `TypeError: unhashable type: 'list'` | a list was used as a key or counted | convert to a `tuple` first |
| `TypeError: sequence index must be integer, not 'slice'` | you sliced a deque | `list(islice(dq, a, b))` |
| `AttributeError: can't set attribute` | you assigned to a namedtuple field | use `_replace()` |
| Program is slow with a queue | `list.pop(0)` is O(n) | use `deque.popleft()` |

In [300]:
# Seeing the real error messages helps you recognise them later
from collections import Counter, deque

try:
    Counter([[1, 2], [3]])            # lists are not hashable
except TypeError as e:
    print("TypeError:", e)            # TypeError: unhashable type: 'list'

try:
    deque([1, 2, 3])[0:2]             # deques do not support slicing
except TypeError as e:
    print("TypeError:", e)

print(Counter(map(tuple, [[1, 2], [3], [1, 2]])))   # Counter({(1, 2): 2, (3,): 1})

TypeError: unhashable type: 'list'
TypeError: sequence index must be integer, not 'slice'
Counter({(1, 2): 2, (3,): 1})


> **Key points**
> * `Counter` counts in O(n); missing items count as 0; `most_common(k)` gives the top k.
> * `defaultdict(factory)` creates missing values; reading a missing key creates it.
> * `deque` gives O(1) at both ends; use it for queues, BFS and sliding windows; `maxlen` keeps the last N items.
> * `namedtuple` / `typing.NamedTuple` give readable, immutable records.
> * `OrderedDict.move_to_end` and `popitem(last=False)` make an O(1) LRU cache; `ChainMap` layers dicts.

### Practice 7.1: collections

**A. Predict the output**

1. What does this print?
   ```python
   from collections import Counter

   print(Counter("banana").most_common(1))
   ```
2. What does this print?
   ```python
   from collections import defaultdict

   dd = defaultdict(list)
   if dd["a"]:
       print("has items")
   print(len(dd))
   ```
3. What does this print?
   ```python
   from collections import deque

   d = deque([1, 2, 3], maxlen=3)
   d.append(4)
   d.appendleft(0)
   print(list(d))
   ```
4. What does this print?
   ```python
   from collections import Counter

   print(Counter("aab") - Counter("abb"))
   ```

**B. Write code**

5. Write `is_anagram(s, t)` that returns `True` if `t` is an anagram of `s`. Use `Counter`.
6. Write `group_anagrams(words)` that returns a list of groups, each group being words with the same letters, e.g. `["eat", "tea", "tan", "ate", "nat"]` gives `[["eat", "tea", "ate"], ["tan", "nat"]]`. Use `defaultdict(list)`.
7. Write `last_n_lines(lines, n)` that returns the last `n` lines of an iterable of lines, reading it only once. Use `deque(maxlen=n)`.
8. Write `top_k_words(words, k)` that returns the `k` most frequent words; ties are broken alphabetically.
9. **[Hard]** Write `moving_average(data, k)` that returns the average of every window of `k` consecutive numbers in O(n) total time, using a `deque` and a running sum. `moving_average([1, 2, 3, 4, 5], 3)` gives `[2.0, 3.0, 4.0]`.
10. **[Hard]** Extend the `LRUCache` idea: write `LRU` with `get`, `put` and `__len__`, and test that the eviction order is correct after a mix of gets and puts.

**C. Find and fix the bug**

11. This should count how many words start with each letter, but it is slow on large input and the result is wrong for empty strings. Find both problems.
    ```python
    def first_letters(words):
        counts = {}
        for w in words:
            if w[0] in counts.keys():
                counts[w[0]] += 1
            else:
                counts[w[0]] = 1
        return counts
    ```
12. This BFS gives the right answer but is O(n^2) on long queues. Why? Fix it.
    ```python
    def bfs(graph, start):
        queue = [start]
        seen = {start}
        order = []
        while queue:
            node = queue.pop(0)
            order.append(node)
            for nxt in graph[node]:
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append(nxt)
        return order
    ```

**D. Interview questions**

13. Why is a `deque` better than a `list` for a queue? Is there anything a list does better?
14. When would you choose `Counter` over `defaultdict(int)`?
15. **[Hard]** How does an LRU cache get O(1) for both `get` and `put`? Which data structures are involved inside `OrderedDict`?

### Solutions 7.1

**A.**
1. `[('a', 3)]`. The letter `a` appears three times.
2. It prints only `1`. The `if dd["a"]` reads a missing key, which creates `dd["a"] = []`. The empty list is falsy, so "has items" is not printed, but the key now exists.
3. `[0, 2, 3]`. After `append(4)` the deque is `[2, 3, 4]` (1 fell off the left). `appendleft(0)` makes it `[0, 2, 3]` (4 falls off the right, because a full deque drops items from the opposite end).
4. `Counter({'a': 1})`. `a`: 2 - 1 = 1, `b`: 1 - 2 = -1 (dropped, only positive counts stay).

**C.**
11. `w[0]` raises `IndexError` for an empty string, so skip empty words. The `.keys()` call is not slow in Python 3 (it is a view), but the whole if/else is unnecessary: `Counter(w[0] for w in words if w)` does it in one line.
12. `list.pop(0)` shifts all remaining items, so each pop is O(n). Use `collections.deque` and `popleft()`.

**D.**
13. `deque.popleft()` and `appendleft()` are O(1); on a list they are O(n). A list is better for random access in the middle (O(1) versus O(n)) and supports slicing.
14. `Counter` when you need counting extras: `most_common`, arithmetic between counters, `total()`, and missing items read as 0 without being inserted. `defaultdict(int)` is fine for simple counting, and `defaultdict(list)` or `defaultdict(set)` for grouping.
15. A hash map gives O(1) lookup by key, and a doubly linked list keeps the usage order: moving a node to the end and removing the first node are O(1) because you have direct pointers. `OrderedDict` combines both.

In [301]:
# Solutions 7.1
from collections import Counter, defaultdict, deque, OrderedDict

# A: checks
assert Counter("banana").most_common(1) == [("a", 3)]
dd = defaultdict(list)
if dd["a"]:
    pass
assert len(dd) == 1
d = deque([1, 2, 3], maxlen=3)
d.append(4)
d.appendleft(0)
assert list(d) == [0, 2, 3]
assert Counter("aab") - Counter("abb") == Counter({"a": 1})

# 5. Anagram check: O(n) time, O(k) space for k distinct letters
def is_anagram(s, t):
    return Counter(s) == Counter(t)

assert is_anagram("listen", "silent")
assert not is_anagram("rat", "car")
assert is_anagram("", "")

# 6. Group anagrams: the sorted letters are the group key
def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())

assert group_anagrams(["eat", "tea", "tan", "ate", "nat"]) == [["eat", "tea", "ate"], ["tan", "nat"]]
assert group_anagrams([]) == []

# 7. Last n lines with a bounded deque: O(n) memory, one pass
def last_n_lines(lines, n):
    return list(deque(lines, maxlen=n))

assert last_n_lines(iter(["l1", "l2", "l3", "l4"]), 2) == ["l3", "l4"]
assert last_n_lines([], 3) == []

# 8. Top k words, ties alphabetical: sort by (-count, word)
def top_k_words(words, k):
    counts = Counter(words)
    return sorted(counts, key=lambda w: (-counts[w], w))[:k]

assert top_k_words(["b", "a", "b", "c", "a", "d"], 2) == ["a", "b"]
assert top_k_words(["x"], 5) == ["x"]

# 9. Moving average: each number enters and leaves the window once -> O(n)
def moving_average(data, k):
    window = deque()
    running = 0
    result = []
    for x in data:
        window.append(x)
        running += x
        if len(window) > k:
            running -= window.popleft()
        if len(window) == k:
            result.append(running / k)
    return result

assert moving_average([1, 2, 3, 4, 5], 3) == [2.0, 3.0, 4.0]
assert moving_average([1, 2], 3) == []
assert moving_average([-2, 2], 1) == [-2.0, 2.0]

# 10. LRU with __len__
class LRU:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return None
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)

    def __len__(self):
        return len(self.data)

lru = LRU(2)
lru.put(1, "a")
lru.put(2, "b")
lru.get(1)              # 1 is now most recent, so 2 is the oldest
lru.put(3, "c")         # evicts 2
assert lru.get(2) is None
assert lru.get(1) == "a" and lru.get(3) == "c"
assert len(lru) == 2

# 11. Fixed first_letters
def first_letters(words):
    return Counter(w[0] for w in words if w)

assert first_letters(["apple", "avocado", "", "banana"]) == {"a": 2, "b": 1}

# 12. Fixed BFS with deque
def bfs(graph, start):
    queue = deque([start])
    seen = {start}
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return order

assert bfs({1: [2, 3], 2: [4], 3: [4], 4: []}, 1) == [1, 2, 3, 4]
print("Practice 7.1: all checks passed")

Practice 7.1: all checks passed


---

## 7.2 heapq: priority queues

A **priority queue** is a collection where you can always take out the **smallest** (or most important) item quickly. The `heapq` module builds a priority queue on top of a normal Python list, called a **heap**.

A **min-heap** is a list that follows one rule: every parent is less than or equal to its children. The children of index `i` are at `2*i + 1` and `2*i + 2`. So the smallest item is always at index 0.

```text
  heap list:  [1, 3, 9, 7, 5]
  index:       0  1  2  3  4

  as a tree:            1          (index 0, the minimum)
                      /   \
                     3     9       (index 1, 2)
                    / \
                   7   5           (index 3, 4)

  Rule: parent <= children. The list is NOT fully sorted.
```

| Operation | Function | Time |
|---|---|---|
| Turn a list into a heap (in place) | `heapq.heapify(h)` | O(n) |
| Add an item | `heapq.heappush(h, x)` | O(log n) |
| Remove and return the smallest | `heapq.heappop(h)` | O(log n) |
| Look at the smallest | `h[0]` | O(1) |
| Pop then push in one step | `heapq.heapreplace(h, x)` | O(log n) |
| Push then pop in one step | `heapq.heappushpop(h, x)` | O(log n) |

In [302]:
# Min-heap basics: heapify, heappush, heappop, peek with h[0]
import heapq

nums = [5, 7, 9, 1, 3]
heapq.heapify(nums)                  # O(n), rearranges the list in place
print(nums)                          # [1, 3, 9, 7, 5]  (a heap, not a sorted list)

heapq.heappush(nums, 2)
print(nums[0])                       # 1  (peek at the smallest)

print(heapq.heappop(nums))           # 1
print(heapq.heappop(nums))           # 2
print(nums[0])                       # 3

[1, 3, 9, 7, 5]
1
1
2
3


In [303]:
# Popping everything gives sorted order: this is heap sort, O(n log n)
import heapq

h = [5, 3, 8, 1, 9, 2]
heapq.heapify(h)
print([heapq.heappop(h) for _ in range(len(h))])   # [1, 2, 3, 5, 8, 9]

[1, 2, 3, 5, 8, 9]


### 7.2.1 Max-heaps

For a long time `heapq` only had public min-heap functions. **Python 3.14** added public max-heap functions: `heapify_max`, `heappush_max`, `heappop_max`, `heapreplace_max` and `heappushpop_max`.

On older versions (and in many interview environments, which may run 3.10 to 3.13) use the **negation trick**: push `-x`, and negate again when you pop. The smallest negative number is the largest original number.

In [304]:
# Max-heap: Python 3.14 functions, with the negation trick as a fallback
import heapq
import sys

values = [5, 7, 9, 1, 3]

if sys.version_info >= (3, 14):
    max_heap = values.copy()
    heapq.heapify_max(max_heap)           # Python 3.14+
    heapq.heappush_max(max_heap, 8)
    print(max_heap[0])                    # 9  (peek at the largest)
    print(heapq.heappop_max(max_heap))    # 9
    print(heapq.heappop_max(max_heap))    # 8
else:
    print("Python < 3.14: use the negation trick below")

9
9
8


In [305]:
# The negation trick works on every Python version
import heapq

values = [5, 7, 9, 1, 3]
neg_heap = [-v for v in values]
heapq.heapify(neg_heap)

heapq.heappush(neg_heap, -8)
print(-neg_heap[0])                  # 9  (peek: negate back)
print(-heapq.heappop(neg_heap))      # 9
print(-heapq.heappop(neg_heap))      # 8

9
9
8


> **Warning:** The negation trick only works for numbers. For strings or other objects, wrap them in a small class with a reversed `__lt__`, or (3.14+) use the `_max` functions.

### 7.2.2 Tuples as (priority, item)

A heap compares its items with `<`. Tuples compare element by element, so a tuple `(priority, item)` is ordered by priority first.

In [306]:
# A task queue ordered by priority (smaller number = more urgent)
import heapq

tasks = []
heapq.heappush(tasks, (2, "write code"))
heapq.heappush(tasks, (1, "plan"))
heapq.heappush(tasks, (3, "test"))

while tasks:
    priority, name = heapq.heappop(tasks)
    print(priority, name)
# 1 plan
# 2 write code
# 3 test

1 plan
2 write code
3 test


In [307]:
# Common mistake: equal priorities make Python compare the second items
import heapq
import itertools

jobs = [(1, {"name": "backup"})]
try:
    heapq.heappush(jobs, (1, {"name": "email"}))    # dicts cannot be compared with <
except TypeError as e:
    print("TypeError:", e)

# Fix: put a unique, increasing counter between priority and item
counter = itertools.count()
queue = []
for priority, job in [(1, {"name": "backup"}), (1, {"name": "email"}), (0, {"name": "alert"})]:
    heapq.heappush(queue, (priority, next(counter), job))

while queue:
    priority, _, job = heapq.heappop(queue)
    print(priority, job["name"])
# 0 alert
# 1 backup   (equal priorities come out in insertion order)
# 1 email

TypeError: '<' not supported between instances of 'dict' and 'dict'
0 alert
1 backup
1 email


### 7.2.3 Top-k, merging and other helpers

* `heapq.nlargest(k, data, key=None)` and `heapq.nsmallest(k, data, key=None)` return the k largest or smallest items. They cost about O(n log k).
* `heapq.merge(*sorted_inputs)` lazily merges already sorted inputs into one sorted stream.

In [308]:
# nlargest, nsmallest (with key=) and merge
import heapq

scores = [52, 91, 78, 64, 99, 85]
print(heapq.nlargest(3, scores))     # [99, 91, 85]
print(heapq.nsmallest(2, scores))    # [52, 64]

people = [("Asha", 31), ("Ravi", 25), ("Meera", 40)]
print(heapq.nsmallest(1, people, key=lambda p: p[1]))   # [('Ravi', 25)]

print(list(heapq.merge([1, 4, 7], [2, 5, 8], [0, 9])))  # [0, 1, 2, 4, 5, 7, 8, 9]

[99, 91, 85]
[52, 64]
[('Ravi', 25)]
[0, 1, 2, 4, 5, 7, 8, 9]


**Streaming top-k pattern.** When numbers arrive one by one (or do not fit in memory), keep a **min-heap of size k**. The heap root is the smallest of the current top k. A new number only enters if it beats the root.

```text
  for each x:
      +-------------------+  Yes  +--------------------+
      | len(heap) < k ?   |------>| heappush(heap, x)  |
      +---------+---------+       +--------------------+
                | No
                v
      +-------------------+  Yes  +------------------------+
      | x > heap[0] ?     |------>| heapreplace(heap, x)   |
      +---------+---------+       +------------------------+
                | No
                v
             skip x
  answer: heap[0] is the k-th largest; the heap holds the top k
```

In [309]:
# Kth largest element in a stream: O(n log k) time, O(k) memory
import heapq

def kth_largest(stream, k):
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)   # pop the smallest, push x
    return heap[0]

print(kth_largest([3, 2, 1, 5, 6, 4], 2))           # 5
print(kth_largest(iter([7, 7, 7]), 1))              # 7

5
7


In [310]:
# heapreplace vs heappushpop: the order of the two steps differs
import heapq

h = [1, 5, 9]                       # already a valid heap
print(heapq.heapreplace(h, 0))      # 1  (pops first, THEN pushes 0)
print(h)                            # [0, 5, 9]

h = [1, 5, 9]
print(heapq.heappushpop(h, 0))      # 0  (pushes first, so 0 comes straight back)
print(h)                            # [1, 5, 9]

1
[0, 5, 9]
0
[1, 5, 9]


> **Interview tip:** Heaps solve "top k", "k-th largest", "merge k sorted lists", "median of a stream" (two heaps) and Dijkstra's shortest path. The full set of heap problems is in Notebook 3, Part 11, and Dijkstra is in Notebook 3, Part 14.

### Common mistakes with heapq

* **Treating the heap list as sorted.** Only `h[0]` is guaranteed. Pop repeatedly or use `sorted(h)` to get order.
* **Forgetting `heapify`** after building a list by hand. `heappush`/`heappop` assume the list is already a heap.
* **Comparing payloads on ties** (see the dict example). Add a counter as a tie-breaker.
* **Forgetting to negate back** when using the negation trick for a max-heap.
* **Using `sorted(data)[-k:]`** for top-k on huge data. `nlargest` or a size-k heap is O(n log k).

### How to debug heap code

| Symptom | Likely cause | Fix |
|---|---|---|
| `TypeError: '<' not supported between instances of 'dict' and 'dict'` | two equal priorities, payloads not comparable | add `next(counter)` as the second tuple item |
| `IndexError: index out of range` on `heappop` | popping an empty heap | check `if heap:` first |
| Wrong order of results | list was never heapified, or you changed `h[i]` directly | call `heapify` again after direct edits |
| Max-heap returns the smallest | forgot to negate on push or pop | negate on both sides |

In [311]:
# The empty-heap error, and the safe check
import heapq

heap = []
try:
    heapq.heappop(heap)
except IndexError as e:
    print("IndexError:", e)          # IndexError: index out of range

print(heapq.heappop(heap) if heap else None)   # None

IndexError: index out of range
None


> **Key points**
> * `heapq` turns a list into a min-heap: `h[0]` is the smallest; push and pop are O(log n); `heapify` is O(n).
> * Max-heap: `heapify_max` / `heappush_max` / `heappop_max` in Python 3.14+, or push `-x` on any version.
> * Use `(priority, counter, item)` tuples so ties never compare the items.
> * `nlargest`/`nsmallest` and a size-k heap give top-k in O(n log k); `merge` merges sorted inputs lazily.

### Practice 7.2: heapq

**A. Predict the output**

1. What does this print?
   ```python
   import heapq

   h = [4, 1, 7]
   heapq.heapify(h)
   heapq.heappush(h, 0)
   print(h[0], len(h))
   ```
2. What does this print?
   ```python
   import heapq

   h = []
   for x in [3, 1, 2]:
       heapq.heappush(h, -x)
   print(-heapq.heappop(h))
   ```
3. What does this print?
   ```python
   import heapq

   print(heapq.nsmallest(2, ["pear", "fig", "banana"], key=len))
   ```

**B. Write code**

4. Write `k_smallest(nums, k)` that returns the k smallest numbers in sorted order, using `heapq`.
5. Write `top_k_frequent(nums, k)` that returns the k most frequent numbers (any order is fine for ties), using `Counter` and `heapq.nlargest`.
6. Write `merge_k_sorted(lists)` that merges k sorted lists into one sorted list using a heap of `(value, list_index, element_index)` tuples (do not use `heapq.merge`).
7. Write `k_closest(points, k)` that returns the k points closest to the origin `(0, 0)`.
8. **[Hard]** Write a class `MedianFinder` with `add(num)` and `median()`, using two heaps (a max-heap for the lower half and a min-heap for the upper half). Each `add` must be O(log n).
9. **[Hard]** Write `min_meeting_rooms(intervals)` that returns how many rooms are needed for a list of `(start, end)` meetings. Hint: sort by start, keep a min-heap of end times.

**C. Find and fix the bug**

10. This should return the 3 largest numbers but returns wrong values. Why?
    ```python
    import heapq

    def top3(nums):
        heap = nums[:]
        heapq.heapify(heap)
        return heap[-3:]
    ```
11. This priority queue crashes when two tasks have the same priority. Fix it.
    ```python
    import heapq

    class Task:
        def __init__(self, name):
            self.name = name

    pq = []
    heapq.heappush(pq, (1, Task("a")))
    heapq.heappush(pq, (1, Task("b")))
    ```

**D. Interview questions**

12. Why is `heapify` O(n) and not O(n log n)?
13. When is `sorted(data)[:k]` a better choice than `heapq.nsmallest(k, data)`?
14. **[Hard]** How would you find the median of a stream of numbers, and what is the time per insertion?

### Solutions 7.2

**A.**
1. `0 4`. The new smallest value is 0 and the heap has 4 items.
2. `3`. The values are stored as `-3, -1, -2`; the smallest is `-3`, negated back to `3`.
3. `['fig', 'pear']`. Sorted by length: `fig` (3), `pear` (4).

**C.**
10. A heap is not sorted. Only `heap[0]` is guaranteed to be the minimum; the last three positions can hold any values. Use `heapq.nlargest(3, nums)`.
11. `Task` objects have no `<`, and Python compares them when the priorities tie. Add a counter: `(priority, next(counter), task)`.

**D.**
12. Most nodes are near the bottom of the tree and only move down a little. Sifting down from the middle of the list to the root adds up to about n / 2 * 1 + n / 4 * 2 + n / 8 * 3 + ... which is at most 2n, so O(n).
13. When k is close to n, or when n is small: sorting is O(n log n) but has small constants and is simple. `nsmallest` wins when k is much smaller than n.
14. Keep a max-heap `low` for the smaller half and a min-heap `high` for the larger half, balanced so their sizes differ by at most 1. The median is the top of the bigger heap, or the average of both tops. Each insertion is O(log n); reading the median is O(1).

In [312]:
# Solutions 7.2
import heapq
import itertools
from collections import Counter

# A: checks
h = [4, 1, 7]
heapq.heapify(h)
heapq.heappush(h, 0)
assert (h[0], len(h)) == (0, 4)
assert heapq.nsmallest(2, ["pear", "fig", "banana"], key=len) == ["fig", "pear"]

# 4. k smallest in sorted order: O(n log k)
def k_smallest(nums, k):
    return heapq.nsmallest(k, nums)

assert k_smallest([5, -1, 3, 3, 0], 3) == [-1, 0, 3]
assert k_smallest([], 2) == []

# 5. Top k frequent: O(n log k)
def top_k_frequent(nums, k):
    counts = Counter(nums)
    return heapq.nlargest(k, counts, key=counts.get)

assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2]
assert top_k_frequent([7], 1) == [7]

# 6. Merge k sorted lists with a heap: O(N log k) for N total items
def merge_k_sorted(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]
    heapq.heapify(heap)
    result = []
    while heap:
        value, i, j = heapq.heappop(heap)
        result.append(value)
        if j + 1 < len(lists[i]):
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return result

assert merge_k_sorted([[1, 4, 7], [2, 5], [], [0, 9]]) == [0, 1, 2, 4, 5, 7, 9]
assert merge_k_sorted([]) == []

# 7. k closest points: compare squared distances (no sqrt needed)
def k_closest(points, k):
    return heapq.nsmallest(k, points, key=lambda p: p[0] ** 2 + p[1] ** 2)

assert k_closest([(1, 3), (-2, 2), (5, 8), (0, 1)], 2) == [(0, 1), (-2, 2)]

# 8. Median of a stream with two heaps
class MedianFinder:
    def __init__(self):
        self.low = []     # max-heap of the smaller half (stored negated)
        self.high = []    # min-heap of the larger half

    def add(self, num):
        heapq.heappush(self.low, -num)
        # move the largest of the low half to the high half
        heapq.heappush(self.high, -heapq.heappop(self.low))
        # keep low the same size as high, or one bigger
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2

mf = MedianFinder()
medians = []
for n in [5, 15, 1, 3]:
    mf.add(n)
    medians.append(mf.median())
assert medians == [5.0, 10.0, 5.0, 4.0]

# 9. Meeting rooms: the heap holds end times of rooms in use
def min_meeting_rooms(intervals):
    ends = []
    for start, end in sorted(intervals):
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)   # reuse the room that frees first
        else:
            heapq.heappush(ends, end)      # need a new room
    return len(ends)

assert min_meeting_rooms([(0, 30), (5, 10), (15, 20)]) == 2
assert min_meeting_rooms([(7, 10), (2, 4)]) == 1
assert min_meeting_rooms([]) == 0

# 10. Fixed top3
def top3(nums):
    return heapq.nlargest(3, nums)

assert top3([5, 1, 9, 3, 7]) == [9, 7, 5]

# 11. Fixed priority queue with a tie-breaker
class Task:
    def __init__(self, name):
        self.name = name

counter = itertools.count()
pq = []
heapq.heappush(pq, (1, next(counter), Task("a")))
heapq.heappush(pq, (1, next(counter), Task("b")))
assert heapq.heappop(pq)[2].name == "a"
print("Practice 7.2: all checks passed")

Practice 7.2: all checks passed


---

## 7.3 bisect: binary search on sorted lists

**Binary search** finds a position in a **sorted** list by halving the search range each step. It takes O(log n) steps: about 20 steps for a million items.

The `bisect` module does binary search for you:

* `bisect_left(a, x)`: the first index where `x` could be inserted to keep `a` sorted. If `x` is present, this is the index of its **first** occurrence.
* `bisect_right(a, x)` (also called `bisect`): the index **after the last** occurrence of `x`.
* `insort(a, x)` (also `insort_left`, `insort_right`): insert `x` in sorted position.

```text
  a = [1, 2, 4, 4, 4, 7]
  index 0  1  2  3  4  5  6
           |     ^        ^
           |     |        +-- bisect_right(a, 4) = 5 (after the last 4)
           |     +----------- bisect_left(a, 4)  = 2 (first 4)
  number of 4s = bisect_right - bisect_left = 3
```

In [313]:
# bisect_left and bisect_right on a list with duplicates
import bisect

a = [1, 2, 4, 4, 4, 7]
print(bisect.bisect_left(a, 4))      # 2
print(bisect.bisect_right(a, 4))     # 5
print(bisect.bisect_left(a, 5))      # 5  (5 is missing: where it would go)
print(bisect.bisect_left(a, 0))      # 0  (smaller than everything)
print(bisect.bisect_left(a, 99))     # 6  (larger than everything: len(a))

# Count occurrences of a value in O(log n)
print(bisect.bisect_right(a, 4) - bisect.bisect_left(a, 4))   # 3

2
5
5
0
6
3


In [314]:
# insort keeps a list sorted while inserting
import bisect

ranks = [10, 20, 30]
bisect.insort(ranks, 25)
bisect.insort(ranks, 5)
print(ranks)                         # [5, 10, 20, 25, 30]

[5, 10, 20, 25, 30]


> **Warning:** `insort` finds the position in O(log n), but inserting into a list shifts the elements after it, so the whole call is O(n). For many inserts, consider a heap, collect and sort once, or the third-party `sortedcontainers` package.

In [315]:
# Search for an exact value with bisect_left (returns -1 if missing)
import bisect

def index_of(sorted_list, x):
    i = bisect.bisect_left(sorted_list, x)
    if i < len(sorted_list) and sorted_list[i] == x:
        return i
    return -1

print(index_of([1, 3, 5, 7], 5))     # 2
print(index_of([1, 3, 5, 7], 4))     # -1
print(index_of([], 1))               # -1

2
-1
-1


In [316]:
# Table lookup: map a score to a grade with bisect_right
import bisect

def letter_grade(score, cutoffs=(60, 70, 80, 90), grades="FDCBA"):
    return grades[bisect.bisect_right(cutoffs, score)]

print([letter_grade(s) for s in (55, 60, 75, 89, 90, 100)])
# ['F', 'D', 'C', 'B', 'A', 'A']

['F', 'D', 'C', 'B', 'A', 'A']


In [317]:
# Python 3.10+: key= lets you search a list of records sorted by a field
import bisect

orders = [("o1", 120), ("o2", 300), ("o3", 450)]      # sorted by amount

def amount_key(order):
    return order[1]

print(bisect.bisect_left(orders, 300, key=amount_key))   # 1
# Note: x is compared with key(item), so pass the AMOUNT, not a tuple.

# Count orders in a price range [200, 500)
lo = bisect.bisect_left(orders, 200, key=amount_key)
hi = bisect.bisect_left(orders, 500, key=amount_key)
print(hi - lo)                                         # 2

1
2


How `bisect_left` works inside. Writing this by hand is a must-know interview skill (Notebook 3, Part 8 covers it in depth).

```text
  lo = 0, hi = len(a)          search range is [lo, hi)
         |
         v
  +----------------+   No   +-------------------+
  |   lo < hi ?    |------->| return lo         |
  +-------+--------+        +-------------------+
          | Yes
          v
  +----------------+
  | mid=(lo+hi)//2 |
  +-------+--------+
          v
  +----------------+  Yes   +-------------------+
  | a[mid] < x ?   |------->| lo = mid + 1      |---+
  +-------+--------+        +-------------------+   |
          | No                                      |
          v                                         |
  +----------------+                                |
  | hi = mid       |--------------------------------+--> loop again
  +----------------+
```

In [318]:
# bisect_left written by hand, checked against the library
import bisect

def my_bisect_left(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1     # answer is to the right of mid
        else:
            hi = mid         # mid might be the answer
    return lo

data = [1, 2, 4, 4, 4, 7]
for x in [0, 1, 3, 4, 7, 8]:
    assert my_bisect_left(data, x) == bisect.bisect_left(data, x)
print("my_bisect_left matches bisect.bisect_left")

my_bisect_left matches bisect.bisect_left


### Common mistakes with bisect

* **Using bisect on an unsorted list.** `bisect` does not check; it just returns a wrong index.
* **Mixing up left and right.** `bisect_left` points at the first equal item; `bisect_right` points after the last equal item.
* **Reading `a[i]` without checking `i < len(a)`.** The insertion point can be `len(a)`.
* **Passing a whole record as `x` when using `key=`.** `x` is compared with `key(item)`, so pass the key value.

### How to debug binary search code

1. Test tiny inputs first: empty list, one item, value smaller than all, larger than all, duplicates.
2. Print `lo`, `mid`, `hi` each loop with `print(f"{lo=} {mid=} {hi=}")`. If they stop changing, you have an infinite loop (often `lo = mid` instead of `lo = mid + 1`).
3. Compare your function with `bisect` on many random sorted lists (a stress test, see Notebook 3, Part 1).

In [319]:
# IndexError when the insertion point is len(a), and the safe version
import bisect

a = [10, 20, 30]
i = bisect.bisect_left(a, 40)
print(i)                                   # 3
try:
    print(a[i])
except IndexError as e:
    print("IndexError:", e)                # IndexError: list index out of range

print(a[i] if i < len(a) else "not found")  # not found

3
IndexError: list index out of range
not found


> **Key points**
> * `bisect` does O(log n) binary search on an already sorted list.
> * `bisect_left` gives the first position of `x`; `bisect_right` gives the position after the last `x`.
> * Count of `x` = `bisect_right - bisect_left`; `insort` keeps a list sorted but is O(n).
> * `key=` (3.10+) searches lists of records; check `i < len(a)` before reading `a[i]`.

### Practice 7.3: bisect

**A. Predict the output**

1. What does this print?
   ```python
   import bisect

   a = [2, 4, 4, 8]
   print(bisect.bisect_left(a, 4), bisect.bisect_right(a, 4), bisect.bisect(a, 5))
   ```
2. What does this print?
   ```python
   import bisect

   a = [3, 1, 2]
   print(bisect.bisect_left(a, 2))
   ```

**B. Write code**

3. Write `count_in_range(sorted_nums, low, high)` that counts values with `low <= v <= high` in O(log n).
4. Write `search_insert(nums, target)` that returns the index of `target`, or the index where it would be inserted.
5. Write `first_and_last(nums, target)` that returns `[first_index, last_index]` of `target`, or `[-1, -1]`.
6. Write `closest_value(sorted_nums, x)` that returns the value nearest to `x` (the smaller one on a tie).
7. **[Hard]** Write `length_of_lis(nums)`: the length of the longest strictly increasing subsequence in O(n log n), using `bisect_left` on a list of "smallest tail values".
8. **[Hard]** Write a class `TimeMap` with `set(key, value, timestamp)` and `get(key, timestamp)` that returns the value with the largest timestamp `<= timestamp` (or `""`). Timestamps for one key arrive in increasing order.

**C. Find and fix the bug**

9. This should return `True` if `x` is in the sorted list, but it crashes for some inputs. Fix it.
   ```python
   import bisect

   def contains(a, x):
       i = bisect.bisect_left(a, x)
       return a[i] == x
   ```

**D. Interview questions**

10. Why does binary search need sorted data? What is its time complexity?
11. **[Hard]** Why does `my_bisect_left` use `hi = mid` and not `hi = mid - 1`?

### Solutions 7.3

**A.**
1. `1 3 3`. The 4s are at indexes 1 and 2, so left is 1 and right is 3; 5 would go at index 3.
2. `2` here, but the answer is meaningless: the list is not sorted, so `bisect` can return a wrong position. Always sort first.

**C.**
9. When `x` is larger than every item, `i == len(a)` and `a[i]` raises `IndexError`. Check `i < len(a) and a[i] == x`.

**D.**
10. Each step throws away half of the range based on one comparison. That is only correct if everything left of a value is smaller and everything right is larger. Time O(log n), space O(1).
11. The range is half-open `[lo, hi)`, and `mid` might itself be the answer when `a[mid] >= x`. `hi = mid` keeps `mid` as a candidate; `hi = mid - 1` could skip the answer.

In [320]:
# Solutions 7.3
import bisect

# A: checks
a = [2, 4, 4, 8]
assert (bisect.bisect_left(a, 4), bisect.bisect_right(a, 4), bisect.bisect(a, 5)) == (1, 3, 3)

# 3. Count values in [low, high]: O(log n)
def count_in_range(sorted_nums, low, high):
    return bisect.bisect_right(sorted_nums, high) - bisect.bisect_left(sorted_nums, low)

assert count_in_range([1, 3, 3, 5, 9], 3, 5) == 3
assert count_in_range([1, 2], 5, 9) == 0
assert count_in_range([], 0, 1) == 0

# 4. Search insert position
def search_insert(nums, target):
    return bisect.bisect_left(nums, target)

assert search_insert([1, 3, 5, 6], 5) == 2
assert search_insert([1, 3, 5, 6], 2) == 1
assert search_insert([1, 3, 5, 6], 7) == 4

# 5. First and last position
def first_and_last(nums, target):
    left = bisect.bisect_left(nums, target)
    if left == len(nums) or nums[left] != target:
        return [-1, -1]
    return [left, bisect.bisect_right(nums, target) - 1]

assert first_and_last([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert first_and_last([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert first_and_last([], 0) == [-1, -1]

# 6. Closest value: compare the neighbours around the insertion point
def closest_value(sorted_nums, x):
    i = bisect.bisect_left(sorted_nums, x)
    candidates = sorted_nums[max(i - 1, 0): i + 1]
    return min(candidates, key=lambda v: (abs(v - x), v))

assert closest_value([1, 4, 6, 10], 5) == 4      # tie 4 and 6 -> smaller
assert closest_value([1, 4, 6, 10], 9) == 10
assert closest_value([1, 4, 6, 10], -3) == 1
assert closest_value([7], 100) == 7

# 7. Longest increasing subsequence in O(n log n)
def length_of_lis(nums):
    tails = []            # tails[k] = smallest tail of an increasing run of length k+1
    for x in nums:
        i = bisect.bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

assert length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert length_of_lis([7, 7, 7]) == 1
assert length_of_lis([]) == 0

# 8. Time-based key-value store: binary search on timestamps
class TimeMap:
    def __init__(self):
        self.times = {}
        self.values = {}

    def set(self, key, value, timestamp):
        self.times.setdefault(key, []).append(timestamp)
        self.values.setdefault(key, []).append(value)

    def get(self, key, timestamp):
        times = self.times.get(key, [])
        i = bisect.bisect_right(times, timestamp)
        return self.values[key][i - 1] if i else ""

tm = TimeMap()
tm.set("price", "100", 1)
tm.set("price", "120", 4)
assert tm.get("price", 3) == "100"
assert tm.get("price", 4) == "120"
assert tm.get("price", 0) == ""
assert tm.get("missing", 5) == ""

# 9. Fixed contains
def contains(a, x):
    i = bisect.bisect_left(a, x)
    return i < len(a) and a[i] == x

assert contains([1, 2, 3], 3)
assert not contains([1, 2, 3], 4)
assert not contains([], 1)
print("Practice 7.3: all checks passed")

Practice 7.3: all checks passed


---

## 7.4 itertools: building blocks for iteration

`itertools` is a set of fast, memory-friendly tools that create and combine **iterators** (objects that give one value at a time with `next()`, see Part 3). Most of them are **lazy**: they produce values only when you ask, so they work on huge or even infinite inputs.

The tools fall into three groups:

| Group | Tools |
|---|---|
| Infinite | `count`, `cycle`, `repeat` |
| Finite (process one or more inputs) | `accumulate`, `chain`, `islice`, `pairwise`, `batched`, `zip_longest`, `groupby`, `takewhile`, `dropwhile`, `filterfalse`, `compress`, `starmap`, `tee` |
| Combinatoric | `product`, `permutations`, `combinations`, `combinations_with_replacement` |

### 7.4.1 Combinatoric tools (very common in interviews)

In [321]:
# product: every combination of one item from each input (nested loops)
from itertools import product

for size, colour in product(["S", "M"], ["red", "blue"]):
    print(size, colour)
# S red
# S blue
# M red
# M blue

# repeat= means "the same input several times": all 3-bit binary strings
print(["".join(bits) for bits in product("01", repeat=3)])
# ['000', '001', '010', '011', '100', '101', '110', '111']

S red
S blue
M red
M blue
['000', '001', '010', '011', '100', '101', '110', '111']


In [322]:
# permutations (order matters) vs combinations (order does not matter)
from itertools import permutations, combinations, combinations_with_replacement

print(list(permutations([1, 2, 3], 2)))
# [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]

print(list(combinations([1, 2, 3], 2)))
# [(1, 2), (1, 3), (2, 3)]

print(list(combinations_with_replacement("ab", 2)))
# [('a', 'a'), ('a', 'b'), ('b', 'b')]

print(len(list(permutations("abcd"))))    # 24  (4! orderings of all 4 letters)

[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
[(1, 2), (1, 3), (2, 3)]
[('a', 'a'), ('a', 'b'), ('b', 'b')]
24


How many results to expect (n items, choose r):

| Tool | Count | Example n=4, r=2 |
|---|---|---|
| `product(..., repeat=r)` | n^r | 16 |
| `permutations(items, r)` | n! / (n-r)! | 12 |
| `combinations(items, r)` | n! / (r! (n-r)!) | 6 |
| `combinations_with_replacement(items, r)` | (n+r-1)! / (r! (n-1)!) | 10 |

> **Warning:** These numbers grow very fast. `permutations(range(12))` has 479,001,600 results. Never turn such an iterator into a list. In interviews, generating all permutations is a brute-force baseline; mention its O(n * n!) cost.

In [323]:
# The counts can be computed without generating anything, using math
import math

print(math.perm(4, 2))     # 12
print(math.comb(4, 2))     # 6
print(math.comb(4 + 2 - 1, 2))   # 10  (combinations with replacement)

12
6
10


In [324]:
# Interview style: all subsets (the power set) using combinations
from itertools import combinations

def all_subsets(items):
    result = []
    for r in range(len(items) + 1):
        result.extend(combinations(items, r))
    return result

print(all_subsets([1, 2, 3]))
# [(), (1,), (2,), (3,), (1, 2), (1, 3), (2, 3), (1, 2, 3)]

[(), (1,), (2,), (3,), (1, 2), (1, 3), (2, 3), (1, 2, 3)]


> **Interview tip:** Interviewers often want you to write subsets and permutations yourself with backtracking (Notebook 3, Part 6). Use `itertools` to check your own answer, or when the problem is about something else and generating combinations is only a small step.

### 7.4.2 Running totals, chaining and slicing

In [325]:
# accumulate: running totals (prefix sums), or any running operation
from itertools import accumulate
import operator

nums = [3, 1, 4, 1, 5]
print(list(accumulate(nums)))                    # [3, 4, 8, 9, 14]  (prefix sums)
print(list(accumulate(nums, initial=0)))         # [0, 3, 4, 8, 9, 14]
print(list(accumulate(nums, max)))               # [3, 3, 4, 4, 5]   (running maximum)
print(list(accumulate(nums, operator.mul)))      # [3, 3, 12, 12, 60] (running product)

[3, 4, 8, 9, 14]
[0, 3, 4, 8, 9, 14]
[3, 3, 4, 4, 5]
[3, 3, 12, 12, 60]


In [326]:
# Prefix sums answer "sum of nums[i:j]" in O(1) per query
from itertools import accumulate

nums = [3, 1, 4, 1, 5]
prefix = list(accumulate(nums, initial=0))       # prefix[k] = sum(nums[:k])

def range_sum(i, j):
    return prefix[j] - prefix[i]                 # sum of nums[i:j]

print(range_sum(1, 4))                           # 6  (1 + 4 + 1)
print(range_sum(0, 5) == sum(nums))              # True

6
True


In [327]:
# chain: join iterables end to end without building a new list
from itertools import chain

print(list(chain([1, 2], (3, 4), "ab")))         # [1, 2, 3, 4, 'a', 'b']

# chain.from_iterable flattens ONE level of nesting
matrix = [[1, 2], [3], [4, 5]]
print(list(chain.from_iterable(matrix)))         # [1, 2, 3, 4, 5]

[1, 2, 3, 4, 'a', 'b']
[1, 2, 3, 4, 5]


In [328]:
# islice: slicing for any iterator (start, stop, step), even infinite ones
from itertools import islice, count

print(list(islice(range(10), 2, 8, 3)))          # [2, 5]
print(list(islice(count(100), 3)))               # [100, 101, 102]

# First 3 lines of a "file" without reading all of it
lines = iter(["line 1", "line 2", "line 3", "line 4"])
print(list(islice(lines, 3)))                    # ['line 1', 'line 2', 'line 3']

[2, 5]
[100, 101, 102]
['line 1', 'line 2', 'line 3']


In [329]:
# pairwise (3.10+): consecutive pairs, great for differences and "is sorted" checks
from itertools import pairwise

temps = [21, 23, 22, 25]
print(list(pairwise(temps)))                     # [(21, 23), (23, 22), (22, 25)]
print([b - a for a, b in pairwise(temps)])       # [2, -1, 3]
print(all(a <= b for a, b in pairwise([1, 2, 2, 5])))   # True (sorted)

[(21, 23), (23, 22), (22, 25)]
[2, -1, 3]
True


In [330]:
# batched (3.12+): split into fixed-size tuples; the last one may be shorter
import sys
from itertools import batched

print(list(batched(range(7), 3)))                # [(0, 1, 2), (3, 4, 5), (6,)]

if sys.version_info >= (3, 13):
    try:
        list(batched(range(7), 3, strict=True))  # strict= is Python 3.13+
    except ValueError as e:
        print("ValueError:", e)                  # the last batch is incomplete

[(0, 1, 2), (3, 4, 5), (6,)]
ValueError: batched(): incomplete batch


In [331]:
# zip vs zip_longest: zip stops at the shortest input
from itertools import zip_longest

names = ["Asha", "Ravi", "Meera"]
scores = [90, 85]
print(list(zip(names, scores)))                       # [('Asha', 90), ('Ravi', 85)]
print(list(zip_longest(names, scores, fillvalue=0)))  # [('Asha', 90), ('Ravi', 85), ('Meera', 0)]

[('Asha', 90), ('Ravi', 85)]
[('Asha', 90), ('Ravi', 85), ('Meera', 0)]


### 7.4.3 groupby

`groupby(iterable, key)` groups **consecutive** items with the same key. It gives pairs of `(key, group_iterator)`.

```text
  input:  a a b b b a          (not sorted)
  groups: (a: a a) (b: b b b) (a: a)   <- 'a' appears twice!
  sort first: a a a b b b
  groups: (a: a a a) (b: b b b)
```

In [332]:
# groupby: run-length encoding (consecutive groups are exactly what we want)
from itertools import groupby

text = "aaabccdd"
print([(ch, len(list(group))) for ch, group in groupby(text)])
# [('a', 3), ('b', 1), ('c', 2), ('d', 2)]

[('a', 3), ('b', 1), ('c', 2), ('d', 2)]


In [333]:
# Common mistake: groupby on unsorted data splits a key into several groups
from itertools import groupby

words = ["apple", "banana", "avocado", "blueberry", "cherry"]

def first_letter(w):
    return w[0]

print([k for k, _ in groupby(words, key=first_letter)])
# ['a', 'b', 'a', 'b', 'c']  (wrong for "group by first letter")

# Fix: sort by the SAME key first
words_sorted = sorted(words, key=first_letter)
print({k: list(g) for k, g in groupby(words_sorted, key=first_letter)})
# {'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}

['a', 'b', 'a', 'b', 'c']
{'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}


> **Tip:** If you only need "group everything by key" (not consecutive runs), `defaultdict(list)` is O(n) and needs no sorting. Use `groupby` for runs, or when the data is already sorted.

### 7.4.4 Infinite iterators and filters

In [334]:
# count, cycle and repeat never stop on their own: always limit them
from itertools import count, cycle, repeat, islice

print(list(islice(count(10, 5), 4)))             # [10, 15, 20, 25]

turns = cycle(["Asha", "Ravi"])
print([next(turns) for _ in range(5)])           # ['Asha', 'Ravi', 'Asha', 'Ravi', 'Asha']

print(list(repeat("-", 3)))                      # ['-', '-', '-']
print(list(map(pow, [2, 3, 4], repeat(2))))      # [4, 9, 16]  (repeat feeds a constant)

[10, 15, 20, 25]
['Asha', 'Ravi', 'Asha', 'Ravi', 'Asha']
['-', '-', '-']
[4, 9, 16]


In [335]:
# takewhile / dropwhile / filterfalse / compress / starmap
from itertools import takewhile, dropwhile, filterfalse, compress, starmap

nums = [1, 3, 5, 2, 7]
print(list(takewhile(lambda x: x < 4, nums)))    # [1, 3]      (stops at the first failure)
print(list(dropwhile(lambda x: x < 4, nums)))    # [5, 2, 7]   (skips until the first failure)
print(list(filterfalse(lambda x: x % 2, nums)))  # [2]         (keeps items where test is false)

print(list(compress("abcd", [1, 0, 1, 0])))      # ['a', 'c']  (select by a mask)
print(list(starmap(pow, [(2, 3), (10, 2)])))     # [8, 100]    (unpacks each tuple as arguments)

[1, 3]
[5, 2, 7]
[2]
['a', 'c']
[8, 100]


In [336]:
# tee: make several independent iterators from one
from itertools import tee

readings = iter([5, 6, 7])
first, second = tee(readings, 2)
print(list(first))       # [5, 6, 7]
print(list(second))      # [5, 6, 7]
# Do not use the original `readings` after tee(); let tee own it.

[5, 6, 7]
[5, 6, 7]


### Common mistakes with itertools

* **Iterators are one-shot.** After you loop over `combinations(...)` or `chain(...)` once, it is empty.
* **`groupby` groups only consecutive items.** Sort by the same key first.
* **Saving `groupby` groups for later.** Each group shares the underlying iterator. When `groupby` moves on, the previous group becomes empty. Convert each group to a list immediately.
* **Calling `list()` on an infinite iterator** (`count()`, `cycle()`) hangs forever. Always limit with `islice`, `takewhile` or `zip`.
* **Exploding sizes.** Check `math.comb` / `math.perm` before generating combinations.

### How to debug iterator code

* If a second loop prints nothing, you are reusing an exhausted iterator. Print `list(it)` once, or rebuild the iterator.
* If the program hangs, look for an infinite iterator without a limit.
* To inspect a lazy pipeline, convert a small slice to a list: `list(islice(it, 5))`.

In [337]:
# Bug demo: one-shot iterators and saved groupby groups
from itertools import combinations, groupby

pairs = combinations([1, 2, 3], 2)
print(len(list(pairs)))     # 3
print(len(list(pairs)))     # 0  (already used up)

# Saving the group iterators without converting them
saved = [(k, g) for k, g in groupby("aabbb")]
print([(k, list(g)) for k, g in saved])   # [('a', []), ('b', [])]  (groups are gone)

# Fix: convert each group to a list immediately
saved = [(k, list(g)) for k, g in groupby("aabbb")]
print(saved)                              # [('a', ['a', 'a']), ('b', ['b', 'b', 'b'])]

3
0
[('a', []), ('b', [])]
[('a', ['a', 'a']), ('b', ['b', 'b', 'b'])]


> **Key points**
> * `product`, `permutations`, `combinations` generate combinatoric results lazily; sizes grow very fast.
> * `accumulate` gives prefix sums and running max; `pairwise` gives neighbours; `batched` (3.12+) gives chunks.
> * `chain` joins, `islice` slices any iterator, `zip_longest` pads the shorter input.
> * `groupby` groups consecutive equal keys: sort first, and convert groups to lists right away.
> * Iterators are one-shot; infinite iterators need a limit.

### Practice 7.4: itertools

**A. Predict the output**

1. What does this print?
   ```python
   from itertools import accumulate

   print(list(accumulate([2, 2, 2], initial=1)))
   ```
2. What does this print?
   ```python
   from itertools import groupby

   print([k for k, g in groupby("aabaa")])
   ```
3. What does this print?
   ```python
   from itertools import combinations

   c = combinations("abc", 2)
   print(len(list(c)), len(list(c)))
   ```
4. How many items does `product("ab", repeat=3)` produce?

**B. Write code**

5. Write `rle(s)` that returns the run-length encoding as a string: `"aaabcc"` becomes `"a3b1c2"`. Use `groupby`.
6. Write `flatten_once(list_of_lists)` using `chain.from_iterable`.
7. Write `chunks(items, n)` that returns lists of size `n` (the last may be shorter). Use `batched` on 3.12+ or `islice` otherwise.
8. Write `two_sum_pairs(nums, target)` that returns all index pairs `(i, j)` with `i < j` and `nums[i] + nums[j] == target`, using `combinations` (this is the O(n^2) brute force).
9. Write `max_drawdown(prices)`: the largest drop from a running peak, using `accumulate(prices, max)`.
10. **[Hard]** Write `letter_combinations(digits)` for phone keypad digits (`"2"` -> `"abc"`, ..., `"9"` -> `"wxyz"`) using `product`. `"23"` gives `["ad", "ae", "af", "bd", ..., "cf"]`.

**C. Find and fix the bug**

11. This should group orders by customer but gives the same customer several times. Fix it.
    ```python
    from itertools import groupby

    orders = [("ravi", 10), ("asha", 5), ("ravi", 7)]
    totals = {}
    for name, group in groupby(orders, key=lambda o: o[0]):
        totals[name] = sum(amount for _, amount in group)
    print(totals)
    ```
12. This hangs. Why? Fix it.
    ```python
    from itertools import count

    evens = [n for n in count(0, 2) if n < 10]
    ```

**D. Interview questions**

13. What is the difference between `permutations` and `combinations`? Give their counts for n=5, r=3.
14. **[Hard]** Why are most `itertools` functions lazy, and when does that matter?

### Solutions 7.4

**A.**
1. `[1, 3, 5, 7]`. The initial value 1 comes first, then 1+2, 3+2, 5+2.
2. `['a', 'b', 'a']`. Groups are consecutive runs.
3. `3 0`. The iterator is used up by the first `list()`.
4. 2^3 = 8 items.

**C.**
11. `groupby` only groups consecutive items, and the orders are not sorted by name, so `ravi` appears twice and the second total overwrites the first (the result is `{'ravi': 7, 'asha': 5}`). Sort by the same key first, or use a `defaultdict(int)`.
12. `count()` never ends; the comprehension keeps asking for more numbers forever because the `if` only filters, it never stops the loop. Use `takewhile(lambda n: n < 10, count(0, 2))` or simply `range(0, 10, 2)`.

**D.**
13. Permutations care about order, combinations do not. For n=5, r=3: permutations 5*4*3 = 60, combinations 60 / 3! = 10.
14. Lazy tools compute one value at a time, so memory stays O(1) even for huge or infinite inputs, and you can stop early. It matters when the data is large (log files, streams) or when you only need the first few results. The cost: you can iterate only once.

In [338]:
# Solutions 7.4
import sys
from collections import defaultdict
from itertools import accumulate, groupby, combinations, chain, islice, product, takewhile, count

# A: checks
assert list(accumulate([2, 2, 2], initial=1)) == [1, 3, 5, 7]
assert [k for k, g in groupby("aabaa")] == ["a", "b", "a"]
assert len(list(product("ab", repeat=3))) == 8

# 5. Run-length encoding
def rle(s):
    return "".join(f"{ch}{len(list(g))}" for ch, g in groupby(s))

assert rle("aaabcc") == "a3b1c2"
assert rle("") == ""
assert rle("z") == "z1"

# 6. Flatten one level
def flatten_once(list_of_lists):
    return list(chain.from_iterable(list_of_lists))

assert flatten_once([[1, 2], [], [3]]) == [1, 2, 3]

# 7. Chunks: batched on 3.12+, islice otherwise
def chunks(items, n):
    if sys.version_info >= (3, 12):
        from itertools import batched
        return [list(b) for b in batched(items, n)]
    it = iter(items)
    result = []
    while block := list(islice(it, n)):
        result.append(block)
    return result

assert chunks([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]
assert chunks([], 3) == []

# 8. Brute-force pairs: O(n^2)
def two_sum_pairs(nums, target):
    return [(i, j) for (i, a), (j, b) in combinations(enumerate(nums), 2) if a + b == target]

assert two_sum_pairs([1, 3, 2, 2], 4) == [(0, 1), (2, 3)]
assert two_sum_pairs([], 0) == []

# 9. Max drawdown with a running peak
def max_drawdown(prices):
    peaks = accumulate(prices, max)
    return max((peak - p for peak, p in zip(peaks, prices)), default=0)

assert max_drawdown([100, 120, 90, 130, 80]) == 50
assert max_drawdown([1, 2, 3]) == 0
assert max_drawdown([]) == 0

# 10. Phone letter combinations
KEYPAD = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
          "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}

def letter_combinations(digits):
    if not digits:
        return []
    return ["".join(p) for p in product(*(KEYPAD[d] for d in digits))]

assert letter_combinations("23") == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert letter_combinations("") == []

# 11. Fixed grouping (two ways)
orders = [("ravi", 10), ("asha", 5), ("ravi", 7)]

def customer(order):
    return order[0]

totals = {name: sum(a for _, a in g) for name, g in groupby(sorted(orders, key=customer), key=customer)}
assert totals == {"asha": 5, "ravi": 17}

totals2 = defaultdict(int)
for name, amount in orders:
    totals2[name] += amount
assert totals2 == totals

# 12. Fixed: stop the infinite iterator
evens = list(takewhile(lambda n: n < 10, count(0, 2)))
assert evens == [0, 2, 4, 6, 8] == list(range(0, 10, 2))
print("Practice 7.4: all checks passed")

Practice 7.4: all checks passed


---

## 7.5 functools: tools for functions

`functools` contains **higher-order functions**: functions that take or return other functions. You met decorators in Part 4; several tools here are decorators.

| Tool | What it does |
|---|---|
| `lru_cache(maxsize=128)`, `cache` | remember (memoise) results of a function |
| `partial(f, ...)` | make a new function with some arguments pre-filled |
| `reduce(f, items, initial)` | fold a sequence into one value |
| `cached_property` | compute an attribute once per object |
| `singledispatch` | choose an implementation by the type of the first argument |
| `cmp_to_key` | turn an old-style compare function into a sort key |
| `total_ordering` | fill in all comparison methods from `__eq__` and one more |
| `wraps` | keep a function's name and docstring in a decorator (Part 4) |

### 7.5.1 lru_cache and cache: memoisation

**Memoisation** means: remember the result for each input, and return the saved result the next time the same input comes. It turns many slow recursive functions into fast ones. This is the "top-down" style of dynamic programming (Notebook 3, Part 16).

* `@lru_cache(maxsize=128)` keeps up to 128 results and drops the least recently used one when full.
* `@lru_cache(maxsize=None)` and `@cache` (Python 3.9+) keep everything (no limit).

```text
  call fib(n)
       |
       v
  +---------------------+  Yes  +-----------------------+
  | n in the cache ?    |------>| return saved result   |  (a "hit")
  +----------+----------+       +-----------------------+
             | No  (a "miss")
             v
  +---------------------+
  | run the function    |
  | save the result     |
  | return it           |
  +---------------------+
```

In [339]:
# Without a cache, fib(n) repeats the same work an exponential number of times
call_count = 0

def slow_fib(n):
    global call_count
    call_count += 1
    if n < 2:
        return n
    return slow_fib(n - 1) + slow_fib(n - 2)

print(slow_fib(20))        # 6765
print(call_count)          # 21891  (calls for only 21 different inputs)

6765
21891


In [340]:
# With @cache each n is computed once: O(n) calls instead of O(2^n)
from functools import cache

@cache
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(20))             # 6765
print(fib(90))             # 2880067194370816120
print(fib.cache_info())    # CacheInfo(hits=89, misses=91, maxsize=None, currsize=91)

6765
2880067194370816120
CacheInfo(hits=89, misses=91, maxsize=None, currsize=91)


In [341]:
# lru_cache with a size limit, cache_info() and cache_clear()
from functools import lru_cache

@lru_cache(maxsize=2)
def square(n):
    return n * n

for n in [1, 2, 1, 3, 2]:
    square(n)

print(square.cache_info())   # CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)
square.cache_clear()
print(square.cache_info())   # CacheInfo(hits=0, misses=0, maxsize=2, currsize=0)

CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)
CacheInfo(hits=0, misses=0, maxsize=2, currsize=0)


Trace of the last cell: `1` miss, `2` miss, `1` hit, `3` miss (evicts `2`, the least recently used), `2` miss again.

In [342]:
# Common mistake: cached functions need hashable arguments
from functools import cache

@cache
def total(numbers):
    return sum(numbers)

try:
    total([1, 2, 3])                 # a list is not hashable
except TypeError as e:
    print("TypeError:", e)           # TypeError: unhashable type: 'list'

print(total((1, 2, 3)))              # 6  (fix: pass a tuple)

TypeError: unhashable type: 'list'
6


> **Warning:** Do not put `@lru_cache` on instance methods without thinking. The cache stores `self` as part of each key, so objects are kept alive as long as the cache lives (a memory leak). Cache a module-level function, use `cached_property`, or keep a dict on the instance.

> **Warning:** Only cache **pure** functions: the same input always gives the same output and there are no side effects. Caching a function that reads the clock or a database gives stale answers.

> **Interview tip:** In Python interviews `@cache` turns a recursive solution into memoised DP in one line. Say clearly that it is memoisation, and give the time (number of states x work per state) and space (cache size + recursion depth). For deep recursion (thousands of levels) a bottom-up loop avoids `RecursionError`.

In [343]:
# Memoised DP on a grid: number of paths from top-left to bottom-right
from functools import cache

def unique_paths(rows, cols):
    @cache
    def paths(r, c):
        if r == 0 or c == 0:
            return 1                       # only one way along the edges
        return paths(r - 1, c) + paths(r, c - 1)
    return paths(rows - 1, cols - 1)

print(unique_paths(3, 7))                  # 28
print(unique_paths(1, 1))                  # 1

28
1


### 7.5.2 partial

`partial(func, *args, **kwargs)` returns a new function with some arguments already filled in. It is like a lambda, but clearer and it can be inspected.

In [344]:
# partial pre-fills arguments
from functools import partial

parse_binary = partial(int, base=2)
print(parse_binary("1011"))          # 11

square = partial(pow, exp=2)
print(square(9))                     # 81

def send(message, *, channel, urgent=False):
    return f"[{channel}] {'URGENT ' if urgent else ''}{message}"

alert = partial(send, channel="ops", urgent=True)
print(alert("disk full"))            # [ops] URGENT disk full
print(alert.func.__name__, alert.keywords)   # send {'channel': 'ops', 'urgent': True}

11
81
[ops] URGENT disk full
send {'channel': 'ops', 'urgent': True}


In [345]:
# Python 3.14+: functools.Placeholder reserves a positional slot in partial
import sys
from functools import partial

if sys.version_info >= (3, 14):
    from functools import Placeholder
    dash_join = partial(str.replace, Placeholder, " ", "-")   # fill args 2 and 3, leave arg 1 open
    print(dash_join("open source tools"))                     # open-source-tools
else:
    print("Placeholder needs Python 3.14+")

open-source-tools


### 7.5.3 reduce

`reduce(func, items, initial)` combines items left to right: `func(func(func(initial, a), b), c)`. Often a built-in (`sum`, `max`, `math.prod`, `"".join`) is clearer, so use `reduce` only when no built-in fits.

In [346]:
# reduce folds a sequence into one value
from functools import reduce
import operator

print(reduce(operator.mul, [1, 2, 3, 4]))          # 24
print(reduce(operator.add, [], 0))                 # 0  (initial value handles empty input)

# Merge a list of dicts (later dicts win)
configs = [{"a": 1}, {"b": 2}, {"a": 3}]
print(reduce(operator.or_, configs, {}))           # {'a': 3, 'b': 2}

try:
    reduce(operator.add, [])                       # empty and no initial value
except TypeError as e:
    print("TypeError:", e)

24
0
{'a': 3, 'b': 2}
TypeError: reduce() of empty iterable with no initial value


### 7.5.4 cached_property

`@cached_property` computes a value the first time you read the attribute and stores it on the object. Later reads are normal attribute reads. It works because it is a **non-data descriptor** (see Part 6): the stored instance attribute wins on later lookups.

In [347]:
# cached_property: expensive computation runs only once per object
from functools import cached_property

class Report:
    def __init__(self, sales):
        self.sales = sales

    @cached_property
    def total(self):
        print("  computing total...")
        return sum(self.sales)

r = Report([120, 80, 50])
print(r.total)           # prints "  computing total..." then 250
print(r.total)           # 250  (no recomputation)

del r.total              # delete the stored value to force a recompute
r.sales.append(50)
print(r.total)           # computes again: 300

  computing total...
250
250
  computing total...
300


> **Note:** `cached_property` needs a normal instance `__dict__`. It does not work on classes that use `__slots__` without a `__dict__` slot.

### 7.5.5 singledispatch

`@singledispatch` makes a **generic function**: one name, with a different implementation chosen by the type of the **first** argument.

In [348]:
# singledispatch: choose the implementation by argument type
from functools import singledispatch

@singledispatch
def describe(value):
    return f"something of type {type(value).__name__}"

@describe.register
def _(value: int):
    return f"the integer {value}"

@describe.register
def _(value: list):
    return f"a list with {len(value)} items"

print(describe(7))           # the integer 7
print(describe([1, 2]))      # a list with 2 items
print(describe(2.5))         # something of type float
print(describe(True))        # the integer True  (bool is a subclass of int)

the integer 7
a list with 2 items
something of type float
the integer True


### 7.5.6 cmp_to_key and total_ordering

Python sorts with a **key** function. Some orderings are easier to write as a **comparison**: "should `a` come before `b`?" returning a negative number, zero, or a positive number. `cmp_to_key` converts such a function into a key.

The classic interview problem is **Largest Number**: arrange numbers to form the largest possible number. `a` should go before `b` when `a + b > b + a` as strings.

In [349]:
# cmp_to_key: Largest Number
from functools import cmp_to_key

def largest_number(nums):
    strs = [str(n) for n in nums]

    def compare(a, b):
        if a + b > b + a:
            return -1        # a first
        if a + b < b + a:
            return 1         # b first
        return 0

    strs.sort(key=cmp_to_key(compare))
    result = "".join(strs)
    return "0" if result[0] == "0" else result   # [0, 0] should give "0"

print(largest_number([3, 30, 34, 5, 9]))   # 9534330
print(largest_number([0, 0]))              # 0

9534330
0


In [350]:
# total_ordering: define __eq__ and __lt__, get <=, >, >= for free
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, text):
        self.parts = tuple(int(p) for p in text.split("."))

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts == other.parts

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts < other.parts

    def __repr__(self):
        return "Version(" + ".".join(map(str, self.parts)) + ")"

print(Version("3.10") > Version("3.9"))      # True  (string comparison would say False)
print(Version("3.12") >= Version("3.12"))    # True
print(sorted([Version("3.14"), Version("3.9"), Version("3.12")]))
# [Version(3.9), Version(3.12), Version(3.14)]

True
True
[Version(3.9), Version(3.12), Version(3.14)]


### Common mistakes with functools

* **Unhashable arguments to a cached function** (`list`, `dict`, `set`) raise `TypeError`. Pass tuples or frozensets.
* **Caching impure functions** gives stale results.
* **`lru_cache` on methods** keeps every `self` alive.
* **`reduce` on an empty sequence without an initial value** raises `TypeError`.
* **A compare function that returns `True`/`False`** instead of negative/zero/positive gives wrong orders with `cmp_to_key`.

### How to debug cached and partial functions

* `f.cache_info()` shows hits and misses. Zero hits means the cache is not helping (for example, every call has new arguments).
* `f.cache_clear()` resets the cache when you change data during testing.
* `f.__wrapped__` gives the original, uncached function.
* For a `partial`, inspect `p.func`, `p.args` and `p.keywords`.

In [351]:
# Inspecting a cached function and the original behind it
from functools import cache

@cache
def double(n):
    return n * 2

double(4)
double(4)
print(double.cache_info().hits)          # 1
print(double.__wrapped__(4))             # 8  (calls the original, bypassing the cache)
print(double.cache_info().hits)          # 1  (unchanged)

1
8
1


> **Key points**
> * `@cache` / `@lru_cache` memoise pure functions with hashable arguments; check with `cache_info()`.
> * `partial` pre-fills arguments; `Placeholder` (3.14+) can leave positional slots open.
> * `reduce` folds a sequence; give an initial value for empty inputs; prefer built-ins when they exist.
> * `cached_property` computes an attribute once; `singledispatch` dispatches on the first argument's type.
> * `cmp_to_key` adapts compare functions (Largest Number); `total_ordering` fills in comparisons.

### Practice 7.5: functools

**A. Predict the output**

1. What does this print?
   ```python
   from functools import lru_cache

   @lru_cache(maxsize=None)
   def f(n):
       return n * 10

   f(1)
   f(2)
   f(1)
   print(f.cache_info().hits, f.cache_info().misses)
   ```
2. What does this print?
   ```python
   from functools import partial

   def power(base, exponent):
       return base ** exponent

   cube = partial(power, exponent=3)
   print(cube(2))
   ```
3. What does this print?
   ```python
   from functools import reduce

   print(reduce(lambda acc, x: acc * 10 + x, [1, 2, 3], 0))
   ```

**B. Write code**

4. Write `climb_stairs(n)`: the number of ways to climb `n` steps taking 1 or 2 steps at a time, using `@cache`.
5. Write `min_coins(coins, amount)`: the fewest coins to make `amount`, or `-1`, using `@cache` (top-down DP).
6. Write `make_multiplier(k)` using `partial(operator.mul, k)` and use it with `map` to triple a list.
7. Write `sort_by_length_then_reverse_alpha(words)` with `cmp_to_key`: shorter words first; equal lengths in reverse alphabetical order.
8. Write a `@singledispatch` function `to_json_value(x)` that returns `x` for `int`/`str`, a list for `set` (sorted), and an ISO string for `datetime.date`.
9. **[Hard]** Write `word_break(s, words)` that returns `True` if `s` can be split into words from the list, using `@cache` on the start index.
10. **[Hard]** Write your own `my_cache` decorator (a simple dict-based memoiser) that also exposes a `.calls` counter of real computations.

**C. Find and fix the bug**

11. This cached function gives a `TypeError`. Fix it without removing the cache.
    ```python
    from functools import cache

    @cache
    def count_unique(items):
        return len(set(items))

    count_unique([1, 2, 2, 3])
    ```
12. This sort with `cmp_to_key` gives a wrong order. Why?
    ```python
    from functools import cmp_to_key

    nums = [3, 1, 2]
    nums.sort(key=cmp_to_key(lambda a, b: a < b))
    ```

**D. Interview questions**

13. What is memoisation, and how does `@cache` change the complexity of naive recursive Fibonacci?
14. **[Hard]** What are the risks of `lru_cache` on an instance method, and what are two alternatives?

### Solutions 7.5

**A.**
1. `1 2`. `f(1)` and `f(2)` are misses; the second `f(1)` is a hit.
2. `8`.
3. `123`. Steps: 0*10+1 = 1, 1*10+2 = 12, 12*10+3 = 123.

**C.**
11. Lists are not hashable. Convert at the call site: `count_unique(tuple(items))`, or write a thin wrapper that converts and then calls the cached function.
12. The compare function returns `True`/`False` (1/0), never a negative number, so "less than" is never reported. Return `a - b` (or -1/0/1). Better: just use `nums.sort()`.

**D.**
13. Memoisation stores the result for each argument so it is computed once. Naive Fibonacci is O(2^n) calls; with `@cache` each `n` is computed once, so O(n) time and O(n) space (cache plus recursion depth).
14. The cache keeps a reference to every `self` used, so objects are never freed while the cache exists (memory leak), and the cache is shared by all instances. Alternatives: `@cached_property` for zero-argument values; a per-instance dict; or a module-level cached function that takes only hashable data (not `self`).

In [352]:
# Solutions 7.5
import operator
from datetime import date
from functools import cache, lru_cache, partial, reduce, cmp_to_key, singledispatch, wraps

# A: checks
@lru_cache(maxsize=None)
def f(n):
    return n * 10

f(1)
f(2)
f(1)
assert (f.cache_info().hits, f.cache_info().misses) == (1, 2)
assert reduce(lambda acc, x: acc * 10 + x, [1, 2, 3], 0) == 123

# 4. Climbing stairs: O(n) time with the cache
def climb_stairs(n):
    @cache
    def ways(i):
        if i <= 1:
            return 1
        return ways(i - 1) + ways(i - 2)
    return ways(n)

assert [climb_stairs(n) for n in range(1, 6)] == [1, 2, 3, 5, 8]
assert climb_stairs(0) == 1

# 5. Minimum coins: O(amount * len(coins)) time
def min_coins(coins, amount):
    @cache
    def best(rest):
        if rest == 0:
            return 0
        if rest < 0:
            return float("inf")
        return 1 + min(best(rest - c) for c in coins)
    answer = best(amount)
    return -1 if answer == float("inf") else answer

assert min_coins((1, 2, 5), 11) == 3
assert min_coins((2,), 3) == -1
assert min_coins((1,), 0) == 0

# 6. partial with operator.mul
def make_multiplier(k):
    return partial(operator.mul, k)

triple = make_multiplier(3)
assert list(map(triple, [1, -2, 0])) == [3, -6, 0]

# 7. Custom comparison
def sort_by_length_then_reverse_alpha(words):
    def compare(a, b):
        if len(a) != len(b):
            return len(a) - len(b)
        if a == b:
            return 0
        return -1 if a > b else 1      # reverse alphabetical
    return sorted(words, key=cmp_to_key(compare))

assert sort_by_length_then_reverse_alpha(["bb", "a", "cc", "aa", "c"]) == ["c", "a", "cc", "bb", "aa"]

# 8. singledispatch by type
@singledispatch
def to_json_value(x):
    raise TypeError(f"cannot convert {type(x).__name__}")

@to_json_value.register(int)
@to_json_value.register(str)
def _(x):
    return x

@to_json_value.register
def _(x: set):
    return sorted(x)

@to_json_value.register
def _(x: date):
    return x.isoformat()

assert to_json_value(5) == 5 and to_json_value("hi") == "hi"
assert to_json_value({3, 1, 2}) == [1, 2, 3]
assert to_json_value(date(2026, 10, 24)) == "2026-10-24"
try:
    to_json_value(1.5)
    raise AssertionError("expected TypeError")
except TypeError:
    pass

# 9. Word break: O(n^2) states and work with the cache
def word_break(s, words):
    word_set = set(words)

    @cache
    def can_split(start):
        if start == len(s):
            return True
        return any(s[start:end] in word_set and can_split(end)
                   for end in range(start + 1, len(s) + 1))

    return can_split(0)

assert word_break("leetcode", ["leet", "code"])
assert word_break("applepenapple", ["apple", "pen"])
assert not word_break("catsandog", ["cats", "dog", "sand", "and", "cat"])
assert word_break("", ["a"])

# 10. A simple memoising decorator
def my_cache(func):
    saved = {}

    @wraps(func)
    def wrapper(*args):
        if args not in saved:
            wrapper.calls += 1
            saved[args] = func(*args)
        return saved[args]

    wrapper.calls = 0
    return wrapper

@my_cache
def slow_square(n):
    return n * n

assert [slow_square(3), slow_square(3), slow_square(4)] == [9, 9, 16]
assert slow_square.calls == 2
assert slow_square.__name__ == "slow_square"

# 11. Fixed: pass a hashable tuple
@cache
def count_unique(items):
    return len(set(items))

assert count_unique(tuple([1, 2, 2, 3])) == 3

# 12. Fixed comparison
nums = [3, 1, 2]
nums.sort(key=cmp_to_key(lambda a, b: a - b))
assert nums == [1, 2, 3]
print("Practice 7.5: all checks passed")

Practice 7.5: all checks passed


---

## 7.6 operator: operators as functions

The `operator` module gives you every Python operator as a normal function: `operator.add(a, b)` is `a + b`, `operator.lt(a, b)` is `a < b`, and so on. Its most useful tools are three **getters** that build key functions:

* `itemgetter(k)`: a function that returns `obj[k]` (works for lists, tuples, dicts).
* `attrgetter("name")`: a function that returns `obj.name` (also `"address.city"` for nested attributes).
* `methodcaller("name", *args)`: a function that calls `obj.name(*args)`.

They are a little faster than the equivalent `lambda`, and they read clearly in `sorted`, `min`, `max`, `heapq` and `groupby`.

In [353]:
# itemgetter: sort records by one or more positions/keys
from operator import itemgetter

rows = [("Ravi", 31, "Pune"), ("Asha", 25, "Delhi"), ("Meera", 31, "Agra")]

print(sorted(rows, key=itemgetter(1)))
# [('Asha', 25, 'Delhi'), ('Ravi', 31, 'Pune'), ('Meera', 31, 'Agra')]  (stable sort keeps Ravi before Meera)

print(sorted(rows, key=itemgetter(1, 0)))
# [('Asha', 25, 'Delhi'), ('Meera', 31, 'Agra'), ('Ravi', 31, 'Pune')]  (age, then name)

get_name_city = itemgetter(0, 2)
print(get_name_city(rows[0]))      # ('Ravi', 'Pune')  (several keys give a tuple)

[('Asha', 25, 'Delhi'), ('Ravi', 31, 'Pune'), ('Meera', 31, 'Agra')]
[('Asha', 25, 'Delhi'), ('Meera', 31, 'Agra'), ('Ravi', 31, 'Pune')]
('Ravi', 'Pune')


In [354]:
# itemgetter on dicts: sort a dict by value, sort dict records
from operator import itemgetter

stock = {"pens": 40, "books": 12, "bags": 25}
print(sorted(stock.items(), key=itemgetter(1)))
# [('books', 12), ('bags', 25), ('pens', 40)]

print(max(stock.items(), key=itemgetter(1))[0])   # pens

people = [{"name": "Ravi", "dept": "ops"}, {"name": "Asha", "dept": "dev"}]
print([p["name"] for p in sorted(people, key=itemgetter("dept"))])   # ['Asha', 'Ravi']

[('books', 12), ('bags', 25), ('pens', 40)]
pens
['Asha', 'Ravi']


In [355]:
# attrgetter and methodcaller
from operator import attrgetter, methodcaller
from dataclasses import dataclass

@dataclass
class Employee:
    name: str
    dept: str
    salary: int

staff = [Employee("Ravi", "ops", 50), Employee("Asha", "dev", 70), Employee("Meera", "dev", 60)]

print([e.name for e in sorted(staff, key=attrgetter("dept", "salary"))])
# ['Meera', 'Asha', 'Ravi']

print(max(staff, key=attrgetter("salary")).name)      # Asha

clean = methodcaller("strip", " .")
print(list(map(clean, ["  hi. ", ".ok"])))            # ['hi', 'ok']

['Meera', 'Asha', 'Ravi']
Asha
['hi', 'ok']


In [356]:
# Arithmetic and comparison operators as functions
import operator
from functools import reduce

print(reduce(operator.mul, range(1, 6)))              # 120
print(list(map(operator.neg, [1, -2, 3])))            # [-1, 2, -3]
print(list(map(operator.add, [1, 2], [10, 20])))      # [11, 22]

# Choose an operator at run time from a table
ops = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
print(ops["*"](6, 7))                                 # 42

120
[-1, 2, -3]
[11, 22]
42


The operator table in the last cell is a clean way to write a calculator, an expression evaluator, or "Evaluate Reverse Polish Notation" (Notebook 3, Part 10), without a long `if/elif` chain.

### Common mistakes with operator

* **`itemgetter(0, 1)` returns a tuple**, but `itemgetter(0)` returns a single value. Do not unpack the single-key version.
* **`attrgetter("a.b")` follows dots**, so a typo gives `AttributeError` at sort time, not when you build the getter.
* **Using `operator.truediv` when you wanted integer division.** Use `operator.floordiv` for `//`.

### How to debug key functions

Call the key function on one item by itself: `itemgetter(1)(rows[0])` or `attrgetter("salary")(staff[0])`. If that works and returns what you expect, the sort will work too. Errors you will meet:

In [357]:
# Typical key-function errors
from operator import itemgetter, attrgetter

try:
    sorted([("a", 1), ("b",)], key=itemgetter(1))   # second tuple is too short
except IndexError as e:
    print("IndexError:", e)          # IndexError: tuple index out of range

try:
    attrgetter("salry")(object())                   # misspelled attribute
except AttributeError as e:
    print("AttributeError:", e)

IndexError: tuple index out of range
AttributeError: 'object' object has no attribute 'salry'


> **Key points**
> * `operator` turns operators into functions (`add`, `mul`, `neg`, `lt`, `truediv`, ...).
> * `itemgetter`, `attrgetter` and `methodcaller` build fast, readable key functions; several keys give a tuple key.
> * A dict of operators replaces long `if/elif` chains.

### Practice 7.6: operator

**A. Predict the output**

1. What does this print?
   ```python
   from operator import itemgetter

   print(itemgetter(-1)("python"), itemgetter(0, -1)("python"))
   ```
2. What does this print?
   ```python
   import operator

   print(sorted([3, -4, 1], key=operator.neg))
   ```

**B. Write code**

3. Sort a list of `(name, score)` tuples by score descending, then name ascending, using `itemgetter` only once (hint: sort twice, relying on stability).
4. Use `attrgetter` to find the youngest person in a list of objects with `name` and `age` attributes.
5. Write `evaluate_rpn(tokens)` for Reverse Polish Notation (`["2", "1", "+", "3", "*"]` gives `9`) using a dict of `operator` functions; division truncates toward zero.
6. Write `dot(a, b)` using `map(operator.mul, a, b)` and `sum`.
7. Write `top_department(employees)` that returns the department with the highest total salary from a list of dicts, using `itemgetter` and a `Counter`.
8. **[Hard]** Write `sort_records(records, spec)` where `spec` is a list like `[("dept", False), ("salary", True)]` (field, descending?). Use repeated stable sorts with `itemgetter`, starting from the last field.

**C. Find and fix the bug**

9. This should return the names of the two oldest people but crashes. Fix it.
   ```python
   from operator import itemgetter

   people = [{"name": "Asha", "age": 30}, {"name": "Ravi", "age": 41}, {"name": "Meera", "age": 35}]
   oldest = sorted(people, key=itemgetter("Age"), reverse=True)[:2]
   ```

**D. Interview questions**

10. When would you use `itemgetter` instead of a `lambda`?

### Solutions 7.6

**A.**
1. `n ('p', 'n')`. One index gives one value; two indexes give a tuple.
2. `[3, 1, -4]`. The keys are `-3, 4, -1`; sorting the keys ascending gives `-3, -1, 4`, which is the order 3, 1, -4.

**C.**
9. The key is `"age"` (lower case); `itemgetter("Age")` raises `KeyError: 'Age'`. Use `itemgetter("age")`, then take the names.

**D.**
10. When the key is just "take field k" or "take attribute x": it is shorter, slightly faster, can take several fields at once, and can be pickled (useful for multiprocessing). Use a `lambda` when the key needs real logic.

In [358]:
# Solutions 7.6
import operator
from collections import Counter
from operator import itemgetter, attrgetter
from types import SimpleNamespace

# A: checks
assert (itemgetter(-1)("python"), itemgetter(0, -1)("python")) == ("n", ("p", "n"))
assert sorted([3, -4, 1], key=operator.neg) == [3, 1, -4]

# 3. Two stable sorts: secondary key first, then primary key
scores = [("Ravi", 80), ("Asha", 95), ("Meera", 80)]
by_name = sorted(scores, key=itemgetter(0))
result = sorted(by_name, key=itemgetter(1), reverse=True)
assert result == [("Asha", 95), ("Meera", 80), ("Ravi", 80)]

# 4. Youngest person
people = [SimpleNamespace(name="Asha", age=30), SimpleNamespace(name="Ravi", age=22)]
assert min(people, key=attrgetter("age")).name == "Ravi"

# 5. Evaluate RPN with an operator table
OPS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": lambda a, b: int(a / b),     # truncate toward zero
}

def evaluate_rpn(tokens):
    stack = []
    for tok in tokens:
        if tok in OPS:
            right = stack.pop()
            left = stack.pop()
            stack.append(OPS[tok](left, right))
        else:
            stack.append(int(tok))
    return stack[0]

assert evaluate_rpn(["2", "1", "+", "3", "*"]) == 9
assert evaluate_rpn(["4", "13", "5", "/", "+"]) == 6
assert evaluate_rpn(["-7", "2", "/"]) == -3

# 6. Dot product
def dot(a, b):
    return sum(map(operator.mul, a, b))

assert dot([1, 2, 3], [4, 5, 6]) == 32
assert dot([], []) == 0

# 7. Department with the highest total salary
def top_department(employees):
    totals = Counter()
    for e in employees:
        totals[e["dept"]] += e["salary"]
    return max(totals.items(), key=itemgetter(1))[0]

staff = [{"dept": "dev", "salary": 70}, {"dept": "ops", "salary": 90}, {"dept": "dev", "salary": 30}]
assert top_department(staff) == "dev"

# 8. Multi-field sort with mixed directions
def sort_records(records, spec):
    result = list(records)
    for field, descending in reversed(spec):
        result.sort(key=itemgetter(field), reverse=descending)
    return result

recs = [{"dept": "ops", "salary": 50}, {"dept": "dev", "salary": 60}, {"dept": "dev", "salary": 90}]
assert sort_records(recs, [("dept", False), ("salary", True)]) == [
    {"dept": "dev", "salary": 90}, {"dept": "dev", "salary": 60}, {"dept": "ops", "salary": 50}]

# 9. Fixed key name
people = [{"name": "Asha", "age": 30}, {"name": "Ravi", "age": 41}, {"name": "Meera", "age": 35}]
oldest = [p["name"] for p in sorted(people, key=itemgetter("age"), reverse=True)[:2]]
assert oldest == ["Ravi", "Meera"]
print("Practice 7.6: all checks passed")

Practice 7.6: all checks passed


---

## 7.7 Numbers: math, statistics, random, secrets, decimal, fractions

| Module | Use it for |
|---|---|
| `math` | square roots, rounding, gcd, combinations, logarithms, constants |
| `statistics` | mean, median, mode, standard deviation, quantiles |
| `random` | simulations, games, shuffling, sampling (not for security) |
| `secrets` | passwords, tokens, keys (security) |
| `decimal` | exact decimal arithmetic, money |
| `fractions` | exact rational numbers like 1/3 |

### 7.7.1 math

In [359]:
# Rounding and roots: note what happens with negative numbers
import math

print(math.floor(-2.5), math.ceil(-2.5), math.trunc(-2.5))   # -3 -2 -2
print(round(2.5), round(3.5))     # 2 4  (round() uses "round half to even")

print(math.sqrt(16))              # 4.0  (always a float)
print(math.isqrt(17))             # 4    (integer square root, exact for huge ints)
print(math.isqrt(10**40) == 10**20)   # True

-3 -2 -2
2 4
4.0
4
True


In [360]:
# Number theory and counting
import math

print(math.gcd(12, 18))           # 6
print(math.gcd(12, 18, 27))       # 3   (several arguments, Python 3.9+)
print(math.lcm(4, 6))             # 12  (Python 3.9+)
print(math.factorial(5))          # 120
print(math.comb(5, 2))            # 10  (choose 2 of 5, order does not matter)
print(math.perm(5, 2))            # 20  (order matters)
print(math.prod([2, 3, 4]))       # 24  (like sum, but multiplies)

6
3
12
120
10
20
24


In [361]:
# Logarithms, distances and constants
import math

print(math.log2(1024))            # 10.0
print(math.log10(1000))           # 3.0
print(math.log(8, 2))             # 3.0  (any base)
print(math.hypot(3, 4))           # 5.0  (length of the vector (3, 4))
print(math.dist((0, 0), (6, 8)))  # 10.0 (distance between two points)
print(round(math.pi, 5), round(math.e, 5))   # 3.14159 2.71828
print(math.inf > 10**100)         # True (useful as "infinity" in min-searches)

10.0
3.0
3.0
5.0
10.0
3.14159 2.71828
True


In [362]:
# Comparing floats: use math.isclose, and math.fsum for exact sums
import math

print(0.1 + 0.2 == 0.3)                  # False (binary floats are approximate)
print(math.isclose(0.1 + 0.2, 0.3))      # True
print(math.isclose(1e-10, 0.0))          # False (relative tolerance fails near zero)
print(math.isclose(1e-10, 0.0, abs_tol=1e-9))   # True

values = [0.1] * 10
print(sum(values) == 1.0)                # True on Python 3.12+ (sum became more accurate)
print(math.fsum(values) == 1.0)          # True on every version (exactly rounded)

False
True
False
True
True
True


> **Note:** `math` works with real numbers. `math.sqrt(-1)` raises `ValueError` (the message is "math domain error" on older versions and "expected a nonnegative input, got -1.0" on Python 3.14). For complex numbers use the `cmath` module.

### 7.7.2 statistics

In [363]:
# Averages and spread
import statistics

data = [2, 4, 4, 4, 5, 5, 7, 9]
print(statistics.mean(data))             # 5
print(statistics.median(data))           # 4.5  (average of the two middle values)
print(statistics.mode(data))             # 4
print(statistics.multimode("aabbc"))     # ['a', 'b']  (all most common values)
print(statistics.pstdev(data))           # 2.0  (population standard deviation)
print(round(statistics.stdev(data), 3))  # 2.138  (sample standard deviation)
print(statistics.quantiles(data, n=4))   # [4.0, 4.5, 6.5]  (quartile cut points)

5
4.5
4
['a', 'b']
2.0
2.138
[4.0, 4.5, 6.5]


* `pstdev` / `pvariance`: use when `data` is the **whole population**.
* `stdev` / `variance`: use when `data` is a **sample** of a bigger population (divides by n - 1).
* `statistics.mean([])` raises `StatisticsError`. Check for empty data first.

### 7.7.3 random

`random` gives **pseudo-random** numbers: they look random, but come from a formula (the Mersenne Twister). With the same **seed** you get the same sequence, which makes tests and simulations reproducible.

In [364]:
# Seeded random numbers are reproducible (on the same Python version)
import random

random.seed(42)
print(random.random())                        # 0.6394267984578837  (0 <= x < 1)
print(random.randint(1, 6))                   # 1  (both ends included)
print(random.choice(["red", "green", "blue"]))  # blue

0.6394267984578837
1
blue


In [365]:
# shuffle, sample, choices and a private generator
import random

random.seed(42)
cards = ["a", "b", "c", "d", "e"]
random.shuffle(cards)                          # shuffles IN PLACE, returns None
print(cards)                                   # ['d', 'b', 'c', 'e', 'a']

print(random.sample(range(1, 50), 3))          # 3 different numbers (no repeats)
print(random.choices(["win", "lose"], weights=[1, 9], k=5))   # with repeats, weighted

rng = random.Random(7)                         # own generator: does not affect the global one
print([rng.randint(1, 6) for _ in range(5)])   # [3, 2, 4, 6, 1]

['d', 'b', 'c', 'e', 'a']
[16, 15, 9]
['lose', 'lose', 'lose', 'win', 'lose']
[3, 2, 4, 6, 1]


> **Tip:** In tests and simulations, create `rng = random.Random(seed)` and pass it around. Other code that uses the global `random` cannot then change your sequence.

> **Warning:** Never use `random` for passwords, tokens, or anything an attacker might guess. Its future values can be predicted from past outputs. Use `secrets`.

### 7.7.4 secrets

In [366]:
# secrets: cryptographically strong randomness for security
import secrets
import string

token = secrets.token_hex(16)             # 16 random bytes as 32 hex characters
print(len(token))                         # 32

url_token = secrets.token_urlsafe(16)     # safe to put in a URL (password reset links)
print(len(url_token) >= 16)               # True

alphabet = string.ascii_letters + string.digits
password = "".join(secrets.choice(alphabet) for _ in range(12))
print(len(password), password.isalnum())  # 12 True

# Compare secrets in constant time (prevents timing attacks)
print(secrets.compare_digest("abc123", "abc123"))   # True

32
True
12 True
True


### 7.7.5 decimal: exact decimal arithmetic

Floats store numbers in binary, so `0.1` is not stored exactly. `Decimal` stores decimal digits exactly, which is what you want for **money**.

In [367]:
# Decimal: always build from STRINGS, not floats
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))   # True
print(0.1 + 0.2 == 0.3)                                    # False

print(Decimal("0.1"))      # 0.1
print(Decimal(0.1))        # 0.1000000000000000055511151231257827021181583404541015625
# Decimal(0.1) copies the float's binary error exactly. Use Decimal("0.1").

True
False
0.1
0.1000000000000000055511151231257827021181583404541015625


In [368]:
# Rounding money: quantize with an explicit rounding rule
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN

price = Decimal("2.675")
print(price.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))    # 2.68
print(price.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN))  # 2.68  (8 is even)
print(round(2.675, 2))     # 2.67  (the float 2.675 is really 2.67499999...)

subtotal = Decimal("19.99") * 3
tax = (subtotal * Decimal("0.18")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
print(subtotal, tax, subtotal + tax)   # 59.97 10.79 70.76

2.68
2.68
2.67
59.97 10.79 70.76


In [369]:
# Precision is set by a context; localcontext() changes it temporarily
from decimal import Decimal, localcontext

print(Decimal(1) / Decimal(3))         # 0.3333333333333333333333333333  (28 digits by default)

with localcontext() as ctx:
    ctx.prec = 5
    print(Decimal(1) / Decimal(3))     # 0.33333

print(Decimal(2) / Decimal(3))         # 0.6666666666666666666666666667  (back to 28)

0.3333333333333333333333333333
0.33333
0.6666666666666666666666666667


### 7.7.6 fractions: exact rational numbers

In [370]:
# Fraction keeps exact numerators and denominators
from fractions import Fraction

print(Fraction(1, 3) + Fraction(1, 6))        # 1/2
print(Fraction("0.75"))                       # 3/4
print(Fraction(6, 8))                         # 3/4  (always reduced)
print(Fraction(1, 3) * 3 == 1)                # True

print(Fraction(0.1))                          # 3602879701896397/36028797018963968
print(Fraction(0.1).limit_denominator(100))   # 1/10

1/2
3/4
3/4
True
3602879701896397/36028797018963968
1/10


| Type | Exact? | Speed | Use for |
|---|---|---|---|
| `int` | yes, any size | fast | counting, indexes, big integers |
| `float` | no (binary, about 15-17 digits) | fastest | science, measurements |
| `Decimal` | yes for decimal values | slower | money, financial rules |
| `Fraction` | yes for rationals | slowest | exact maths with ratios |

### Common mistakes with numbers

* **`Decimal(0.1)` instead of `Decimal("0.1")`**: copies the float error.
* **Expecting `round(2.5) == 3`**: Python rounds half to even. Use `Decimal.quantize(..., ROUND_HALF_UP)` for "school" rounding.
* **Comparing floats with `==`**: use `math.isclose`.
* **Using `random` for tokens**: use `secrets`.
* **`random.shuffle` returns `None`**: `cards = random.shuffle(cards)` destroys your list variable.
* **Mixing `Decimal` and `float`** in arithmetic raises `TypeError`.

### How to debug numeric code

In [371]:
# The typical numeric errors, caught and printed
import math
import random
import statistics
from decimal import Decimal

try:
    math.sqrt(-4)
except ValueError as e:
    print("ValueError:", e)                     # message wording differs between versions

try:
    Decimal("1.5") + 0.5
except TypeError as e:
    print("TypeError:", e)

try:
    statistics.mean([])
except statistics.StatisticsError as e:
    print("StatisticsError:", e)

cards = [1, 2, 3]
print(random.shuffle(cards))                    # None  (shuffle works in place)

ValueError: expected a nonnegative input, got -4.0
TypeError: unsupported operand type(s) for +: 'decimal.Decimal' and 'float'
StatisticsError: mean requires at least one data point
None


When a calculation gives a surprising result:

1. Print the exact value with `repr(x)` or `f"{x!r}"`; `print` may hide digits.
2. Print the types: `type(x).__name__`. A `float` sneaking into `Decimal` code is a common cause.
3. Check for integer vs true division (`//` vs `/`) and for rounding at the wrong step (round only at the end).
4. Write an `assert` with a known answer computed by hand.

> **Key points**
> * `math`: `isqrt`, `gcd`, `lcm`, `comb`, `perm`, `prod`, `isclose`, `fsum`, `inf`.
> * `statistics`: `mean`, `median`, `mode`, `stdev` (sample) vs `pstdev` (population), `quantiles`.
> * `random` is reproducible with a seed; use `random.Random(seed)` for isolation; never for security.
> * `secrets` for tokens and passwords; `compare_digest` for safe comparisons.
> * `Decimal` (built from strings) for money with explicit rounding; `Fraction` for exact ratios.

### Practice 7.7: numbers

**A. Predict the output**

1. What does this print?
   ```python
   import math

   print(math.floor(-0.5), math.ceil(-0.5), round(0.5), round(1.5))
   ```
2. What does this print?
   ```python
   from decimal import Decimal

   print(Decimal("1.10") + Decimal("2.205"))
   ```
3. What does this print?
   ```python
   from fractions import Fraction

   print(Fraction(2, 4) + Fraction(1, 4))
   ```
4. What does this print?
   ```python
   import random

   items = [3, 1, 2]
   result = random.shuffle(items)
   print(result)
   ```

**B. Write code**

5. Write `is_prime(n)` using `math.isqrt` (check divisors up to the integer square root).
6. Write `lcm_of_list(nums)` using `math.lcm` (it accepts several arguments).
7. Write `summary(data)` that returns a dict with `mean`, `median` and `stdev` (rounded to 2 places), or `None` when there are fewer than 2 values.
8. Write `split_bill(total, people)` with `Decimal` that returns a list of shares in rupees with 2 decimals, and the shares must add up to exactly `total` (give the extra paise to the first people).
9. Write `make_token()` that returns a 32-character hex token using `secrets`.
10. Write `simulate_dice(rolls, seed)` that returns a `Counter` of results for `rolls` dice throws, using `random.Random(seed)`, so the result is reproducible.
11. **[Hard]** Write `reservoir_sample(stream, k, rng)`: pick `k` items uniformly from an iterator of unknown length in one pass, using O(k) memory.

**C. Find and fix the bug**

12. This money calculation prints `0.30000000000000004`. Fix it.
    ```python
    price = 0.1
    qty = 3
    print(price * qty)
    ```
13. This token generator is insecure. Why? Fix it.
    ```python
    import random
    import string

    def reset_code():
        return "".join(random.choice(string.digits) for _ in range(6))
    ```

**D. Interview questions**

14. Why is `0.1 + 0.2 != 0.3` in Python (and almost every language)? How do you compare floats correctly?
15. **[Hard]** What is the difference between `random` and `secrets`? Name a situation for each.

### Solutions 7.7

**A.**
1. `-1 0 0 2`. `floor` goes down, `ceil` goes up; `round` uses half-to-even, so 0.5 becomes 0 and 1.5 becomes 2.
2. `3.305`. Decimal adds exactly.
3. `3/4`. `2/4` is reduced to `1/2`, plus `1/4`.
4. `None`. `shuffle` changes the list in place and returns `None`.

**C.**
12. Floats are binary approximations. Use `Decimal("0.1") * 3`, which gives `Decimal('0.3')`; for display only, `f"{price * qty:.2f}"` also works.
13. `random` is predictable: an attacker who sees some outputs can predict others. Use `secrets.choice(string.digits)` (or `f"{secrets.randbelow(10**6):06d}"`).

**D.**
14. 0.1, 0.2 and 0.3 have no exact binary form, so each is stored with a tiny error, and the errors do not cancel. Compare with `math.isclose(a, b)` (with `abs_tol` near zero), or use `Decimal`/`Fraction` when exactness matters.
15. `random` is a fast, seedable, predictable generator: good for simulations, games, tests and shuffling data. `secrets` uses the operating system's secure source and cannot be seeded: use it for passwords, reset tokens, API keys and session ids.

In [372]:
# Solutions 7.7
import math
import random
import secrets
import statistics
from collections import Counter
from decimal import Decimal, ROUND_DOWN
from fractions import Fraction

# A: checks
assert (math.floor(-0.5), math.ceil(-0.5), round(0.5), round(1.5)) == (-1, 0, 0, 2)
assert Decimal("1.10") + Decimal("2.205") == Decimal("3.305")
assert Fraction(2, 4) + Fraction(1, 4) == Fraction(3, 4)
assert random.shuffle([3, 1, 2]) is None

# 5. Prime check: O(sqrt(n))
def is_prime(n):
    if n < 2:
        return False
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

assert [n for n in range(20) if is_prime(n)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert not is_prime(-7)

# 6. LCM of a list
def lcm_of_list(nums):
    return math.lcm(*nums)

assert lcm_of_list([4, 6, 10]) == 60
assert lcm_of_list([7]) == 7

# 7. Summary statistics
def summary(data):
    if len(data) < 2:
        return None
    return {
        "mean": round(statistics.mean(data), 2),
        "median": statistics.median(data),
        "stdev": round(statistics.stdev(data), 2),
    }

assert summary([2, 4, 4, 4, 5, 5, 7, 9]) == {"mean": 5, "median": 4.5, "stdev": 2.14}
assert summary([1]) is None

# 8. Split a bill exactly
def split_bill(total, people):
    total = Decimal(total)
    base = (total / people).quantize(Decimal("0.01"), rounding=ROUND_DOWN)
    shares = [base] * people
    leftover_paise = int((total - base * people) / Decimal("0.01"))
    for i in range(leftover_paise):
        shares[i] += Decimal("0.01")
    return shares

shares = split_bill("100.00", 3)
assert shares == [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")]
assert sum(shares) == Decimal("100.00")
assert split_bill("10", 2) == [Decimal("5.00"), Decimal("5.00")]

# 9. Secure token
def make_token():
    return secrets.token_hex(16)

tok = make_token()
assert len(tok) == 32 and all(c in "0123456789abcdef" for c in tok)
assert make_token() != make_token()

# 10. Reproducible dice simulation
def simulate_dice(rolls, seed):
    rng = random.Random(seed)
    return Counter(rng.randint(1, 6) for _ in range(rolls))

first = simulate_dice(600, seed=1)
assert first == simulate_dice(600, seed=1)       # same seed, same result
assert first.total() == 600 and set(first) <= set(range(1, 7))

# 11. Reservoir sampling: item i (0-based) replaces a slot with probability k / (i + 1)
def reservoir_sample(stream, k, rng):
    sample = []
    for i, item in enumerate(stream):
        if i < k:
            sample.append(item)
        else:
            j = rng.randint(0, i)
            if j < k:
                sample[j] = item
    return sample

picked = reservoir_sample(iter(range(1000)), 5, random.Random(3))
assert len(picked) == 5 and len(set(picked)) == 5
assert reservoir_sample(iter([1, 2]), 5, random.Random(0)) == [1, 2]
# Rough fairness check: each of 10 items should be picked about 30% of the time
counts = Counter()
rng = random.Random(11)
for _ in range(3000):
    counts.update(reservoir_sample(range(10), 3, rng))
assert all(700 < counts[i] < 1100 for i in range(10))

# 12. Fixed money calculation
assert Decimal("0.1") * 3 == Decimal("0.3")

# 13. Fixed secure reset code
def reset_code():
    return f"{secrets.randbelow(10**6):06d}"

code = reset_code()
assert len(code) == 6 and code.isdigit()
print("Practice 7.7: all checks passed")

Practice 7.7: all checks passed


---

## 7.8 Dates and times: datetime, zoneinfo, time

The `datetime` module has five main types:

| Type | Holds | Example |
|---|---|---|
| `date` | year, month, day | `date(2026, 10, 24)` |
| `time` | hour, minute, second, microsecond | `time(14, 30)` |
| `datetime` | date + time (+ optional time zone) | `datetime(2026, 10, 24, 14, 30)` |
| `timedelta` | a duration | `timedelta(days=2, hours=3)` |
| `timezone` | a fixed offset from UTC | `timezone.utc`, `timezone(timedelta(hours=5, minutes=30))` |

### 7.8.1 date, datetime and timedelta

In [373]:
# Dates and date arithmetic with timedelta
from datetime import date, timedelta

d = date(2026, 10, 24)
print(d)                              # 2026-10-24
print(d.year, d.month, d.day)         # 2026 10 24
print(d.weekday())                    # 5  (Monday is 0, so 5 is Saturday)
print(d + timedelta(days=10))         # 2026-11-03
print((date(2026, 12, 25) - d).days)  # 62  (subtracting dates gives a timedelta)
print(d.replace(day=1))               # 2026-10-01  (dates are immutable; replace returns a new one)

2026-10-24
2026 10 24
5
2026-11-03
62
2026-10-01


In [374]:
# datetime and timedelta in more detail
from datetime import datetime, timedelta

start = datetime(2026, 10, 24, 9, 15)
end = datetime(2026, 10, 25, 11, 45)

gap = end - start
print(gap)                         # 1 day, 2:30:00
print(gap.days, gap.seconds)       # 1 9000  (seconds is only the part beyond whole days)
print(gap.total_seconds() / 3600)  # 26.5  (use total_seconds() for the full length)
print(start + timedelta(hours=36)) # 2026-10-25 21:15:00

1 day, 2:30:00
1 9000
26.5
2026-10-25 21:15:00


> **Warning:** `timedelta` has no `months` or `years`, because months have different lengths. To add a month, change the month field yourself (and handle day 31), or use the third-party `dateutil.relativedelta`.

### 7.8.2 Formatting and parsing

* `strftime` = **str**ing **f**rom **time**: datetime -> text.
* `strptime` = **str**ing **p**arse **time**: text -> datetime.
* `isoformat()` / `fromisoformat()` use the standard ISO 8601 format (`2026-10-24T14:30:00`). Prefer ISO for files and APIs.

| Code | Meaning | Example |
|---|---|---|
| `%Y` / `%m` / `%d` | year / month / day (zero padded) | 2026 / 10 / 24 |
| `%H` / `%M` / `%S` | hour (24h) / minute / second | 14 / 05 / 09 |
| `%I` / `%p` | hour (12h) / AM or PM | 02 / PM |
| `%A` / `%a` | weekday name / short | Saturday / Sat |
| `%B` / `%b` | month name / short | October / Oct |
| `%z` | UTC offset | +0530 |

In [375]:
# strftime (datetime -> text) and strptime (text -> datetime)
from datetime import datetime

meeting = datetime(2026, 10, 24, 14, 5, 9)
print(meeting.strftime("%d/%m/%Y %H:%M"))        # 24/10/2026 14:05
print(meeting.strftime("%A, %d %B %Y, %I:%M %p")) # Saturday, 24 October 2026, 02:05 PM
print(f"{meeting:%Y-%m-%d}")                     # 2026-10-24  (format codes work in f-strings)

parsed = datetime.strptime("24/10/2026 09:05", "%d/%m/%Y %H:%M")
print(parsed)                                    # 2026-10-24 09:05:00
print(repr(parsed))                              # datetime.datetime(2026, 10, 24, 9, 5)

24/10/2026 14:05
Saturday, 24 October 2026, 02:05 PM
2026-10-24
2026-10-24 09:05:00
datetime.datetime(2026, 10, 24, 9, 5)


In [376]:
# ISO 8601 round trip, and parsing errors
from datetime import datetime, date

stamp = datetime(2026, 10, 24, 14, 30)
text = stamp.isoformat()
print(text)                                      # 2026-10-24T14:30:00
print(datetime.fromisoformat(text) == stamp)     # True
print(date.fromisoformat("2026-12-25"))          # 2026-12-25
print(datetime.fromisoformat("2026-10-24T09:00:00Z"))   # 2026-10-24 09:00:00+00:00 ("Z" accepted since 3.11)

try:
    datetime.strptime("2026-13-01", "%Y-%m-%d")  # there is no month 13
except ValueError as e:
    print("ValueError:", e)

2026-10-24T14:30:00
True
2026-12-25
2026-10-24 09:00:00+00:00
ValueError: time data '2026-13-01' does not match format '%Y-%m-%d'


In [377]:
# Python 3.14+: date.strptime and time.strptime class methods
import sys
from datetime import date

if sys.version_info >= (3, 14):
    print(date.strptime("24 Oct 2026", "%d %b %Y"))   # 2026-10-24
else:
    from datetime import datetime
    print(datetime.strptime("24 Oct 2026", "%d %b %Y").date())   # same result on older versions

2026-10-24


### 7.8.3 Naive vs aware datetimes and UTC

* A **naive** datetime has no time zone (`tzinfo is None`). Python does not know which moment in the world it means.
* An **aware** datetime has a `tzinfo`. It is an exact moment.

You **cannot compare or subtract** a naive and an aware datetime: Python raises `TypeError`.

**Best practice:** store and exchange times as **aware UTC** values. Convert to the user's local time zone only when you display them.

```text
   user input (local)      store / compute         display (local)
  +------------------+    +-----------------+    +------------------+
  | 09:00 IST        |--->| 03:30 UTC       |--->| 23:30 EDT (prev) |
  | (aware, +05:30)  |    | (aware, +00:00) |    | (aware, -04:00)  |
  +------------------+    +-----------------+    +------------------+
          astimezone(timezone.utc)      astimezone(user_zone)
```

In [378]:
# Aware datetimes with timezone.utc and fixed offsets
from datetime import datetime, timedelta, timezone

utc_moment = datetime(2026, 10, 24, 3, 30, tzinfo=timezone.utc)
ist = timezone(timedelta(hours=5, minutes=30), "IST")

print(utc_moment.isoformat())                  # 2026-10-24T03:30:00+00:00
print(utc_moment.astimezone(ist).isoformat())  # 2026-10-24T09:00:00+05:30
print(utc_moment.astimezone(ist).tzname())     # IST

# The same moment written in two zones is equal
print(utc_moment == datetime(2026, 10, 24, 9, 0, tzinfo=ist))   # True

2026-10-24T03:30:00+00:00
2026-10-24T09:00:00+05:30
IST
True


In [379]:
# Common mistake: mixing naive and aware datetimes
from datetime import datetime, timezone

naive = datetime(2026, 10, 24, 9, 0)
aware = datetime(2026, 10, 24, 9, 0, tzinfo=timezone.utc)
print(naive.tzinfo, aware.tzinfo)        # None UTC

try:
    print(naive < aware)
except TypeError as e:
    print("TypeError:", e)               # TypeError: can't compare offset-naive and offset-aware datetimes

# Fix: attach the correct zone if you KNOW the naive value is UTC
print(naive.replace(tzinfo=timezone.utc) == aware)   # True

None UTC
TypeError: can't compare offset-naive and offset-aware datetimes
True


In [380]:
# The current time: use datetime.now(timezone.utc), not utcnow()
import warnings
from datetime import datetime, timezone

now_utc = datetime.now(timezone.utc)
print(now_utc.tzinfo)                    # UTC  (aware)

# datetime.utcnow() is deprecated since 3.12 and returns a NAIVE value
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old_style = datetime.utcnow()
print(old_style.tzinfo)                  # None
print(caught[0].category.__name__)       # DeprecationWarning

UTC
None
DeprecationWarning


### 7.8.4 Named time zones with zoneinfo

A fixed offset like +05:30 is fine for India, which has no **daylight saving time (DST)**. Many zones change their offset during the year (New York is -05:00 in winter and -04:00 in summer). For those you need the **IANA time zone database** and `zoneinfo.ZoneInfo("America/New_York")` (Python 3.9+).

> **Warning:** On Windows (and in some slim containers) the time zone database may be missing, and `ZoneInfo(...)` raises `ZoneInfoNotFoundError`. Fix it with `python -m pip install tzdata`. Robust code catches the error and falls back to something sensible.

In [381]:
# Getting a named zone safely, with a fixed-offset fallback
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

def get_zone(name, fallback_hours):
    """Return ZoneInfo(name), or a fixed-offset zone if the tz database is missing."""
    try:
        return ZoneInfo(name)
    except ZoneInfoNotFoundError:
        return timezone(timedelta(hours=fallback_hours), name)

kolkata = get_zone("Asia/Kolkata", 5.5)
tokyo = get_zone("Asia/Tokyo", 9)

flight = datetime(2026, 10, 24, 3, 30, tzinfo=timezone.utc)
print(flight.astimezone(kolkata).isoformat())   # 2026-10-24T09:00:00+05:30
print(flight.astimezone(tokyo).isoformat())     # 2026-10-24T12:30:00+09:00
# (Kolkata and Tokyo have no DST, so the fallback gives the same answer.)

2026-10-24T09:00:00+05:30
2026-10-24T12:30:00+09:00


In [382]:
# Daylight saving time needs the real tz database
from datetime import datetime
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

try:
    new_york = ZoneInfo("America/New_York")
except ZoneInfoNotFoundError:
    new_york = None

if new_york is None:
    print("tz database not installed: run  python -m pip install tzdata")
else:
    winter = datetime(2026, 1, 15, 12, 0, tzinfo=new_york)
    summer = datetime(2026, 7, 15, 12, 0, tzinfo=new_york)
    print(winter.utcoffset(), winter.tzname())   # -1 day, 19:00:00 EST  (that is -5 hours)
    print(summer.utcoffset(), summer.tzname())   # -1 day, 20:00:00 EDT  (that is -4 hours)

tz database not installed: run  python -m pip install tzdata


### 7.8.5 Timestamps and the time module

A **Unix timestamp** is the number of seconds since 1970-01-01 00:00:00 UTC (the "epoch").

* `time.time()`: current timestamp (wall clock). It can jump when the system clock is changed.
* `time.perf_counter()`: a high-resolution clock for **measuring durations**. Use this for timing code.
* `time.monotonic()`: a clock that never goes backwards. Use it for timeouts.
* `time.sleep(seconds)`: pause the program.

In [383]:
# Converting between timestamps and datetimes
from datetime import datetime, timezone

epoch = datetime.fromtimestamp(0, tz=timezone.utc)
print(epoch)                                        # 1970-01-01 00:00:00+00:00

moment = datetime(2026, 10, 24, tzinfo=timezone.utc)
print(moment.timestamp())                           # 1792800000.0
print(datetime.fromtimestamp(1792800000, tz=timezone.utc) == moment)   # True

1970-01-01 00:00:00+00:00
1792800000.0
True


In [384]:
# Measuring how long code takes with perf_counter
import time

start = time.perf_counter()
total = sum(range(100_000))
elapsed = time.perf_counter() - start

print(total)                     # 4999950000
print(elapsed < 1.0)             # True  (exact timings vary from run to run)

4999950000
True


### Common mistakes with dates and times

* **Mixing naive and aware datetimes** raises `TypeError`. Make everything aware (UTC) at the edges of your program.
* **Using `datetime.utcnow()`**: deprecated, and returns a naive value. Use `datetime.now(timezone.utc)`.
* **Using `gap.seconds` instead of `gap.total_seconds()`**: `seconds` ignores whole days.
* **Using `time.time()` to measure code speed**: use `time.perf_counter()`.
* **Storing local times** in databases: daylight saving makes some local times happen twice or never.
* **Assuming `ZoneInfo` always works**: on Windows install `tzdata`, and catch `ZoneInfoNotFoundError`.

### How to debug date and time code

1. Print `repr(dt)`, not `print(dt)`: the repr shows whether `tzinfo` is set.
2. Check `dt.tzinfo` and `dt.utcoffset()` on every value involved in a comparison.
3. For parsing errors, compare the input text and the format string character by character (`"%d/%m"` will not parse `"24-10"`).
4. Test edge cases: end of month, leap day (`date(2028, 2, 29)`), year end, DST change days.

> **Key points**
> * `date`, `datetime`, `timedelta` handle calendar maths; `timedelta` has no months.
> * `strftime` formats, `strptime` parses; ISO 8601 (`isoformat` / `fromisoformat`) is the safest text format.
> * Naive vs aware: they cannot be mixed; store aware UTC, convert for display.
> * `datetime.now(timezone.utc)`, never `utcnow()`; `ZoneInfo` for named zones (guard with `ZoneInfoNotFoundError`).
> * `perf_counter` for timing, `monotonic` for timeouts, `time.time` for timestamps.

### Practice 7.8: dates and times

**A. Predict the output**

1. What does this print?
   ```python
   from datetime import date, timedelta

   print(date(2028, 2, 28) + timedelta(days=1))
   ```
2. What does this print?
   ```python
   from datetime import datetime

   gap = datetime(2026, 1, 2, 1, 0) - datetime(2026, 1, 1, 0, 0)
   print(gap.seconds, gap.total_seconds())
   ```
3. Does this raise an error? Why?
   ```python
   from datetime import datetime, timezone

   print(datetime.now() - datetime.now(timezone.utc))
   ```

**B. Write code**

4. Write `days_until(iso_date, today)` that returns how many days remain from `today` (a `date`) to the ISO date string.
5. Write `age_on(birth, on_day)` that returns a person's age in whole years.
6. Write `to_utc(iso_local, offset_hours)` that takes a naive ISO string and its UTC offset in hours (may be fractional, like 5.5) and returns an aware UTC ISO string.
7. Write `parse_flexible(text)` that tries the formats `"%Y-%m-%d"`, `"%d/%m/%Y"` and `"%d %b %Y"` and returns a `date`, or raises `ValueError` if none match.
8. Write `weekdays_between(start, end)` that counts Monday-to-Friday days in the half-open range `[start, end)`.
9. **[Hard]** Write `add_months(d, months)` that adds months to a date and clamps the day to the end of the month (`2026-01-31` + 1 month = `2026-02-28`). Hint: `calendar.monthrange(year, month)[1]` gives the number of days.

**C. Find and fix the bug**

10. This should print the number of hours in the gap, but prints a small number.
    ```python
    from datetime import datetime

    gap = datetime(2026, 10, 26, 6, 0) - datetime(2026, 10, 24, 6, 0)
    print(gap.seconds / 3600)
    ```
11. This crashes with `TypeError`. Fix it.
    ```python
    from datetime import datetime, timezone

    deadline = datetime(2026, 12, 31, 23, 59)
    now = datetime.now(timezone.utc)
    print(now < deadline)
    ```

**D. Interview questions**

12. What is the difference between a naive and an aware datetime? Why store UTC?
13. **[Hard]** Why should you use `time.perf_counter()` or `time.monotonic()` and not `time.time()` to measure durations?

### Solutions 7.8

**A.**
1. `2028-02-29`. 2028 is a leap year.
2. `3600 90000.0`. The gap is 1 day and 1 hour; `.seconds` holds only the 1 hour beyond whole days.
3. Yes, `TypeError: can't subtract offset-naive and offset-aware datetimes`. `datetime.now()` is naive, `datetime.now(timezone.utc)` is aware.

**C.**
10. `.seconds` ignores the 2 whole days, so it prints `0.0`. Use `gap.total_seconds() / 3600`, which gives `48.0`.
11. `deadline` is naive. Make it aware: `datetime(2026, 12, 31, 23, 59, tzinfo=timezone.utc)` (or the correct local zone).

**D.**
12. A naive datetime has no time zone, so it does not identify a single moment; an aware one does. UTC has no daylight saving jumps, so every UTC time is unique and easy to compare and sort; local time is only for display.
13. `time.time()` follows the system clock, which can be changed by the user or by time synchronisation (it can even go backwards). `perf_counter` is high resolution and `monotonic` never goes backwards, so the difference between two readings is a true duration.

In [385]:
# Solutions 7.8
import calendar
from datetime import date, datetime, timedelta, timezone

# A: checks
assert date(2028, 2, 28) + timedelta(days=1) == date(2028, 2, 29)
gap = datetime(2026, 1, 2, 1, 0) - datetime(2026, 1, 1, 0, 0)
assert (gap.seconds, gap.total_seconds()) == (3600, 90000.0)
try:
    datetime.now() - datetime.now(timezone.utc)
    raise AssertionError("expected TypeError")
except TypeError:
    pass

# 4. Days until a date
def days_until(iso_date, today):
    return (date.fromisoformat(iso_date) - today).days

assert days_until("2026-12-25", date(2026, 10, 24)) == 62
assert days_until("2026-10-24", date(2026, 10, 24)) == 0

# 5. Age in whole years: subtract 1 if the birthday has not happened yet this year
def age_on(birth, on_day):
    before_birthday = (on_day.month, on_day.day) < (birth.month, birth.day)
    return on_day.year - birth.year - before_birthday

assert age_on(date(2000, 10, 25), date(2026, 10, 24)) == 25
assert age_on(date(2000, 10, 24), date(2026, 10, 24)) == 26

# 6. Local ISO string with an offset -> aware UTC ISO string
def to_utc(iso_local, offset_hours):
    zone = timezone(timedelta(hours=offset_hours))
    local = datetime.fromisoformat(iso_local).replace(tzinfo=zone)
    return local.astimezone(timezone.utc).isoformat()

assert to_utc("2026-10-24T09:00:00", 5.5) == "2026-10-24T03:30:00+00:00"
assert to_utc("2026-01-01T01:00:00", 2) == "2025-12-31T23:00:00+00:00"

# 7. Try several formats
FORMATS = ("%Y-%m-%d", "%d/%m/%Y", "%d %b %Y")

def parse_flexible(text):
    for fmt in FORMATS:
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            continue
    raise ValueError(f"unrecognised date: {text!r}")

assert parse_flexible("2026-10-24") == parse_flexible("24/10/2026") == parse_flexible("24 Oct 2026")
try:
    parse_flexible("yesterday")
    raise AssertionError("expected ValueError")
except ValueError as e:
    assert "yesterday" in str(e)

# 8. Count weekdays in [start, end)
def weekdays_between(start, end):
    days = (end - start).days
    return sum(1 for i in range(days) if (start + timedelta(days=i)).weekday() < 5)

assert weekdays_between(date(2026, 10, 19), date(2026, 10, 26)) == 5   # Monday to next Monday
assert weekdays_between(date(2026, 10, 24), date(2026, 10, 24)) == 0

# 9. Add months with clamping
def add_months(d, months):
    index = d.month - 1 + months
    year = d.year + index // 12
    month = index % 12 + 1
    day = min(d.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)

assert add_months(date(2026, 1, 31), 1) == date(2026, 2, 28)
assert add_months(date(2028, 1, 31), 1) == date(2028, 2, 29)
assert add_months(date(2026, 11, 15), 3) == date(2027, 2, 15)
assert add_months(date(2026, 3, 31), -1) == date(2026, 2, 28)

# 10. Fixed: total_seconds
gap = datetime(2026, 10, 26, 6, 0) - datetime(2026, 10, 24, 6, 0)
assert gap.seconds / 3600 == 0.0
assert gap.total_seconds() / 3600 == 48.0

# 11. Fixed: make both aware
deadline = datetime(2026, 12, 31, 23, 59, tzinfo=timezone.utc)
assert isinstance(datetime.now(timezone.utc) < deadline, bool)
print("Practice 7.8: all checks passed")

Practice 7.8: all checks passed


---

## 7.9 Regular expressions: re

A **regular expression** (regex) is a small pattern language for text. With one pattern you can ask "does this look like a date?", "find every order number", or "replace all runs of spaces with one space".

**In simple words:** string methods like `find`, `split` and `replace` work with fixed text. A regex works with a *shape* of text, like "4 digits, a dash, 2 digits".

> **Tip:** Always write patterns as **raw strings**: `r"\d+"`. In a raw string the backslash is kept as is, so `\d` reaches the `re` module unchanged. In a normal string, `"\b"` is a backspace character, not a word boundary.

### 7.9.1 Pattern syntax

Characters and classes:

| Pattern | Matches |
|---|---|
| `.` | any character except a newline |
| `\d` / `\D` | a digit / not a digit |
| `\w` / `\W` | a word character (letter, digit, `_`) / not one |
| `\s` / `\S` | whitespace (space, tab, newline) / not whitespace |
| `[abc]` / `[^abc]` | one of a, b, c / any character except a, b, c |
| `[a-z0-9]` | a range of characters |
| `\.` `\$` `\(` | a literal `.`, `$`, `(` (escape special characters) |

Quantifiers, anchors and groups:

| Pattern | Meaning |
|---|---|
| `*` / `+` / `?` | 0 or more / 1 or more / 0 or 1 |
| `{3}` / `{2,4}` | exactly 3 / between 2 and 4 |
| `*?` `+?` `??` | **lazy** versions (as few as possible) |
| `^` / `$` | start / end of the string (of each line with `re.MULTILINE`) |
| `\b` | word boundary |
| `(...)` | capturing group |
| `(?P<name>...)` | named group |
| `(?:...)` | group without capturing |
| `(?=...)` / `(?!...)` | lookahead: followed by / not followed by |

Alternation: the pattern `cat|dog` matches "cat" or "dog" (the `|` means "or").

### 7.9.2 The main functions

| Function | Returns |
|---|---|
| `re.search(p, s)` | first match **anywhere**, or `None` |
| `re.match(p, s)` | match only at the **start** of `s`, or `None` |
| `re.fullmatch(p, s)` | match only if the **whole** string matches, or `None` |
| `re.findall(p, s)` | list of all matches (strings, or tuples of groups) |
| `re.finditer(p, s)` | iterator of match objects (with positions) |
| `re.sub(p, repl, s)` | new string with matches replaced |
| `re.split(p, s)` | list of pieces between matches |
| `re.compile(p)` | a pattern object to reuse |

```text
  What do you need?
        |
        +-- yes/no: does the WHOLE string match?    -> re.fullmatch
        +-- yes/no: is the pattern somewhere?       -> re.search
        +-- every match as text                     -> re.findall
        +-- every match with positions / groups     -> re.finditer
        +-- replace matches                         -> re.sub
        +-- split on a pattern                      -> re.split
```

In [386]:
# search vs match vs fullmatch
import re

text = "order 66 shipped"
print(re.search(r"\d+", text))             # <re.Match object; span=(6, 8), match='66'>
print(re.match(r"\d+", text))              # None  (match only looks at the start)
print(re.match(r"order", text) is not None)   # True

print(re.fullmatch(r"\d+", "12345") is not None)   # True
print(re.fullmatch(r"\d+", "123a5"))               # None

<re.Match object; span=(6, 8), match='66'>
None
True
True
None


In [387]:
# A match object: the matched text, its position and its groups
import re

m = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Shipped on 2026-10-24 by courier")
print(m.group())        # 2026-10-24  (the whole match, same as group(0))
print(m.group(1))       # 2026
print(m.groups())       # ('2026', '10', '24')
print(m.span())         # (11, 21)
print(m.start(), m.end())   # 11 21

2026-10-24
2026
('2026', '10', '24')
(11, 21)
11 21


In [388]:
# Named groups make patterns self-documenting
import re

line = "2026-10-24 14:30:01 ERROR payments: card declined"
pattern = r"(?P<date>\S+) (?P<time>\S+) (?P<level>[A-Z]+) (?P<source>\w+): (?P<message>.*)"

m = re.match(pattern, line)
print(m.group("level"))     # ERROR
print(m["source"])          # payments  (m["name"] is the same as m.group("name"))
print(m.groupdict())
# {'date': '2026-10-24', 'time': '14:30:01', 'level': 'ERROR', 'source': 'payments', 'message': 'card declined'}

ERROR
payments
{'date': '2026-10-24', 'time': '14:30:01', 'level': 'ERROR', 'source': 'payments', 'message': 'card declined'}


In [389]:
# findall: without groups you get the matches, with groups you get the groups
import re

text = "Order #123 on 2026-10-24; order #456 on 2026-11-02."
print(re.findall(r"#\d+", text))           # ['#123', '#456']
print(re.findall(r"#(\d+)", text))         # ['123', '456']  (only the group)
print(re.findall(r"(\d{4})-(\d{2})", text))   # [('2026', '10'), ('2026', '11')]  (tuples)

# finditer gives match objects, so you also get positions
for m in re.finditer(r"#(\d+)", text):
    print(m.group(1), m.start())
# 123 6
# 456 32

['#123', '#456']
['123', '456']
[('2026', '10'), ('2026', '11')]
123 6
456 32


In [390]:
# sub: replace with text, with backreferences, or with a function
import re

text = "Shipped 2026-10-24, delivered 2026-10-27"
print(re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", text))
# Shipped 24/10/2026, delivered 27/10/2026

print(re.sub(r"\s+", " ", "too    many \t spaces"))    # too many spaces

def double(m):
    return str(int(m.group()) * 2)

print(re.sub(r"\d+", double, "3 apples and 10 pears"))  # 6 apples and 20 pears

Shipped 24/10/2026, delivered 27/10/2026
too many spaces
6 apples and 20 pears


In [391]:
# split on a pattern
import re

print(re.split(r"[;,]\s*", "red, green;blue,  black"))   # ['red', 'green', 'blue', 'black']
print(re.split(r"(\d)", "a1b2c"))                         # ['a', '1', 'b', '2', 'c']  (a group keeps the separators)

['red', 'green', 'blue', 'black']
['a', '1', 'b', '2', 'c']


### 7.9.3 Greedy vs lazy

Quantifiers are **greedy** by default: they match as much as they can, then give back only what is needed. Adding `?` makes them **lazy**: they match as little as possible.

```text
  text:      <b>bold</b> and <i>italic</i>
  <.+>   ->  <b>bold</b> and <i>italic</i>     (greedy: one huge match)
  <.+?>  ->  <b>   </b>   <i>   </i>           (lazy: four small matches)
```

In [392]:
# Greedy .+ swallows too much; lazy .+? stops early
import re

html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.+>", html))     # ['<b>bold</b> and <i>italic</i>']
print(re.findall(r"<.+?>", html))    # ['<b>', '</b>', '<i>', '</i>']
print(re.findall(r"<[^>]+>", html))  # ['<b>', '</b>', '<i>', '</i>']  (a negated class is often clearer)

['<b>bold</b> and <i>italic</i>']
['<b>', '</b>', '<i>', '</i>']
['<b>', '</b>', '<i>', '</i>']


### 7.9.4 Flags, compile, lookahead and escape

* `re.IGNORECASE` (`re.I`): ignore upper/lower case.
* `re.MULTILINE` (`re.M`): `^` and `$` match at every line.
* `re.DOTALL` (`re.S`): `.` also matches a newline.
* `re.VERBOSE` (`re.X`): allows spaces and `#` comments inside the pattern.

In [393]:
# Flags change how a pattern matches
import re

print(re.findall(r"python", "Python PYTHON python", re.IGNORECASE))   # ['Python', 'PYTHON', 'python']

log = "INFO start\nERROR disk\nINFO done"
print(re.findall(r"^\w+", log))                 # ['INFO']  (only the start of the string)
print(re.findall(r"^\w+", log, re.MULTILINE))   # ['INFO', 'ERROR', 'INFO']

print(re.search(r"start.*done", log))                     # None (. stops at newlines)
print(re.search(r"start.*done", log, re.DOTALL) is not None)   # True

['Python', 'PYTHON', 'python']
['INFO']
['INFO', 'ERROR', 'INFO']
None
True


In [394]:
# compile + VERBOSE: a readable, reusable pattern
import re

EMAIL = re.compile(r"""
    ^[\w.+-]+          # user name: letters, digits, dot, plus, dash
    @
    [\w-]+             # domain name
    (\.[\w-]+)+$       # one or more .parts, like .co.in
""", re.VERBOSE)

for address in ["asha@example.com", "ravi.k+news@mail.co.in", "not-an-email", "a@b"]:
    print(address, bool(EMAIL.match(address)))
# asha@example.com True
# ravi.k+news@mail.co.in True
# not-an-email False
# a@b False

asha@example.com True
ravi.k+news@mail.co.in True
not-an-email False
a@b False


> **Note:** This email pattern is a practical check, not a full implementation of the email standard. For real sign-ups, check the basic shape and then send a confirmation email.

In [395]:
# Lookahead: check several rules at one position without consuming text
import re

STRONG = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")

for pw in ["Secret123", "secret123", "Sh0rt", "LongerPassword9"]:
    print(pw, bool(STRONG.match(pw)))
# Secret123 True
# secret123 False      (no upper-case letter)
# Sh0rt False          (too short)
# LongerPassword9 True

Secret123 True
secret123 False
Sh0rt False
LongerPassword9 True


In [396]:
# re.escape: search for user text literally (dots, plus signs, brackets...)
import re

user_text = "3.5+"
print(re.findall(user_text, "3.5+ or 345"))            # ['3.5', '345']  (wrong: . and + are special)
print(re.findall(re.escape(user_text), "3.5+ or 345")) # ['3.5+']

['3.5', '345']
['3.5+']


### Common mistakes with regular expressions

* **`match` instead of `search`**: `match` only checks the start of the string.
* **Forgetting the `r` prefix**: `"\bword\b"` contains backspace characters, so it never matches.
* **Unescaped `.`**: `r"3.5"` also matches `"345"`. Write `r"3\.5"`.
* **Greedy `.*`** matching too much. Use `.*?` or a negated class like `[^"]*`.
* **`findall` with groups** returns the groups, not the full match. Use `(?:...)` for groups you do not want returned.
* **Catastrophic backtracking**: nested quantifiers like `(a+)+$` can take exponential time on inputs like `"aaaa...ab"`. That can freeze a server (a "ReDoS" attack). Keep patterns simple, avoid nested quantifiers, limit input length. Python 3.11+ also supports atomic groups `(?>...)` and possessive quantifiers `*+`, `++` that prevent backtracking.
* **Parsing HTML, JSON or CSV with regex**: use real parsers (`html.parser`, `json`, `csv`).

### How to debug a regex

1. Test the pattern on the smallest string that should match and one that should not.
2. Build the pattern up piece by piece, checking each piece with `re.findall`.
3. Print `m.groups()` or `m.groupdict()` to see what each group captured.
4. Use `re.VERBOSE` with comments for anything longer than about 20 characters.
5. Read `re.error` messages: they include the position of the problem.

In [397]:
# A broken pattern raises re.error (also called re.PatternError in 3.13+)
import re

try:
    re.compile(r"(\d+")                 # missing closing parenthesis
except re.error as e:
    print("re.error:", e)               # re.error: missing ), unterminated subpattern at position 0
    print("position:", e.pos)           # position: 0

# Calling .group() on None is the most common regex crash
m = re.search(r"\d+", "no digits here")
try:
    m.group()
except AttributeError as e:
    print("AttributeError:", e)         # AttributeError: 'NoneType' object has no attribute 'group'

# Fix: always check for None first
print(m.group() if m else "no match")   # no match

re.error: missing ), unterminated subpattern at position 0
position: 0
AttributeError: 'NoneType' object has no attribute 'group'
no match


> **Key points**
> * Write patterns as raw strings; `\d \w \s`, classes `[...]`, quantifiers `* + ? {m,n}`, anchors `^ $ \b`.
> * `search` finds anywhere, `match` only at the start, `fullmatch` the whole string; check for `None`.
> * Named groups `(?P<name>...)` with `groupdict()` make parsing code readable.
> * Greedy vs lazy (`.*` vs `.*?`); use `re.escape` for literal user text; `re.VERBOSE` for long patterns.
> * Avoid nested quantifiers on untrusted input; do not parse structured formats with regex.

### Practice 7.9: regular expressions

**A. Predict the output**

1. What does this print?
   ```python
   import re

   print(re.findall(r"\d+", "a1b22c333"))
   ```
2. What does this print?
   ```python
   import re

   print(re.match(r"\d+", "abc123"), re.search(r"\d+", "abc123").group())
   ```
3. What does this print?
   ```python
   import re

   print(re.findall(r"(\w)(\d)", "a1 b2 c"))
   ```
4. What does this print?
   ```python
   import re

   print(re.sub(r"a+", "-", "caaandy and aa"))
   ```

**B. Write code**

5. Write `extract_emails(text)` that returns all email addresses in a text.
6. Write `is_valid_pin(s)`: `True` if `s` is exactly 4 or 6 digits.
7. Write `parse_log_line(line)` that uses named groups to parse `"2026-10-24 14:30:01 ERROR payments: card declined"` into a dict with `date`, `time`, `level`, `source`, `message`, or returns `None` if the line does not match.
8. Write `normalise_spaces(text)` that trims the text and collapses every run of whitespace into one space.
9. Write `camel_to_snake(name)`: `"parseHttpResponse"` becomes `"parse_http_response"`.
10. Write `mask_card(text)` that replaces every 16-digit card number (optionally in groups of 4 separated by spaces or dashes) with `****-****-****-1234` keeping the last 4 digits.
11. **[Hard]** Write `word_frequencies(text)` that lower-cases, extracts words with a regex (letters and apostrophes), and returns the top 3 as `(word, count)` pairs, ties alphabetical.

**C. Find and fix the bug**

12. This should find the whole word "cat" but finds nothing. Why?
    ```python
    import re

    print(re.findall("\bcat\b", "the cat sat"))
    ```
13. This should extract the text inside double quotes, but returns one long string. Fix it.
    ```python
    import re

    print(re.findall(r'"(.*)"', 'say "hi" and "bye"'))
    ```

**D. Interview questions**

14. What is the difference between `re.match`, `re.search` and `re.fullmatch`?
15. **[Hard]** What is catastrophic backtracking, and how do you avoid it?

### Solutions 7.9

**A.**
1. `['1', '22', '333']`.
2. `None 123`. `match` only checks the start; `search` scans the whole string.
3. `[('a', '1'), ('b', '2')]`. Two groups give tuples; `c` has no digit after it.
4. `c-ndy -nd -`. Each run of one or more `a` becomes one `-`.

**C.**
12. Without the `r` prefix, `"\b"` is the backspace character, so the pattern looks for backspaces. Use `r"\bcat\b"`.
13. `.*` is greedy, so it runs from the first quote to the last one, giving `['hi" and "bye']`. Use the lazy `r'"(.*?)"'` or the negated class `r'"([^"]*)"'`.

**D.**
14. `match` must match at the start of the string, `search` finds the first match anywhere, `fullmatch` must match the entire string. For validation, use `fullmatch`.
15. Some patterns (usually nested or overlapping quantifiers such as `(a+)+` or `(a|aa)*`) can match the same text in very many ways. When the overall match fails, the engine tries every way, which can take exponential time. Avoid nested quantifiers, make alternatives exclusive, use atomic groups or possessive quantifiers (3.11+), limit the input length, and test with long non-matching input.

In [398]:
# Solutions 7.9
import re
from collections import Counter

# A: checks
assert re.findall(r"\d+", "a1b22c333") == ["1", "22", "333"]
assert re.findall(r"(\w)(\d)", "a1 b2 c") == [("a", "1"), ("b", "2")]
assert re.sub(r"a+", "-", "caaandy and aa") == "c-ndy -nd -"

# 5. Emails (a non-capturing group so findall returns whole matches)
def extract_emails(text):
    return re.findall(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", text)

assert extract_emails("mail a.b@x.org or c@d.co.in, not @nobody") == ["a.b@x.org", "c@d.co.in"]
assert extract_emails("") == []

# 6. PIN: exactly 4 or 6 digits
def is_valid_pin(s):
    return re.fullmatch(r"\d{4}|\d{6}", s) is not None

assert is_valid_pin("1234") and is_valid_pin("123456")
assert not is_valid_pin("12345") and not is_valid_pin("12a4") and not is_valid_pin("")

# 7. Log line parser
LOG_LINE = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) "
    r"(?P<level>[A-Z]+) (?P<source>\w+): (?P<message>.*)"
)

def parse_log_line(line):
    m = LOG_LINE.fullmatch(line)
    return m.groupdict() if m else None

assert parse_log_line("2026-10-24 14:30:01 ERROR payments: card declined") == {
    "date": "2026-10-24", "time": "14:30:01", "level": "ERROR",
    "source": "payments", "message": "card declined"}
assert parse_log_line("garbage") is None

# 8. Normalise whitespace
def normalise_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

assert normalise_spaces("  hello \t\n  world  ") == "hello world"
assert normalise_spaces("") == ""

# 9. camelCase -> snake_case: put "_" before each upper-case letter that follows a lower-case letter or digit
def camel_to_snake(name):
    return re.sub(r"(?<=[a-z0-9])([A-Z])", r"_\1", name).lower()

assert camel_to_snake("parseHttpResponse") == "parse_http_response"
assert camel_to_snake("already_snake") == "already_snake"
assert camel_to_snake("userID2Name") == "user_id2_name"

# 10. Mask card numbers
CARD = re.compile(r"\b(?:\d{4}[ -]?){3}(\d{4})\b")

def mask_card(text):
    return CARD.sub(lambda m: "****-****-****-" + m.group(1), text)

assert mask_card("card 1234 5678 9012 3456 ok") == "card ****-****-****-3456 ok"
assert mask_card("1234-5678-9012-3456") == "****-****-****-3456"
assert mask_card("1234567890123456") == "****-****-****-3456"
assert mask_card("call 12345") == "call 12345"

# 11. Top 3 words, ties alphabetical
def word_frequencies(text):
    counts = Counter(re.findall(r"[a-z']+", text.lower()))
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:3]

assert word_frequencies("The cat. The dog! A cat, the end") == [("the", 3), ("cat", 2), ("a", 1)]

# 12 and 13. Fixed patterns
assert re.findall("\bcat\b", "the cat sat") == []
assert re.findall(r"\bcat\b", "the cat sat") == ["cat"]
assert re.findall(r'"(.*)"', 'say "hi" and "bye"') == ['hi" and "bye']
assert re.findall(r'"(.*?)"', 'say "hi" and "bye"') == ["hi", "bye"]
print("Practice 7.9: all checks passed")

Practice 7.9: all checks passed


---

## 7.10 Logging and pprint

### 7.10.1 Why logging instead of print

`print` is fine while learning. Real programs use the **`logging`** module because a log message has:

* a **level** (how serious it is), so you can show only warnings in production and everything while debugging;
* a **logger name** (which part of the program wrote it);
* a **format** (time, level, module, line number) chosen in one place;
* a **destination** (console, file, log service) chosen by configuration, without changing the code.

| Level | Number | Use for |
|---|---|---|
| `DEBUG` | 10 | detailed information for diagnosing problems |
| `INFO` | 20 | normal events ("order placed") |
| `WARNING` | 30 | something unexpected, but the program still works (the default threshold) |
| `ERROR` | 40 | an operation failed |
| `CRITICAL` | 50 | the program may not be able to continue |

A message is shown only if its level is **at or above** the logger's level (and the handler's level).

### 7.10.2 The pieces: logger, handler, formatter

```text
  logger.info("order %s placed", 42)
          |
          v
  +--------------------+   level too low?  +-----------+
  | Logger "shop.cart" |------------------>| dropped   |
  +---------+----------+                   +-----------+
            | passes
            v
  +--------------------+     +--------------------------+
  | Handler(s)         |---->| Formatter: how it looks  |
  | where it goes      |     | "INFO shop.cart: ..."    |
  +---------+----------+     +--------------------------+
            | propagate=True
            v
  parent logger "shop" -> its handlers -> root logger
```

* **Logger**: what your code calls. Create one per module with `logging.getLogger(__name__)`. Names with dots form a tree: `shop.cart` is a child of `shop`.
* **Handler**: where messages go (`StreamHandler` for the console, `FileHandler` for a file, and many more).
* **Formatter**: how a message looks, for example `"%(asctime)s %(levelname)s %(name)s: %(message)s"`.

> **Note:** `logging.StreamHandler()` writes to standard error by default. The demos below pass `sys.stdout` so the messages appear as normal cell output, and they leave out the time stamp so the output is the same on every run.

In [399]:
# A logger with its own handler and formatter (printing to stdout for this notebook)
import logging
import sys

def make_console_logger(name, level=logging.DEBUG):
    """Return a logger that prints 'LEVEL name: message' lines to the cell output."""
    log = logging.getLogger(name)
    log.handlers.clear()                   # avoid duplicate handlers if a cell runs twice
    log.propagate = False                  # keep the demo independent of the root logger
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(levelname)-8s %(name)s: %(message)s"))
    log.addHandler(handler)
    log.setLevel(level)
    return log

logger = make_console_logger("shop")

logger.debug("cart has %d items", 3)       # DEBUG    shop: cart has 3 items
logger.info("order %s placed", "A-42")     # INFO     shop: order A-42 placed
logger.warning("stock low for %s", "pens") # WARNING  shop: stock low for pens

logger.setLevel(logging.WARNING)
logger.info("this INFO message is filtered out")   # nothing is printed
logger.error("payment service unavailable")         # ERROR    shop: payment service unavailable

DEBUG    shop: cart has 3 items
INFO     shop: order A-42 placed
WARNING  shop: stock low for pens
ERROR    shop: payment service unavailable


In [400]:
# Child loggers send their messages up to the parent's handlers
import logging

make_console_logger("shop", logging.WARNING)     # helper from the previous cell
payments = logging.getLogger("shop.payments")    # child of "shop"
print(payments.parent.name)                      # shop
print(payments.getEffectiveLevel() == logging.WARNING)   # True (inherited from "shop")

payments.warning("card %s expires soon", "****1234")
# WARNING  shop.payments: card ****1234 expires soon

shop
True
WARNING  shop.payments: card ****1234 expires soon


In [401]:
# logger.exception() inside except: logs at ERROR level and adds the traceback
import logging

logger = make_console_logger("shop")    # helper defined two cells above
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("could not compute discount")
# ERROR    shop: could not compute discount
# Traceback (most recent call last):
#   ...
# ZeroDivisionError: division by zero

ERROR    shop: could not compute discount
Traceback (most recent call last):
  File "<nb2_c.txt:4147>", line 6, in <module>
ZeroDivisionError: division by zero


### 7.10.3 Configuring logging in a real program

In a script, configure logging **once**, at program start, usually with `basicConfig`:

```python
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

logger = logging.getLogger(__name__)
logger.info("program started")
# 2026-10-24 14:30:01,123 INFO __main__: program started
```

Best practices:

* **Libraries** only create loggers (`logging.getLogger(__name__)`) and never add handlers or call `basicConfig`. The **application** decides where logs go.
* Use **lazy formatting**: `logger.info("user %s", name)`, not `logger.info(f"user {name}")`. The text is only built if the message is actually shown.
* Use `logger.exception(...)` inside `except` blocks to keep the traceback.
* In Jupyter, `basicConfig` does nothing if the root logger already has handlers; pass `force=True` to replace them.
* Never log passwords, tokens or full card numbers.
* For larger applications use `logging.config.dictConfig` with a settings dict.

In [402]:
# Writing logs to a file in a temporary folder (cleaned up afterwards)
import logging
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    log_path = Path(folder) / "app.log"

    file_logger = logging.getLogger("demo.file")
    file_logger.propagate = False
    file_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(levelname)s|%(message)s"))
    file_logger.addHandler(file_handler)

    file_logger.info("job started")
    file_logger.debug("not written: below INFO")
    file_logger.error("job failed")

    file_handler.close()                      # release the file (needed on Windows)
    file_logger.removeHandler(file_handler)

    print(log_path.read_text(encoding="utf-8"))
# INFO|job started
# ERROR|job failed

INFO|job started
ERROR|job failed



### 7.10.4 pprint: pretty-printing nested data

`print` puts a nested dict on one long line. **`pprint`** ("pretty print") breaks it over several lines and indents it, which is much easier to read when you debug API responses or configuration.

In [403]:
# pprint vs print on nested data
from pprint import pprint, pformat

data = {
    "users": [
        {"name": "Asha", "roles": ["admin", "dev"], "active": True},
        {"name": "Ravi", "roles": [], "active": False},
    ],
    "count": 2,
}

print(data)
pprint(data, width=50)            # wraps lines at about 50 characters

{'users': [{'name': 'Asha', 'roles': ['admin', 'dev'], 'active': True}, {'name': 'Ravi', 'roles': [], 'active': False}], 'count': 2}
{'count': 2,
 'users': [{'active': True,
            'name': 'Asha',
            'roles': ['admin', 'dev']},
           {'active': False,
            'name': 'Ravi',
            'roles': []}]}


In [404]:
# Useful pprint options: sort_dicts, depth, and pformat for a string
from pprint import pprint, pformat

config = {"zeta": 1, "alpha": {"beta": {"gamma": [1, 2, 3]}}}

pprint(config)                     # dict keys are SORTED by default
pprint(config, sort_dicts=False)   # keep insertion order
pprint(config, depth=2)            # deeper levels are shown as ...

text = pformat(config, width=30)   # a string, for example to put in a log message
print(text.count("\n") >= 1)       # True (it spans several lines)

{'alpha': {'beta': {'gamma': [1, 2, 3]}}, 'zeta': 1}
{'zeta': 1, 'alpha': {'beta': {'gamma': [1, 2, 3]}}}
{'alpha': {'beta': {...}}, 'zeta': 1}
True


### Common mistakes with logging

* **Nothing appears**: the default level is `WARNING`, so `info` and `debug` are dropped. Set the level on the logger (and handler).
* **Every message appears twice**: a handler was added twice (cell run twice), or both your logger and the root logger have handlers. Clear handlers or set `propagate = False`.
* **`basicConfig` has no effect**: logging was already configured. Use `force=True` (Python 3.8+).
* **Using f-strings in log calls**: the string is built even when the message is filtered out.
* **Libraries calling `basicConfig`**: this takes control away from the application.

### How to debug logging problems

1. Print `logger.level`, `logger.getEffectiveLevel()` and `logger.handlers` to see what is configured.
2. Check `logger.propagate` and the root logger's handlers (`logging.getLogger().handlers`).
3. Remember that both the logger **and** each handler have a level; the message must pass both.

In [405]:
# Inspecting a logger's configuration
import logging

shop = make_console_logger("shop", logging.WARNING)
print(logging.getLevelName(shop.level))       # WARNING
print(len(shop.handlers), shop.propagate)     # 1 False

quiet = logging.getLogger("not.configured")
print(logging.getLevelName(quiet.getEffectiveLevel()))   # WARNING (inherited from the root logger)

WARNING
1 False
WARNING


> **Key points**
> * Levels: DEBUG < INFO < WARNING (default) < ERROR < CRITICAL.
> * Logger (what you call) -> Handler (where) -> Formatter (how); child loggers propagate to parents.
> * `getLogger(__name__)` in modules; configure once in the application (`basicConfig`, `dictConfig`).
> * Lazy `%s` formatting; `logger.exception` in `except` blocks; never log secrets.
> * `pprint` / `pformat` make nested data readable.

### Practice 7.10: logging and pprint

**A. Predict the output**

1. A logger has level `WARNING`. Which of these calls produce output: `debug`, `info`, `warning`, `error`, `critical`?
2. What does this print?
   ```python
   import logging

   print(logging.getLevelName(logging.ERROR), logging.ERROR > logging.WARNING)
   ```
3. What does this print?
   ```python
   from pprint import pformat

   print(pformat({"b": 1, "a": 2}))
   ```

**B. Write code**

4. Write `make_logger(name, stream, level)` that returns a logger writing `"LEVEL:name:message"` lines to `stream`, with no duplicate handlers if called twice.
5. Using a logger writing to an `io.StringIO`, check that `DEBUG` messages are filtered at level `INFO` and that `logger.exception` adds the traceback.
6. Write `safe_divide(a, b, logger)` that returns `a / b`, or logs a warning and returns `None` when `b` is 0.
7. Add a second handler that only receives `ERROR` and above (set the handler's level), and check that an `INFO` message reaches only the first handler.
8. **[Hard]** Write a `logging.Filter` subclass that hides any message containing the word `"password"`, and test it.

**C. Find and fix the bug**

9. This prints every message twice. Why? Fix it.
   ```python
   import logging
   import sys

   def get_logger():
       log = logging.getLogger("app")
       log.addHandler(logging.StreamHandler(sys.stdout))
       return log

   get_logger().warning("hello")
   get_logger().warning("again")
   ```

**D. Interview questions**

10. Give three advantages of `logging` over `print`.
11. **[Hard]** Why should libraries not call `logging.basicConfig()`?

### Solutions 7.10

**A.**
1. `warning`, `error` and `critical`. Messages below the logger's level are dropped.
2. `ERROR True`. ERROR is 40, WARNING is 30.
3. `{'a': 2, 'b': 1}`. `pformat` sorts dict keys by default.

**C.**
9. Each call to `get_logger()` adds another handler to the same logger (`getLogger("app")` always returns the same object). The second message is printed twice (and the third would be printed three times). Add the handler only if `not log.handlers`, or configure logging once at startup.

**D.**
10. Levels let you filter messages without changing code; one format can add time, module and line number; messages can go to files or log services; `logger.exception` records tracebacks; logging is thread-safe; and you can turn modules up or down individually.
11. `basicConfig` configures the **root** logger for the whole process. A library that calls it forces its choices (level, format, destination) on every application that imports it, and the application's own `basicConfig` call is then ignored. Libraries should only create named loggers (optionally with a `NullHandler`).

In [406]:
# Solutions 7.10
import io
import logging
from pprint import pformat

# A: checks
assert logging.getLevelName(logging.ERROR) == "ERROR" and logging.ERROR > logging.WARNING
assert pformat({"b": 1, "a": 2}) == "{'a': 2, 'b': 1}"

# 4. Logger factory without duplicate handlers
def make_logger(name, stream, level=logging.INFO):
    log = logging.getLogger(name)
    log.setLevel(level)
    log.propagate = False
    if not log.handlers:
        handler = logging.StreamHandler(stream)
        handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
        log.addHandler(handler)
    return log

buf = io.StringIO()
log = make_logger("practice.p4", buf)
make_logger("practice.p4", buf)          # second call adds no handler
log.info("hello")
assert buf.getvalue() == "INFO:practice.p4:hello\n"

# 5. Filtering and exception tracebacks
buf = io.StringIO()
log = make_logger("practice.p5", buf, logging.INFO)
log.debug("debug hidden")
log.info("info shown")
try:
    1 / 0
except ZeroDivisionError:
    log.exception("failed")
out = buf.getvalue()
assert "debug hidden" not in out and "info shown" in out
assert "Traceback" in out and "ZeroDivisionError" in out

# 6. Log and return None instead of crashing
def safe_divide(a, b, logger):
    if b == 0:
        logger.warning("division by zero: %s / %s", a, b)
        return None
    return a / b

buf = io.StringIO()
log = make_logger("practice.p6", buf)
assert safe_divide(6, 3, log) == 2.0
assert safe_divide(1, 0, log) is None
assert "division by zero: 1 / 0" in buf.getvalue()

# 7. Handler-level filtering
all_buf, error_buf = io.StringIO(), io.StringIO()
log = make_logger("practice.p7", all_buf)
error_handler = logging.StreamHandler(error_buf)
error_handler.setLevel(logging.ERROR)
log.addHandler(error_handler)
log.info("routine")
log.error("broken")
assert "routine" in all_buf.getvalue() and "broken" in all_buf.getvalue()
assert "routine" not in error_buf.getvalue() and "broken" in error_buf.getvalue()

# 8. A filter that hides sensitive messages
class HidePasswords(logging.Filter):
    def filter(self, record):
        return "password" not in record.getMessage().lower()

buf = io.StringIO()
log = make_logger("practice.p8", buf)
log.addFilter(HidePasswords())
log.info("user asha logged in")
log.info("Password was %s", "secret")
assert "asha" in buf.getvalue() and "secret" not in buf.getvalue()

# 9. Fixed: add the handler only once
buf = io.StringIO()

def get_logger():
    log = logging.getLogger("practice.p9")
    log.propagate = False
    if not log.handlers:
        log.addHandler(logging.StreamHandler(buf))
    return log

get_logger().warning("hello")
get_logger().warning("again")
assert buf.getvalue() == "hello\nagain\n"
print("Practice 7.10: all checks passed")

Practice 7.10: all checks passed


---

## 7.11 Command-line programs: argparse and sys

A **command-line program** is started from a terminal with **arguments**, for example:

```text
python wordcount.py report.txt notes.txt --top 5 --ignore-case
```

* `sys.argv` is the raw list of words: `['wordcount.py', 'report.txt', 'notes.txt', '--top', '5', '--ignore-case']`. Item 0 is the script name.
* `argparse` turns those words into checked Python values, writes `--help` text for you, and prints clear error messages.
* `sys.exit(code)` ends the program. Code `0` means success; any other number means an error (`2` is the usual code for wrong arguments).

### 7.11.1 sys.argv and exit codes

A notebook has no real command line, so the next cell writes a tiny script to a temporary folder and runs it as a separate program. (`subprocess.run` is explained in section 7.12.)

In [407]:
# sys.argv and sys.exit seen from a real script
import subprocess
import sys
import tempfile
from pathlib import Path

script = """
import sys

print("argv:", sys.argv[1:])          # skip the script name
if len(sys.argv) < 2:
    print("usage: greet.py NAME", file=sys.stderr)
    sys.exit(2)                        # non-zero: an error
print(f"Hello, {sys.argv[1]}!")
sys.exit(0)                            # zero: success
"""

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "greet.py"
    path.write_text(script, encoding="utf-8")

    ok = subprocess.run([sys.executable, str(path), "Asha"], capture_output=True, text=True)
    print(ok.stdout.strip().splitlines())   # ['argv: [\'Asha\']', 'Hello, Asha!']
    print(ok.returncode)                     # 0

    bad = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(bad.stderr.strip())                # usage: greet.py NAME
    print(bad.returncode)                    # 2

["argv: ['Asha']", 'Hello, Asha!']
0
usage: greet.py NAME
2


> **Note:** `sys.exit(code)` works by raising the `SystemExit` exception. That is why `try/except SystemExit` can catch it, and why a bare `except:` (which you should not write) would accidentally stop a program from exiting.

### 7.11.2 argparse step by step

1. Create a parser: `argparse.ArgumentParser(prog=..., description=...)`.
2. Add arguments:
   * **positional** arguments (no dashes): required, identified by position;
   * **optional** arguments (`-n` / `--top`): may be left out, often with a default.
3. Call `parser.parse_args()`. In a real script it reads `sys.argv[1:]`. In tests and notebooks, pass a list of strings.

In [408]:
# Building a parser and parsing a simulated command line
import argparse

parser = argparse.ArgumentParser(prog="wordcount", description="Count words in text files")
parser.add_argument("paths", nargs="+", help="one or more files")
parser.add_argument("-n", "--top", type=int, default=3, help="how many words to show")
parser.add_argument("-i", "--ignore-case", action="store_true", help="treat 'The' and 'the' as equal")
parser.add_argument("--format", choices=["text", "json"], default="text", help="output format")
parser.add_argument("-v", "--verbose", action="count", default=0, help="-v, -vv for more detail")

args = parser.parse_args(["report.txt", "notes.txt", "-n", "5", "--ignore-case", "-vv"])
print(args)
# Namespace(paths=['report.txt', 'notes.txt'], top=5, ignore_case=True, format='text', verbose=2)

print(args.paths)          # ['report.txt', 'notes.txt']
print(args.top + 1)        # 6  (type=int already converted "5" to 5)
print(args.ignore_case)    # True  (dashes become underscores in the attribute name)

Namespace(paths=['report.txt', 'notes.txt'], top=5, ignore_case=True, format='text', verbose=2)
['report.txt', 'notes.txt']
6
True


Useful `add_argument` options:

| Option | Meaning |
|---|---|
| `type=int` / `float` / a function | convert and validate the text |
| `default=...` | value when the option is not given |
| `required=True` | an option that must be given |
| `choices=[...]` | only these values are allowed |
| `action="store_true"` | a flag: `True` if present, else `False` |
| `action="count"` | count repeats: `-vvv` gives 3 |
| `action="append"` | collect repeated options into a list |
| `nargs="+"` / `"*"` / `"?"` / `2` | one or more / zero or more / optional / exactly 2 values |
| `dest="name"` | the attribute name to use |
| `metavar="FILE"` | the name shown in the help text |

In [409]:
# The automatic --help text (format_help returns the text that -h prints)
import argparse

parser = argparse.ArgumentParser(prog="wordcount", description="Count words in text files")
parser.add_argument("paths", nargs="+", help="one or more files")
parser.add_argument("-n", "--top", type=int, default=3, help="how many words to show")
print(parser.format_help())

usage: wordcount [-h] [-n TOP] paths [paths ...]

Count words in text files

positional arguments:
  paths          one or more files

options:
  -h, --help     show this help message and exit
  -n, --top TOP  how many words to show



### 7.11.3 Errors from argparse

When the arguments are wrong, `argparse` prints a usage line and an error message to **standard error**, then calls `sys.exit(2)`. In a notebook or a test, catch `SystemExit` and capture the message.

In [410]:
# Bad arguments: argparse prints an error and raises SystemExit(2)
import argparse
import contextlib
import io

parser = argparse.ArgumentParser(prog="wordcount")
parser.add_argument("paths", nargs="+")
parser.add_argument("-n", "--top", type=int, default=3)

for bad_args in (["-n", "five", "a.txt"], []):
    captured = io.StringIO()
    with contextlib.redirect_stderr(captured):
        try:
            parser.parse_args(bad_args)
        except SystemExit as e:
            print("exit code:", e.code)               # exit code: 2
    print(captured.getvalue().strip().splitlines()[-1])
# exit code: 2
# wordcount: error: argument -n/--top: invalid int value: 'five'
# exit code: 2
# wordcount: error: the following arguments are required: paths

exit code: 2
wordcount: error: argument -n/--top: invalid int value: 'five'
exit code: 2
wordcount: error: the following arguments are required: paths


In [411]:
# Python 3.14+: suggest_on_error=True suggests the closest valid choice
import argparse
import contextlib
import io
import sys

if sys.version_info >= (3, 14):
    parser = argparse.ArgumentParser(prog="paint", suggest_on_error=True)
    parser.add_argument("--colour", choices=["red", "green"])
    captured = io.StringIO()
    with contextlib.redirect_stderr(captured):
        try:
            parser.parse_args(["--colour", "gren"])
        except SystemExit:
            pass
    print(captured.getvalue().strip().splitlines()[-1])
    # paint: error: argument --colour: invalid choice: 'gren', maybe you meant 'green'? (choose from 'red', 'green')
else:
    print("suggest_on_error needs Python 3.14+")

paint: error: argument --colour: invalid choice: 'gren', maybe you meant 'green'? (choose from 'red', 'green')


### 7.11.4 Sub-commands, exclusive options and the main() pattern

Tools like `git` have **sub-commands** (`git add`, `git commit`), each with its own arguments. `add_subparsers` builds that. `add_mutually_exclusive_group` allows at most one option of a group.

In [412]:
# Sub-commands with add_subparsers, and a mutually exclusive group
import argparse

parser = argparse.ArgumentParser(prog="todo")
sub = parser.add_subparsers(dest="command", required=True)

add_cmd = sub.add_parser("add", help="add a task")
add_cmd.add_argument("title")
add_cmd.add_argument("--priority", type=int, default=2)

list_cmd = sub.add_parser("list", help="list tasks")
group = list_cmd.add_mutually_exclusive_group()
group.add_argument("--done", action="store_true")
group.add_argument("--pending", action="store_true")

print(parser.parse_args(["add", "Pay rent", "--priority", "1"]))
# Namespace(command='add', title='Pay rent', priority=1)
print(parser.parse_args(["list", "--pending"]))
# Namespace(command='list', done=False, pending=True)

Namespace(command='add', title='Pay rent', priority=1)
Namespace(command='list', done=False, pending=True)


The standard structure of a command-line script. `main()` takes an optional list of arguments, so tests can call it directly, and returns an exit code:

```python
import argparse
import sys


def build_parser():
    parser = argparse.ArgumentParser(prog="wordcount")
    parser.add_argument("paths", nargs="+")
    parser.add_argument("-n", "--top", type=int, default=3)
    return parser


def main(argv=None):
    args = build_parser().parse_args(argv)   # argv=None means "use sys.argv[1:]"
    ...                                      # the real work
    return 0                                 # exit code


if __name__ == "__main__":
    sys.exit(main())
```

`if __name__ == "__main__":` is explained in Notebook 1, Part 14. Part 10 of this notebook builds a complete log-analysis command-line tool with this structure.

In [413]:
# The main(argv) pattern is easy to test without a terminal
import argparse
from collections import Counter

def build_parser():
    parser = argparse.ArgumentParser(prog="topwords")
    parser.add_argument("words", nargs="+")
    parser.add_argument("-n", "--top", type=int, default=2)
    return parser

def main(argv=None):
    args = build_parser().parse_args(argv)
    for word, count in Counter(args.words).most_common(args.top):
        print(word, count)
    return 0

exit_code = main(["red", "blue", "red", "green", "red", "blue"])
print("exit code:", exit_code)
# red 3
# blue 2
# exit code: 0

red 3
blue 2
exit code: 0


### 7.11.5 Other useful parts of sys

In [414]:
# A few sys attributes you will use often
import sys

print(sys.version_info >= (3, 12))        # True  (use version_info for version checks)
print(sys.platform in ("win32", "linux", "darwin"))   # True
print(sys.maxsize == 2**63 - 1)           # True on 64-bit builds (Python ints themselves have no limit)
print(sys.getrecursionlimit())            # 1000  (the default; deep recursion raises RecursionError)
print(sys.getsizeof([]) < sys.getsizeof([1, 2, 3]))   # True  (size of the list object in bytes)

# Write error messages to standard error, normal output to standard output
print("normal output", file=sys.stdout)   # normal output

True
True
True
1000
True
normal output


> **Interview tip:** Deep recursive DFS on a 10^5-node path can hit the default recursion limit of 1000. Mention `sys.setrecursionlimit(...)` as a quick fix and an iterative version with an explicit stack as the robust fix (Notebook 3, Part 6).

### Common mistakes with command-line programs

* **Reading `sys.argv[1]` without checking the length**: `IndexError` when no argument is given. Use `argparse`.
* **Forgetting `type=int`**: the value stays a string, so `args.top + 1` raises `TypeError`.
* **Calling `parse_args()` with no list in a notebook**: it reads the notebook kernel's own arguments and fails. Pass a list.
* **Returning error codes as `print`**: scripts called by other programs must use exit codes (`sys.exit(1)`).
* **Option names with dashes**: `--ignore-case` becomes `args.ignore_case`.

### How to debug argument parsing

1. Print the whole `Namespace`: `print(args)` shows every parsed value and its type.
2. Run the script with `-h` to see what the parser really expects.
3. Test `main([...])` or `parser.parse_args([...])` with a list of strings instead of typing in a terminal.
4. When a script "exits silently", catch `SystemExit` and print `e.code`.

In [415]:
# Common mistake: reading sys.argv positions directly
argv = ["greet.py"]                      # simulated: the user gave no name
try:
    name = argv[1]
except IndexError as e:
    print("IndexError:", e)              # IndexError: list index out of range

IndexError: list index out of range


> **Key points**
> * `sys.argv` is the raw list of arguments (`argv[0]` is the script); `sys.exit(code)` ends the program (0 = success).
> * `argparse`: positional and optional arguments, `type`, `default`, `choices`, `store_true`, `count`, `nargs`.
> * Wrong arguments print a message to stderr and raise `SystemExit(2)`.
> * Structure scripts as `main(argv=None) -> exit code` with `if __name__ == "__main__": sys.exit(main())`.
> * Test parsers with `parse_args([...])`.

### Practice 7.11: argparse and sys

**A. Predict the output**

1. What does this print?
   ```python
   import argparse

   p = argparse.ArgumentParser()
   p.add_argument("--dry-run", action="store_true")
   p.add_argument("--level", type=int, default=1)
   args = p.parse_args(["--level", "3"])
   print(args.dry_run, args.level * 2)
   ```
2. What does this print?
   ```python
   import argparse

   p = argparse.ArgumentParser()
   p.add_argument("-v", action="count", default=0)
   print(p.parse_args(["-vvv"]).v)
   ```
3. What exception does `sys.exit(3)` raise, and what is its `code`?

**B. Write code**

4. Write `build_parser()` for a tool with a positional `files` (one or more), `-o/--output` (default `"out.txt"`) and a counting `-v/--verbose` flag. Test it with `parse_args`.
5. Write a parser with a `--size` option that only accepts positive integers. Hint: `type=` can be your own function that raises `argparse.ArgumentTypeError`.
6. Write a parser with a repeatable `--tag` option (`action="append"`) so `--tag a --tag b` gives `["a", "b"]`.
7. Write `main(argv=None)` for a `sum` tool that adds the numbers given (floats) and prints the total; it returns `0`, or `1` if no numbers were given (use `nargs="*"`).
8. **[Hard]** Write a parser with sub-commands `deposit AMOUNT` and `withdraw AMOUNT [--force]`, and a function `run(argv, balance)` that returns the new balance (withdraw without `--force` must not go below zero: return the old balance).

**C. Find and fix the bug**

9. This crashes with `TypeError`. Fix it.
   ```python
   import argparse

   p = argparse.ArgumentParser()
   p.add_argument("--retries", default=3)
   args = p.parse_args(["--retries", "5"])
   print(args.retries + 1)
   ```

**D. Interview questions**

10. What do exit codes mean, and why do they matter for scripts?

### Solutions 7.11

**A.**
1. `False 6`. `--dry-run` was not given; `--level` was converted to the int 3.
2. `3`.
3. `SystemExit`, with `code == 3`.

**C.**
9. Without `type=int` the value is the string `"5"`, so `"5" + 1` raises `TypeError`. Add `type=int` (the default `3` is then also an int).

**D.**
10. The exit code tells the calling program (a shell, a CI pipeline, a scheduler, `subprocess`) whether the program succeeded: `0` is success, non-zero is failure (often 1 for general errors, 2 for usage errors). Scripts chained with `&&`, or CI jobs, rely on it to stop at the first failure.

In [416]:
# Solutions 7.11
import argparse
import contextlib
import io

# A: checks
p = argparse.ArgumentParser()
p.add_argument("--dry-run", action="store_true")
p.add_argument("--level", type=int, default=1)
a = p.parse_args(["--level", "3"])
assert (a.dry_run, a.level * 2) == (False, 6)
try:
    raise SystemExit(3)                  # what sys.exit(3) does
except SystemExit as e:
    assert e.code == 3

# 4. A parser with files, output and verbosity
def build_parser():
    parser = argparse.ArgumentParser(prog="tool")
    parser.add_argument("files", nargs="+")
    parser.add_argument("-o", "--output", default="out.txt")
    parser.add_argument("-v", "--verbose", action="count", default=0)
    return parser

ns = build_parser().parse_args(["a.log", "b.log", "-vv", "-o", "report.txt"])
assert (ns.files, ns.output, ns.verbose) == (["a.log", "b.log"], "report.txt", 2)
assert build_parser().parse_args(["x"]).output == "out.txt"

# 5. A custom type function for validation
def positive_int(text):
    value = int(text)
    if value <= 0:
        raise argparse.ArgumentTypeError(f"{text} is not a positive integer")
    return value

sized = argparse.ArgumentParser(prog="sized")
sized.add_argument("--size", type=positive_int, default=10)
assert sized.parse_args(["--size", "4"]).size == 4
captured = io.StringIO()
with contextlib.redirect_stderr(captured):
    try:
        sized.parse_args(["--size", "-2"])
    except SystemExit as e:
        assert e.code == 2
assert "not a positive integer" in captured.getvalue()

# 6. Repeatable option
tagger = argparse.ArgumentParser()
tagger.add_argument("--tag", action="append", default=[])
assert tagger.parse_args(["--tag", "a", "--tag", "b"]).tag == ["a", "b"]
assert tagger.parse_args([]).tag == []

# 7. main() returning an exit code
def main(argv=None):
    parser = argparse.ArgumentParser(prog="sum")
    parser.add_argument("numbers", nargs="*", type=float)
    args = parser.parse_args(argv)
    if not args.numbers:
        print("no numbers given")
        return 1
    print(sum(args.numbers))
    return 0

with contextlib.redirect_stdout(io.StringIO()) as out:
    assert main(["1.5", "2", "-0.5"]) == 0
assert out.getvalue().strip() == "3.0"
with contextlib.redirect_stdout(io.StringIO()):
    assert main([]) == 1

# 8. Sub-commands
def build_bank_parser():
    parser = argparse.ArgumentParser(prog="bank")
    sub = parser.add_subparsers(dest="command", required=True)
    dep = sub.add_parser("deposit")
    dep.add_argument("amount", type=float)
    wd = sub.add_parser("withdraw")
    wd.add_argument("amount", type=float)
    wd.add_argument("--force", action="store_true")
    return parser

def run(argv, balance):
    args = build_bank_parser().parse_args(argv)
    if args.command == "deposit":
        return balance + args.amount
    if args.amount > balance and not args.force:
        return balance
    return balance - args.amount

assert run(["deposit", "50"], 100) == 150
assert run(["withdraw", "30"], 100) == 70
assert run(["withdraw", "300"], 100) == 100
assert run(["withdraw", "300", "--force"], 100) == -200

# 9. Fixed: type=int
p = argparse.ArgumentParser()
p.add_argument("--retries", type=int, default=3)
assert p.parse_args(["--retries", "5"]).retries + 1 == 6
print("Practice 7.11: all checks passed")

Practice 7.11: all checks passed


---

## 7.12 Running other programs: subprocess

The `subprocess` module starts other programs (a **child process**), waits for them, and collects their output and exit code. Use it to call tools like `git`, `ffmpeg`, compilers, or another Python script.

`subprocess.run(...)` is the function you need almost always:

| Argument | Meaning |
|---|---|
| `[program, arg1, arg2]` | the command as a **list** (recommended) |
| `capture_output=True` | collect stdout and stderr instead of printing them |
| `text=True` | give `str` instead of `bytes` |
| `check=True` | raise `CalledProcessError` if the exit code is not 0 |
| `timeout=seconds` | kill the child and raise `TimeoutExpired` if it takes too long |
| `input="..."` | send text to the child's standard input |
| `cwd=folder`, `env={...}` | working folder and environment variables for the child |

It returns a `CompletedProcess` object with `.returncode`, `.stdout` and `.stderr`.

```text
  subprocess.run([...], capture_output=True, text=True, check=True, timeout=5)
        |
        v
  +---------------------+  not found  +---------------------------+
  | start the program   |------------>| FileNotFoundError         |
  +----------+----------+             +---------------------------+
             v
  +---------------------+  too slow   +---------------------------+
  | wait for it         |------------>| kill it, TimeoutExpired   |
  +----------+----------+             +---------------------------+
             v
  +---------------------+  not 0      +---------------------------+
  | exit code == 0 ?    |------------>| CalledProcessError        |
  +----------+----------+  (check)    +---------------------------+
             | yes
             v
     CompletedProcess(returncode, stdout, stderr)
```

> **Tip:** To run Python code in a child process, use `sys.executable` (the path of the Python that is running now) instead of `"python"`. It works on every operating system and inside virtual environments.

In [417]:
# Run a program and capture its output
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-c", "print('hello from the child'); print(6 * 7)"],
    capture_output=True,
    text=True,
)
print(result.returncode)               # 0
print(result.stdout.splitlines())      # ['hello from the child', '42']
print(repr(result.stderr))             # ''

0
['hello from the child', '42']
''


In [418]:
# Exit codes, check=True and CalledProcessError
import subprocess
import sys

code = "import sys; print('working'); sys.exit(3)"
result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(result.returncode)               # 3  (no exception without check=True)

try:
    subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as e:
    print("CalledProcessError: exit status", e.returncode)   # CalledProcessError: exit status 3
    print("child printed:", e.stdout.strip())                # child printed: working

3
CalledProcessError: exit status 3
child printed: working


In [419]:
# Timeouts and missing programs
import subprocess
import sys

try:
    subprocess.run([sys.executable, "-c", "import time; time.sleep(5)"], timeout=0.5)
except subprocess.TimeoutExpired as e:
    print("TimeoutExpired after", e.timeout, "seconds")   # TimeoutExpired after 0.5 seconds

try:
    subprocess.run(["no-such-program-xyz"])
except FileNotFoundError as e:
    print("FileNotFoundError: the program is not installed or not on the PATH")

TimeoutExpired after 0.5 seconds
FileNotFoundError: the program is not installed or not on the PATH


In [420]:
# Sending input to the child's standard input
import subprocess
import sys

upper = "import sys; print(sys.stdin.read().upper(), end='')"
result = subprocess.run([sys.executable, "-c", upper], input="hello asha\n",
                        capture_output=True, text=True, check=True)
print(result.stdout)                   # HELLO ASHA

HELLO ASHA



### 7.12.1 Security: never use shell=True with user input

With `shell=True`, the command is a single string that a **shell** (cmd.exe, bash) reads. The shell treats characters like `;`, `&`, `|` and `$( )` as instructions. If any part of the string comes from a user, they can run any command they like. This is called **command injection**.

```python
# DANGEROUS: do not do this
filename = request_value            # imagine the user sends: "report.txt; rm -rf ~"
subprocess.run(f"cat {filename}", shell=True)
# The shell runs "cat report.txt" AND THEN "rm -rf ~"
```

The safe form passes a **list**. Each item is one argument, given directly to the program, and no shell reads it:

```python
subprocess.run(["cat", filename], check=True)     # the whole text is just one file name
```

In [421]:
# With a list, "dangerous" text is just an ordinary argument
import subprocess
import sys

user_value = "report.txt; echo HACKED"
show_args = "import sys; print(sys.argv[1:])"
result = subprocess.run([sys.executable, "-c", show_args, user_value],
                        capture_output=True, text=True, check=True)
print(result.stdout.strip())   # ['report.txt; echo HACKED']  (one argument, nothing extra ran)

['report.txt; echo HACKED']


> **Tip:** If you have a command as one string from a trusted config file, `shlex.split("git log -n 5")` turns it into a list safely (POSIX shell rules). If you truly need shell features, quote each user value with `shlex.quote` (POSIX shells only), but the list form is almost always better. More secure-coding rules are in Part 9.

> **Note:** For long-running children whose output you want to read while they run, use `subprocess.Popen`. For running many tasks in parallel inside Python, see Part 8 (concurrency).

### Common mistakes with subprocess

* **Passing a string without `shell=True`**: `subprocess.run("git status")` fails on Linux/macOS (it looks for a program literally named `git status`). Use a list.
* **Using `shell=True` with user input**: command injection.
* **Forgetting `text=True`**: you get `bytes` (`b'...'`), and comparing them with `str` is always `False`.
* **Forgetting `check=True`**: a failing command goes unnoticed.
* **No `timeout`**: a stuck child blocks your program forever.
* **Hard-coding `"python"`**: may be the wrong Python, or missing on Windows. Use `sys.executable`.

### How to debug subprocess calls

1. Print the exact list you pass: `print(cmd)`. Every argument must be its own item.
2. Capture and print **stderr**: the child's error message is usually there.
3. Check `returncode` (or use `check=True`) and read `e.stderr` in the `CalledProcessError`.
4. Run the same command by hand in a terminal to compare.
5. On `FileNotFoundError`, check that the program is installed and on the `PATH` (`shutil.which("git")`).

In [422]:
# Reading the child's error message from stderr
import shutil
import subprocess
import sys

try:
    subprocess.run([sys.executable, "-c", "1 / 0"], capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as e:
    print("exit status:", e.returncode)                  # exit status: 1
    print("last stderr line:", e.stderr.strip().splitlines()[-1])
    # last stderr line: ZeroDivisionError: division by zero

print(shutil.which("no-such-program-xyz"))               # None  (not on the PATH)

exit status: 1
last stderr line: ZeroDivisionError: division by zero
None


> **Key points**
> * `subprocess.run([...], capture_output=True, text=True, check=True, timeout=...)` covers most needs.
> * Pass the command as a list; never `shell=True` with user input (command injection).
> * `returncode` 0 means success; `check=True` raises `CalledProcessError`; `timeout` raises `TimeoutExpired`.
> * Use `sys.executable` to run Python; read `stderr` when something fails.

### Practice 7.12: subprocess

**A. Predict the output**

1. What does this print?
   ```python
   import subprocess
   import sys

   r = subprocess.run([sys.executable, "-c", "print('hi')"], capture_output=True)
   print(type(r.stdout).__name__, r.stdout.strip() == "hi", r.stdout.strip() == b"hi")
   ```
2. What happens here?
   ```python
   import subprocess
   import sys

   subprocess.run([sys.executable, "-c", "import sys; sys.exit(1)"])
   print("still running")
   ```

**B. Write code**

3. Write `run_python(code, timeout=5)` that runs a Python snippet in a child process and returns its stdout as a string; it must raise on a non-zero exit code or a timeout.
4. Write `python_version_of_child()` that returns the child's `sys.version_info[:2]` as a tuple of ints, using `subprocess` and `sys.executable`.
5. Write `count_lines_via_child(text)` that sends `text` to a child Python process on stdin, and the child prints the number of lines.
6. **[Hard]** Write `run_all(snippets)` that runs several snippets and returns a list of `(returncode, first_line_of_stdout_or_stderr)` without ever raising.

**C. Find and fix the bug**

7. This is unsafe. Explain why and rewrite it.
   ```python
   import subprocess

   def show_file(name):
       subprocess.run("type " + name, shell=True)
   ```

**D. Interview questions**

8. What is command injection, and how do you prevent it in Python?
9. What is the difference between `check=True` and checking `returncode` yourself?

### Solutions 7.12

**A.**
1. `bytes False True`. Without `text=True`, `stdout` is `bytes`, and bytes never equal a `str`. Add `text=True` to get a string (it also converts Windows `\r\n` line endings to `\n`).
2. It prints `still running`. Without `check=True`, a non-zero exit code does not raise; it is only stored in `returncode`.

**C.**
7. With `shell=True`, a name like `report.txt & del /q *` makes the shell run a second command. Pass a list and avoid the shell. `type` is a cmd.exe built-in (not a program), so the portable fix is to read the file in Python: `Path(name).read_text(encoding="utf-8")`. For a real program, use `subprocess.run(["program", name], check=True)`.

**D.**
8. Command injection happens when untrusted text becomes part of a shell command, so the attacker can add their own commands with characters like `;`, `&`, `|`. Prevent it by passing arguments as a list with no shell, validating inputs (for example with an allow-list), and avoiding `os.system` and `shell=True`.
9. `check=True` raises `CalledProcessError` automatically, which you cannot forget to handle and which carries `returncode`, `stdout` and `stderr`. Checking `returncode` yourself is useful when some non-zero codes are expected (for example `grep` returns 1 for "no match").

In [423]:
# Solutions 7.12
import subprocess
import sys

# A: checks
r = subprocess.run([sys.executable, "-c", "print('hi')"], capture_output=True)
assert type(r.stdout).__name__ == "bytes"
assert r.stdout.strip() != "hi" and r.stdout.strip() == b"hi"
r = subprocess.run([sys.executable, "-c", "import sys; sys.exit(1)"])
assert r.returncode == 1

# 3. Run a snippet, raising on failure or timeout
def run_python(code, timeout=5):
    result = subprocess.run([sys.executable, "-c", code], capture_output=True,
                            text=True, timeout=timeout, check=True)
    return result.stdout

assert run_python("print(6 * 7)").strip() == "42"
try:
    run_python("raise SystemExit(1)")
    raise AssertionError("expected CalledProcessError")
except subprocess.CalledProcessError as e:
    assert e.returncode == 1

# 4. Ask the child for its version
def python_version_of_child():
    out = run_python("import sys; print(sys.version_info[0], sys.version_info[1])")
    major, minor = map(int, out.split())
    return (major, minor)

assert python_version_of_child() == tuple(sys.version_info[:2])

# 5. Send data on stdin
def count_lines_via_child(text):
    code = "import sys; print(len(sys.stdin.read().splitlines()))"
    result = subprocess.run([sys.executable, "-c", code], input=text,
                            capture_output=True, text=True, check=True, timeout=5)
    return int(result.stdout)

assert count_lines_via_child("a\nb\nc\n") == 3
assert count_lines_via_child("") == 0

# 6. Run several snippets and never raise
def run_all(snippets, timeout=5):
    results = []
    for code in snippets:
        try:
            r = subprocess.run([sys.executable, "-c", code], capture_output=True,
                               text=True, timeout=timeout)
            text = (r.stdout or r.stderr).strip()
            first = text.splitlines()[0] if text else ""
            results.append((r.returncode, first))
        except subprocess.TimeoutExpired:
            results.append((None, "timeout"))
    return results

out = run_all(["print('ok')", "import sys; sys.exit(2)", "raise ValueError('bad')"])
assert out[0] == (0, "ok")
assert out[1] == (2, "")
assert out[2][0] == 1 and out[2][1].startswith("Traceback")

# 7. Safe version: no shell; read the file in Python
import tempfile
from pathlib import Path

def show_file(name):
    return Path(name).read_text(encoding="utf-8")

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "notes.txt"
    path.write_text("safe\n", encoding="utf-8")
    assert show_file(path) == "safe\n"
print("Practice 7.12: all checks passed")

Practice 7.12: all checks passed


---

## 7.13 Choosing the right tool and debugging library code

### 7.13.1 Problem signal to tool

| When the problem says... | Reach for |
|---|---|
| "count", "frequency", "anagram", "most common" | `Counter` |
| "group by", "adjacency list", "map key to many values" | `defaultdict(list)` / `defaultdict(set)` |
| "queue", "BFS", "level by level", "last N items" | `deque` (`maxlen` for last N) |
| "top k", "k-th largest", "merge k sorted", "always the smallest next" | `heapq` |
| "sorted list", "insert position", "first index >= x", "count in a range" | `bisect` |
| "all combinations / permutations / subsets", "prefix sums", "consecutive pairs" | `itertools` |
| "overlapping subproblems", "recursive with repeated inputs" | `functools.cache` |
| "sort by field", "sort by several keys" | `operator.itemgetter` / `attrgetter` |
| "money", "exact decimal" | `decimal.Decimal` |
| "secure token", "password" | `secrets` |
| "time zones", "dates", "durations" | `datetime`, `zoneinfo` |
| "extract / validate text patterns" | `re` |
| "diagnostics in a real program" | `logging` |
| "command-line options" | `argparse` |
| "run another program" | `subprocess` |
| "read/write JSON or CSV" | `json`, `csv` (Notebook 1, Part 13) |

### 7.13.2 A debugging routine for standard library code

```text
  +-------------------------------+
  | Read the error: type + message|
  +---------------+---------------+
                  v
  +-------------------------------+
  | Find the line (traceback,     |
  | read it bottom-up)            |
  +---------------+---------------+
                  v
  +-------------------------------+   unsure what a function  +-------------------+
  | Do I know what the library    |-------------------------->| help(func), docs, |
  | function expects and returns? |   expects or returns      | try it on 1 value |
  +---------------+---------------+                           +---------+---------+
                  | yes                                                 |
                  v  <--------------------------------------------------+
  +-------------------------------+
  | Print inputs: repr() and type |
  | (None? str vs int? naive dt?) |
  +---------------+---------------+
                  v
  +-------------------------------+
  | Fix, re-run, add an assert    |
  | test for this case            |
  +-------------------------------+
```

Most bugs with library code come from four places:

1. **Wrong input type**: a list where a tuple is needed (unhashable), a string where an int is needed (argparse without `type=`), a float in `Decimal` code, a naive datetime.
2. **Wrong assumption about the return value**: `re.search` can return `None`; `random.shuffle` and `list.sort` return `None`; `heapq` lists are not sorted.
3. **One-shot iterators**: `itertools` results, `groupby` groups and `map`/`filter` objects are used up after one loop.
4. **Hidden state**: a `defaultdict` that grows on reads, a cache holding old results, a logger with duplicate handlers.

In [424]:
# help() and the docstring tell you what a function expects and returns
import bisect

print(bisect.bisect_left.__doc__.splitlines()[0])
# Return the index where to insert item x in list a, assuming a is sorted.

Return the index where to insert item x in list a, assuming a is sorted.


In [425]:
# Check types and values quickly with repr() and type()
from datetime import datetime, timezone

values = ["42", 42, None, datetime(2026, 10, 24), datetime(2026, 10, 24, tzinfo=timezone.utc)]
for v in values:
    print(f"{v!r:60} {type(v).__name__}")

'42'                                                         str
42                                                           int
None                                                         NoneType
datetime.datetime(2026, 10, 24, 0, 0)                        datetime
datetime.datetime(2026, 10, 24, 0, 0, tzinfo=datetime.timezone.utc) datetime


### Summary of Part 7

* `collections`: `Counter` for counting (`most_common`, arithmetic), `defaultdict` for grouping (reading creates keys), `deque` for O(1) queues and BFS, `namedtuple` for small records, `OrderedDict` for an O(1) LRU cache, `ChainMap` for layered settings.
* `heapq` is a min-heap on a list: O(log n) push/pop, O(n) heapify, `(priority, counter, item)` tuples; max-heap with the 3.14 `_max` functions or by negating values.
* `bisect` does O(log n) binary search on sorted lists: `bisect_left` for the first position, `bisect_right` for after the last; `insort` is O(n).
* `itertools` gives lazy combinatorics, `accumulate`, `chain`, `islice`, `pairwise`, `batched`, `groupby` (sort first); iterators are one-shot.
* `functools.cache` / `lru_cache` memoise pure functions with hashable arguments; `partial`, `reduce`, `cached_property`, `singledispatch`, `cmp_to_key`, `total_ordering` cover the rest; `operator` getters make clean sort keys.
* Numbers: `math.isclose`/`fsum` for floats, `statistics` for summaries, `random` with a seed for simulations, `secrets` for security, `Decimal` from strings for money, `Fraction` for exact ratios.
* Dates: never mix naive and aware datetimes; store aware UTC; `datetime.now(timezone.utc)`; `ZoneInfo` for named zones (catch `ZoneInfoNotFoundError`, install `tzdata` on Windows).
* `re`: raw strings, `search` vs `match` vs `fullmatch`, named groups, greedy vs lazy, `re.escape`, check for `None`.
* `logging` replaces `print` for diagnostics: levels, logger -> handler -> formatter, lazy `%s` formatting, `logger.exception`.
* `argparse` builds tested command-line interfaces (`main(argv=None)`); `subprocess.run` with a list, `check=True`, `timeout`, and never `shell=True` with user input.

Next: Part 8 (Concurrency and Async Programming).

# Part 8: Concurrency and Async Programming

**What these notes cover**

* The basic words: concurrency, parallelism, process, thread, coroutine, I/O-bound and CPU-bound work.
* What the GIL (Global Interpreter Lock) really does, and what changed in Python 3.13 and 3.14.
* How to run work in threads with `threading` and `ThreadPoolExecutor`, and how to get results and errors back.
* Race conditions, locks, queues (producer/consumer) and deadlocks.
* How to use several CPU cores with `ProcessPoolExecutor`.
* How `asyncio` works: the event loop, `async`/`await`, `gather`, `create_task`, `TaskGroup`, timeouts, cancellation, semaphores and `to_thread`.
* How to choose between threads, processes and asyncio, and how to debug concurrent code.

**Before you start:** Notebook 1 (especially Part 11 Functions and Part 12 Errors), and in this notebook Part 3 (Iterators and Generators), Part 4 (Decorators) and Part 5 (Context Managers).

## 8.1 The vocabulary

Most programs do one thing at a time. Sometimes this is too slow:

* A program downloads 100 web pages. Each download spends most of its time **waiting** for the network.
* A program resizes 5,000 images. Each resize spends its time **computing** on the CPU.

Concurrency tools let a program make progress on many tasks at once. First we need clear words.

| Term | Meaning in simple words |
|---|---|
| **Concurrency** | Dealing with many tasks at once. Their lifetimes overlap, but they may take turns. |
| **Parallelism** | Running many tasks at the same instant, on different CPU cores. |
| **Process** | A running program with its own memory. Heavy to start. Truly parallel. |
| **Thread** | A flow of work inside a process. Threads share the process memory. Cheap to start. |
| **Coroutine** | A function that can pause at `await` and let other code run, all on one thread. |

**In simple words:** concurrency is one cook switching between several pots. Parallelism is several cooks, each stirring a pot at the same moment.

```text
Concurrency (one core, tasks take turns):

  core 1:  [A][B][A][C][B][A][C]      time ------>

Parallelism (three cores, tasks run at the same time):

  core 1:  [A A A A A A A]
  core 2:  [B B B B B B B]
  core 3:  [C C C C C C C]
```

Two more words decide which tool you should use:

* **I/O-bound** work spends most of its time *waiting*: network calls, disk reads, database queries. "I/O" means input/output.
* **CPU-bound** work spends most of its time *computing*: maths, parsing, compression, image processing.

> **Tip:** Before you add concurrency, ask: "Is my program slow because it waits, or because it computes?" The answer picks the tool.

In [426]:
# Every program starts with one process and one thread: the main thread
import os
import threading

print(threading.current_thread().name)         # MainThread
print(threading.active_count() >= 1)           # True
print(isinstance(os.getpid(), int))            # True  (the process id is a number)
print((os.cpu_count() or 1) >= 1)              # True  (how many CPU cores the OS reports)

MainThread
True
True
True


The three tools Python gives you, and where each one fits:

| Tool | Module | Good for | Runs in parallel? |
|---|---|---|---|
| Threads | `threading`, `concurrent.futures` | I/O-bound work, a moderate number of tasks | Not for Python code in the default build (GIL) |
| Processes | `multiprocessing`, `concurrent.futures` | CPU-bound pure Python work | Yes, one process per core |
| Async | `asyncio` | Many (thousands of) I/O waits, e.g. network servers | No, one thread takes turns |

> **Key points**
> * Concurrency means overlapping tasks; parallelism means tasks running at the same instant.
> * A process has its own memory; threads inside a process share memory.
> * I/O-bound work waits; CPU-bound work computes. The kind of work decides the tool.

---

## 8.2 The GIL (Global Interpreter Lock)

The standard Python you download from python.org is called **CPython** (it is written in C). CPython has a **GIL**: a lock that allows only **one thread at a time to run Python bytecode** (the small instructions your code is compiled into).

What this means in practice:

* **CPU-bound pure Python code does not get faster with threads.** Only one thread computes at a time. Threads even add a little overhead.
* **I/O-bound code does get faster with threads.** A thread that waits (for `time.sleep`, a socket, a file) releases the GIL, so another thread can run.
* **Native libraries** such as NumPy can release the GIL inside their C code, so threads can help them too.

**What the GIL does not do:** it does not make your code thread-safe. It protects the interpreter's own data. A line such as `counter += 1` is several bytecode steps (read, add, write). A thread switch can happen between those steps, so updates can be lost. You still need locks (see 8.5).

**Version facts (important for interviews):**

* The GIL is a detail of CPython, not of the Python language.
* **Python 3.13** added an *experimental* **free-threaded build** with the GIL disabled (PEP 703).
* **Python 3.14** made the free-threaded build *officially supported but optional* (PEP 779). The normal build still has the GIL. The free-threaded build is installed separately (often named `python3.14t`).
* `sys._is_gil_enabled()` (Python 3.13+) tells you if the GIL is on in the running interpreter.

> **Interview tip:** When you talk about the GIL, say which build you mean: "In standard CPython, threads do not speed up CPU-bound Python code."

In [427]:
# Check the GIL and the thread switch interval of this interpreter
import sys

if sys.version_info >= (3, 13):
    print("GIL enabled:", sys._is_gil_enabled())       # GIL enabled: True  (on the standard build)
else:
    print("GIL enabled: True (always before 3.13)")

# The GIL is handed to another thread about every 5 ms
print("switch interval:", sys.getswitchinterval())    # switch interval: 0.005

GIL enabled: True
switch interval: 0.005


**Common mistakes**

* Thinking the GIL makes `counter += 1` safe. It does not; use a lock.
* Using threads to speed up a CPU-heavy pure Python loop. Use processes instead (see 8.7).
* Saying "Python has no parallelism". Processes, native libraries and free-threaded builds all run in parallel.

> **Key points**
> * The GIL lets one thread run Python bytecode at a time (standard CPython).
> * Waiting (I/O) releases the GIL, so threads help I/O-bound work.
> * The GIL does not make your code thread-safe.
> * 3.13: experimental free-threaded build; 3.14: supported but optional; the default build keeps the GIL.

---

### Practice 8.1: Vocabulary and the GIL

**A. Predict the output**

1. What does this print in a normal script?
   ```python
   import threading
   print(threading.current_thread().name)
   ```
2. On the standard CPython 3.14 build, what does this print?
   ```python
   import sys
   print(sys._is_gil_enabled())
   ```

**B. Write code**

3. Write `classify(task)` that returns `"io"` or `"cpu"` for these task names: `"download files"`, `"query database"`, `"resize images"`, `"compute primes"`, `"read disk"`, `"compress video"`. Use a dict.
4. Write `gil_status()` that returns the string `"on"`, `"off"` or `"unknown"`. Use `sys._is_gil_enabled()` only if it exists (check with `hasattr`).

**C. Find and fix the bug**

5. A teammate says: "This code is safe because Python has a GIL." Is it? Explain.
   ```python
   counter = 0

   def add_many():
       global counter
       for _ in range(100_000):
           counter += 1
   ```

**D. Interview questions**

6. Define concurrency and parallelism with an everyday example. Can you have concurrency without parallelism?
7. Pick the best tool (threads, processes, asyncio or none) for: (a) download 200 files, (b) resize 5,000 images, (c) find primes below 10 million, (d) keep 10,000 idle chat connections open, (e) parse 20 small JSON strings.
8. What does the GIL protect, and what does it not protect?
9. **[Hard]** Why might threads speed up NumPy code but not a pure Python loop?
10. What changed about the GIL in Python 3.13 and 3.14?

### Solutions 8.1

1. `MainThread`. Every program starts in the main thread.
2. `True`. The standard build has the GIL. Only the separate free-threaded build prints `False`.
5. It is **not** safe. `counter += 1` is a read, an add and a write. If several threads run `add_many`, a switch can happen between the read and the write, and an update is lost. Protect the update with a `threading.Lock` (see 8.5).
6. Concurrency: one cook switching between pots. Parallelism: several cooks at once. Yes: threads on one core, or asyncio, give concurrency without parallelism.
7. (a) threads or asyncio (waiting on the network); (b) processes (CPU work); (c) processes; (d) asyncio (very cheap per connection); (e) none: the work is tiny and the overhead of any pool is bigger than the gain.
8. It protects the interpreter's internal state (for example reference counts). It does not make a sequence of operations atomic, so your own shared data still needs locks.
9. NumPy runs its loops in C and releases the GIL while doing so, so several threads can compute at the same time. A Python loop holds the GIL while it runs bytecode.
10. 3.13 added an experimental free-threaded build (PEP 703). 3.14 made it officially supported but still optional (PEP 779). The default build still has the GIL.

In [428]:
# Solutions 8.1 (B) with checks
import sys

def classify(task):
    kinds = {
        "download files": "io",
        "query database": "io",
        "read disk": "io",
        "resize images": "cpu",
        "compute primes": "cpu",
        "compress video": "cpu",
    }
    return kinds[task]

def gil_status():
    if not hasattr(sys, "_is_gil_enabled"):
        return "unknown"
    return "on" if sys._is_gil_enabled() else "off"

assert classify("download files") == "io"
assert classify("compute primes") == "cpu"
assert gil_status() in {"on", "off", "unknown"}
print("Practice 8.1: all checks passed")

Practice 8.1: all checks passed


## 8.3 Threads with the threading module

A **thread** runs a function next to the main thread, inside the same process. The basic steps are:

1. Create it: `t = threading.Thread(target=func, args=(...))`.
2. Start it: `t.start()`. The function now runs in the new thread.
3. Wait for it: `t.join()`. The main thread stops here until `t` has finished.

```text
  main thread:  create t1, t2 --> start t1 --> start t2 --> join t1 --> join t2 --> continue
                                     |              |           ^           ^
                                     v              v           |           |
  thread t1:                      [ work ...........]-----------+           |
  thread t2:                                     [ work .........]----------+
```

In [429]:
# Start two threads and wait for both with join()
import threading
import time

finished = []                                   # threads share this list

def download(name, seconds):
    time.sleep(seconds)                         # stands in for a network wait
    finished.append(name)

t1 = threading.Thread(target=download, args=("orders.csv", 0.10))
t2 = threading.Thread(target=download, args=("products.csv", 0.05))
t1.start()
t2.start()
t1.join()                                       # wait until t1 is done
t2.join()                                       # wait until t2 is done

print(finished)                                 # ['products.csv', 'orders.csv']  (the faster one ends first)

['products.csv', 'orders.csv']


A thread function cannot `return` a value to you directly; `Thread` ignores the return value. Common ways to get results back:

* write into a shared list or dict (as above; safe for simple `append` calls, but see 8.5 for updates),
* use a `queue.Queue` (see 8.6),
* or, best for most code, use `ThreadPoolExecutor` (see 8.4), which gives you the return value.

**Daemon threads.** A **daemon** thread is a background thread that does not keep the program alive. When only daemon threads are left, Python exits and kills them. A normal (non-daemon) thread keeps the process alive until it ends.

> **Warning:** If a program ends before its threads print anything, the threads were daemon threads (or the process was killed). Non-daemon threads always finish. Call `join()` when you need the work to be done before you continue.

In [430]:
# Thread objects: names, daemon flag, and is_alive()
import threading
import time

def short_job():
    time.sleep(0.05)

worker = threading.Thread(target=short_job, name="report-worker")
print(worker.name)            # report-worker
print(worker.daemon)          # False  (normal threads keep the program alive)

worker.start()
print(worker.is_alive())      # True   (it is still sleeping)
worker.join()
print(worker.is_alive())      # False  (finished)

report-worker
False
True
False


**Exceptions in threads.** If the target function raises an exception, the thread prints the traceback (through `threading.excepthook`) and stops. The main thread does **not** get the exception. This is one more reason to prefer `ThreadPoolExecutor`, which hands errors back to you.

In [431]:
# An exception inside a plain thread does not reach the main thread
import threading

errors = []

def record_error(args):                     # called by Python when a thread crashes
    errors.append(f"{args.exc_type.__name__}: {args.exc_value}")

old_hook = threading.excepthook
threading.excepthook = record_error         # catch crashes instead of printing them

t = threading.Thread(target=lambda: 1 / 0)
t.start()
t.join()
threading.excepthook = old_hook             # always restore global settings

print("main thread is still running")       # main thread is still running
print(errors)                               # ['ZeroDivisionError: division by zero']

main thread is still running
['ZeroDivisionError: division by zero']


> **Key points**
> * `Thread(target=..., args=(...))`, then `start()`, then `join()`.
> * Threads share the memory of their process.
> * A plain thread throws away its return value and its exceptions do not reach you.
> * Daemon threads are killed when the program exits; normal threads keep it alive.

---

## 8.4 Thread pools with ThreadPoolExecutor

Creating threads by hand is error-prone. The `concurrent.futures` module gives you a **thread pool**: a fixed group of worker threads that take tasks from a waiting line.

* `executor.submit(func, *args)` schedules one call and returns a **Future**: an object that will hold the result (or the exception) later.
* `future.result()` waits for the call and returns its value. If the call raised an exception, `result()` raises it again here.
* `executor.map(func, items)` runs `func` on every item and gives results **in input order**.
* `as_completed(futures)` gives futures **in the order they finish**.
* The `with` block waits for all tasks when it ends.

How a task flows through a thread pool:

```text
   +------------------+     submit()     +---------------------+
   |   Main thread    |----------------->|  Task queue (FIFO)  |
   +--------+---------+                  +----------+----------+
            |                                       |
            | gets a Future at once                 | a free worker takes the task
            v                                       v
   +------------------+                  +---------------------+
   | future.result()  |                  | Worker thread runs  |
   | waits here ...   |                  | func(*args)         |
   +--------+---------+                  +----------+----------+
            ^                                       |
            |        result OR exception            |
            +-------- stored in the Future <--------+
```

In [432]:
# Sequential calls versus a thread pool for I/O-bound work
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_table(name):
    time.sleep(0.1)                      # stands in for a database query (waiting releases the GIL)
    return f"{name}: ok"

tables = ["orders", "products", "customers", "reviews"]

start = time.perf_counter()
sequential = [fetch_table(t) for t in tables]
seq_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    threaded = list(executor.map(fetch_table, tables))   # map keeps the input order
pool_time = time.perf_counter() - start

print(threaded)                          # ['orders: ok', 'products: ok', 'customers: ok', 'reviews: ok']
print(sequential == threaded)            # True
print("pool is faster:", pool_time < seq_time)    # pool is faster: True  (about 0.1 s instead of 0.4 s)

['orders: ok', 'products: ok', 'customers: ok', 'reviews: ok']
True
pool is faster: True


In [433]:
# submit() + as_completed(): handle each result as soon as it is ready
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def job(name, seconds):
    time.sleep(seconds)
    return name

work = [("slow", 0.30), ("fast", 0.05), ("medium", 0.15)]

with ThreadPoolExecutor(max_workers=3) as executor:
    future_to_name = {executor.submit(job, n, s): n for n, s in work}
    done_order = [f.result() for f in as_completed(future_to_name)]

print(done_order)                        # ['fast', 'medium', 'slow']  (finish order, not input order)

['fast', 'medium', 'slow']


**Errors are stored in the Future.** If a task raises an exception, the pool does not crash and nothing is printed. The exception waits inside the future. You see it only when you call `future.result()` (or when you read the results of `executor.map`).

In [434]:
# Exceptions come back when you call result()
from concurrent.futures import ThreadPoolExecutor

def parse_price(text):
    return float(text)                   # raises ValueError for bad text

inputs = ["9.99", "abc", "5"]

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(parse_price, t) for t in inputs]

print("the with-block ended and no error was raised yet")

for text, future in zip(inputs, futures):
    try:
        print(text, "->", future.result())
    except ValueError as e:
        print(text, "-> failed:", e)
# 9.99 -> 9.99
# abc -> failed: could not convert string to float: 'abc'
# 5 -> 5.0

the with-block ended and no error was raised yet
9.99 -> 9.99
abc -> failed: could not convert string to float: 'abc'
5 -> 5.0


**How many workers?** The default `max_workers` is `min(32, number_of_CPUs + 4)`. For I/O-bound work you can use more threads than cores, because most threads are waiting. Do not create thousands of threads: each one uses memory. For thousands of waits, use asyncio (see 8.8).

**Common mistakes**

* Calling `submit` and never calling `result()`. Failed tasks then fail silently.
* Relying on completion order. Real network calls finish in a random order. Use `map` (input order) or keep a dict from future to input.
* Using a thread pool for CPU-heavy pure Python code. It is not faster under the GIL.

> **Key points**
> * `submit` returns a Future; `result()` returns the value or raises the stored exception.
> * `map` keeps input order; `as_completed` gives finish order.
> * The `with` block waits for all tasks to finish.
> * Thread pools are the simple, safe default for I/O-bound work.

---

### Practice 8.2: Threads and thread pools

**A. Predict the output**

1. What does this print?
   ```python
   import threading

   results = []

   def work(n):
       results.append(n * n)

   threads = [threading.Thread(target=work, args=(i,)) for i in range(4)]
   for t in threads:
       t.start()
   for t in threads:
       t.join()
   print(sorted(results))
   ```
2. What does this print?
   ```python
   from concurrent.futures import ThreadPoolExecutor

   with ThreadPoolExecutor(2) as ex:
       print(list(ex.map(len, ["a", "bbb", "cc"])))
   ```
3. Does this code print an error? Why or why not?
   ```python
   from concurrent.futures import ThreadPoolExecutor

   with ThreadPoolExecutor(2) as ex:
       ex.submit(int, "not a number")
   print("done")
   ```

**B. Write code**

4. Write `fetch_all(names)` that "fetches" each name with `time.sleep(0.05)` and returns `name.upper()`. Use `ThreadPoolExecutor.map`. Return the results in input order.
5. Write `run_all(funcs)` that runs zero-argument functions in a thread pool and returns `(results, errors)`. `results` keeps input order with `None` for failed calls; `errors` maps the index to the exception.
6. Write `first_finished(delays)` that submits `time.sleep(d)` style jobs returning `d`, and returns the delay of the job that finishes first. Use `as_completed`.
7. **[Hard]** Write `thread_results(func, items)` that uses plain `threading.Thread` objects (no executor) and still returns the results in input order. Hint: give each thread its index and write into a pre-sized list.

**C. Find and fix the bug**

8. The program ends but the downloads never happen. Why? Fix it.
   ```python
   import threading
   import time

   def download():
       time.sleep(0.1)
       print("downloaded")

   t = threading.Thread(target=download, daemon=True)
   t.start()
   # the script ends here
   ```
9. The results are in a random order. Fix it so they match `urls`.
   ```python
   with ThreadPoolExecutor(4) as ex:
       futures = [ex.submit(fetch, u) for u in urls]
       results = [f.result() for f in as_completed(futures)]
   ```

**D. Interview questions**

10. What is a Future? What happens to an exception raised inside a task?
11. When would you use `as_completed` instead of `map`?
12. Why can you use more threads than CPU cores for I/O-bound work?

### Solutions 8.2

1. `[0, 1, 4, 9]`. We sort because the threads may append in any order.
2. `[1, 3, 2]`. `map` returns results in input order.
3. It prints only `done`. The `ValueError` is stored in the future, and nobody calls `result()`, so the error is lost silently.
8. The thread is a daemon thread. When the main thread ends, Python kills daemon threads, so `print` never runs. Fix: call `t.join()` before the end, or create the thread without `daemon=True` (normal threads keep the program alive).
9. `as_completed` yields futures in finish order. Use `results = [f.result() for f in futures]` (input order), or `list(ex.map(fetch, urls))`.
10. A Future is a placeholder for a result that is not ready yet. If the task raises, the exception is stored in the Future and raised again when you call `result()`.
11. When you want to react to each result as soon as it is ready (show progress, stop early, or handle the fast ones first).
12. Threads that wait on I/O release the GIL and use no CPU, so many of them can wait at the same time. The limit is memory and the other system, not the number of cores.

In [435]:
# Solutions 8.2 (B and checks)
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# 4
def fetch_all(names):
    def fetch(name):
        time.sleep(0.05)
        return name.upper()
    with ThreadPoolExecutor(max_workers=max(1, len(names))) as ex:
        return list(ex.map(fetch, names))

assert fetch_all(["asha", "ravi", "meera"]) == ["ASHA", "RAVI", "MEERA"]
assert fetch_all([]) == []

# 5
def run_all(funcs):
    results, errors = [None] * len(funcs), {}
    with ThreadPoolExecutor(max_workers=max(1, len(funcs))) as ex:
        futures = [ex.submit(f) for f in funcs]
    for i, fut in enumerate(futures):
        try:
            results[i] = fut.result()
        except Exception as e:          # we collect every task error on purpose
            errors[i] = e
    return results, errors

res, errs = run_all([lambda: 1, lambda: 1 / 0, lambda: "x"])
assert res == [1, None, "x"]
assert list(errs) == [1] and isinstance(errs[1], ZeroDivisionError)
assert run_all([]) == ([], {})

# 6
def first_finished(delays):
    def job(d):
        time.sleep(d)
        return d
    with ThreadPoolExecutor(max_workers=len(delays)) as ex:
        futures = [ex.submit(job, d) for d in delays]
        return next(as_completed(futures)).result()

assert first_finished([0.2, 0.02, 0.1]) == 0.02

# 7
def thread_results(func, items):
    out = [None] * len(items)
    def run(i, item):
        out[i] = func(item)             # each thread writes its own slot: no conflict
    threads = [threading.Thread(target=run, args=(i, x)) for i, x in enumerate(items)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return out

assert thread_results(lambda x: x * 10, [3, 1, 2]) == [30, 10, 20]
assert thread_results(str, []) == []

# 3 (check that the error really was stored and not raised)
with ThreadPoolExecutor(2) as ex:
    fut = ex.submit(int, "not a number")
assert isinstance(fut.exception(), ValueError)
print("Practice 8.2: all checks passed")

Practice 8.2: all checks passed


## 8.5 Race conditions and locks

Threads share memory. When two threads **read and then write** the same data, the result can depend on timing. This bug is a **race condition**.

Example: two threads both want to add 1 to `balance = 100`.

```text
   thread A                 thread B                 balance
   read balance (100)                                  100
                            read balance (100)         100
   write 100 + 1                                       101
                            write 100 + 1              101   <- one update lost
```

Race conditions are hard to see because they happen only sometimes. The demo below forces the bad timing with a `threading.Barrier` (a meeting point: every thread waits there until all have arrived), so the bug appears every time.

In [436]:
# A race condition made visible: both threads read before either writes
import threading

balance = 100
both_have_read = threading.Barrier(2)       # each thread waits here until both arrive

def unsafe_deposit(amount):
    global balance
    current = balance                       # 1. read
    both_have_read.wait()                   # force the bad timing for the demo
    balance = current + amount              # 2. write a stale value

threads = [threading.Thread(target=unsafe_deposit, args=(1,)) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(balance)          # 101  (expected 102: one deposit was lost)

101


**The fix: a lock.** A `threading.Lock` lets only one thread at a time run a block of code. The read and the write then happen together, as one step. A block protected like this is called a **critical section**.

Use the lock with `with lock:`. The `with` statement always releases the lock, even if an exception happens (see Part 5, Context Managers).

In [437]:
# The same deposit, protected by a lock
import threading
import time

balance = 100
balance_lock = threading.Lock()

def safe_deposit(amount):
    global balance
    with balance_lock:                      # only one thread at a time inside
        current = balance
        time.sleep(0.01)                    # even a slow step between read and write is now safe
        balance = current + amount

threads = [threading.Thread(target=safe_deposit, args=(1,)) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(balance)          # 105

105


**Other synchronisation tools in `threading`:**

| Tool | What it does |
|---|---|
| `Lock` | One thread at a time in a block. |
| `RLock` | A lock the same thread may take again (re-entrant), e.g. in recursive methods. |
| `Semaphore(n)` | At most `n` threads at a time in a block (limit connections). |
| `Event` | One thread sets a flag; others wait for it (`wait()`, `set()`). |
| `Barrier(n)` | `n` threads wait until all of them arrive. |
| `queue.Queue` | Thread-safe waiting line for passing data (see 8.6). Often the best choice. |

In [438]:
# Event: worker threads wait for a "go" signal from the main thread
import threading

go = threading.Event()
started = []
started_lock = threading.Lock()

def worker(name):
    go.wait()                           # block until the event is set
    with started_lock:
        started.append(name)

threads = [threading.Thread(target=worker, args=(n,)) for n in ["Asha", "Ravi"]]
for t in threads:
    t.start()

print(started)                          # []   (nobody can start yet)
go.set()                                # signal all waiting threads
for t in threads:
    t.join()
print(sorted(started))                  # ['Asha', 'Ravi']

[]
['Asha', 'Ravi']


In [439]:
# Semaphore: at most 2 threads inside the block at the same time
import threading
import time

slots = threading.Semaphore(2)
state = {"active": 0, "peak": 0}
state_lock = threading.Lock()

def use_connection():
    with slots:                         # take one of the 2 slots
        with state_lock:
            state["active"] += 1
            state["peak"] = max(state["peak"], state["active"])
        time.sleep(0.05)
        with state_lock:
            state["active"] -= 1

threads = [threading.Thread(target=use_connection) for _ in range(6)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("most connections at once:", state["peak"])     # most connections at once: 2

most connections at once: 2


**Common mistakes**

* Protecting the write but not the read. The lock must cover the whole read-change-write.
* Using a different lock object in each thread (for example creating the lock inside the function). All threads must share **the same** lock.
* Calling `lock.acquire()` without a guaranteed `release()`. Use `with lock:`.
* Holding a lock while doing slow I/O. Other threads wait for no reason. Keep critical sections short.

> **Key points**
> * A race condition is a bug where the result depends on thread timing.
> * Protect shared read-change-write with one shared `Lock`, using `with`.
> * `Semaphore`, `Event` and `Barrier` coordinate threads; queues pass data safely.

---

## 8.6 Queues, producer/consumer and deadlocks

The safest way for threads to work together is to **not share changing data at all**, but to **send messages**. `queue.Queue` is a thread-safe waiting line:

* `q.put(item)` adds an item. With `maxsize`, `put` waits when the queue is full (this is called **back-pressure**: a fast producer is slowed down).
* `q.get()` removes an item and waits if the queue is empty.
* `q.task_done()` and `q.join()` let you wait until every item has been processed.

This gives the classic **producer/consumer** pattern:

```text
  +-----------+   put()   +-------------------+   get()   +-----------+
  | Producer  |---------->|  queue.Queue      |---------->| Consumer  |
  | (makes    |           |  [item][item][..] |           | (handles  |
  |  work)    |           +-------------------+           |  work)    |
  +-----------+         a sentinel says "stop"            +-----------+
```

A **sentinel** is a special value that means "no more items". We use a unique `object()`, so it can never be confused with real data.

In [440]:
# Producer/consumer with a bounded queue and a sentinel
import queue
import threading

tasks = queue.Queue(maxsize=3)          # bounded: the producer waits when 3 items are queued
STOP = object()                         # unique sentinel
squares = []

def producer():
    for n in range(1, 7):
        tasks.put(n)
    tasks.put(STOP)                     # tell the consumer to finish

def consumer():
    while True:
        item = tasks.get()
        if item is STOP:
            break
        squares.append(item * item)

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
p.start()
c.start()
p.join()
c.join()

print(squares)                          # [1, 4, 9, 16, 25, 36]

[1, 4, 9, 16, 25, 36]


With several consumers, put **one sentinel per consumer**. Each consumer stops after it reads one sentinel.

In [441]:
# Two consumers sum numbers from one queue
import queue
import threading

numbers = queue.Queue()
STOP = object()
partial_sums = []
sums_lock = threading.Lock()

def summer():
    total = 0
    while True:
        item = numbers.get()
        if item is STOP:
            break
        total += item
    with sums_lock:
        partial_sums.append(total)

consumers = [threading.Thread(target=summer) for _ in range(2)]
for c in consumers:
    c.start()
for n in range(1, 101):
    numbers.put(n)
for _ in consumers:
    numbers.put(STOP)                   # one sentinel per consumer
for c in consumers:
    c.join()

print(sum(partial_sums))                # 5050

5050


### Deadlock

A **deadlock** happens when threads wait for each other forever. Classic case: thread 1 holds lock A and wants lock B; thread 2 holds lock B and wants lock A. Neither can continue.

```text
     thread 1                         thread 2
   holds lock A                     holds lock B
        |                                |
        | wants B  ------ waits ------>  |
        |  <------ waits ------  wants A |
        v                                v
            both wait forever: DEADLOCK
```

A real deadlock would freeze the notebook. The demo uses `acquire(timeout=0.1)`, so each thread gives up instead of waiting forever.

In [442]:
# Detect a deadlock safely with acquire(timeout=...)
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()
both_hold_first = threading.Barrier(2)
both_tried = threading.Barrier(2)
outcome = {}

def task(name, first, second):
    with first:
        both_hold_first.wait()                       # make sure each thread holds its first lock
        got_it = second.acquire(timeout=0.1)         # would wait forever without a timeout
        both_tried.wait()                            # keep holding 'first' until both have tried
        if got_it:
            second.release()
        outcome[name] = got_it

t1 = threading.Thread(target=task, args=("t1", lock_a, lock_b))
t2 = threading.Thread(target=task, args=("t2", lock_b, lock_a))   # opposite order: danger
t1.start()
t2.start()
t1.join()
t2.join()

print(outcome["t1"], outcome["t2"])    # False False  (each waited for the other: a deadlock)

False False


**The fix: always take locks in the same global order.** If every thread takes A before B, a cycle of waiting cannot form.

In [443]:
# Same work, but both threads lock A first and then B
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()
log = []

def transfer(name):
    with lock_a:                       # fixed order: A, then B
        with lock_b:
            log.append(name)

threads = [threading.Thread(target=transfer, args=(n,)) for n in ["t1", "t2"]]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(sorted(log))                     # ['t1', 't2']  (both finished, no deadlock)

['t1', 't2']


**How to avoid deadlocks:**

* Take several locks in one fixed order everywhere.
* Hold locks for a short time; do not call unknown code while holding a lock.
* Use `with` so locks are always released.
* Use timeouts (`acquire(timeout=...)`) where a hang would be worse than an error.
* Prefer queues and immutable data, so you need fewer locks.

> **Key points**
> * `queue.Queue` passes work between threads safely; use sentinels to stop consumers.
> * A bounded queue gives back-pressure.
> * Deadlock = threads waiting for each other in a cycle. Fixed lock order prevents it.

---

### Practice 8.3: Race conditions, locks, queues and deadlocks

**A. Predict the output**

1. What does this print, and is the answer always the same?
   ```python
   import threading

   total = 0
   lock = threading.Lock()

   def add(n):
       global total
       for _ in range(n):
           with lock:
               total += 1

   threads = [threading.Thread(target=add, args=(1000,)) for _ in range(4)]
   for t in threads:
       t.start()
   for t in threads:
       t.join()
   print(total)
   ```
2. What does this print?
   ```python
   import queue

   q = queue.Queue()
   for word in ["a", "b", "c"]:
       q.put(word)
   print(q.get(), q.qsize())
   ```

**B. Write code**

3. Write a class `SafeCounter` with `increment()` and a `value` property that is safe to use from many threads. Test it with 4 threads times 1,000 increments.
4. Producer/consumer: one producer puts the numbers 1 to 20, **two** consumers sum them. The final total must be 210.
5. Write `bank_transfer(accounts, locks, src, dst, amount)` that moves money between two accounts and always takes the two locks in sorted order of the account names, to avoid deadlock.
6. **[Hard]** Write `parallel_word_count(lines, workers)` that counts words with several threads. Each thread counts into its own `collections.Counter`, and you merge the counters at the end (no lock needed). Compare with a single-threaded count.

**C. Find and fix the bug**

7. The total is sometimes wrong. Why?
   ```python
   def add(n):
       global total
       for _ in range(n):
           lock = threading.Lock()
           with lock:
               total += 1
   ```
8. This consumer never stops. Why?
   ```python
   def consumer():
       while True:
           item = q.get()
           handle(item)
   ```

**D. Interview questions**

9. What is a race condition? Give an example.
10. Name the four conditions needed for a deadlock and two ways to prevent it.
11. Why are queues often better than shared variables plus locks?
12. What is the difference between `Lock` and `RLock`?

### Solutions 8.3

1. `4000`, always. Every `+= 1` happens inside the same lock.
2. `a 2`. A `Queue` is first-in, first-out; after one `get` two items remain.
7. A **new** lock is created on every loop step, so each thread locks its own private lock. The threads never block each other. Create one lock outside the function and share it.
8. Nothing tells it to stop, and `q.get()` waits forever on an empty queue. Put a sentinel after the last item and `break` when it arrives (one sentinel per consumer).
9. A bug where the result depends on the timing of threads, e.g. two threads read the same balance and both write `balance + 1`, losing one update.
10. Mutual exclusion (a lock is held by one thread), hold and wait (a thread holds one lock while waiting for another), no pre-emption (locks are not taken away), circular wait. Prevent it with a fixed lock order, timeouts, or by using fewer locks (queues).
11. Each item is owned by one thread at a time, so there is no shared changing data. The queue handles all locking inside, and it adds back-pressure and a clean way to stop (sentinels).
12. A `Lock` cannot be taken twice, even by the same thread (it would deadlock). An `RLock` counts how many times the owning thread took it and must be released the same number of times.

In [444]:
# Solutions 8.3 (B and checks)
import queue
import threading
from collections import Counter

# 3
class SafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:
            self._value += 1

    @property
    def value(self):
        with self._lock:
            return self._value

c = SafeCounter()
threads = [threading.Thread(target=lambda: [c.increment() for _ in range(1000)]) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
assert c.value == 4000

# 4
work_q, STOP = queue.Queue(), object()
totals, totals_lock = [], threading.Lock()

def consume():
    s = 0
    while True:
        item = work_q.get()
        if item is STOP:
            break
        s += item
    with totals_lock:
        totals.append(s)

consumers = [threading.Thread(target=consume) for _ in range(2)]
for t in consumers:
    t.start()
for i in range(1, 21):
    work_q.put(i)
for _ in consumers:
    work_q.put(STOP)
for t in consumers:
    t.join()
assert sum(totals) == 210

# 5
def bank_transfer(accounts, locks, src, dst, amount):
    first, second = sorted([src, dst])          # same order for every thread
    with locks[first], locks[second]:
        if accounts[src] < amount:
            raise ValueError("insufficient funds")
        accounts[src] -= amount
        accounts[dst] += amount

accounts = {"asha": 100, "ravi": 100}
locks = {name: threading.Lock() for name in accounts}
jobs = [threading.Thread(target=bank_transfer, args=(accounts, locks, "asha", "ravi", 10)),
        threading.Thread(target=bank_transfer, args=(accounts, locks, "ravi", "asha", 30))]
for t in jobs:
    t.start()
for t in jobs:
    t.join()
assert accounts == {"asha": 120, "ravi": 80}

# 6
def parallel_word_count(lines, workers=2):
    chunks = [lines[i::workers] for i in range(workers)]   # split the work
    partial = [Counter() for _ in range(workers)]      # one private counter per thread

    def count(i):
        for line in chunks[i]:
            partial[i].update(line.split())

    threads = [threading.Thread(target=count, args=(i,)) for i in range(workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return sum(partial, Counter())

text = ["the cat sat", "the dog sat", "a cat ran", ""]
expected = Counter(word for line in text for word in line.split())
assert parallel_word_count(text, 3) == expected
assert parallel_word_count([], 2) == Counter()
print("Practice 8.3: all checks passed")

Practice 8.3: all checks passed


## 8.7 Processes with ProcessPoolExecutor

For **CPU-bound** pure Python work, threads do not help (the GIL). **Processes** do: each process has its own Python interpreter, its own memory and its own GIL, so several processes really compute at the same time on different cores.

The price you pay:

* Starting a process is slow (tens of milliseconds or more), much slower than a thread.
* Memory is **not shared**. A change to a global variable in a child process does not change the parent.
* Arguments and results are sent between processes as bytes using **pickle** (see Notebook 1, Part 13). Things that cannot be pickled (lambdas, open files, locks) cannot be sent.

`ProcessPoolExecutor` has the same interface as `ThreadPoolExecutor`: `submit`, `map`, `result`, `as_completed`. Switching between them is easy.

**Start methods.** How does Python create a child process?

| Method | How it works | Default on |
|---|---|---|
| `spawn` | Starts a fresh Python, then imports your module to find the function | Windows, macOS |
| `fork` | Copies the parent process (fast, but unsafe with threads) | Linux before 3.14 |
| `forkserver` | A small clean server process forks the children | Linux from Python 3.14 |

Because of `spawn`, two rules keep your code portable:

1. Put the code that starts processes under `if __name__ == "__main__":` in a script. Otherwise each child would import the script and start new processes again.
2. The worker function must live in a module that the child can **import**. A function defined in a notebook cell cannot be found by a spawned child.

So below we write a small module file into a temporary folder, add the folder to `sys.path`, and import the worker from it.

In [445]:
# Write an importable worker module into a temporary folder
import pathlib
import sys
import tempfile

workers_dir = pathlib.Path(tempfile.mkdtemp(prefix="nb2_workers_"))
(workers_dir / "cpu_workers.py").write_text(
    "def sum_of_squares(n):\n"
    "    total = 0\n"
    "    for i in range(n):\n"
    "        total += i * i\n"
    "    return total\n"
    "\n"
    "def chunk_sum(bounds):\n"
    "    lo, hi = bounds\n"
    "    return sum(i * i for i in range(lo, hi))\n",
    encoding="utf-8",
)
sys.path.insert(0, str(workers_dir))         # child processes inherit sys.path

import cpu_workers
print(cpu_workers.sum_of_squares(4))         # 14  (0 + 1 + 4 + 9)

14


In [446]:
# Run CPU-bound work in 4 processes
from concurrent.futures import ProcessPoolExecutor
import cpu_workers

jobs = [300_000] * 4

sequential = [cpu_workers.sum_of_squares(n) for n in jobs]

with ProcessPoolExecutor(max_workers=4) as executor:
    parallel = list(executor.map(cpu_workers.sum_of_squares, jobs))

print(parallel[0])                  # 8999955000050000
print(sequential == parallel)       # True

8999955000050000
True


**What about speed?** On a machine with 4 or more cores, the 4 processes take roughly a quarter of the sequential time for large jobs, while 4 threads take about the same time as sequential code (the GIL). For small jobs, the start-up cost can make processes **slower**. Timings vary from machine to machine, so always measure your own case.

Splitting one big job into chunks is a common pattern. Each process sums one range, and the parent adds the partial sums.

In [447]:
# Split one big range into chunks, sum each chunk in a separate process
from concurrent.futures import ProcessPoolExecutor
import cpu_workers

def make_chunks(n, parts):
    step = -(-n // parts)                        # ceiling division
    return [(lo, min(lo + step, n)) for lo in range(0, n, step)]

print(make_chunks(10, 3))                        # [(0, 4), (4, 8), (8, 10)]

n = 1_000_000
with ProcessPoolExecutor(max_workers=4) as executor:
    total = sum(executor.map(cpu_workers.chunk_sum, make_chunks(n, 4)))

print(total == sum(i * i for i in range(n)))     # True

[(0, 4), (4, 8), (8, 10)]
True


**Things that cannot be sent to a process.** A lambda cannot be pickled by reference, so the call fails. The error is stored in the future, like with threads.

In [448]:
# A lambda cannot be pickled, so it cannot run in another process
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor(max_workers=1) as executor:
    future = executor.submit(lambda x: x * 2, 21)
    try:
        future.result()
    except Exception as e:                        # the exact message differs between versions
        print(type(e).__name__)                  # PicklingError

PicklingError


**Sharing data between processes.** Processes do not share variables. When they must talk, use:

* `multiprocessing.Queue` or `Pipe` to send messages,
* `multiprocessing.Value`, `Array` or `shared_memory` for shared raw data,
* a `Manager` for shared lists and dicts (slower, but easy).

Usually the simplest design is best: send inputs in, get results back from `map`, and combine them in the parent.

**A complete script** (save as `squares.py` and run `python squares.py`). Notice the `__main__` guard:

```python
from concurrent.futures import ProcessPoolExecutor


def sum_of_squares(n):
    return sum(i * i for i in range(n))


def main():
    jobs = [5_000_000] * 4
    with ProcessPoolExecutor() as executor:
        print(list(executor.map(sum_of_squares, jobs)))


if __name__ == "__main__":       # required for spawn: children import this file
    main()
```

In a script the worker can live in the same file, because the child can import the script by name.

In [449]:
# Clean up: remove the worker module and its folder
import shutil
import sys

sys.path.remove(str(workers_dir))
sys.modules.pop("cpu_workers", None)
shutil.rmtree(workers_dir, ignore_errors=True)
print(workers_dir.exists())                      # False

False


**Common mistakes**

* Forgetting the `if __name__ == "__main__":` guard in a script. With `spawn` this can start processes again and again, or fail with a `RuntimeError`.
* Passing lambdas, open files, locks or database connections to processes (they cannot be pickled).
* Expecting a child to change the parent's global variables.
* Using processes for tiny tasks. The start-up and pickling cost is bigger than the work.

> **Key points**
> * Processes give real parallelism for CPU-bound Python code; each has its own GIL.
> * Arguments and results are pickled; memory is not shared.
> * `spawn` (Windows, macOS) needs importable workers and the `__main__` guard; Linux uses `forkserver` from 3.14.
> * `ProcessPoolExecutor` has the same API as `ThreadPoolExecutor`.

---

### Practice 8.4: Processes

**A. Predict the output**

1. What does this print, if `work` lives in an importable module and runs in a child process?
   ```python
   # in module workers.py
   seen = []

   def work(x):
       seen.append(x)
       return x * 2

   # in the main program
   with ProcessPoolExecutor(2) as ex:
       print(list(ex.map(workers.work, [1, 2, 3])))
   print(workers.seen)
   ```
2. What is the default start method on Windows? On Linux with Python 3.14?

**B. Write code**

3. Write `make_chunks(n, parts)` that splits `range(n)` into at most `parts` `(lo, hi)` pairs that cover every number once. Handle `n = 0`.
4. Write `parallel_sum_of_squares(n, workers)` with `ProcessPoolExecutor`. Put the worker into a module in a temp folder. It must return the same value as the sequential sum.
5. Write `count_primes_parallel(limits)` that counts primes below each limit in `limits` using a process pool, and returns a list of counts. Check: `count_primes_parallel([10, 100])` is `[4, 25]`.

**C. Find and fix the bug**

6. This script works on Linux before 3.14 but fails or misbehaves on Windows. Why?
   ```python
   from concurrent.futures import ProcessPoolExecutor

   def square(x):
       return x * x

   with ProcessPoolExecutor() as ex:
       print(list(ex.map(square, range(5))))
   ```
7. Why does this fail?
   ```python
   with ProcessPoolExecutor() as ex:
       print(list(ex.map(lambda x: x + 1, [1, 2, 3])))
   ```

**D. Interview questions**

8. Why do processes speed up CPU-bound Python code when threads do not (standard CPython)?
9. What are the costs of using processes?
10. **[Hard]** When can a process pool be slower than a simple loop? How would you check?

### Solutions 8.4

1. `[2, 4, 6]` and then `[]`. The `append` happens in the child processes, which have their own copy of the module. The parent's list is unchanged.
2. Windows uses `spawn`. Linux uses `forkserver` from Python 3.14 (it used `fork` before).
6. There is no `if __name__ == "__main__":` guard. With `spawn`, every child imports the script, and the import runs the pool code again. Put the pool code inside `main()` and call it under the guard.
7. Lambdas cannot be pickled by reference, so they cannot be sent to a child process (`PicklingError`). Use a normal function defined at the top level of a module.
8. Each process has its own interpreter and its own GIL, so they run Python bytecode in parallel on different cores. Threads in one standard CPython process share one GIL.
9. Slow start-up, more memory (one interpreter per process), pickling of inputs and outputs, no shared variables, more complex debugging.
10. When each task is tiny (the pickling and start-up cost dominates), or when the data to send is huge. Measure both versions with `time.perf_counter` or `timeit` on realistic data, and try a bigger `chunksize` in `map`.

In [450]:
# Solutions 8.4 (B and checks)
import pathlib
import shutil
import sys
import tempfile
from concurrent.futures import ProcessPoolExecutor

# 3
def make_chunks(n, parts):
    if n <= 0:
        return []
    step = -(-n // parts)
    return [(lo, min(lo + step, n)) for lo in range(0, n, step)]

assert make_chunks(10, 3) == [(0, 4), (4, 8), (8, 10)]
assert make_chunks(0, 4) == []
assert make_chunks(3, 5) == [(0, 1), (1, 2), (2, 3)]
assert sum(hi - lo for lo, hi in make_chunks(1001, 7)) == 1001

# 4 and 5: workers in an importable module
practice_dir = pathlib.Path(tempfile.mkdtemp(prefix="nb2_p84_"))
(practice_dir / "p84_workers.py").write_text(
    "def chunk_sum(bounds):\n"
    "    lo, hi = bounds\n"
    "    return sum(i * i for i in range(lo, hi))\n"
    "\n"
    "def count_primes(limit):\n"
    "    count = 0\n"
    "    for n in range(2, limit):\n"
    "        if all(n % d for d in range(2, int(n ** 0.5) + 1)):\n"
    "            count += 1\n"
    "    return count\n",
    encoding="utf-8",
)
sys.path.insert(0, str(practice_dir))
import p84_workers

def parallel_sum_of_squares(n, workers=2):
    with ProcessPoolExecutor(max_workers=workers) as ex:
        return sum(ex.map(p84_workers.chunk_sum, make_chunks(n, workers)))

def count_primes_parallel(limits):
    with ProcessPoolExecutor(max_workers=2) as ex:
        return list(ex.map(p84_workers.count_primes, limits))

try:
    assert parallel_sum_of_squares(10_000, 3) == sum(i * i for i in range(10_000))
    assert parallel_sum_of_squares(0, 2) == 0
    assert count_primes_parallel([10, 100, 2]) == [4, 25, 0]
finally:
    sys.path.remove(str(practice_dir))
    sys.modules.pop("p84_workers", None)
    shutil.rmtree(practice_dir, ignore_errors=True)
print("Practice 8.4: all checks passed")

Practice 8.4: all checks passed


## 8.8 asyncio: many waits on one thread

`asyncio` runs many tasks on **one thread** using an **event loop**. It is built for programs that wait a lot: web servers, chat servers, web crawlers, API clients with thousands of requests.

The key words:

* A **coroutine function** is defined with `async def`. Calling it does not run it; it returns a **coroutine object**.
* `await x` means: "pause me here until `x` is done, and let other tasks run meanwhile".
* The **event loop** is the scheduler. It runs one task until that task reaches an `await`, then switches to another task that is ready.
* A **task** wraps a coroutine so the event loop can run it concurrently with others.

**In simple words:** asyncio is one waiter serving many tables. The waiter takes an order, and while the kitchen cooks, serves other tables. Nobody stands and waits.

How the event loop runs tasks:

```text
   +---------------------------+
   | Event loop: pick a task   |<-----------------------------+
   | that is ready to run      |                              |
   +-------------+-------------+                              |
                 |                                            |
                 v                                            |
   +---------------------------+                              |
   | Run the task until it     |                              |
   | reaches an await          |                              |
   +-------------+-------------+                              |
                 |                                            |
                 v                                            |
   +---------------------------+  Yes   +------------------+  |
   | Did the task finish?      |------->| Save its result  |--+
   +-------------+-------------+        +------------------+  |
                 | No: it waits for I/O or a timer            |
                 v                                            |
   +---------------------------+                              |
   | Park the task. Wake it up |                              |
   | when its I/O is ready     |------------------------------+
   +---------------------------+
```

The switching is **cooperative**: a task gives control back only at an `await`. A task that never awaits (or calls a blocking function like `time.sleep`) stops every other task.

### Running async code: notebooks versus scripts

* In a **script**, start the event loop with `asyncio.run(main())`. It creates a loop, runs `main()` until it finishes, and closes the loop.
* In **Jupyter**, an event loop is already running, so you write `await main()` directly in a cell (top-level `await`). Calling `asyncio.run()` inside Jupyter raises `RuntimeError: asyncio.run() cannot be called from a running event loop`.

The cells below use top-level `await`, like Jupyter. A script version looks like this:

```python
import asyncio


async def main():
    await asyncio.sleep(0.1)
    print("done")


if __name__ == "__main__":
    asyncio.run(main())
```

In [451]:
# Calling a coroutine function only creates a coroutine object
import asyncio

async def greet(name):
    await asyncio.sleep(0.01)            # pause; other tasks could run here
    return f"Hello, {name}"

coro = greet("Asha")
print(type(coro).__name__)               # coroutine  (nothing has run yet)
coro.close()                             # we will not run it; close it to avoid a warning

message = await greet("Ravi")            # top-level await (Jupyter); use asyncio.run(...) in a script
print(message)                           # Hello, Ravi

coroutine
Hello, Ravi


### Sequential awaits versus gather

`await` on one coroutine after another runs them **one after another**. To run them **at the same time**, schedule them together with `asyncio.gather(...)`. `gather` returns the results **in the order you passed them**, not in finish order.

In [452]:
# Three 0.1 s waits: one after another versus together
import asyncio
import time

async def fetch(name, delay):
    await asyncio.sleep(delay)           # non-blocking wait
    return f"{name} done"

start = time.perf_counter()
one_by_one = [await fetch("a", 0.1), await fetch("b", 0.1), await fetch("c", 0.1)]
sequential_time = time.perf_counter() - start

start = time.perf_counter()
together = await asyncio.gather(fetch("a", 0.1), fetch("b", 0.1), fetch("c", 0.1))
gather_time = time.perf_counter() - start

print(together)                          # ['a done', 'b done', 'c done']
print(one_by_one == together)            # True
print("gather is faster:", gather_time < sequential_time)   # gather is faster: True  (about 0.1 s vs 0.3 s)

['a done', 'b done', 'c done']
True
gather is faster: True


### create_task: start now, await later

`asyncio.create_task(coro)` schedules a coroutine to start **right away** as a task. You can do other work and `await` the task later. Keep a reference to every task you create; otherwise it may be garbage-collected before it finishes.

In [453]:
# create_task starts work in the background of the event loop
import asyncio

events = []

async def download(name, delay):
    events.append(f"start {name}")
    await asyncio.sleep(delay)
    events.append(f"end {name}")
    return name

task_a = asyncio.create_task(download("A", 0.05))
task_b = asyncio.create_task(download("B", 0.02))
events.append("main continues")            # tasks have not started yet: main has not awaited
await asyncio.sleep(0)                      # give the loop a chance to start the tasks

print(await task_a, await task_b)          # A B
print(events)
# ['main continues', 'start A', 'start B', 'end B', 'end A']

A B
['main continues', 'start A', 'start B', 'end B', 'end A']


### TaskGroup: structured concurrency (Python 3.11+)

`asyncio.TaskGroup` is the modern, safer way to run several tasks:

* Tasks are created with `tg.create_task(...)` inside `async with asyncio.TaskGroup() as tg:`.
* The `async with` block waits until all tasks are finished.
* If one task fails, the group **cancels the other tasks**, and the errors are raised together as an `ExceptionGroup`. You catch them with `except*` (see Notebook 1, Part 12).

In [454]:
# TaskGroup: all tasks finish before the block ends
import asyncio

async def price(item, delay):
    await asyncio.sleep(delay)
    return {"pen": 10, "book": 250}[item]

async with asyncio.TaskGroup() as tg:
    pen_task = tg.create_task(price("pen", 0.02))
    book_task = tg.create_task(price("book", 0.01))

print(pen_task.result() + book_task.result())      # 260

260


In [455]:
# TaskGroup: one failure cancels the others and raises an ExceptionGroup
import asyncio

cancelled = []

async def slow_job():
    try:
        await asyncio.sleep(5)
    except asyncio.CancelledError:
        cancelled.append("slow_job")
        raise                                    # always re-raise CancelledError

async def failing_job():
    await asyncio.sleep(0.02)
    raise ValueError("bad record")

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(slow_job())
        tg.create_task(failing_job())
except* ValueError as group:
    print("errors:", [str(e) for e in group.exceptions])    # errors: ['bad record']

print("cancelled:", cancelled)                               # cancelled: ['slow_job']

errors: ['bad record']
cancelled: ['slow_job']


**gather and errors.** With `gather`, the first exception is raised to the caller, but the other tasks keep running. With `return_exceptions=True`, exceptions are returned as values in the result list, which is handy when you want "all results, failed or not".

In [456]:
# gather(return_exceptions=True) collects errors as values
import asyncio

async def parse(text):
    await asyncio.sleep(0.01)
    return int(text)

results = await asyncio.gather(parse("1"), parse("x"), parse("3"), return_exceptions=True)
for r in results:
    print(type(r).__name__, r)
# int 1
# ValueError invalid literal for int() with base 10: 'x'
# int 3

int 1
ValueError invalid literal for int() with base 10: 'x'
int 3


### Timeouts and cancellation

* `async with asyncio.timeout(seconds):` (Python 3.11+) cancels the work inside if it takes too long and raises `TimeoutError`.
* `await asyncio.wait_for(coro, seconds)` does the same for a single awaitable (works on older versions too).
* `task.cancel()` asks a task to stop. Inside the task, `asyncio.CancelledError` is raised at the current `await`. Do cleanup in `except` or `finally`, then **re-raise** it.

In [457]:
# Timeouts with asyncio.timeout and asyncio.wait_for
import asyncio

try:
    async with asyncio.timeout(0.05):
        await asyncio.sleep(5)                     # too slow
except TimeoutError:
    print("timed out")                             # timed out

async def quick():
    await asyncio.sleep(0.01)
    return "fast result"

print(await asyncio.wait_for(quick(), timeout=1))  # fast result

timed out
fast result


In [458]:
# Cancelling a task: clean up, then re-raise CancelledError
import asyncio

async def worker():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("worker: cleaning up")               # close files, connections ...
        raise                                      # let the cancellation finish

task = asyncio.create_task(worker())
await asyncio.sleep(0.02)                          # let the worker start
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("cancelled:", task.cancelled())          # cancelled: True
# worker: cleaning up
# cancelled: True

worker: cleaning up
cancelled: True


### Semaphore: limit how many run at once

If you start 1,000 requests at the same time, a server may refuse them. An `asyncio.Semaphore(n)` lets at most `n` tasks into a block at the same time.

In [459]:
# At most 3 "requests" at the same time
import asyncio

limit = asyncio.Semaphore(3)
state = {"active": 0, "peak": 0}

async def request(i):
    async with limit:                              # wait for a free slot
        state["active"] += 1
        state["peak"] = max(state["peak"], state["active"])
        await asyncio.sleep(0.02)
        state["active"] -= 1
        return i * 10

results = await asyncio.gather(*(request(i) for i in range(10)))
print(results)                                     # [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
print("most at once:", state["peak"])              # most at once: 3

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
most at once: 3


### Never block the event loop

A normal blocking call such as `time.sleep()`, `requests.get()` or a heavy calculation **freezes all tasks**, because the loop runs on one thread and only switches at `await`.

* For waiting, use the async version: `await asyncio.sleep(...)`, an async HTTP library, and so on.
* For a blocking function you cannot change, use `await asyncio.to_thread(func, *args)` (Python 3.9+). It runs the function in a worker thread and lets the loop continue.
* For heavy CPU work, use a process pool (`loop.run_in_executor` with a `ProcessPoolExecutor`).

In [460]:
# Blocking time.sleep in async code versus asyncio.sleep and to_thread
import asyncio
import time

async def blocking_wait():
    time.sleep(0.05)                    # Common mistake: blocks the whole event loop

async def friendly_wait():
    await asyncio.sleep(0.05)           # gives control back to the loop

def legacy_call():                      # a blocking library function we cannot change
    time.sleep(0.05)
    return "legacy ok"

start = time.perf_counter()
await asyncio.gather(*(blocking_wait() for _ in range(4)))
blocked = time.perf_counter() - start

start = time.perf_counter()
await asyncio.gather(*(friendly_wait() for _ in range(4)))
friendly = time.perf_counter() - start

start = time.perf_counter()
answers = await asyncio.gather(*(asyncio.to_thread(legacy_call) for _ in range(4)))
in_threads = time.perf_counter() - start

print("blocking version is much slower:", blocked > 2 * friendly)       # True  (about 0.2 s vs 0.05 s)
print(answers[0], "| to_thread ran them together:", in_threads < blocked)  # legacy ok | ... True

blocking version is much slower: True
legacy ok | to_thread ran them together: True


### asyncio.Queue

`asyncio.Queue` is the async version of `queue.Queue`. Its `put` and `get` are awaited. It is used for async producer/consumer pipelines, for example a web crawler with a fixed number of worker tasks.

In [461]:
# Async producer/consumer with a fixed number of workers
import asyncio

async def crawler_worker(name, q, done):
    while True:
        url = await q.get()
        await asyncio.sleep(0.01)                  # pretend to download
        done.append(url)
        q.task_done()

q = asyncio.Queue()
done = []
for i in range(6):
    q.put_nowait(f"page{i}")

workers = [asyncio.create_task(crawler_worker(f"w{i}", q, done)) for i in range(2)]
await q.join()                                     # wait until every item is processed
for w in workers:
    w.cancel()                                     # workers loop forever: stop them
await asyncio.gather(*workers, return_exceptions=True)

print(sorted(done))     # ['page0', 'page1', 'page2', 'page3', 'page4', 'page5']

['page0', 'page1', 'page2', 'page3', 'page4', 'page5']


**Useful asyncio tools at a glance:**

| Tool | Purpose |
|---|---|
| `async def` / `await` | Define a coroutine / wait for an awaitable |
| `asyncio.run(main())` | Start the loop in a script |
| `asyncio.create_task(coro)` | Run a coroutine concurrently as a task |
| `asyncio.gather(*aws)` | Run several awaitables, results in input order |
| `asyncio.TaskGroup()` (3.11+) | Structured concurrency; cancels the rest on failure |
| `asyncio.timeout(s)` (3.11+), `wait_for` | Time limits |
| `asyncio.Semaphore(n)` | Limit concurrency |
| `asyncio.to_thread(f, ...)` | Run a blocking function without freezing the loop |
| `asyncio.Queue` | Async producer/consumer |

**Common mistakes**

* Calling a coroutine without `await`: nothing runs, and Python warns "coroutine ... was never awaited".
* Using `time.sleep` or other blocking calls inside `async def`.
* Catching `CancelledError` and not re-raising it.
* Calling `asyncio.run()` inside Jupyter or inside another coroutine.
* Creating a task and losing the reference to it.

> **Key points**
> * asyncio is cooperative multitasking on one thread; tasks switch only at `await`.
> * `gather` and `TaskGroup` run coroutines concurrently; `TaskGroup` is the safer modern choice.
> * Use `asyncio.timeout`, `Semaphore` and `to_thread` for limits, throttling and blocking code.
> * Scripts use `asyncio.run(main())`; Jupyter uses top-level `await`.

---

### Practice 8.5: asyncio

**A. Predict the output**

1. What does this print?
   ```python
   import asyncio

   async def double(x):
       await asyncio.sleep(0.01)
       return x * 2

   print(await asyncio.gather(double(3), double(1), double(2)))
   ```
2. About how long does this take: 0.1 s or 0.3 s?
   ```python
   async def wait():
       await asyncio.sleep(0.1)

   await wait()
   await wait()
   await wait()
   ```
3. What is printed?
   ```python
   async def hello():
       return "hi"

   x = hello()
   print(type(x).__name__)
   x.close()
   ```

**B. Write code**

4. Write `fetch_all(items, limit)` that "fetches" each item with `await asyncio.sleep(0.01)` and returns `item * 2`. Run all items concurrently, but at most `limit` at a time. Return `(results, peak)`: the results in input order, and the largest number of fetches that were running at the same time (it must be at most `limit`).
5. Write `with_timeout(coro, seconds)` that returns the result, or `None` if it does not finish in time.
6. Write `first_result(delays)` that starts one task per delay (each returns its delay after sleeping) and returns the first one to finish. Cancel the rest. Hint: `asyncio.wait(..., return_when=asyncio.FIRST_COMPLETED)`.
7. **[Hard]** Write `retry_async(func, attempts)` that awaits `func()` and retries when it raises `ConnectionError`, up to `attempts` times, then re-raises the last error.

**C. Find and fix the bug**

8. All tasks run one after another. Why? Fix it.
   ```python
   async def job():
       time.sleep(0.1)

   await asyncio.gather(job(), job(), job())
   ```
9. The cancelled task never stops cleanly and `task.cancelled()` is `False`. Why?
   ```python
   async def worker():
       try:
           await asyncio.sleep(10)
       except asyncio.CancelledError:
           print("cleaning up")
   ```

**D. Interview questions**

10. What is the difference between threads (pre-emptive) and asyncio (cooperative) for shared data and for blocking calls?
11. What does `TaskGroup` do when one task fails? How is that different from `gather`?
12. Why does `asyncio.run()` fail inside Jupyter, and what do you write instead?

### Solutions 8.5

1. `[6, 2, 4]`. `gather` keeps the order of its arguments.
2. About 0.3 s. Each `await` waits for the previous call to finish. Use `gather` to overlap them.
3. `coroutine`. Calling an `async def` function only creates a coroutine object.
8. `time.sleep` blocks the event loop, so nothing else can run. Use `await asyncio.sleep(0.1)`. If the blocking call is a real library function, use `await asyncio.to_thread(func)`.
9. The `except` block swallows `CancelledError`, so the task finishes normally instead of being cancelled. Add `raise` after the cleanup.
10. Threads can be switched at almost any point, so any shared data needs locks; but a blocking call only blocks its own thread. asyncio switches only at `await`, so code between two awaits cannot be interrupted (simpler shared state), but one blocking call freezes every task.
11. It cancels the other tasks in the group and raises all errors together as an `ExceptionGroup` (catch with `except*`). `gather` raises the first error to the caller but does not cancel the other tasks (unless the gather itself is cancelled).
12. Jupyter already runs an event loop, and `asyncio.run` must create a new one. Write `await main()` directly in the cell.

In [462]:
# Solutions 8.5 (B and checks)
import asyncio

# 4
async def fetch_all(items, limit):
    sem = asyncio.Semaphore(limit)
    state = {"active": 0, "peak": 0}

    async def one(item):
        async with sem:
            state["active"] += 1
            state["peak"] = max(state["peak"], state["active"])
            await asyncio.sleep(0.01)
            state["active"] -= 1
            return item * 2

    results = await asyncio.gather(*(one(x) for x in items))
    return results, state["peak"]

out, peak = await fetch_all(list(range(8)), 3)
assert out == [0, 2, 4, 6, 8, 10, 12, 14] and peak <= 3
assert await fetch_all([], 2) == ([], 0)

# 5
async def with_timeout(coro, seconds):
    try:
        return await asyncio.wait_for(coro, seconds)
    except TimeoutError:
        return None

assert await with_timeout(asyncio.sleep(0.01, result="fast"), 0.5) == "fast"
assert await with_timeout(asyncio.sleep(1, result="slow"), 0.02) is None

# 6
async def first_result(delays):
    async def job(d):
        await asyncio.sleep(d)
        return d
    tasks = [asyncio.create_task(job(d)) for d in delays]
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
    for t in pending:
        t.cancel()
    await asyncio.gather(*pending, return_exceptions=True)   # wait for the cancellations
    return done.pop().result()

assert await first_result([0.2, 0.01, 0.1]) == 0.01

# 7
async def retry_async(func, attempts):
    for attempt in range(1, attempts + 1):
        try:
            return await func()
        except ConnectionError:
            if attempt == attempts:
                raise

calls = {"n": 0}
async def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("server busy")
    return "ok"

assert await retry_async(flaky, 5) == "ok" and calls["n"] == 3

async def always_fails():
    raise ConnectionError("down")

try:
    await retry_async(always_fails, 2)
    raise AssertionError("should have failed")
except ConnectionError as e:
    assert str(e) == "down"
print("Practice 8.5: all checks passed")

Practice 8.5: all checks passed


## 8.9 Choosing an approach

Use this chart when you decide how to speed up a program:

```text
              +----------------------------------+
              | Is the program too slow?         |
              | Measure first (Part 9, 9.8)      |
              +----------------+-----------------+
                               |
                               v
              +----------------------------------+
              | Where does the time go?          |
              +--------+----------------+--------+
                       |                |
             waiting (I/O)        computing (CPU)
                       |                |
                       v                v
      +-------------------------+  +---------------------------+
      | Thousands of waits and  |  | Pure Python code?         |
      | async libraries exist?  |  +-----+---------------+-----+
      +-----+-------------+-----+        | Yes           | No (NumPy etc.)
            | Yes         | No           v               v
            v             v       +-------------+  +------------------+
      +-----------+ +------------+| Processes   |  | Vectorise first; |
      | asyncio   | | Threads    || (Process-   |  | threads may help |
      |           | | (ThreadPool||  PoolExec.) |  | (GIL released)   |
      +-----------+ |  Executor) |+-------------+  +------------------+
                    +------------+
```

**Checklist**

1. **Measure first.** Profile to see whether time goes to waiting or computing (see Part 9, section 9.8).
2. **Waiting, a moderate number of tasks:** `ThreadPoolExecutor`. Simple and works with normal blocking libraries.
3. **Waiting, thousands of connections:** `asyncio` with async libraries. Wrap blocking calls with `asyncio.to_thread`.
4. **Computing in pure Python:** `ProcessPoolExecutor`. Computing in NumPy/pandas: vectorise first.
5. **Shared data?** Prefer queues and immutable data. Otherwise protect it with a lock.
6. **Tiny tasks?** Pools add overhead (a thread costs microseconds, a process milliseconds, plus pickling). Maybe stay sequential.

| | Threads | Processes | asyncio |
|---|---|---|---|
| Best for | I/O, tens to hundreds of tasks | CPU-bound Python | I/O, thousands of tasks |
| Memory | Shared | Separate | Shared (one thread) |
| Switching | Pre-emptive (any time) | OS runs them in parallel | Only at `await` |
| Main risk | Race conditions | Pickling, start-up cost | Blocking the loop |

> **Interview tip:** Always explain *why*: "This is I/O-bound, so threads or asyncio; the GIL is released while waiting."

---

## 8.10 How to debug concurrent code

Concurrency bugs are hard because they depend on timing. A bug may appear once in a hundred runs. Use this process:

```text
  +-------------------------------+
  | Symptom: wrong result, hang,  |
  | missing output, slow program  |
  +---------------+---------------+
                  |
                  v
  +-------------------------------+      Yes     +-------------------------+
  | Does it also fail with ONE    |------------->| Normal bug: debug it    |
  | worker / sequential code?     |              | without concurrency     |
  +---------------+---------------+              +-------------------------+
                  | No
                  v
  +-------------------------------+
  | Look for: shared data without |
  | a lock, lost exceptions,      |
  | blocking calls, lock order    |
  +---------------+---------------+
                  |
                  v
  +-------------------------------+
  | Add logging with thread/task  |
  | names; add timeouts to hangs  |
  +---------------+---------------+
                  |
                  v
  +-------------------------------+
  | Fix, then run many times      |
  | (and add a test)              |
  +-------------------------------+
```

**Typical problems and what they mean:**

| Symptom or message | Meaning | Fix |
|---|---|---|
| Total is sometimes too small | Race condition (lost update) | One shared `Lock` around read-change-write |
| Program hangs forever | Deadlock, or a consumer waiting on an empty queue | Fixed lock order, timeouts, sentinels |
| Task "failed silently" | Exception stored in a Future | Call `future.result()` |
| `RuntimeWarning: coroutine '...' was never awaited` | You called an `async def` without `await` | Add `await` or create a task |
| `RuntimeError: asyncio.run() cannot be called from a running event loop` | `asyncio.run` inside Jupyter or a coroutine | Use `await` |
| All async tasks freeze | A blocking call in async code | `await asyncio.sleep`, `asyncio.to_thread` |
| `PicklingError` / `BrokenProcessPool` | Worker or data cannot be sent to a process | Top-level function in an importable module, `__main__` guard |

In [463]:
# Reproduce two async errors safely and print their messages
import asyncio

async def try_asyncio_run():
    coro = asyncio.sleep(0)
    try:
        asyncio.run(coro)                      # wrong: a loop is already running
    except RuntimeError as e:
        print(type(e).__name__ + ":", e)
    finally:
        coro.close()                           # avoid "never awaited" warning for this demo

await try_asyncio_run()
# RuntimeError: asyncio.run() cannot be called from a running event loop

async def compute():
    return 42

result = compute()                             # Common mistake: forgot await
print(type(result).__name__)                   # coroutine  (not 42)
result.close()
print(await compute())                         # 42  (fixed)

RuntimeError: asyncio.run() cannot be called from a running event loop
coroutine
42


**Logging helps.** Put the thread name or task name in every log message, so you can see who did what. The `logging` module has `%(threadName)s` built in (see Part 7, logging).

In [464]:
# Log messages that show which thread did the work
import io
import logging
import threading

stream = io.StringIO()
logger = logging.getLogger("concurrency_demo")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(threadName)s: %(message)s"))
logger.addHandler(handler)
logger.propagate = False

def job(n):
    logger.info("processing item %d", n)

threads = [threading.Thread(target=job, args=(i,), name=f"worker-{i}") for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()
logger.removeHandler(handler)

for line in sorted(stream.getvalue().splitlines()):
    print(line)
# worker-0: processing item 0
# worker-1: processing item 1
# worker-2: processing item 2

worker-0: processing item 0
worker-1: processing item 1
worker-2: processing item 2


**More debugging habits**

* **Make it sequential first.** Run with `max_workers=1`. If the bug stays, it is not a concurrency bug.
* **Add timeouts** (`future.result(timeout=5)`, `asyncio.timeout`, `lock.acquire(timeout=...)`) so a hang becomes an error you can read.
* **Run many times.** A race may show up only under load. Repeat the test in a loop.
* **asyncio debug mode:** `asyncio.run(main(), debug=True)` or the environment variable `PYTHONASYNCIODEBUG=1` reports slow callbacks and never-awaited coroutines.
* **`faulthandler.dump_traceback_later(10)`** prints the stack of every thread if the program is still running after 10 seconds, which shows where a hang is.

> **Key points**
> * First check whether the bug also happens without concurrency.
> * Lost exceptions, races, deadlocks and blocking calls are the usual suspects.
> * Name your threads and tasks and log them; add timeouts to find hangs.

---

### Practice 8.6: Choosing and debugging

**A. Predict the output**

1. What type is printed?
   ```python
   async def total():
       return 10

   value = total()
   print(type(value).__name__)
   value.close()
   ```
2. What does `future.result(timeout=0.01)` raise if the task takes 1 second?

**B. Write code**

3. Write `choose_tool(kind, many_connections=False, pure_python=True)` that returns `"threads"`, `"asyncio"`, `"processes"` or `"vectorise"` following the chart in 8.9. `kind` is `"io"` or `"cpu"`.
4. Write `safe_results(futures)` that returns a list of `("ok", value)` or `("error", "TypeName: message")` tuples, one per future, in order.
5. **[Hard]** Write `run_with_deadline(funcs, seconds)` that runs functions in a thread pool and returns the results of those that finish within `seconds` (in input order), with `None` for the ones that timed out. Use `concurrent.futures.wait(..., timeout=seconds)`.

**C. Find and fix the bug**

6. This code hangs. Find the reason.
   ```python
   lock = threading.Lock()

   def outer():
       with lock:
           inner()

   def inner():
       with lock:
           print("inside")
   ```
7. This async code is no faster than sequential code. Why?
   ```python
   async def main():
       for url in urls:
           await fetch(url)
   ```

**D. Interview questions**

8. You must call 500 REST APIs and then compute statistics on 2 GB of numbers. Which tools do you use for each step?
9. How do you find out where a program hangs?
10. Why is "make it sequential first" a good first debugging step?

### Solutions 8.6

1. `coroutine`.
2. `TimeoutError` (in `concurrent.futures`, `TimeoutError` is the built-in `TimeoutError` since Python 3.11).
6. `inner` tries to take the same `Lock` that `outer` already holds. A `Lock` cannot be taken twice, so the thread waits for itself forever. Use `threading.RLock()` (re-entrant), or restructure so `inner` does not lock.
7. The loop awaits each fetch before starting the next, so the requests run one at a time. Use `await asyncio.gather(*(fetch(u) for u in urls))` or a `TaskGroup`.
8. The API calls are I/O-bound: a thread pool (or asyncio with a semaphore to limit concurrent requests). The statistics are CPU-bound: use NumPy (vectorised), or processes on chunks for pure Python.
9. Add timeouts so the hang becomes an error, use `faulthandler.dump_traceback_later` or a debugger to see the stack of every thread, and log thread names to see the last action of each one.
10. It separates logic bugs from timing bugs. If the bug remains with one worker, you can debug normal code; if it disappears, look at shared data, locks and ordering.

In [465]:
# Solutions 8.6 (B and checks)
import time
from concurrent.futures import ThreadPoolExecutor, wait

# 3
def choose_tool(kind, many_connections=False, pure_python=True):
    if kind == "io":
        return "asyncio" if many_connections else "threads"
    if kind == "cpu":
        return "processes" if pure_python else "vectorise"
    raise ValueError(f"unknown kind: {kind!r}")

assert choose_tool("io") == "threads"
assert choose_tool("io", many_connections=True) == "asyncio"
assert choose_tool("cpu") == "processes"
assert choose_tool("cpu", pure_python=False) == "vectorise"

# 4
def safe_results(futures):
    out = []
    for f in futures:
        try:
            out.append(("ok", f.result()))
        except Exception as e:
            out.append(("error", f"{type(e).__name__}: {e}"))
    return out

with ThreadPoolExecutor(2) as ex:
    futs = [ex.submit(int, "5"), ex.submit(int, "x")]
assert safe_results(futs) == [("ok", 5), ("error", "ValueError: invalid literal for int() with base 10: 'x'")]

# 5
def run_with_deadline(funcs, seconds):
    ex = ThreadPoolExecutor(max_workers=max(1, len(funcs)))
    futures = [ex.submit(f) for f in funcs]
    wait(futures, timeout=seconds)
    results = [f.result() if f.done() else None for f in futures]
    ex.shutdown(wait=False, cancel_futures=True)   # do not wait for the slow ones
    return results

res = run_with_deadline([lambda: "quick", lambda: time.sleep(0.3) or "slow"], 0.1)
assert res == ["quick", None]
time.sleep(0.25)                                   # let the slow demo thread finish before moving on
print("Practice 8.6: all checks passed")

Practice 8.6: all checks passed


### Summary of Part 8

* Concurrency is overlapping tasks; parallelism is tasks running at the same instant.
* I/O-bound work: threads or asyncio. CPU-bound pure Python: processes.
* The GIL (standard CPython) lets one thread run bytecode at a time; it does not make your code thread-safe. Free-threaded builds are experimental in 3.13 and supported but optional in 3.14.
* `ThreadPoolExecutor` and `ProcessPoolExecutor` share one API; always call `result()` to see errors.
* Protect shared data with a `Lock`, pass work through queues, and take locks in a fixed order to avoid deadlocks.
* Processes need picklable, importable workers and the `__main__` guard (`spawn` on Windows/macOS, `forkserver` on Linux from 3.14).
* asyncio is cooperative: tasks switch only at `await`. Use `gather`/`TaskGroup`, `timeout`, `Semaphore` and `to_thread`, and never block the loop.
* Scripts call `asyncio.run(main())`; Jupyter uses top-level `await`.

---

# Part 9: Testing, Code Quality, Security and Performance

**What these notes cover**

* Why automated tests matter, and the words used in testing.
* How to write tests with `unittest`: assertions, `setUp`, `subTest`, `assertRaises`.
* How `pytest` tests look (plain `assert`, fixtures, parametrize) and how to run them.
* How to isolate code with mocks: `Mock`, `return_value`, `side_effect`, `patch` (patch where the name is looked up), autospec.
* `doctest`, good test design, and coverage.
* Code quality tools (ruff, black, mypy) and safe refactoring.
* Secure coding habits: SQL injection, password hashing, `secrets`, `literal_eval`, safe `subprocess`.
* How to measure speed and memory: `timeit`, `cProfile`, `tracemalloc`, plus practical performance tips.
* How to debug a failing test with the test-debug-fix loop.

**Before you start:** Notebook 1 Part 11 (Functions), Part 12 (Errors and Debugging) and Part 14 (Modules); in this notebook Part 1 (Classes) and Part 5 (Context Managers).

## 9.1 Why we test

Code that is not tested is code you *hope* works. A **test** is a small piece of code that calls your code with known inputs and checks the result automatically.

Tests help you to:

* find bugs early, while the code is fresh in your mind,
* change (refactor) code safely: if the tests still pass, behaviour did not change,
* document how a function should behave,
* catch **regressions**: old bugs that come back after a change.

**Words you will meet:**

| Term | Meaning |
|---|---|
| **Unit test** | Tests one small unit (a function or a class) alone. Fast, no network or disk. |
| **Integration test** | Tests several parts together (code plus a database or an API). |
| **Assertion** | One check, such as "the result equals 5". |
| **Fixture** | Preparation shared by tests (test data, a temp folder). |
| **Mock** | A fake object that stands in for a real dependency and records how it was used. |
| **Coverage** | Which lines of code ran during the tests. |
| **Regression** | A bug that returns after it was fixed. |

> **Note:** The `assert` statement inside a program checks an internal assumption while the program runs, and it is removed when Python runs with `-O`. A *test* is separate code that calls your program from outside. Do not use `assert` to validate user input (see Notebook 1, Part 12).

**The test pyramid.** Write many fast unit tests, fewer integration tests, and only a few slow end-to-end tests.

```text
            /\
           /  \        End-to-end tests (few, slow)
          /----\
         /      \      Integration tests (some)
        /--------\
       /          \    Unit tests (many, fast)
      /____________\
```

Good tests are **FIRST**: Fast, Independent (no test depends on another), Repeatable (same result every run), Self-checking (pass or fail, no reading output by eye), Timely (written with the code).

> **Key points**
> * Tests give you confidence to change code.
> * Unit tests are small and fast; integration tests check parts working together.
> * Tests must be independent and repeatable.

---

## 9.2 unittest

`unittest` is in the standard library. The rules:

1. Write a class that inherits from `unittest.TestCase`.
2. Every method whose name starts with `test` is one test.
3. Inside a test, use assertion methods such as `self.assertEqual(actual, expected)`.
4. From a terminal, run `python -m unittest` in the project folder. It discovers files named `test*.py` (usually in a `tests/` folder).

In a notebook there is no test file, so we load the test classes ourselves. This helper runs them and prints a short report:

In [466]:
# A helper to run unittest classes inside a notebook
import io
import unittest

def run_tests(*test_classes):
    """Run TestCase classes and print a one-line summary."""
    suite = unittest.TestSuite()
    for cls in test_classes:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    stream = io.StringIO()                                   # keep the runner output quiet
    result = unittest.TextTestRunner(stream=stream, verbosity=0).run(suite)
    status = "OK" if result.wasSuccessful() else "FAILED"
    print(f"tests run: {result.testsRun}, {status} "
          f"(failures={len(result.failures)}, errors={len(result.errors)})")
    return result

print(callable(run_tests))       # True

True


In [467]:
# The code under test and its first tests
import unittest

def slugify(text):
    """Turn '  Hello, World!  ' into 'hello-world'."""
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    cleaned = "".join(ch if ch.isalnum() else " " for ch in text.lower())
    return "-".join(cleaned.split())

class TestSlugify(unittest.TestCase):
    def test_basic_sentence(self):
        self.assertEqual(slugify("  Hello, World!  "), "hello-world")

    def test_empty_string(self):
        self.assertEqual(slugify(""), "")

    def test_rejects_non_strings(self):
        with self.assertRaises(TypeError):          # the error is part of the contract
            slugify(None)

result = run_tests(TestSlugify)        # tests run: 3, OK (failures=0, errors=0)

tests run: 3, OK (failures=0, errors=0)


**Common assertion methods:**

| Method | Checks that |
|---|---|
| `assertEqual(a, b)` / `assertNotEqual` | `a == b` / `a != b` |
| `assertTrue(x)` / `assertFalse(x)` | `bool(x)` is True / False |
| `assertIs(a, b)` / `assertIsNone(x)` | `a is b` / `x is None` |
| `assertIn(a, b)` | `a in b` |
| `assertAlmostEqual(a, b)` | floats are equal to 7 decimal places |
| `assertRaises(Err)` | the block raises `Err` (use as `with`) |
| `assertRaisesRegex(Err, "text")` | it raises `Err` and the message matches |

Prefer the specific method (`assertEqual`, not `assertTrue(a == b)`): when it fails, it prints both values.

### setUp, tearDown and subTest

* `setUp()` runs **before each test**; `tearDown()` runs **after each test**. Each test gets fresh objects, so tests stay independent.
* `setUpClass` / `tearDownClass` (class methods) run once per class, for expensive setup.
* `with self.subTest(...)` checks many cases in one test, and reports each failing case separately.

In [468]:
# setUp gives every test a fresh object; subTest checks many cases
import unittest

class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def __len__(self):
        return len(self._items)

class TestStack(unittest.TestCase):
    def setUp(self):
        self.stack = Stack()                       # new stack for EVERY test

    def test_last_in_first_out(self):
        self.stack.push(1)
        self.stack.push(2)
        self.assertEqual(self.stack.pop(), 2)
        self.assertEqual(self.stack.pop(), 1)

    def test_pop_empty_raises(self):
        with self.assertRaisesRegex(IndexError, "empty"):
            self.stack.pop()

    def test_len_after_pushes(self):
        for count in [0, 1, 5]:
            with self.subTest(count=count):        # each count is reported on its own
                stack = Stack()
                for i in range(count):
                    stack.push(i)
                self.assertEqual(len(stack), count)

result = run_tests(TestStack)          # tests run: 3, OK (failures=0, errors=0)

tests run: 3, OK (failures=0, errors=0)


### Reading a test failure

A failing test shows the test name, a traceback, and an `AssertionError` with the **actual** and the **expected** value. Here is a test with a wrong expectation, so you can see the shape:

In [469]:
# A failing test, and the important line of its report
import unittest

class TestBroken(unittest.TestCase):
    def test_wrong_expectation(self):
        self.assertEqual(slugify("A b"), "a_b")     # the code returns "a-b"

result = run_tests(TestBroken)                     # tests run: 1, FAILED (failures=1, errors=0)

test, report = result.failures[0]
print(test.id().split(".")[-1])                    # test_wrong_expectation
for line in report.splitlines():
    if line.startswith("AssertionError"):
        print(line)                                # AssertionError: 'a-b' != 'a_b'

tests run: 1, FAILED (failures=1, errors=0)
test_wrong_expectation
AssertionError: 'a-b' != 'a_b'


**Failure versus error.** A *failure* means an assertion was false (the code gave a wrong answer). An *error* means the test crashed with some other exception (for example a `TypeError` in the code or in the test).

**Common mistakes**

* Test method names that do not start with `test`. They are silently skipped.
* Tests that share a module-level list or object. They pass or fail depending on order. Create state in `setUp`.
* Wrapping an assertion in `try/except Exception: pass`. The test can then never fail.
* Testing only the "happy path". Test empty input, one element, wrong types and error cases too.

> **Key points**
> * A test class inherits `unittest.TestCase`; methods named `test_...` are tests.
> * Use specific assertions; use `assertRaises` as a context manager for errors.
> * `setUp` keeps tests independent; `subTest` reports each case separately.
> * Run from a terminal with `python -m unittest`.

---

### Practice 9.1: unittest

**A. Predict the output**

1. How many tests does this class have?
   ```python
   class TestMath(unittest.TestCase):
       def test_add(self):
           self.assertEqual(1 + 1, 2)

       def check_sub(self):
           self.assertEqual(3 - 1, 2)

       def test_mul(self):
           self.assertEqual(2 * 3, 6)
   ```
2. Is this a failure or an error?
   ```python
   class TestX(unittest.TestCase):
       def test_it(self):
           self.assertEqual(int("abc"), 0)
   ```
3. How many times does `setUp` run for a class with 4 tests?

**B. Write code**

4. Write `is_palindrome(s)` that ignores case and punctuation and raises `TypeError` for non-strings. Write a `unittest` class that covers: a normal palindrome, mixed case, punctuation, the empty string, one character, a non-palindrome and the wrong type. Use `subTest`.
5. Write a `BankAccount` class with `deposit`, `withdraw` and a read-only `balance` property. Withdrawing too much or depositing a non-positive amount raises `ValueError`. Write tests using `setUp` and `assertRaises`.
6. Write a test that checks `0.1 + 0.2` equals `0.3` using the right assertion.

**C. Find and fix the bug**

7. This test can never fail. Why?
   ```python
   def test_total(self):
       try:
           self.assertEqual(total([1, 2]), 4)
       except Exception:
           pass
   ```
8. `test_b` passes alone but fails when run after `test_a`. Why?
   ```python
   ITEMS = []

   class TestCart(unittest.TestCase):
       def test_a(self):
           ITEMS.append("pen")
           self.assertEqual(len(ITEMS), 1)

       def test_b(self):
           self.assertEqual(ITEMS, [])
   ```

**D. Interview questions**

9. What is the difference between a unit test and an integration test?
10. What does "tests must be independent" mean, and why does it matter?
11. Why is `assertEqual(a, b)` better than `assertTrue(a == b)`?

### Solutions 9.1

1. Two. `check_sub` does not start with `test`, so it is not run.
2. An **error**: `int("abc")` raises `ValueError` before the assertion runs.
3. Four times: `setUp` runs before every test.
7. The `except Exception` catches the `AssertionError`, so the failure is hidden. Remove the `try/except`.
8. Both tests share the module-level list `ITEMS`. After `test_a` it is not empty. Create the list in `setUp` (for example `self.items = []`) so each test starts clean.
9. A unit test checks one small piece alone and runs fast; an integration test checks that several pieces (for example code and a real database) work together; it is slower but catches wiring problems.
10. Every test must pass or fail on its own, in any order. Otherwise a failure in one test causes confusing failures in others, and running a single test gives a different result.
11. On failure, `assertEqual` prints both values (`'a-b' != 'a_b'`). `assertTrue` only says `False is not true`.

In [470]:
# Solutions 9.1 (B) with checks
import unittest

# 4
def is_palindrome(s):
    if not isinstance(s, str):
        raise TypeError("s must be a string")
    cleaned = [c.lower() for c in s if c.isalnum()]
    return cleaned == cleaned[::-1]

class TestPalindrome(unittest.TestCase):
    def test_cases(self):
        cases = [("racecar", True), ("RaceCar", True),
                 ("A man, a plan, a canal: Panama", True),
                 ("", True), ("x", True), ("hello", False)]
        for text, expected in cases:
            with self.subTest(text=text):
                self.assertEqual(is_palindrome(text), expected)

    def test_wrong_type(self):
        with self.assertRaises(TypeError):
            is_palindrome(121)

# 5
class BankAccount:
    def __init__(self):
        self._balance = 0

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount

class TestBankAccount(unittest.TestCase):
    def setUp(self):
        self.account = BankAccount()

    def test_deposit_then_withdraw(self):
        self.account.deposit(100)
        self.account.withdraw(30)
        self.assertEqual(self.account.balance, 70)

    def test_invalid_amounts(self):
        for bad in [0, -5]:
            with self.subTest(amount=bad):
                with self.assertRaises(ValueError):
                    self.account.deposit(bad)

    def test_overdraw_keeps_balance(self):
        with self.assertRaisesRegex(ValueError, "insufficient"):
            self.account.withdraw(1)
        self.assertEqual(self.account.balance, 0)

# 6
class TestFloat(unittest.TestCase):
    def test_sum(self):
        self.assertAlmostEqual(0.1 + 0.2, 0.3)

result = run_tests(TestPalindrome, TestBankAccount, TestFloat)
assert result.wasSuccessful() and result.testsRun == 6
print("Practice 9.1: all checks passed")

tests run: 6, OK (failures=0, errors=0)
Practice 9.1: all checks passed


## 9.3 pytest

`pytest` is the most popular test tool in the Python world. It is a **third-party** package: install it with `python -m pip install pytest`. It can also run `unittest` tests.

Why people like it:

* Tests are plain functions, and checks are plain `assert` statements. pytest shows the values when an assert fails.
* **Fixtures** provide test data or resources; a test asks for a fixture by naming it as a parameter.
* `@pytest.mark.parametrize` runs one test with many inputs.
* `pytest.raises(Error)` checks exceptions.
* Built-in fixtures such as `tmp_path` (a fresh temp folder) and `monkeypatch` (temporarily change attributes or environment variables).

A typical test file `tests/test_prices.py`:

```python
import pytest

from shop.prices import apply_discount


@pytest.fixture
def cart():
    return [{"name": "pen", "price": 10.0}, {"name": "book", "price": 250.0}]


def test_total_without_discount(cart):          # "cart" is filled in by the fixture
    assert apply_discount(cart, 0) == 260.0


@pytest.mark.parametrize("percent, expected", [(10, 234.0), (50, 130.0), (100, 0.0)])
def test_discounts(cart, percent, expected):
    assert apply_discount(cart, percent) == pytest.approx(expected)


def test_invalid_percent(cart):
    with pytest.raises(ValueError, match="percent"):
        apply_discount(cart, 120)
```

Common commands (run in the project folder):

| Command | What it does |
|---|---|
| `python -m pytest` | Find and run all `test_*.py` files |
| `python -m pytest -q` | Quiet output |
| `python -m pytest -k discount` | Only tests whose name contains "discount" |
| `python -m pytest -x` | Stop at the first failure |
| `python -m pytest --pdb` | Open the debugger at a failure |
| `python -m pytest -v` | One line per test |

**A runnable demo.** We write a small module and a test file into a temp folder, then run pytest as a separate program with `subprocess` (see Part 7), exactly like you would in a terminal. Timings in pytest's last line change on every run, so we remove them before printing.

In [471]:
# Write a module and a pytest test file into a temp folder
import pathlib
import tempfile
import textwrap

pytest_dir = pathlib.Path(tempfile.mkdtemp(prefix="nb2_pytest_"))

(pytest_dir / "prices.py").write_text(textwrap.dedent("""\
    def apply_discount(items, percent):
        if not 0 <= percent <= 100:
            raise ValueError("percent must be between 0 and 100")
        total = sum(item["price"] for item in items)
        return round(total * (100 - percent) / 100, 2)
"""), encoding="utf-8")

(pytest_dir / "test_prices.py").write_text(textwrap.dedent("""\
    import pytest
    from prices import apply_discount

    @pytest.fixture
    def cart():
        return [{"name": "pen", "price": 10.0}, {"name": "book", "price": 250.0}]

    def test_no_discount(cart):
        assert apply_discount(cart, 0) == 260.0

    @pytest.mark.parametrize("percent, expected", [(10, 234.0), (50, 130.0), (100, 0.0)])
    def test_discounts(cart, percent, expected):
        assert apply_discount(cart, percent) == pytest.approx(expected)

    def test_invalid_percent(cart):
        with pytest.raises(ValueError, match="percent"):
            apply_discount(cart, 120)
"""), encoding="utf-8")

print(sorted(p.name for p in pytest_dir.iterdir()))    # ['prices.py', 'test_prices.py']

['prices.py', 'test_prices.py']


In [472]:
# Run pytest in a separate process and print the summary line without the timing
import os
import re
import subprocess
import sys

def run_pytest(folder):
    env = dict(os.environ,
               PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",   # ignore extra installed plugins
               PYTHONDONTWRITEBYTECODE="1",          # no stale .pyc files after we edit a module
               PYTHONIOENCODING="utf-8")
    proc = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
        cwd=folder, capture_output=True, text=True, encoding="utf-8", env=env,
    )
    lines = [line for line in proc.stdout.splitlines() if line.strip()]
    summary = re.sub(r" in [\d.]+s.*$", "", lines[-1]).strip("= ")   # drop " in 0.05s"
    return proc.returncode, summary, proc.stdout

code, summary, _ = run_pytest(pytest_dir)
print("exit code:", code)          # exit code: 0
print(summary)                     # 5 passed

exit code: 0
5 passed


The parametrized test counts as three tests, so 5 tests ran. Now we add a bug to the module and run again. Lines that start with `E` in pytest's report explain the failure.

In [473]:
# Introduce a bug, run pytest again, show the explanation lines
import shutil

prices_file = pytest_dir / "prices.py"
buggy = prices_file.read_text(encoding="utf-8").replace("(100 - percent)", "(100 + percent)")
prices_file.write_text(buggy, encoding="utf-8")

code, summary, output = run_pytest(pytest_dir)
print("exit code:", code)          # exit code: 1
print(summary)                     # 3 failed, 2 passed
first_e_line = next(line for line in output.splitlines() if line.startswith("E "))
print(first_e_line.replace("\u00b1", "+-").strip())
# E       assert 286.0 == 234.0 +- 2.3e-04   (approx shows the allowed tolerance after +-)

shutil.rmtree(pytest_dir, ignore_errors=True)  # clean up the temp folder

exit code: 1
3 failed, 2 passed
E       assert 286.0 == 234.0 +- 2.3e-04


> **Tip:** pytest rewrites `assert` statements, so a failure shows the actual values (`286.0 == 234.0`). That is why plain `assert` is enough in pytest tests.

> **Key points**
> * pytest uses plain functions and `assert`; it is third-party (`pip install pytest`).
> * Fixtures provide data; parametrize runs one test with many inputs; `pytest.raises` checks errors.
> * Run with `python -m pytest`; exit code 0 means all tests passed.

---

## 9.4 Mocking: replacing dependencies in tests

Unit tests should be fast and repeatable. They should not call a real network, database, clock or random generator. A **mock** stands in for such a dependency.

`unittest.mock.Mock` creates an object that accepts any call and remembers it:

* `mock.method.return_value = x` sets what a call returns.
* `mock.method.side_effect = SomeError` makes the call raise; a list makes successive calls return successive values.
* `mock.method.assert_called_once_with(args)`, `assert_not_called()`, `call_count` and `call_args` check how it was used.

In [474]:
# Mock basics: return_value, side_effect and call checks
from unittest import mock

gateway = mock.Mock()
gateway.charge.return_value = {"status": "ok"}

print(gateway.charge(50))                      # {'status': 'ok'}
gateway.charge.assert_called_once_with(50)     # passes silently
print(gateway.charge.call_count)               # 1

dice = mock.Mock(side_effect=[6, 1, 3])        # successive calls return successive values
print(dice(), dice(), dice())                  # 6 1 3

network = mock.Mock(side_effect=ConnectionError("timeout"))
try:
    network()
except ConnectionError as e:
    print("ConnectionError:", e)               # ConnectionError: timeout

{'status': 'ok'}
1
6 1 3
ConnectionError: timeout


### Technique 1: dependency injection

The easiest way to make code testable is to **pass the dependency in** (this is called **dependency injection**). Then a test can pass a mock.

In [475]:
# Testing a class that receives its payment gateway as a parameter
import unittest
from unittest import mock

class Checkout:
    def __init__(self, gateway):
        self.gateway = gateway                    # injected dependency

    def pay(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        try:
            response = self.gateway.charge(amount)
        except ConnectionError:
            return False                          # degrade gracefully
        return response["status"] == "ok"

class TestCheckout(unittest.TestCase):
    def test_success(self):
        gateway = mock.Mock()
        gateway.charge.return_value = {"status": "ok"}
        self.assertTrue(Checkout(gateway).pay(50))
        gateway.charge.assert_called_once_with(50)

    def test_declined(self):
        gateway = mock.Mock()
        gateway.charge.return_value = {"status": "declined"}
        self.assertFalse(Checkout(gateway).pay(10))

    def test_network_error_is_handled(self):
        gateway = mock.Mock()
        gateway.charge.side_effect = ConnectionError    # a failure we cannot trigger for real
        self.assertFalse(Checkout(gateway).pay(10))

    def test_invalid_amount_never_calls_gateway(self):
        gateway = mock.Mock()
        with self.assertRaises(ValueError):
            Checkout(gateway).pay(0)
        gateway.charge.assert_not_called()

result = run_tests(TestCheckout)        # tests run: 4, OK (failures=0, errors=0)

tests run: 4, OK (failures=0, errors=0)


### Technique 2: patch, and "patch where it is looked up"

When you cannot pass the dependency in, `mock.patch("module.name")` replaces a name **temporarily** (inside a `with` block or a decorated test) and restores it afterwards.

The golden rule: **patch the name where it is looked up, not where it is defined.**

* If `report.py` does `from weather import get_temperature`, then `report` has its own name `get_temperature`. Patch `"report.get_temperature"`.
* If `report.py` does `import weather` and calls `weather.get_temperature()`, patch `"weather.get_temperature"`.

```text
  weather.py                      report.py
  +--------------------+          +-------------------------------+
  | get_temperature ---+---+      | from weather import           |
  +--------------------+   |      |     get_temperature           |
                           +----->| get_temperature  (own name!)  |
  patch("weather.get_...")        | describe() uses THIS name     |
  changes only the left box       +-------------------------------+
                                  patch("report.get_temperature")
                                  changes the name describe() uses
```

The demo writes two small modules into a temp folder so we can patch them by module name.

In [476]:
# Two real modules: report.py imports get_temperature from weather.py
import pathlib
import sys
import tempfile
import textwrap

mock_dir = pathlib.Path(tempfile.mkdtemp(prefix="nb2_mock_"))
(mock_dir / "weather.py").write_text(textwrap.dedent("""\
    def get_temperature(city):
        raise ConnectionError("no network during tests")
"""), encoding="utf-8")
(mock_dir / "report.py").write_text(textwrap.dedent("""\
    from weather import get_temperature

    def describe(city):
        return "hot" if get_temperature(city) >= 30 else "mild"
"""), encoding="utf-8")
sys.path.insert(0, str(mock_dir))

import report
print(report.describe.__module__)       # report

report


In [477]:
# Patching the wrong place has no effect; patching where it is looked up works
from unittest import mock
import report

with mock.patch("weather.get_temperature", return_value=35):     # Common mistake: wrong target
    try:
        report.describe("Delhi")
    except ConnectionError as e:
        print("wrong target, real code ran:", e)   # wrong target, real code ran: no network during tests

with mock.patch("report.get_temperature", return_value=35) as fake:  # correct target
    print(report.describe("Delhi"))                # hot
    fake.assert_called_once_with("Delhi")

with mock.patch("report.get_temperature", return_value=20):
    print(report.describe("Shimla"))               # mild

wrong target, real code ran: no network during tests
hot
mild


### autospec: mocks that refuse wrong calls

A plain `Mock()` accepts **any** attribute and **any** arguments. A typo like `gateway.chrage(50)` passes silently. `mock.create_autospec(real_thing)` (or `patch(..., autospec=True)`) builds a mock with the same methods and signatures as the real object, so wrong calls fail.

In [478]:
# autospec catches typos and wrong arguments
from unittest import mock
import report

loose = mock.Mock()
loose.chrage(50)                                    # typo, but no error: dangerous
print("plain Mock accepted a typo")                 # plain Mock accepted a typo

strict = mock.create_autospec(report.get_temperature, return_value=25)
print(strict("Pune"))                               # 25
try:
    strict("Pune", "extra argument")                # the real function takes one argument
except TypeError as e:
    print("TypeError:", e)                          # TypeError: too many positional arguments

plain Mock accepted a typo
25
TypeError: too many positional arguments


In [479]:
# Clean up the temp modules
import shutil
import sys

sys.path.remove(str(mock_dir))
for name in ["report", "weather"]:
    sys.modules.pop(name, None)
shutil.rmtree(mock_dir, ignore_errors=True)
print(mock_dir.exists())                            # False

False


**Making time and randomness testable.** The clock and random numbers are hidden inputs. Either inject them (pass `now` or a `random.Random` object as a parameter) or patch them. Injection is usually cleaner:

In [480]:
# Inject the clock instead of calling datetime.now() inside the logic
from datetime import datetime, timezone

def greeting(now=None):
    now = now or datetime.now(timezone.utc)        # real clock by default
    return "Good morning" if now.hour < 12 else "Good afternoon"

print(greeting(datetime(2026, 1, 1, 9, 0, tzinfo=timezone.utc)))    # Good morning
print(greeting(datetime(2026, 1, 1, 15, 0, tzinfo=timezone.utc)))   # Good afternoon

Good morning
Good afternoon


**Common mistakes**

* Patching the wrong target: the test passes, but the real code still runs (or the mock is never used).
* Mocking so much that the test only tests the mock. Mock the edges (network, disk, clock), not your own logic.
* Using a plain `Mock` where a typo would go unnoticed. Use `autospec`.
* Asserting internal details instead of behaviour, so every refactor breaks the tests.

> **Key points**
> * `Mock` records calls; `return_value` and `side_effect` control what it does.
> * Prefer dependency injection; use `patch` when you must.
> * Patch where the name is looked up.
> * Use `create_autospec` / `autospec=True` so mocks match the real signature.

---

### Practice 9.2: pytest and mocking

**A. Predict the output**

1. What does this print?
   ```python
   from unittest import mock

   m = mock.Mock(return_value=7)
   print(m(1, 2), m.call_count)
   ```
2. What does this print?
   ```python
   m = mock.Mock(side_effect=[1, 2])
   print(m() + m())
   ```
3. How many tests does pytest count here?
   ```python
   @pytest.mark.parametrize("n", [1, 2, 3, 4])
   def test_positive(n):
       assert n > 0
   ```

**B. Write code**

4. Write `notify(user, mailer)` that sends a welcome email only if `user` has an `"email"` key and returns `True`/`False`. Test it with a `Mock` mailer: check the call arguments, and check that no call happens without an email.
5. Write `flip_coin()` using `random.random() < 0.5` for heads. Make it deterministic in a test with `mock.patch("random.random", ...)`.
6. Write `fetch_with_retry(client, url, attempts=3)` that calls `client.get(url)` and retries on `ConnectionError`. Test it with a mock whose `side_effect` is `[ConnectionError, ConnectionError, "data"]`.
7. **[Hard]** Write `is_weekend(today=None)` that uses `date.today()` by default. Test it for a Saturday and a Monday without patching, by injecting the date.

**C. Find and fix the bug**

8. The patch has no effect. Why? What is the right target?
   ```python
   # in shop.py
   from random import randint

   def roll():
       return randint(1, 6)

   # in the test
   with mock.patch("random.randint", return_value=6):
       assert shop.roll() == 6
   ```
9. This test passes even though `send_email` is misspelled in the code. Why, and how do you prevent it?
   ```python
   mailer = mock.Mock()
   mailer.sned_email("a@b.c")    # bug in the code under test
   ```

**D. Interview questions**

10. What is a fixture in pytest? Give an example of a built-in one.
11. Why should unit tests not call a real network?
12. What is dependency injection, and why does it make testing easier?

### Solutions 9.2

1. `7 1`.
2. `3`: the first call returns 1, the second returns 2.
3. Four tests: one per parameter value.
8. `shop.py` did `from random import randint`, so `shop` has its own name `randint`. Patching `random.randint` does not change it. Patch `"shop.randint"`.
9. A plain `Mock` creates any attribute on demand. Use `mock.create_autospec(RealMailer)` or `Mock(spec=RealMailer)`, so an unknown attribute raises `AttributeError`.
10. A function that prepares something a test needs; a test receives it by naming it as a parameter. Built-in examples: `tmp_path` (a temp folder), `monkeypatch`, `capsys` (captured output).
11. A network is slow and can fail or change, so tests become slow and flaky. You also cannot easily test error cases. Mock it and test the real network in a few integration tests.
12. Passing the dependencies (gateway, clock, client) into a function or class instead of creating them inside. Tests can then pass fakes or mocks without patching.

In [481]:
# Solutions 9.2 (B) with checks
import random
import unittest
from datetime import date
from unittest import mock

# 4
def notify(user, mailer):
    if not user.get("email"):
        return False
    mailer.send(user["email"], "Welcome!")
    return True

class TestNotify(unittest.TestCase):
    def test_sends_when_email_present(self):
        mailer = mock.Mock()
        self.assertTrue(notify({"email": "asha@example.org"}, mailer))
        mailer.send.assert_called_once_with("asha@example.org", "Welcome!")

    def test_skips_without_email(self):
        mailer = mock.Mock()
        self.assertFalse(notify({"name": "Ravi"}, mailer))
        mailer.send.assert_not_called()

# 5
def flip_coin():
    return "heads" if random.random() < 0.5 else "tails"

class TestCoin(unittest.TestCase):
    def test_heads_and_tails(self):
        with mock.patch("random.random", return_value=0.1):
            self.assertEqual(flip_coin(), "heads")
        with mock.patch("random.random", return_value=0.9):
            self.assertEqual(flip_coin(), "tails")

# 6
def fetch_with_retry(client, url, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return client.get(url)
        except ConnectionError:
            if attempt == attempts:
                raise

class TestRetry(unittest.TestCase):
    def test_succeeds_on_third_try(self):
        client = mock.Mock()
        client.get.side_effect = [ConnectionError, ConnectionError, "data"]
        self.assertEqual(fetch_with_retry(client, "/items"), "data")
        self.assertEqual(client.get.call_count, 3)

    def test_gives_up(self):
        client = mock.Mock()
        client.get.side_effect = ConnectionError("down")
        with self.assertRaises(ConnectionError):
            fetch_with_retry(client, "/items", attempts=2)
        self.assertEqual(client.get.call_count, 2)

# 7
def is_weekend(today=None):
    today = today or date.today()
    return today.weekday() >= 5               # 5 = Saturday, 6 = Sunday

class TestWeekend(unittest.TestCase):
    def test_days(self):
        self.assertTrue(is_weekend(date(2026, 10, 24)))    # a Saturday
        self.assertFalse(is_weekend(date(2026, 10, 26)))   # a Monday

result = run_tests(TestNotify, TestCoin, TestRetry, TestWeekend)
assert result.wasSuccessful() and result.testsRun == 6
print("Practice 9.2: all checks passed")

tests run: 6, OK (failures=0, errors=0)
Practice 9.2: all checks passed


## 9.5 doctest, test design and coverage

### doctest

A **doctest** is an example inside a docstring, written like an interactive session (`>>>` then the expected output). The `doctest` module runs these examples and checks the output. It keeps your documentation correct.

* From a terminal: `python -m doctest mymodule.py -v`, or `python -m pytest --doctest-modules`.
* Good for short, clear examples. Not a replacement for real tests (edge cases and errors become hard to read).

In [482]:
# Run the examples in a docstring with doctest
import doctest

def add(a, b):
    """Add two numbers or join two strings.

    >>> add(2, 3)
    5
    >>> add("data", "base")
    'database'
    """
    return a + b

runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(add, "add", globs={"add": add}):
    runner.run(test)
print(runner.summarize(verbose=False))     # TestResults(failed=0, attempted=2)

TestResults(failed=0, attempted=2)


### Designing good tests

* **Arrange, Act, Assert (AAA):** prepare the inputs, call the code once, check the result. Keep the three steps visible.
* **One behaviour per test**, with a name that says what it checks: `test_withdraw_more_than_balance_raises`.
* **Test the edges**, not only the normal case. Use this checklist:

| Input type | Edge cases to try |
|---|---|
| Numbers | 0, 1, negative, very large, float vs int |
| Strings | empty, one character, spaces only, Unicode, mixed case |
| Lists | empty, one element, duplicates, already sorted, reversed |
| Dicts / objects | missing keys, `None` values, wrong types |
| Errors | invalid input raises the right exception with a helpful message |

* **Independent and repeatable:** no shared state, no real clock, network or randomness (inject or seed them).
* **Test behaviour, not implementation:** check what a function returns or does, not which private helpers it calls.

In [483]:
# Arrange, Act, Assert in one small test
import unittest

def average(numbers):
    if not numbers:
        raise ValueError("average of an empty list")
    return sum(numbers) / len(numbers)

class TestAverage(unittest.TestCase):
    def test_average_of_three_numbers(self):
        numbers = [2, 4, 9]              # Arrange
        result = average(numbers)        # Act
        self.assertEqual(result, 5)      # Assert

    def test_single_element(self):
        self.assertEqual(average([7]), 7)

    def test_negative_numbers(self):
        self.assertEqual(average([-2, 2]), 0)

    def test_empty_list_raises(self):
        with self.assertRaisesRegex(ValueError, "empty"):
            average([])

result = run_tests(TestAverage)          # tests run: 4, OK (failures=0, errors=0)

tests run: 4, OK (failures=0, errors=0)


### Test-driven development (TDD) in one paragraph

In **TDD** you write a failing test first ("red"), then the simplest code that makes it pass ("green"), then clean the code while the tests stay green ("refactor"). Even if you do not follow TDD strictly, writing a test **before fixing a bug** is a great habit: the test proves the bug exists, and later proves it is fixed and stays fixed.

### Coverage

**Coverage** measures which lines (and branches) ran during your tests. The `coverage` package is third-party (`python -m pip install coverage`):

```text
python -m coverage run -m pytest          # run the tests while recording lines
python -m coverage report -m              # table with the missing line numbers
python -m coverage html                   # a browsable report in htmlcov/
```

> **Warning:** 100% line coverage proves only that every line ran. It does not prove the results were checked, or that every edge case was tried. Use coverage to find **untested** code, not as proof of quality.

**Property-based testing** (third-party `hypothesis`) is a further step: you describe the *kind* of input ("any list of integers") and a property that must always hold ("sorting twice gives the same result as sorting once"). The tool generates hundreds of inputs and shrinks a failure to the smallest example.

> **Key points**
> * doctest keeps docstring examples correct.
> * Arrange, Act, Assert; one behaviour per test; test edge and error cases.
> * Write a failing test before you fix a bug.
> * Coverage finds untested lines; it does not prove correctness.

---

## 9.6 Code quality: tools, readability and refactoring

### Tools that check your code automatically

| Tool | Kind | What it does | Command |
|---|---|---|---|
| `ruff` | Linter (and formatter) | Finds bugs and style problems; very fast | `ruff check .` |
| `black` | Formatter | Rewrites code into one consistent style | `black .` |
| `ruff format` | Formatter | Black-compatible formatting in ruff | `ruff format .` |
| `mypy` / `pyright` | Type checker | Checks type hints without running code | `mypy src/` |
| `pre-commit` | Hook runner | Runs the tools before every `git commit` | `pre-commit install` |

All of these are third-party (`python -m pip install ruff mypy`). A **linter** finds suspicious code (unused imports, undefined names, shadowed built-ins). A **formatter** changes layout only. A **type checker** reads type hints (which Python itself does not enforce) and finds type mistakes before you run the code.

What a type checker catches (shown as text, because Python itself runs this without complaint until the bad call):

```python
def total_price(prices: list[float]) -> float:
    return sum(prices)

total_price("12.5")      # mypy: Argument 1 to "total_price" has incompatible type "str"; expected "list[float]"
```

Typical project setup in `pyproject.toml` (see Notebook 1, Part 14):

```text
[tool.ruff]
line-length = 100

[tool.ruff.lint]
select = ["E", "F", "B", "I"]    # pycodestyle errors, pyflakes, bugbear, import sorting

[tool.mypy]
strict = true
```

### Readable code in a few rules

* **Names say what things are.** `remaining_stock`, not `rs`. Verbs for functions (`load_orders`), nouns for classes (`Order`).
* **Small functions, one job each.** If you need "and" to describe a function, split it.
* **Keep logic apart from I/O.** A pure calculation (no files, no printing, no network) is easy to test. Put reading and printing at the edges.
* **Shallow nesting.** Return early (guard clauses) instead of deep `if` blocks.
* **No magic numbers.** Give constants names: `VAT_RATE = 0.18`.
* **DRY (Don't Repeat Yourself)**, but do not abstract too early.
* **Docstrings and type hints** on public functions; comments explain *why*.

### Refactoring under test

**Refactoring** means changing the structure of code **without changing its behaviour**. It is only safe when tests (or a comparison) prove the behaviour is the same.

In [484]:
# Before: nested ifs, magic numbers, one long function
def total_before(order):
    t = 0
    for i in order["items"]:
        if i["qty"] > 0:
            if i["price"] > 0:
                t = t + i["qty"] * i["price"]
    if order["customer"] == "vip":
        t = t - t * 0.1
    else:
        if t > 1000:
            t = t - t * 0.05
    t = t + t * 0.18
    return round(t, 2)

print(total_before({"customer": "vip", "items": [{"qty": 2, "price": 500}]}))   # 1062.0

1062.0


In [485]:
# After: named constants and small single-purpose functions
VAT_RATE = 0.18
VIP_DISCOUNT = 0.10
BULK_THRESHOLD = 1000
BULK_DISCOUNT = 0.05

def subtotal(items):
    return sum(i["qty"] * i["price"] for i in items if i["qty"] > 0 and i["price"] > 0)

def discount_rate(customer, amount):
    if customer == "vip":
        return VIP_DISCOUNT
    return BULK_DISCOUNT if amount > BULK_THRESHOLD else 0.0

def total_after(order):
    amount = subtotal(order["items"])
    amount *= 1 - discount_rate(order["customer"], amount)
    return round(amount * (1 + VAT_RATE), 2)

# Prove that behaviour did not change, including edge cases
orders = [
    {"customer": "vip", "items": [{"qty": 2, "price": 500}, {"qty": 0, "price": 99}]},
    {"customer": "regular", "items": [{"qty": 3, "price": 400}]},
    {"customer": "regular", "items": [{"qty": 1, "price": 100}, {"qty": 1, "price": -5}]},
    {"customer": "regular", "items": []},
]
print([total_after(o) for o in orders])                        # [1062.0, 1345.2, 118.0, 0.0]
print(all(total_before(o) == total_after(o) for o in orders))  # True

[1062.0, 1345.2, 118.0, 0.0]
True


**Common mistakes**

* Refactoring and changing behaviour in the same step. Do one, test, then do the other.
* Refactoring without tests ("it looks the same").
* Ignoring linter warnings until there are hundreds. Fix them as you go, or let a pre-commit hook enforce them.

> **Key points**
> * Linters find problems, formatters fix layout, type checkers check hints.
> * Readable code: clear names, small functions, logic apart from I/O, named constants.
> * Refactor only under tests, and in small steps.

---

### Practice 9.3: doctest, test design and code quality

**A. Predict the output**

1. How many examples does doctest attempt here?
   ```python
   def square(x):
       """
       >>> square(3)
       9
       >>> square(-2)
       4
       """
       return x * x
   ```
2. A doctest expects `0.30000000000000004` but the docstring says `0.3`. Does it pass?

**B. Write code**

3. Write `fahrenheit(c)` with two doctest examples (`100` gives `212.0`, `-40` gives `-40.0`) and run them with `doctest`.
4. List at least six edge-case inputs you would test for `second_largest(numbers)`. Then implement it (it raises `ValueError` if there are fewer than two distinct values) and test them.
5. **[Hard]** Refactor the function below into clear code and prove that it behaves the same on many inputs.
   ```python
   def f(l):
       r = []
       for i in l:
           if i % 2 == 0:
               if i > 10:
                   r.append(i * 2)
               else:
                   r.append(i)
       return r
   ```

**C. Find and fix the bug**

6. Why does this doctest fail even though the function is correct?
   ```python
   def greet(name):
       """
       >>> greet("Asha")
       Hello, Asha
       """
       return f"Hello, {name}"
   ```
7. The team has 100% coverage but a bug slipped through: `discount(1000)` returns the wrong value. How can that be?

**D. Interview questions**

8. What is the difference between a linter, a formatter and a type checker?
9. What is refactoring, and why do you need tests before you start?
10. What does "Arrange, Act, Assert" mean?

### Solutions 9.3

1. Two examples.
2. No. doctest compares the printed text exactly. Write a stable example such as `round(0.1 + 0.2, 2)` giving `0.3`.
6. In the interactive prompt, a returned string is shown with quotes (its `repr`): `'Hello, Asha'`. The expected output must include the quotes.
7. Coverage only shows that lines ran. A test may have called `discount(500)` and run every line, but no test checked the boundary value 1000.
8. A linter finds suspicious code and style problems; a formatter rewrites layout only; a type checker checks that types match the type hints, without running the code.
9. Changing the structure of code without changing behaviour. Tests show that the behaviour really stayed the same.
10. Prepare the inputs (Arrange), call the code under test (Act), check the result (Assert).

In [486]:
# Solutions 9.3 (B) with checks
import doctest

# 3
def fahrenheit(c):
    """Convert Celsius to Fahrenheit.

    >>> fahrenheit(100)
    212.0
    >>> fahrenheit(-40)
    -40.0
    """
    return c * 9 / 5 + 32

runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(fahrenheit, "fahrenheit", globs={"fahrenheit": fahrenheit}):
    runner.run(test)
assert runner.summarize(verbose=False).failed == 0

# 4  edge cases: [1, 2], [2, 1], duplicates [5, 5, 3], negatives, all equal, one element, empty
def second_largest(numbers):
    distinct = sorted(set(numbers), reverse=True)
    if len(distinct) < 2:
        raise ValueError("need at least two distinct values")
    return distinct[1]

assert second_largest([1, 2]) == 1
assert second_largest([2, 1]) == 1
assert second_largest([5, 5, 3]) == 3
assert second_largest([-1, -7, -3]) == -3
for bad in ([], [4], [9, 9, 9]):
    try:
        second_largest(bad)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass

# 5
def f_old(l):
    r = []
    for i in l:
        if i % 2 == 0:
            if i > 10:
                r.append(i * 2)
            else:
                r.append(i)
    return r

def double_large_evens(numbers, threshold=10):
    """Keep even numbers; double the ones above the threshold."""
    return [n * 2 if n > threshold else n for n in numbers if n % 2 == 0]

for data in ([], [1, 3], [2, 4, 12, 14, 9], list(range(-20, 40)), [10, 11, 12]):
    assert double_large_evens(data) == f_old(data), data
print("Practice 9.3: all checks passed")

Practice 9.3: all checks passed


## 9.7 Secure coding basics

Security bugs often come from trusting input. **Never trust data from outside** (users, files, the network, URLs, environment): validate it, and never run it as code.

| Risk | Do not | Do instead |
|---|---|---|
| Code injection | `eval()` / `exec()` on input | `ast.literal_eval` for literals, `json.loads`, real parsers |
| SQL injection | Build SQL with f-strings or `+` | Parameterised queries (`?` placeholders) |
| Command injection | `subprocess.run(text, shell=True)` with user data | Argument lists, no shell |
| Unsafe loading | `pickle.loads` / `yaml.load` on untrusted data | `json`, `yaml.safe_load` |
| Path traversal | `open(base + user_name)` | Resolve the path and check it stays inside the base folder |
| Weak tokens | `random` for passwords or tokens | `secrets` |
| Password storage | Plain text, MD5, SHA-1 | Salted slow hash: `hashlib.scrypt`, `pbkdf2_hmac`, or bcrypt/argon2 libraries |
| Secrets in code | API keys in source or git | Environment variables or a secrets manager |
| Timing leaks | `==` on secrets | `hmac.compare_digest` |
| Security checks with `assert` | `assert user.is_admin` | `if not user.is_admin: raise PermissionError(...)` |

### SQL injection

If you paste user input into an SQL string, the input can change the query itself. With **parameterised queries**, the database receives the value separately, so it is always treated as data, never as SQL.

```text
  unsafe:  "SELECT * FROM users WHERE name = '" + name + "'"
           name = x' OR '1'='1
           -> SELECT * FROM users WHERE name = 'x' OR '1'='1'   (matches every row)

  safe:    execute("SELECT * FROM users WHERE name = ?", (name,))
           -> the whole text "x' OR '1'='1" is compared as a name (matches nothing)
```

In [487]:
# SQL injection with string building versus a parameterised query
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE users (name TEXT, role TEXT)")
con.executemany("INSERT INTO users VALUES (?, ?)", [("asha", "admin"), ("ravi", "user")])

attack = "x' OR '1'='1"                       # input typed by an attacker

unsafe_sql = f"SELECT name FROM users WHERE name = '{attack}'"      # Common mistake
print(con.execute(unsafe_sql).fetchall())     # [('asha',), ('ravi',)]  (every user leaked)

safe = con.execute("SELECT name FROM users WHERE name = ?", (attack,)).fetchall()
print(safe)                                   # []  (treated as a plain name)
con.close()

[('asha',), ('ravi',)]
[]


### Storing passwords

Never store passwords as plain text, and never with a fast hash like MD5 or SHA-256 alone (attackers can try billions per second). Use a **salted, slow** hash:

* A **salt** is random bytes stored next to the hash. Two users with the same password get different hashes.
* A **slow** hash (many iterations) makes guessing expensive. For PBKDF2-SHA256, current guidance (OWASP) is about 600,000 iterations.
* Compare hashes with `hmac.compare_digest`, which takes the same time whether the first or last byte differs (no timing leak).

In [488]:
# Salted password hashing with PBKDF2 and a constant-time comparison
import hashlib
import hmac
import os

ITERATIONS = 600_000

def hash_password(password, salt=None):
    salt = salt or os.urandom(16)                 # new random salt per user
    digest = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, ITERATIONS)
    return salt, digest

def verify_password(password, salt, expected_digest):
    _, digest = hash_password(password, salt)
    return hmac.compare_digest(digest, expected_digest)

salt, stored = hash_password("correct horse battery")
print(verify_password("correct horse battery", salt, stored))   # True
print(verify_password("wrong guess", salt, stored))             # False

_, other = hash_password("correct horse battery")               # same password, new salt
print(other == stored)                                           # False

True
False
False


### secrets, literal_eval and safe subprocess calls

* `secrets` makes tokens that attackers cannot predict. The `random` module is predictable and only for simulations and games.
* `ast.literal_eval` reads Python literals (numbers, strings, lists, dicts ...) and refuses anything else. Never use `eval` on input.
* `subprocess.run` with a **list** of arguments passes each item as one argument. No shell is involved, so `;` or `&&` in user input are just characters.

In [489]:
# secrets for tokens, literal_eval instead of eval
import ast
import secrets

token = secrets.token_hex(16)                    # 16 random bytes as 32 hex characters
print(len(token))                                # 32
print(len(secrets.token_urlsafe(16)) >= 16)      # True  (URL-safe text, e.g. reset links)
print(secrets.choice(["A", "B", "C"]) in "ABC")  # True

print(ast.literal_eval("[1, 2, {'a': 3}]"))      # [1, 2, {'a': 3}]
try:
    ast.literal_eval("__import__('os').getcwd()")
except ValueError as e:
    print("refused:", type(e).__name__)          # refused: ValueError

32
True
True
[1, 2, {'a': 3}]
refused: ValueError


In [490]:
# A list of arguments keeps user input as one argument (no shell)
import subprocess
import sys

user_file = "report.txt; del *.*"                # dangerous if passed to a shell
result = subprocess.run(
    [sys.executable, "-c", "import sys; print(sys.argv[1])", user_file],
    capture_output=True, text=True, check=True,
)
print(result.stdout.strip())                     # report.txt; del *.*  (just text)

report.txt; del *.*


In [491]:
# Path traversal: keep user-chosen file names inside a base folder
from pathlib import Path

BASE = Path("uploads").resolve()

def safe_path(user_name):
    candidate = (BASE / user_name).resolve()
    if not candidate.is_relative_to(BASE):       # Python 3.9+
        raise PermissionError(f"path escapes the upload folder: {user_name!r}")
    return candidate

print(safe_path("photo.png").name)               # photo.png
try:
    safe_path("../../secret.txt")
except PermissionError as e:
    print("PermissionError:", e)    # PermissionError: path escapes the upload folder: '../../secret.txt'

photo.png
PermissionError: path escapes the upload folder: '../../secret.txt'


In [492]:
# Read secrets from the environment, never from source code
import os

api_key = os.environ.get("NOTES_DEMO_API_KEY")   # set outside the program, e.g. in the shell
print(api_key is None)                           # True  (not set in this demo)
if api_key is None:
    print("No API key configured; running in offline mode")

True
No API key configured; running in offline mode


**More habits:** pin and update dependencies and scan them (`pip-audit`); never log passwords or tokens; use HTTPS and timeouts for network calls; give each service only the permissions it needs (least privilege); treat `pickle` files as executable code.

> **Key points**
> * Never run input as code: no `eval`, no SQL string building, no `shell=True` with user data.
> * Parameterised SQL, `literal_eval`, argument lists.
> * Passwords: salted slow hash and `hmac.compare_digest`; tokens: `secrets`.
> * Secrets live in the environment or a secrets manager, not in code.

---

### Practice 9.4: Secure coding

**A. Predict the output**

1. What does this print?
   ```python
   import ast
   print(ast.literal_eval("(1, 'two', [3])"))
   ```
2. What does the query return if `name` is `"x' OR '1'='1"`?
   ```python
   con.execute("SELECT * FROM users WHERE name = ?", (name,)).fetchall()
   ```

**B. Write code**

3. Write `find_user(con, name)` using a parameterised query. Show that `"x' OR '1'='1"` finds nothing and `"asha"` finds one row.
4. Write `make_reset_token()` that returns a URL-safe random token of at least 32 characters.
5. Write `hash_password` / `verify_password` yourself with `hashlib.scrypt` (use `n=2**14, r=8, p=1`) and a 16-byte random salt.
6. **[Hard]** Write `parse_config_value(text)` that returns a Python literal (int, float, list, dict, bool, None) or the original string if it is not a literal. It must never run code.

**C. Find and fix the bug**

7. What is wrong here?
   ```python
   def is_admin(user):
       assert user.role == "admin", "not allowed"
       return True
   ```
8. What is wrong here?
   ```python
   import random
   token = "".join(random.choice("abcdef0123456789") for _ in range(32))
   ```
9. What is wrong here?
   ```python
   subprocess.run(f"convert {filename} out.png", shell=True)
   ```

**D. Interview questions**

10. What is SQL injection and how do you prevent it?
11. Why is a salt needed, and why must password hashing be slow?
12. Why is `hmac.compare_digest` better than `==` for comparing tokens?

### Solutions 9.4

1. `(1, 'two', [3])`.
2. An empty list: the whole text is compared as one name.
7. `assert` statements are removed when Python runs with `-O`, so the check disappears. Use `if user.role != "admin": raise PermissionError("not allowed")`.
8. `random` is predictable (its internal state can be reconstructed). Use `secrets.token_hex(16)`.
9. With `shell=True`, a file name like `a.png; rm -rf ~` runs a second command. Use a list: `subprocess.run(["convert", filename, "out.png"], check=True)`.
10. An attacker's input changes the structure of an SQL statement because it was pasted into the string. Prevent it with parameterised queries (and least-privilege database users).
11. The salt makes equal passwords hash differently, which defeats precomputed tables and reveals nothing when two users share a password. Slowness makes every guess expensive for an attacker who stole the hashes.
12. `==` may stop at the first different byte, so response times can leak how many leading bytes were right. `compare_digest` takes the same time either way.

In [493]:
# Solutions 9.4 (B) with checks
import ast
import hashlib
import hmac
import os
import secrets
import sqlite3

# 3
def find_user(con, name):
    return con.execute("SELECT name, role FROM users WHERE name = ?", (name,)).fetchall()

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE users (name TEXT, role TEXT)")
con.execute("INSERT INTO users VALUES ('asha', 'admin')")
assert find_user(con, "asha") == [("asha", "admin")]
assert find_user(con, "x' OR '1'='1") == []
con.close()

# 4
def make_reset_token():
    return secrets.token_urlsafe(32)          # 32 bytes -> about 43 characters

assert len(make_reset_token()) >= 32
assert make_reset_token() != make_reset_token()

# 5
def hash_password(password, salt=None):
    salt = salt or os.urandom(16)
    digest = hashlib.scrypt(password.encode("utf-8"), salt=salt, n=2**14, r=8, p=1)
    return salt, digest

def verify_password(password, salt, expected):
    return hmac.compare_digest(hash_password(password, salt)[1], expected)

salt, digest = hash_password("s3cret!")
assert verify_password("s3cret!", salt, digest)
assert not verify_password("S3cret!", salt, digest)
assert hash_password("s3cret!")[1] != digest     # different salt, different hash

# 6
def parse_config_value(text):
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return text

assert parse_config_value("42") == 42
assert parse_config_value("[1, 2]") == [1, 2]
assert parse_config_value("True") is True
assert parse_config_value("None") is None
assert parse_config_value("hello world") == "hello world"
assert parse_config_value("__import__('os').getcwd()") == "__import__('os').getcwd()"
print("Practice 9.4: all checks passed")

Practice 9.4: all checks passed


## 9.8 Measuring and improving performance

Two rules:

1. **Measure before you optimise.** Programmers are bad at guessing where time goes.
2. **Improve the algorithm and data structure first** (see Notebook 3, Part 2). Changing O(n^2) to O(n log n) beats any micro-trick.

The standard tools:

| Tool | Question it answers |
|---|---|
| `timeit` | How fast is this small snippet? (repeats it many times) |
| `cProfile` + `pstats` | Which functions use the most time in a whole program? |
| `tracemalloc` | How much memory is allocated, and where? |
| `time.perf_counter()` | How long did this block take? (quick check) |

> **Note:** Timings depend on the machine and on other running programs, so the cells below print comparisons (True/False) instead of exact times. Run them yourself to see your numbers.

In [494]:
# timeit: compare two ways to do the same thing
import timeit

setup = "data = list(range(10_000)); target = 9_999; data_set = set(data)"
list_time = min(timeit.repeat("target in data", setup=setup, number=200, repeat=3))
set_time = min(timeit.repeat("target in data_set", setup=setup, number=200, repeat=3))

print("set lookup is faster:", set_time < list_time)     # set lookup is faster: True
print("at least 10x faster:", list_time > 10 * set_time)  # at least 10x faster: True

set lookup is faster: True
at least 10x faster: True


From a terminal you can time a snippet without writing code: `python -m timeit -s "data = list(range(1000))" "sum(data)"`. In Jupyter, `%timeit sum(data)` does the same.

**cProfile** runs a whole program and records every function call. Sort by `tottime` (time inside the function itself) or `cumulative` (including the functions it calls). From a terminal: `python -m cProfile -s cumulative my_script.py`.

In [495]:
# cProfile: find the function that uses the most time
import cProfile
import pstats

def slow_part():
    return sum(len(str(i)) for i in range(150_000))   # the heavy work

def fast_part():
    return sum(range(1_000))

def program():
    return slow_part() + fast_part()

profiler = cProfile.Profile()
profiler.enable()
program()
profiler.disable()

stats = pstats.Stats(profiler)
# stats.sort_stats("cumulative").print_stats(5)  prints the full table (times vary per run)
cumulative = {func[2]: info[3] for func, info in stats.stats.items()}   # name -> cumulative time
ours = {name: t for name, t in cumulative.items() if name in {"slow_part", "fast_part"}}
print(max(ours, key=ours.get))                 # slow_part

slow_part


In [496]:
# tracemalloc: a list holds everything, a generator holds one item at a time
import tracemalloc

def peak_bytes(func):
    tracemalloc.start()
    func()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak

list_peak = peak_bytes(lambda: sum([i * i for i in range(200_000)]))
gen_peak = peak_bytes(lambda: sum(i * i for i in range(200_000)))

print("list needs more memory:", list_peak > 10 * gen_peak)   # list needs more memory: True

list needs more memory: True


**Performance tips (after measuring):**

| Slow pattern | Faster pattern | Why |
|---|---|---|
| `x in some_list` inside a loop | Build a `set` once, then `x in some_set` | O(1) average lookup instead of O(n) |
| `list.pop(0)` / `insert(0, x)` | `collections.deque` | O(1) at both ends |
| `s += part` in a big loop | `"".join(parts)` | One allocation |
| Recomputing the same result | `functools.cache` / a dict | Do the work once |
| Python loops over numbers | Built-ins (`sum`, `max`, `sorted`) or NumPy | Loops run in C |
| Reading a whole huge file | Iterate line by line / generators | Constant memory |
| Sorting to get the top k | `heapq.nlargest(k, data)` | O(n log k) |
| Many tiny network calls | Batch them; threads or asyncio (Part 8) | Overlap the waiting |

In [497]:
# deque versus list for removing from the front
import timeit

list_time = timeit.timeit("while q: q.pop(0)", setup="q = list(range(20_000))", number=1)
deque_time = timeit.timeit(
    "while q: q.popleft()",
    setup="from collections import deque; q = deque(range(20_000))",
    number=1,
)
print("deque is faster:", deque_time < list_time)     # deque is faster: True

deque is faster: True


**Common mistakes**

* Optimising code that is not the bottleneck. Profile first.
* Measuring once. Use `timeit.repeat` and take the minimum; close other programs.
* Making code unreadable for a tiny gain. Readability matters more unless the code is really hot.
* Forgetting the algorithm: no micro-optimisation saves an O(n^2) loop on a million items.

> **Key points**
> * Measure first: `timeit` for snippets, `cProfile` for programs, `tracemalloc` for memory.
> * Fix the algorithm and data structure before micro-optimising.
> * Use sets for membership, deque for queues, join for strings, generators for big data.

---

## 9.9 How to debug failing tests

A failing test is good news: it found a problem before your users did. Use this loop every time:

```text
   +---------------------------+
   | Run the tests             |<-------------------------------+
   +-------------+-------------+                                |
                 |                                              |
                 v                                              |
   +---------------------------+   All pass   +---------------+ |
   | Any test failing?         |------------->| Commit / done | |
   +-------------+-------------+              +---------------+ |
                 | Yes                                          |
                 v                                              |
   +---------------------------+                                |
   | Read the report: which    |                                |
   | test, expected vs actual  |                                |
   +-------------+-------------+                                |
                 |                                              |
                 v                                              |
   +---------------------------+   test is    +---------------+ |
   | Is the TEST wrong or the  |------------->| Fix the test  |-+
   | CODE wrong?               |   wrong      +---------------+ |
   +-------------+-------------+                                |
                 | code is wrong                                |
                 v                                              |
   +---------------------------+                                |
   | Reproduce with only this  |                                |
   | test; inspect values      |                                |
   | (print, f"{x=}", pdb)     |                                |
   +-------------+-------------+                                |
                 |                                              |
                 v                                              |
   +---------------------------+                                |
   | Fix the smallest thing,   |--------------------------------+
   | keep the test as guard    |
   +---------------------------+
```

**Useful commands while debugging tests:**

| Goal | unittest | pytest |
|---|---|---|
| Run one test | `python -m unittest tests.test_cart.TestCart.test_total` | `python -m pytest tests/test_cart.py::test_total` |
| Stop at the first failure | `python -m unittest -f` | `python -m pytest -x` |
| Show more detail | `python -m unittest -v` | `python -m pytest -vv` |
| Open the debugger at the failure | `breakpoint()` in the test | `python -m pytest --pdb` |
| Show print output | printed by default | `python -m pytest -s` |

**Typical test problems and their messages:**

| Message | Meaning | Fix |
|---|---|---|
| `AssertionError: 'a-b' != 'a_b'` | The code gave a different value | Decide which side is right; fix code or test |
| `AssertionError: ValueError not raised` | The code did not raise the expected error | Add the validation, or fix the test input |
| `ModuleNotFoundError: No module named 'shop'` | Tests cannot import your code | Run from the project root; use a `src/` layout and `pip install -e .` |
| `AttributeError: Mock object has no attribute ...` | A `spec`/autospec mock refused an unknown name | Use the real method name |
| Test passes alone, fails in the full run | Shared state between tests | Create state in `setUp` / fixtures |
| Test fails sometimes (flaky) | Time, randomness, order or threads | Inject or seed them; add timeouts |

In [498]:
# Debugging a failing test: print the values the assertion compared
import unittest

def word_count(text):
    return len(text.split(" "))          # bug: double spaces create empty words

class TestWordCount(unittest.TestCase):
    def test_double_space(self):
        self.assertEqual(word_count("data  science"), 2)

result = run_tests(TestWordCount)        # tests run: 1, FAILED (failures=1, errors=0)
print(result.failures[0][1].strip().splitlines()[-1])   # AssertionError: 3 != 2

text = "data  science"
print(f"{text.split(' ')=}")             # text.split(' ')=['data', '', 'science']  <- the empty word

def word_count(text):                    # fix: split() with no argument skips runs of spaces
    return len(text.split())

result = run_tests(TestWordCount)        # tests run: 1, OK (failures=0, errors=0)

tests run: 1, FAILED (failures=1, errors=0)
AssertionError: 3 != 2
text.split(' ')=['data', '', 'science']
tests run: 1, OK (failures=0, errors=0)


> **Tip:** When you fix a bug, first write a test that fails because of the bug. Then fix the code. The test stays in the suite as a guard against the bug coming back.

> **Key points**
> * Read the failure: test name, expected versus actual.
> * Decide whether the test or the code is wrong.
> * Reproduce with one test, inspect values, fix, and re-run everything.

---

### Practice 9.5: Performance and debugging tests

**A. Predict the output**

1. Which is usually faster for 10,000 lookups in a collection of 10,000 numbers: a `list` or a `set`? Why?
2. What does this print?
   ```python
   print("a  b".split(" "), "a  b".split())
   ```
3. In `cProfile` output, what is the difference between `tottime` and `cumtime`?

**B. Write code**

4. Write `common_items(a, b)` that returns the items of `a` that also appear in `b`, keeping the order of `a`. It must be O(len(a) + len(b)).
5. Write `measure(func, repeat=3)` that returns the best of `repeat` runs in seconds using `time.perf_counter`. Use it to check that `sorted` on 50,000 numbers takes less than 1 second.
6. Write `peak_kib(func)` using `tracemalloc`, and show that building `list(range(100_000))` uses more memory than `sum(range(100_000))`.
7. **[Hard]** `fib(30)` written naively is slow. Use `cProfile` to count how many times `fib` is called (`stats.stats` has the call count), then fix it with `functools.cache` and show the call count drops to 31.

**C. Find and fix the bug**

8. This test fails with `AssertionError: [3, 1, 2] != [1, 2, 3]`. Is the code or the test wrong?
   ```python
   def unique(items):
       return list(set(items))

   def test_unique():
       assert unique([3, 1, 3, 2]) == [1, 2, 3]
   ```
9. `ModuleNotFoundError: No module named 'shop'` appears when you run `python -m pytest` from inside the `tests/` folder. What should you do?

**D. Interview questions**

10. Your API endpoint is slow. Describe the steps you take before changing any code.
11. Why do you take the minimum of several timing runs instead of the average?
12. What is a flaky test? Give two causes.

### Solutions 9.5

1. The `set`: membership in a set is O(1) on average (hashing); in a list it is O(n) (it scans).
2. `['a', '', 'b'] ['a', 'b']`. `split(" ")` splits on every single space; `split()` treats runs of whitespace as one separator.
3. `tottime` is time spent inside the function itself; `cumtime` also includes the time of every function it calls.
8. It depends on the contract. Sets have no order, so `unique` does not keep any order. If the function should keep the first-seen order, fix the code (`list(dict.fromkeys(items))`). If order does not matter, fix the test (`sorted(unique(...)) == [1, 2, 3]`).
9. Run tests from the project root, and make the package importable: use a `src/` layout and `python -m pip install -e .`, or configure `pythonpath` in pytest settings.
10. Reproduce and measure (timing, logs); profile to find the hot spot (database, network, CPU); check the algorithm and number of queries; fix the biggest cost; measure again and add a performance check.
11. Noise (other programs, CPU frequency, garbage collection) only ever makes a run slower. The minimum is the closest to the true cost of the code.
12. A test that sometimes passes and sometimes fails without code changes. Causes: real time or dates, randomness without a seed, order dependence between tests, network calls, thread timing.

In [499]:
# Solutions 9.5 (B) with checks
import cProfile
import functools
import pstats
import random
import time
import tracemalloc

# 4
def common_items(a, b):
    b_set = set(b)                       # O(len(b)) once
    return [x for x in a if x in b_set]  # O(len(a))

assert common_items([3, 1, 2, 3], [3, 2]) == [3, 2, 3]
assert common_items([], [1]) == []
assert common_items([1, 2], []) == []

# 5
def measure(func, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        func()
        best = min(best, time.perf_counter() - start)
    return best

random.seed(7)
numbers = [random.random() for _ in range(50_000)]
assert measure(lambda: sorted(numbers)) < 1.0

# 6
def peak_kib(func):
    tracemalloc.start()
    func()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024

assert peak_kib(lambda: list(range(100_000))) > peak_kib(lambda: sum(range(100_000)))

# 7
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

def call_count(func, arg, name):
    profiler = cProfile.Profile()
    profiler.enable()
    func(arg)
    profiler.disable()
    stats = pstats.Stats(profiler)
    return sum(info[1] for key, info in stats.stats.items() if key[2] == name)   # info[1] = total calls

slow_calls = call_count(fib, 22, "fib")          # 22 keeps the demo fast; 30 would be over a million calls

@functools.cache
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

fast_calls = call_count(fib_cached, 30, "fib_cached")
assert slow_calls == 57313                       # 2 * fib(23) - 1 calls for n = 22
assert fast_calls == 31                          # each n from 0 to 30 computed once
print("Practice 9.5: all checks passed")

Practice 9.5: all checks passed


### Summary of Part 9

* Tests let you change code with confidence; keep them fast, independent and repeatable.
* `unittest`: `TestCase`, `test_` methods, specific assertions, `setUp`, `subTest`, `assertRaises`.
* `pytest`: plain `assert`, fixtures, parametrize, `pytest.raises`; run with `python -m pytest`.
* Mocks replace slow or unpredictable dependencies; prefer injection; patch where the name is looked up; use autospec.
* doctest keeps examples correct; coverage finds untested lines but does not prove correctness.
* Linters, formatters and type checkers catch problems early; refactor only under tests.
* Security: parameterised SQL, no `eval`/`pickle`/`shell=True` on untrusted data, salted slow password hashes, `secrets`, `compare_digest`, secrets in the environment.
* Performance: measure first (`timeit`, `cProfile`, `tracemalloc`), fix the algorithm, then use the right data structure.
* Debug failing tests with the loop: read, decide test or code, reproduce, inspect, fix, re-run.

---

# Part 10: Real-World Mini Projects

**What these notes cover**

* How the pieces of this notebook fit together in real programs.
* Project A: a log-analysis command-line tool (generator pipeline, `argparse`, `logging`, tests).
* Project B: a small ETL pipeline (dataclasses, custom exceptions, a retry decorator, a context manager).
* Project C: an inventory manager (classes, custom exceptions, JSON persistence, tests).
* How to turn a script into a proper project, and how to debug a whole program.

**Before you start:** Notebook 1 Parts 12-14 (errors, files, modules) and in this notebook Parts 1-9. Each project names the Parts it uses.

## 10.1 How to build a small project

Real projects are not written top to bottom in one go. Work in small steps and test each step:

```text
   +-----------------------------+
   | Understand the goal:        |
   | inputs, outputs, examples   |
   +--------------+--------------+
                  |
                  v
   +-----------------------------+
   | Split into small pieces:    |
   | read -> parse -> compute -> |
   | output                      |
   +--------------+--------------+
                  |
                  v
   +-----------------------------+
   | Write ONE piece as a pure   |<-------------+
   | function where possible     |              |
   +--------------+--------------+              |
                  |                             |
                  v                             |
   +-----------------------------+   No         |
   | Test it (normal, edge,      |----> fix ----+
   | error cases). Pass?         |
   +--------------+--------------+
                  | Yes
                  v
   +-----------------------------+   more pieces
   | Connect the pieces; add the |----------------> (back to "Write ONE piece")
   | CLI or entry point last     |
   +-----------------------------+
```

The three projects below follow the same design ideas:

* **Separation of concerns:** reading, parsing, computing and printing live in different functions.
* **Pure core, thin edges:** the logic does not print or read files, so it is easy to test.
* **Errors are part of the design:** bad input is reported with a clear message, not a crash.
* **Tests next to the code.**

We run the tests with a small helper (the same idea as in Part 9):

In [500]:
# Helper to run unittest classes in the notebook and print a one-line summary
import io
import unittest

def run_tests(*test_classes):
    suite = unittest.TestSuite()
    for cls in test_classes:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
    status = "OK" if result.wasSuccessful() else "FAILED"
    print(f"tests run: {result.testsRun}, {status} "
          f"(failures={len(result.failures)}, errors={len(result.errors)})")
    for test, report in result.failures + result.errors:      # show problems if any
        print(test.id(), report.strip().splitlines()[-1])
    return result

print(callable(run_tests))     # True

True


> **Key points**
> * Build a project piece by piece, testing each piece.
> * Keep logic pure; keep files, printing and command-line handling at the edges.

---

## 10.2 Project A: a log-analysis command-line tool

**Goal:** a command `loganalyzer app.log --min-level INFO --top 3 --json` that counts lines per log level and lists the sources with the most errors.

**Uses:** generators (Part 3), regular expressions, `Counter`, `argparse` and `logging` (Part 7), custom exceptions (Notebook 1, Part 12), modules and `__main__` (Notebook 1, Part 14), tests (Part 9).

**Design:**

```text
  log file --> read_records() --> parse_line() --> summarize() --> main() prints
               (generator,        (regex,          (pure logic,     (argparse,
                one line at a     raises           Counter)         text or JSON)
                time)             LogFormatError)
```

* `read_records` is a **generator**, so even a 10 GB log never sits in memory.
* Bad lines are **logged and skipped**, not fatal.
* `main(argv)` takes the argument list, so tests can call it without a real terminal. It returns an exit code (0 = success, 2 = bad input).

The module is written into a temp folder, so the notebook stays self-contained.

In [501]:
# Write the loganalyzer module into a temporary project folder
import pathlib
import tempfile
import textwrap

proj_dir = pathlib.Path(tempfile.mkdtemp(prefix="nb2_loganalyzer_"))

(proj_dir / "loganalyzer.py").write_text(textwrap.dedent(r'''
    """Summarise application log files."""
    import argparse
    import json
    import logging
    import re
    import sys
    from collections import Counter
    from pathlib import Path

    LOG_RE = re.compile(
        r"^(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) "
        r"(?P<level>[A-Z]+) (?P<source>[\w.]+): (?P<message>.*)$"
    )
    LEVELS = ("DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL")
    log = logging.getLogger(__name__)


    class LogFormatError(ValueError):
        """A line does not look like a log record."""


    def parse_line(line):
        match = LOG_RE.match(line.rstrip("\n"))
        if not match:
            raise LogFormatError(f"malformed line: {line.strip()!r}")
        return match.groupdict()


    def read_records(path):
        """Yield parsed records one by one; bad lines are logged and skipped."""
        with open(path, encoding="utf-8") as f:
            for number, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                try:
                    yield parse_line(line)
                except LogFormatError as exc:
                    log.warning("line %d: %s", number, exc)


    def summarize(records, min_level="DEBUG", top=3):
        threshold = LEVELS.index(min_level)
        by_level, error_sources = Counter(), Counter()
        for rec in records:
            if rec["level"] not in LEVELS:
                log.warning("unknown level %r ignored", rec["level"])
                continue
            if LEVELS.index(rec["level"]) < threshold:
                continue
            by_level[rec["level"]] += 1
            if rec["level"] in ("ERROR", "CRITICAL"):
                error_sources[rec["source"]] += 1
        return {"by_level": dict(by_level),
                "top_error_sources": error_sources.most_common(top)}


    def build_parser():
        parser = argparse.ArgumentParser(prog="loganalyzer", description=__doc__)
        parser.add_argument("path", type=Path)
        parser.add_argument("--min-level", choices=LEVELS, default="DEBUG")
        parser.add_argument("--top", type=int, default=3)
        parser.add_argument("--json", action="store_true", dest="as_json")
        return parser


    def main(argv=None):
        args = build_parser().parse_args(argv)
        logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")
        if not args.path.is_file():
            print(f"error: {args.path} is not a file", file=sys.stderr)
            return 2
        result = summarize(read_records(args.path), args.min_level, args.top)
        if args.as_json:
            print(json.dumps(result, indent=2))
        else:
            for level in LEVELS:
                if level in result["by_level"]:
                    print(f"{level:<9}{result['by_level'][level]:>4}")
            for source, n in result["top_error_sources"]:
                print(f"  error source {source}: {n}")
        return 0


    if __name__ == "__main__":
        sys.exit(main())
''').lstrip(), encoding="utf-8")

print((proj_dir / "loganalyzer.py").exists())    # True

True


In [502]:
# A small sample log file with one bad line and one unknown level
sample_log = proj_dir / "app.log"
sample_log.write_text(
    "2026-10-23 09:00:01 INFO app.start: booting\n"
    "2026-10-23 09:00:02 DEBUG db: connecting\n"
    "this line is garbage\n"
    "2026-10-23 09:00:03 ERROR payments: card declined\n"
    "2026-10-23 09:00:04 ERROR payments: timeout\n"
    "2026-10-23 09:00:05 WARNING db: slow query\n"
    "2026-10-23 09:00:06 ERROR search: index missing\n"
    "2026-10-23 09:00:07 FATAL core: unknown level\n",
    encoding="utf-8",
)
print(len(sample_log.read_text(encoding="utf-8").splitlines()))    # 8

8


Now we run the tool exactly like a user would, as a separate program, with `subprocess.run` and an argument **list** (Part 7 and Part 9, 9.7):

In [503]:
# Run the command-line tool and look at stdout, stderr and the exit code
import subprocess
import sys

def run_cli(*args):
    return subprocess.run(
        [sys.executable, str(proj_dir / "loganalyzer.py"), *args],
        capture_output=True, text=True,
    )

r = run_cli(str(sample_log), "--min-level", "INFO")
print("exit code:", r.returncode)        # exit code: 0
print(r.stdout.rstrip())
# INFO        1
# WARNING     1
# ERROR       3
#   error source payments: 2
#   error source search: 1
print(r.stderr.rstrip())
# WARNING line 3: malformed line: 'this line is garbage'
# WARNING unknown level 'FATAL' ignored

exit code: 0
INFO        1
WARNING     1
ERROR       3
  error source payments: 2
  error source search: 1
WARNING line 3: malformed line: 'this line is garbage'
WARNING unknown level 'FATAL' ignored


In [504]:
# JSON output, and the error path for a missing file
r = run_cli(str(sample_log), "--json", "--top", "1")
print(r.stdout.rstrip())
# {
#   "by_level": {
#     "INFO": 1,
#     "DEBUG": 1,
#     "ERROR": 3,
#     "WARNING": 1
#   },
#   "top_error_sources": [
#     [
#       "payments",
#       2
#     ]
#   ]
# }

missing = run_cli(str(proj_dir / "nope.log"))
print(missing.returncode)                         # 2
print(missing.stderr.strip().startswith("error:"))   # True

{
  "by_level": {
    "INFO": 1,
    "DEBUG": 1,
    "ERROR": 3,
    "WARNING": 1
  },
  "top_error_sources": [
    [
      "payments",
      2
    ]
  ]
}
2
True


**Testing Project A.** The tests import the module and call its functions directly. They cover a valid line, a bad line, the level filter, the JSON output of `main`, and the missing-file error path. `contextlib.redirect_stdout` captures what `main` prints.

In [505]:
# Unit tests for the log analyzer
import contextlib
import io
import json
import sys
import unittest

sys.path.insert(0, str(proj_dir))
import loganalyzer as la

class TestLogAnalyzer(unittest.TestCase):
    def test_parse_valid_line(self):
        rec = la.parse_line("2026-10-23 09:00:03 ERROR payments: card declined")
        self.assertEqual((rec["level"], rec["source"], rec["message"]),
                         ("ERROR", "payments", "card declined"))

    def test_parse_invalid_line_raises(self):
        with self.assertRaises(la.LogFormatError):
            la.parse_line("garbage")

    def test_summarize_filters_by_level(self):
        records = [{"level": "DEBUG", "source": "a"},
                   {"level": "ERROR", "source": "a"},
                   {"level": "ERROR", "source": "b"}]
        result = la.summarize(records, min_level="INFO", top=1)
        self.assertEqual(result["by_level"], {"ERROR": 2})
        self.assertEqual(result["top_error_sources"], [("a", 1)])

    def test_summarize_empty_input(self):
        self.assertEqual(la.summarize([]), {"by_level": {}, "top_error_sources": []})

    def test_main_json_output(self):
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(io.StringIO()):
            status = la.main([str(sample_log), "--json", "--min-level", "ERROR"])
        self.assertEqual(status, 0)
        self.assertEqual(json.loads(buf.getvalue())["by_level"], {"ERROR": 3})

    def test_main_missing_file_returns_2(self):
        with contextlib.redirect_stderr(io.StringIO()):
            self.assertEqual(la.main([str(proj_dir / "missing.log")]), 2)

result = run_tests(TestLogAnalyzer)      # tests run: 6, OK (failures=0, errors=0)

tests run: 6, OK (failures=0, errors=0)


> **Note:** The project folder and the imported `loganalyzer` module stay available until the end of Practice 10.4, where they are removed.

> **Key points**
> * A generator pipeline keeps memory use constant for any file size.
> * `main(argv)` returning an exit code makes the CLI easy to test.
> * Bad input is logged and skipped; a missing file gives exit code 2 and a message on stderr.

---

### Practice 10.1: Project A

**A. Predict the output**

1. Using the sample log above, what is `la.summarize(la.read_records(sample_log), "ERROR")["by_level"]`?
2. What exit code does `la.main(["missing.log"])` return?

**B. Write code**

3. Write `records_since(records, since)` (a generator) that keeps only records whose `time` is at or after `since` (format `HH:MM:SS`). Use it with `summarize` on the sample log for `"09:00:05"`.
4. Write `messages_for(records, source)` that returns the list of messages from one source, in file order.
5. **[Hard]** Write `busiest_minute(records)` that returns the `"HH:MM"` minute with the most records (ties: the earliest minute).

**C. Find and fix the bug**

6. A teammate changed `read_records` to `return [parse_line(l) for l in open(path)]`. Name two problems.

**D. Interview questions**

7. Why is `read_records` a generator and not a function returning a list?
8. Why does `main` take `argv` as a parameter and return an exit code instead of calling `sys.exit` inside?

### Solutions 10.1

1. `{'ERROR': 3}`. Only ERROR and CRITICAL pass the filter, and there are 3 ERROR lines (the FATAL line is ignored as unknown).
2. `2`.
6. (a) The file is never closed (no `with`). (b) One malformed line raises `LogFormatError` and stops everything instead of being skipped. (c) The whole file is loaded into memory.
7. A generator reads one line at a time, so memory stays small for any file size, and `summarize` can start working immediately.
8. Tests can pass arguments directly and check the returned code without the process exiting. Only the `if __name__ == "__main__":` block turns the code into `sys.exit(main())`.

In [506]:
# Solutions 10.1 (B) with checks
import contextlib
import io
from collections import Counter

# 3
def records_since(records, since):
    return (r for r in records if r["time"] >= since)   # "HH:MM:SS" strings compare correctly

with contextlib.redirect_stderr(io.StringIO()):          # hide the warnings for bad lines
    filtered = la.summarize(records_since(la.read_records(sample_log), "09:00:05"))
assert filtered["by_level"] == {"WARNING": 1, "ERROR": 1}
assert filtered["top_error_sources"] == [("search", 1)]

# 4
def messages_for(records, source):
    return [r["message"] for r in records if r["source"] == source]

with contextlib.redirect_stderr(io.StringIO()):
    assert messages_for(la.read_records(sample_log), "payments") == ["card declined", "timeout"]
assert messages_for([], "db") == []

# 5
def busiest_minute(records):
    counts = Counter(r["time"][:5] for r in records)
    if not counts:
        return None
    return min(counts, key=lambda minute: (-counts[minute], minute))

recs = [{"time": "09:01:10"}, {"time": "09:00:59"}, {"time": "09:01:20"}, {"time": "09:00:01"}]
assert busiest_minute(recs) == "09:00"          # tie 2-2: earliest minute wins
assert busiest_minute(recs + [{"time": "09:01:30"}]) == "09:01"
assert busiest_minute([]) is None
print("Practice 10.1: all checks passed")

Practice 10.1: all checks passed


## 10.3 Project B: a small ETL pipeline

**ETL** means **Extract, Transform, Load**: read raw data, clean and convert it, and store the result. It is the daily work of data engineers.

**Goal:** read raw CSV sales rows, turn each valid row into a typed `Sale` object, add up revenue per region, and write a JSON report. Bad rows are **reported, not fatal**. The load step is flaky (like a network share), so it is **retried**.

**Uses:** generators (Part 3), a retry decorator (Part 4), a timing context manager (Part 5), dataclasses (Part 2), custom exceptions with `raise ... from` (Notebook 1, Part 12), `csv`, `json`, `defaultdict` (Part 7).

```text
  raw CSV --> extract() --> transform() --+--> valid Sale --> totals per region --> load() --> JSON
              (generator)   (validate,    |                                        (@retry)
                             convert)     +--> RowError --> rejected_rows list
```

In [507]:
# The data model and the extract/transform steps
import csv
import io
from dataclasses import dataclass

@dataclass(frozen=True)
class Sale:
    region: str
    units: int
    unit_price: float

    @property
    def revenue(self):
        return self.units * self.unit_price

class RowError(ValueError):
    """A CSV row failed validation."""

def extract(stream):
    """Yield (line_number, row_dict) one row at a time."""
    for number, row in enumerate(csv.DictReader(stream), start=2):   # line 1 is the header
        yield number, row

def transform(number, row):
    try:
        sale = Sale(row["region"].strip(), int(row["units"]), float(row["unit_price"]))
    except (KeyError, ValueError, AttributeError) as exc:
        raise RowError(f"line {number}: {type(exc).__name__}: {exc}") from exc
    if not sale.region or sale.units < 0 or sale.unit_price < 0:
        raise RowError(f"line {number}: invalid values {sale}")
    return sale

print(transform(2, {"region": " North ", "units": "10", "unit_price": "2.5"}))
# Sale(region='North', units=10, unit_price=2.5)
try:
    transform(5, {"region": "West", "units": "abc", "unit_price": "1"})
except RowError as e:
    print(e)    # line 5: ValueError: invalid literal for int() with base 10: 'abc'

Sale(region='North', units=10, unit_price=2.5)
line 5: ValueError: invalid literal for int() with base 10: 'abc'


In [508]:
# Reusable tools: a retry decorator and a stopwatch context manager
import functools
import time
from contextlib import contextmanager

def retry(times=3, delay=0.0, exceptions=(Exception,)):
    """Call the function again when it raises one of `exceptions`."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise                       # give up: show the real error
                    time.sleep(delay)
        return wrapper
    return decorator

@contextmanager
def stopwatch(label, sink):
    """Store the time taken by the with-block in sink[label]."""
    start = time.perf_counter()
    try:
        yield
    finally:
        sink[label] = time.perf_counter() - start

timings = {}
with stopwatch("demo", timings):
    sum(range(1000))
print(sorted(timings))          # ['demo']

['demo']


In [509]:
# The pipeline: transform every row, collect errors, load with retry
import io
import json
from collections import defaultdict

attempts = {"n": 0}

@retry(times=3, exceptions=(OSError,))
def load(report, out):
    attempts["n"] += 1
    if attempts["n"] < 3:                           # simulate two temporary failures
        raise OSError("target temporarily unavailable")
    json.dump(report, out, indent=2, sort_keys=True)

def run_pipeline(raw_csv):
    timings, totals, errors = {}, defaultdict(float), []
    with stopwatch("transform", timings):
        for number, row in extract(io.StringIO(raw_csv)):
            try:
                sale = transform(number, row)
            except RowError as exc:
                errors.append(str(exc))             # collect, do not crash
                continue
            totals[sale.region] += sale.revenue
    report = {"revenue_by_region": dict(totals), "rejected_rows": errors}
    out = io.StringIO()                             # stands in for a real file
    with stopwatch("load", timings):
        load(report, out)
    return json.loads(out.getvalue()), timings

raw = """region,units,unit_price
North,10,2.5
South,4,10
North,3,2.5
West,abc,1
,5,5
East,-1,3
South,1,0.5
"""
report, timings = run_pipeline(raw)
print(report["revenue_by_region"])       # {'North': 32.5, 'South': 40.5}
for line in report["rejected_rows"]:
    print(line)
# line 5: ValueError: invalid literal for int() with base 10: 'abc'
# line 6: invalid values Sale(region='', units=5, unit_price=5.0)
# line 7: invalid values Sale(region='East', units=-1, unit_price=3.0)
print("load attempts:", attempts["n"], "| timed steps:", sorted(timings))
# load attempts: 3 | timed steps: ['load', 'transform']

{'North': 32.5, 'South': 40.5}
line 5: ValueError: invalid literal for int() with base 10: 'abc'
line 6: invalid values Sale(region='', units=5, unit_price=5.0)
line 7: invalid values Sale(region='East', units=-1, unit_price=3.0)
load attempts: 3 | timed steps: ['load', 'transform']


In [510]:
# Tests for the ETL pieces
import unittest

class TestEtl(unittest.TestCase):
    def test_transform_valid_row(self):
        sale = transform(2, {"region": "North", "units": "2", "unit_price": "1.5"})
        self.assertEqual(sale.revenue, 3.0)

    def test_transform_rejects_negative_units(self):
        with self.assertRaisesRegex(RowError, "line 9"):
            transform(9, {"region": "East", "units": "-1", "unit_price": "3"})

    def test_transform_missing_column(self):
        with self.assertRaises(RowError) as ctx:
            transform(3, {"region": "North"})
        self.assertIsInstance(ctx.exception.__cause__, KeyError)   # raise ... from keeps the cause

    def test_retry_gives_up_after_three_attempts(self):
        calls = []

        @retry(times=3, exceptions=(OSError,))
        def always_fails():
            calls.append(1)
            raise OSError("down")

        with self.assertRaises(OSError):
            always_fails()
        self.assertEqual(len(calls), 3)

    def test_empty_input(self):
        attempts["n"] = 2                       # the next load attempt succeeds at once
        report, _ = run_pipeline("region,units,unit_price\n")
        self.assertEqual(report, {"revenue_by_region": {}, "rejected_rows": []})

result = run_tests(TestEtl)      # tests run: 5, OK (failures=0, errors=0)

tests run: 5, OK (failures=0, errors=0)


> **Key points**
> * ETL: extract lazily, transform with validation, load safely.
> * A custom exception (`RowError`) with `raise ... from` keeps the original cause.
> * Collect bad rows instead of crashing; retry only errors that may be temporary (`OSError`), not bugs.

---

### Practice 10.2: Project B

**A. Predict the output**

1. What does `transform(4, {"region": "North", "units": "2", "unit_price": "x"})` raise, and what is its message?
2. With `@retry(times=3, exceptions=(OSError,))`, what happens if the function raises `ValueError` on the first call?

**B. Write code**

3. Write `top_region_by_units(raw_csv)` that returns the region with the most units sold (valid rows only).
4. Write `dedupe_rows(rows)` that drops exact duplicate row dicts but keeps the first one, in order. Hint: turn each dict into a tuple of its items.
5. **[Hard]** Add a `max_errors` limit: write `run_strict(raw_csv, max_errors)` that raises `RowError("too many bad rows")` as soon as more than `max_errors` rows are rejected.

**C. Find and fix the bug**

6. Why is this retry decorator dangerous?
   ```python
   def retry(func):
       def wrapper(*args):
           while True:
               try:
                   return func(*args)
               except Exception:
                   pass
       return wrapper
   ```

**D. Interview questions**

7. Why is `Sale` a frozen dataclass?
8. Which errors should be retried, and which should not?

### Solutions 10.2

1. `RowError` with the message `line 4: ValueError: could not convert string to float: 'x'`.
2. The `ValueError` is not in `exceptions`, so it propagates at once; no retry.
6. It retries forever, hides every error (even bugs like `TypeError`), never waits between attempts, drops keyword arguments, and loses the function's name and docstring (no `functools.wraps`). Limit the attempts, catch only specific errors, and re-raise the last one.
7. A sale is a fact that should not change after validation. `frozen=True` prevents accidental changes and makes objects hashable.
8. Retry temporary problems (timeouts, connection errors, "service busy"). Do not retry bugs or permanent problems (`ValueError` from bad data, authentication failures), because the result will never change.

In [511]:
# Solutions 10.2 (B) with checks
import io
from collections import Counter

# 3
def top_region_by_units(raw_csv):
    units = Counter()
    for number, row in extract(io.StringIO(raw_csv)):
        try:
            sale = transform(number, row)
        except RowError:
            continue
        units[sale.region] += sale.units
    return units.most_common(1)[0][0] if units else None

assert top_region_by_units(raw) == "North"           # North 13 units, South 5
assert top_region_by_units("region,units,unit_price\n") is None

# 4
def dedupe_rows(rows):
    seen, out = set(), []
    for row in rows:
        key = tuple(sorted(row.items()))
        if key not in seen:
            seen.add(key)
            out.append(row)
    return out

rows = [{"a": "1"}, {"a": "2"}, {"a": "1"}]
assert dedupe_rows(rows) == [{"a": "1"}, {"a": "2"}]
assert dedupe_rows([]) == []

# 5
def run_strict(raw_csv, max_errors):
    errors, totals = 0, Counter()
    for number, row in extract(io.StringIO(raw_csv)):
        try:
            sale = transform(number, row)
        except RowError:
            errors += 1
            if errors > max_errors:
                raise RowError("too many bad rows") from None
            continue
        totals[sale.region] += sale.revenue
    return dict(totals)

assert run_strict(raw, 3) == {"North": 32.5, "South": 40.5}
try:
    run_strict(raw, 2)
    raise AssertionError("expected RowError")
except RowError as e:
    assert str(e) == "too many bad rows"
print("Practice 10.2: all checks passed")

Practice 10.2: all checks passed


## 10.4 Project C: an inventory manager

**Goal:** a small shop inventory. You can add products, restock, sell, see low-stock items and the total stock value, and save/load everything as JSON.

**Uses:** classes and properties (Part 1), dataclasses (Part 2), custom exception hierarchy (Notebook 1, Part 12), JSON and atomic file writing (Notebook 1, Part 13), tests (Part 9).

```text
  +--------------------------------+          +-------------------------+
  | Inventory                      |  holds   | Product (dataclass)     |
  |  _products: dict[sku, Product] |--------->|  sku, name, price,      |
  |  add_product() restock() sell()|  0..n    |  quantity               |
  |  low_stock()  total_value      |          |  validates itself       |
  |  save(path)   load(path)       |          +-------------------------+
  +--------------------------------+
         | raises
         v
  InventoryError
    +-- UnknownProductError
    +-- OutOfStockError
```

In [512]:
# Product and the exception hierarchy
from dataclasses import dataclass

class InventoryError(Exception):
    """Base class for all inventory problems."""

class UnknownProductError(InventoryError):
    pass

class OutOfStockError(InventoryError):
    pass

@dataclass
class Product:
    sku: str
    name: str
    price: float
    quantity: int = 0

    def __post_init__(self):                 # validate once, when the object is created
        if not self.sku:
            raise ValueError("sku must not be empty")
        if self.price < 0:
            raise ValueError("price must not be negative")
        if self.quantity < 0:
            raise ValueError("quantity must not be negative")

pen = Product("P-1", "Pen", 10.0, 5)
print(pen)                       # Product(sku='P-1', name='Pen', price=10.0, quantity=5)
try:
    Product("P-2", "Broken", -1.0)
except ValueError as e:
    print("ValueError:", e)      # ValueError: price must not be negative

Product(sku='P-1', name='Pen', price=10.0, quantity=5)
ValueError: price must not be negative


In [513]:
# The Inventory class with JSON persistence
import json
import os
import tempfile
from dataclasses import asdict
from pathlib import Path

class Inventory:
    def __init__(self):
        self._products = {}

    def add_product(self, product):
        if product.sku in self._products:
            raise InventoryError(f"duplicate sku: {product.sku}")
        self._products[product.sku] = product

    def _get(self, sku):
        try:
            return self._products[sku]
        except KeyError:
            raise UnknownProductError(f"unknown sku: {sku}") from None

    def restock(self, sku, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._get(sku).quantity += amount

    def sell(self, sku, amount):
        """Remove stock and return the price of the sale."""
        product = self._get(sku)
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > product.quantity:
            raise OutOfStockError(f"only {product.quantity} x {product.name} left")
        product.quantity -= amount
        return round(amount * product.price, 2)

    def low_stock(self, threshold=3):
        return sorted(p.sku for p in self._products.values() if p.quantity <= threshold)

    @property
    def total_value(self):
        return round(sum(p.price * p.quantity for p in self._products.values()), 2)

    def __len__(self):
        return len(self._products)

    def save(self, path):
        """Write JSON atomically: write a temp file, then replace the target."""
        path = Path(path)
        data = [asdict(p) for p in self._products.values()]
        fd, tmp = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        os.replace(tmp, path)                  # a crash never leaves a half-written file

    @classmethod
    def load(cls, path):
        inventory = cls()
        for item in json.loads(Path(path).read_text(encoding="utf-8")):
            inventory.add_product(Product(**item))   # Product validates the loaded data
        return inventory

shop = Inventory()
shop.add_product(Product("P-1", "Pen", 10.0, 5))
shop.add_product(Product("B-7", "Notebook", 45.5, 2))
print(shop.sell("P-1", 3))           # 30.0
print(shop.low_stock())              # ['B-7', 'P-1']
print(shop.total_value)              # 111.0
try:
    shop.sell("B-7", 10)
except OutOfStockError as e:
    print("OutOfStockError:", e)     # OutOfStockError: only 2 x Notebook left

30.0
['B-7', 'P-1']
111.0
OutOfStockError: only 2 x Notebook left


In [514]:
# Tests for the inventory, including saving and loading in a temp folder
import tempfile
import unittest
from pathlib import Path

class TestInventory(unittest.TestCase):
    def setUp(self):
        self.inv = Inventory()
        self.inv.add_product(Product("P-1", "Pen", 10.0, 5))

    def test_sell_reduces_stock_and_returns_price(self):
        self.assertEqual(self.inv.sell("P-1", 2), 20.0)
        self.assertEqual(self.inv.total_value, 30.0)

    def test_cannot_oversell(self):
        with self.assertRaises(OutOfStockError):
            self.inv.sell("P-1", 6)
        self.assertEqual(self.inv.total_value, 50.0)       # nothing changed

    def test_unknown_sku(self):
        with self.assertRaises(UnknownProductError):
            self.inv.restock("X-9", 1)

    def test_errors_share_a_base_class(self):
        with self.assertRaises(InventoryError):             # callers can catch all at once
            self.inv.sell("X-9", 1)

    def test_duplicate_sku(self):
        with self.assertRaisesRegex(InventoryError, "duplicate"):
            self.inv.add_product(Product("P-1", "Other pen", 1.0))

    def test_invalid_amounts(self):
        for bad in [0, -3]:
            with self.subTest(amount=bad):
                with self.assertRaises(ValueError):
                    self.inv.restock("P-1", bad)

    def test_save_and_load_round_trip(self):
        with tempfile.TemporaryDirectory() as folder:
            path = Path(folder) / "inventory.json"
            self.inv.save(path)
            loaded = Inventory.load(path)
        self.assertEqual(len(loaded), 1)
        self.assertEqual(loaded.total_value, self.inv.total_value)

result = run_tests(TestInventory)      # tests run: 7, OK (failures=0, errors=0)

tests run: 7, OK (failures=0, errors=0)


> **Key points**
> * A dataclass with `__post_init__` validates every object, including loaded ones.
> * One base exception (`InventoryError`) lets callers catch all inventory problems together.
> * Save with an atomic replace, and test persistence with a temporary folder.

---

### Practice 10.3: Project C

**A. Predict the output**

1. What does this print?
   ```python
   inv = Inventory()
   inv.add_product(Product("A", "Apple", 2.0, 10))
   inv.sell("A", 4)
   print(inv.total_value, inv.low_stock(5))
   ```
2. Is `OutOfStockError` caught by `except InventoryError:`?

**B. Write code**

3. Add a function `apply_discount(inventory, sku, percent)` that lowers one product's price (0-100%), using the inventory's `_get` helper, and raises `ValueError` for a bad percent.
4. Write `sell_many(inventory, orders)` that takes a list of `(sku, amount)` and sells all of them **or none** (if any sale would fail, nothing changes). Hint: check everything first.
5. **[Hard]** Write `class AuditedInventory(Inventory)` that records every successful sale as `(sku, amount, price)` in a `history` list, using `super().sell(...)`.

**C. Find and fix the bug**

6. Why is this `save` method risky?
   ```python
   def save(self, path):
       with open(path, "w") as f:
           f.write("[")
           for p in self._products.values():
               f.write(json.dumps(asdict(p)) + ",")
           f.write("]")
   ```

**D. Interview questions**

7. Why does `load` create `Product(**item)` instead of copying the dicts directly?
8. How would you change this design if many threads sell at the same time?

### Solutions 10.3

1. `12.0 []`. 6 apples left at 2.0 = 12.0; 6 is above the threshold 5, so the low-stock list is empty.
2. Yes: `OutOfStockError` is a subclass of `InventoryError`.
6. It writes invalid JSON (a trailing comma before `]`), uses no explicit encoding, and if the program crashes in the middle the old file is already destroyed. Use `json.dump` on a list, `encoding="utf-8"`, and write to a temp file then `os.replace` it.
7. So that loaded data goes through the same validation as new data (`__post_init__`), and the rest of the code always works with real `Product` objects.
8. Protect `sell` and `restock` with a `threading.Lock` (one shared lock in the inventory) so the check and the update happen together; otherwise two threads could both see enough stock and oversell (a race condition, see Part 8).

In [515]:
# Solutions 10.3 (B) with checks
# 3
def apply_discount(inventory, sku, percent):
    if not 0 <= percent <= 100:
        raise ValueError("percent must be between 0 and 100")
    product = inventory._get(sku)
    product.price = round(product.price * (100 - percent) / 100, 2)

inv = Inventory()
inv.add_product(Product("A", "Apple", 2.0, 10))
inv.add_product(Product("B", "Bread", 40.0, 1))
apply_discount(inv, "B", 25)
assert inv.total_value == 50.0                    # 2*10 + 30*1
try:
    apply_discount(inv, "A", 120)
    raise AssertionError("expected ValueError")
except ValueError:
    pass

# 4
def sell_many(inventory, orders):
    needed = {}
    for sku, amount in orders:                    # check everything first
        if amount <= 0:
            raise ValueError("amount must be positive")
        needed[sku] = needed.get(sku, 0) + amount
    for sku, amount in needed.items():
        product = inventory._get(sku)
        if amount > product.quantity:
            raise OutOfStockError(f"not enough {product.name}")
    return round(sum(inventory.sell(sku, amount) for sku, amount in orders), 2)

assert sell_many(inv, [("A", 2), ("A", 3)]) == 10.0
before = inv.total_value
try:
    sell_many(inv, [("A", 1), ("B", 5)])          # B has only 1: nothing may change
    raise AssertionError("expected OutOfStockError")
except OutOfStockError:
    pass
assert inv.total_value == before

# 5
class AuditedInventory(Inventory):
    def __init__(self):
        super().__init__()
        self.history = []

    def sell(self, sku, amount):
        price = super().sell(sku, amount)         # only reached if the sale succeeded
        self.history.append((sku, amount, price))
        return price

audited = AuditedInventory()
audited.add_product(Product("P", "Pen", 10.0, 3))
audited.sell("P", 2)
try:
    audited.sell("P", 5)
except OutOfStockError:
    pass
assert audited.history == [("P", 2, 20.0)]
print("Practice 10.3: all checks passed")

Practice 10.3: all checks passed


## 10.5 From script to project, and how to debug a whole program

### From script to project

To turn Project A into an installable tool (see Notebook 1, Part 14):

```text
loganalyzer/
    pyproject.toml
    README.md
    src/
        loganalyzer/
            __init__.py
            cli.py          <- main(argv), build_parser()
            parsing.py      <- parse_line, read_records, LogFormatError
            summary.py      <- summarize
    tests/
        test_parsing.py
        test_summary.py
        test_cli.py
```

```text
[project]
name = "loganalyzer"
version = "0.1.0"
requires-python = ">=3.12"

[project.scripts]
loganalyzer = "loganalyzer.cli:main"
```

Then: create a virtual environment, run `python -m pip install -e .` (editable install), run `python -m pytest`, `ruff check .` and `mypy src`, and set up the same commands in CI (a server that runs them on every change).

**Checklist before you call code "done":**

* Names are clear; functions are short and do one job.
* Inputs are validated at the boundary; errors use specific exception types.
* No swallowed errors (`except: pass`); diagnostics use `logging`, not `print`.
* No hard-coded secrets or machine-specific paths.
* Files and connections are closed with `with`.
* Tests cover normal, edge and error cases, and they pass.
* Public functions have docstrings and type hints.
* Dependencies are pinned; the README says how to install and run.

### How to debug a whole program

When a full program misbehaves, narrow the problem down step by step:

```text
  +-------------------------------+
  | Reproduce with the smallest   |
  | input that still fails        |
  +---------------+---------------+
                  |
                  v
  +-------------------------------+
  | Find the stage: check the     |
  | output after read, parse,     |
  | compute, write (one by one)   |
  +---------------+---------------+
                  |
                  v
  +-------------------------------+
  | Test that stage alone with    |
  | the bad input (unit test)     |
  +---------------+---------------+
                  |
                  v
  +-------------------------------+
  | Inspect values (f"{x=}",      |
  | logging, pdb); fix; keep the  |
  | test; run ALL tests again     |
  +-------------------------------+
```

In [516]:
# Debugging by stages. Symptom: a row with an extra column is accepted silently. Why?
import io

bad_csv = "region,units,unit_price\nNorth,7,2,EXTRA\n"

stage1 = list(extract(io.StringIO(bad_csv)))          # stage 1: what did extract produce?
print(f"{stage1=}")
# stage1=[(2, {'region': 'North', 'units': '7', 'unit_price': '2', None: ['EXTRA']})]

number, row = stage1[0]
sale = transform(number, row)                         # stage 2: transform never looks at key None
print(f"{sale=}")            # sale=Sale(region='North', units=7, unit_price=2.0)

# Found it: csv.DictReader stores extra values under the key None.
# Fix, if extra columns must be rejected: check for that key inside transform().
if None in row:
    print(RowError(f"line {number}: unexpected extra columns {row[None]}"))
# line 2: unexpected extra columns ['EXTRA']

stage1=[(2, {'region': 'North', 'units': '7', 'unit_price': '2', None: ['EXTRA']})]
sale=Sale(region='North', units=7, unit_price=2.0)
line 2: unexpected extra columns ['EXTRA']


**Typical errors in projects like these:**

| Message | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'loganalyzer'` | Wrong working folder or package not installed | Run from the project root; `pip install -e .` |
| `FileNotFoundError: [Errno 2] No such file or directory` | Relative path from a different working folder | Build paths with `pathlib` from a known base |
| `UnicodeDecodeError` | File read with the wrong encoding | Pass `encoding="utf-8"` (or the real encoding) |
| `json.decoder.JSONDecodeError` | Half-written or hand-edited JSON | Atomic writes; show the line and column from the error |
| `KeyError: 'units'` | A CSV column is missing or misspelled | Validate the header first; wrap in a clear custom error |
| `SystemExit: 2` in tests | `argparse` rejected the arguments | Check the argument list you passed to `main(argv)` |

> **Key points**
> * A project has a `src/` package, `tests/`, a `pyproject.toml` and a README.
> * Debug a pipeline stage by stage with the smallest failing input.
> * Every fixed bug gets a test.

---

### Practice 10.4: Capstone tasks

These tasks combine several Parts. Try each one before you look at the solution.

**B. Write code**

1. **Parallel files.** Write `analyze_many(paths, workers)` that summarises several log files concurrently with `ThreadPoolExecutor` (Part 8) and merges the results (add up the per-level counts and the error-source counts; keep the top 3).
2. **Streaming proof.** Generate a log with 50,000 lines in the project folder. Using `tracemalloc` (Part 9), show that `summarize(read_records(path))` has a much lower peak memory than `summarize(list(read_records(path)))`.
3. **Async prices.** Write `async def fetch_prices(skus)` that "fetches" a price for each SKU with `await asyncio.sleep(0.01)` (price = length of the SKU times 10), runs all fetches concurrently with a `TaskGroup`, and returns a dict.
4. **Package check.** Parse the `pyproject.toml` text from 10.5 with `tomllib` and check it has a console script named `loganalyzer`.
5. **[Hard] Thread-safe inventory.** Write `LockedInventory(Inventory)` whose `sell` uses a `threading.Lock`. Start 10 threads that each try to sell 1 unit of a product with 5 in stock. Exactly 5 sales must succeed.
6. **[Hard] CLI test with pytest.** Write a pytest file that runs `loganalyzer.main` with `--json` on a small log in `tmp_path` and checks the result. Run it with `python -m pytest -q` in a subprocess.

**D. Interview questions**

7. In task 1, why are threads a good choice? When would you switch to processes?
8. In task 5, what goes wrong without the lock?

### Solutions 10.4

7. Reading files is mostly waiting on the disk and `summarize` is light, so threads overlap the waiting with little overhead. If parsing becomes CPU-heavy (huge files, complex regexes), switch to `ProcessPoolExecutor`; the functions are importable because `loganalyzer` is a module.
8. Two threads can both check `amount <= quantity` before either subtracts, so both sell the last unit (a race condition) and the stock may go negative. The lock makes the check and the update one step.

The code below solves tasks 1-6 and checks the results. At the end it removes the temporary project folder.

In [517]:
# Solutions 10.4, tasks 1-2: parallel analysis and the streaming memory proof
import contextlib
import io
import tracemalloc
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

# 1
def analyze_many(paths, workers=4):
    def one(path):
        return la.summarize(la.read_records(path), "DEBUG", top=100)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        results = list(ex.map(one, paths))
    levels, sources = Counter(), Counter()
    for res in results:
        levels.update(res["by_level"])
        sources.update(dict(res["top_error_sources"]))
    return {"by_level": dict(levels), "top_error_sources": sources.most_common(3)}

second_log = proj_dir / "second.log"
second_log.write_text("2026-10-23 10:00:00 ERROR payments: again\n"
                      "2026-10-23 10:00:01 INFO web: ok\n", encoding="utf-8")
with contextlib.redirect_stderr(io.StringIO()):
    merged = analyze_many([sample_log, second_log])
assert merged["by_level"]["ERROR"] == 4 and merged["by_level"]["INFO"] == 2
assert merged["top_error_sources"][0] == ("payments", 3)

# 2
big_log = proj_dir / "big.log"
with open(big_log, "w", encoding="utf-8") as f:
    for i in range(50_000):
        level = "ERROR" if i % 10 == 0 else "INFO"
        f.write(f"2026-10-23 09:00:00 {level} svc{i % 5}: message number {i}\n")

def peak_kib(func):
    tracemalloc.start()
    func()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024

streamed = peak_kib(lambda: la.summarize(la.read_records(big_log)))
loaded = peak_kib(lambda: la.summarize(list(la.read_records(big_log))))
print("streaming uses far less memory:", loaded > 10 * streamed)   # True

streaming uses far less memory: True


In [518]:
# Solutions 10.4, tasks 3-4: async prices and the pyproject check
import asyncio
import tomllib

# 3
async def fetch_prices(skus):
    async def fetch(sku):
        await asyncio.sleep(0.01)
        return len(sku) * 10
    async with asyncio.TaskGroup() as tg:
        tasks = {sku: tg.create_task(fetch(sku)) for sku in skus}
    return {sku: task.result() for sku, task in tasks.items()}

assert await fetch_prices(["P-1", "BOOK-7"]) == {"P-1": 30, "BOOK-7": 60}
assert await fetch_prices([]) == {}

# 4
pyproject_text = """
[project]
name = "loganalyzer"
version = "0.1.0"
requires-python = ">=3.12"

[project.scripts]
loganalyzer = "loganalyzer.cli:main"
"""
config = tomllib.loads(pyproject_text)
assert config["project"]["scripts"]["loganalyzer"] == "loganalyzer.cli:main"
print("tasks 3-4: checks passed")

tasks 3-4: checks passed


In [519]:
# Solutions 10.4, task 5: a thread-safe inventory
import threading

class LockedInventory(Inventory):
    def __init__(self):
        super().__init__()
        self._lock = threading.Lock()

    def sell(self, sku, amount):
        with self._lock:                         # check and update as one step
            return super().sell(sku, amount)

locked = LockedInventory()
locked.add_product(Product("T", "Ticket", 100.0, 5))
outcomes = []
outcomes_lock = threading.Lock()

def buyer():
    try:
        locked.sell("T", 1)
        result = "sold"
    except OutOfStockError:
        result = "refused"
    with outcomes_lock:
        outcomes.append(result)

threads = [threading.Thread(target=buyer) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()
assert outcomes.count("sold") == 5 and outcomes.count("refused") == 5
assert locked.total_value == 0.0
print("task 5: checks passed")

task 5: checks passed


In [520]:
# Solutions 10.4, task 6: a pytest test for the CLI, run in a subprocess
import os
import re
import subprocess
import sys
import textwrap

(proj_dir / "test_cli.py").write_text(textwrap.dedent("""\
    import json
    import loganalyzer

    def test_json_output(tmp_path, capsys):
        log = tmp_path / "mini.log"
        log.write_text("2026-10-23 09:00:00 ERROR db: down\\n"
                       "2026-10-23 09:00:01 INFO web: up\\n", encoding="utf-8")
        assert loganalyzer.main([str(log), "--json"]) == 0
        data = json.loads(capsys.readouterr().out)
        assert data["by_level"] == {"ERROR": 1, "INFO": 1}
        assert data["top_error_sources"] == [["db", 1]]
"""), encoding="utf-8")

env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", PYTHONDONTWRITEBYTECODE="1")
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "test_cli.py"],
    cwd=proj_dir, capture_output=True, text=True, encoding="utf-8", env=env,
)
last_line = [line for line in proc.stdout.splitlines() if line.strip()][-1]
print(proc.returncode, re.sub(r" in [\d.]+s.*$", "", last_line).strip("= "))   # 0 1 passed

0 1 passed


In [521]:
# Clean up the Project A folder and module
import shutil
import sys

sys.path.remove(str(proj_dir))
sys.modules.pop("loganalyzer", None)
shutil.rmtree(proj_dir, ignore_errors=True)
print("project folder removed:", not proj_dir.exists())    # project folder removed: True
print("Practice 10.4: all checks passed")

project folder removed: True
Practice 10.4: all checks passed


### Summary of Part 10

* Build projects in small, tested pieces: read, parse, compute, output.
* Keep the core logic pure; put files, printing and the command line at the edges.
* Generators keep memory flat; `main(argv)` with an exit code makes CLIs testable.
* Custom exception hierarchies and `raise ... from` give clear, catchable errors.
* Decorators (retry) and context managers (stopwatch, atomic save) remove repeated code.
* Dataclasses with `__post_init__` validate data, including data loaded from JSON.
* A real project has `src/`, `tests/`, `pyproject.toml`, linters, type checks and CI.
* Debug a pipeline stage by stage, and keep a test for every bug you fix.

---

## What you learned in Notebook 2

You now know the tools that separate a beginner from a professional Python developer:

* **Object-oriented programming** (Parts 1-2): classes, properties, inheritance, special methods, dataclasses, enums and design principles.
* **Advanced functions and language features** (Parts 3-6): iterators and generators, closures and decorators, context managers, descriptors, typing and memory management.
* **The standard library toolbox** (Part 7): `collections`, `heapq`, `bisect`, `itertools`, `functools`, `datetime`, `re`, `logging`, `subprocess` and more.
* **Concurrency** (Part 8): threads, locks and queues, processes, and `asyncio`.
* **Professional practice** (Part 9): testing with `unittest`, `pytest` and mocks, code quality tools, secure coding and performance measurement.
* **Real projects** (Part 10): a CLI tool, an ETL pipeline and an inventory manager, with tests.

> **Tip:** Before you move on, rebuild one of the Part 10 projects from memory, in a real project folder with `src/` and `tests/`. Doing it without looking is the best test of what you learned.

## What next: Notebook 3

**Notebook 3: DSA and Interview Preparation** (`Python_Notes_3_DSA_and_Interviews.ipynb`) turns your Python skills into interview skills:

* a step-by-step framework for solving coding problems and talking through them,
* complexity analysis (Big-O) for every solution,
* the core patterns: arrays and hashing, two pointers, sliding window, recursion and backtracking, sorting, binary search,
* data structures: linked lists, stacks and queues, heaps, trees, tries and graphs,
* greedy algorithms, dynamic programming and bit manipulation,
* cheat sheets, code templates and mock interviews.

From Notebook 2 you will use most: generators (Part 3), `functools.cache` (Parts 4 and 7), and `collections`, `heapq`, `bisect` and `itertools` (Part 7).